# Notebook 06C — Hybrid Adaptive Quiz Generation


## Plan C v2 — Generic token/complexity-adaptive batching

Notebook 06C keeps the stable Notebook 06 quality-control pipeline and changes only how fresh quiz-generation work is grouped for model calls.

### Generic batching policy — no quiz-size mapping

There is **no rule such as 5 questions → 3+2**, no small/medium/large question-count table, and no topic-specific split.

For each next blueprint question, Notebook 06C temporarily adds it to the current candidate batch and calculates:

- estimated prompt/input tokens after batch-specific evidence compaction;
- dynamic output-token reservation for the selected model;
- average question complexity from marks, task family and visual/code demand;
- visually-heavy question fraction;
- higher-mark question fraction;
- a risk-adjusted safe token ceiling below model/provider limits.

If the candidate remains safe, the question stays in the current batch. If it becomes unsafe, the current batch closes and a new one starts. `LLM_MAX_GENERATION_BATCH_QUESTIONS` remains only a technical emergency ceiling, not a quiz-size mapping.

Therefore the **same 5-question request may become 5, 4+1, 3+2, 2+2+1, etc.** depending on the actual blueprint, prompt, model and provider budget. A 10-, 20- or 30-question request uses the same algorithm.

### Cost-control changes

- Initial batching is generic and greedy rather than count-bucket driven.
- Automatic learner-facing repair is limited to one normal corrective round.
- The separate structural-PDF repair loop defaults to **0 extra LLM rounds**; unresolved failures go to HITL.
- Marking-only repair is attempted automatically only when the detected failure is exclusively marking/answer-guidance related.
- Targeted question repair uses a compact repair prompt and a smaller dynamic output budget.
- Model-call audit records stage, trigger, plan indexes, prompt profile and whether the call is initial, repair or transport recovery.

All deterministic blueprint rules, AQA constraints, structural/semantic validators, special-instruction checks, answer verification, visual logic, HITL, PostgreSQL/Qdrant feedback memory, release gates and PDF generation remain unchanged.
### v3 cost-control correction

The previous generic planner still had one hidden non-generic bottleneck: a fixed 24K soft token ceiling. On large-context models such as GPT-5 mini (400K), that ceiling could fragment a 10-question quiz into many small initial batches even when the model context was mostly unused. v3 removes that fixed default and derives the batching ceiling from the selected model.

The planner now closes a batch for two generic reasons only: (1) model/provider token safety, or (2) predicted completion/output pressure approaching the configured generation ceiling. There is still no question-count mapping. Automatic LLM repair also has one shared budget across marking-only and learner-facing repair paths, and restored candidates are revalidated without silently spending another model call.


### v2.43.3 — generic validation-authoritative repair

Targeted regeneration now resolves contradictions through contract-level invariants rather than topic-specific rules. The deterministic blueprint remains immutable, while only the smallest invalid learner-facing fragment is rewritten when validation proves a contradiction. This change adds no topic name, syllabus concept, question ID, or specific visual-type pair.


### v2.43.5 — supplied visual vs learner response scaffold

Generic referential validation now distinguishes the visual stimulus supplied by the renderer from a learner response scaffold/output format. Only strong learner-facing references that explicitly present an object as supplied/shown/provided stimulus are compared with `visual_requirement`. A bare instruction to complete a response object is no longer assumed to identify the renderer-supplied visual. No topic, syllabus concept, question ID, or visual-type pair is hardcoded.


### v2.44 — truth-table response scaffolds + render-safe logic-gate tasks

- If learner wording says **complete/fill in the truth table**, Notebook 06C deterministically adds a blank truth-table response scaffold when the primary supplied visual is something else (for example a logic circuit). Output-answer cells remain blank and are rendered in the student PDF.
- LLM generation is forbidden from creating **L1/L2/L3/L4, G1/G2, or equivalent unknown gate-position tasks** where the learner must assign gate types to labelled slots. A deterministic validator rejects any such output and routes it through normal targeted regeneration/HITL instead.
- Fully specified logic circuits remain allowed for truth-table completion, Boolean-expression interpretation, and output-evaluation questions.


### v2.43.4 — targeted-regeneration context transport fix

Targeted repair context now survives request compaction into provider batches. Only the active failed plan indexes and their original-question context are carried into that repair batch. This is a generic transport-layer fix; no topic, question ID, or visual-type pair is hardcoded.


# Model selection history — GPT-OSS 20B test evidence

Before selecting Gemini 3.5 Flash, **Notebook 06 was tested with
`openai/gpt-oss-20b` through the Groq on-demand service tier**. The quiz
blueprint and generation flow reached the model API, but repeated provider
quota/throughput limits blocked reliable end-to-end quiz generation.

The following failures were observed during the actual Agent 2 tests:

1. **Daily token limit (TPD) — HTTP 429**
   - Model: `openai/gpt-oss-20b`
   - Organization: `org_01kvjdassdf8xb1qyaz7a1j1gt`
   - TPD limit: **200,000**
   - Used: **198,135**
   - Requested: **16,136**
   - Groq response: retry after approximately **1h 42m 45s**

2. **Daily token limit after another run/restart — HTTP 429**
   - TPD limit: **200,000**
   - Used: **199,272**
   - Requested: **16,136**
   - Groq response: retry after approximately **1h 50m 56s**

3. **Single-request tokens-per-minute limit — HTTP 413**
   - TPM limit: **8,000**
   - Requested: **10,305**
   - Groq response: `Request too large for model openai/gpt-oss-20b`

4. **After prompt compaction/evidence de-duplication — HTTP 429**
   - TPD limit: **200,000**
   - Used: **198,633**
   - Requested: **6,064**
   - Groq response: retry after approximately **33m 49s**

This showed that prompt compaction successfully reduced the request size, but
the **Groq free/on-demand quota remained the runtime bottleneck**. These were
provider/service-tier quota failures, not failures in the Agent 1 handoff,
deterministic quiz blueprint, topic coverage logic, or structural validators.

That migration established the provider-neutral transport used now. Notebook 06 can
benchmark **Gemini 3.5 Flash**, **GPT-OSS 120B / Groq**, and **GPT-5 mini** without
changing the existing Agent 2 workflow: deterministic blueprint planning, exact
question/mark constraints, multi-topic coverage, evidence de-duplication,
dynamic batching, structural validation, semantic validation, human review,
regeneration, and final release checks are unchanged.

### Gemini runtime setup

Add the Gemini key to `Agent2/.env`:

```text
GEMINI_API_KEY=<your Gemini API key>
```

Notebook 06 uses the official Google GenAI Python SDK:

```text
pip install -U google-genai
```

The default thinking level is `medium`, and it can be changed without editing
the notebook by setting:

```text
AGENT2_GEMINI_THINKING_LEVEL=medium
```

### OpenAI runtime setup

For GPT-5 mini or GPT-5.4 mini, add the OpenAI API key to `Agent2/.env`:

```text
OPENAI_API_KEY=<your OpenAI API key>
```

Notebook 06 uses the official OpenAI Python SDK:

```text
pip install -U openai
```

Model choice in Streamlit is saved to the current run's
`integration/quiz_model_selection.json`; no model ID needs to be edited manually in `.env`.


## Why the previous run showed `EMPTY / 0 marks / 0 questions`

In the failed run, the Google GenAI API returned `400 INVALID_ARGUMENT`
**during the Gemini generation request itself**. Notebook 06 therefore never
received a generated quiz payload and never reached the point where
`generated_questions.json` could be saved.

For that run, the frontend's `EMPTY / 0 marks / 0 questions` state reflected
the real backend state: **no candidate questions existed yet**.

The generation transport now uses a deliberately shallow Gemini response
schema and retries once in JSON MIME-only mode if Gemini rejects the structured
schema. All exact quiz requirements remain enforced afterwards by Notebook 06's
deterministic validators.

## Codebase-grounded AQA difficulty calibration — v2.11

For `complete_quiz`, Notebook 06 now performs a **read-only** lookup of a small
set of reviewed questions already stored in the Agent 2 assessment question
bank for the same official AQA reference and similar mark value. Those examples
are sent to the selected generation model only as compact difficulty/style calibration. They are not
used as question templates and Notebook 05 is still not called.

The existing deterministic similarity checks remain active, so a generated
question that is too close to a supplied bank example is failed/warned rather
than silently accepted. The examples are de-duplicated before LLM transport and
bounded to two compact examples per official reference to keep prompt tokens
small. No additional LLM call is introduced by this calibration step.


## Task-family refinement — v2.12

The complete-quiz grounding now calibrates each blueprint slot against the existing
Agent 2 question bank by **official reference + marks + task family**, rather than
reference/marks alone.  Task-family classification is deterministic Python; it adds
no LLM call and does not send the full question bank to the selected generation model.

Additional deterministic safeguards now check:

- one-mark questions for excessive multi-part demand,
- unsupported implementation/runtime jargon that is absent from lesson/style evidence,
- excessive repetition of the same learner task family, and
- collapsed multi-statement code/pseudocode formatting.

These checks do not change Agent 1 topic handoff, Notebook 05 routing, mark allocation,
HITL, visual handling, or the existing release flow. Visual architecture remains a
separate next step.



## Config-driven model selection — v2.19

Notebook 06 now uses a single model registry in `Agent2/config/quiz_model_config.json`.
The current run selects **Gemini 3.5 Flash**, **GPT-OSS 120B / Groq**, or **GPT-5 mini**
without maintaining separate Notebook 06 files.

Before every provider request, Notebook 06 performs a conservative local token
preflight. If estimated input plus reserved output would exceed the selected model context
window, generation stops locally and prints the token estimate + model limit; **no API
request is sent**. A configured provider/service-tier TPM ceiling is handled separately
and may still use local batching/output-budget reduction. Every actual provider request
is counted as an API hit, and every provider response records
input/output/reasoning/total token usage in
`model_call_usage.json` and the final quiz manifest.

All existing deterministic blueprint, topic coverage, validation, HITL,
visual architecture, regeneration, and automatic questions + marking-scheme
PDF behavior remains unchanged.


## 1. Imports, project paths and execution controls

In [ ]:
from __future__ import annotations

import ast
import hashlib
import json
import math
import os
import re
import time

from datetime import datetime, timezone
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any

import pandas as pd
import numpy as np
from IPython.display import display

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None


# ================================================================
# PROJECT ROOT
# ================================================================

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name.casefold() in {"notebooks", "notebook"}:
    AGENT2_ROOT = CURRENT_DIR.parent
else:
    AGENT2_ROOT = Path(
        os.getenv("AGENT2_PROJECT_ROOT", str(CURRENT_DIR))
    ).expanduser().resolve()

if load_dotenv is not None:
    for env_file in [
        AGENT2_ROOT / ".env",
        AGENT2_ROOT.parent / ".env",
    ]:
        if env_file.is_file():
            load_dotenv(env_file, override=False)

OUTPUT_DIR = Path(
    os.getenv(
        "AGENT2_QUIZ_OUTPUT_DIR",
        str(AGENT2_ROOT / "OUTPUT" / "notebook_06_quiz"),
    )
).expanduser().resolve()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ================================================================
# SMALL ENV HELPERS
# ================================================================

def env_flag(name: str, default: bool = False) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return bool(default)

    return str(raw).strip().casefold() in {
        "1",
        "true",
        "yes",
        "on",
    }


def env_json(
    name: str,
    default: Any,
) -> Any:
    raw = str(
        os.getenv(name, "")
        or ""
    ).strip()

    if not raw:
        return default

    return json.loads(raw)


# ================================================================
# MODE
# ================================================================
# complete_quiz  -> Generate Complete Quiz button.
# fill_shortfall -> Generate Missing Quiz Coverage with AI button.
# ================================================================

QUIZ_MODE = "complete_quiz"

env_mode = str(
    os.getenv("AGENT2_QUIZ_MODE", "")
    or ""
).strip()

if env_mode:
    QUIZ_MODE = env_mode

QUIZ_MODE = QUIZ_MODE.strip().casefold()

VALID_QUIZ_MODES = {
    "complete_quiz",
    "fill_shortfall",
}

if QUIZ_MODE not in VALID_QUIZ_MODES:
    raise ValueError(
        f"QUIZ_MODE must be one of {sorted(VALID_QUIZ_MODES)}"
    )


# ================================================================
# HUMAN / EXECUTION SWITCHES
# ================================================================
# Safe notebook defaults are OFF.
# Streamlit can override them with environment variables.
# ================================================================

USER_APPROVED_GENERATION = env_flag(
    "AGENT2_QUIZ_USER_APPROVED_GENERATION",
    False,
)

RUN_GENERATION = env_flag(
    "AGENT2_QUIZ_RUN_GENERATION",
    False,
)

HUMAN_REVIEW_DECISION = str(
    os.getenv(
        "AGENT2_QUIZ_REVIEW_DECISION",
        "pending",
    )
    or "pending"
).strip().casefold()

HUMAN_REVIEW_REASON = str(
    os.getenv(
        "AGENT2_QUIZ_REVIEW_REASON",
        "",
    )
    or ""
).strip()

# Optional question-level HITL actions. Streamlit can pass a compact JSON list
# directly or write a JSON file and provide its path. These are deliberately
# optional so the existing whole-quiz approve/regenerate/reject contract remains
# backward compatible.
HUMAN_REVIEW_ACTIONS_JSON_RAW = str(
    os.getenv(
        "AGENT2_QUIZ_REVIEW_ACTIONS_JSON",
        "",
    )
    or ""
).strip()

HUMAN_REVIEW_ACTIONS_PATH = str(
    os.getenv(
        "AGENT2_QUIZ_REVIEW_ACTIONS_PATH",
        "",
    )
    or ""
).strip()

# Question-level HITL persistence/memory. PostgreSQL is the source of truth;
# Qdrant is an optional semantic retrieval index for future targeted
# regenerations. Both are fail-soft so a temporary memory-store outage never
# destroys the human decision: JSONL audit remains the local fallback.
HITL_FEEDBACK_DB_ENABLED = env_flag(
    "AGENT2_HITL_FEEDBACK_DB_ENABLED",
    True,
)

HITL_MEMORY_QDRANT_ENABLED = env_flag(
    "AGENT2_HITL_MEMORY_QDRANT_ENABLED",
    True,
)

HITL_MEMORY_QDRANT_URL = str(
    os.getenv("QDRANT_URL", os.getenv("AGENT2_QDRANT_URL", ""))
    or ""
).strip()

if not HITL_MEMORY_QDRANT_URL:
    _hitl_qdrant_host = str(
        os.getenv("QDRANT_HOST", "")
        or ""
    ).strip()
    _hitl_qdrant_port = str(
        os.getenv("QDRANT_PORT", "6333")
        or "6333"
    ).strip()
    if _hitl_qdrant_host:
        HITL_MEMORY_QDRANT_URL = (
            f"http://{_hitl_qdrant_host}:{_hitl_qdrant_port}"
        )

if (
    HITL_MEMORY_QDRANT_URL
    and "://" not in HITL_MEMORY_QDRANT_URL
):
    HITL_MEMORY_QDRANT_URL = "http://" + HITL_MEMORY_QDRANT_URL

HITL_MEMORY_QDRANT_API_KEY = str(
    os.getenv("QDRANT_API_KEY", os.getenv("AGENT2_QDRANT_API_KEY", ""))
    or ""
).strip()

HITL_MEMORY_COLLECTION = str(
    os.getenv(
        "AGENT2_HITL_MEMORY_COLLECTION",
        "agent2_generation_feedback_memory_v1",
    )
    or "agent2_generation_feedback_memory_v1"
).strip()

HITL_MEMORY_TOP_K = max(
    1,
    int(
        os.getenv(
            "AGENT2_HITL_MEMORY_TOP_K",
            "8",
        )
    ),
)

HITL_MEMORY_MAX_PROMPT_HINTS = max(
    1,
    int(
        os.getenv(
            "AGENT2_HITL_MEMORY_MAX_PROMPT_HINTS",
            "2",
        )
    ),
)

HITL_MEMORY_MIN_SIMILARITY = min(
    1.0,
    max(
        0.0,
        float(
            os.getenv(
                "AGENT2_HITL_MEMORY_MIN_SIMILARITY",
                "0.78",
            )
        ),
    ),
)

# Qdrant now supplies a broader same-reference candidate pool. Selection is
# then performed locally using current validator issues + action compatibility,
# rather than blindly taking the highest semantic-similarity point.
HITL_MEMORY_CANDIDATE_MIN_SIMILARITY = min(
    1.0,
    max(
        0.0,
        float(
            os.getenv(
                "AGENT2_HITL_MEMORY_CANDIDATE_MIN_SIMILARITY",
                "0.55",
            )
        ),
    ),
)

HITL_MEMORY_MIN_ISSUE_SIMILARITY = min(
    1.0,
    max(
        0.0,
        float(
            os.getenv(
                "AGENT2_HITL_MEMORY_MIN_ISSUE_SIMILARITY",
                "0.35",
            )
        ),
    ),
)

HITL_MEMORY_COMPATIBILITY_THRESHOLD = min(
    1.0,
    max(
        0.0,
        float(
            os.getenv(
                "AGENT2_HITL_MEMORY_COMPATIBILITY_THRESHOLD",
                "0.55",
            )
        ),
    ),
)

MAX_HUMAN_TARGETED_REGENERATION_ATTEMPTS = max(
    1,
    int(
        os.getenv(
            "AGENT2_MAX_HUMAN_TARGETED_REGENERATION_ATTEMPTS",
            "2",
        )
    ),
)


# ================================================================
# MODEL / QUALITY POLICY — CONFIG DRIVEN
# ================================================================
# One Notebook 06 supports every configured generation model.
#
# Selection priority:
# 1) AGENT2_QUIZ_MODEL_KEY environment override,
# 2) current-run integration/quiz_model_selection.json written by Streamlit,
# 3) default_model_key in Agent2/config/quiz_model_config.json.
#
# Adding a future model only requires a config entry plus a provider adapter
# if the provider itself is new. Gemini, Groq, and OpenAI are supported here.
# ================================================================

MODEL_CONFIG_PATH = Path(
    os.getenv(
        "AGENT2_QUIZ_MODEL_CONFIG_PATH",
        str(
            AGENT2_ROOT
            / "config"
            / "quiz_model_config.json"
        ),
    )
).expanduser().resolve()


def _default_quiz_model_config() -> dict[str, Any]:
    """Safe fallback used only if the external config file is missing."""
    return {
        "schema_version": "agent2-quiz-model-config-v1.2.0",
        "default_model_key": "gemini_3_5_flash",
        "models": {
            "gemini_3_5_flash": {
                "display_name": "Gemini 3.5 Flash",
                "provider": "google_gemini",
                "model_id": "gemini-3.5-flash",
                "api_key_env": "GEMINI_API_KEY",
                "context_window_tokens": 1_048_576,
                "hard_max_output_tokens": 65_536,
                "generation_output_min_tokens": 8_192,
                "generation_output_max_tokens": 16_384,
                "thinking_level": "medium",
                "transient_retry_max_attempts": 5,
                "transient_retry_initial_backoff_seconds": 5,
                "transient_retry_max_backoff_seconds": 40,
                "preflight_chars_per_token": 3.0,
                "preflight_safety_multiplier": 1.15,
                "provider_tpm_limit_tokens": None,
                "provider_tpm_limit_env": "GEMINI_TPM_LIMIT_TOKENS",
            },
            "gpt_oss_120b_groq": {
                "display_name": "GPT-OSS 120B / Groq",
                "provider": "groq",
                "model_id": "openai/gpt-oss-120b",
                "api_key_env": "GROQ_API_KEY",
                "context_window_tokens": 131_072,
                "hard_max_output_tokens": 65_536,
                "generation_output_min_tokens": 1_024,
                "generation_output_max_tokens": 2_048,
                "reasoning_effort": "medium",
                "transient_retry_max_attempts": 5,
                "transient_retry_initial_backoff_seconds": 5,
                "transient_retry_max_backoff_seconds": 40,
                "preflight_chars_per_token": 3.0,
                "preflight_safety_multiplier": 1.15,
                "provider_tpm_limit_tokens": 8_000,
                "provider_tpm_limit_env": "GROQ_TPM_LIMIT_TOKENS",
            },
            "gpt_5_mini": {
                "display_name": "GPT-5 mini",
                "provider": "openai",
                "model_id": "gpt-5-mini",
                "api_key_env": "OPENAI_API_KEY",
                "context_window_tokens": 400_000,
                "hard_max_output_tokens": 128_000,
                "generation_output_min_tokens": 8_192,
                "generation_output_max_tokens": 16_384,
                "reasoning_effort": "medium",
                "transient_retry_max_attempts": 5,
                "transient_retry_initial_backoff_seconds": 5,
                "transient_retry_max_backoff_seconds": 40,
                "preflight_chars_per_token": 3.0,
                "preflight_safety_multiplier": 1.15,
                "provider_tpm_limit_tokens": None,
                "provider_tpm_limit_env": "OPENAI_TPM_LIMIT_TOKENS",
            },
            "gpt_5_4_mini": {
                "display_name": "GPT-5.4 mini",
                "provider": "openai",
                "model_id": "gpt-5.4-mini",
                "api_key_env": "OPENAI_API_KEY",
                "context_window_tokens": 400_000,
                "hard_max_output_tokens": 128_000,
                "generation_output_min_tokens": 8_192,
                "generation_output_max_tokens": 16_384,
                "reasoning_effort": "medium",
                "transient_retry_max_attempts": 5,
                "transient_retry_initial_backoff_seconds": 5,
                "transient_retry_max_backoff_seconds": 40,
                "preflight_chars_per_token": 3.0,
                "preflight_safety_multiplier": 1.15,
                "provider_tpm_limit_tokens": None,
                "provider_tpm_limit_env": "OPENAI_TPM_LIMIT_TOKENS",
            },
        },
    }


def _load_quiz_model_config() -> dict[str, Any]:
    if MODEL_CONFIG_PATH.is_file():
        payload = json.loads(
            MODEL_CONFIG_PATH.read_text(
                encoding="utf-8"
            )
        )
    else:
        payload = _default_quiz_model_config()
        print(
            "WARNING: quiz model config file was not found; "
            "using the built-in four-model fallback:",
            MODEL_CONFIG_PATH,
        )

    if not isinstance(payload, dict):
        raise ValueError(
            "Quiz model config must be a JSON object."
        )

    models = payload.get("models")
    if not isinstance(models, dict) or not models:
        raise ValueError(
            "Quiz model config must contain a non-empty 'models' object."
        )

    return payload


QUIZ_MODEL_CONFIG = _load_quiz_model_config()
QUIZ_MODEL_REGISTRY = QUIZ_MODEL_CONFIG["models"]


def _current_run_model_selection_key() -> str:
    env_key = str(
        os.getenv(
            "AGENT2_QUIZ_MODEL_KEY",
            "",
        )
        or ""
    ).strip()

    if env_key:
        return env_key

    candidate_paths: list[Path] = []

    explicit_path = str(
        os.getenv(
            "AGENT2_QUIZ_MODEL_SELECTION_PATH",
            "",
        )
        or ""
    ).strip()

    if explicit_path:
        candidate_paths.append(
            Path(
                explicit_path
            ).expanduser()
        )

    for parent in [
        OUTPUT_DIR.parent,
        OUTPUT_DIR.parent.parent,
        OUTPUT_DIR.parent.parent.parent,
    ]:
        candidate_paths.append(
            parent
            / "integration"
            / "quiz_model_selection.json"
        )

    seen_paths: set[str] = set()

    for candidate in candidate_paths:
        try:
            resolved = candidate.resolve()
        except OSError:
            resolved = candidate

        path_key = str(resolved).casefold()
        if path_key in seen_paths:
            continue
        seen_paths.add(path_key)

        if not resolved.is_file():
            continue

        try:
            payload = json.loads(
                resolved.read_text(
                    encoding="utf-8"
                )
            )
        except (
            OSError,
            json.JSONDecodeError,
        ):
            continue

        if not isinstance(payload, dict):
            continue

        selected = str(
            payload.get(
                "model_key",
                "",
            )
            or ""
        ).strip()

        if selected:
            return selected

    return str(
        QUIZ_MODEL_CONFIG.get(
            "default_model_key",
            "",
        )
        or ""
    ).strip()


SELECTED_MODEL_KEY = _current_run_model_selection_key()

if SELECTED_MODEL_KEY not in QUIZ_MODEL_REGISTRY:
    raise ValueError(
        "Unknown quiz model key "
        f"{SELECTED_MODEL_KEY!r}. Available models: "
        f"{sorted(QUIZ_MODEL_REGISTRY)}"
    )

ACTIVE_MODEL_CONFIG = dict(
    QUIZ_MODEL_REGISTRY[
        SELECTED_MODEL_KEY
    ]
)

GENERATION_PROVIDER = str(
    ACTIVE_MODEL_CONFIG.get(
        "provider",
        "",
    )
    or ""
).strip().casefold()

GENERATION_MODEL = str(
    ACTIVE_MODEL_CONFIG.get(
        "model_id",
        "",
    )
    or ""
).strip()

GENERATION_MODEL_DISPLAY_NAME = str(
    ACTIVE_MODEL_CONFIG.get(
        "display_name",
        GENERATION_MODEL,
    )
    or GENERATION_MODEL
).strip()

if not GENERATION_PROVIDER or not GENERATION_MODEL:
    raise ValueError(
        "Selected quiz model is missing provider/model_id in the config."
    )

# Prevent a stale legacy model override from silently mismatching the selected
# model's context limits. Matching values remain backwards compatible.
legacy_model_override = str(
    os.getenv(
        "AGENT2_GENERATION_MODEL",
        "",
    )
    or ""
).strip()

if (
    legacy_model_override
    and legacy_model_override
    != GENERATION_MODEL
):
    raise ValueError(
        "AGENT2_GENERATION_MODEL does not match the model selected by "
        "quiz_model_config.json. Remove the legacy override or select the "
        "matching config model."
    )

ACTIVE_MODEL_CONTEXT_WINDOW_TOKENS = int(
    ACTIVE_MODEL_CONFIG.get(
        "context_window_tokens",
        0,
    )
    or 0
)

ACTIVE_MODEL_HARD_MAX_OUTPUT_TOKENS = int(
    ACTIVE_MODEL_CONFIG.get(
        "hard_max_output_tokens",
        0,
    )
    or 0
)

ACTIVE_MODEL_MIN_GENERATION_OUTPUT_TOKENS = int(
    ACTIVE_MODEL_CONFIG.get(
        "generation_output_min_tokens",
        8192,
    )
    or 8192
)

ACTIVE_MODEL_MAX_GENERATION_OUTPUT_TOKENS = int(
    ACTIVE_MODEL_CONFIG.get(
        "generation_output_max_tokens",
        16384,
    )
    or 16384
)

if (
    ACTIVE_MODEL_CONTEXT_WINDOW_TOKENS <= 0
    or ACTIVE_MODEL_HARD_MAX_OUTPUT_TOKENS <= 0
):
    raise ValueError(
        "Selected model must define positive context_window_tokens and "
        "hard_max_output_tokens."
    )

if (
    ACTIVE_MODEL_MIN_GENERATION_OUTPUT_TOKENS <= 0
    or ACTIVE_MODEL_MAX_GENERATION_OUTPUT_TOKENS
    < ACTIVE_MODEL_MIN_GENERATION_OUTPUT_TOKENS
):
    raise ValueError(
        "Invalid generation output-token limits in quiz model config."
    )

if (
    ACTIVE_MODEL_MAX_GENERATION_OUTPUT_TOKENS
    > ACTIVE_MODEL_HARD_MAX_OUTPUT_TOKENS
):
    raise ValueError(
        "Configured Notebook 06 output ceiling exceeds the selected model's "
        "hard max output tokens."
    )

MODEL_PREFLIGHT_CHARS_PER_TOKEN = max(
    2.0,
    float(
        ACTIVE_MODEL_CONFIG.get(
            "preflight_chars_per_token",
            3.0,
        )
        or 3.0
    ),
)

MODEL_PREFLIGHT_SAFETY_MULTIPLIER = max(
    1.0,
    float(
        ACTIVE_MODEL_CONFIG.get(
            "preflight_safety_multiplier",
            1.15,
        )
        or 1.15
    ),
)

# Optional account/service-tier TPM ceiling.
# This is separate from model context. Example: GPT-OSS 120B can have a
# 131K context window while the current Groq on-demand organization allows
# only 8000 tokens/minute.
_PROVIDER_TPM_LIMIT_ENV = str(
    ACTIVE_MODEL_CONFIG.get(
        "provider_tpm_limit_env",
        "",
    )
    or ""
).strip()

_provider_tpm_env_value = (
    str(os.getenv(_PROVIDER_TPM_LIMIT_ENV, "") or "").strip()
    if _PROVIDER_TPM_LIMIT_ENV
    else ""
)

if _provider_tpm_env_value:
    ACTIVE_MODEL_PROVIDER_TPM_LIMIT_TOKENS = int(
        _provider_tpm_env_value
    )
else:
    _configured_provider_tpm = ACTIVE_MODEL_CONFIG.get(
        "provider_tpm_limit_tokens"
    )
    ACTIVE_MODEL_PROVIDER_TPM_LIMIT_TOKENS = (
        int(_configured_provider_tpm)
        if _configured_provider_tpm not in {None, "", 0, "0"}
        else None
    )

if (
    ACTIVE_MODEL_PROVIDER_TPM_LIMIT_TOKENS is not None
    and ACTIVE_MODEL_PROVIDER_TPM_LIMIT_TOKENS <= 0
):
    raise ValueError(
        "provider_tpm_limit_tokens must be positive when configured."
    )

ACTIVE_MODEL_API_KEY_ENV = str(
    ACTIVE_MODEL_CONFIG.get(
        "api_key_env",
        "",
    )
    or ""
).strip()

# Quality checking after generation is deterministic Python + mandatory HITL.
# No second LLM semantic-review request is made.
SEMANTIC_REVIEW_MODEL = None
SEMANTIC_REVIEW_ENABLED = False

GEMINI_THINKING_LEVEL = str(
    os.getenv(
        "AGENT2_GEMINI_THINKING_LEVEL",
        ACTIVE_MODEL_CONFIG.get(
            "thinking_level",
            "medium",
        ),
    )
    or "medium"
).strip().casefold()

if GEMINI_THINKING_LEVEL not in {
    "minimal",
    "low",
    "medium",
    "high",
}:
    raise ValueError(
        "Gemini thinking level must be one of: "
        "minimal, low, medium, high."
    )

GROQ_REASONING_EFFORT = str(
    os.getenv(
        "AGENT2_GROQ_REASONING_EFFORT",
        ACTIVE_MODEL_CONFIG.get(
            "reasoning_effort",
            "medium",
        ),
    )
    or "medium"
).strip().casefold()

if GROQ_REASONING_EFFORT not in {
    "low",
    "medium",
    "high",
}:
    raise ValueError(
        "Groq reasoning effort must be one of: low, medium, high."
    )

OPENAI_REASONING_EFFORT = str(
    os.getenv(
        "AGENT2_OPENAI_REASONING_EFFORT",
        ACTIVE_MODEL_CONFIG.get(
            "reasoning_effort",
            "medium",
        ),
    )
    or "medium"
).strip().casefold()

if OPENAI_REASONING_EFFORT not in {
    "minimal",
    "low",
    "medium",
    "high",
}:
    raise ValueError(
        "OpenAI reasoning effort must be one of: minimal, low, medium, high."
    )

MODEL_CALL_USAGE_PATH = (
    OUTPUT_DIR
    / "model_call_usage.json"
)

MODEL_CALL_USAGE_ROWS: list[
    dict[str, Any]
] = []

MODEL_API_HIT_ROWS: list[
    dict[str, Any]
] = []


# One corrective generation after the initial generation.
MAX_REGENERATION_ATTEMPTS = 1

# FAIL may automatically consume the single corrective attempt.
AUTO_REGENERATE_ON_FAIL = True

# Relative mark-allocation importance.
#
# These are general role weights, not fixed percentages. The allocator uses
# them proportionally, so primary questions receive greater emphasis without
# starving supporting questions of meaningful marks.
ROLE_WEIGHTS = {
    "primary": 2.0,
    "supporting": 1.0,
}

# Quiz-wide cognitive/task patterns used by the deterministic blueprint.
#
# These are topic-neutral assessment intents, not hardcoded topic mappings.
# The planner distributes them across the whole quiz so repeated topics do not
# default to the same "execute code and state the final value" template.
# If a quiz contains more questions than this pool, the cycle can repeat.
# These patterns are diversity TARGETS, not hard uniqueness constraints.
# Python only flags strong near-duplicate templates; it does not reject a
# question merely because a broad assessment pattern is reused when that is
# appropriate for the topic.
ASSESSMENT_PATTERN_CYCLE = (
    "apply_or_predict",
    "explain_or_reason",
    "analyse_or_interpret",
    "compare_or_select",
    "diagnose_or_correct",
    "construct_or_complete",
    "scenario_application",
    "predict_consequence",
    "evaluate_or_justify",
    "adapt_or_modify",
    "classify_or_decide",
    "multi_step_synthesis",
)


# ================================================================
# PROVIDER-NEUTRAL LLM TRANSPORT / BATCHING POLICY
# ================================================================
# These are technical transport controls only. They do NOT hardcode or
# constrain the user's quiz choices. The deterministic quiz blueprint,
# question count, marks, roles, topic coverage, code/visual choices, and
# paper/language filters remain driven by the frontend request.
#
# The selected model context is read from quiz_model_config.json. We still keep
# evidence de-duplication and dynamic batching because they reduce latency,
# quota usage, and repeated context.
# ================================================================

LLM_ESTIMATED_CHARS_PER_TOKEN = max(
    2.5,
    float(
        os.getenv(
            "AGENT2_LLM_ESTIMATED_CHARS_PER_TOKEN",
            "4.0",
        )
    ),
)

LLM_TOPIC_EVIDENCE_CHAR_BUDGET = max(
    600,
    int(
        os.getenv(
            "AGENT2_LLM_TOPIC_EVIDENCE_CHAR_BUDGET",
            "1400",
        )
    ),
)

LLM_MAX_GENERATION_BATCH_QUESTIONS = max(
    1,
    int(
        os.getenv(
            "AGENT2_LLM_MAX_GENERATION_BATCH_QUESTIONS",
            "12",
        )
    ),
)

LLM_MAX_REVIEW_BATCH_QUESTIONS = max(
    1,
    int(
        os.getenv(
            "AGENT2_LLM_MAX_REVIEW_BATCH_QUESTIONS",
            "5",
        )
    ),
)

# Model-aware soft request-size guard.
#
# IMPORTANT: the previous fixed 24K default was the main source of excessive
# initial API calls on large-context models. A user may still override this via
# AGENT2_LLM_BATCH_SOFT_TOKEN_LIMIT, but the default now scales with the
# selected model instead of pretending every provider has a ~24K practical
# request ceiling. Provider TPM limits are still enforced separately below.
try:
    _raw_batch_soft_limit = int(
        str(
            os.getenv(
                "AGENT2_LLM_BATCH_SOFT_TOKEN_LIMIT",
                "0",
            )
            or "0"
        ).strip()
    )
except ValueError as exc:
    raise ValueError(
        "AGENT2_LLM_BATCH_SOFT_TOKEN_LIMIT must be an integer token count."
    ) from exc

if _raw_batch_soft_limit > 0:
    LLM_BATCH_SOFT_TOKEN_LIMIT = max(8000, _raw_batch_soft_limit)
else:
    LLM_BATCH_SOFT_TOKEN_LIMIT = max(
        8000,
        min(
            int(ACTIVE_MODEL_CONTEXT_WINDOW_TOKENS * 0.25),
            max(
                64000,
                int(ACTIVE_MODEL_MAX_GENERATION_OUTPUT_TOKENS * 4.0),
            ),
        ),
    )


# Active-model completion ceilings are technical transport safeguards only.
# We preserve the existing Notebook 06 operational budget (8K -> max 16K by
# default) even though both configured models support larger hard output limits.
MODEL_MIN_GENERATION_OUTPUT_TOKENS = max(
    512,
    ACTIVE_MODEL_MIN_GENERATION_OUTPUT_TOKENS,
)

MODEL_MAX_GENERATION_OUTPUT_TOKENS = max(
    MODEL_MIN_GENERATION_OUTPUT_TOKENS,
    ACTIVE_MODEL_MAX_GENERATION_OUTPUT_TOKENS,
)

# Backwards-compatible aliases used inside the existing provider recovery code.
GEMINI_MIN_GENERATION_OUTPUT_TOKENS = MODEL_MIN_GENERATION_OUTPUT_TOKENS
GEMINI_MAX_GENERATION_OUTPUT_TOKENS = MODEL_MAX_GENERATION_OUTPUT_TOKENS
GROQ_MIN_GENERATION_OUTPUT_TOKENS = MODEL_MIN_GENERATION_OUTPUT_TOKENS
GROQ_MAX_GENERATION_OUTPUT_TOKENS = MODEL_MAX_GENERATION_OUTPUT_TOKENS
OPENAI_MIN_GENERATION_OUTPUT_TOKENS = MODEL_MIN_GENERATION_OUTPUT_TOKENS
OPENAI_MAX_GENERATION_OUTPUT_TOKENS = MODEL_MAX_GENERATION_OUTPUT_TOKENS

# Provider-specific transient retry settings come from the selected model's
# config entry. Client/config/quota errors are not hidden by these retries.
MODEL_TRANSIENT_MAX_ATTEMPTS = max(
    1,
    int(
        ACTIVE_MODEL_CONFIG.get(
            "transient_retry_max_attempts",
            5,
        )
        or 5
    ),
)

MODEL_TRANSIENT_INITIAL_BACKOFF_SECONDS = max(
    0.0,
    float(
        ACTIVE_MODEL_CONFIG.get(
            "transient_retry_initial_backoff_seconds",
            5,
        )
        or 5
    ),
)

MODEL_TRANSIENT_MAX_BACKOFF_SECONDS = max(
    MODEL_TRANSIENT_INITIAL_BACKOFF_SECONDS,
    float(
        ACTIVE_MODEL_CONFIG.get(
            "transient_retry_max_backoff_seconds",
            40,
        )
        or 40
    ),
)

GEMINI_503_MAX_ATTEMPTS = MODEL_TRANSIENT_MAX_ATTEMPTS
GEMINI_503_INITIAL_BACKOFF_SECONDS = MODEL_TRANSIENT_INITIAL_BACKOFF_SECONDS
GEMINI_503_MAX_BACKOFF_SECONDS = MODEL_TRANSIENT_MAX_BACKOFF_SECONDS

GROQ_TRANSIENT_MAX_ATTEMPTS = MODEL_TRANSIENT_MAX_ATTEMPTS
GROQ_TRANSIENT_INITIAL_BACKOFF_SECONDS = MODEL_TRANSIENT_INITIAL_BACKOFF_SECONDS
GROQ_TRANSIENT_MAX_BACKOFF_SECONDS = MODEL_TRANSIENT_MAX_BACKOFF_SECONDS

OPENAI_TRANSIENT_MAX_ATTEMPTS = MODEL_TRANSIENT_MAX_ATTEMPTS
OPENAI_TRANSIENT_INITIAL_BACKOFF_SECONDS = MODEL_TRANSIENT_INITIAL_BACKOFF_SECONDS
OPENAI_TRANSIENT_MAX_BACKOFF_SECONDS = MODEL_TRANSIENT_MAX_BACKOFF_SECONDS

NOTEBOOK06_PIPELINE_VERSION = (
    "agent2-notebook06-plan-c-v3.8-preserve-notebook05-pdf-pages"
)


# ================================================================
# OPTIONAL NOTEBOOK 05 PATHS
# Only used in fill_shortfall mode.
# ================================================================

NOTEBOOK05_PACKAGE_PATH: str | None = (
    str(
        os.getenv(
            "AGENT2_NOTEBOOK05_PACKAGE_PATH",
            "",
        )
        or ""
    ).strip()
    or None
)

NOTEBOOK05_SELECTED_CSV_PATH: str | None = (
    str(
        os.getenv(
            "AGENT2_NOTEBOOK05_SELECTED_CSV_PATH",
            "",
        )
        or ""
    ).strip()
    or None
)


print("Agent2 root:", AGENT2_ROOT)
print("Quiz output:", OUTPUT_DIR)
print("QUIZ_MODE:", QUIZ_MODE)
print("SELECTED_MODEL_KEY:", SELECTED_MODEL_KEY)
print("GENERATION_PROVIDER:", GENERATION_PROVIDER)
print("GENERATION_MODEL:", GENERATION_MODEL)
print("MODEL_CONTEXT_WINDOW_TOKENS:", ACTIVE_MODEL_CONTEXT_WINDOW_TOKENS)
print("MODEL_HARD_MAX_OUTPUT_TOKENS:", ACTIVE_MODEL_HARD_MAX_OUTPUT_TOKENS)
print("USER_APPROVED_GENERATION:", USER_APPROVED_GENERATION)
print("RUN_GENERATION:", RUN_GENERATION)


## 2. Agent 1 topics and the shared assessment/quiz filters

For standalone testing, edit the sample values below.

For Streamlit, pass the same values that were already selected in the assessment form through:

- `AGENT2_AGENT1_TOPICS_JSON`
- `AGENT2_ASSESSMENT_REQUEST_JSON`
- `AGENT2_LESSON_SUMMARY`

In `fill_shortfall` mode, the notebook uses the request and approved topics stored in the exact Notebook 05 run instead.

In [ ]:
# ================================================================
# STANDALONE SAMPLE INPUT
# ================================================================

AGENT1_TOPIC_OUTPUT = [
    {
        "topic": "Algorithm tracing and program execution",
        "role": "primary",
        "official_reference": "3.1.1",
        "confidence": 0.8172,
        "ranking_score": 0.6502,
        "source_chunks": [3, 4, 5, 6, 7, 8],
    },
    {
        "topic": "One- and two-dimensional arrays",
        "role": "supporting",
        "official_reference": "3.2.6",
        "confidence": 0.8477,
        "ranking_score": 0.6109,
        "source_chunks": [2, 3, 5, 6, 10],
    },
    {
        "topic": "Iteration",
        "role": "supporting",
        "official_reference": "3.2.2",
        "confidence": 0.5943,
        "ranking_score": 0.3768,
        "source_chunks": [3, 8],
    },
]


LESSON_SUMMARY = (
    "The lesson focused on tracing algorithms and following "
    "program execution. It also covered one- and "
    "two-dimensional arrays and iteration."
)


ASSESSMENT_REQUEST = {
    "number_of_questions": 5,
    "target_total_marks": 20,

    "minimum_question_marks": 1,
    "maximum_question_marks": 4,

    "minimum_primary_questions": 3,
    "minimum_supporting_questions": 1,
    "minimum_distinct_official_references": 2,

    "cover_all_approved_topics": True,

    "include_code_questions": True,
    "include_visual_questions": True,

    # "1", "1B", "2", or None
    "paper_code": None,

    # "Python" or None
    "programming_language": None,

    # Optional free-text instructions from the user.
    # In complete_quiz mode these are MANDATORY when they do not conflict
    # with approved AQA scope or explicit hard frontend controls. Machine-
    # checkable directives (for example exact primary/supporting counts and
    # distinct-style requests) are promoted into deterministic constraints
    # before any generation API call.
    "special_instructions": "",

    # Complete quiz can satisfy the count exactly.
    # Hybrid mode normally treats count as a minimum because official
    # retrieval may already have reached the requested count but still
    # be short on marks.
    "exact_question_count": True,
}


# ================================================================
# STREAMLIT OVERRIDES
# ================================================================

AGENT1_TOPIC_OUTPUT = env_json(
    "AGENT2_AGENT1_TOPICS_JSON",
    AGENT1_TOPIC_OUTPUT,
)

ASSESSMENT_REQUEST = env_json(
    "AGENT2_ASSESSMENT_REQUEST_JSON",
    ASSESSMENT_REQUEST,
)

LESSON_SUMMARY = str(
    os.getenv(
        "AGENT2_LESSON_SUMMARY",
        LESSON_SUMMARY,
    )
    or ""
).strip()


# ================================================================
# OPTIONAL USER SPECIAL INSTRUCTIONS
# ================================================================
# Priority:
# 1) explicit environment variable from an execution adapter,
# 2) current-run Streamlit sidecar file,
# 3) value already present in ASSESSMENT_REQUEST.
#
# The sidecar makes this feature compatible with the current LangGraph/MCP
# request path without changing Agent 1 or Notebook 05.
def _load_current_run_special_instructions() -> str:
    env_value = str(
        os.getenv(
            "AGENT2_QUIZ_SPECIAL_INSTRUCTIONS",
            "",
        )
        or ""
    ).strip()

    if env_value:
        return env_value

    candidates: list[Path] = []

    explicit_path = str(
        os.getenv(
            "AGENT2_QUIZ_SPECIAL_INSTRUCTIONS_PATH",
            "",
        )
        or ""
    ).strip()

    if explicit_path:
        candidates.append(
            Path(
                explicit_path
            ).expanduser()
        )

    # Typical current-run output layouts:
    #   run/output/agent2_quiz
    #   run/output/agent2_quiz/complete_quiz
    #   run/output/agent2_quiz/fill_shortfall
    for parent in [
        OUTPUT_DIR.parent,
        OUTPUT_DIR.parent.parent,
        OUTPUT_DIR.parent.parent.parent,
    ]:
        candidates.append(
            parent
            / "integration"
            / "quiz_special_instructions.json"
        )

    seen_paths: set[str] = set()

    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except OSError:
            resolved = candidate

        key = str(
            resolved
        ).casefold()

        if key in seen_paths:
            continue

        seen_paths.add(
            key
        )

        if not resolved.is_file():
            continue

        try:
            payload = json.loads(
                resolved.read_text(
                    encoding="utf-8"
                )
            )
        except (
            OSError,
            json.JSONDecodeError,
        ):
            continue

        if not isinstance(
            payload,
            dict,
        ):
            continue

        value = str(
            payload.get(
                "special_instructions",
                "",
            )
            or ""
        ).strip()

        if value:
            return value

    return str(
        ASSESSMENT_REQUEST.get(
            "special_instructions",
            "",
        )
        or ""
    ).strip()


SPECIAL_QUIZ_INSTRUCTIONS = (
    _load_current_run_special_instructions()
)

ASSESSMENT_REQUEST = {
    **ASSESSMENT_REQUEST,
    "special_instructions":
        SPECIAL_QUIZ_INSTRUCTIONS,
}


display(
    pd.DataFrame(
        AGENT1_TOPIC_OUTPUT
    )
)

display(
    pd.DataFrame(
        [
            ASSESSMENT_REQUEST
        ]
    )
)

## 3. Shared normalization and validation helpers

In [ ]:
def normalize_text(value: Any) -> str:
    return " ".join(
        re.sub(
            r"[^a-z0-9_]+",
            " ",
            str(value or "").casefold(),
        ).split()
    )


def safe_int(
    value: Any,
    default: int = 0,
) -> int:
    try:
        return int(
            float(value)
        )
    except (
        TypeError,
        ValueError,
    ):
        return int(default)


def safe_float(
    value: Any,
    default: float = 0.0,
) -> float:
    try:
        number = float(value)
        return number if math.isfinite(number) else default
    except (
        TypeError,
        ValueError,
    ):
        return float(default)


def safe_list(value: Any) -> list[Any]:
    if value is None:
        return []

    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    if isinstance(value, str):
        text = value.strip()

        if not text:
            return []

        try:
            parsed = ast.literal_eval(text)

            if isinstance(
                parsed,
                (
                    list,
                    tuple,
                ),
            ):
                return list(parsed)

        except Exception:
            pass

    return [value]


def stable_json_fingerprint(
    value: Any,
) -> str:
    canonical = json.dumps(
        value,
        sort_keys=True,
        ensure_ascii=False,
        default=str,
        separators=(
            ",",
            ":",
        ),
    )

    return hashlib.sha256(
        canonical.encode(
            "utf-8"
        )
    ).hexdigest()


def normalize_paper_code(
    value: Any,
) -> str | None:
    text = str(
        value or ""
    ).strip().upper()

    if not text:
        return None

    if text in {
        "1",
        "1A",
        "1B",
        "PAPER 1",
    }:
        return "1"

    if text in {
        "2",
        "PAPER 2",
    }:
        return "2"

    raise ValueError(
        f"Unsupported paper_code: {value!r}"
    )


# ================================================================
# AQA SYLLABUS PAPER ROUTING — ONE POSTGRESQL READ PER RUN
# ================================================================
# PostgreSQL remains the authoritative syllabus source. The notebook loads
# active syllabus concepts once through Agent 1's SyllabusStore and then uses
# in-memory lookups for all paper-routing checks. No database lookup occurs
# inside paper_code_for_reference().
import sys

from sqlalchemy import create_engine


AGENT1_CODE_ROOT = (
    AGENT2_ROOT.parent
    / "Agent_1"
).resolve()

SYLLABUS_STORE_PATH = (
    AGENT1_CODE_ROOT
    / "app"
    / "services"
    / "syllabus_store.py"
)

if not SYLLABUS_STORE_PATH.is_file():
    raise RuntimeError(
        "Could not locate Agent 1 SyllabusStore at "
        f"{SYLLABUS_STORE_PATH}"
    )

if str(AGENT1_CODE_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(AGENT1_CODE_ROOT),
    )

from app.services.syllabus_store import SyllabusStore


_syllabus_database_url = str(
    os.getenv("AGENT2_DATABASE_URL", "")
    or os.getenv("DATABASE_URL", "")
    or ""
).strip()

if not _syllabus_database_url:
    raise RuntimeError(
        "AGENT2_DATABASE_URL / DATABASE_URL is required to load "
        "AQA syllabus paper ownership from PostgreSQL."
    )

_syllabus_engine = create_engine(
    _syllabus_database_url,
    pool_pre_ping=True,
    future=True,
)

_syllabus_store = SyllabusStore(
    engine=_syllabus_engine
)

_syllabus_concepts = tuple(
    _syllabus_store.get_all_concepts()
)

if not _syllabus_concepts:
    raise RuntimeError(
        "PostgreSQL returned zero active AQA syllabus concepts."
    )

SYLLABUS_PAPER_CODE_BY_REFERENCE: dict[str, str] = {}

for _concept in _syllabus_concepts:
    _reference = str(
        _concept.official_reference
        or ""
    ).strip()

    if not _reference:
        continue

    _paper_code = normalize_paper_code(
        _concept.paper
    )

    if _paper_code is None:
        continue

    _existing_code = SYLLABUS_PAPER_CODE_BY_REFERENCE.get(
        _reference
    )

    if (
        _existing_code is not None
        and _existing_code != _paper_code
    ):
        raise RuntimeError(
            "Conflicting syllabus paper ownership in PostgreSQL for "
            f"{_reference}: {_existing_code} vs {_paper_code}"
        )

    SYLLABUS_PAPER_CODE_BY_REFERENCE[
        _reference
    ] = _paper_code

if not SYLLABUS_PAPER_CODE_BY_REFERENCE:
    raise RuntimeError(
        "No usable AQA syllabus paper mappings were loaded from PostgreSQL."
    )

SYLLABUS_PAPER_REFERENCE_ROWS = tuple(
    SYLLABUS_PAPER_CODE_BY_REFERENCE.items()
)

print(
    "Syllabus paper routing loaded once from PostgreSQL via SyllabusStore: "
    f"{len(SYLLABUS_PAPER_CODE_BY_REFERENCE)} official reference(s)."
)


def paper_code_for_reference(
    reference: Any,
) -> str | None:
    """
    Resolve paper ownership from the in-memory PostgreSQL-backed syllabus map.

    Compatibility behaviour is preserved for broader or more-specific AQA
    references:
      1) exact reference match;
      2) nearest stored parent reference for a more-specific input;
      3) unanimous stored child references for a broader input.

    No hardcoded 3.1/3.2 rule and no per-reference database query are used.
    """
    cleaned = str(
        reference or ""
    ).strip()

    if not cleaned:
        return None

    exact = SYLLABUS_PAPER_CODE_BY_REFERENCE.get(
        cleaned
    )
    if exact is not None:
        return exact

    # More-specific input than the stored concept, e.g. 3.2.2.1 when 3.2.2
    # is stored. Choose the longest matching stored parent.
    parent_matches = [
        (
            stored_reference,
            paper_code,
        )
        for stored_reference, paper_code
        in SYLLABUS_PAPER_REFERENCE_ROWS
        if cleaned.startswith(
            stored_reference + "."
        )
    ]

    if parent_matches:
        parent_matches.sort(
            key=lambda item: len(
                item[0]
            ),
            reverse=True,
        )
        longest_length = len(
            parent_matches[0][0]
        )
        longest_codes = {
            paper_code
            for stored_reference, paper_code
            in parent_matches
            if len(
                stored_reference
            )
            == longest_length
        }

        if len(longest_codes) == 1:
            return next(
                iter(
                    longest_codes
                )
            )

        return None

    # Broader input than stored concepts, e.g. 3.6.2 when only 3.6.2.1 and
    # 3.6.2.2 are stored. Resolve only when every matching child agrees.
    child_codes = {
        paper_code
        for stored_reference, paper_code
        in SYLLABUS_PAPER_REFERENCE_ROWS
        if stored_reference.startswith(
            cleaned + "."
        )
    }

    if len(child_codes) == 1:
        return next(
            iter(
                child_codes
            )
        )

    return None


def paper_label_for_code(
    paper_code: Any,
) -> str | None:
    normalized = normalize_paper_code(
        paper_code
    )

    if normalized == "1":
        return "Paper 1"

    if normalized == "2":
        return "Paper 2"

    return None


def paper_routing_preflight(
    raw_topics: list[dict[str, Any]],
    selected_paper_code: Any,
) -> dict[str, Any]:
    """
    Validate the selected paper against approved Agent 1 topics before any
    blueprint construction or LLM/API call.

    This is routing/input validation only; it does not constrain question
    generation. Mixed-paper topic sets may proceed using the compatible topics,
    but an all-mismatch (or no compatible primary topic) is blocked with a
    clear user-facing message.
    """
    selected = normalize_paper_code(selected_paper_code)

    result: dict[str, Any] = {
        "status": "NOT_FILTERED" if selected is None else "MATCH",
        "selected_paper_code": selected,
        "selected_paper_label": paper_label_for_code(selected),
        "compatible_topics": [],
        "excluded_topics": [],
        "unknown_reference_topics": [],
        "user_message": "",
        "block_generation": False,
    }

    if selected is None:
        result["user_message"] = (
            "No paper filter selected; all approved Agent 1 topics remain eligible."
        )
        return result

    if not isinstance(raw_topics, list):
        raise TypeError("Agent 1 topics must be a list.")

    for index, item in enumerate(raw_topics, start=1):
        if not isinstance(item, dict):
            continue

        topic = str(
            item.get(
                "topic",
                item.get(
                    "detected_topic",
                    item.get("topic_name", item.get("name", "")),
                ),
            )
            or ""
        ).strip()
        reference = str(
            item.get(
                "official_reference",
                item.get(
                    "agent1_official_reference",
                    item.get("reference", ""),
                ),
            )
            or ""
        ).strip()
        role = str(item.get("role", "supporting") or "supporting").strip().casefold()
        detected_paper = paper_code_for_reference(reference)

        summary = {
            "agent1_topic_index": index,
            "topic": topic,
            "official_reference": reference,
            "role": role,
            "paper_code": detected_paper,
            "paper_label": paper_label_for_code(detected_paper),
        }

        if detected_paper is None:
            result["unknown_reference_topics"].append(summary)
        elif detected_paper == selected:
            result["compatible_topics"].append(summary)
        else:
            result["excluded_topics"].append(summary)

    compatible = result["compatible_topics"]
    excluded = result["excluded_topics"]
    compatible_primary = [
        item for item in compatible
        if str(item.get("role", "")).casefold() == "primary"
    ]

    selected_label = result["selected_paper_label"] or f"Paper {selected}"

    if not compatible:
        available_labels = sorted({
            str(item.get("paper_label") or "").strip()
            for item in excluded
            if str(item.get("paper_label") or "").strip()
        })
        available_text = " / ".join(available_labels) or "the other paper"
        result["status"] = "BLOCKED_PAPER_MISMATCH"
        result["block_generation"] = True
        result["user_message"] = (
            f"You selected {selected_label}, but the approved Agent 1 topics do not "
            f"belong to {selected_label}; they belong to {available_text}. "
            "Quiz generation was stopped before any LLM/API call. "
            f"Select the matching paper or provide approved {selected_label} topics."
        )
        return result

    if not compatible_primary:
        result["status"] = "BLOCKED_NO_COMPATIBLE_PRIMARY"
        result["block_generation"] = True
        result["user_message"] = (
            f"You selected {selected_label}. Some compatible topics exist, but none "
            f"is an approved primary {selected_label} topic. Quiz generation was "
            "stopped before any LLM/API call. Approve/select at least one compatible "
            "primary topic first."
        )
        return result

    if excluded:
        excluded_names = [
            str(item.get("topic") or item.get("official_reference") or "topic")
            for item in excluded
        ]
        result["status"] = "PARTIAL_MATCH"
        result["user_message"] = (
            f"{selected_label} selected. Quiz generation will use only compatible "
            f"{selected_label} topics. Excluded topics: "
            + ", ".join(excluded_names)
            + "."
        )
    else:
        result["status"] = "MATCH"
        result["user_message"] = (
            f"{selected_label} selected and all approved topics are compatible."
        )

    return result


def normalize_agent1_topics(
    raw_topics: list[dict[str, Any]],
    paper_code: str | None,
) -> list[dict[str, Any]]:
    rows: list[
        dict[str, Any]
    ] = []

    if not isinstance(
        raw_topics,
        list,
    ):
        raise TypeError(
            "Agent 1 topics must be a list."
        )

    for index, item in enumerate(
        raw_topics,
        start=1,
    ):
        if not isinstance(
            item,
            dict,
        ):
            continue

        # Complete-quiz Agent 1 handoff uses `topic`, while Notebook 05's
        # assessment package intentionally stores the same approved topic as
        # `detected_topic`. Accept both schemas so fill_shortfall consumes the
        # current Notebook 05 package directly rather than requiring a frontend
        # reshape/copy.
        topic = str(
            item.get(
                "topic",
                item.get(
                    "detected_topic",
                    item.get(
                        "topic_name",
                        item.get(
                            "name",
                            "",
                        ),
                    ),
                ),
            )
            or ""
        ).strip()

        reference = str(
            item.get(
                "official_reference",
                item.get(
                    "agent1_official_reference",
                    item.get(
                        "reference",
                        "",
                    ),
                ),
            )
            or ""
        ).strip()

        role = str(
            item.get(
                "role",
                "supporting",
            )
            or "supporting"
        ).strip().casefold()

        if not topic or not reference:
            raise ValueError(
                f"Agent 1 topic {index} is missing topic/reference."
            )

        if role not in {
            "primary",
            "supporting",
        }:
            raise ValueError(
                f"Invalid Agent 1 role: {role!r}"
            )

        if (
            paper_code
            and paper_code_for_reference(
                reference
            )
            != paper_code
        ):
            continue

        rows.append(
            {
                "agent1_topic_index":
                    index,

                "topic":
                    topic,

                "topic_norm":
                    normalize_text(
                        topic
                    ),

                "role":
                    role,

                "official_reference":
                    reference,

                "confidence":
                    safe_float(
                        item.get(
                            "confidence"
                        )
                    ),

                "ranking_score":
                    safe_float(
                        item.get(
                            "ranking_score",
                            item.get(
                                "confidence",
                                0.0,
                            ),
                        )
                    ),

                "source_chunks":
                    safe_list(
                        item.get(
                            "source_chunks"
                        )
                    ),

                "source_chunk_texts":
                    [
                        str(value).strip()
                        for value in safe_list(
                            item.get(
                                "source_chunk_texts"
                            )
                        )
                        if str(
                            value
                        ).strip()
                    ],
            }
        )

    if not rows:
        raise RuntimeError(
            "No approved Agent 1 topics remain after the current filters."
        )

    if not any(
        item[
            "role"
        ]
        == "primary"
        for item in rows
    ):
        raise RuntimeError(
            "At least one primary Agent 1 topic is required."
        )

    return rows



# ================================================================
# STRICT SPECIAL-INSTRUCTION DIRECTIVES
# ================================================================
# Free-text special instructions are treated as mandatory in complete_quiz
# when they do not conflict with approved AQA scope or explicit hard controls.
# High-confidence machine-checkable directives are promoted into deterministic
# constraints before blueprint construction/API use. Qualitative instructions
# remain mandatory generation instructions and are surfaced to HITL.
# ================================================================

def _extract_role_question_count(
    instruction_text: str,
    role: str,
) -> int | None:
    """
    Conservative deterministic preflight helper.

    This is NOT the natural-language special-instruction interpreter. The same
    quiz-generation LLM call receives the raw instruction and interprets its
    full meaning. Python promotes only very explicit exact/only/remaining role
    counts here so the deterministic blueprint can be shaped before generation
    without paying for a separate interpreter call.

    Ambiguous wording such as "around 7", "prefer 7", "up to 7", or a bare
    "7 questions for supporting topics" is intentionally NOT converted into an
    exact hard count here.
    """
    text = re.sub(
        r"\s+",
        " ",
        str(instruction_text or "").casefold(),
    ).strip()

    role_word = re.escape(
        str(role or "").strip().casefold()
    )

    patterns = [
        rf"\b(?:only|exactly|just)\s+(\d+)\s+(?:different\s+(?:style|styles|type|types|pattern|patterns)\s+)?questions?\s+(?:for|on)\s+(?:the\s+)?{role_word}(?:\s+topics?)?\b",
        rf"\b(?:the\s+)?remaining\s+(\d+)\s+(?:different\s+(?:style|styles|type|types|pattern|patterns)\s+)?questions?\s+(?:for|on)\s+(?:the\s+)?{role_word}(?:\s+topics?)?\b",
        rf"\b{role_word}(?:\s+topics?)?\s*[:=\-]?\s*(?:only|exactly|just)\s+(\d+)\s+questions?\b",
        rf"\b(?:only|exactly|just)\s+(\d+)\s+{role_word}\s+questions?\b",
    ]

    values: list[int] = []

    for pattern in patterns:
        for match in re.finditer(
            pattern,
            text,
            flags=re.IGNORECASE,
        ):
            values.append(
                int(
                    match.group(1)
                )
            )

    if not values:
        return None

    unique_values = sorted(
        set(
            values
        )
    )

    if len(
        unique_values
    ) > 1:
        raise ValueError(
            "Special quiz instructions contain conflicting explicit exact "
            f"{role} question counts: {unique_values}."
        )

    return unique_values[
        0
    ]



def _special_distinct_style_roles(
    instruction_text: str,
) -> list[str]:
    text = re.sub(
        r"\s+",
        " ",
        str(instruction_text or "").casefold(),
    ).strip()

    if not text:
        return []

    style_phrase = (
        r"(?:different|distinct|unique)\s+"
        r"(?:style|styles|styled|type|types|pattern|patterns)"
    )

    roles: list[str] = []

    for role in [
        "primary",
        "supporting",
    ]:
        role_pattern = re.escape(
            role
        )

        # High-confidence role attachment only. Keep the style phrase and role
        # in the same local clause so "primary ... while remaining 7 different
        # style questions for supporting" does not incorrectly mark primary.
        role_specific_patterns = [
            rf"{style_phrase}\s+questions?\s+(?:for|on)\s+(?:the\s+)?{role_pattern}(?:\s+topics?)?\b",
            rf"\b{role_pattern}\s+questions?.{{0,35}}{style_phrase}\b",
            rf"\b{role_pattern}(?:\s+topics?)?\s+(?:should|must|need\s+to|are\s+to|with|using)\s+.{{0,35}}{style_phrase}\b",
        ]

        if any(
            re.search(
                pattern,
                text,
                flags=re.IGNORECASE,
            )
            for pattern in role_specific_patterns
        ):
            roles.append(
                role
            )

    if (
        not roles
        and re.search(
            rf"\b(?:all|every)\s+questions?.{{0,80}}{style_phrase}\b|\b{style_phrase}\s+(?:for\s+)?(?:all|every)\s+questions?\b",
            text,
            flags=re.IGNORECASE,
        )
    ):
        roles = [
            "primary",
            "supporting",
        ]

    return roles

def _apply_special_instruction_directives(
    normalized_request: dict[str, Any],
) -> dict[str, Any]:
    """
    Add cheap deterministic PRE-GENERATION hints only.

    Final architecture:
      - raw special_instructions are interpreted by the SAME LLM call that
        generates the quiz;
      - no separate instruction-interpreter LLM call is used;
      - Python promotes only high-confidence machine-checkable hints that are
        useful before generation;
      - post-generation Python validators independently enforce objective
        constraints, while HITL confirms qualitative instructions.

    Therefore this function must stay deliberately narrow and must never try to
    become a general natural-language parser.
    """
    normalized = dict(
        normalized_request
    )

    instruction_text = str(
        normalized.get(
            "special_instructions",
            "",
        )
        or ""
    ).strip()

    active_quiz_mode = str(
        globals().get(
            "QUIZ_MODE",
            "complete_quiz",
        )
        or "complete_quiz"
    ).strip().casefold()

    directives: dict[str, Any] = {
        "mode":
            (
                "same_generation_call_mandatory"
                if instruction_text
                else "none"
            ),
        "interpreter_strategy":
            "same_generation_call",
        "separate_instruction_llm_call":
            False,
        "scope":
            (
                "entire_generated_quiz"
                if active_quiz_mode == "complete_quiz"
                else "ai_generated_shortfall_only"
            ),
        "exact_primary_questions":
            None,
        "exact_supporting_questions":
            None,
        "distinct_styles_for_roles":
            [],
        "requires_human_confirmation_for_qualitative_parts":
            bool(
                instruction_text
            ),
        "deterministic_hint_parser_is_exhaustive":
            False,
    }

    if not instruction_text:
        normalized[
            "special_instruction_directives"
        ] = directives
        return normalized

    # In fill_shortfall mode official AQA questions are already fixed by
    # Notebook 05. Raw special instructions are still mandatory for the
    # AI-generated shortfall, but exact whole-quiz role counts are not promoted
    # here because they may refer to content the generator does not own.
    if active_quiz_mode != "complete_quiz":
        normalized[
            "special_instruction_directives"
        ] = directives
        return normalized

    question_count = safe_int(
        normalized.get(
            "number_of_questions",
            0,
        )
    )

    exact_primary = _extract_role_question_count(
        instruction_text,
        "primary",
    )

    exact_supporting = _extract_role_question_count(
        instruction_text,
        "supporting",
    )

    if (
        exact_primary is not None
        and exact_supporting is None
    ):
        exact_supporting = (
            question_count
            - exact_primary
        )

    elif (
        exact_supporting is not None
        and exact_primary is None
    ):
        exact_primary = (
            question_count
            - exact_supporting
        )

    if (
        exact_primary is not None
        or exact_supporting is not None
    ):
        if (
            exact_primary is None
            or exact_supporting is None
        ):
            raise AssertionError(
                "Internal special-instruction role resolution failed."
            )

        if (
            exact_primary < 0
            or exact_supporting < 0
        ):
            raise ValueError(
                "Special quiz instructions request more role-specific "
                "questions than the selected total number_of_questions."
            )

        if (
            exact_primary
            + exact_supporting
            != question_count
        ):
            raise ValueError(
                "Special quiz instructions conflict with the selected total "
                "question count: explicit exact primary + supporting must equal "
                f"{question_count}."
            )

        minimum_primary = safe_int(
            normalized.get(
                "minimum_primary_questions",
                0,
            )
        )

        minimum_supporting = safe_int(
            normalized.get(
                "minimum_supporting_questions",
                0,
            )
        )

        if exact_primary < minimum_primary:
            raise ValueError(
                "Special quiz instructions conflict with the frontend controls: "
                f"exact primary questions ({exact_primary}) are below the "
                f"selected minimum ({minimum_primary})."
            )

        if exact_supporting < minimum_supporting:
            raise ValueError(
                "Special quiz instructions conflict with the frontend controls: "
                f"exact supporting questions ({exact_supporting}) are below the "
                f"selected minimum ({minimum_supporting})."
            )

        directives[
            "exact_primary_questions"
        ] = exact_primary

        directives[
            "exact_supporting_questions"
        ] = exact_supporting

    directives[
        "distinct_styles_for_roles"
    ] = _special_distinct_style_roles(
        instruction_text
    )

    normalized[
        "special_instruction_directives"
    ] = directives

    return normalized



def validate_request(
    request: dict[str, Any],
) -> dict[str, Any]:
    """
    Normalize the Streamlit request without silently changing the user's
    quiz design choices.

    The frontend controls are authoritative. Notebook 06 validates the
    same domains again because Streamlit passes the request through JSON
    environment variables and the notebook must remain safe when run
    independently.
    """
    if not isinstance(
        request,
        dict,
    ):
        raise TypeError(
            "Assessment request must be a dictionary."
        )

    normalized = dict(
        request
    )

    def bounded_int(
        field: str,
        default: int,
        minimum: int,
        maximum: int,
    ) -> int:
        value = safe_int(
            normalized.get(
                field,
                default,
            ),
            default,
        )

        if not (
            minimum
            <= value
            <= maximum
        ):
            raise ValueError(
                f"{field} must be between {minimum} and {maximum}; "
                f"received {value}."
            )

        return value

    normalized[
        "number_of_questions"
    ] = bounded_int(
        "number_of_questions",
        5,
        1,
        30,
    )

    normalized[
        "target_total_marks"
    ] = bounded_int(
        "target_total_marks",
        20,
        1,
        200,
    )

    normalized[
        "minimum_question_marks"
    ] = bounded_int(
        "minimum_question_marks",
        1,
        1,
        20,
    )

    normalized[
        "maximum_question_marks"
    ] = bounded_int(
        "maximum_question_marks",
        12,
        1,
        30,
    )

    if (
        normalized[
            "maximum_question_marks"
        ]
        < normalized[
            "minimum_question_marks"
        ]
    ):
        raise ValueError(
            "maximum_question_marks must be greater than or equal to "
            "minimum_question_marks."
        )

    normalized[
        "minimum_primary_questions"
    ] = bounded_int(
        "minimum_primary_questions",
        1,
        1,
        30,
    )

    normalized[
        "minimum_supporting_questions"
    ] = bounded_int(
        "minimum_supporting_questions",
        0,
        0,
        30,
    )

    question_count = normalized[
        "number_of_questions"
    ]

    minimum_primary = normalized[
        "minimum_primary_questions"
    ]

    minimum_supporting = normalized[
        "minimum_supporting_questions"
    ]

    if minimum_primary > question_count:
        raise ValueError(
            "minimum_primary_questions cannot exceed number_of_questions."
        )

    if minimum_supporting > question_count:
        raise ValueError(
            "minimum_supporting_questions cannot exceed number_of_questions."
        )

    if (
        minimum_primary
        + minimum_supporting
        > question_count
    ):
        raise ValueError(
            "minimum_primary_questions + minimum_supporting_questions "
            "cannot exceed number_of_questions."
        )

    normalized[
        "minimum_distinct_official_references"
    ] = max(
        0,
        safe_int(
            normalized.get(
                "minimum_distinct_official_references",
                0,
            )
        ),
    )

    if (
        normalized[
            "minimum_distinct_official_references"
        ]
        > question_count
    ):
        raise ValueError(
            "minimum_distinct_official_references cannot exceed "
            "number_of_questions."
        )

    normalized[
        "cover_all_approved_topics"
    ] = bool(
        normalized.get(
            "cover_all_approved_topics",
            False,
        )
    )

    normalized[
        "include_code_questions"
    ] = bool(
        normalized.get(
            "include_code_questions",
            True,
        )
    )

    normalized[
        "include_visual_questions"
    ] = bool(
        normalized.get(
            "include_visual_questions",
            True,
        )
    )

    normalized[
        "paper_code"
    ] = normalize_paper_code(
        normalized.get(
            "paper_code"
        )
    )

    language = str(
        normalized.get(
            "programming_language",
            "",
        )
        or ""
    ).strip()

    normalized[
        "programming_language"
    ] = (
        None
        if language.casefold()
        in {
            "",
            "automatic",
            "auto",
        }
        else language
    )

    special_instructions = str(
        normalized.get(
            "special_instructions",
            "",
        )
        or ""
    ).strip()

    # This is only a transport/safety ceiling for free text. The content
    # itself is never hardcoded or interpreted as a fixed rule here.
    if len(
        special_instructions
    ) > 6000:
        raise ValueError(
            "special_instructions must be 6000 characters or fewer."
        )

    normalized[
        "special_instructions"
    ] = special_instructions

    normalized = _apply_special_instruction_directives(
        normalized
    )

    return normalized



## 4. Load the correct input branch

In [ ]:
PACKAGE_RE = re.compile(
    r"agent2_assessment_package_(\d{8}_\d{6})\.json$"
)

SELECTED_RE = re.compile(
    r"agent2_selected_questions_(\d{8}_\d{6})\.csv$"
)


def timestamp_from_path(
    path: Path,
    pattern: re.Pattern,
) -> str | None:
    match = pattern.search(
        path.name
    )

    return (
        match.group(1)
        if match
        else None
    )


def resolve_notebook05_pair() -> tuple[
    Path,
    Path,
    str,
]:
    explicit_package = (
        Path(
            NOTEBOOK05_PACKAGE_PATH
        ).expanduser().resolve()
        if NOTEBOOK05_PACKAGE_PATH
        else None
    )

    explicit_selected = (
        Path(
            NOTEBOOK05_SELECTED_CSV_PATH
        ).expanduser().resolve()
        if NOTEBOOK05_SELECTED_CSV_PATH
        else None
    )

    if explicit_package:
        if not explicit_package.is_file():
            raise FileNotFoundError(
                explicit_package
            )

        package_ts = timestamp_from_path(
            explicit_package,
            PACKAGE_RE,
        )

        if not package_ts:
            raise RuntimeError(
                "Could not read the Notebook 05 package timestamp."
            )

        if explicit_selected:
            selected_path = explicit_selected
        else:
            selected_path = (
                explicit_package.parent
                / f"agent2_selected_questions_{package_ts}.csv"
            )

        if not selected_path.is_file():
            raise FileNotFoundError(
                selected_path
            )

        selected_ts = timestamp_from_path(
            selected_path,
            SELECTED_RE,
        )

        if (
            selected_ts
            and selected_ts
            != package_ts
        ):
            raise RuntimeError(
                "Notebook 05 package and selected CSV are from different runs."
            )

        return (
            explicit_package,
            selected_path,
            package_ts,
        )

    if explicit_selected:
        if not explicit_selected.is_file():
            raise FileNotFoundError(
                explicit_selected
            )

        selected_ts = timestamp_from_path(
            explicit_selected,
            SELECTED_RE,
        )

        if not selected_ts:
            raise RuntimeError(
                "Could not read the Notebook 05 CSV timestamp."
            )

        package_path = (
            explicit_selected.parent
            / f"agent2_assessment_package_{selected_ts}.json"
        )

        if not package_path.is_file():
            raise FileNotFoundError(
                package_path
            )

        return (
            package_path,
            explicit_selected,
            selected_ts,
        )

    output_root = (
        AGENT2_ROOT
        / "OUTPUT"
    )

    paired: list[
        tuple[
            str,
            Path,
            Path,
        ]
    ] = []

    for package_path in output_root.rglob(
        "agent2_assessment_package_*.json"
    ):
        run_ts = timestamp_from_path(
            package_path,
            PACKAGE_RE,
        )

        if not run_ts:
            continue

        selected_path = (
            package_path.parent
            / f"agent2_selected_questions_{run_ts}.csv"
        )

        if selected_path.is_file():
            paired.append(
                (
                    run_ts,
                    package_path,
                    selected_path,
                )
            )

    if not paired:
        raise FileNotFoundError(
            "No paired Notebook 05 package + selected CSV was found."
        )

    paired.sort(
        key=lambda item: item[
            0
        ],
        reverse=True,
    )

    run_ts, package_path, selected_path = paired[
        0
    ]

    return (
        package_path,
        selected_path,
        run_ts,
    )


def _resolve_notebook05_student_pdf(
    *,
    package: dict[str, Any],
    package_path: Path,
    run_timestamp: str | None,
) -> tuple[Path | None, dict[str, Any]]:
    """
    Resolve Notebook 05's ORIGINAL student-facing question-paper PDF.

    This intentionally follows the original Notebook 06 shortfall export
    architecture: Notebook 06C does NOT redraw/reconstruct official questions.
    It merges the original Notebook 05 PDF pages so diagrams, figure crops,
    tables, answer grids, code figures and layout survive unchanged.

    Resolution is deliberately backward-compatible because older Notebook 05
    packages did not always store the PDF under the same manifest field.
    """
    package_path = Path(package_path).expanduser().resolve()
    package_dir = package_path.parent

    candidates: list[tuple[str, Path]] = []

    def _add_candidate(source: str, raw: Any) -> None:
        value = str(raw or "").strip()
        if not value:
            return

        candidate = Path(value).expanduser()
        if not candidate.is_absolute():
            candidate = package_dir / candidate

        try:
            candidate = candidate.resolve()
        except Exception:
            return

        if candidate.is_file() and candidate.suffix.casefold() == ".pdf":
            candidates.append((source, candidate))

    # Primary/original Notebook 06 contract.
    output_files = package.get("output_files", {}) or {}
    if isinstance(output_files, dict):
        for key in (
            "student_question_paper_pdf",
            "official_student_question_paper_pdf",
            "student_pdf",
        ):
            _add_candidate(
                f"package.output_files.{key}",
                output_files.get(key),
            )

    # Backward-/forward-compatible manifest shapes.
    source_artifacts = package.get("source_artifacts", {}) or {}
    if isinstance(source_artifacts, dict):
        for key in (
            "student_question_paper_pdf",
            "notebook05_student_question_paper_pdf",
            "official_retrieval_student_pdf",
        ):
            _add_candidate(
                f"package.source_artifacts.{key}",
                source_artifacts.get(key),
            )

    # Exact run-timestamp fallback used by the original Notebook 06.
    if run_timestamp:
        exact_names = (
            f"agent2_student_question_paper_{run_timestamp}.pdf",
            f"Agent2_student_question_paper_{run_timestamp}.pdf",
            f"Agent2_Student_Question_Paper_{run_timestamp}.pdf",
        )

        for exact_name in exact_names:
            for match in package_dir.rglob(exact_name):
                if match.is_file():
                    candidates.append(
                        ("timestamp_filename_fallback", match.resolve())
                    )

    # Older/variant Notebook 05 builds: same output directory, student-paper
    # naming convention, choose newest only when an exact timestamp match was
    # unavailable.
    for pattern in (
        "agent2_student_question_paper_*.pdf",
        "Agent2_student_question_paper_*.pdf",
        "*student*question*paper*.pdf",
    ):
        for match in package_dir.rglob(pattern):
            if match.is_file():
                candidates.append(
                    ("directory_student_pdf_fallback", match.resolve())
                )

    # De-duplicate while preserving the strongest source ordering.
    deduped: list[tuple[str, Path]] = []
    seen: set[str] = set()

    for source, candidate in candidates:
        key = str(candidate).casefold()
        if key in seen:
            continue
        seen.add(key)
        deduped.append((source, candidate))

    if not deduped:
        return (
            None,
            {
                "status": "NOT_FOUND",
                "selected_source": None,
                "selected_path": None,
                "candidate_count": 0,
                "policy": "preserve_original_notebook05_pdf_pages",
            },
        )

    # Source-priority first; newest mtime only breaks ties within the same
    # fallback class.
    priority = {
        "package.output_files.student_question_paper_pdf": 0,
        "package.output_files.official_student_question_paper_pdf": 1,
        "package.output_files.student_pdf": 2,
        "package.source_artifacts.student_question_paper_pdf": 3,
        "package.source_artifacts.notebook05_student_question_paper_pdf": 4,
        "package.source_artifacts.official_retrieval_student_pdf": 5,
        "timestamp_filename_fallback": 6,
        "directory_student_pdf_fallback": 7,
    }

    deduped.sort(
        key=lambda item: (
            priority.get(item[0], 99),
            -item[1].stat().st_mtime,
        )
    )

    selected_source, selected_path = deduped[0]

    return (
        selected_path,
        {
            "status": "FOUND",
            "selected_source": selected_source,
            "selected_path": str(selected_path),
            "candidate_count": len(deduped),
            "all_candidates": [
                {
                    "source": source,
                    "path": str(path),
                }
                for source, path in deduped
            ],
            "policy": "preserve_original_notebook05_pdf_pages",
        },
    )


request: dict[
    str,
    Any,
]

approved_topics_raw: list[
    dict[str, Any]
]

official_questions_df = pd.DataFrame()

notebook05_package_path: Path | None = None
notebook05_selected_path: Path | None = None
notebook05_run_timestamp: str | None = None
notebook05_student_pdf_path: Path | None = None
notebook05_teacher_pdf_path: Path | None = None


if QUIZ_MODE == "complete_quiz":

    # IMPORTANT:
    # Complete Quiz does NOT call Notebook 05 and does NOT read
    # any past-paper retrieval output.
    request = validate_request(
        ASSESSMENT_REQUEST
    )

    approved_topics_raw = (
        AGENT1_TOPIC_OUTPUT
    )

    request[
        "exact_question_count"
    ] = bool(
        request.get(
            "exact_question_count",
            True,
        )
    )


else:

    (
        notebook05_package_path,
        notebook05_selected_path,
        notebook05_run_timestamp,
    ) = resolve_notebook05_pair()

    package = json.loads(
        notebook05_package_path.read_text(
            encoding="utf-8"
        )
    )

    if not isinstance(
        package,
        dict,
    ):
        raise RuntimeError(
            "Notebook 05 package must contain a JSON object."
        )

    request = validate_request(
        package.get(
            "assessment_request",
            package.get(
                "request",
                {},
            ),
        )
    )

    # Hybrid mode treats number_of_questions as the requested/minimum
    # count unless the incoming request explicitly says otherwise.
    request[
        "exact_question_count"
    ] = bool(
        request.get(
            "exact_question_count",
            False,
        )
    )

    approved_topics_raw = package.get(
        "agent1_topics",
        [],
    )

    if not isinstance(
        approved_topics_raw,
        list,
    ):
        raise RuntimeError(
            "Notebook 05 package does not contain Agent 1 topics."
        )

    if not approved_topics_raw:
        raise RuntimeError(
            "Notebook 05 package contains no approved Agent 1 topic handoff. "
            "Rerun Notebook 05 from the current Agent 1 approval state before "
            "requesting AI shortfall generation."
        )

    official_questions_df = pd.read_csv(
        notebook05_selected_path
    )

    # IMPORTANT — preserve the exact original Notebook 06 shortfall PDF
    # architecture. Official retrieval pages are never redrawn from CSV text
    # when the Notebook 05 student paper exists; we merge those original pages
    # directly so all retrieved diagrams/figures/tables/code crops survive.
    (
        notebook05_student_pdf_path,
        notebook05_student_pdf_resolution,
    ) = _resolve_notebook05_student_pdf(
        package=package,
        package_path=notebook05_package_path,
        run_timestamp=notebook05_run_timestamp,
    )

    # Resolve Notebook 05's already-generated official teacher mark-scheme PDF.
    # This is presentation/export-only; retrieval, ranking, HITL and release logic
    # remain unchanged.
    notebook05_output_files = package.get("output_files", {}) or {}
    if isinstance(notebook05_output_files, dict):
        raw_teacher_pdf = str(
            notebook05_output_files.get("teacher_mark_scheme_pdf") or ""
        ).strip()
        if raw_teacher_pdf:
            teacher_candidate = Path(raw_teacher_pdf).expanduser()
            if not teacher_candidate.is_absolute():
                teacher_candidate = notebook05_package_path.parent / teacher_candidate
            try:
                teacher_candidate = teacher_candidate.resolve()
            except Exception:
                teacher_candidate = None
            if (
                teacher_candidate is not None
                and teacher_candidate.is_file()
                and teacher_candidate.suffix.casefold() == ".pdf"
            ):
                notebook05_teacher_pdf_path = teacher_candidate

    notebook05_pdf_resolution_path = (
        OUTPUT_DIR
        / "notebook05_student_pdf_resolution.json"
    )

    notebook05_pdf_resolution_path.write_text(
        json.dumps(
            notebook05_student_pdf_resolution,
            indent=2,
            ensure_ascii=False,
            default=str,
        ),
        encoding="utf-8",
    )

    print(
        "Notebook 05 handoff topics loaded:",
        len(approved_topics_raw),
    )
    if notebook05_student_pdf_path is not None:
        print(
            "Notebook 05 ORIGINAL official student PDF:",
            notebook05_student_pdf_path,
        )
        print(
            "Official shortfall PDF mode: preserve original Notebook 05 pages "
            "(diagrams/figures/layout included)."
        )
    else:
        print(
            "WARNING: Notebook 05 original student PDF was not resolved. "
            "A review fallback may be shown, but it is not the preferred "
            "diagram-preserving hybrid export."
        )


paper_routing_preflight_result = paper_routing_preflight(
    approved_topics_raw,
    request.get(
        "paper_code"
    ),
)

paper_routing_preflight_path = (
    OUTPUT_DIR
    / "paper_routing_preflight.json"
)

paper_routing_preflight_path.write_text(
    json.dumps(
        paper_routing_preflight_result,
        indent=2,
        ensure_ascii=False,
        default=str,
    ),
    encoding="utf-8",
)

print(
    "Paper routing preflight:",
    paper_routing_preflight_result.get(
        "user_message",
        "",
    ),
)

PAPER_ROUTING_BLOCKED = bool(
    paper_routing_preflight_result.get(
        "block_generation",
        False,
    )
)

PAPER_ROUTING_BLOCK_REASON = str(
    paper_routing_preflight_result.get(
        "user_message",
        "",
    )
    or ""
).strip()

if PAPER_ROUTING_BLOCKED:
    # IMPORTANT: a paper mismatch is a valid user/input state, not a notebook
    # execution failure. Keep Notebook 06 alive so Streamlit/LangGraph can read
    # the routing artifact and show the actionable message. No generation
    # blueprint will be built and no LLM/API call will be sent.
    print()
    print("QUIZ GENERATION BLOCKED BY PAPER ROUTING")
    print(PAPER_ROUTING_BLOCK_REASON)
    print("Notebook 06 will complete normally with a blocked status artifact.")
    print("No LLM/API call will be sent for this request.")
    print()

    # Preserve the approved Agent 1 topics for reporting/debugging. We
    # intentionally do NOT silently switch the user's selected paper.
    approved_topics = normalize_agent1_topics(
        approved_topics_raw,
        None,
    )
else:
    approved_topics = normalize_agent1_topics(
        approved_topics_raw,
        request.get(
            "paper_code"
        ),
    )


print("Input branch:", QUIZ_MODE)
print(
    "Notebook 05 used:",
    QUIZ_MODE == "fill_shortfall",
)

if notebook05_run_timestamp:
    print(
        "Notebook 05 run:",
        notebook05_run_timestamp,
    )

display(
    pd.DataFrame(
        approved_topics
    )
)

## 5. Normalize official questions for hybrid mode

In [ ]:
def first_available_column(
    frame: pd.DataFrame,
    names: list[str],
) -> str | None:
    for name in names:
        if name in frame.columns:
            return name
    return None


def records_json_safe(
    frame: pd.DataFrame,
) -> list[
    dict[str, Any]
]:
    if frame.empty:
        return []

    safe = (
        frame.copy()
        .astype(object)
        .where(
            pd.notna(
                frame
            ),
            None,
        )
    )

    return safe.to_dict(
        orient="records"
    )


official_questions: list[
    dict[str, Any]
] = []

if QUIZ_MODE == "fill_shortfall":

    if official_questions_df.empty:
        official_questions = []

    else:

        official_questions = records_json_safe(
            official_questions_df
        )

        for question in official_questions:
            question.setdefault(
                "source_type",
                "official_aqa",
            )
            question.setdefault(
                "official_aqa_question",
                True,
            )


def question_marks(
    question: dict[str, Any],
) -> int:
    for key in [
        "marks_numeric",
        "marks_postgres",
        "marks_retrieval",
        "maximum_marks",
        "marks",
    ]:
        if key in question:
            value = safe_int(
                question.get(
                    key
                )
            )

            if value > 0:
                return value

    return 0


def question_topic(
    question: dict[str, Any],
) -> str:
    return str(
        question.get(
            "detected_topic",
            question.get(
                "topic",
                "",
            ),
        )
        or ""
    ).strip()


def question_role(
    question: dict[str, Any],
) -> str:
    return str(
        question.get(
            "agent1_role",
            question.get(
                "role",
                "",
            ),
        )
        or ""
    ).strip().casefold()


def question_reference(
    question: dict[str, Any],
) -> str:
    for key in [
        "agent1_official_reference",
        "official_reference_canonical",
        "official_reference_postgres",
        "official_reference_retrieval",
        "official_reference",
    ]:
        value = str(
            question.get(
                key,
                "",
            )
            or ""
        ).strip()

        if value:
            return value

    return ""


def question_text(
    question: dict[str, Any],
) -> str:
    for key in [
        "question_text_canonical",
        "question_text_postgres",
        "question_text_retrieval",
        "question_text",
    ]:
        value = str(
            question.get(
                key,
                "",
            )
            or ""
        ).strip()

        if value:
            return value

    return ""


official_marks = sum(
    question_marks(
        question
    )
    for question in official_questions
)

official_question_count = len(
    official_questions
)

print(
    "Official question count:",
    official_question_count,
)

print(
    "Official marks:",
    official_marks,
)

## 6. Deterministic quiz blueprint planner

For **Generate Complete Quiz**, the user's frontend request is authoritative.
The requested question count and total marks remain exact, while
primary/supporting counts are minimums.

Each generated question has one **anchor topic** that determines its role.
The same question can also cover additional approved Agent 1 topics through
`covered_topics`. Therefore **Cover all approved topics does not mean one
separate question per topic**. Topic/reference coverage is distributed across
the exact number of questions requested by the user.

All topic assignments are derived dynamically from the current Agent 1 handoff;
no particular topic names, counts, or role combination is hardcoded.


## Quiz-wide diversity and balanced role weighting

Two general rules are now applied to every generated quiz.

### 1. Question-pattern diversity

Questions are treated as parts of one assessment rather than independent
generations. The selected generation model is instructed not to reuse the same underlying assessment
template, reasoning route, or response structure simply by changing values,
variable names, strings, identifiers, or scenario wording.

If a topic appears more than once, the questions should assess it in
meaningfully different suitable ways where possible. Examples include tracing,
prediction, explanation, debugging/completion, comparison, scenario
application, construction/modification, or data-structure reasoning. These
are examples only; the selected pattern must remain appropriate to the current
topic, marks, GCSE level, and user filters.

Local MiniLM/lexical/task-family checks also compare generated questions against one another.
Repeated topic coverage is allowed; repeated question structure is what is
flagged.

### 2. Weighted but balanced marks

Primary topics remain more important than supporting topics, but extra marks
are no longer allocated by filling primary questions first.

The deterministic allocator uses the existing role weights
`primary = 2.0` and `supporting = 1.0` as **relative weights**. The next
available mark is repeatedly assigned to the question that is currently most
under-served relative to its role weight.

Therefore:
- primary questions normally receive more marks on average;
- supporting questions still receive meaningful assessment weight;
- the user's exact total marks are preserved;
- the user's minimum and maximum marks per question are preserved;
- no fixed primary/supporting percentage, topic count, or quiz-specific mark
  split is hardcoded.


## Deterministic blueprint v2.29 — hard controls + one-call natural-language instructions

This revision keeps the deterministic blueprint for objective quiz controls while
moving **arbitrary free-text instruction understanding to the same LLM call that
generates the quiz**.

1. **Hard frontend controls remain deterministic.** Question count, marks,
   paper routing, approved topics, role minima, code/visual settings and
   programming language are validated in Python.
2. **No separate instruction-interpreter LLM call is used.** The raw
   `special_instructions` text is sent to the selected generation model, which
   interprets it and generates the quiz in the same response.
3. **A deliberately small preflight helper promotes only explicit,
   machine-checkable wording** such as `only/exactly/remaining` role counts.
   This helper is an optimisation for shaping the blueprint before generation;
   it is not the natural-language interpreter and is intentionally not
   exhaustive.
4. **Special instructions are mandatory when their wording is mandatory.**
   The model returns a structured `instruction_interpretation` and
   `special_instruction_compliance` report in the same JSON response. Python
   independently verifies objective constraints; HITL confirms qualitative
   requirements.
5. **No quiz-specific values or topics are hardcoded.** The same generic logic
   applies to any approved AQA topic and any user-entered special instruction.


In [ ]:
def topic_priority(
    topic: dict[str, Any],
) -> tuple:
    return (
        -ROLE_WEIGHTS.get(
            topic[
                "role"
            ],
            1.0,
        ),
        -safe_float(
            topic.get(
                "ranking_score"
            )
        ),
        -safe_float(
            topic.get(
                "confidence"
            )
        ),
        topic[
            "topic"
        ].casefold(),
    )


def choose_topic_for_role(
    topics: list[
        dict[str, Any]
    ],
    role: str,
    assigned_counts: dict[
        str,
        int,
    ],
) -> dict[str, Any]:
    candidates = [
        item
        for item in topics
        if item[
            "role"
        ]
        == role
    ]

    if not candidates:
        raise RuntimeError(
            f"No approved {role} topic is available."
        )

    return sorted(
        candidates,
        key=lambda item: (
            assigned_counts.get(
                item[
                    "topic_norm"
                ],
                0,
            ),
            topic_priority(
                item
            ),
        ),
    )[
        0
    ]


def choose_general_topic(
    topics: list[
        dict[str, Any]
    ],
    assigned_counts: dict[
        str,
        int,
    ],
) -> dict[str, Any]:
    return sorted(
        topics,
        key=lambda item: (
            (
                assigned_counts.get(
                    item[
                        "topic_norm"
                    ],
                    0,
                )
                / ROLE_WEIGHTS.get(
                    item[
                        "role"
                    ],
                    1.0,
                )
            ),
            topic_priority(
                item
            ),
        ),
    )[
        0
    ]


def add_slot(
    slots: list[
        dict[str, Any]
    ],
    topic: dict[str, Any],
    assigned_counts: dict[
        str,
        int,
    ],
) -> None:
    paper_code = paper_code_for_reference(
        topic[
            "official_reference"
        ]
    )

    slots.append(
        {
            "topic":
                topic[
                    "topic"
                ],

            "topic_norm":
                topic[
                    "topic_norm"
                ],

            "role":
                topic[
                    "role"
                ],

            "official_reference":
                topic[
                    "official_reference"
                ],

            "paper_code":
                paper_code,

            "paper_label":
                paper_label_for_code(
                    paper_code
                ),

            "assessment_pattern":
                "",

            "marks":
                0,
        }
    )

    assigned_counts[
        topic[
            "topic_norm"
        ]
    ] = (
        assigned_counts.get(
            topic[
                "topic_norm"
            ],
            0,
        )
        + 1
    )


def assign_assessment_patterns(
    slots: list[
        dict[str, Any]
    ],
) -> list[
    dict[str, Any]
]:
    """
    Deterministically spread assessment patterns across the whole quiz.

    The pool is topic-neutral. Selection minimizes:
    1) reuse for the same anchor topic,
    2) reuse across the whole quiz,
    3) the fixed cycle order as a deterministic tie-break.

    This means repeated topics receive different task/cognitive forms before
    any pattern is reused where possible. No topic-specific pattern is
    hardcoded.
    """
    topic_pattern_counts: dict[
        str,
        dict[str, int],
    ] = {}

    global_pattern_counts = {
        pattern:
            0
        for pattern in ASSESSMENT_PATTERN_CYCLE
    }

    for slot in slots:
        topic_norm = normalize_text(
            slot.get(
                "topic_norm",
                slot.get(
                    "topic",
                    "",
                ),
            )
        )

        per_topic = (
            topic_pattern_counts
            .setdefault(
                topic_norm,
                {
                    pattern:
                        0
                    for pattern
                    in ASSESSMENT_PATTERN_CYCLE
                },
            )
        )

        chosen_pattern = min(
            ASSESSMENT_PATTERN_CYCLE,
            key=lambda pattern: (
                per_topic[
                    pattern
                ],
                global_pattern_counts[
                    pattern
                ],
                ASSESSMENT_PATTERN_CYCLE.index(
                    pattern
                ),
            ),
        )

        slot[
            "assessment_pattern"
        ] = chosen_pattern

        per_topic[
            chosen_pattern
        ] += 1

        global_pattern_counts[
            chosen_pattern
        ] += 1

    return slots


def allocate_exact_marks(
    slots: list[
        dict[str, Any]
    ],
    total_marks: int,
    minimum_marks: int,
    maximum_marks: int,
) -> list[
    dict[str, Any]
]:
    """
    Allocate the exact requested total in TWO deterministic levels.

    Level 1 — role budget:
    - every question receives the user's minimum first;
    - remaining marks are allocated between roles according to
      question_count_for_role × ROLE_WEIGHTS[role];
    - primary therefore receives more marks PER QUESTION on average when
      primary has the larger configured role weight;
    - supporting questions keep a meaningful budget;
    - role totals are not fixed percentages.

    Level 2 — within-role distribution:
    - each role's budget is spread as evenly as possible across that role's
      question slots while respecting the user's per-question maximum.

    This keeps the rule general for any topic names, question count, mark
    total, and primary/supporting mix.
    """
    if not slots:
        if total_marks == 0:
            return []

        raise RuntimeError(
            "Marks cannot be allocated without generated questions."
        )

    minimum_possible = (
        len(
            slots
        )
        * minimum_marks
    )

    maximum_possible = (
        len(
            slots
        )
        * maximum_marks
    )

    if not (
        minimum_possible
        <= total_marks
        <= maximum_possible
    ):
        raise RuntimeError(
            "Requested marks cannot be distributed across the "
            "generated question count while respecting the "
            "per-question mark range."
        )

    role_indexes: dict[
        str,
        list[int],
    ] = {}

    for index, slot in enumerate(
        slots
    ):
        role = str(
            slot.get(
                "role",
                "supporting",
            )
            or "supporting"
        ).strip().casefold()

        role_indexes.setdefault(
            role,
            [],
        ).append(
            index
        )

        slot[
            "marks"
        ] = minimum_marks

    role_marks = {
        role:
            len(
                indexes
            )
            * minimum_marks
        for role, indexes
        in role_indexes.items()
    }

    role_caps = {
        role:
            len(
                indexes
            )
            * maximum_marks
        for role, indexes
        in role_indexes.items()
    }

    role_weight_units = {
        role:
            max(
                0.01,
                float(
                    ROLE_WEIGHTS.get(
                        role,
                        1.0,
                    )
                ),
            )
            * len(
                indexes
            )
        for role, indexes
        in role_indexes.items()
    }

    remaining = (
        total_marks
        - minimum_possible
    )

    # Allocate the remaining total to ROLES first.
    while remaining > 0:
        eligible_roles = [
            role
            for role in role_indexes
            if role_marks[
                role
            ] < role_caps[
                role
            ]
        ]

        if not eligible_roles:
            raise RuntimeError(
                "No role-level mark-allocation capacity remains."
            )

        chosen_role = min(
            eligible_roles,
            key=lambda role: (
                role_marks[
                    role
                ]
                / role_weight_units[
                    role
                ],
                role_marks[
                    role
                ],
                role,
            ),
        )

        role_marks[
            chosen_role
        ] += 1

        remaining -= 1

    # Spread each role budget across that role's own questions.
    for role, indexes in role_indexes.items():
        role_remaining = (
            role_marks[
                role
            ]
            - (
                len(
                    indexes
                )
                * minimum_marks
            )
        )

        while role_remaining > 0:
            eligible_indexes = [
                index
                for index in indexes
                if slots[
                    index
                ][
                    "marks"
                ] < maximum_marks
            ]

            if not eligible_indexes:
                raise RuntimeError(
                    f"No per-question capacity remains for role {role!r}."
                )

            chosen_index = min(
                eligible_indexes,
                key=lambda index: (
                    slots[
                        index
                    ][
                        "marks"
                    ],
                    index,
                ),
            )

            slots[
                chosen_index
            ][
                "marks"
            ] += 1

            role_remaining -= 1

    for index, slot in enumerate(
        slots,
        start=1,
    ):
        slot[
            "plan_index"
        ] = index

    assign_assessment_patterns(
        slots
    )

    return slots


def _coverage_item(
    topic: dict[str, Any],
) -> dict[str, Any]:
    return {
        "topic":
            topic[
                "topic"
            ],

        "topic_norm":
            topic[
                "topic_norm"
            ],

        "role":
            topic[
                "role"
            ],

        "official_reference":
            topic[
                "official_reference"
            ],
    }


def _slot_covered_norms(
    slot: dict[str, Any],
) -> set[str]:
    return {
        normalize_text(
            item.get(
                "topic_norm",
                item.get(
                    "topic",
                    "",
                ),
            )
        )
        for item in slot.get(
            "covered_topics",
            [],
        )
        if isinstance(
            item,
            dict,
        )
    }


def _source_chunk_overlap(
    left: dict[str, Any],
    right: dict[str, Any],
) -> int:
    left_chunks = {
        str(
            value
        )
        for value in safe_list(
            left.get(
                "source_chunks"
            )
        )
    }

    right_chunks = {
        str(
            value
        )
        for value in safe_list(
            right.get(
                "source_chunks"
            )
        )
    }

    return len(
        left_chunks
        & right_chunks
    )


def _attach_topic_to_best_slot(
    slots: list[
        dict[str, Any]
    ],
    topic: dict[str, Any],
    topic_by_norm: dict[
        str,
        dict[str, Any]
    ],
) -> None:
    """
    Add an approved topic as secondary coverage without creating a new
    question. Prefer a slot whose anchor topic appeared in overlapping
    Agent 1 source chunks; otherwise distribute coverage evenly.
    """
    topic_norm = topic[
        "topic_norm"
    ]

    if any(
        topic_norm
        in _slot_covered_norms(
            slot
        )
        for slot in slots
    ):
        return

    candidates = []

    for index, slot in enumerate(
        slots
    ):
        anchor = topic_by_norm[
            slot[
                "topic_norm"
            ]
        ]

        overlap = _source_chunk_overlap(
            anchor,
            topic,
        )

        candidates.append(
            (
                -overlap,
                len(
                    slot.get(
                        "covered_topics",
                        [],
                    )
                ),
                index,
            )
        )

    chosen_index = sorted(
        candidates
    )[
        0
    ][
        2
    ]

    slots[
        chosen_index
    ].setdefault(
        "covered_topics",
        [],
    ).append(
        _coverage_item(
            topic
        )
    )



def enforce_distinct_style_patterns_for_roles(
    slots: list[dict[str, Any]],
    roles: list[str],
) -> list[dict[str, Any]]:
    """
    Enforce an explicit user request for different question styles by role.

    Blueprint pattern labels must be unique first; semantic diversity is checked
    again after generation so different labels cannot hide near-duplicate tasks.
    """
    normalized_roles = {
        str(role or "").strip().casefold()
        for role in roles
        if str(role or "").strip()
    }

    for role in sorted(
        normalized_roles
    ):
        role_slots = [
            slot
            for slot in slots
            if str(
                slot.get(
                    "role",
                    "",
                )
                or ""
            ).strip().casefold()
            == role
        ]

        if len(
            role_slots
        ) > len(
            ASSESSMENT_PATTERN_CYCLE
        ):
            raise ValueError(
                "Special quiz instructions require different assessment styles "
                f"for {len(role_slots)} {role} questions, but the deterministic "
                f"style pool contains only {len(ASSESSMENT_PATTERN_CYCLE)} "
                "distinct patterns."
            )

        for index, slot in enumerate(
            role_slots
        ):
            slot[
                "assessment_pattern"
            ] = ASSESSMENT_PATTERN_CYCLE[
                index
            ]

        patterns = [
            str(
                slot.get(
                    "assessment_pattern",
                    "",
                )
                or ""
            )
            for slot in role_slots
        ]

        if len(
            patterns
        ) != len(
            set(
                patterns
            )
        ):
            raise AssertionError(
                "Internal planner invariant failed: a mandatory distinct-style "
                f"instruction for role {role!r} produced duplicate patterns."
            )

    return slots


def build_complete_quiz_blueprint(
    topics: list[
        dict[str, Any]
    ],
    request_payload: dict[str, Any],
) -> list[
    dict[str, Any]
]:
    """
    Build the exact complete-quiz blueprint from the user's controls.

    A question has one ANCHOR topic. The anchor determines that question's
    primary/supporting role for role-minimum counting.

    A question may also assess additional approved topics through
    covered_topics. This means "cover all approved topics" does not require
    one separate question per topic. The user-selected question count remains
    exact and the LLM receives the complete coverage assignment.
    """
    question_count = request_payload[
        "number_of_questions"
    ]

    target_marks = request_payload[
        "target_total_marks"
    ]

    minimum_marks = request_payload[
        "minimum_question_marks"
    ]

    maximum_marks = request_payload[
        "maximum_question_marks"
    ]

    minimum_primary = request_payload[
        "minimum_primary_questions"
    ]

    minimum_supporting = request_payload[
        "minimum_supporting_questions"
    ]

    special_directives = request_payload.get(
        "special_instruction_directives",
        {},
    )

    if not isinstance(
        special_directives,
        dict,
    ):
        special_directives = {}

    exact_primary = special_directives.get(
        "exact_primary_questions"
    )

    exact_supporting = special_directives.get(
        "exact_supporting_questions"
    )

    strict_role_allocation = bool(
        exact_primary is not None
        or exact_supporting is not None
    )

    if strict_role_allocation:
        exact_primary = safe_int(
            exact_primary,
            -1,
        )
        exact_supporting = safe_int(
            exact_supporting,
            -1,
        )

        if (
            exact_primary < 0
            or exact_supporting < 0
            or exact_primary + exact_supporting != question_count
        ):
            raise ValueError(
                "Resolved special-instruction exact role counts are invalid "
                "for the selected total question count."
            )

        if exact_primary < minimum_primary:
            raise ValueError(
                "Exact primary count from special instructions is below the "
                "selected minimum primary count."
            )

        if exact_supporting < minimum_supporting:
            raise ValueError(
                "Exact supporting count from special instructions is below the "
                "selected minimum supporting count."
            )

    minimum_distinct_refs = request_payload[
        "minimum_distinct_official_references"
    ]

    cover_all = bool(
        request_payload[
            "cover_all_approved_topics"
        ]
    )

    minimum_total_marks = (
        question_count
        * minimum_marks
    )

    maximum_total_marks = (
        question_count
        * maximum_marks
    )

    if not (
        minimum_total_marks
        <= target_marks
        <= maximum_total_marks
    ):
        raise ValueError(
            "The requested quiz cannot satisfy the selected marks settings. "
            f"{question_count} questions with {minimum_marks}-{maximum_marks} "
            f"marks each can total only {minimum_total_marks}-"
            f"{maximum_total_marks} marks, but the requested total is "
            f"{target_marks}."
        )

    if (
        minimum_primary
        + minimum_supporting
        > question_count
    ):
        raise ValueError(
            "The requested minimum primary and supporting question counts "
            "do not fit inside the selected number of questions."
        )

    primary_topics = [
        item
        for item in topics
        if item[
            "role"
        ]
        == "primary"
    ]

    supporting_topics = [
        item
        for item in topics
        if item[
            "role"
        ]
        == "supporting"
    ]

    required_primary_for_availability = (
        exact_primary
        if strict_role_allocation
        else minimum_primary
    )

    required_supporting_for_availability = (
        exact_supporting
        if strict_role_allocation
        else minimum_supporting
    )

    if (
        required_primary_for_availability > 0
        and not primary_topics
    ):
        raise ValueError(
            "Primary questions were requested, but no approved primary "
            "Agent 1 topic is available after the current filters."
        )

    if (
        required_supporting_for_availability > 0
        and not supporting_topics
    ):
        raise ValueError(
            "Supporting questions were requested, but no approved supporting "
            "Agent 1 topic is available after the current filters."
        )

    available_refs = {
        item[
            "official_reference"
        ]
        for item in topics
        if str(
            item.get(
                "official_reference",
                "",
            )
            or ""
        ).strip()
    }

    if (
        minimum_distinct_refs
        > len(
            available_refs
        )
    ):
        raise ValueError(
            "The requested minimum distinct official references exceeds "
            "the references available in the approved Agent 1 topics."
        )

    # ----------------------------------------------------------
    # Step 1: create EXACTLY the requested number of questions.
    # Role minima apply to ANCHOR roles, not to topic coverage.
    # ----------------------------------------------------------
    slots: list[
        dict[str, Any]
    ] = []

    assigned_counts = {
        item[
            "topic_norm"
        ]:
            0
        for item in topics
    }

    def add_anchor_slot(
        topic: dict[str, Any],
    ) -> None:
        add_slot(
            slots,
            topic,
            assigned_counts,
        )

        slots[
            -1
        ][
            "covered_topics"
        ] = [
            _coverage_item(
                topic
            )
        ]

    target_primary = (
        exact_primary
        if strict_role_allocation
        else minimum_primary
    )

    target_supporting = (
        exact_supporting
        if strict_role_allocation
        else minimum_supporting
    )

    while sum(
        1
        for slot in slots
        if slot[
            "role"
        ]
        == "primary"
    ) < target_primary:
        add_anchor_slot(
            choose_topic_for_role(
                topics,
                "primary",
                assigned_counts,
            )
        )

    while sum(
        1
        for slot in slots
        if slot[
            "role"
        ]
        == "supporting"
    ) < target_supporting:
        add_anchor_slot(
            choose_topic_for_role(
                topics,
                "supporting",
                assigned_counts,
            )
        )

    if strict_role_allocation:
        if len(
            slots
        ) != question_count:
            raise AssertionError(
                "Internal planner invariant failed: exact role allocation from "
                "special instructions does not fill the requested question count."
            )
    else:
        while len(
            slots
        ) < question_count:
            add_anchor_slot(
                choose_general_topic(
                    topics,
                    assigned_counts,
                )
            )

    if len(
        slots
    ) != question_count:
        raise AssertionError(
            "Internal planner invariant failed: anchor question count "
            "does not equal the user's requested question count."
        )

    # ----------------------------------------------------------
    # Step 2: distribute topic/reference coverage OVER the
    # existing questions. No extra question slots are created.
    # ----------------------------------------------------------
    topic_by_norm = {
        item[
            "topic_norm"
        ]:
            item
        for item in topics
    }

    if cover_all:
        for topic in sorted(
            topics,
            key=topic_priority,
        ):
            _attach_topic_to_best_slot(
                slots,
                topic,
                topic_by_norm,
            )

    def covered_refs() -> set[str]:
        refs: set[
            str
        ] = set()

        for slot in slots:
            for item in slot.get(
                "covered_topics",
                [],
            ):
                if not isinstance(
                    item,
                    dict,
                ):
                    continue

                reference = str(
                    item.get(
                        "official_reference",
                        "",
                    )
                    or ""
                ).strip()

                if reference:
                    refs.add(
                        reference
                    )

        return refs

    # Distinct-reference coverage can also be attached to an existing
    # question instead of inflating the question count.
    while len(
        covered_refs()
    ) < minimum_distinct_refs:
        missing_ref_topics = [
            item
            for item in topics
            if item[
                "official_reference"
            ]
            not in covered_refs()
        ]

        if not missing_ref_topics:
            raise ValueError(
                "Could not satisfy the distinct-reference requirement from "
                "the approved Agent 1 topics."
            )

        topic = sorted(
            missing_ref_topics,
            key=topic_priority,
        )[
            0
        ]

        _attach_topic_to_best_slot(
            slots,
            topic,
            topic_by_norm,
        )

    # ----------------------------------------------------------
    # Step 3: exact mark allocation over the exact question count.
    # ----------------------------------------------------------
    slots = allocate_exact_marks(
        slots,
        total_marks=target_marks,
        minimum_marks=minimum_marks,
        maximum_marks=maximum_marks,
    )

    distinct_style_roles = special_directives.get(
        "distinct_styles_for_roles",
        [],
    )

    if isinstance(
        distinct_style_roles,
        list,
    ) and distinct_style_roles:
        slots = enforce_distinct_style_patterns_for_roles(
            slots,
            distinct_style_roles,
        )

    return slots

def _shortfall_exact_fit_slot_count(
    *,
    missing_marks: int,
    missing_question_count: int,
    minimum_marks: int,
    maximum_marks: int,
) -> tuple[int, str | None]:
    """
    Return the MINIMUM number of positive-mark AI questions needed to satisfy
    the remaining fill_shortfall coverage targets.

    In fill_shortfall, BOTH requested questions and requested marks are minimum
    coverage targets. Question count is completed first. The mark target is met
    exactly when that is compatible with the required positive-mark questions;
    otherwise the planner allows only the minimum unavoidable mark overage.

    Examples:
      missing 1 mark, 0 questions -> 1 one-mark question
      missing 3 marks, 0 questions -> 1 three-mark question
      missing 6 marks, 1 question  -> 2 questions if max/question is 5
      missing 2 marks, 4 questions -> 4 questions; minimum marks may exceed gap
      missing 0 marks, 1 question  -> 1 minimum-mark question
    """
    missing_marks = max(0, safe_int(missing_marks))
    missing_question_count = max(
        0,
        safe_int(missing_question_count),
    )
    minimum_marks = max(1, safe_int(minimum_marks, 1))
    maximum_marks = max(
        minimum_marks,
        safe_int(maximum_marks, minimum_marks),
    )

    if missing_marks == 0 and missing_question_count == 0:
        return 0, None

    slots_for_marks = (
        math.ceil(missing_marks / maximum_marks)
        if missing_marks > 0
        else 0
    )

    required_slots = max(
        missing_question_count,
        slots_for_marks,
    )

    # If required question count forces more minimum marks than the remaining
    # mark gap, build_shortfall_blueprint allows only the minimum unavoidable
    # overage instead of blocking generation.
    return required_slots, None


def _shortfall_exact_fit_regression_probe() -> dict[str, Any]:
    """
    Zero-API local regression probe for the shortfall planner.
    """
    cases = {
        "19_of_20_and_5_of_5":
            _shortfall_exact_fit_slot_count(
                missing_marks=1,
                missing_question_count=0,
                minimum_marks=1,
                maximum_marks=5,
            ),

        "17_of_20_and_5_of_5":
            _shortfall_exact_fit_slot_count(
                missing_marks=3,
                missing_question_count=0,
                minimum_marks=1,
                maximum_marks=5,
            ),

        "14_of_20_and_4_of_5":
            _shortfall_exact_fit_slot_count(
                missing_marks=6,
                missing_question_count=1,
                minimum_marks=1,
                maximum_marks=5,
            ),

        "20_of_20_and_4_of_5_question_shortfall":
            _shortfall_exact_fit_slot_count(
                missing_marks=0,
                missing_question_count=1,
                minimum_marks=1,
                maximum_marks=5,
            ),

        "2_marks_and_4_questions_minimal_overage":
            _shortfall_exact_fit_slot_count(
                missing_marks=2,
                missing_question_count=4,
                minimum_marks=1,
                maximum_marks=5,
            ),
    }

    return {
        "cases": cases,
        "passed": bool(
            cases["19_of_20_and_5_of_5"] == (1, None)
            and cases["17_of_20_and_5_of_5"] == (1, None)
            and cases["14_of_20_and_4_of_5"] == (2, None)
            and cases["20_of_20_and_4_of_5_question_shortfall"] == (1, None)
            and cases["2_marks_and_4_questions_minimal_overage"] == (4, None)
        ),
    }


def build_shortfall_blueprint(
    topics: list[
        dict[str, Any]
    ],
    request_payload: dict[str, Any],
    official: list[
        dict[str, Any]
    ],
) -> tuple[
    list[
        dict[str, Any]
    ],
    dict[
        str,
        Any,
    ],
]:
    """
    Build the MINIMUM AI extension for fill_shortfall mode.

    Shortfall policy:
    - Notebook 05 official questions are fixed.
    - Requested question count and target marks are MINIMUM coverage targets.
    - Complete the missing question count first; every AI question must still
      respect the configured positive per-question mark range.
    - Meet the remaining mark target exactly when possible. If the missing
      question count requires more minimum marks than the remaining mark gap,
      allow only the minimum unavoidable mark overage.
    - Topic / role / reference coverage NEVER creates extra AI question slots.
      If generation is already required, those coverage needs are attached to
      the minimum supplemental slots as anchors / covered_topics.
    """
    target_marks = request_payload[
        "target_total_marks"
    ]

    requested_questions = request_payload[
        "number_of_questions"
    ]

    minimum_marks = request_payload[
        "minimum_question_marks"
    ]

    maximum_marks = request_payload[
        "maximum_question_marks"
    ]

    official_marks_value = sum(
        question_marks(question)
        for question in official
    )

    official_count_value = len(
        official
    )

    missing_marks = max(
        0,
        target_marks - official_marks_value,
    )

    missing_count = max(
        0,
        requested_questions - official_count_value,
    )

    official_topic_norms = {
        normalize_text(
            question_topic(question)
        )
        for question in official
        if question_topic(question)
    }

    official_role_counts = {
        "primary":
            sum(
                1
                for question in official
                if question_role(question) == "primary"
            ),

        "supporting":
            sum(
                1
                for question in official
                if question_role(question) == "supporting"
            ),
    }

    official_refs = {
        question_reference(question)
        for question in official
        if question_reference(question)
    }

    uncovered_topics = [
        item
        for item in topics
        if (
            request_payload[
                "cover_all_approved_topics"
            ]
            and item["topic_norm"]
            not in official_topic_norms
        )
    ]

    missing_primary = max(
        0,
        request_payload[
            "minimum_primary_questions"
        ]
        - official_role_counts[
            "primary"
        ],
    )

    missing_supporting = max(
        0,
        request_payload[
            "minimum_supporting_questions"
        ]
        - official_role_counts[
            "supporting"
        ],
    )

    distinct_ref_shortage = max(
        0,
        request_payload[
            "minimum_distinct_official_references"
        ]
        - len(official_refs),
    )

    # Only the quantitative user shortfall can create AI question slots.
    # Coverage constraints are fitted INTO those slots; they cannot inflate
    # a 1-mark shortfall into two 1-mark questions.
    required_slots, constraint_conflict = (
        _shortfall_exact_fit_slot_count(
            missing_marks=missing_marks,
            missing_question_count=missing_count,
            minimum_marks=minimum_marks,
            maximum_marks=maximum_marks,
        )
    )

    base_summary = {
        "official_marks":
            official_marks_value,

        "official_question_count":
            official_count_value,

        "missing_marks":
            missing_marks,

        "missing_question_count":
            missing_count,

        "uncovered_topics":
            [
                item["topic"]
                for item in uncovered_topics
            ],

        "missing_primary_questions":
            missing_primary,

        "missing_supporting_questions":
            missing_supporting,

        "distinct_reference_shortage":
            distinct_ref_shortage,

        "shortfall_completion_policy":
            "question_count_first_minimum_coverage",

        "exact_mark_target":
            False,
    }

    if constraint_conflict:
        return (
            [],
            {
                **base_summary,

                "retrieval_sufficient":
                    False,

                "constraint_conflict":
                    True,

                "constraint_conflict_reason":
                    constraint_conflict,

                "generated_mark_target":
                    0,

                "generated_question_count":
                    0,

                "projected_total_marks":
                    official_marks_value,

                "projected_total_questions":
                    official_count_value,

                "target_mark_overage":
                    0,

                "target_question_overage":
                    max(
                        0,
                        official_count_value
                        - requested_questions,
                    ),
            },
        )

    if required_slots == 0:
        return (
            [],
            {
                **base_summary,

                "retrieval_sufficient":
                    True,

                "constraint_conflict":
                    False,

                "generated_mark_target":
                    0,

                "generated_question_count":
                    0,

                "projected_total_marks":
                    official_marks_value,

                "projected_total_questions":
                    official_count_value,

                "target_mark_overage":
                    0,

                "target_question_overage":
                    max(
                        0,
                        official_count_value
                        - requested_questions,
                    ),
            },
        )

    assigned_counts = {
        item["topic_norm"]:
            0
        for item in topics
    }

    remaining_role_need = {
        "primary":
            missing_primary,
        "supporting":
            missing_supporting,
    }

    uncovered_norms = {
        item["topic_norm"]
        for item in uncovered_topics
    }

    anchor_topics: list[
        dict[str, Any]
    ] = []

    for _ in range(required_slots):
        item = min(
            topics,
            key=lambda candidate: (
                0
                if remaining_role_need.get(
                    candidate["role"],
                    0,
                ) > 0
                else 1,

                0
                if candidate["topic_norm"]
                in uncovered_norms
                else 1,

                assigned_counts[
                    candidate["topic_norm"]
                ],

                topic_priority(candidate),
            ),
        )

        anchor_topics.append(
            item
        )

        assigned_counts[
            item["topic_norm"]
        ] += 1

        role = item[
            "role"
        ]

        if remaining_role_need.get(
            role,
            0,
        ) > 0:
            remaining_role_need[
                role
            ] -= 1

    slots: list[
        dict[str, Any]
    ] = []

    local_counts: dict[
        str,
        int
    ] = {}

    for item in anchor_topics:
        add_slot(
            slots,
            item,
            local_counts,
        )

    topic_by_norm = {
        item["topic_norm"]:
            item
        for item in topics
    }

    # Attach ALL still-uncovered approved topics to the existing minimum slots.
    # This preserves Agent 1 coverage instructions without creating extra Qs.
    for item in sorted(
        uncovered_topics,
        key=topic_priority,
    ):
        _attach_topic_to_best_slot(
            slots,
            item,
            topic_by_norm,
        )

    def projected_covered_refs() -> set[str]:
        refs = set(
            official_refs
        )

        for slot in slots:
            anchor_ref = str(
                slot.get(
                    "official_reference",
                    "",
                )
                or ""
            ).strip()

            if anchor_ref:
                refs.add(
                    anchor_ref
                )

            for covered in slot.get(
                "covered_topics",
                [],
            ):
                if not isinstance(
                    covered,
                    dict,
                ):
                    continue

                reference = str(
                    covered.get(
                        "official_reference",
                        "",
                    )
                    or ""
                ).strip()

                if reference:
                    refs.add(
                        reference
                    )

        return refs

    # Distinct-reference coverage may be attached to existing slots but may
    # never add slots or marks.
    while (
        len(projected_covered_refs())
        < request_payload[
            "minimum_distinct_official_references"
        ]
    ):
        current_refs = projected_covered_refs()

        candidates = [
            item
            for item in topics
            if item[
                "official_reference"
            ]
            not in current_refs
        ]

        if not candidates:
            break

        item = sorted(
            candidates,
            key=topic_priority,
        )[0]

        _attach_topic_to_best_slot(
            slots,
            item,
            topic_by_norm,
        )

    # Fill-shortfall targets are minimum coverage targets. Prefer the exact
    # remaining mark gap, but if the required question count would need more
    # positive marks, allow only the minimum unavoidable overage.
    minimum_marks_for_required_slots = (
        required_slots * minimum_marks
    )

    generated_mark_target = max(
        missing_marks,
        minimum_marks_for_required_slots,
    )

    slots = allocate_exact_marks(
        slots,
        total_marks=generated_mark_target,
        minimum_marks=minimum_marks,
        maximum_marks=maximum_marks,
    )

    projected_total_marks = (
        official_marks_value
        + generated_mark_target
    )

    projected_total_questions = (
        official_count_value
        + required_slots
    )

    if projected_total_marks < target_marks:
        raise AssertionError(
            "Shortfall minimum-coverage invariant failed: projected combined "
            "marks are below the requested target."
        )

    if sum(
        safe_int(
            slot.get(
                "marks",
                0,
            )
        )
        for slot in slots
    ) != generated_mark_target:
        raise AssertionError(
            "Shortfall minimum-coverage invariant failed: generated blueprint "
            "marks do not equal the planned AI mark target."
        )

    generated_primary = sum(
        1
        for slot in slots
        if str(
            slot.get(
                "role",
                "",
            )
            or ""
        ).strip().casefold()
        == "primary"
    )

    generated_supporting = sum(
        1
        for slot in slots
        if str(
            slot.get(
                "role",
                "",
            )
            or ""
        ).strip().casefold()
        == "supporting"
    )

    summary = {
        **base_summary,

        "retrieval_sufficient":
            False,

        "constraint_conflict":
            False,

        "generated_mark_target":
            generated_mark_target,

        "generated_question_count":
            required_slots,

        "projected_total_marks":
            projected_total_marks,

        "projected_total_questions":
            projected_total_questions,

        "target_mark_overage":
            max(
                0,
                projected_total_marks
                - target_marks,
            ),

        "target_question_overage":
            max(
                0,
                projected_total_questions
                - requested_questions,
            ),

        "generated_primary_questions":
            generated_primary,

        "generated_supporting_questions":
            generated_supporting,

        "unresolved_primary_question_shortage":
            max(
                0,
                missing_primary
                - generated_primary,
            ),

        "unresolved_supporting_question_shortage":
            max(
                0,
                missing_supporting
                - generated_supporting,
            ),

        "projected_distinct_references":
            len(
                projected_covered_refs()
            ),
    }

    return (
        slots,
        summary,
    )


if PAPER_ROUTING_BLOCKED:

    generation_blueprint = []

    shortfall_summary = {
        "official_marks":
            official_marks,

        "official_question_count":
            official_question_count,

        "missing_marks":
            max(
                0,
                request["target_total_marks"] - official_marks,
            ),

        "missing_question_count":
            max(
                0,
                request["number_of_questions"] - official_question_count,
            ),

        "retrieval_sufficient":
            False,

        "routing_blocked":
            True,

        "routing_status":
            paper_routing_preflight_result.get(
                "status",
                "BLOCKED_PAPER_MISMATCH",
            ),

        "routing_message":
            PAPER_ROUTING_BLOCK_REASON,
    }


elif QUIZ_MODE == "complete_quiz":

    generation_blueprint = (
        build_complete_quiz_blueprint(
            approved_topics,
            request,
        )
    )

    shortfall_summary = {
        "official_marks":
            0,

        "official_question_count":
            0,

        "missing_marks":
            request[
                "target_total_marks"
            ],

        "missing_question_count":
            request[
                "number_of_questions"
            ],

        "retrieval_sufficient":
            False,
    }


else:

    (
        generation_blueprint,
        shortfall_summary,
    ) = build_shortfall_blueprint(
        approved_topics,
        request,
        official_questions,
    )


# ---------------------------------------------------------------
# Shortfall minimum-coverage hard guard
# ---------------------------------------------------------------
# This is local/deterministic and uses zero provider calls.
if (
    QUIZ_MODE == "fill_shortfall"
    and generation_blueprint
):
    _planned_shortfall_marks = sum(
        safe_int(
            slot.get(
                "marks",
                0,
            )
        )
        for slot in generation_blueprint
        if isinstance(
            slot,
            dict,
        )
    )

    _expected_shortfall_marks = safe_int(
        shortfall_summary.get(
            "generated_mark_target",
            0,
        )
    )

    if (
        _planned_shortfall_marks
        != _expected_shortfall_marks
    ):
        raise AssertionError(
            "Shortfall generation blocked locally: planned AI marks "
            "do not equal the planner's generated mark target."
        )

    if (
        safe_int(
            shortfall_summary.get(
                "projected_total_marks",
                0,
            )
        )
        < request[
            "target_total_marks"
        ]
    ):
        raise AssertionError(
            "Shortfall generation blocked locally: projected combined "
            "marks are below the requested minimum target."
        )


generation_required = bool(
    generation_blueprint
)


print(
    "Generation required:",
    generation_required,
)

if QUIZ_MODE == "fill_shortfall":
    print(
        "Shortfall completion:",
        f"missing={shortfall_summary.get('missing_marks', 0)} mark(s), ",
        f"{shortfall_summary.get('missing_question_count', 0)} question(s); ",
        f"AI plan={shortfall_summary.get('generated_mark_target', 0)} mark(s), ",
        f"{shortfall_summary.get('generated_question_count', 0)} question(s); ",
        f"model={GENERATION_MODEL_DISPLAY_NAME}."
    )

display(
    pd.DataFrame(
        generation_blueprint
    )
)

## Visual architecture — Notebook 06 → Notebook 08 handoff (v2.35)

The visual pipeline is now separated into **generation/specification** and **rendering**.

### Notebook 06 — visual intent + compact specification

Notebook 06 does not make a paid image-generation request.

```text
Approved Agent 1 topics
        ↓
Deterministic quiz blueprint
        ↓
Visual need + visual type planned
        ↓
Same quiz-generation LLM call
        ↓
question + marking guidance + compact visual.spec
        ↓
Strict visual-contract validation
        ↓
HITL / candidate state
        ↓
visual_tool_handoff.json
```

The visual specification schema remains unchanged so existing quiz-generation,
validation and HITL logic are not unnecessarily disturbed.

### Notebook 08 — canonical visual renderer layer

Notebook 08 routes each valid visual to one of **three backends**:

```text
                        Notebook 08
                             │
                 renderer registry / router
            ┌────────────────┼────────────────┐
            │                │                │
            ▼                ▼                ▼
        SchemDraw           Kroki       Local structured
            │                │              renderer
            │                │                │
     logic circuits    network diagrams      arrays
     Boolean gates     flowcharts            trace tables
                       CPU/block diagrams     truth tables
                       future technical       database tables
                       diagram engines        memory grids
                                              binary registers
                                              code blocks
```

### Why this split

- **SchemDraw** is specialist handling for proper logic-gate symbols.
- **Kroki** provides automatic layout for technical diagrams and lets us use
  GraphViz through a single HTTP rendering interface.
- **Local structured rendering** is retained only where it already works well:
  tables, grids, arrays, code and other exact-value assessment visuals.
- We do **not** maintain hand-positioned matplotlib network/flowchart/CPU layouts
  in Notebook 08.
- No generative-image model is introduced.

### Migration rule

Notebook 06 still contains its existing embedded deterministic visual renderer
for backward compatibility with the current PDF/Streamlit path. It is now a
**legacy compatibility path**, not the canonical future renderer architecture.

After Notebook 08 is verified, the existing MCP/controller can expose the same
three approved tool contracts:

```text
render_logic_visual()
render_technical_visual()
render_structured_visual()
```

No MCP implementation is added in this notebook version.


In [ ]:
# ================================================================
# COMPLETE VISUAL ARCHITECTURE — PHASE 1 + PHASE 2 + FINAL
# ================================================================
# Design rules:
# - Visual need/type is chosen deterministically BEFORE the LLM call.
# - The LLM returns only a compact visual spec in the SAME generation call.
# - Python validates every visual; the embedded renderer is retained only as a legacy compatibility path.
# - Canonical rendering is delegated by the Notebook 08 handoff to SchemDraw / Kroki / local structured rendering.
# - No fixed visual quota is forced. Text-only remains valid when appropriate.
# - Student-facing visual specs must not contain answer/solution fields.
# ================================================================

VISUAL_SCHEMA_VERSION = "agent2-visual-architecture-v3.0.0"
VISUAL_RENDERER = "deterministic_matplotlib_v3"

VISUAL_PHASE1_TYPES = (
    "code_block",
    "trace_table",
    "array_grid",
    "simple_flowchart",
)

VISUAL_PHASE2_TYPES = (
    "logic_gate_diagram",
    "truth_table",
    "network_diagram",
    "database_table",
)

VISUAL_FINAL_TYPES = (
    "cpu_block_diagram",
    "memory_grid",
    "binary_register",
)

VISUAL_TYPES = (
    "none",
    *VISUAL_PHASE1_TYPES,
    *VISUAL_PHASE2_TYPES,
    *VISUAL_FINAL_TYPES,
)

VISUAL_DIR = OUTPUT_DIR / "generated_visuals"

# Backward-compatible aliases so existing Streamlit/notebook consumers do not
# break merely because Phase 1 has now been expanded to the final architecture.
PHASE1_VISUAL_SCHEMA_VERSION = VISUAL_SCHEMA_VERSION
PHASE1_VISUAL_RENDERER = VISUAL_RENDERER
PHASE1_VISUAL_TYPES = VISUAL_TYPES
PHASE1_VISUAL_DIR = VISUAL_DIR


def normalize_visual_requirement(value: Any) -> str:
    normalized = str(value or "none").strip().casefold()

    aliases = {
        "": "none",
        "no": "none",
        "false": "none",
        "flowchart": "simple_flowchart",
        "flow_chart": "simple_flowchart",
        "array": "array_grid",
        "table": "trace_table",
        "code": "code_block",
        "logic": "logic_gate_diagram",
        "logic_gate": "logic_gate_diagram",
        "logic_gates": "logic_gate_diagram",
        "truth": "truth_table",
        "network": "network_diagram",
        "database": "database_table",
        "cpu": "cpu_block_diagram",
        "memory": "memory_grid",
        "binary": "binary_register",
        "bit_register": "binary_register",
    }

    normalized = aliases.get(normalized, normalized)
    return normalized if normalized in VISUAL_TYPES else "none"


def visual_phase_for_type(value: Any) -> str:
    visual_type = normalize_visual_requirement(value)
    if visual_type == "none":
        return "none"
    if visual_type in VISUAL_PHASE1_TYPES:
        return "phase_1"
    if visual_type in VISUAL_PHASE2_TYPES:
        return "phase_2"
    if visual_type in VISUAL_FINAL_TYPES:
        return "final"
    return "none"


def visual_requirement_for_slot(
    slot: dict[str, Any],
    style_calibration: dict[str, Any],
    assessment_filters: dict[str, Any],
    source_topic: dict[str, Any] | None = None,
) -> str:
    """
    Generic visual planner.

    Visual form is selected only when a reviewed knowledge-base / official
    exemplar for the current official reference provides a concrete visual-form
    signal.

    No topic-name keyword routing is used here.
    """
    if not bool(
        assessment_filters.get(
            "include_visual_questions",
            True,
        )
    ):
        return "none"

    metadata = style_calibration.get(
        "kb_exemplar_visual",
        {},
    )

    if not isinstance(
        metadata,
        dict,
    ):
        metadata = {}

    evidence_present = bool(
        metadata.get(
            "visual_evidence_present",
            False,
        )
    )

    preferred = str(
        metadata.get(
            "preferred_visual_type",
            "",
        )
        or ""
    ).strip()

    if (
        evidence_present
        and preferred in VISUAL_TYPES
        and preferred != "none"
    ):
        return preferred

    # Conservative generic fallback:
    # if reviewed evidence does not identify a concrete visual form, do not
    # invent one from topic keywords.
    return "none"


def phase1_visual_requirement_for_slot(
    slot: dict[str, Any],
    style_calibration: dict[str, Any],
    assessment_filters: dict[str, Any],
    source_topic: dict[str, Any] | None = None,
) -> str:
    return visual_requirement_for_slot(
        slot=slot,
        style_calibration=style_calibration,
        assessment_filters=assessment_filters,
        source_topic=source_topic,
    )

_VISUAL_FORBIDDEN_STUDENT_KEYS = {
    "answer",
    "answers",
    "solution",
    "solutions",
    "correct_answer",
    "correct_value",
    "mark_scheme",
    "marking_guidance",
}


def _visual_contains_forbidden_student_key(value: Any) -> bool:
    if isinstance(value, dict):
        for key, child in value.items():
            if str(key).strip().casefold() in _VISUAL_FORBIDDEN_STUDENT_KEYS:
                return True
            if _visual_contains_forbidden_student_key(child):
                return True
        return False
    if isinstance(value, list):
        return any(_visual_contains_forbidden_student_key(item) for item in value)
    return False


def _validate_rectangular_table(
    rows: Any,
    *,
    min_rows: int,
    max_rows: int,
    min_cols: int,
    max_cols: int,
    label: str,
) -> tuple[list[str], list[list[Any]], int]:
    errors: list[str] = []
    if not isinstance(rows, list) or not (min_rows <= len(rows) <= max_rows):
        return ([f"{label} requires {min_rows}-{max_rows} rows"], [], 0)

    normalized_rows: list[list[Any]] = []
    for index, row in enumerate(rows, start=1):
        if not isinstance(row, list):
            errors.append(f"{label} row {index} must be a list")
            continue
        normalized_rows.append(row)

    if not normalized_rows:
        return (errors, [], 0)

    width = len(normalized_rows[0])
    if not (min_cols <= width <= max_cols):
        errors.append(f"{label} requires {min_cols}-{max_cols} columns")

    if any(len(row) != width for row in normalized_rows):
        errors.append(f"{label} rows must be rectangular")

    return (errors, normalized_rows, width)


def validate_visual_spec(
    visual: Any,
    required_type: str,
) -> list[str]:
    """Validate the complete visual contract without rendering it."""
    errors: list[str] = []
    required_type = normalize_visual_requirement(required_type)

    if required_type == "none":
        if visual is None or visual == "":
            return []
        if not isinstance(visual, dict):
            return ["visual must be null/empty when visual_requirement=none"]
        actual_type = normalize_visual_requirement(visual.get("type", "none"))
        if actual_type != "none":
            errors.append("visual.type must be none when visual_requirement=none")
        if visual.get("spec", {}) not in (None, {}):
            errors.append("visual.spec must be empty when visual_requirement=none")
        return errors

    if not isinstance(visual, dict):
        return [f"visual must be an object for visual_requirement={required_type}"]

    actual_type = normalize_visual_requirement(visual.get("type", ""))
    if actual_type != required_type:
        errors.append(f"visual.type mismatch ({actual_type!r} != {required_type!r})")

    spec = visual.get("spec", {})
    if not isinstance(spec, dict):
        return errors + ["visual.spec must be an object"]

    if _visual_contains_forbidden_student_key(spec):
        errors.append("student-facing visual.spec contains answer/solution fields")

    if any(key in spec for key in ["image_url", "url", "external_image"]):
        errors.append("visual.spec cannot depend on an external image URL")

    if required_type == "code_block":
        code = str(spec.get("code", "") or "").strip("\n")
        if not code.strip():
            errors.append("code_block spec.code is empty")
        if len(code) > 5000:
            errors.append("code_block spec.code is unexpectedly long")
        if not str(spec.get("language", "") or "").strip():
            errors.append("code_block spec.language is empty")

    elif required_type == "trace_table":
        columns = spec.get("columns", [])
        rows = spec.get("rows", [])
        if not isinstance(columns, list) or not (2 <= len(columns) <= 8):
            errors.append("trace_table requires 2-8 columns")
            columns = []
        if columns and any(not str(v or "").strip() for v in columns):
            errors.append("trace_table column labels cannot be empty")
        table_errors, _, width = _validate_rectangular_table(
            rows,
            min_rows=1,
            max_rows=14,
            min_cols=2,
            max_cols=8,
            label="trace_table",
        )
        errors.extend(table_errors)
        if columns and width and width != len(columns):
            errors.append("trace_table rows do not match the column count")

    elif required_type == "array_grid":
        values = spec.get("values", [])
        if not isinstance(values, list) or not values:
            errors.append("array_grid spec.values must be a non-empty list")
        else:
            rows = values if isinstance(values[0], list) else [values]
            if any(not isinstance(row, list) for row in rows):
                errors.append("array_grid cannot mix 1D and 2D rows")
            else:
                width = len(rows[0]) if rows else 0
                if not (1 <= len(rows) <= 8 and 1 <= width <= 12):
                    errors.append("array_grid supports at most 8 rows x 12 columns")
                if any(len(row) != width for row in rows):
                    errors.append("array_grid rows must be rectangular")
                row_labels = spec.get("row_labels")
                col_labels = spec.get("column_labels")
                if row_labels is not None and (
                    not isinstance(row_labels, list) or len(row_labels) != len(rows)
                ):
                    errors.append("array_grid row_labels must match the number of rows")
                if col_labels is not None and (
                    not isinstance(col_labels, list) or len(col_labels) != width
                ):
                    errors.append("array_grid column_labels must match the number of columns")

    elif required_type == "simple_flowchart":
        nodes = spec.get("nodes", [])
        edges = spec.get("edges", [])
        if not isinstance(nodes, list) or not (2 <= len(nodes) <= 10):
            errors.append("simple_flowchart requires 2-10 nodes")
            nodes = []
        allowed = {"start_end", "process", "decision", "input_output"}
        ids: list[str] = []
        for i, node in enumerate(nodes, start=1):
            if not isinstance(node, dict):
                errors.append(f"simple_flowchart node {i} must be an object")
                continue
            node_id = str(node.get("id", "") or "").strip()
            node_type = str(node.get("type", "") or "").strip().casefold()
            text = str(node.get("text", "") or "").strip()
            if not node_id:
                errors.append(f"simple_flowchart node {i} has no id")
            else:
                ids.append(node_id)
            if node_type not in allowed:
                errors.append(f"simple_flowchart node {i} has unsupported type={node_type!r}")
            if not text:
                errors.append(f"simple_flowchart node {i} has empty text")
        if len(ids) != len(set(ids)):
            errors.append("simple_flowchart node ids must be unique")
        if not isinstance(edges, list) or not edges:
            errors.append("simple_flowchart requires at least one edge")
            edges = []
        id_set = set(ids)
        for i, edge in enumerate(edges, start=1):
            if not isinstance(edge, dict):
                errors.append(f"simple_flowchart edge {i} must be an object")
                continue
            source = str(edge.get("from", "") or "").strip()
            target = str(edge.get("to", "") or "").strip()
            if source not in id_set:
                errors.append(f"simple_flowchart edge {i} has unknown from={source!r}")
            if target not in id_set:
                errors.append(f"simple_flowchart edge {i} has unknown to={target!r}")
            if source and source == target:
                errors.append(f"simple_flowchart edge {i} cannot point to itself")

    elif required_type == "truth_table":
        columns = spec.get("columns", [])
        rows = spec.get("rows", [])
        if not isinstance(columns, list) or not (2 <= len(columns) <= 8):
            errors.append("truth_table requires 2-8 columns")
            columns = []
        table_errors, normalized_rows, width = _validate_rectangular_table(
            rows,
            min_rows=2,
            max_rows=16,
            min_cols=2,
            max_cols=8,
            label="truth_table",
        )
        errors.extend(table_errors)
        if columns and width and width != len(columns):
            errors.append("truth_table rows do not match the column count")
        allowed_values = {0, 1, "0", "1", "", None, "T", "F", "t", "f"}
        for r, row in enumerate(normalized_rows, start=1):
            for c, value in enumerate(row, start=1):
                if value not in allowed_values:
                    errors.append(f"truth_table cell ({r},{c}) must be 0/1/T/F/blank")

    elif required_type == "logic_gate_diagram":
        inputs = spec.get("inputs", [])
        gates = spec.get("gates", [])
        output = spec.get("output", {})
        if not isinstance(inputs, list) or not (1 <= len(inputs) <= 6):
            errors.append("logic_gate_diagram requires 1-6 input labels")
            inputs = []
        input_names = [str(v or "").strip() for v in inputs]
        if any(not v for v in input_names) or len(input_names) != len(set(input_names)):
            errors.append("logic_gate_diagram input labels must be non-empty and unique")
        if not isinstance(gates, list) or not (1 <= len(gates) <= 7):
            errors.append("logic_gate_diagram requires 1-7 gates")
            gates = []
        allowed_gate_types = {"AND", "OR", "NOT", "NAND", "NOR", "XOR"}
        gate_ids: list[str] = []
        for i, gate in enumerate(gates, start=1):
            if not isinstance(gate, dict):
                errors.append(f"logic_gate_diagram gate {i} must be an object")
                continue
            gate_id = str(gate.get("id", "") or "").strip()
            gate_type = str(gate.get("type", "") or "").strip().upper()
            refs = gate.get("inputs", [])
            if not gate_id:
                errors.append(f"logic_gate_diagram gate {i} has no id")
            else:
                gate_ids.append(gate_id)
            if gate_type not in allowed_gate_types:
                errors.append(f"logic_gate_diagram gate {i} has unsupported type={gate_type!r}")
            expected_inputs = 1 if gate_type == "NOT" else 2
            if not isinstance(refs, list) or len(refs) != expected_inputs:
                errors.append(f"logic_gate_diagram gate {i} requires {expected_inputs} input reference(s)")
        if len(gate_ids) != len(set(gate_ids)):
            errors.append("logic_gate_diagram gate ids must be unique")
        known = set(input_names)
        for i, gate in enumerate(gates, start=1):
            if not isinstance(gate, dict):
                continue
            gate_id = str(gate.get("id", "") or "").strip()
            for ref in gate.get("inputs", []) if isinstance(gate.get("inputs", []), list) else []:
                ref_text = str(ref or "").strip()
                if ref_text not in known:
                    errors.append(f"logic_gate_diagram gate {i} references unknown/forward input={ref_text!r}")
            if gate_id:
                known.add(gate_id)
        if not isinstance(output, dict):
            errors.append("logic_gate_diagram output must be an object")
        else:
            source = str(output.get("from", "") or "").strip()
            if source not in set(gate_ids):
                errors.append("logic_gate_diagram output.from must reference a gate id")

    elif required_type == "network_diagram":
        nodes = spec.get("nodes", [])
        edges = spec.get("edges", [])
        allowed_types = {"computer", "laptop", "server", "switch", "router", "printer", "access_point", "cloud", "firewall", "device"}
        if not isinstance(nodes, list) or not (2 <= len(nodes) <= 12):
            errors.append("network_diagram requires 2-12 nodes")
            nodes = []
        ids: list[str] = []
        for i, node in enumerate(nodes, start=1):
            if not isinstance(node, dict):
                errors.append(f"network_diagram node {i} must be an object")
                continue
            node_id = str(node.get("id", "") or "").strip()
            node_type = str(node.get("type", "device") or "device").strip().casefold()
            label = str(node.get("label", "") or "").strip()
            if not node_id:
                errors.append(f"network_diagram node {i} has no id")
            else:
                ids.append(node_id)
            if node_type not in allowed_types:
                errors.append(f"network_diagram node {i} has unsupported type={node_type!r}")
            if not label:
                errors.append(f"network_diagram node {i} has empty label")
        if len(ids) != len(set(ids)):
            errors.append("network_diagram node ids must be unique")
        id_set = set(ids)
        if not isinstance(edges, list) or not (1 <= len(edges) <= 20):
            errors.append("network_diagram requires 1-20 edges")
            edges = []
        for i, edge in enumerate(edges, start=1):
            if not isinstance(edge, dict):
                errors.append(f"network_diagram edge {i} must be an object")
                continue
            source = str(edge.get("from", "") or "").strip()
            target = str(edge.get("to", "") or "").strip()
            if source not in id_set or target not in id_set:
                errors.append(f"network_diagram edge {i} references an unknown node")
            if source and source == target:
                errors.append(f"network_diagram edge {i} cannot point to itself")

    elif required_type == "database_table":
        columns = spec.get("columns", [])
        rows = spec.get("rows", [])
        if not isinstance(columns, list) or not (2 <= len(columns) <= 8):
            errors.append("database_table requires 2-8 columns")
            columns = []
        table_errors, _, width = _validate_rectangular_table(
            rows,
            min_rows=1,
            max_rows=12,
            min_cols=2,
            max_cols=8,
            label="database_table",
        )
        errors.extend(table_errors)
        if columns and width and width != len(columns):
            errors.append("database_table rows do not match the column count")
        pk = spec.get("primary_key")
        if pk not in (None, "") and str(pk) not in [str(v) for v in columns]:
            errors.append("database_table primary_key must name one of the columns")

    elif required_type == "cpu_block_diagram":
        components = spec.get("components", [])
        connections = spec.get("connections", [])
        allowed_types = {"alu", "control_unit", "register", "memory", "cache", "bus", "input", "output", "component"}
        if not isinstance(components, list) or not (3 <= len(components) <= 10):
            errors.append("cpu_block_diagram requires 3-10 components")
            components = []
        ids: list[str] = []
        for i, component in enumerate(components, start=1):
            if not isinstance(component, dict):
                errors.append(f"cpu_block_diagram component {i} must be an object")
                continue
            component_id = str(component.get("id", "") or "").strip()
            component_type = str(component.get("type", "component") or "component").strip().casefold()
            label = str(component.get("label", "") or "").strip()
            if not component_id:
                errors.append(f"cpu_block_diagram component {i} has no id")
            else:
                ids.append(component_id)
            if component_type not in allowed_types:
                errors.append(f"cpu_block_diagram component {i} has unsupported type={component_type!r}")
            if not label:
                errors.append(f"cpu_block_diagram component {i} has empty label")
        if len(ids) != len(set(ids)):
            errors.append("cpu_block_diagram component ids must be unique")
        id_set = set(ids)
        if not isinstance(connections, list) or not connections:
            errors.append("cpu_block_diagram requires at least one connection")
            connections = []
        for i, connection in enumerate(connections, start=1):
            if not isinstance(connection, dict):
                errors.append(f"cpu_block_diagram connection {i} must be an object")
                continue
            source = str(connection.get("from", "") or "").strip()
            target = str(connection.get("to", "") or "").strip()
            if source not in id_set or target not in id_set:
                errors.append(f"cpu_block_diagram connection {i} references an unknown component")

    elif required_type == "memory_grid":
        addresses = spec.get("addresses", [])
        values = spec.get("values", [])
        if not isinstance(addresses, list) or not (2 <= len(addresses) <= 16):
            errors.append("memory_grid requires 2-16 addresses")
            addresses = []
        if not isinstance(values, list) or len(values) != len(addresses):
            errors.append("memory_grid values must match the address count")
        if len([str(v) for v in addresses]) != len(set(str(v) for v in addresses)):
            errors.append("memory_grid addresses must be unique")

    elif required_type == "binary_register":
        bits = spec.get("bits", [])
        if not isinstance(bits, list) or not (4 <= len(bits) <= 16):
            errors.append("binary_register requires 4-16 bit cells")
            bits = []
        allowed_bits = {0, 1, "0", "1", "", None}
        if any(bit not in allowed_bits for bit in bits):
            errors.append("binary_register bits must be 0/1/blank")
        place_values = spec.get("place_values")
        if place_values is not None and (
            not isinstance(place_values, list) or len(place_values) != len(bits)
        ):
            errors.append("binary_register place_values must match the bit count")
        labels = spec.get("labels")
        if labels is not None and (
            not isinstance(labels, list) or len(labels) != len(bits)
        ):
            errors.append("binary_register labels must match the bit count")

    return errors


# Backward-compatible validator used elsewhere in the Phase 1 notebook.
def validate_phase1_visual_spec(visual: Any, required_type: str) -> list[str]:
    return validate_visual_spec(visual, required_type)


VISUAL_CONTRACT_EXAMPLES: dict[str, dict[str, Any]] = {
    "none": {"type": "none", "spec": {}},
    "code_block": {
        "type": "code_block",
        "spec": {"code": "multi-line code/pseudocode", "language": "Python or pseudocode", "caption": ""},
    },
    "trace_table": {
        "type": "trace_table",
        "spec": {"columns": ["Step", "i", "total"], "rows": [["Initial", "0", "0"], ["1", "", ""]], "caption": ""},
    },
    "array_grid": {
        "type": "array_grid",
        "spec": {"values": [[4, 8, 2], [7, 5, 9]], "row_labels": ["0", "1"], "column_labels": ["0", "1", "2"], "caption": ""},
    },
    "simple_flowchart": {
        "type": "simple_flowchart",
        "spec": {"nodes": [{"id": "n1", "type": "start_end", "text": "Start"}, {"id": "n2", "type": "process", "text": "count ← 0"}], "edges": [{"from": "n1", "to": "n2", "label": ""}], "caption": ""},
    },
    "logic_gate_diagram": {
        "type": "logic_gate_diagram",
        "spec": {"inputs": ["A", "B"], "gates": [{"id": "g1", "type": "AND", "inputs": ["A", "B"]}], "output": {"from": "g1", "label": "Q"}, "caption": ""},
    },
    "truth_table": {
        "type": "truth_table",
        "spec": {"columns": ["A", "B", "Q"], "rows": [[0, 0, ""], [0, 1, ""], [1, 0, ""], [1, 1, ""]], "caption": ""},
    },
    "network_diagram": {
        "type": "network_diagram",
        "spec": {"nodes": [{"id": "r1", "type": "router", "label": "Router"}, {"id": "c1", "type": "computer", "label": "PC 1"}], "edges": [{"from": "r1", "to": "c1", "label": ""}], "caption": ""},
    },
    "database_table": {
        "type": "database_table",
        "spec": {"columns": ["StudentID", "Name", "Score"], "rows": [[101, "Ali", 18], [102, "Sam", 15]], "primary_key": "StudentID", "caption": ""},
    },
    "cpu_block_diagram": {
        "type": "cpu_block_diagram",
        "spec": {"components": [{"id": "cu", "type": "control_unit", "label": "Control Unit"}, {"id": "alu", "type": "alu", "label": "ALU"}, {"id": "reg", "type": "register", "label": "Registers"}], "connections": [{"from": "cu", "to": "alu", "label": ""}, {"from": "alu", "to": "reg", "label": ""}], "caption": ""},
    },
    "memory_grid": {
        "type": "memory_grid",
        "spec": {"addresses": ["100", "101", "102", "103"], "values": ["12", "7", "", "4"], "caption": ""},
    },
    "binary_register": {
        "type": "binary_register",
        "spec": {"bits": [1, 0, 1, 1, 0, 0, 1, 0], "place_values": [128, 64, 32, 16, 8, 4, 2, 1], "caption": ""},
    },
}


def visual_contract_for_blueprint(
    blueprint_items: list[dict[str, Any]],
) -> dict[str, dict[str, Any]]:
    """Return only contracts needed by the current batch to keep prompts small."""
    required = {
        normalize_visual_requirement(item.get("visual_requirement", "none"))
        for item in blueprint_items
        if isinstance(item, dict)
    }
    required.add("none")
    return {
        visual_type: VISUAL_CONTRACT_EXAMPLES[visual_type]
        for visual_type in VISUAL_TYPES
        if visual_type in required
    }


## 7. Build grounded generation request

In [ ]:
def topic_evidence(
    topic: dict[str, Any],
) -> list[str]:
    """
    Return a compact, unique grounding set for one approved Agent 1 topic.

    The full Agent 1 handoff is unchanged. Only the LLM transport copy is
    compressed: source chunks are ranked by lexical relevance to the approved
    topic, then a small number of the strongest snippets are carried forward.
    """
    direct = [
        str(value).strip()
        for value in topic.get("source_chunk_texts", [])
        if str(value).strip()
    ]

    if direct:
        raw_values = direct
    elif LESSON_SUMMARY:
        raw_values = [str(LESSON_SUMMARY).strip()]
    else:
        raw_values = [
            f"Approved Agent 1 topic: {topic['topic']} "
            f"({topic['official_reference']})."
        ]

    unique_values: list[str] = []
    seen: set[str] = set()

    for value in raw_values:
        normalized = normalize_text(value)
        if not normalized or normalized in seen:
            continue
        seen.add(normalized)
        unique_values.append(value)

    # Rank evidence before trimming so a smaller prompt retains the snippets
    # most directly connected to the approved topic instead of simply keeping
    # the first transcript chunk encountered.
    stop_words = {
        "and", "or", "the", "a", "an", "of", "to", "in", "on",
        "for", "with", "computer", "science",
    }
    topic_tokens = {
        token
        for token in re.findall(r"[a-z0-9]+", normalize_text(topic.get("topic", "")))
        if len(token) >= 3 and token not in stop_words
    }

    ranked_pairs = []
    for index, value in enumerate(unique_values):
        value_tokens = set(re.findall(r"[a-z0-9]+", normalize_text(value)))
        overlap = len(topic_tokens & value_tokens) if topic_tokens else 0
        ranked_pairs.append((-overlap, index, value))

    ranked_values = [
        value
        for _, _, value in sorted(ranked_pairs)
    ]

    remaining = LLM_TOPIC_EVIDENCE_CHAR_BUDGET
    compact: list[str] = []
    max_snippets = 3
    per_snippet_cap = (
        remaining
        if len(ranked_values) <= 1
        else min(700, remaining)
    )

    for value in ranked_values:
        if remaining <= 0 or len(compact) >= max_snippets:
            break

        take = min(len(value), remaining, per_snippet_cap)
        compact_value = value[:take].rstrip()
        if take < len(value):
            compact_value += " …"

        if compact_value:
            compact.append(compact_value)
            remaining -= take

    return compact


# ---------------------------------------------------------------
# AQA question-bank style calibration for COMPLETE quiz generation.
#
# The LLM must not invent its idea of "AQA-like" difficulty from the syllabus
# name alone.  Instead, Notebook 06 reads a very small set of already-ingested,
# reviewed assessment questions from the Agent 2 question bank and uses them
# ONLY as calibration examples for level, command wording, context/code density
# and the amount of learner work expected for the assigned marks.
#
# Important:
# - This is a READ-ONLY lookup.
# - Notebook 05 is still NOT called in complete_quiz mode.
# - No extra LLM call is made.
# - Examples are compact and bounded so they do not inflate the prompt.
# - Existing deterministic anti-copy checks compare generated questions against
#   these examples and flag close copying.
# ---------------------------------------------------------------

STYLE_REFERENCE_EXAMPLES_PER_SLOT = 2
STYLE_REFERENCE_CONTEXT_CHAR_LIMIT = 400
STYLE_REFERENCE_QUESTION_CHAR_LIMIT = 1100
STYLE_REFERENCE_TOTAL_CHAR_LIMIT = 2000

_style_reference_bank_cache: pd.DataFrame | None = None
_style_reference_bank_source = "not_loaded"


def _style_bool(value: Any) -> bool:
    if isinstance(value, bool):
        return value

    return str(value or "").strip().casefold() in {
        "1",
        "true",
        "yes",
        "y",
        "on",
    }


def _style_optional_column(table: Any, names: list[str]):
    for name in names:
        if name in table.c:
            return table.c[name]
    return None


def _load_style_reference_bank_from_postgres(
    required_references: list[str],
) -> pd.DataFrame:
    database_url = str(
        os.getenv("AGENT2_DATABASE_URL", "")
        or ""
    ).strip()

    if not database_url or not required_references:
        return pd.DataFrame()

    from sqlalchemy import MetaData, Table, create_engine, literal, select

    engine = create_engine(
        database_url,
        pool_pre_ping=True,
        future=True,
    )

    metadata = MetaData()

    topics = Table(
        "assessment_topical_topics",
        metadata,
        autoload_with=engine,
    )

    questions = Table(
        "assessment_topical_questions",
        metadata,
        autoload_with=engine,
    )

    mappings = Table(
        "assessment_topic_official_mappings",
        metadata,
        autoload_with=engine,
    )

    def selected_or_null(column: Any, label: str):
        return (
            column.label(label)
            if column is not None
            else literal(None).label(label)
        )

    q_context = _style_optional_column(
        questions,
        ["context_text"],
    )
    q_uid = _style_optional_column(
        questions,
        ["question_uid", "uid"],
    )
    q_review = _style_optional_column(
        questions,
        ["review_status"],
    )
    q_retrieval_enabled = _style_optional_column(
        questions,
        ["retrieval_enabled"],
    )
    q_is_active = _style_optional_column(
        questions,
        ["is_active"],
    )
    q_is_legacy = _style_optional_column(
        questions,
        ["is_legacy"],
    )
    q_record_type = _style_optional_column(
        questions,
        ["record_type"],
    )
    q_has_code = _style_optional_column(
        questions,
        ["has_code"],
    )
    q_has_visual = _style_optional_column(
        questions,
        ["has_visual"],
    )

    t_paper = _style_optional_column(
        topics,
        ["paper_code"],
    )
    t_language = _style_optional_column(
        topics,
        ["programming_language"],
    )

    m_reference = _style_optional_column(
        mappings,
        ["official_reference"],
    )
    m_status = _style_optional_column(
        mappings,
        ["mapping_status"],
    )
    m_human_approved = _style_optional_column(
        mappings,
        ["human_approved"],
    )

    if m_reference is None:
        return pd.DataFrame()

    statement = (
        select(
            questions.c.id.label("question_id"),
            selected_or_null(q_uid, "question_uid"),
            m_reference.label("official_reference"),
            questions.c.question_number.label("question_number"),
            questions.c.marks.label("marks"),
            questions.c.question_text.label("question_text"),
            selected_or_null(q_context, "context_text"),
            selected_or_null(q_has_code, "has_code"),
            selected_or_null(q_has_visual, "has_visual"),
            selected_or_null(t_paper, "paper_code"),
            selected_or_null(t_language, "programming_language"),
        )
        .select_from(
            questions
            .join(
                topics,
                questions.c.topic_id == topics.c.id,
            )
            .join(
                mappings,
                mappings.c.topic_id == topics.c.id,
            )
        )
        .where(
            m_reference.in_(required_references)
        )
        .where(
            questions.c.marks.is_not(None)
        )
        .where(
            questions.c.question_text.is_not(None)
        )
    )

    if m_status is not None:
        statement = statement.where(
            m_status == "approved"
        )

    if m_human_approved is not None:
        statement = statement.where(
            m_human_approved.is_(True)
        )

    if q_review is not None:
        statement = statement.where(
            q_review.in_(
                [
                    "human_approved",
                    "human_corrected",
                ]
            )
        )

    if q_retrieval_enabled is not None:
        statement = statement.where(
            q_retrieval_enabled.is_(True)
        )

    if q_is_active is not None:
        statement = statement.where(
            q_is_active.is_(True)
        )

    if q_is_legacy is not None:
        statement = statement.where(
            q_is_legacy.is_(False)
        )

    if q_record_type is not None:
        statement = statement.where(
            q_record_type == "scored_item"
        )

    with engine.connect() as connection:
        frame = pd.read_sql(
            statement,
            connection,
        )

    return frame


def _load_style_reference_bank_from_outputs(
    required_references: list[str],
) -> pd.DataFrame:
    """
    Local read-only fallback for notebook/repo runs where PostgreSQL is not
    reachable.  It uses the latest existing parsed-question export plus the
    approved official topic mapping already present in Agent2/OUTPUT.
    """
    output_root = AGENT2_ROOT / "OUTPUT"

    parsed_files = sorted(
        output_root.glob(
            "pmt_topical_parsed_questions_batch_*.csv"
        ),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )

    mapping_files = sorted(
        output_root.glob(
            "agent2_official_topic_mapping_apply_*.csv"
        ),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )

    if (
        not parsed_files
        or not mapping_files
        or not required_references
    ):
        return pd.DataFrame()

    questions_df = pd.read_csv(
        parsed_files[0]
    )
    mappings_df = pd.read_csv(
        mapping_files[0]
    )

    if "mapping_status" in mappings_df.columns:
        mappings_df = mappings_df[
            mappings_df["mapping_status"]
            .fillna("")
            .astype(str)
            .str.casefold()
            .eq("approved")
        ]

    if "human_approved" in mappings_df.columns:
        mappings_df = mappings_df[
            mappings_df["human_approved"].map(
                _style_bool
            )
        ]

    mappings_df = mappings_df[
        mappings_df["official_reference"]
        .fillna("")
        .astype(str)
        .isin(required_references)
    ].copy()

    keep_mapping_columns = [
        column
        for column in [
            "pmt_subtopic_name",
            "official_reference",
        ]
        if column in mappings_df.columns
    ]

    mappings_df = mappings_df[
        keep_mapping_columns
    ].drop_duplicates()

    if (
        "subtopic" not in questions_df.columns
        or "pmt_subtopic_name" not in mappings_df.columns
    ):
        return pd.DataFrame()

    frame = questions_df.merge(
        mappings_df,
        left_on="subtopic",
        right_on="pmt_subtopic_name",
        how="inner",
    )

    if "record_type" in frame.columns:
        frame = frame[
            frame["record_type"]
            .fillna("")
            .astype(str)
            .eq("scored_item")
        ]

    if "review_status" in frame.columns:
        frame = frame[
            frame["review_status"]
            .fillna("")
            .astype(str)
            .isin(
                [
                    "auto_valid",
                    "human_approved",
                    "human_corrected",
                ]
            )
        ]

    frame = frame[
        frame["marks"].notna()
        & frame["question_text"].notna()
    ].copy()

    if "question_uid" not in frame.columns:
        frame["question_uid"] = ""

    if "question_number" not in frame.columns:
        frame["question_number"] = ""

    if "context_text" not in frame.columns:
        frame["context_text"] = ""

    if "has_code" not in frame.columns:
        frame["has_code"] = False

    if "has_visual" not in frame.columns:
        frame["has_visual"] = False

    if "paper_code" not in frame.columns:
        frame["paper_code"] = None

    if "programming_language" not in frame.columns:
        frame["programming_language"] = None

    return frame[
        [
            "question_uid",
            "official_reference",
            "question_number",
            "marks",
            "question_text",
            "context_text",
            "has_code",
            "has_visual",
            "paper_code",
            "programming_language",
        ]
    ].copy()


def load_style_reference_bank(
    required_references: list[str],
) -> pd.DataFrame:
    global _style_reference_bank_cache
    global _style_reference_bank_source

    if _style_reference_bank_cache is not None:
        return _style_reference_bank_cache

    required_references = sorted(
        {
            str(value or "").strip()
            for value in required_references
            if str(value or "").strip()
        }
    )

    if not required_references:
        _style_reference_bank_cache = pd.DataFrame()
        _style_reference_bank_source = "none_required"
        return _style_reference_bank_cache

    try:
        frame = _load_style_reference_bank_from_postgres(
            required_references
        )

        if not frame.empty:
            _style_reference_bank_cache = frame.copy()
            _style_reference_bank_source = "postgresql_question_bank"
            return _style_reference_bank_cache

    except Exception as exc:
        print(
            "Style calibration PostgreSQL lookup unavailable; "
            "trying local Agent 2 outputs instead:",
            str(exc)[:300],
        )

    try:
        frame = _load_style_reference_bank_from_outputs(
            required_references
        )

        if not frame.empty:
            _style_reference_bank_cache = frame.copy()
            _style_reference_bank_source = "local_agent2_question_bank_export"
            return _style_reference_bank_cache

    except Exception as exc:
        print(
            "Style calibration local fallback unavailable:",
            str(exc)[:300],
        )

    _style_reference_bank_cache = pd.DataFrame()
    _style_reference_bank_source = "unavailable"
    return _style_reference_bank_cache


def _compact_style_reference(
    row: pd.Series,
) -> str:
    marks_value = safe_int(
        row.get("marks")
    )

    raw_question_value = row.get(
        "question_text",
        "",
    )
    raw_context_value = row.get(
        "context_text",
        "",
    )

    question_value = (
        ""
        if pd.isna(raw_question_value)
        else str(raw_question_value).strip()
    )

    context_value = (
        ""
        if pd.isna(raw_context_value)
        else str(raw_context_value).strip()
    )

    question_value = question_value[
        :STYLE_REFERENCE_QUESTION_CHAR_LIMIT
    ].rstrip()

    context_value = context_value[
        :STYLE_REFERENCE_CONTEXT_CHAR_LIMIT
    ].rstrip()

    pieces = [
        f"{marks_value}m|task={_style_task_family(question_value, context_value)}"
    ]

    if context_value:
        pieces.append("C:" + context_value)

    pieces.append("Q:" + question_value)

    return "\n".join(pieces).strip()


def _style_topic_tokens(value: Any) -> set[str]:
    raw_tokens = re.findall(
        r"[a-z0-9]+",
        normalize_text(
            value
        ),
    )

    stop_words = {
        "and",
        "or",
        "the",
        "a",
        "an",
        "of",
        "to",
        "in",
        "on",
        "for",
        "with",
        "one",
        "two",
        "computer",
        "science",
    }

    stems: set[str] = set()

    for token in raw_tokens:
        if token in stop_words or len(token) < 3:
            continue

        stem = token

        for suffix in (
            "ational",
            "ation",
            "ments",
            "ment",
            "ingly",
            "ing",
            "ions",
            "ion",
            "ers",
            "er",
            "ed",
            "es",
            "s",
        ):
            if (
                stem.endswith(suffix)
                and len(stem) - len(suffix) >= 4
            ):
                stem = stem[:-len(suffix)]
                break

        stems.add(
            stem
        )

    return stems


def _style_topic_relevance(
    topic_name: str,
    question_text_value: Any,
    context_text_value: Any,
) -> float:
    topic_tokens = _style_topic_tokens(
        topic_name
    )

    if not topic_tokens:
        return 0.0

    candidate_tokens = _style_topic_tokens(
        (
            str(question_text_value or "")
            + " "
            + str(context_text_value or "")
        )
    )

    if not candidate_tokens:
        return 0.0

    overlap = topic_tokens & candidate_tokens

    return len(overlap) / len(topic_tokens)


def _style_task_family(
    question_text_value: Any,
    context_text_value: Any = "",
) -> str:
    """
    Deterministically classify the learner work demanded by an existing
    question.  This is deliberately coarse: it is used only to calibrate task
    shape/difficulty, never to infer syllabus content.
    """
    text = normalize_text(
        str(question_text_value or "")
        + " "
        + str(context_text_value or "")
    )

    if not text:
        return "other"

    # Error-finding/correction is checked first because many such questions
    # also contain generic verbs such as identify, state or write.
    if re.search(
        r"\b(?:logic\s+error|syntax\s+error|error|incorrect|wrong|debug|fix|correct\s+(?:the|this|line|statement|code|algorithm))\b",
        text,
    ):
        return "diagnose_or_correct"

    if re.search(
        r"\b(?:evaluate|justify|discuss|assess)\b",
        text,
    ):
        return "evaluate_or_justify"

    if re.search(
        r"\b(?:compare|comparison|difference|differences|similarity|similarities|contrast)\b",
        text,
    ):
        return "compare_or_contrast"

    if re.search(
        r"\b(?:shade\s+one\s+lozenge|select|choose|which\s+of\s+the\s+following|identify|name)\b",
        text,
    ):
        return "select_or_identify"

    if re.search(
        r"\b(?:complete|write\s+(?:one\s+)?line|write\s+(?:an?\s+)?algorithm|write\s+(?:the\s+)?code|construct|create|amend|adapt|modify|add\s+(?:a\s+)?line)\b",
        text,
    ):
        return "construct_or_complete"

    if re.search(
        r"\b(?:trace|final\s+value|final\s+values|final\s+output|what\s+is\s+the\s+output|what\s+will\s+be\s+printed|what\s+is\s+printed|displayed|how\s+many\s+times|value\s+of\s+[a-z_][a-z0-9_]*\s+after|values\s+in\s+the\s+(?:trace\s+)?table)\b",
        text,
    ):
        return "trace_or_predict"

    if re.search(
        r"\b(?:calculate|work\s+out|determine)\b",
        text,
    ):
        return "calculate_or_apply"

    if re.search(
        r"\b(?:explain|give\s+(?:one\s+)?reason|why|describe\s+how|describe\s+why)\b",
        text,
    ):
        return "explain_or_reason"

    if re.search(
        r"\b(?:state|describe|give)\b",
        text,
    ):
        return "state_or_describe"

    return "scenario_application"


def _preferred_style_task_families(
    assessment_pattern: str | None,
) -> tuple[str, ...]:
    mapping = {
        "apply_or_predict": (
            "trace_or_predict",
            "calculate_or_apply",
            "scenario_application",
        ),
        "explain_or_reason": (
            "explain_or_reason",
            "state_or_describe",
        ),
        "analyse_or_interpret": (
            "trace_or_predict",
            "explain_or_reason",
            "scenario_application",
        ),
        "compare_or_select": (
            "select_or_identify",
            "compare_or_contrast",
        ),
        "diagnose_or_correct": (
            "diagnose_or_correct",
        ),
        "construct_or_complete": (
            "construct_or_complete",
        ),
        "scenario_application": (
            "scenario_application",
            "calculate_or_apply",
            "select_or_identify",
        ),
        "predict_consequence": (
            "trace_or_predict",
            "explain_or_reason",
        ),
        "evaluate_or_justify": (
            "evaluate_or_justify",
            "explain_or_reason",
        ),
        "adapt_or_modify": (
            "construct_or_complete",
            "diagnose_or_correct",
        ),
        "classify_or_decide": (
            "select_or_identify",
            "state_or_describe",
        ),
        "multi_step_synthesis": (
            "construct_or_complete",
            "trace_or_predict",
            "explain_or_reason",
        ),
    }

    return mapping.get(
        normalize_text(assessment_pattern),
        (),
    )


def _style_command_demand(
    question_text_value: Any,
) -> int:
    """Small deterministic proxy for how many distinct commands are asked."""
    text = normalize_text(question_text_value)

    command_patterns = (
        r"\bstate\b",
        r"\bidentify\b",
        r"\bselect\b",
        r"\bchoose\b",
        r"\bname\b",
        r"\bdescribe\b",
        r"\bexplain\b",
        r"\bjustify\b",
        r"\bcompare\b",
        r"\bcalculate\b",
        r"\bdetermine\b",
        r"\bcorrect\b",
        r"\bcomplete\b",
        r"\bwrite\b",
        r"\bevaluate\b",
    )

    return sum(
        1
        for pattern in command_patterns
        if re.search(pattern, text)
    )


def codebase_style_calibration_for_slot(
    topic_name: str,
    official_reference: str,
    target_marks: int,
    paper_code: str | None,
    assessment_pattern: str | None = None,
) -> dict[str, Any]:
    bank = load_style_reference_bank(
        [official_reference]
    )

    if bank.empty:
        return {
            "target_task_family": "",
            "examples": [],
        }

    candidates = bank[
        bank["official_reference"]
        .fillna("")
        .astype(str)
        .eq(str(official_reference))
    ].copy()

    if candidates.empty:
        return {
            "target_task_family": "",
            "examples": [],
        }

    candidates["_marks"] = pd.to_numeric(
        candidates["marks"],
        errors="coerce",
    )

    candidates = candidates[
        candidates["_marks"].notna()
    ].copy()

    if candidates.empty:
        return {
            "target_task_family": "",
            "examples": [],
        }

    requested_paper = str(
        paper_code or ""
    ).strip()

    if requested_paper and "paper_code" in candidates.columns:
        exact_paper = candidates[
            candidates["paper_code"]
            .fillna("")
            .astype(str)
            .str.strip()
            .eq(requested_paper)
        ]

        if not exact_paper.empty:
            candidates = exact_paper.copy()

    candidates["_mark_distance"] = (
        candidates["_marks"]
        - int(target_marks)
    ).abs()

    candidates["_topic_relevance"] = candidates.apply(
        lambda row: _style_topic_relevance(
            topic_name=topic_name,
            question_text_value=row.get(
                "question_text",
                "",
            ),
            context_text_value=(
                ""
                if pd.isna(
                    row.get("context_text")
                )
                else row.get(
                    "context_text",
                    "",
                )
            ),
        ),
        axis=1,
    )

    candidates["_task_family"] = candidates.apply(
        lambda row: _style_task_family(
            row.get("question_text", ""),
            ""
            if pd.isna(row.get("context_text"))
            else row.get("context_text", ""),
        ),
        axis=1,
    )

    preferred_families = _preferred_style_task_families(
        assessment_pattern
    )

    preference_rank = {
        family: index
        for index, family in enumerate(
            preferred_families
        )
    }

    candidates["_task_family_match"] = candidates[
        "_task_family"
    ].map(
        lambda family: (
            max(
                0.0,
                1.0 - 0.18 * preference_rank[family],
            )
            if family in preference_rank
            else 0.0
        )
    )

    candidates["_command_demand"] = candidates[
        "question_text"
    ].map(
        _style_command_demand
    )

    expected_commands = (
        1
        if int(target_marks) <= 1
        else min(2, int(target_marks))
    )

    candidates["_demand_penalty"] = (
        candidates["_command_demand"]
        - expected_commands
    ).abs()

    # Visual rendering is a later feature. Prefer textual/code examples when
    # equally useful, without excluding a visual example as a fallback.
    candidates["_visual_penalty"] = candidates[
        "has_visual"
    ].map(
        lambda value: 1 if _style_bool(value) else 0
    )

    candidates["_stable_id"] = candidates.apply(
        lambda row: str(
            row.get("question_uid")
            or row.get("question_id")
            or row.get("question_number")
            or ""
        ),
        axis=1,
    )

    candidates["_calibration_score"] = (
        (3.4 * candidates["_task_family_match"])
        + (2.6 * candidates["_topic_relevance"])
        - (0.7 * candidates["_mark_distance"])
        - (0.20 * candidates["_demand_penalty"])
        - (0.15 * candidates["_visual_penalty"])
    )

    near_mark_candidates = candidates[
        candidates["_mark_distance"] <= 2
    ].copy()

    if near_mark_candidates.empty:
        near_mark_candidates = candidates.copy()

    # Pick a target task family only from families that actually exist in the
    # reviewed bank.  This keeps the target codebase-grounded rather than
    # inventing a synthetic task type from the assessment_pattern alone.
    preferred_present = near_mark_candidates[
        near_mark_candidates["_task_family_match"] > 0
    ].copy()

    family_source = (
        preferred_present
        if not preferred_present.empty
        else near_mark_candidates
    )

    family_order = family_source.sort_values(
        [
            "_calibration_score",
            "_task_family_match",
            "_topic_relevance",
            "_mark_distance",
            "_stable_id",
        ],
        ascending=[
            False,
            False,
            False,
            True,
            True,
        ],
        kind="stable",
    )

    target_task_family = str(
        family_order.iloc[0]["_task_family"]
        if not family_order.empty
        else ""
    ).strip()

    candidates["_target_family_match"] = candidates[
        "_task_family"
    ].eq(target_task_family)

    topical_order = near_mark_candidates.assign(
        _target_family_match=near_mark_candidates[
            "_task_family"
        ].eq(target_task_family)
    ).sort_values(
        [
            "_target_family_match",
            "_calibration_score",
            "_topic_relevance",
            "_mark_distance",
            "_visual_penalty",
            "_stable_id",
        ],
        ascending=[
            False,
            False,
            False,
            True,
            True,
            True,
        ],
        kind="stable",
    )

    fallback_order = candidates.sort_values(
        [
            "_target_family_match",
            "_mark_distance",
            "_topic_relevance",
            "_visual_penalty",
            "_stable_id",
        ],
        ascending=[
            False,
            True,
            False,
            True,
            True,
        ],
        kind="stable",
    )

    ordered_candidates = pd.concat(
        [
            topical_order,
            fallback_order,
        ],
        ignore_index=True,
    )

    if target_task_family:
        target_family_candidates = ordered_candidates[
            ordered_candidates["_task_family"].eq(
                target_task_family
            )
        ].copy()

        # Once a real bank family has been selected, do not dilute its task
        # signal with an unrelated second example merely to fill the quota.
        if not target_family_candidates.empty:
            ordered_candidates = target_family_candidates

    examples: list[str] = []
    seen_questions: set[str] = set()
    used_chars = 0

    for _, row in ordered_candidates.iterrows():
        raw_question = str(
            row.get("question_text", "")
            or ""
        ).strip()

        normalized_question = normalize_text(
            raw_question
        )

        if (
            not normalized_question
            or normalized_question in seen_questions
        ):
            continue

        example = _compact_style_reference(
            row
        )

        if not example:
            continue

        if (
            examples
            and used_chars + len(example)
            > STYLE_REFERENCE_TOTAL_CHAR_LIMIT
        ):
            break

        seen_questions.add(
            normalized_question
        )
        examples.append(
            example
        )
        used_chars += len(example)

        if len(examples) >= STYLE_REFERENCE_EXAMPLES_PER_SLOT:
            break

    return {
        "target_task_family": target_task_family,
        "examples": examples,
    }


def codebase_style_examples_for_slot(
    topic_name: str,
    official_reference: str,
    target_marks: int,
    paper_code: str | None,
    assessment_pattern: str | None = None,
) -> list[str]:
    """Backward-compatible wrapper returning only compact examples."""
    return codebase_style_calibration_for_slot(
        topic_name=topic_name,
        official_reference=official_reference,
        target_marks=target_marks,
        paper_code=paper_code,
        assessment_pattern=assessment_pattern,
    )["examples"]


def official_style_calibration_for_topic(
    topic_name: str,
    official_reference: str,
    target_marks: int,
    paper_code: str | None = None,
    assessment_pattern: str | None = None,
) -> dict[str, Any]:
    # Preserve fill_shortfall source behavior: selected Notebook 05 official
    # questions remain the calibration evidence.
    if QUIZ_MODE == "fill_shortfall":
        topic_norm = normalize_text(
            topic_name
        )

        examples: list[str] = []

        for question in official_questions:
            if normalize_text(
                question_topic(
                    question
                )
            ) != topic_norm:
                continue

            text = question_text(
                question
            )

            if text:
                examples.append(
                    text
                )

        examples = examples[:3]

        return {
            "target_task_family": (
                _style_task_family(examples[0])
                if examples
                else ""
            ),
            "examples": examples,
        }

    return codebase_style_calibration_for_slot(
        topic_name=topic_name,
        official_reference=str(
            official_reference or ""
        ).strip(),
        target_marks=int(
            target_marks
        ),
        paper_code=paper_code,
        assessment_pattern=assessment_pattern,
    )


def official_style_examples_for_topic(
    topic_name: str,
    official_reference: str = "",
    target_marks: int = 1,
    paper_code: str | None = None,
    assessment_pattern: str | None = None,
) -> list[str]:
    return official_style_calibration_for_topic(
        topic_name=topic_name,
        official_reference=official_reference,
        target_marks=target_marks,
        paper_code=paper_code,
        assessment_pattern=assessment_pattern,
    )["examples"]


topic_by_norm = {
    item[
        "topic_norm"
    ]:
        item
    for item in approved_topics
}


# ---------------------------------------------------------------
# Unique topic grounding bank.
# Evidence appears ONCE per approved topic rather than being repeated
# inside every blueprint question that references the topic.
# ---------------------------------------------------------------

topic_grounding = []

for topic in sorted(
    approved_topics,
    key=topic_priority,
):
    topic_grounding.append(
        {
            "topic":
                topic[
                    "topic"
                ],

            "topic_norm":
                topic[
                    "topic_norm"
                ],

            "role":
                topic[
                    "role"
                ],

            "official_reference":
                topic[
                    "official_reference"
                ],

            "lesson_evidence":
                topic_evidence(
                    topic
                ),
        }
    )


# Load all required references once so repeated slot lookups share one compact
# read-only bank query instead of caching only the first reference encountered.
if QUIZ_MODE == "complete_quiz":
    load_style_reference_bank(
        sorted(
            {
                str(
                    slot.get(
                        "official_reference",
                        "",
                    )
                    or ""
                ).strip()
                for slot in generation_blueprint
                if str(
                    slot.get(
                        "official_reference",
                        "",
                    )
                    or ""
                ).strip()
            }
        )
    )


# ================================================================
# v2.37 — KNOWLEDGE-BASE EXEMPLAR VISUAL METADATA
# ================================================================

_STYLE_VISUAL_TYPES = {
    "none",
    "code_block",
    "trace_table",
    "array_grid",
    "simple_flowchart",
    "logic_gate_diagram",
    "truth_table",
    "network_diagram",
    "database_table",
    "cpu_block_diagram",
    "memory_grid",
    "binary_register",
}


def _infer_exemplar_visual_type(
    *,
    question_text_value: Any,
    context_text_value: Any = "",
    topic_name: str = "",
    has_visual: Any = False,
    has_code: Any = False,
) -> str:
    """
    Infer only an explicit assessment visual form from a reviewed exemplar.

    This function deliberately ignores topic_name for routing. It looks only
    for explicit form cues present in the official/reviewed question/context.

    If a reviewed row says `has_visual=True` but its compact text does not
    identify the visual form safely, return an unresolved empty string rather
    than guessing from syllabus-topic keywords.
    """
    has_visual_flag = _style_bool(
        has_visual
    )
    has_code_flag = _style_bool(
        has_code
    )

    corpus = normalize_text(
        " ".join(
            [
                str(question_text_value or ""),
                str(context_text_value or ""),
            ]
        )
    )

    explicit_patterns = [
        (
            "simple_flowchart",
            [
                r"\bflowchart\b",
                r"\bflow chart\b",
            ],
        ),
        (
            "logic_gate_diagram",
            [
                r"\blogic gate diagram\b",
                r"\blogic circuit\b",
                r"\blogic gate\b",
            ],
        ),
        (
            "truth_table",
            [
                r"\btruth table\b",
            ],
        ),
        (
            "network_diagram",
            [
                r"\bnetwork diagram\b",
                r"\bnetwork topology diagram\b",
            ],
        ),
        (
            "database_table",
            [
                r"\bdatabase table\b",
                r"\btable of records\b",
            ],
        ),
        (
            "cpu_block_diagram",
            [
                r"\bcpu block diagram\b",
                r"\bprocessor block diagram\b",
            ],
        ),
        (
            "memory_grid",
            [
                r"\bmemory grid\b",
                r"\bmemory table\b",
            ],
        ),
        (
            "binary_register",
            [
                r"\bbinary register\b",
                r"\bbit register\b",
                r"\bregister diagram\b",
            ],
        ),
        (
            "trace_table",
            [
                r"\btrace table\b",
            ],
        ),
        (
            "array_grid",
            [
                r"\barray grid\b",
                r"\bgrid of values\b",
            ],
        ),
        (
            "code_block",
            [
                r"\bpseudocode shown\b",
                r"\bcode shown\b",
                r"\bprogram shown\b",
                r"\bcode listing\b",
            ],
        ),
    ]

    for visual_type, patterns in explicit_patterns:
        if any(
            re.search(
                pattern,
                corpus,
                flags=re.IGNORECASE,
            )
            for pattern in patterns
        ):
            return visual_type

    # Code is explicit metadata, so it is safe to use without topic routing.
    if has_code_flag:
        return "code_block"

    # Reviewed visual exists but exact form is not safely inferable.
    if has_visual_flag:
        return ""

    return "none"


def _kb_visual_metadata_from_rows(
    rows: Any,
    *,
    topic_name: str,
    source: str,
) -> dict[str, Any]:
    if rows is None:
        return {
            "visual_evidence_present": False,
            "preferred_visual_type": "",
            "visual_example_count": 0,
            "source": source,
            "evidence_ids": [],
        }

    inferred: list[tuple[str, str]] = []

    try:
        iterator = rows.iterrows()
    except Exception:
        iterator = []

    for _, row in iterator:
        if not _style_bool(row.get("has_visual", False)):
            continue

        visual_type = _infer_exemplar_visual_type(
            question_text_value=row.get("question_text", ""),
            context_text_value=(
                ""
                if pd.isna(row.get("context_text"))
                else row.get("context_text", "")
            ),
            topic_name=topic_name,
            has_visual=row.get("has_visual", False),
            has_code=row.get("has_code", False),
        )

        evidence_id = str(
            row.get("question_uid")
            or row.get("question_id")
            or row.get("question_number")
            or ""
        ).strip()

        inferred.append((visual_type, evidence_id))

    specific = [
        visual_type
        for visual_type, _ in inferred
        if visual_type in _STYLE_VISUAL_TYPES
        and visual_type != "none"
        and visual_type
    ]

    preferred = ""
    if specific:
        counts: dict[str, int] = {}
        for visual_type in specific:
            counts[visual_type] = counts.get(visual_type, 0) + 1
        preferred = sorted(
            counts,
            key=lambda visual_type: (
                -counts[visual_type],
                visual_type,
            ),
        )[0]

    return {
        "visual_evidence_present": bool(inferred),
        "preferred_visual_type": preferred,
        "visual_example_count": len(inferred),
        "source": source,
        "evidence_ids": [
            evidence_id
            for _, evidence_id in inferred
            if evidence_id
        ][:5],
    }


# Keep the existing task-shape selector intact, but augment its output with
# explicit visual metadata from the same reviewed KB/reference bank.
_codebase_style_calibration_v236 = codebase_style_calibration_for_slot


def codebase_style_calibration_for_slot(
    topic_name: str,
    official_reference: str,
    target_marks: int,
    paper_code: str | None,
    assessment_pattern: str | None = None,
) -> dict[str, Any]:
    result = _codebase_style_calibration_v236(
        topic_name=topic_name,
        official_reference=official_reference,
        target_marks=target_marks,
        paper_code=paper_code,
        assessment_pattern=assessment_pattern,
    )

    bank = load_style_reference_bank(
        [official_reference]
    )

    if bank.empty:
        result["kb_exemplar_visual"] = {
            "visual_evidence_present": False,
            "preferred_visual_type": "",
            "visual_example_count": 0,
            "source": "knowledge_base_unavailable",
            "evidence_ids": [],
        }
        return result

    candidates = bank[
        bank["official_reference"]
        .fillna("")
        .astype(str)
        .eq(str(official_reference))
    ].copy()

    if candidates.empty:
        result["kb_exemplar_visual"] = {
            "visual_evidence_present": False,
            "preferred_visual_type": "",
            "visual_example_count": 0,
            "source": "knowledge_base_no_reference_match",
            "evidence_ids": [],
        }
        return result

    requested_paper = str(
        paper_code or ""
    ).strip()

    if requested_paper and "paper_code" in candidates.columns:
        exact_paper = candidates[
            candidates["paper_code"]
            .fillna("")
            .astype(str)
            .str.strip()
            .eq(requested_paper)
        ]
        if not exact_paper.empty:
            candidates = exact_paper.copy()

    candidates["_marks"] = pd.to_numeric(
        candidates["marks"],
        errors="coerce",
    )

    candidates["_mark_distance"] = (
        candidates["_marks"]
        - int(target_marks)
    ).abs()

    candidates["_topic_relevance"] = candidates.apply(
        lambda row: _style_topic_relevance(
            topic_name=topic_name,
            question_text_value=row.get("question_text", ""),
            context_text_value=(
                ""
                if pd.isna(row.get("context_text"))
                else row.get("context_text", "")
            ),
        ),
        axis=1,
    )

    target_family = str(
        result.get(
            "target_task_family",
            "",
        )
        or ""
    ).strip()

    candidates["_task_family"] = candidates.apply(
        lambda row: _style_task_family(
            row.get("question_text", ""),
            ""
            if pd.isna(row.get("context_text"))
            else row.get("context_text", ""),
        ),
        axis=1,
    )

    candidates["_family_match"] = (
        candidates["_task_family"].eq(target_family)
        if target_family
        else True
    )

    # Visual metadata is a style signal, not an invitation to broaden scope.
    # Prefer close-mark, same-task-family, topic-relevant reviewed rows.
    candidates = candidates.sort_values(
        [
            "_family_match",
            "_topic_relevance",
            "_mark_distance",
        ],
        ascending=[
            False,
            False,
            True,
        ],
        kind="stable",
    )

    result["kb_exemplar_visual"] = _kb_visual_metadata_from_rows(
        candidates.head(8),
        topic_name=topic_name,
        source=_style_reference_bank_source or "knowledge_base",
    )

    return result


def _official_question_has_visual(question: dict[str, Any]) -> bool:
    explicit = question.get("has_visual")
    if explicit is not None:
        return _style_bool(explicit)

    visual_requirement = str(
        question.get("visual_requirement", "")
        or ""
    ).strip().casefold()
    if visual_requirement and visual_requirement != "none":
        return True

    visual = question.get("visual")
    if isinstance(visual, dict):
        visual_type = str(
            visual.get("type", "")
            or ""
        ).strip().casefold()
        if visual_type and visual_type != "none":
            return True

    text_value = question_text(question)
    context_value = str(
        question.get("context_text", "")
        or ""
    )
    signal = normalize_text(
        f"{text_value} {context_value}"
    )
    return bool(
        re.search(
            r"\b(diagram|flowchart|flow chart|grid|truth table|"
            r"trace table|network topology|shown below|figure)\b",
            signal,
        )
    )


# Preserve fill_shortfall behavior (Notebook 05 selected official questions),
# but carry visual exemplar metadata alongside task-family calibration.
_official_style_calibration_v236 = official_style_calibration_for_topic


def official_style_calibration_for_topic(
    topic_name: str,
    official_reference: str,
    target_marks: int,
    paper_code: str | None = None,
    assessment_pattern: str | None = None,
) -> dict[str, Any]:
    if QUIZ_MODE != "fill_shortfall":
        return codebase_style_calibration_for_slot(
            topic_name=topic_name,
            official_reference=official_reference,
            target_marks=target_marks,
            paper_code=paper_code,
            assessment_pattern=assessment_pattern,
        )

    topic_norm = normalize_text(
        topic_name
    )

    matched = [
        question
        for question in official_questions
        if normalize_text(
            question_topic(question)
        )
        == topic_norm
    ]

    examples = [
        question_text(question)
        for question in matched
        if question_text(question)
    ][:3]

    visual_types: list[str] = []
    evidence_ids: list[str] = []

    for question in matched:
        has_visual = _official_question_has_visual(
            question
        )

        visual_type = _infer_exemplar_visual_type(
            question_text_value=question_text(question),
            context_text_value=question.get("context_text", ""),
            topic_name=topic_name,
            has_visual=has_visual,
            has_code=question.get("has_code", False),
        )

        if has_visual:
            if visual_type:
                visual_types.append(visual_type)
            evidence_ids.append(
                str(
                    question.get("question_uid")
                    or question.get("question_id")
                    or question.get("question_number")
                    or ""
                ).strip()
            )

    preferred = ""
    if visual_types:
        counts: dict[str, int] = {}
        for value in visual_types:
            counts[value] = counts.get(value, 0) + 1
        preferred = sorted(
            counts,
            key=lambda value: (-counts[value], value),
        )[0]

    return {
        "target_task_family": (
            _style_task_family(examples[0])
            if examples
            else ""
        ),
        "examples": examples,
        "kb_exemplar_visual": {
            "visual_evidence_present": bool(evidence_ids),
            "preferred_visual_type": preferred,
            "visual_example_count": len(evidence_ids),
            "source": "notebook05_selected_official_questions",
            "evidence_ids": [
                value
                for value in evidence_ids
                if value
            ][:5],
        },
    }

generation_topics = []

for slot in generation_blueprint:

    covered_topic_details = []

    for covered in slot.get(
        "covered_topics",
        [],
    ):
        covered_norm = normalize_text(
            covered.get(
                "topic_norm",
                covered.get(
                    "topic",
                    "",
                ),
            )
        )

        source_topic = topic_by_norm.get(
            covered_norm
        )

        if source_topic is None:
            continue

        # No evidence here. It lives once in topic_grounding.
        covered_topic_details.append(
            {
                "topic":
                    source_topic[
                        "topic"
                    ],

                "role":
                    source_topic[
                        "role"
                    ],

                "official_reference":
                    source_topic[
                        "official_reference"
                    ],
            }
        )

    style_calibration = official_style_calibration_for_topic(
        topic_name=slot[
            "topic"
        ],
        official_reference=slot[
            "official_reference"
        ],
        target_marks=slot[
            "marks"
        ],
        paper_code=slot.get(
            "paper_code"
        ),
        assessment_pattern=slot.get(
            "assessment_pattern"
        ),
    )

    anchor_topic_norm = normalize_text(
        slot.get(
            "topic_norm",
            slot.get(
                "topic",
                "",
            ),
        )
    )

    visual_requirement = visual_requirement_for_slot(
        slot=slot,
        style_calibration=style_calibration,
        assessment_filters=request,
        source_topic=topic_by_norm.get(
            anchor_topic_norm
        ),
    )

    # Store the deterministic decision on the raw blueprint as well so the
    # final manifest shows the exact visual plan used for generation.
    slot[
        "visual_requirement"
    ] = visual_requirement

    slot[
        "kb_exemplar_visual"
    ] = style_calibration.get(
        "kb_exemplar_visual",
        {},
    )

    slot[
        "visual_requirement_source"
    ] = (
        "kb_exemplar_visual"
        if str(
            (
                style_calibration.get(
                    "kb_exemplar_visual",
                    {},
                )
                or {}
            ).get(
                "preferred_visual_type",
                "",
            )
            or ""
        ).strip()
        == visual_requirement
        and visual_requirement != "none"
        else "no_resolved_kb_visual_evidence"
    )

    generation_topics.append(
        {
            "plan_index":
                slot[
                    "plan_index"
                ],

            # Anchor topic determines this question's role count.
            "topic":
                slot[
                    "topic"
                ],

            "role":
                slot[
                    "role"
                ],

            "official_reference":
                slot[
                    "official_reference"
                ],

            "paper_code":
                slot.get(
                    "paper_code"
                ),

            "paper_label":
                slot.get(
                    "paper_label"
                ),

            "assessment_pattern":
                slot.get(
                    "assessment_pattern"
                ),

            "visual_requirement":
                visual_requirement,

            "covered_topics":
                covered_topic_details,

            "marks":
                slot[
                    "marks"
                ],

            "target_task_family":
                style_calibration.get(
                    "target_task_family",
                    "",
                ),

            "official_question_style_examples":
                style_calibration.get(
                    "examples",
                    [],
                ),

            "kb_exemplar_visual":
                style_calibration.get(
                    "kb_exemplar_visual",
                    {},
                ),

            "visual_requirement_source":
                (
                    "kb_exemplar_visual"
                    if str(
                        (
                            style_calibration.get(
                                "kb_exemplar_visual",
                                {},
                            )
                            or {}
                        ).get(
                            "preferred_visual_type",
                            "",
                        )
                        or ""
                    ).strip()
                    == visual_requirement
                    and visual_requirement != "none"
                    else "no_resolved_kb_visual_evidence"
                ),
        }
    )


generation_request = {
    "schema_version":
        "agent2-quiz-generation-request-v2.9.0",

    "pipeline_version":
        NOTEBOOK06_PIPELINE_VERSION,

    "generated_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "specification":
        "AQA GCSE Computer Science 8525",

    "quiz_mode":
        QUIZ_MODE,

    "generation_scope":
        (
            "entire_quiz"
            if QUIZ_MODE
            == "complete_quiz"
            else "missing_coverage_only"
        ),

    "notebook05_used":
        bool(
            QUIZ_MODE
            == "fill_shortfall"
        ),

    "notebook05_run_timestamp":
        notebook05_run_timestamp,

    "assessment_filters":
        request,

    "special_instruction_policy":
        {
            "strategy":
                "same_generation_call_interpret_and_generate",
            "separate_instruction_llm_call":
                False,
            "raw_instruction":
                str(
                    request.get(
                        "special_instructions",
                        "",
                    )
                    or ""
                ).strip(),
            "scope":
                (
                    "entire_generated_quiz"
                    if QUIZ_MODE == "complete_quiz"
                    else "ai_generated_shortfall_only"
                ),
            "deterministic_preflight_hints":
                request.get(
                    "special_instruction_directives",
                    {},
                ),
            "policy":
                (
                    "follow_fully_when_compatible_with_aqa_scope_and_hard_controls"
                    if str(
                        request.get(
                            "special_instructions",
                            "",
                        )
                        or ""
                    ).strip()
                    else "none"
                ),
        },

    "target_generated_question_count":
        len(
            generation_blueprint
        ),

    "target_generated_marks":
        sum(
            item[
                "marks"
            ]
            for item in generation_blueprint
        ),

    "blueprint":
        generation_topics,

    # Grounding is de-duplicated here.
    "topic_grounding":
        topic_grounding,

    "official_question_count":
        len(
            official_questions
        ),

    "official_marks":
        sum(
            question_marks(
                question
            )
            for question in official_questions
        ),

    "shortfall_summary":
        shortfall_summary,
}


if QUIZ_MODE == "complete_quiz":
    style_example_count = sum(
        len(
            item.get(
                "official_question_style_examples",
                [],
            )
            or []
        )
        for item in generation_topics
    )

    print(
        "Codebase style calibration source:",
        _style_reference_bank_source,
    )
    print(
        "Codebase style calibration examples attached:",
        style_example_count,
    )
    print(
        "Codebase task-family calibration:",
        [
            {
                "plan_index": item.get("plan_index"),
                "task_family": item.get("target_task_family"),
            }
            for item in generation_topics
        ],
    )
    print(
        "Final visual architecture plan:",
        [
            {
                "plan_index": item.get("plan_index"),
                "visual_requirement": item.get("visual_requirement", "none"),
            }
            for item in generation_topics
        ],
    )



def request_identity(
    payload: dict[str, Any],
) -> dict[str, Any]:
    return {
        key:
            value
        for key, value in payload.items()
        if key
        not in {
            "generated_at_utc",
        }
    }


generation_request_fingerprint = (
    stable_json_fingerprint(
        request_identity(
            generation_request
        )
    )
)


generation_request[
    "request_fingerprint"
] = generation_request_fingerprint


generation_request_path = (
    OUTPUT_DIR
    / "generation_request.json"
)

generation_request_path.write_text(
    json.dumps(
        generation_request,
        indent=2,
        ensure_ascii=False,
        default=str,
    ),
    encoding="utf-8",
)


print(
    "Generation request fingerprint:",
    generation_request_fingerprint,
)

print(
    "Unique grounded topics:",
    len(
        topic_grounding
    ),
)

print(
    "Saved:",
    generation_request_path,
)



## Gemini structured-output truncation safeguard

During the first Gemini 3.5 Flash end-to-end run, the model call succeeded but
the returned JSON ended part-way through a string, producing:

```text
JSONDecodeError: Unterminated string
```

This was a **response-completion / transport issue**, not a failure of the
Agent 1 handoff, quiz blueprint, marks allocation, topic coverage, or
deterministic validation.

The Gemini transport now:
- uses Gemini JSON Schema structured output rather than MIME type alone;
- provides dynamic output headroom for both model thinking and visible JSON;
- checks the Gemini finish reason before attempting JSON parsing;
- treats `MAX_TOKENS` as an explicit truncated-response condition;
- distinguishes malformed completed JSON from true token truncation;
- conservatively repairs only safe trailing-comma syntax defects;
- gives a malformed single-question response one bounded strict MIME-only retry;
- dynamically splits only the affected generation batch and retries it;
- gives a single-question batch one larger retry allowance before failing;
- applies schema-constrained JSON output to semantic review as well.

The actual quiz constraints remain unchanged and continue to come from the
frontend request and deterministic Notebook 06 blueprint.


## 8. Compact, normally single-call config-selected generation

Notebook 06 sends one compact grounded quiz request to the model selected in
`quiz_model_config.json`. Lesson evidence is de-duplicated into a single
`topic_grounding` bank instead of being repeated inside every question
blueprint.

For a normal 10-question quiz, the default transport cap is now 12 questions so
the target path is **one generation call**. If the selected provider/context
budget cannot safely carry the request, the existing local token preflight can
split only as a technical transport safeguard. This is not an extra
instruction-understanding or review stage.

The same generation call performs both:
- natural-language interpretation of `special_instructions`; and
- quiz + marking-guidance + visual-spec generation.

The response also contains a compact structured instruction interpretation and
compliance report. No separate LLM interpreter and no secondary LLM reviewer
are used.

The deterministic blueprint, exact marks/question-count logic, topic coverage,
structural validation, local MiniLM grounding/diversity checks, mandatory HITL,
targeted corrective regeneration, final acceptance rules, and Streamlit
contract remain authoritative.


### Prompt transport compression

The LLM boundary stays small without weakening quiz rules:
- system/output contracts are compact;
- lesson evidence is relevance-ranked and de-duplicated;
- reviewed style examples are bounded and used only for task-shape calibration;
- visual contracts and corrective feedback are sent only when needed;
- raw special instructions are sent once with the generation request, rather
  than through a separate interpreter API call.


## Gemini 400 `INVALID_ARGUMENT` structured-output fix

After adding Paper 1/2 metadata, deterministic assessment patterns, and richer
quiz validation, the Gemini request returned:

```text
400 INVALID_ARGUMENT
Request contains an invalid argument.
```

The failure occurred inside the Google GenAI API call, before quiz generation.
Google documents that structured-output schemas support only a subset of JSON
Schema and that very large or deeply nested schemas may be rejected by the
API.

The fix keeps all existing Agent 2 logic but separates responsibilities:

- **Gemini provider schema:** deliberately shallow; guarantees only the outer
  JSON structure (`questions`, and semantic-review outer fields).
- **Notebook 06 deterministic validation:** remains the authority for exact
  question count, marks, primary/supporting roles, AQA references, Paper 1/2,
  assessment patterns, covered topics, marking guidance, user filters, and
  release conditions.
- **Fallback:** if Gemini still rejects the minimal response schema with HTTP
  400, the same call is retried once with `application/json` MIME mode only.
  The deterministic validators then perform the same strict checks.

No quiz constraint, topic allocation, mark allocation, paper routing, special
instruction handling, HITL rule, or release rule is relaxed by this change.


In [ ]:
SYSTEM_PROMPT = """
Generate AI practice questions for AQA GCSE Computer Science 8525. Return JSON only.

HARD RULES:
1. The deterministic blueprint is authoritative: generate exactly one question per item and preserve plan_index, topic, role, official_reference, paper_code, paper_label, assessment_pattern, visual_requirement, covered_topics and marks exactly.
2. Assess only the anchor topic plus covered_topics. Integrate multiple covered topics coherently; never infer extra syllabus content from style examples. Priority order is: anchor topic + approved covered_topics first, target_task_family second, diversity third. Never leave the approved topic scope merely to create a more varied-looking question.
3. Style grounding calibrates only task shape, difficulty, scaffolding and code density. target_task_family guides learner work only. Never copy source wording, scenario, values, names, identifiers, code/layout or answer pattern.
4. Use concise GCSE/AQA-like commands. For AQA GCSE Computer Science 8525 algorithm efficiency, keep assessment within the specification's qualitative time-efficiency scope: do not require Big-O/asymptotic notation, formal time-complexity notation, or space-complexity analysis. Comparisons, passes, swaps, operations and relative time efficiency may be assessed when self-contained and relevant. A 1-mark item normally asks for one independent learner demand. Every question must be self-contained: explicitly initialise every state variable needed to derive an answer and explicitly state every operation/state transition needed to determine the requested result. Never rely on unstated assumptions such as a loop counter automatically incrementing. Marking guidance must contain independent explicit points whose marks sum exactly to the question marks. If the question explicitly requests N reasons, advantages, disadvantages, examples, values, answers, factors, features, steps, causes, effects, methods, ways, points, benefits, drawbacks, differences or similarities, the marking guidance must contain exactly N separately markable criteria of that requested type; never ask for two and award four. For genuinely open-ended tasks with multiple valid responses (for example suitable test data, examples, scenarios or methods), marking guidance must describe what makes a response acceptable and allow equivalent valid answers; do not present arbitrary sample responses as the only correct answers. For numbered-code edits, distinguish precisely between changing/replacing an existing line and inserting a new line: if the required correction is an insertion after line N, ask where the line should be inserted (for example, "after which line?") rather than asking which existing line should be changed. For objective/numeric/binary/index/final-value/code-correction points, include the exact expected answer/value/code in the criterion (for example "178", "row index = 17", or "index <- index + 1"), never only generic wording such as "correct value" or "correct answer", and never use "e.g."/"etc." where one exact answer is required. Do not invent unsupported runtime/implementation jargon beyond lesson grounding.
5. Respect assessment_filters. SPECIAL INSTRUCTIONS ARE INTERPRETED IN THIS SAME GENERATION CALL: there is no separate instruction-interpreter LLM request. Read the complete raw special instruction, preserve its actual modality, and follow every mandatory requirement fully whenever compatible with approved AQA scope and deterministic hard controls. deterministic_preflight_hints are only conservative machine-checkable hints and are not the full meaning of the instruction. Return instruction_interpretation and special_instruction_compliance in the same JSON response. Never silently ignore, weaken, or invent a special requirement. If code is disabled, require no code task; if a language is set, use it. Multi-statement code/pseudocode must remain genuinely multiline and fenced.
6. Preserve visual_requirement exactly in the generation response. none => requires_visual=false and visual={"type":"none","spec":{}}. Otherwise requires_visual=true and use only the matching supplied visual contract. A non-none visual must be functionally relevant to that specific question: question_text must explicitly refer to/use the visual, and the learner should need or materially benefit from the visual to answer the task. The learner-facing description of the supplied visual must be semantically consistent with the blueprint visual_requirement: never ask the learner to inspect, complete, label, or otherwise use a different visual object from the one the deterministic blueprint supplies. Never add a decorative or merely topic-generic visual. Student visuals must contain no answers, solutions, marking data or external URLs; use blanks for learner-completed cells/bits/outputs. For code_block, keep code in visual.spec.code rather than duplicating it in question_text. Visuals must be new, compact and black-and-white readable. A deterministic post-generation relevance gate may safely downgrade an irrelevant planned visual to none before rendering.
7. Treat the batch as one quiz. Diversity is a third-priority preference after topic fidelity and target_task_family. For repeated anchor topics, make the learner action genuinely different when that can be done naturally within scope; do not force novelty by introducing unrelated concepts. Avoid near-duplicate/superficial variants, but prefer a legitimate repeated assessment form over an off-topic question.
8. Follow the output contract exactly, including instruction_interpretation and special_instruction_compliance even when there are no special instructions (use status="none", empty requirements and an empty compliance list). Material must remain clearly AI-generated practice, never an official AQA question. Do not leak source-paper response-layout instructions such as "shade one lozenge", "tick one box", "cross one box" or "write in the box below" unless that exact response control is actually represented in the generated student material; otherwise use a normal command such as state, select or write.
9. TARGETED REGENERATION IS A MINIMAL-REPAIR OPERATION. When REQUEST contains targeted_regeneration_context, use the matching original question as the baseline. Preserve its approved topic/reference/role/marks, learner-task intent, scenario/concept, task family, and every part of the wording that remains valid. CURRENT HUMAN FEEDBACK and CURRENT DETERMINISTIC VALIDATION FEEDBACK are authoritative over invalid baseline content. Change only the component required to satisfy the active feedback and hard rules. If feedback concerns only marking guidance, acceptance of equivalent answers, mark allocation, metadata/pattern, or a visual property that does not contradict learner-facing wording, keep question_text unchanged. If deterministic validation identifies a contradiction between question_text and an immutable blueprint field, preserve the blueprint and minimally rewrite only the conflicting learner-facing wording. Do not evade a correction by replacing the task with an unrelated/easier question. Rewrite the learner task more broadly only when the current human explicitly asks for a new/different question or when a narrower repair cannot satisfy the hard rules. Past memory is secondary guidance and must never override current feedback, the approved AQA scope, or the deterministic blueprint.
11. VISUAL COMPATIBILITY AND RELEVANCE: A visual should only be used when it genuinely helps assess the approved topic in an official-question style. Do not force a CPU diagram, network diagram, flowchart, or other visual if it is not a natural fit for the topic and learner task. If you use a visual, make it necessary: the learner should need the visual to answer correctly. Do not restate all of the visual's decisive information in the question text. Avoid generating two questions that are effectively the same learner task even if their pattern labels differ.
13. GENERIC VISUAL POLICY: Do not infer a visual type merely from the syllabus-topic name. Use the visual_requirement already planned from reviewed KB/official exemplar evidence. If visual_requirement='none', do not refer to a diagram/visual/table/grid/code shown elsewhere. A pseudocode/program listing that is written directly inside question_text is self-contained textual question content, not an external visual dependency, so visual_requirement='none' is valid for that case. When code is embedded directly in question_text, refer to it as the following/preceding pseudocode rather than implying that a separate visual asset exists. If a visual is required, make the learner explicitly use or inspect it. Keep all visual.spec enum/type fields valid for the supplied VISUAL_CONTRACT; do not use display placeholders such as UNLABELLED/HIDDEN/UNKNOWN as semantic object types. Do not generate a decorative visual that the learner can ignore.
14. GENERIC ANSWER VERIFICATION: For each question return answer_verification. Use mode='machine' or 'mixed' only when one or more marking criteria can be derived from explicit data already present in this generated question/visual. Each machine check must use a pure Python expression and its 1-based criterion_index. The expression may use literals plus the safe context variables visual_spec, question_text and code, and simple pure built-ins such as len/sum/min/max/sorted/all/any/abs/round/range/int/float/str/bool. Never use the marking guidance itself inside the expression and never use a constant-only expression as 'verification'. For conceptual/explanatory answers or answers that cannot be safely machine-derived, use mode='manual' and leave checks empty. Do not invent a machine check just to avoid HITL. Question wording, visual/code references, and marking-guidance answer targets must agree exactly on referenced line numbers and named student-completion objects.
15. MARKING-SCHEME QUALITY: Treat the generated question/visual evidence as the source of truth. Every objective marking criterion must contain the exact expected value/code/line/selection. The marking scheme must answer exactly what the learner-facing question asks. Do not change the learner-facing question merely to rescue an incorrect marking answer; a valid question with a wrong marking answer should have its marking scheme corrected.
16. GENERIC DIVERSITY: Return diversity_signature for each generated question. Within the same official reference, vary task intent, scenario/context and answer form where the blueprint permits. Do not create two questions that test essentially the same reasoning merely by changing numbers, names, wording or pattern labels. Diversity must remain inside approved topic, marks, paper, difficulty and official-style constraints.
10. VISUAL QUALITY: If the blueprint contains kb_exemplar_visual metadata, treat it as reviewed assessment-form evidence from the knowledge base. Use it only for visual/task structure and difficulty; never copy source content. When visual_requirement is non-none, the question must genuinely depend on that visual. If the learner is asked to name/identify a gate, device, component, symbol or other object, do not print that answer as a visible label/caption in the visual. Keep question wording, visual labels, visual node types and marking guidance internally consistent. Do not reuse the exact same visual values/structure in multiple independent generated questions; vary the visual spec unless the questions intentionally share one source visual. Never refer to a diagram/grid/table/visual that is not present in the returned visual contract.
17. ANSWER LEAKAGE: question_text must never contain or reveal an answer expected in marking_guidance, including through examples, hints, required forms, code, tables or visual annotations. Permitted notation/operators may be specified, but never the completed solution. Before returning each question, compare question_text with marking_guidance and rewrite the learner-facing wording if it gives away any expected answer.
18. TRUTH-TABLE RESPONSE SCAFFOLD: Whenever the learner is asked to complete, fill in, or finish a truth table, do not write the truth-table rows as prose and do not provide completed output values in question_text. State the input/output variable names clearly enough for the deterministic renderer to build the response area. The application will attach a separate blank learner response_scaffold when the supplied stimulus is another visual such as a logic circuit. A fully supplied truth_table visual may itself be the response scaffold, but every learner-completed output cell must remain blank.
19. FORBIDDEN UNSUPPORTED LOGIC TASKS: Never generate a question that asks the learner to determine, name, choose, select, place, or fill in logic-gate TYPES at labelled/placeholder circuit positions such as L1, L2, L3, L4, G1, G2, or equivalent labelled gate slots. Never ask the learner to choose between multiple separate labelled circuits (for example X, Y and Z), because the current logic visual contract represents one output-connected circuit per question. Never create a partially specified circuit whose missing answers are the gate types. This prohibition applies even when a reviewed style exemplar uses that format. Fully specified single logic circuits are still allowed for tasks such as completing a truth table, deriving/interpreting a Boolean expression, or determining outputs.
""".strip()


def compact_feedback(
    validation_feedback: list[str] | None,
) -> list[str]:
    if not validation_feedback:
        return []

    compact = []
    seen = set()

    for value in validation_feedback:
        text = str(
            value
            or ""
        ).strip()

        if not text:
            continue

        normalized = normalize_text(
            text
        )

        if normalized in seen:
            continue

        seen.add(
            normalized
        )

        compact.append(
            text[
                :350
            ]
        )

        if len(
            compact
        ) >= 8:
            break

    return compact


def estimate_text_tokens(
    value: Any,
) -> int:
    text = str(
        value
        or ""
    )

    return max(
        1,
        math.ceil(
            len(
                text
            )
            / LLM_ESTIMATED_CHARS_PER_TOKEN
        ),
    )


def estimate_messages_tokens(
    messages: list[
        dict[str, Any]
    ],
) -> int:
    # Conservative small per-message overhead.
    total = 12

    for message in messages:
        total += 8

        total += estimate_text_tokens(
            message.get(
                "content",
                "",
            )
        )

    return total


def topic_norms_for_blueprint_item(
    item: dict[str, Any],
) -> set[str]:
    values = {
        normalize_text(
            item.get(
                "topic",
                "",
            )
        )
    }

    for covered in safe_list(
        item.get(
            "covered_topics",
            [],
        )
    ):
        if not isinstance(
            covered,
            dict,
        ):
            continue

        values.add(
            normalize_text(
                covered.get(
                    "topic",
                    "",
                )
            )
        )

    return {
        value
        for value in values
        if value
    }


def select_grounding_for_blueprint(
    request_payload: dict[str, Any],
    blueprint_items: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    required_norms: set[str] = set()

    for item in blueprint_items:
        required_norms.update(topic_norms_for_blueprint_item(item))

    selected: list[dict[str, Any]] = []

    for grounding in request_payload.get("topic_grounding", []):
        if not isinstance(grounding, dict):
            continue

        grounding_norm = normalize_text(
            grounding.get("topic_norm", grounding.get("topic", ""))
        )

        if grounding_norm not in required_norms:
            continue

        # Role/reference metadata already lives in the blueprint. Send only the
        # topic label and evidence needed to ground content scope.
        selected.append(
            {
                "topic": grounding.get("topic", ""),
                "lesson_evidence": safe_list(grounding.get("lesson_evidence", [])),
            }
        )

    return selected


def compact_assessment_filters(
    filters: Any,
) -> dict[str, Any]:
    if not isinstance(filters, dict):
        return {}

    # Counts/marks/roles/paper routing are already encoded exactly in the
    # blueprint and deterministic validators. Only controls that still affect
    # wording/content generation need to cross the LLM boundary.
    allowed = {
        "include_code_questions",
        "include_visual_questions",
        "programming_language",
        "special_instructions",
        "special_instruction_directives",
    }

    compact: dict[str, Any] = {}
    for key, value in filters.items():
        if key not in allowed or value is None or value == "":
            continue

        if key == "special_instructions":
            value = re.sub(r"\s+", " ", str(value)).strip()
            # validate_request already enforces a 6000-character transport ceiling.
            # Do not silently truncate mandatory user instructions here.
            value = value[:6000]

        compact[key] = value

    return compact


def compact_blueprint_for_transport(
    blueprint_items: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """
    Remove the repeated style-example copies from each blueprint item before
    sending the batch to the selected model. The same examples are transported once in
    official_style_grounding instead, keeping token use small.
    """
    compact_items = []

    for item in blueprint_items:
        compact_item = {
            key: value
            for key, value in item.items()
            if key != "official_question_style_examples"
        }
        compact_items.append(
            compact_item
        )

    return compact_items


def select_style_grounding_for_blueprint(
    blueprint_items: list[dict[str, Any]],
) -> dict[str, Any]:
    """De-duplicate style examples and transport slot-level visual metadata."""
    example_bank: list[dict[str, Any]] = []
    example_id_by_norm: dict[str, str] = {}
    assignments: list[dict[str, Any]] = []

    for item in blueprint_items:
        example_ids: list[str] = []

        for example in safe_list(
            item.get(
                "official_question_style_examples",
                [],
            )
        ):
            text = str(
                example or ""
            ).strip()

            normalized = normalize_text(
                text
            )

            if not normalized:
                continue

            example_id = example_id_by_norm.get(
                normalized
            )

            if example_id is None:
                example_id = (
                    f"STYLE_{len(example_bank) + 1:03d}"
                )
                example_id_by_norm[
                    normalized
                ] = example_id
                example_bank.append(
                    {
                        "id": example_id,
                        "text": text,
                    }
                )

            if example_id not in example_ids:
                example_ids.append(
                    example_id
                )

            if len(
                example_ids
            ) >= 2:
                break

        kb_visual = item.get(
            "kb_exemplar_visual",
            {},
        )

        if not isinstance(
            kb_visual,
            dict,
        ):
            kb_visual = {}

        if (
            example_ids
            or kb_visual.get(
                "visual_evidence_present",
                False,
            )
        ):
            assignments.append(
                {
                    "plan_index": safe_int(
                        item.get(
                            "plan_index"
                        )
                    ),
                    "example_ids": example_ids,
                    "kb_exemplar_visual": {
                        "visual_evidence_present": bool(
                            kb_visual.get(
                                "visual_evidence_present",
                                False,
                            )
                        ),
                        "preferred_visual_type": str(
                            kb_visual.get(
                                "preferred_visual_type",
                                "",
                            )
                            or ""
                        ),
                        "visual_example_count": safe_int(
                            kb_visual.get(
                                "visual_example_count",
                                0,
                            )
                        ),
                        "source": str(
                            kb_visual.get(
                                "source",
                                "",
                            )
                            or ""
                        ),
                    },
                }
            )

    return {
        "example_bank": example_bank,
        "slot_assignments": assignments,
    }


def build_generation_batch_request(
    request_payload: dict[str, Any],
    blueprint_items: list[dict[str, Any]],
) -> dict[str, Any]:
    batch = {
        "target_generated_question_count": len(blueprint_items),
        "target_generated_marks": sum(
            safe_int(item.get("marks"))
            for item in blueprint_items
        ),
        "blueprint": compact_blueprint_for_transport(blueprint_items),
        "topic_grounding": select_grounding_for_blueprint(
            request_payload, blueprint_items
        ),
    }

    filters = compact_assessment_filters(
        request_payload.get("assessment_filters", {})
    )
    if filters:
        batch["assessment_filters"] = filters

    special_policy = request_payload.get(
        "special_instruction_policy",
        {},
    )
    if isinstance(special_policy, dict):
        raw_instruction = str(
            special_policy.get(
                "raw_instruction",
                "",
            )
            or ""
        ).strip()

        batch["special_instruction_policy"] = {
            "strategy":
                "same_generation_call_interpret_and_generate",
            "separate_instruction_llm_call":
                False,
            "scope":
                str(
                    special_policy.get(
                        "scope",
                        "entire_generated_quiz",
                    )
                    or "entire_generated_quiz"
                ),
            "raw_instruction":
                raw_instruction,
            "deterministic_preflight_hints":
                special_policy.get(
                    "deterministic_preflight_hints",
                    {},
                ),
        }

    # Global targets remain visible even when an operational token/context
    # safeguard splits a large quiz into more than one transport batch.
    global_blueprint = [
        item
        for item in request_payload.get(
            "blueprint",
            [],
        )
        if isinstance(
            item,
            dict,
        )
    ]

    batch["global_quiz_targets"] = {
        "target_generated_question_count":
            safe_int(
                request_payload.get(
                    "target_generated_question_count",
                    len(global_blueprint)
                    or len(blueprint_items),
                )
            ),
        "target_generated_marks":
            safe_int(
                request_payload.get(
                    "target_generated_marks",
                    sum(
                        safe_int(item.get("marks"))
                        for item in global_blueprint
                    )
                    if global_blueprint
                    else sum(
                        safe_int(item.get("marks"))
                        for item in blueprint_items
                    ),
                )
            ),
        "quiz_mode":
            str(
                request_payload.get(
                    "quiz_mode",
                    "",
                )
                or ""
            ),
        "global_role_counts": {
            "primary":
                sum(
                    1
                    for item in global_blueprint
                    if str(
                        item.get(
                            "role",
                            "",
                        )
                        or ""
                    ).strip().casefold()
                    == "primary"
                ),
            "supporting":
                sum(
                    1
                    for item in global_blueprint
                    if str(
                        item.get(
                            "role",
                            "",
                        )
                        or ""
                    ).strip().casefold()
                    == "supporting"
                ),
        },
    }

    batch["transport_batch_scope"] = {
        "plan_indexes":
            [
                safe_int(
                    item.get(
                        "plan_index"
                    )
                )
                for item in blueprint_items
            ],
        "is_full_generation_blueprint":
            bool(
                global_blueprint
                and len(
                    blueprint_items
                )
                == len(
                    global_blueprint
                )
            ),
        "is_targeted_regeneration":
            bool(
                request_payload.get(
                    "targeted_regeneration_plan_indexes",
                    [],
                )
            ),
    }

    # v2.43.4 — preserve targeted-repair context through transport compaction.
    # Keep only the failed plan indexes represented by this transport batch.
    targeted_indexes = {
        safe_int(value)
        for value in safe_list(
            request_payload.get(
                "targeted_regeneration_plan_indexes",
                [],
            )
        )
        if safe_int(value) > 0
    }

    if targeted_indexes:
        batch_plan_indexes = {
            safe_int(
                item.get(
                    "plan_index"
                )
            )
            for item in blueprint_items
            if (
                isinstance(
                    item,
                    dict,
                )
                and safe_int(
                    item.get(
                        "plan_index"
                    )
                ) > 0
            )
        }

        active_targeted_indexes = sorted(
            targeted_indexes
            & batch_plan_indexes
        )

        if active_targeted_indexes:
            batch[
                "targeted_regeneration_plan_indexes"
            ] = active_targeted_indexes

            batch["targeted_regeneration_trigger"] = str(
                request_payload.get(
                    "targeted_regeneration_trigger",
                    "validator_targeted_regeneration",
                )
                or "validator_targeted_regeneration"
            )
            batch["repair_transport_profile"] = "compact_targeted_repair_v1"

            targeted_context = request_payload.get(
                "targeted_regeneration_context",
                {},
            )

            if isinstance(
                targeted_context,
                dict,
            ):
                active_index_set = set(
                    active_targeted_indexes
                )

                original_questions = [
                    item
                    for item in safe_list(
                        targeted_context.get(
                            "original_questions",
                            [],
                        )
                    )
                    if (
                        isinstance(
                            item,
                            dict,
                        )
                        and safe_int(
                            item.get(
                                "plan_index"
                            )
                        )
                        in active_index_set
                    )
                ]

                batch[
                    "targeted_regeneration_context"
                ] = {
                    "repair_mode":
                        str(
                            targeted_context.get(
                                "repair_mode",
                                "minimal_change",
                            )
                            or "minimal_change"
                        ),
                    "original_questions":
                        original_questions,
                    "policy":
                        str(
                            targeted_context.get(
                                "policy",
                                "",
                            )
                            or ""
                        ),
                }

    style_grounding = select_style_grounding_for_blueprint(blueprint_items)
    if style_grounding.get("example_bank"):
        batch["official_style_grounding"] = style_grounding

    return batch


def generation_output_token_need(
    blueprint_items: list[dict[str, Any]],
) -> int:
    """Generic uncapped completion estimate used for batching only.

    This is not a quiz-size mapping. It scales continuously with the amount of
    learner-facing work, marks and blueprint complexity. The extra headroom
    accounts for models whose reasoning/thinking tokens share the completion
    budget.
    """
    question_count = max(1, len(blueprint_items))
    marks = sum(
        max(1, safe_int(item.get("marks")))
        for item in blueprint_items
        if isinstance(item, dict)
    )
    complexities = [
        hybrid_question_complexity(item)
        for item in blueprint_items
        if isinstance(item, dict)
    ]
    average_complexity = (
        sum(complexities) / len(complexities)
        if complexities
        else 1.0
    )

    base = 3000 + 650 * question_count + 180 * marks
    complexity_multiplier = 1.0 + min(0.20, max(0.0, average_complexity - 1.0) * 0.08)

    if GENERATION_PROVIDER == "openai":
        effort = OPENAI_REASONING_EFFORT
    elif GENERATION_PROVIDER == "groq":
        effort = GROQ_REASONING_EFFORT
    else:
        effort = GEMINI_THINKING_LEVEL

    reasoning_headroom = {
        "minimal": 1.05,
        "low": 1.12,
        "medium": 1.25,
        "high": 1.35,
    }.get(str(effort or "medium").casefold(), 1.25)

    return max(1, int(math.ceil(base * complexity_multiplier * reasoning_headroom)))


def generation_output_token_budget(
    blueprint_items: list[dict[str, Any]],
) -> int:
    """Dynamic selected-model completion ceiling.

    The planner uses the uncapped need separately to decide whether a batch is
    too completion-heavy. The provider request itself is clipped to the model's
    configured Notebook 06 generation ceiling.
    """
    calculated = generation_output_token_need(blueprint_items)
    return max(
        MODEL_MIN_GENERATION_OUTPUT_TOKENS,
        min(MODEL_MAX_GENERATION_OUTPUT_TOKENS, calculated),
    )


def generation_response_schema(
    expected_question_count: int,
) -> dict[str, Any]:
    """
    Provider-friendly shallow JSON schema.

    The same generation call must return:
      1. a concise semantic interpretation of any raw special instructions;
      2. a concise compliance report;
      3. the generated questions.

    The provider schema stays intentionally shallow because Notebook 06 performs
    strict deterministic validation after the response. This avoids a second LLM
    interpreter/reviewer call while keeping provider schema complexity low.
    """
    return {
        "type":
            "object",

        "properties": {
            "instruction_interpretation": {
                "type":
                    "object",
            },
            "special_instruction_compliance": {
                "type":
                    "array",
                "items": {
                    "type":
                        "object",
                },
            },
            "questions": {
                "type":
                    "array",

                "items": {
                    "type":
                        "object"
                },
            }
        },

        "required": [
            "instruction_interpretation",
            "special_instruction_compliance",
            "questions",
        ],
    }



def build_generation_prompt(
    request_payload: dict[str, Any],
    validation_feedback: list[str] | None = None,
) -> str:
    feedback = compact_feedback(validation_feedback)
    blueprint_items = [
        item
        for item in request_payload.get("blueprint", [])
        if isinstance(item, dict)
    ]

    output_contract = {
        "instruction_interpretation": {
            "status": "none | applied | conflict",
            "summary": "concise interpretation of the user's raw special instructions",
            "requirements": [
                {
                    "requirement_id": "SI_1",
                    "instruction": "one atomic interpreted requirement",
                    "priority": "mandatory | preference",
                    "scope": "whole_quiz | role | topic | question_subset | generated_shortfall",
                    "machine_checkable": "boolean",
                    "resolved_target": {},
                }
            ],
        },
        "special_instruction_compliance": [
            {
                "requirement_id": "SI_1",
                "satisfied": "boolean",
                "evidence": "concise evidence from this generated batch",
            }
        ],
        "questions": [
            {
                "plan_index": "=blueprint",
                "generated_question_id": "GEN_<3-digit plan_index>",
                "topic": "=blueprint",
                "official_reference": "=blueprint",
                "paper_code": "=blueprint",
                "paper_label": "=blueprint",
                "assessment_pattern": "=blueprint",
                "visual_requirement": "=blueprint",
                "role": "=blueprint",
                "covered_topics": "=blueprint",
                "question_type": "short_answer",
                "question_text": "string",
                "marks": "=blueprint",
                "requires_code": "boolean",
                "requires_visual": "boolean",
                "visual": {"type": "string", "spec": {}},
                "programming_language": "requested language or null",
                "marking_guidance": [{"marks": "int", "criterion": "string"}],
                "answer_verification": {
                    "mode": "machine | mixed | manual",
                    "checks": [
                        {
                            "criterion_index": "1-based int",
                            "expression": "restricted pure Python expression",
                            "expected": "JSON scalar/list/dict"
                        }
                    ],
                    "manual_reason": "short reason when manual/mixed"
                },
                "diversity_signature": {
                    "task_intent": "short abstract learner task",
                    "scenario": "short scenario/context summary",
                    "answer_form": "short answer-form summary"
                },
                "source_type": "ai_generated_aqa_aligned",
                "official_aqa_question": False,
            }
        ]
    }

    parts = [
        (
            "Generate this batch exactly. IMPORTANT SPECIAL-INSTRUCTION ARCHITECTURE: "
            "there is NO separate instruction-interpreter LLM call. In THIS SAME "
            "generation response, first understand the full natural-language meaning "
            "of special_instruction_policy.raw_instruction, then generate the quiz "
            "accordingly, and return instruction_interpretation plus "
            "special_instruction_compliance. The deterministic_preflight_hints are "
            "only conservative machine-checkable hints; they are NOT an exhaustive "
            "interpretation of the user's wording. Preserve the user's modality: "
            "only/exactly/must/avoid are mandatory; prefer/where possible remain "
            "preferences. Any non-empty mandatory instruction must be followed fully "
            "when compatible with the approved AQA scope and hard controls. "
            "If an instruction is genuinely incompatible, set interpretation.status="
            "'conflict' and mark the affected compliance item false rather than "
            "silently weakening it. If transport_batch_scope says this is a split "
            "batch or targeted regeneration, interpret global instructions against "
            "global_quiz_targets and judge compliance by whether THIS BATCH preserves "
            "its assigned contribution; do not mark a global requirement false merely "
            "because this transport batch contains only part of the quiz. "
            "The deterministic blueprint remains authoritative for its fixed slot "
            "fields; preflight hints that affect those fields have already been "
            "resolved into the blueprint. If targeted_regeneration_context is present, perform a MINIMAL REPAIR "
            "against the supplied original question. Preserve learner-task intent, scenario, task family, and "
            "every part of question wording that remains structurally valid. CURRENT HUMAN FEEDBACK and CURRENT "
            "DETERMINISTIC VALIDATION FEEDBACK are authoritative when the baseline conflicts with a hard rule or "
            "immutable blueprint field. In that case preserve the blueprint and rewrite only the smallest learner-facing "
            "fragment needed to remove the contradiction. Keep question_text unchanged when a marking-guidance/acceptance/"
            "metadata repair, or a visual-only repair with no learner-facing contradiction, is sufficient. Never dodge "
            "the requested fix by substituting a different learner task. "
            "topic_grounding defines allowed lesson "
            "scope. If official_style_grounding exists, use only each plan_index's "
            "assigned example_ids for difficulty/task-shape calibration; do not copy. "
            "Keep 1-mark demands atomic, make each task fully self-contained, preserve "
            "multiline code/pseudocode, and put exact expected answers/values/code in "
            "objective marking criteria rather than generic 'correct value' wording."
        ),
        "REQUEST:" + json.dumps(
            request_payload,
            separators=(",", ":"),
            ensure_ascii=False,
            default=str,
        ),
    ]

    if any(
        normalize_visual_requirement(item.get("visual_requirement", "none")) != "none"
        for item in blueprint_items
    ):
        parts.append(
            "VISUAL_CONTRACT:"
            + json.dumps(
                visual_contract_for_blueprint(blueprint_items),
                separators=(",", ":"),
                ensure_ascii=False,
            )
        )

    if feedback:
        parts.append(
            "CORRECTIVE_FEEDBACK:"
            + json.dumps(feedback, separators=(",", ":"), ensure_ascii=False)
        )

    parts.append(
        "OUTPUT_CONTRACT:"
        + json.dumps(output_contract, separators=(",", ":"), ensure_ascii=False)
    )
    parts.append("Return JSON only.")

    return "\n\n".join(parts)



def parse_json_response(
    text: str,
) -> dict[str, Any]:
    value_text = str(
        text or ""
    ).strip()

    if not value_text:
        raise ValueError(
            "Model returned an empty response."
        )

    try:
        value = json.loads(
            value_text
        )

    except json.JSONDecodeError:

        fenced = re.search(
            r"```(?:json)?\s*(\{.*\})\s*```",
            value_text,
            flags=(
                re.DOTALL
                | re.IGNORECASE
            ),
        )

        if not fenced:
            raise

        value = json.loads(
            fenced.group(1)
        )

    if not isinstance(
        value,
        dict,
    ):
        raise ValueError(
            "Model output must be a JSON object."
        )

    return value


def _instruction_report_from_payload(
    payload: dict[str, Any],
    plan_indexes: list[int],
) -> dict[str, Any]:
    interpretation = payload.get(
        "instruction_interpretation",
        {},
    )
    if not isinstance(interpretation, dict):
        interpretation = {}

    compliance = payload.get(
        "special_instruction_compliance",
        [],
    )
    if not isinstance(compliance, list):
        compliance = []

    return {
        "blueprint_plan_indexes":
            list(
                plan_indexes
            ),
        "instruction_interpretation":
            interpretation,
        "special_instruction_compliance":
            [
                item
                for item in compliance
                if isinstance(
                    item,
                    dict,
                )
            ],
    }


def _merge_instruction_batch_reports(
    reports: list[dict[str, Any]],
) -> dict[str, Any]:
    """
    Merge same-call instruction reports across operational transport batches.

    A normal 10-question quiz is intended to fit in one generation batch. If a
    provider/context safeguard splits the request, every batch still receives
    the same raw instruction and contributes one report. No extra interpreter
    or reviewer call is created.
    """
    clean_reports = [
        item
        for item in reports
        if isinstance(
            item,
            dict,
        )
    ]

    if not clean_reports:
        return {
            "instruction_interpretation": {
                "status": "none",
                "summary": "",
                "requirements": [],
            },
            "special_instruction_compliance": [],
            "_special_instruction_batch_reports": [],
        }

    statuses: list[str] = []
    summaries: list[str] = []
    requirements_by_key: dict[str, dict[str, Any]] = {}
    compliance_by_key: dict[str, dict[str, Any]] = {}

    for report in clean_reports:
        interpretation = report.get(
            "instruction_interpretation",
            {},
        )
        if not isinstance(
            interpretation,
            dict,
        ):
            interpretation = {}

        status = str(
            interpretation.get(
                "status",
                "",
            )
            or ""
        ).strip().casefold()

        if status:
            statuses.append(
                status
            )

        summary = str(
            interpretation.get(
                "summary",
                "",
            )
            or ""
        ).strip()

        if summary:
            summaries.append(
                summary
            )

        for requirement in safe_list(
            interpretation.get(
                "requirements",
                [],
            )
        ):
            if not isinstance(
                requirement,
                dict,
            ):
                continue

            requirement_id = str(
                requirement.get(
                    "requirement_id",
                    "",
                )
                or ""
            ).strip()

            instruction = str(
                requirement.get(
                    "instruction",
                    "",
                )
                or ""
            ).strip()

            key = (
                normalize_text(
                    requirement_id
                )
                or normalize_text(
                    instruction
                )
                or stable_json_fingerprint(
                    requirement
                )
            )

            if key not in requirements_by_key:
                requirements_by_key[
                    key
                ] = requirement

        for item in safe_list(
            report.get(
                "special_instruction_compliance",
                [],
            )
        ):
            if not isinstance(
                item,
                dict,
            ):
                continue

            requirement_id = str(
                item.get(
                    "requirement_id",
                    "",
                )
                or ""
            ).strip()

            key = (
                normalize_text(
                    requirement_id
                )
                or stable_json_fingerprint(
                    item
                )
            )

            existing = compliance_by_key.get(
                key
            )

            if existing is None:
                compliance_by_key[
                    key
                ] = dict(
                    item
                )
                continue

            # Across split batches a global requirement is considered satisfied
            # only when every contributing batch reports compliance.
            existing[
                "satisfied"
            ] = bool(
                existing.get(
                    "satisfied",
                    False,
                )
                and item.get(
                    "satisfied",
                    False,
                )
            )

            evidence_parts = [
                str(
                    existing.get(
                        "evidence",
                        "",
                    )
                    or ""
                ).strip(),
                str(
                    item.get(
                        "evidence",
                        "",
                    )
                    or ""
                ).strip(),
            ]

            existing[
                "evidence"
            ] = " | ".join(
                dict.fromkeys(
                    value
                    for value in evidence_parts
                    if value
                )
            )

    merged_status = (
        "conflict"
        if "conflict" in statuses
        else "applied"
        if "applied" in statuses
        else "none"
    )

    return {
        "instruction_interpretation": {
            "status":
                merged_status,
            "summary":
                " | ".join(
                    dict.fromkeys(
                        summaries
                    )
                ),
            "requirements":
                list(
                    requirements_by_key.values()
                ),
        },
        "special_instruction_compliance":
            list(
                compliance_by_key.values()
            ),
        "_special_instruction_batch_reports":
            clean_reports,
    }



# ================================================================
# PLAN C — HYBRID ADAPTIVE BATCHING
# ================================================================

QUIZ_GENERATION_STRATEGY = "plan_c_hybrid_adaptive_batching_v1"
HYBRID_BATCHING_VERSION = "agent2-plan-c-generic-adaptive-v3.0.0"

# Technical ceiling only. This is NOT a quiz-size mapping: the generic token /
# complexity fit test below normally closes a batch first when necessary.
HYBRID_TECHNICAL_MAX_BATCH_QUESTIONS = max(
    1,
    LLM_MAX_GENERATION_BATCH_QUESTIONS,
)

HYBRID_MIN_SAFE_UTILIZATION = min(
    0.95,
    max(
        0.50,
        float(os.getenv("AGENT2_HYBRID_MIN_SAFE_UTILIZATION", "0.68")),
    ),
)
HYBRID_MAX_SAFE_UTILIZATION = min(
    0.98,
    max(
        HYBRID_MIN_SAFE_UTILIZATION,
        float(os.getenv("AGENT2_HYBRID_MAX_SAFE_UTILIZATION", "0.92")),
    ),
)
HYBRID_HIGH_MARK_THRESHOLD = max(
    2,
    safe_int(os.getenv("AGENT2_HYBRID_HIGH_MARK_THRESHOLD", "6")),
)

# Completion/output pressure is evaluated independently from input context.
# This is generic and model-aware: it prevents a very large batch from using
# nearly all completion headroom and triggering truncation/recovery calls,
# without imposing any fixed question-count split.
HYBRID_MAX_OUTPUT_PRESSURE = min(
    0.95,
    max(
        0.55,
        float(os.getenv("AGENT2_HYBRID_MAX_OUTPUT_PRESSURE", "0.80")),
    ),
)

# Targeted repair must not inherit the normal 8K generation floor. These values
# are output ceilings, not forced usage, and are clipped to the active model.
TARGETED_REPAIR_MIN_OUTPUT_TOKENS = min(
    MODEL_MAX_GENERATION_OUTPUT_TOKENS,
    max(
        1024,
        safe_int(os.getenv("AGENT2_TARGETED_REPAIR_MIN_OUTPUT_TOKENS", "2048")),
    ),
)
TARGETED_REPAIR_MAX_OUTPUT_TOKENS = min(
    MODEL_MAX_GENERATION_OUTPUT_TOKENS,
    max(
        TARGETED_REPAIR_MIN_OUTPUT_TOKENS,
        safe_int(os.getenv("AGENT2_TARGETED_REPAIR_MAX_OUTPUT_TOKENS", "6144")),
    ),
)

TARGETED_REPAIR_SYSTEM_PROMPT = (
    "You are repairing only the specified AQA GCSE Computer Science quiz question "
    "slots. The supplied deterministic blueprint is immutable for plan_index, topic, "
    "official_reference, paper routing, marks, role, assessment_pattern and "
    "visual_requirement. Preserve every part of the original learner task that is "
    "already valid and make the smallest change required by CURRENT validation or "
    "human feedback. topic_grounding defines allowed lesson scope. If a visual "
    "contract is supplied, obey it exactly. Never generate unknown/placeholder logic-gate "
    "slot tasks where the learner must assign gate types to labels such as L1/L2/L3/L4, "
    "G1/G2, or equivalent positions; use a fully specified circuit/task instead. If the "
    "learner is asked to complete a truth table, keep learner-completed output cells blank "
    "and leave response-table rendering to the deterministic response scaffold when the "
    "primary visual is not itself a truth table. Marking guidance must sum to the exact "
    "question marks and objective criteria must contain the exact expected answer, "
    "value or code. Return JSON only and return replacements only for the supplied "
    "plan indexes."
)


def is_targeted_generation_request(
    request_payload: dict[str, Any],
) -> bool:
    scope = request_payload.get("transport_batch_scope", {}) or {}
    if isinstance(scope, dict) and bool(scope.get("is_targeted_regeneration")):
        return True
    return bool(
        safe_list(
            request_payload.get(
                "targeted_regeneration_plan_indexes",
                [],
            )
        )
    )


def targeted_repair_output_token_budget(
    blueprint_items: list[dict[str, Any]],
) -> int:
    question_count = max(1, len(blueprint_items))
    marks = sum(
        max(1, safe_int(item.get("marks")))
        for item in blueprint_items
        if isinstance(item, dict)
    )
    calculated = (
        1200
        + 500 * question_count
        + 120 * marks
    )
    return max(
        TARGETED_REPAIR_MIN_OUTPUT_TOKENS,
        min(
            TARGETED_REPAIR_MAX_OUTPUT_TOKENS,
            calculated,
        ),
    )


def _compact_targeted_repair_output_contract() -> dict[str, Any]:
    return {
        "instruction_interpretation": {
            "status": "none | applied | conflict",
            "summary": "concise",
            "requirements": [],
        },
        "special_instruction_compliance": [],
        "questions": [
            {
                "plan_index": "=blueprint",
                "generated_question_id": "GEN_<3-digit plan_index>",
                "topic": "=blueprint",
                "official_reference": "=blueprint",
                "paper_code": "=blueprint",
                "paper_label": "=blueprint",
                "assessment_pattern": "=blueprint",
                "visual_requirement": "=blueprint",
                "role": "=blueprint",
                "covered_topics": "=blueprint",
                "question_type": "short_answer",
                "question_text": "string",
                "marks": "=blueprint",
                "requires_code": "boolean",
                "requires_visual": "boolean",
                "visual": {"type": "string", "spec": {}},
                "programming_language": "requested language or null",
                "marking_guidance": [{"marks": "int", "criterion": "string"}],
                "answer_verification": {
                    "mode": "machine | mixed | manual",
                    "checks": [],
                    "manual_reason": "string",
                },
                "diversity_signature": {
                    "task_intent": "string",
                    "scenario": "string",
                    "answer_form": "string",
                },
                "source_type": "ai_generated_aqa_aligned",
                "official_aqa_question": False,
            }
        ],
    }


def build_targeted_repair_prompt(
    request_payload: dict[str, Any],
    validation_feedback: list[str] | None = None,
) -> str:
    """Compact repair transport: no full SYSTEM_PROMPT or style exemplar bank."""
    blueprint_items = [
        item
        for item in request_payload.get("blueprint", [])
        if isinstance(item, dict)
    ]
    compact_request: dict[str, Any] = {
        "target_generated_question_count": safe_int(
            request_payload.get("target_generated_question_count", len(blueprint_items))
        ),
        "target_generated_marks": safe_int(
            request_payload.get(
                "target_generated_marks",
                sum(safe_int(item.get("marks")) for item in blueprint_items),
            )
        ),
        "blueprint": blueprint_items,
        "topic_grounding": request_payload.get("topic_grounding", []),
        "assessment_filters": request_payload.get("assessment_filters", {}),
        "special_instruction_policy": request_payload.get("special_instruction_policy", {}),
        "transport_batch_scope": request_payload.get("transport_batch_scope", {}),
        "targeted_regeneration_plan_indexes": request_payload.get(
            "targeted_regeneration_plan_indexes", []
        ),
        "targeted_regeneration_context": request_payload.get(
            "targeted_regeneration_context", {}
        ),
    }
    parts = [
        "TARGETED_REPAIR_REQUEST:"
        + json.dumps(
            compact_request,
            separators=(",", ":"),
            ensure_ascii=False,
            default=str,
        )
    ]
    if any(
        normalize_visual_requirement(item.get("visual_requirement", "none")) != "none"
        for item in blueprint_items
    ):
        parts.append(
            "VISUAL_CONTRACT:"
            + json.dumps(
                visual_contract_for_blueprint(blueprint_items),
                separators=(",", ":"),
                ensure_ascii=False,
                default=str,
            )
        )
    feedback = compact_feedback(validation_feedback)
    if feedback:
        parts.append(
            "CURRENT_FEEDBACK:"
            + json.dumps(feedback, separators=(",", ":"), ensure_ascii=False)
        )
    parts.append(
        "OUTPUT_CONTRACT:"
        + json.dumps(
            _compact_targeted_repair_output_contract(),
            separators=(",", ":"),
            ensure_ascii=False,
        )
    )
    parts.append("Return JSON only.")
    return "\n\n".join(parts)


def hybrid_question_complexity(
    item: dict[str, Any],
) -> float:
    """Topic-neutral complexity estimate used only for transport batching."""
    marks = max(1, safe_int(item.get("marks")))
    pattern = str(item.get("assessment_pattern", "") or "").strip().casefold()
    visual_requirement = str(
        item.get("visual_requirement", "none") or "none"
    ).strip().casefold()
    covered_topics = [
        value
        for value in safe_list(item.get("covered_topics", []))
        if str(value or "").strip()
    ]

    score = 1.0 + max(0, marks - 2) * 0.30
    if pattern in {
        "analyse_or_interpret",
        "evaluate_or_justify",
        "construct_or_complete",
        "diagnose_or_correct",
        "scenario_application",
        "adapt_or_modify",
        "multi_step_synthesis",
    }:
        score += 0.60
    if visual_requirement not in {"", "none", "no_visual", "not_required"}:
        score += 0.75
    score += min(0.75, 0.20 * len(covered_topics))
    return round(score, 3)


def generation_batch_metrics(
    request_payload: dict[str, Any],
    blueprint_items: list[dict[str, Any]],
    validation_feedback: list[str] | None = None,
) -> dict[str, Any]:
    """Estimate the real candidate batch and derive a risk-adjusted safe ceiling."""
    batch_request = build_generation_batch_request(
        request_payload,
        blueprint_items,
    )
    targeted = is_targeted_generation_request(batch_request)
    if targeted:
        prompt = build_targeted_repair_prompt(
            batch_request,
            validation_feedback=validation_feedback,
        )
        system_prompt = TARGETED_REPAIR_SYSTEM_PROMPT
        output_budget = targeted_repair_output_token_budget(blueprint_items)
    else:
        prompt = build_generation_prompt(
            batch_request,
            validation_feedback=validation_feedback,
        )
        system_prompt = SYSTEM_PROMPT
        output_budget = generation_output_token_budget(blueprint_items)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]
    estimated_input_tokens = estimate_messages_tokens(messages)
    estimated_total = estimated_input_tokens + output_budget
    uncapped_output_need = (
        targeted_repair_output_token_budget(blueprint_items)
        if targeted
        else generation_output_token_need(blueprint_items)
    )
    output_pressure_ratio = (
        float(uncapped_output_need)
        / max(1, MODEL_MAX_GENERATION_OUTPUT_TOKENS)
    )

    count = max(1, len(blueprint_items))
    complexities = [hybrid_question_complexity(item) for item in blueprint_items]
    average_complexity = sum(complexities) / count
    visual_fraction = (
        sum(
            1
            for item in blueprint_items
            if normalize_visual_requirement(item.get("visual_requirement", "none")) != "none"
        )
        / count
    )
    high_mark_fraction = (
        sum(
            1
            for item in blueprint_items
            if safe_int(item.get("marks")) >= HYBRID_HIGH_MARK_THRESHOLD
        )
        / count
    )

    complexity_component = min(1.0, average_complexity / 5.0)
    risk_score = min(
        1.0,
        0.55 * complexity_component
        + 0.25 * visual_fraction
        + 0.20 * high_mark_fraction,
    )
    safe_utilization = max(
        HYBRID_MIN_SAFE_UTILIZATION,
        min(
            HYBRID_MAX_SAFE_UTILIZATION,
            HYBRID_MAX_SAFE_UTILIZATION
            - (HYBRID_MAX_SAFE_UTILIZATION - HYBRID_MIN_SAFE_UTILIZATION)
            * risk_score,
        ),
    )

    risk_adjusted_soft_ceiling = max(
        1,
        int(LLM_BATCH_SOFT_TOKEN_LIMIT * safe_utilization),
    )
    context_ceiling = max(
        1,
        int(ACTIVE_MODEL_CONTEXT_WINDOW_TOKENS * safe_utilization),
    )
    safe_token_ceiling = min(risk_adjusted_soft_ceiling, context_ceiling)
    if ACTIVE_MODEL_PROVIDER_TPM_LIMIT_TOKENS is not None:
        safe_token_ceiling = min(
            safe_token_ceiling,
            max(1, int(ACTIVE_MODEL_PROVIDER_TPM_LIMIT_TOKENS * 0.90)),
        )

    return {
        "targeted_repair": targeted,
        "question_count": len(blueprint_items),
        "marks": sum(safe_int(item.get("marks")) for item in blueprint_items),
        "estimated_input_tokens": estimated_input_tokens,
        "reserved_output_tokens": output_budget,
        "uncapped_output_need_tokens": uncapped_output_need,
        "output_pressure_ratio": round(output_pressure_ratio, 3),
        "max_output_pressure": HYBRID_MAX_OUTPUT_PRESSURE,
        "estimated_total_tokens": estimated_total,
        "average_complexity": round(average_complexity, 3),
        "visual_fraction": round(visual_fraction, 3),
        "high_mark_fraction": round(high_mark_fraction, 3),
        "risk_score": round(risk_score, 3),
        "safe_utilization": round(safe_utilization, 3),
        "safe_token_ceiling": safe_token_ceiling,
        "technical_question_ceiling": HYBRID_TECHNICAL_MAX_BATCH_QUESTIONS,
    }


def generation_batch_fits(
    request_payload: dict[str, Any],
    blueprint_items: list[dict[str, Any]],
    validation_feedback: list[str] | None = None,
) -> bool:
    if not blueprint_items:
        return True
    if len(blueprint_items) > HYBRID_TECHNICAL_MAX_BATCH_QUESTIONS:
        return False
    metrics = generation_batch_metrics(
        request_payload,
        blueprint_items,
        validation_feedback=validation_feedback,
    )
    return bool(
        metrics["estimated_total_tokens"]
        <= metrics["safe_token_ceiling"]
        and (
            metrics["targeted_repair"]
            or metrics["output_pressure_ratio"]
            <= HYBRID_MAX_OUTPUT_PRESSURE
        )
    )


def hybrid_batch_diagnostics(
    request_payload: dict[str, Any],
    blueprint: list[dict[str, Any]],
    batches: list[list[dict[str, Any]]],
    validation_feedback: list[str] | None = None,
) -> dict[str, Any]:
    return {
        "strategy": QUIZ_GENERATION_STRATEGY,
        "batching_version": HYBRID_BATCHING_VERSION,
        "question_count": len(blueprint),
        "batch_count": len(batches),
        "quiz_size_mapping_used": False,
        "technical_max_batch_questions": HYBRID_TECHNICAL_MAX_BATCH_QUESTIONS,
        "soft_token_limit": LLM_BATCH_SOFT_TOKEN_LIMIT,
        "batches": [
            {
                "batch_number": index,
                "plan_indexes": [safe_int(item.get("plan_index")) for item in batch],
                **generation_batch_metrics(
                    request_payload,
                    batch,
                    validation_feedback=validation_feedback,
                ),
            }
            for index, batch in enumerate(batches, start=1)
        ],
    }


def build_generation_batches(
    request_payload: dict[str, Any],
    validation_feedback: list[str] | None = None,
) -> list[list[dict[str, Any]]]:
    """
    Generic Plan C greedy batching.

    There are no quiz-size buckets. Each next question is added only if the real
    candidate batch remains inside the risk-adjusted token/complexity ceiling.
    """
    blueprint = [
        item
        for item in request_payload.get("blueprint", [])
        if isinstance(item, dict)
    ]
    if not blueprint:
        return []

    batches: list[list[dict[str, Any]]] = []
    current: list[dict[str, Any]] = []

    for item in blueprint:
        candidate = current + [item]
        if current and not generation_batch_fits(
            request_payload,
            candidate,
            validation_feedback=validation_feedback,
        ):
            batches.append(current)
            current = [item]
            if not generation_batch_fits(
                request_payload,
                current,
                validation_feedback=validation_feedback,
            ):
                metrics = generation_batch_metrics(
                    request_payload,
                    current,
                    validation_feedback=validation_feedback,
                )
                raise RuntimeError(
                    "A single quiz question request exceeds the generic Plan C safety "
                    "budget after evidence compaction. Metrics="
                    + json.dumps(metrics, ensure_ascii=False, default=str)
                )
        else:
            current = candidate

    if current:
        batches.append(current)

    diagnostics = hybrid_batch_diagnostics(
        request_payload,
        blueprint,
        batches,
        validation_feedback=validation_feedback,
    )
    print()
    print("PLAN C GENERIC ADAPTIVE BATCHING")
    print(json.dumps(diagnostics, indent=2, ensure_ascii=False))
    print()

    try:
        audit_path = OUTPUT_DIR / "hybrid_batching_diagnostics.json"
        audit_path.write_text(
            json.dumps(diagnostics, indent=2, ensure_ascii=False, default=str),
            encoding="utf-8",
        )
    except Exception as exc:
        print("Hybrid batching diagnostics could not be written:", exc)

    return batches


# ================================================================
# MODEL CONTEXT PREFLIGHT + TOKEN USAGE AUDIT
# ================================================================

class ModelPreflightBlockedError(RuntimeError):
    """Base class for local token-budget blocks before any provider API call."""


class ModelContextWindowExceededError(ModelPreflightBlockedError):
    """Selected model context/output limit would be exceeded."""


class ModelProviderTokenBudgetExceededError(ModelPreflightBlockedError):
    """Configured provider/service-tier token budget would be exceeded."""


ACTIVE_MODEL_CALL_AUDIT_CONTEXT: dict[str, Any] = {}


def _set_active_model_call_audit_context(
    value: dict[str, Any] | None,
) -> None:
    ACTIVE_MODEL_CALL_AUDIT_CONTEXT.clear()
    if isinstance(value, dict):
        ACTIVE_MODEL_CALL_AUDIT_CONTEXT.update(value)


def _generation_call_audit_context(
    batch_request: dict[str, Any],
    *,
    recovery_attempt: int = 0,
    transport_retry: str = "",
) -> dict[str, Any]:
    scope = batch_request.get("transport_batch_scope", {}) or {}
    if not isinstance(scope, dict):
        scope = {}
    plan_indexes = [
        safe_int(value)
        for value in safe_list(scope.get("plan_indexes", []))
        if safe_int(value) > 0
    ]
    targeted = bool(scope.get("is_targeted_regeneration"))
    recovery_attempt = max(0, safe_int(recovery_attempt))

    if targeted:
        call_stage = "targeted_question_repair"
        trigger = str(
            batch_request.get("targeted_regeneration_trigger")
            or "validator_targeted_regeneration"
        )
        prompt_profile = "compact_targeted_repair_v1"
    elif recovery_attempt == 0:
        call_stage = "initial_generation_batch"
        trigger = "initial_generation"
        prompt_profile = "full_generation"
    else:
        call_stage = "transport_recovery_batch"
        trigger = str(transport_retry or "provider_transport_recovery")
        prompt_profile = "full_generation"

    return {
        "call_stage": call_stage,
        "trigger": trigger,
        "plan_indexes": plan_indexes,
        "is_initial_generation": bool(not targeted and recovery_attempt == 0),
        "is_targeted_regeneration": bool(targeted),
        "prompt_profile": prompt_profile,
        "repair_transport_profile": str(
            batch_request.get("repair_transport_profile") or ""
        ),
        "recovery_attempt": recovery_attempt,
        "transport_retry": str(transport_retry or ""),
    }


def _persist_model_call_usage() -> None:
    payload = {
        "schema_version": "agent2-model-call-usage-v1.1.0",
        "selected_model_key": SELECTED_MODEL_KEY,
        "provider": GENERATION_PROVIDER,
        "model": GENERATION_MODEL,
        "display_name": GENERATION_MODEL_DISPLAY_NAME,
        "generation_strategy": QUIZ_GENERATION_STRATEGY,
        "batching_version": HYBRID_BATCHING_VERSION,
        "context_window_tokens": ACTIVE_MODEL_CONTEXT_WINDOW_TOKENS,
        "hard_max_output_tokens": ACTIVE_MODEL_HARD_MAX_OUTPUT_TOKENS,
        "provider_tpm_limit_tokens": ACTIVE_MODEL_PROVIDER_TPM_LIMIT_TOKENS,
        "api_hit_count": len(MODEL_API_HIT_ROWS),
        "api_hits": MODEL_API_HIT_ROWS,
        "calls": MODEL_CALL_USAGE_ROWS,
    }

    MODEL_CALL_USAGE_PATH.write_text(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
            default=str,
        ),
        encoding="utf-8",
    )


def _append_model_usage_row(
    row: dict[str, Any],
) -> None:
    MODEL_CALL_USAGE_ROWS.append(
        {
            "call_number": len(MODEL_CALL_USAGE_ROWS) + 1,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            **ACTIVE_MODEL_CALL_AUDIT_CONTEXT,
            **row,
        }
    )
    _persist_model_call_usage()


def _record_model_api_hit(
    *,
    call_kind: str = "quiz_generation",
) -> None:
    """Count every actual provider request attempt, including retries/fallbacks."""
    MODEL_API_HIT_ROWS.append(
        {
            "hit_number": len(MODEL_API_HIT_ROWS) + 1,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "call_kind": call_kind,
            **ACTIVE_MODEL_CALL_AUDIT_CONTEXT,
            "provider": GENERATION_PROVIDER,
            "model": GENERATION_MODEL,
            "model_display_name": GENERATION_MODEL_DISPLAY_NAME,
        }
    )
    _persist_model_call_usage()


def _preflight_estimated_input_tokens(
    messages: list[dict[str, Any]],
) -> tuple[int, int]:
    """
    Return (base estimate, conservative estimate).

    Notebook 06 already has a lightweight char/token estimator for batching.
    The preflight uses a more conservative model-config ratio + safety factor
    so the check errs on the side of not spending an API request.
    """
    serialized = json.dumps(
        messages,
        ensure_ascii=False,
        default=str,
    )

    base_estimate = max(
        1,
        int(
            math.ceil(
                len(serialized)
                / MODEL_PREFLIGHT_CHARS_PER_TOKEN
            )
        ),
    )

    conservative_estimate = max(
        base_estimate,
        int(
            math.ceil(
                base_estimate
                * MODEL_PREFLIGHT_SAFETY_MULTIPLIER
            )
        ),
    )

    return (
        base_estimate,
        conservative_estimate,
    )


def model_context_preflight(
    messages: list[dict[str, Any]],
    *,
    max_output_tokens: int,
    call_kind: str = "quiz_generation",
    exact_input_tokens: int | None = None,
    token_count_source: str = "local_estimate",
) -> dict[str, Any]:
    """
    Check token safety BEFORE any generation API request.

    Enforced independently:
    - model hard output limit;
    - model context window;
    - optional provider/service-tier TPM limit from config.
    """
    if exact_input_tokens is not None:
        base_input_estimate = max(1, int(exact_input_tokens))
        conservative_input_estimate = base_input_estimate
        token_count_source = str(token_count_source or "provider_exact")
    else:
        (
            base_input_estimate,
            conservative_input_estimate,
        ) = _preflight_estimated_input_tokens(messages)
        token_count_source = str(token_count_source or "local_estimate")

    requested_output = int(max_output_tokens)
    conservative_total = (
        conservative_input_estimate
        + requested_output
    )

    result = {
        "call_kind": call_kind,
        "api_called": False,
        "status": "PREFLIGHT_OK",
        "provider": GENERATION_PROVIDER,
        "model": GENERATION_MODEL,
        "model_display_name": GENERATION_MODEL_DISPLAY_NAME,
        "estimated_input_tokens": base_input_estimate,
        "conservative_estimated_input_tokens": conservative_input_estimate,
        "token_count_source": token_count_source,
        "provider_exact_input_tokens": (
            base_input_estimate
            if exact_input_tokens is not None
            else None
        ),
        "requested_max_output_tokens": requested_output,
        "conservative_total_reserved_tokens": conservative_total,
        "context_window_tokens": ACTIVE_MODEL_CONTEXT_WINDOW_TOKENS,
        "hard_max_output_tokens": ACTIVE_MODEL_HARD_MAX_OUTPUT_TOKENS,
        "provider_tpm_limit_tokens": ACTIVE_MODEL_PROVIDER_TPM_LIMIT_TOKENS,
    }

    blocker = None
    status = None
    exc_type = None

    if requested_output > ACTIVE_MODEL_HARD_MAX_OUTPUT_TOKENS:
        status = "BLOCKED_HARD_OUTPUT_LIMIT"
        exc_type = ModelContextWindowExceededError
        blocker = (
            "Requested output budget exceeds the selected model hard output "
            f"limit ({requested_output} > "
            f"{ACTIVE_MODEL_HARD_MAX_OUTPUT_TOKENS})."
        )

    elif conservative_total > ACTIVE_MODEL_CONTEXT_WINDOW_TOKENS:
        status = "BLOCKED_CONTEXT_WINDOW"
        exc_type = ModelContextWindowExceededError
        blocker = (
            "Estimated request exceeds the selected model context window. "
            f"estimated_input={conservative_input_estimate}, "
            f"reserved_output={requested_output}, "
            f"estimated_total={conservative_total}, "
            f"context_window={ACTIVE_MODEL_CONTEXT_WINDOW_TOKENS}."
        )

    elif (
        ACTIVE_MODEL_PROVIDER_TPM_LIMIT_TOKENS is not None
        and conservative_total > ACTIVE_MODEL_PROVIDER_TPM_LIMIT_TOKENS
    ):
        status = "BLOCKED_PROVIDER_TPM_LIMIT"
        exc_type = ModelProviderTokenBudgetExceededError
        blocker = (
            "Estimated request exceeds the configured provider/service-tier "
            "token budget. "
            f"estimated_input={conservative_input_estimate}, "
            f"reserved_output={requested_output}, "
            f"estimated_total={conservative_total}, "
            f"provider_tpm_limit={ACTIVE_MODEL_PROVIDER_TPM_LIMIT_TOKENS}."
        )

    if blocker is not None:
        result["status"] = status
        result["blocker"] = blocker
        _append_model_usage_row(result)

        print()
        print("MODEL CALL BLOCKED BEFORE API")
        print("Model:", GENERATION_MODEL_DISPLAY_NAME)
        print("Provider:", GENERATION_PROVIDER)
        print("Estimated input tokens:", conservative_input_estimate)
        print("Reserved output tokens:", requested_output)
        print("Estimated total reserved:", conservative_total)
        print("Context window:", ACTIVE_MODEL_CONTEXT_WINDOW_TOKENS)
        print(
            "Provider/service-tier TPM:",
            ACTIVE_MODEL_PROVIDER_TPM_LIMIT_TOKENS
            if ACTIVE_MODEL_PROVIDER_TPM_LIMIT_TOKENS is not None
            else "not configured",
        )
        print(blocker)
        print("No provider API request was sent.")
        print()

        raise exc_type(blocker)

    return result


def provider_safe_output_budget_for_messages(
    messages: list[dict[str, Any]],
    *,
    desired_output_tokens: int,
) -> int:
    """
    Calculate a smaller locally safe completion reservation without API usage.
    """
    if ACTIVE_MODEL_PROVIDER_TPM_LIMIT_TOKENS is None:
        return int(desired_output_tokens)

    _, conservative_input = _preflight_estimated_input_tokens(messages)

    remaining = (
        ACTIVE_MODEL_PROVIDER_TPM_LIMIT_TOKENS
        - conservative_input
    )

    # Extra local buffer for estimation error.
    remaining = int(max(0, remaining) * 0.92)

    return max(
        0,
        min(
            int(desired_output_tokens),
            remaining,
            ACTIVE_MODEL_HARD_MAX_OUTPUT_TOKENS,
        ),
    )


def _provider_usage_snapshot(
    response: Any,
    *,
    provider: str,
) -> dict[str, int | None]:
    provider_key = str(
        provider
        or ""
    ).strip().casefold()

    def _int_or_none(
        value: Any,
    ) -> int | None:
        try:
            return (
                int(value)
                if value is not None
                else None
            )
        except (
            TypeError,
            ValueError,
        ):
            return None

    if provider_key == "google_gemini":
        usage = getattr(
            response,
            "usage_metadata",
            None,
        )

        return {
            "input_tokens":
                _int_or_none(
                    getattr(
                        usage,
                        "prompt_token_count",
                        None,
                    )
                    if usage is not None
                    else None
                ),
            "output_tokens":
                _int_or_none(
                    getattr(
                        usage,
                        "candidates_token_count",
                        None,
                    )
                    if usage is not None
                    else None
                ),
            "reasoning_tokens":
                _int_or_none(
                    getattr(
                        usage,
                        "thoughts_token_count",
                        None,
                    )
                    if usage is not None
                    else None
                ),
            "total_tokens":
                _int_or_none(
                    getattr(
                        usage,
                        "total_token_count",
                        None,
                    )
                    if usage is not None
                    else None
                ),
        }

    if provider_key in {"groq", "openai"}:
        usage = getattr(
            response,
            "usage",
            None,
        )

        details = (
            getattr(
                usage,
                "completion_tokens_details",
                None,
            )
            if usage is not None
            else None
        )

        return {
            "input_tokens":
                _int_or_none(
                    getattr(
                        usage,
                        "prompt_tokens",
                        None,
                    )
                    if usage is not None
                    else None
                ),
            "output_tokens":
                _int_or_none(
                    getattr(
                        usage,
                        "completion_tokens",
                        None,
                    )
                    if usage is not None
                    else None
                ),
            "reasoning_tokens":
                _int_or_none(
                    getattr(
                        details,
                        "reasoning_tokens",
                        None,
                    )
                    if details is not None
                    else None
                ),
            "total_tokens":
                _int_or_none(
                    getattr(
                        usage,
                        "total_tokens",
                        None,
                    )
                    if usage is not None
                    else None
                ),
        }

    return {
        "input_tokens": None,
        "output_tokens": None,
        "reasoning_tokens": None,
        "total_tokens": None,
    }


def record_model_response_usage(
    response: Any,
    *,
    preflight: dict[str, Any],
    call_kind: str = "quiz_generation",
) -> None:
    usage = _provider_usage_snapshot(
        response,
        provider=GENERATION_PROVIDER,
    )

    row = {
        **preflight,
        "call_kind": call_kind,
        "api_called": True,
        "status": "API_RESPONSE_RECEIVED",
        "actual_input_tokens": usage.get(
            "input_tokens"
        ),
        "actual_output_tokens": usage.get(
            "output_tokens"
        ),
        "actual_reasoning_tokens": usage.get(
            "reasoning_tokens"
        ),
        "actual_total_tokens": usage.get(
            "total_tokens"
        ),
    }

    _append_model_usage_row(
        row
    )

    print()
    print("MODEL CALL TOKEN USAGE")
    print("Model:", GENERATION_MODEL_DISPLAY_NAME)
    print(
        "Input tokens:",
        usage.get(
            "input_tokens"
        ),
    )
    print(
        "Output/completion tokens:",
        usage.get(
            "output_tokens"
        ),
    )

    reasoning_tokens = usage.get(
        "reasoning_tokens"
    )

    if reasoning_tokens is not None:
        print(
            "Reasoning/thinking tokens:",
            reasoning_tokens,
        )

    print(
        "Total tokens:",
        usage.get(
            "total_tokens"
        ),
    )
    print(
        "Context window:",
        ACTIVE_MODEL_CONTEXT_WINDOW_TOKENS,
    )
    print()


def model_usage_summary() -> dict[str, Any]:
    api_rows = [
        row
        for row in MODEL_CALL_USAGE_ROWS
        if bool(
            row.get(
                "api_called"
            )
        )
    ]

    blocked_rows = [
        row
        for row in MODEL_CALL_USAGE_ROWS
        if str(
            row.get(
                "status",
                "",
            )
        ).startswith("BLOCKED_")
    ]

    context_blocked_rows = [
        row
        for row in blocked_rows
        if str(row.get("status", "")) == "BLOCKED_CONTEXT_WINDOW"
    ]

    per_response_context_tokens = [
        safe_int(row.get("actual_input_tokens"))
        + safe_int(row.get("actual_output_tokens"))
        for row in api_rows
    ]

    max_actual_context_tokens = max(
        per_response_context_tokens,
        default=0,
    )

    def _sum_field(
        field: str,
    ) -> int:
        return sum(
            safe_int(
                row.get(
                    field
                )
            )
            for row in api_rows
        )

    return {
        "selected_model_key": SELECTED_MODEL_KEY,
        "provider": GENERATION_PROVIDER,
        "model": GENERATION_MODEL,
        "display_name": GENERATION_MODEL_DISPLAY_NAME,
        "context_window_tokens": ACTIVE_MODEL_CONTEXT_WINDOW_TOKENS,
        "hard_max_output_tokens": ACTIVE_MODEL_HARD_MAX_OUTPUT_TOKENS,
        "api_hits": len(
            MODEL_API_HIT_ROWS
        ),
        "generation_api_hits": sum(
            1 for row in MODEL_API_HIT_ROWS
            if str(row.get("call_kind", "")) == "quiz_generation"
        ),
        "utility_api_hits": sum(
            1 for row in MODEL_API_HIT_ROWS
            if str(row.get("call_kind", "")) != "quiz_generation"
        ),
        "api_hit_breakdown": {
            kind: sum(
                1 for row in MODEL_API_HIT_ROWS
                if str(row.get("call_kind", "")) == kind
            )
            for kind in sorted({
                str(row.get("call_kind", "unknown"))
                for row in MODEL_API_HIT_ROWS
            })
        },
        "completed_api_responses": len(
            api_rows
        ),
        "preflight_blocks": len(
            blocked_rows
        ),
        "context_window_blocks": len(
            context_blocked_rows
        ),
        "max_actual_context_tokens_per_response": max_actual_context_tokens,
        "max_context_utilization_pct": (
            round(
                100.0
                * max_actual_context_tokens
                / ACTIVE_MODEL_CONTEXT_WINDOW_TOKENS,
                4,
            )
            if ACTIVE_MODEL_CONTEXT_WINDOW_TOKENS > 0
            else 0.0
        ),
        "actual_input_tokens": _sum_field(
            "actual_input_tokens"
        ),
        "actual_output_tokens": _sum_field(
            "actual_output_tokens"
        ),
        "actual_reasoning_tokens": _sum_field(
            "actual_reasoning_tokens"
        ),
        "actual_total_tokens": _sum_field(
            "actual_total_tokens"
        ),
        "usage_log_path": str(
            MODEL_CALL_USAGE_PATH
        ),
    }


class GeminiOutputTruncatedError(
    RuntimeError
):
    """Raised when Gemini stops before a complete structured JSON response."""


class GeminiMalformedJSONError(
    RuntimeError
):
    """Raised when Gemini finishes but the returned JSON is not parseable."""

    def __init__(
        self,
        message: str,
        *,
        response_text: str = "",
        finish_reason: str = "",
    ) -> None:
        super().__init__(
            message
        )

        self.response_text = str(
            response_text
            or ""
        )

        self.finish_reason = str(
            finish_reason
            or ""
        ).strip().upper()


def _conservative_json_repair(
    text: str,
    *,
    allow_top_level_array: bool,
) -> tuple[
    dict[str, Any],
    bool,
] | None:
    """
    Repair only a small, deterministic JSON syntax defect.

    This intentionally fixes trailing commas only. It does NOT invent missing
    strings, values, questions, answers or closing content. If the response is
    genuinely incomplete, the normal retry/fail-closed path remains in control.
    """
    value_text = str(
        text
        or ""
    ).strip().lstrip(
        "\ufeff"
    )

    if not value_text:
        return None

    fenced = re.search(
        r"```(?:json)?\s*([\{\[].*[\}\]])\s*```",
        value_text,
        flags=(
            re.DOTALL
            | re.IGNORECASE
        ),
    )

    candidate = (
        fenced.group(
            1
        ).strip()
        if fenced
        else value_text
    )

    # Only repair a response that already has a complete outer JSON boundary.
    # This prevents a cut-off response from being "completed" by guesswork.
    has_complete_boundary = bool(
        (
            candidate.startswith(
                "{"
            )
            and candidate.endswith(
                "}"
            )
        )
        or (
            candidate.startswith(
                "["
            )
            and candidate.endswith(
                "]"
            )
        )
    )

    if not has_complete_boundary:
        return None

    repaired_text = re.sub(
        r",\s*([}\]])",
        r"\1",
        candidate,
    )

    if repaired_text == candidate:
        return None

    try:
        value = json.loads(
            repaired_text
        )

    except json.JSONDecodeError:
        return None

    if isinstance(
        value,
        dict,
    ):
        return (
            value,
            False,
        )

    if (
        allow_top_level_array
        and isinstance(
            value,
            list,
        )
        and all(
            isinstance(
                item,
                dict,
            )
            for item in value
        )
    ):
        return (
            {
                "questions":
                    value,
            },
            True,
        )

    return None


def _gemini_finish_reason(
    response: Any,
) -> str:
    candidates = getattr(
        response,
        "candidates",
        None,
    )

    if not candidates:
        return ""

    candidate = candidates[
        0
    ]

    reason = getattr(
        candidate,
        "finish_reason",
        None,
    )

    if reason is None:
        return ""

    name = getattr(
        reason,
        "name",
        None,
    )

    if name:
        return str(
            name
        ).strip().upper()

    text = str(
        reason
    ).strip().upper()

    if "." in text:
        text = text.split(
            "."
        )[
            -1
        ]

    return text


def _is_retryable_gemini_503(exc: Exception) -> bool:
    """Return True only for temporary Gemini HTTP 503 UNAVAILABLE errors."""
    status_code = getattr(
        exc,
        "status_code",
        None,
    )

    if status_code is None:
        status_code = getattr(
            exc,
            "code",
            None,
        )

    try:
        normalized_status = int(
            status_code
        )
    except (TypeError, ValueError):
        normalized_status = None

    error_text = str(
        exc
    ).upper()

    return (
        normalized_status == 503
        or (
            "503" in error_text
            and "UNAVAILABLE" in error_text
        )
    )


def _gemini_generate_content_with_503_retry(
    client: Any,
    **kwargs: Any,
) -> Any:
    """
    Execute one Gemini generate_content call with bounded outer retry for 503.

    Retry schedule defaults to 5s -> 10s -> 20s -> 40s (five total
    attempts). Non-503 exceptions are raised immediately, preserving the
    notebook's existing schema fallback and validation behaviour.
    """
    delay_seconds = (
        GEMINI_503_INITIAL_BACKOFF_SECONDS
    )

    for attempt in range(
        1,
        GEMINI_503_MAX_ATTEMPTS + 1,
    ):
        try:
            _record_model_api_hit(
                call_kind="quiz_generation",
            )
            return client.models.generate_content(
                **kwargs
            )

        except Exception as exc:
            if not _is_retryable_gemini_503(
                exc
            ):
                raise

            if attempt >= GEMINI_503_MAX_ATTEMPTS:
                print(
                    "Gemini HTTP 503 UNAVAILABLE persisted after "
                    f"{attempt} attempts; propagating the final service error."
                )
                raise

            print(
                "Gemini HTTP 503 UNAVAILABLE / high demand; "
                f"retrying attempt {attempt + 1}/"
                f"{GEMINI_503_MAX_ATTEMPTS} after "
                f"{delay_seconds:g}s."
            )

            if delay_seconds > 0:
                time.sleep(
                    delay_seconds
                )

            delay_seconds = min(
                GEMINI_503_MAX_BACKOFF_SECONDS,
                max(
                    GEMINI_503_INITIAL_BACKOFF_SECONDS,
                    delay_seconds * 2,
                ),
            )


def _gemini_json_call(
    client: Any,
    *,
    model: str,
    messages: list[
        dict[str, Any]
    ],
    max_tokens: int,
    response_schema: dict[str, Any],
    force_mime_only: bool = False,
) -> tuple[
    dict[str, Any],
    dict[str, Any],
]:
    """
    Execute one schema-constrained Gemini JSON call.

    The finish reason is checked before parsing. A MAX_TOKENS response is
    treated as truncation rather than as a generic JSON parsing failure.
    """
    try:
        from google.genai import types

    except ImportError as exc:
        raise RuntimeError(
            "Install the Google GenAI SDK first: pip install -U google-genai"
        ) from exc

    system_instruction = "\n\n".join(
        str(
            message.get(
                "content",
                "",
            )
            or ""
        )
        for message in messages
        if str(
            message.get(
                "role",
                "",
            )
            or ""
        ).strip().casefold()
        == "system"
    ).strip()

    user_content = "\n\n".join(
        str(
            message.get(
                "content",
                "",
            )
            or ""
        )
        for message in messages
        if str(
            message.get(
                "role",
                "",
            )
            or ""
        ).strip().casefold()
        == "user"
    ).strip()

    if not user_content:
        raise ValueError(
            "Gemini request has no user content."
        )

    estimated_prompt_tokens = (
        estimate_messages_tokens(
            messages
        )
    )

    exact_prompt_tokens: int | None = None
    token_count_source = "local_estimate"

    # Gemini exposes an official count_tokens endpoint. Use it by default for
    # the actual preflight input count; fall back safely to the existing local
    # estimate if token counting itself is unavailable/transiently fails.
    if env_flag(
        "AGENT2_GEMINI_EXACT_TOKEN_PREFLIGHT",
        True,
    ):
        try:
            count_kwargs: dict[str, Any] = {
                "model": model,
                "contents": user_content,
            }
            if system_instruction:
                count_kwargs["config"] = types.CountTokensConfig(
                    system_instruction=system_instruction,
                )

            _record_model_api_hit(
                call_kind="token_count_preflight",
            )
            count_response = client.models.count_tokens(
                **count_kwargs
            )
            counted_value = getattr(
                count_response,
                "total_tokens",
                None,
            )
            if counted_value is not None:
                exact_prompt_tokens = max(
                    1,
                    int(counted_value),
                )
                token_count_source = "gemini_count_tokens"

        except Exception as token_count_exc:
            print(
                "WARNING: Gemini exact token counting failed; using the "
                "conservative local estimator instead:",
                token_count_exc,
            )

    preflight = model_context_preflight(
        messages,
        max_output_tokens=max_tokens,
        call_kind="quiz_generation",
        exact_input_tokens=exact_prompt_tokens,
        token_count_source=token_count_source,
    )

    started = time.perf_counter()

    def make_config(
        *,
        include_schema: bool,
    ) -> Any:
        kwargs = {
            "system_instruction":
                (
                    system_instruction
                    or None
                ),

            "response_mime_type":
                "application/json",

            "max_output_tokens":
                max_tokens,

            "thinking_config":
                types.ThinkingConfig(
                    thinking_level=(
                        GEMINI_THINKING_LEVEL
                    )
                ),
        }

        if include_schema:
            kwargs[
                "response_json_schema"
            ] = response_schema

        return types.GenerateContentConfig(
            **kwargs
        )

    schema_fallback_used = False

    if force_mime_only:
        response = _gemini_generate_content_with_503_retry(
            client,
            model=model,
            contents=user_content,
            config=make_config(
                include_schema=False
            ),
        )

    else:
        try:
            response = _gemini_generate_content_with_503_retry(
                client,
                model=model,
                contents=user_content,
                config=make_config(
                    include_schema=True
                ),
            )

        except Exception as schema_exc:
            status_code = getattr(
                schema_exc,
                "status_code",
                None,
            )

            error_text = str(
                schema_exc
            ).upper()

            is_invalid_argument = (
                status_code == 400
                or (
                    "400" in error_text
                    and "INVALID_ARGUMENT" in error_text
                )
            )

            if not is_invalid_argument:
                raise

            print(
                "Gemini rejected the structured-output schema with HTTP 400; "
                "retrying this call once with application/json MIME mode only. "
                "Notebook 06 deterministic validation remains authoritative."
            )

            schema_fallback_used = True

            response = _gemini_generate_content_with_503_retry(
                client,
                model=model,
                contents=user_content,
                config=make_config(
                    include_schema=False
                ),
            )

    elapsed = (
        time.perf_counter()
        - started
    )

    finish_reason = (
        _gemini_finish_reason(
            response
        )
    )

    response_text = str(
        getattr(
            response,
            "text",
            "",
        )
        or ""
    )

    record_model_response_usage(
        response,
        preflight=preflight,
        call_kind="quiz_generation",
    )

    if finish_reason == "MAX_TOKENS":
        raise GeminiOutputTruncatedError(
            "Gemini reached MAX_TOKENS before completing the structured "
            f"JSON response (max_output_tokens={max_tokens})."
        )

    parsed = getattr(
        response,
        "parsed",
        None,
    )

    # MIME-only generation occasionally returns the requested question array
    # directly instead of wrapping it in {"questions": [...]}. Keep the global
    # JSON parser strict because _gemini_json_call is also used by semantic
    # review; normalize this provider-shape quirk only for the explicitly
    # forced MIME-only generation fallback.
    mime_top_level_array_normalized = False
    conservative_json_repair_used = False

    if isinstance(
        parsed,
        dict,
    ):
        payload = parsed

    elif (
        force_mime_only
        and isinstance(
            parsed,
            list,
        )
        and all(
            isinstance(
                item,
                dict,
            )
            for item in parsed
        )
    ):
        payload = {
            "questions":
                parsed,
        }

        mime_top_level_array_normalized = True

    else:
        try:
            payload = parse_json_response(
                response_text
            )

        except json.JSONDecodeError as exc:
            repaired = _conservative_json_repair(
                response_text,
                allow_top_level_array=(
                    force_mime_only
                ),
            )

            if repaired is not None:
                (
                    payload,
                    repaired_array_normalized,
                ) = repaired

                conservative_json_repair_used = True
                mime_top_level_array_normalized = bool(
                    mime_top_level_array_normalized
                    or repaired_array_normalized
                )

            else:
                raise GeminiMalformedJSONError(
                    "Gemini returned malformed JSON after completing the "
                    "response. This is handled separately from MAX_TOKENS "
                    "truncation. "
                    f"finish_reason={finish_reason or 'UNKNOWN'}, "
                    f"line={exc.lineno}, column={exc.colno}.",
                    response_text=(
                        response_text
                    ),
                    finish_reason=(
                        finish_reason
                    ),
                ) from exc

        except ValueError as exc:
            # A complete JSON array is not truncation. Gemini can emit this
            # shape in MIME-only mode even though the prompt requests an object.
            # Accept it only when it is clearly a list of question objects;
            # downstream deterministic validation remains unchanged.
            if (
                force_mime_only
                and str(
                    exc
                ).strip()
                == "Model output must be a JSON object."
            ):
                value_text = str(
                    response_text
                    or ""
                ).strip()

                raw_value = None

                try:
                    raw_value = json.loads(
                        value_text
                    )

                except json.JSONDecodeError:
                    fenced_array = re.search(
                        r"```(?:json)?\\s*(\\[.*\\])\\s*```",
                        value_text,
                        flags=(
                            re.DOTALL
                            | re.IGNORECASE
                        ),
                    )

                    if fenced_array:
                        raw_value = json.loads(
                            fenced_array.group(
                                1
                            )
                        )

                if (
                    isinstance(
                        raw_value,
                        list,
                    )
                    and all(
                        isinstance(
                            item,
                            dict,
                        )
                        for item in raw_value
                    )
                ):
                    payload = {
                        "questions":
                            raw_value,
                    }

                    mime_top_level_array_normalized = True

                else:
                    raise

            else:
                raise

    usage = getattr(
        response,
        "usage_metadata",
        None,
    )

    def usage_value(
        name: str,
    ) -> int | None:
        if usage is None:
            return None

        value = getattr(
            usage,
            name,
            None,
        )

        try:
            return (
                int(
                    value
                )
                if value is not None
                else None
            )

        except (
            TypeError,
            ValueError,
        ):
            return None

    return (
        payload,
        {
            "estimated_prompt_tokens":
                estimated_prompt_tokens,

            "preflight_input_tokens":
                preflight.get("estimated_input_tokens"),

            "preflight_token_count_source":
                preflight.get("token_count_source"),

            "max_output_tokens":
                max_tokens,

            "structured_schema_fallback_used":
                schema_fallback_used,

            "response_schema_used":
                bool(
                    not force_mime_only
                    and not schema_fallback_used
                ),

            "mime_only_forced":
                bool(
                    force_mime_only
                ),

            "mime_top_level_array_normalized":
                bool(
                    mime_top_level_array_normalized
                ),

            "conservative_json_repair_used":
                bool(
                    conservative_json_repair_used
                ),

            "finish_reason":
                finish_reason,

            "prompt_token_count":
                usage_value(
                    "prompt_token_count"
                ),

            "candidate_token_count":
                usage_value(
                    "candidates_token_count"
                ),

            "thoughts_token_count":
                usage_value(
                    "thoughts_token_count"
                ),

            "total_token_count":
                usage_value(
                    "total_token_count"
                ),

            "latency_seconds":
                round(
                    elapsed,
                    4,
                ),
        },
    )

def generate_with_gemini(
    request_payload: dict[str, Any],
    validation_feedback: list[str] | None = None,
) -> dict[str, Any]:

    try:
        from google import genai

    except ImportError as exc:
        raise RuntimeError(
            "Install the Google GenAI SDK first: pip install -U google-genai"
        ) from exc

    api_key = str(
        os.getenv(
            "GEMINI_API_KEY"
        )
        or os.getenv(
            "GOOGLE_API_KEY"
        )
        or ""
    ).strip()

    if not api_key:
        raise RuntimeError(
            "GEMINI_API_KEY is not set in Agent2/.env."
        )

    client = genai.Client(
        api_key=api_key
    )

    initial_batches = build_generation_batches(
        request_payload,
        validation_feedback=(
            validation_feedback
        ),
    )

    if not initial_batches:
        return {
            "questions":
                [],

            "_generation_metadata": {
                "provider":
                    GENERATION_PROVIDER,

                "model":
                    GENERATION_MODEL,

                "thinking_level":
                    GEMINI_THINKING_LEVEL,

                "batch_count":
                    0,
            },
        }

    generated_questions = []

    call_metadata_rows = []

    instruction_batch_reports: list[
        dict[str, Any]
    ] = []

    def execute_blueprint_batch(
        blueprint_batch: list[
            dict[str, Any]
        ],
        *,
        depth: int = 0,
        forced_output_budget: int | None = None,
        strict_mime_retry: bool = False,
    ) -> None:
        """
        Generate one batch.

        Transport recovery is deliberately bounded:
        1. true MAX_TOKENS truncation keeps the existing split/headroom logic;
        2. malformed JSON is treated separately;
        3. a malformed single-question response gets one strict MIME-only retry;
        4. repeated malformed output still fails closed.

        No quiz constraint is changed by these transport retries.
        """
        batch_request = (
            build_generation_batch_request(
                request_payload,
                blueprint_batch,
            )
        )

        targeted_transport = is_targeted_generation_request(
            batch_request
        )
        _set_active_model_call_audit_context(
            _generation_call_audit_context(
                batch_request,
                recovery_attempt=depth,
                transport_retry=(
                    "strict_transport_retry"
                    if strict_mime_retry
                    else ""
                ),
            )
        )

        prompt = (
            build_targeted_repair_prompt(
                batch_request,
                validation_feedback=validation_feedback,
            )
            if targeted_transport
            else build_generation_prompt(
                batch_request,
                validation_feedback=(
                    validation_feedback
                ),
            )
        )
        system_prompt_for_call = (
            TARGETED_REPAIR_SYSTEM_PROMPT
            if targeted_transport
            else SYSTEM_PROMPT
        )

        messages = [
            {
                "role":
                    "system",

                "content":
                    system_prompt_for_call,
            },
            {
                "role":
                    "user",

                "content":
                    prompt,
            },
        ]

        if strict_mime_retry:
            messages.append(
                {
                    "role":
                        "user",

                    "content":
                        (
                            "TRANSPORT RETRY ONLY. Return exactly one valid "
                            "JSON object matching the already supplied quiz "
                            "request. Do not use markdown or prose outside JSON. "
                            "Do not change the blueprint, marks, topics, paper "
                            "routing, visual requirements or question count. "
                            "Before returning, verify commas, quotes, braces and "
                            "brackets are syntactically complete."
                        ),
                }
            )

        normal_budget = (
            targeted_repair_output_token_budget(blueprint_batch)
            if targeted_transport
            else generation_output_token_budget(
                blueprint_batch
            )
        )

        output_budget = (
            int(
                forced_output_budget
            )
            if forced_output_budget
            is not None
            else normal_budget
        )

        output_budget = min(
            GEMINI_MAX_GENERATION_OUTPUT_TOKENS,
            max(
                (
                    TARGETED_REPAIR_MIN_OUTPUT_TOKENS
                    if targeted_transport
                    else GEMINI_MIN_GENERATION_OUTPUT_TOKENS
                ),
                output_budget,
            ),
        )

        plan_indexes = [
            safe_int(
                item.get(
                    "plan_index"
                )
            )
            for item in blueprint_batch
        ]

        print(
            "Gemini generation batch",
            plan_indexes,
            "- questions:",
            len(
                blueprint_batch
            ),
            "- max output tokens:",
            output_budget,
        )

        try:
            payload, call_metadata = (
                _gemini_json_call(
                    client,
                    model=GENERATION_MODEL,
                    messages=messages,
                    max_tokens=output_budget,
                    response_schema=(
                        generation_response_schema(
                            len(
                                blueprint_batch
                            )
                        )
                    ),
                    force_mime_only=(
                        strict_mime_retry
                    ),
                )
            )

            call_metadata[
                "strict_mime_retry"
            ] = bool(
                strict_mime_retry
            )

        except ModelContextWindowExceededError:
            # User-requested fail-fast behaviour: if this exact provider request
            # would exceed the selected model context window/output ceiling,
            # stop locally. Do not split and do not spend an API hit.
            raise

        except ModelProviderTokenBudgetExceededError as preflight_exc:
            if len(blueprint_batch) > 1:
                midpoint = max(1, len(blueprint_batch) // 2)
                left = blueprint_batch[:midpoint]
                right = blueprint_batch[midpoint:]

                print(
                    "Token preflight blocked batch; splitting locally BEFORE API:",
                    plan_indexes,
                    "->",
                    [safe_int(item.get("plan_index")) for item in left],
                    "and",
                    [safe_int(item.get("plan_index")) for item in right],
                )

                execute_blueprint_batch(left, depth=depth + 1)
                execute_blueprint_batch(right, depth=depth + 1)
                return

            safe_output_budget = provider_safe_output_budget_for_messages(
                messages,
                desired_output_tokens=output_budget,
            )

            if 768 <= safe_output_budget < output_budget:
                print(
                    "Single-question request would exceed configured provider "
                    "token budget. Reducing only the reserved output budget "
                    "locally BEFORE API to:",
                    safe_output_budget,
                )
                execute_blueprint_batch(
                    blueprint_batch,
                    depth=depth + 1,
                    forced_output_budget=safe_output_budget,
                    strict_json_retry=strict_json_retry,
                )
                return

            raise preflight_exc

        except GeminiOutputTruncatedError:

            if len(
                blueprint_batch
            ) > 1:
                midpoint = max(
                    1,
                    len(
                        blueprint_batch
                    )
                    // 2,
                )

                left = blueprint_batch[
                    :midpoint
                ]

                right = blueprint_batch[
                    midpoint:
                ]

                print(
                    "Gemini JSON was truncated; splitting batch",
                    plan_indexes,
                    "into",
                    [
                        safe_int(
                            item.get(
                                "plan_index"
                            )
                        )
                        for item in left
                    ],
                    "and",
                    [
                        safe_int(
                            item.get(
                                "plan_index"
                            )
                        )
                        for item in right
                    ],
                )

                execute_blueprint_batch(
                    left,
                    depth=(
                        depth
                        + 1
                    ),
                )

                execute_blueprint_batch(
                    right,
                    depth=(
                        depth
                        + 1
                    ),
                )

                return

            if (
                output_budget
                < GEMINI_MAX_GENERATION_OUTPUT_TOKENS
            ):
                expanded_budget = min(
                    GEMINI_MAX_GENERATION_OUTPUT_TOKENS,
                    max(
                        output_budget
                        + 4096,
                        output_budget
                        * 2,
                    ),
                )

                print(
                    "Gemini single-question JSON was truncated; "
                    "retrying with max output tokens:",
                    expanded_budget,
                )

                execute_blueprint_batch(
                    blueprint_batch,
                    depth=(
                        depth
                        + 1
                    ),
                    forced_output_budget=(
                        expanded_budget
                    ),
                    strict_mime_retry=(
                        strict_mime_retry
                    ),
                )

                return

            raise

        except GeminiMalformedJSONError as malformed_exc:
            if len(
                blueprint_batch
            ) > 1:
                midpoint = max(
                    1,
                    len(
                        blueprint_batch
                    )
                    // 2,
                )

                left = blueprint_batch[
                    :midpoint
                ]

                right = blueprint_batch[
                    midpoint:
                ]

                print(
                    "Gemini returned malformed JSON; splitting batch",
                    plan_indexes,
                    "into",
                    [
                        safe_int(
                            item.get(
                                "plan_index"
                            )
                        )
                        for item in left
                    ],
                    "and",
                    [
                        safe_int(
                            item.get(
                                "plan_index"
                            )
                        )
                        for item in right
                    ],
                )

                execute_blueprint_batch(
                    left,
                    depth=(
                        depth
                        + 1
                    ),
                )

                execute_blueprint_batch(
                    right,
                    depth=(
                        depth
                        + 1
                    ),
                )

                return

            if not strict_mime_retry:
                print(
                    "Gemini single-question response contained malformed JSON; "
                    "retrying that same blueprint once in strict MIME-only mode."
                )

                execute_blueprint_batch(
                    blueprint_batch,
                    depth=(
                        depth
                        + 1
                    ),
                    forced_output_budget=(
                        output_budget
                    ),
                    strict_mime_retry=True,
                )

                return

            raise GeminiMalformedJSONError(
                "Gemini returned malformed JSON again after the bounded "
                "single-question strict MIME-only retry. Failing closed "
                "instead of accepting invalid quiz data.",
                response_text=(
                    malformed_exc.response_text
                ),
                finish_reason=(
                    malformed_exc.finish_reason
                ),
            ) from malformed_exc

        batch_questions = payload.get(
            "questions",
            [],
        )

        if not isinstance(
            batch_questions,
            list,
        ):
            raise ValueError(
                "Gemini returned questions in an invalid format."
            )

        # A deliberately shallow Gemini response schema can be accepted by the
        # provider yet still collapse array items to incomplete/empty objects.
        # That produces the frontend symptom "0 marks / only a few questions"
        # even though the generation request asked for the full blueprint.
        #
        # Keep the existing schema path, but if its returned batch is visibly
        # incomplete, retry ONLY that same batch once in JSON MIME-only mode.
        # All deterministic structural/semantic/HITL validation below remains
        # unchanged and authoritative.
        schema_batch_incomplete = bool(
            call_metadata.get(
                "response_schema_used"
            )
            and (
                len(
                    batch_questions
                )
                != len(
                    blueprint_batch
                )
                or any(
                    (
                        not isinstance(
                            question,
                            dict,
                        )
                        or not str(
                            question.get(
                                "question_text",
                                "",
                            )
                            or ""
                        ).strip()
                        or safe_int(
                            question.get(
                                "marks"
                            )
                        )
                        <= 0
                    )
                    for question in batch_questions
                )
            )
        )

        if schema_batch_incomplete:
            print(
                "Gemini structured-output schema returned an incomplete "
                "question batch; retrying the same blueprint batch once with "
                "application/json MIME mode only."
            )

            try:
                payload, mime_metadata = (
                    _gemini_json_call(
                        client,
                        model=GENERATION_MODEL,
                        messages=messages,
                        max_tokens=output_budget,
                        response_schema=(
                            generation_response_schema(
                                len(
                                    blueprint_batch
                                )
                            )
                        ),
                        force_mime_only=True,
                    )
                )

            except GeminiOutputTruncatedError:
                # The schema-integrity fallback is still a Gemini generation
                # call, so it must use the same adaptive truncation handling as
                # the primary structured-output call. No quiz constraint is
                # changed by this transport retry.
                if len(
                    blueprint_batch
                ) > 1:
                    midpoint = max(
                        1,
                        len(
                            blueprint_batch
                        )
                        // 2,
                    )

                    left = blueprint_batch[
                        :midpoint
                    ]

                    right = blueprint_batch[
                        midpoint:
                    ]

                    print(
                        "Gemini MIME-only JSON retry was truncated; splitting batch",
                        plan_indexes,
                        "into",
                        [
                            safe_int(
                                item.get(
                                    "plan_index"
                                )
                            )
                            for item in left
                        ],
                        "and",
                        [
                            safe_int(
                                item.get(
                                    "plan_index"
                                )
                            )
                            for item in right
                        ],
                    )

                    execute_blueprint_batch(
                        left,
                        depth=(
                            depth
                            + 1
                        ),
                    )

                    execute_blueprint_batch(
                        right,
                        depth=(
                            depth
                            + 1
                        ),
                    )

                    return

                if (
                    output_budget
                    < GEMINI_MAX_GENERATION_OUTPUT_TOKENS
                ):
                    expanded_budget = min(
                        GEMINI_MAX_GENERATION_OUTPUT_TOKENS,
                        max(
                            output_budget
                            + 4096,
                            output_budget
                            * 2,
                        ),
                    )

                    print(
                        "Gemini single-question MIME-only JSON retry was "
                        "truncated; retrying with max output tokens:",
                        expanded_budget,
                    )

                    execute_blueprint_batch(
                        blueprint_batch,
                        depth=(
                            depth
                            + 1
                        ),
                        forced_output_budget=(
                            expanded_budget
                        ),
                        strict_mime_retry=True,
                    )

                    return

                raise

            except GeminiMalformedJSONError as malformed_exc:
                if len(
                    blueprint_batch
                ) > 1:
                    midpoint = max(
                        1,
                        len(
                            blueprint_batch
                        )
                        // 2,
                    )

                    left = blueprint_batch[
                        :midpoint
                    ]

                    right = blueprint_batch[
                        midpoint:
                    ]

                    print(
                        "Gemini MIME-only fallback returned malformed JSON; "
                        "splitting batch",
                        plan_indexes,
                        "into",
                        [
                            safe_int(
                                item.get(
                                    "plan_index"
                                )
                            )
                            for item in left
                        ],
                        "and",
                        [
                            safe_int(
                                item.get(
                                    "plan_index"
                                )
                            )
                            for item in right
                        ],
                    )

                    execute_blueprint_batch(
                        left,
                        depth=(
                            depth
                            + 1
                        ),
                    )

                    execute_blueprint_batch(
                        right,
                        depth=(
                            depth
                            + 1
                        ),
                    )

                    return

                print(
                    "Gemini single-question MIME-only fallback returned "
                    "malformed JSON; retrying that same blueprint once with "
                    "the strict JSON transport reminder."
                )

                execute_blueprint_batch(
                    blueprint_batch,
                    depth=(
                        depth
                        + 1
                    ),
                    forced_output_budget=(
                        output_budget
                    ),
                    strict_mime_retry=True,
                )

                return

            batch_questions = payload.get(
                "questions",
                [],
            )

            if not isinstance(
                batch_questions,
                list,
            ):
                raise ValueError(
                    "Gemini returned questions in an invalid format."
                )

            mime_metadata[
                "schema_integrity_retry_used"
            ] = True

            mime_metadata[
                "strict_mime_retry"
            ] = False

            call_metadata = (
                mime_metadata
            )

        instruction_batch_reports.append(
            _instruction_report_from_payload(
                payload,
                plan_indexes,
            )
        )

        generated_questions.extend(
            batch_questions
        )

        parsed_question_objects = [
            question
            for question in batch_questions
            if isinstance(question, dict)
        ]

        parsed_plan_indexes = [
            safe_int(question.get("plan_index"))
            for question in parsed_question_objects
        ]

        call_metadata_rows.append(
            {
                "blueprint_plan_indexes":
                    plan_indexes,

                "question_count":
                    len(
                        blueprint_batch
                    ),

                "parsed_question_count":
                    len(parsed_question_objects),

                "parsed_plan_indexes":
                    parsed_plan_indexes,

                "batch_cardinality_match":
                    len(parsed_question_objects)
                    == len(blueprint_batch),

                "adaptive_split_depth":
                    depth,

                **call_metadata,
            }
        )

    for blueprint_batch in initial_batches:
        execute_blueprint_batch(
            blueprint_batch
        )

    generated_questions.sort(
        key=lambda item: safe_int(
            item.get(
                "plan_index"
            )
        )
        if isinstance(
            item,
            dict,
        )
        else 0
    )

    merged_instruction_report = (
        _merge_instruction_batch_reports(
            instruction_batch_reports
        )
    )

    return {
        "questions":
            generated_questions,

        "instruction_interpretation":
            merged_instruction_report[
                "instruction_interpretation"
            ],

        "special_instruction_compliance":
            merged_instruction_report[
                "special_instruction_compliance"
            ],

        "_special_instruction_batch_reports":
            merged_instruction_report[
                "_special_instruction_batch_reports"
            ],

        "_generation_metadata": {
            "provider":
                GENERATION_PROVIDER,

            "model":
                GENERATION_MODEL,

            "thinking_level":
                GEMINI_THINKING_LEVEL,

            "initial_batch_count":
                len(
                    initial_batches
                ),

            "completed_api_call_count":
                len(
                    call_metadata_rows
                ),

            "batch_soft_token_limit":
                LLM_BATCH_SOFT_TOKEN_LIMIT,

            "batches":
                call_metadata_rows,
        },
    }


# ================================================================
# GROQ / GPT-OSS 120B PROVIDER ADAPTER
# ================================================================

class GroqOutputTruncatedError(RuntimeError):
    """Raised when Groq stops before a complete structured JSON response."""


class GroqMalformedJSONError(RuntimeError):
    """Raised when Groq finishes but the returned JSON is not parseable."""

    def __init__(
        self,
        message: str,
        *,
        response_text: str = "",
        finish_reason: str = "",
    ) -> None:
        super().__init__(message)
        self.response_text = str(response_text or "")
        self.finish_reason = str(finish_reason or "").strip().upper()


def _groq_status_code(exc: Exception) -> int | None:
    status_code = getattr(exc, "status_code", None)
    if status_code is None:
        status_code = getattr(exc, "code", None)
    try:
        return int(status_code) if status_code is not None else None
    except (TypeError, ValueError):
        return None


def _is_retryable_groq_transient(exc: Exception) -> bool:
    """Retry only temporary server-side Groq failures; do not hide 429 quota errors."""
    status = _groq_status_code(exc)
    if status in {500, 502, 503, 504}:
        return True
    text = str(exc).upper()
    return any(code in text for code in ("500", "502", "503", "504")) and (
        "SERVER" in text or "UNAVAILABLE" in text or "TIMEOUT" in text
    )


def _groq_chat_completion_with_retry(
    client: Any,
    **kwargs: Any,
) -> Any:
    """Execute one Groq chat-completion call with bounded 5xx backoff."""
    delay_seconds = GROQ_TRANSIENT_INITIAL_BACKOFF_SECONDS

    for attempt in range(1, GROQ_TRANSIENT_MAX_ATTEMPTS + 1):
        try:
            _record_model_api_hit(
                call_kind="quiz_generation",
            )
            return client.chat.completions.create(**kwargs)
        except Exception as exc:
            if not _is_retryable_groq_transient(exc):
                raise
            if attempt >= GROQ_TRANSIENT_MAX_ATTEMPTS:
                print(
                    "Groq transient server error persisted after "
                    f"{attempt} attempts; propagating the final service error."
                )
                raise
            print(
                "Groq transient server error; retrying attempt "
                f"{attempt + 1}/{GROQ_TRANSIENT_MAX_ATTEMPTS} after "
                f"{delay_seconds:g}s."
            )
            if delay_seconds > 0:
                time.sleep(delay_seconds)
            delay_seconds = min(
                GROQ_TRANSIENT_MAX_BACKOFF_SECONDS,
                max(
                    GROQ_TRANSIENT_INITIAL_BACKOFF_SECONDS,
                    delay_seconds * 2,
                ),
            )


def _groq_finish_reason(response: Any) -> str:
    try:
        reason = response.choices[0].finish_reason
    except Exception:
        return ""
    return str(reason or "").strip().upper()


def _groq_json_call(
    client: Any,
    *,
    model: str,
    messages: list[dict[str, Any]],
    max_tokens: int,
    response_schema: dict[str, Any],
    force_json_object: bool = False,
) -> tuple[dict[str, Any], dict[str, Any]]:
    """
    Execute one Groq GPT-OSS 120B JSON call.

    Primary transport uses Groq JSON Schema mode (best-effort shallow schema)
    because Notebook 06's deterministic validators enforce the full contract.
    If the provider rejects/cannot satisfy that transport schema, retry the same
    request in JSON Object mode. A LENGTH finish reason is treated as truncation.
    """
    if not messages:
        raise ValueError("Groq request has no messages.")

    estimated_prompt_tokens = estimate_messages_tokens(messages)

    preflight = model_context_preflight(
        messages,
        max_output_tokens=max_tokens,
        call_kind="quiz_generation",
    )

    started = time.perf_counter()
    schema_fallback_used = False

    schema_response_format = {
        "type": "json_schema",
        "json_schema": {
            "name": "agent2_quiz_batch",
            "strict": False,
            "schema": response_schema,
        },
    }
    json_object_response_format = {"type": "json_object"}

    def make_call(response_format: dict[str, Any]) -> Any:
        return _groq_chat_completion_with_retry(
            client,
            model=model,
            messages=messages,
            max_completion_tokens=max_tokens,
            reasoning_effort=GROQ_REASONING_EFFORT,
            response_format=response_format,
        )

    if force_json_object:
        response = make_call(json_object_response_format)
    else:
        try:
            response = make_call(schema_response_format)
        except Exception as schema_exc:
            status = _groq_status_code(schema_exc)
            text = str(schema_exc).casefold()
            schema_related = (
                status == 400
                and any(
                    token in text
                    for token in (
                        "schema",
                        "response_format",
                        "structured output",
                        "generated json",
                        "json does not match",
                    )
                )
            )
            if not schema_related:
                raise

            print(
                "Groq JSON Schema transport was rejected/not satisfied; "
                "retrying the same batch in JSON Object mode."
            )
            schema_fallback_used = True
            response = make_call(json_object_response_format)

    elapsed = time.perf_counter() - started
    finish_reason = _groq_finish_reason(response)

    try:
        response_text = str(response.choices[0].message.content or "")
    except Exception as exc:
        raise GroqMalformedJSONError(
            "Groq returned no readable assistant content.",
            response_text="",
            finish_reason=finish_reason,
        ) from exc

    record_model_response_usage(
        response,
        preflight=preflight,
        call_kind="quiz_generation",
    )

    if finish_reason in {"LENGTH", "MAX_TOKENS"}:
        raise GroqOutputTruncatedError(
            "Groq stopped because the completion token ceiling was reached. "
            "The affected batch will be retried with more output headroom or split."
        )

    top_level_array_normalized = False
    conservative_json_repair_used = False

    try:
        payload = parse_json_response(response_text)
    except (json.JSONDecodeError, ValueError) as exc:
        repaired = _conservative_json_repair(
            response_text,
            allow_top_level_array=force_json_object,
        )
        if repaired is not None:
            payload, top_level_array_normalized = repaired
            conservative_json_repair_used = True
        else:
            # In JSON Object mode Groq should return an object, but preserve the
            # old provider-shape tolerance for a clearly valid question array.
            if force_json_object:
                value_text = str(response_text or "").strip()
                raw_value = None
                try:
                    raw_value = json.loads(value_text)
                except json.JSONDecodeError:
                    fenced_array = re.search(
                        r"```(?:json)?\s*(\[.*\])\s*```",
                        value_text,
                        flags=(re.DOTALL | re.IGNORECASE),
                    )
                    if fenced_array:
                        raw_value = json.loads(fenced_array.group(1))

                if (
                    isinstance(raw_value, list)
                    and all(isinstance(item, dict) for item in raw_value)
                ):
                    payload = {"questions": raw_value}
                    top_level_array_normalized = True
                else:
                    raise GroqMalformedJSONError(
                        "Groq returned JSON that could not be parsed.",
                        response_text=response_text,
                        finish_reason=finish_reason,
                    ) from exc
            else:
                raise GroqMalformedJSONError(
                    "Groq returned JSON that could not be parsed.",
                    response_text=response_text,
                    finish_reason=finish_reason,
                ) from exc

    usage = getattr(response, "usage", None)

    def usage_value(name: str) -> int | None:
        if usage is None:
            return None
        value = getattr(usage, name, None)
        try:
            return int(value) if value is not None else None
        except (TypeError, ValueError):
            return None

    reasoning_tokens = None
    if usage is not None:
        details = getattr(usage, "completion_tokens_details", None)
        if details is not None:
            value = getattr(details, "reasoning_tokens", None)
            try:
                reasoning_tokens = int(value) if value is not None else None
            except (TypeError, ValueError):
                reasoning_tokens = None

    return (
        payload,
        {
            "estimated_prompt_tokens": estimated_prompt_tokens,
            "max_output_tokens": max_tokens,
            "structured_schema_fallback_used": schema_fallback_used,
            "response_schema_used": bool(
                not force_json_object and not schema_fallback_used
            ),
            "json_object_forced": bool(force_json_object),
            "top_level_array_normalized": bool(top_level_array_normalized),
            "conservative_json_repair_used": bool(conservative_json_repair_used),
            "finish_reason": finish_reason,
            "prompt_token_count": usage_value("prompt_tokens"),
            "candidate_token_count": usage_value("completion_tokens"),
            "reasoning_token_count": reasoning_tokens,
            "total_token_count": usage_value("total_tokens"),
            "latency_seconds": round(elapsed, 4),
        },
    )


def generate_with_groq(
    request_payload: dict[str, Any],
    validation_feedback: list[str] | None = None,
) -> dict[str, Any]:

    try:
        from groq import Groq

    except ImportError as exc:
        raise RuntimeError(
            "Install the Groq SDK first: pip install -U groq"
        ) from exc

    api_key = str(
        os.getenv(
            "GROQ_API_KEY"
        )
        or ""
    ).strip()

    if not api_key:
        raise RuntimeError(
            "GROQ_API_KEY is not set in Agent2/.env."
        )

    client = Groq(
        api_key=api_key
    )

    initial_batches = build_generation_batches(
        request_payload,
        validation_feedback=(
            validation_feedback
        ),
    )

    if not initial_batches:
        return {
            "questions":
                [],

            "_generation_metadata": {
                "provider":
                    GENERATION_PROVIDER,

                "model":
                    GENERATION_MODEL,

                "reasoning_effort":
                    GROQ_REASONING_EFFORT,

                "batch_count":
                    0,
            },
        }

    generated_questions = []

    call_metadata_rows = []

    instruction_batch_reports: list[
        dict[str, Any]
    ] = []

    def execute_blueprint_batch(
        blueprint_batch: list[
            dict[str, Any]
        ],
        *,
        depth: int = 0,
        forced_output_budget: int | None = None,
        strict_json_retry: bool = False,
    ) -> None:
        """
        Generate one batch.

        Transport recovery is deliberately bounded:
        1. true MAX_TOKENS truncation keeps the existing split/headroom logic;
        2. malformed JSON is treated separately;
        3. a malformed single-question response gets one strict JSON-object retry;
        4. repeated malformed output still fails closed.

        No quiz constraint is changed by these transport retries.
        """
        batch_request = (
            build_generation_batch_request(
                request_payload,
                blueprint_batch,
            )
        )

        targeted_transport = is_targeted_generation_request(
            batch_request
        )
        _set_active_model_call_audit_context(
            _generation_call_audit_context(
                batch_request,
                recovery_attempt=depth,
                transport_retry=(
                    "strict_transport_retry"
                    if strict_json_retry
                    else ""
                ),
            )
        )

        prompt = (
            build_targeted_repair_prompt(
                batch_request,
                validation_feedback=validation_feedback,
            )
            if targeted_transport
            else build_generation_prompt(
                batch_request,
                validation_feedback=(
                    validation_feedback
                ),
            )
        )
        system_prompt_for_call = (
            TARGETED_REPAIR_SYSTEM_PROMPT
            if targeted_transport
            else SYSTEM_PROMPT
        )

        messages = [
            {
                "role":
                    "system",

                "content":
                    system_prompt_for_call,
            },
            {
                "role":
                    "user",

                "content":
                    prompt,
            },
        ]

        if strict_json_retry:
            messages.append(
                {
                    "role":
                        "user",

                    "content":
                        (
                            "TRANSPORT RETRY ONLY. Return exactly one valid "
                            "JSON object matching the already supplied quiz "
                            "request. Do not use markdown or prose outside JSON. "
                            "Do not change the blueprint, marks, topics, paper "
                            "routing, visual requirements or question count. "
                            "Before returning, verify commas, quotes, braces and "
                            "brackets are syntactically complete."
                        ),
                }
            )

        normal_budget = (
            targeted_repair_output_token_budget(blueprint_batch)
            if targeted_transport
            else generation_output_token_budget(
                blueprint_batch
            )
        )

        output_budget = (
            int(
                forced_output_budget
            )
            if forced_output_budget
            is not None
            else normal_budget
        )

        output_budget = min(
            GROQ_MAX_GENERATION_OUTPUT_TOKENS,
            max(
                (
                    TARGETED_REPAIR_MIN_OUTPUT_TOKENS
                    if targeted_transport
                    else GROQ_MIN_GENERATION_OUTPUT_TOKENS
                ),
                output_budget,
            ),
        )

        plan_indexes = [
            safe_int(
                item.get(
                    "plan_index"
                )
            )
            for item in blueprint_batch
        ]

        print(
            "Groq GPT-OSS 120B generation batch",
            plan_indexes,
            "- questions:",
            len(
                blueprint_batch
            ),
            "- max output tokens:",
            output_budget,
        )

        try:
            payload, call_metadata = (
                _groq_json_call(
                    client,
                    model=GENERATION_MODEL,
                    messages=messages,
                    max_tokens=output_budget,
                    response_schema=(
                        generation_response_schema(
                            len(
                                blueprint_batch
                            )
                        )
                    ),
                    force_json_object=(
                        strict_json_retry
                    ),
                )
            )

            call_metadata[
                "strict_json_retry"
            ] = bool(
                strict_json_retry
            )

        except ModelContextWindowExceededError:
            # User-requested fail-fast behaviour: context-window overflow stops
            # locally before any Groq request is sent.
            raise

        except ModelProviderTokenBudgetExceededError as preflight_exc:
            if len(blueprint_batch) > 1:
                midpoint = max(1, len(blueprint_batch) // 2)
                left = blueprint_batch[:midpoint]
                right = blueprint_batch[midpoint:]

                print(
                    "Groq token preflight blocked batch; splitting locally BEFORE API:",
                    plan_indexes,
                    "->",
                    [safe_int(item.get("plan_index")) for item in left],
                    "and",
                    [safe_int(item.get("plan_index")) for item in right],
                )

                execute_blueprint_batch(left, depth=depth + 1)
                execute_blueprint_batch(right, depth=depth + 1)
                return

            safe_output_budget = provider_safe_output_budget_for_messages(
                messages,
                desired_output_tokens=output_budget,
            )

            if 768 <= safe_output_budget < output_budget:
                print(
                    "Single-question Groq request would exceed configured TPM. "
                    "Reducing only the reserved output budget locally BEFORE API to:",
                    safe_output_budget,
                )
                execute_blueprint_batch(
                    blueprint_batch,
                    depth=depth + 1,
                    forced_output_budget=safe_output_budget,
                    strict_json_retry=strict_json_retry,
                )
                return

            raise preflight_exc

        except GroqOutputTruncatedError:

            if len(
                blueprint_batch
            ) > 1:
                midpoint = max(
                    1,
                    len(
                        blueprint_batch
                    )
                    // 2,
                )

                left = blueprint_batch[
                    :midpoint
                ]

                right = blueprint_batch[
                    midpoint:
                ]

                print(
                    "Groq JSON was truncated; splitting batch",
                    plan_indexes,
                    "into",
                    [
                        safe_int(
                            item.get(
                                "plan_index"
                            )
                        )
                        for item in left
                    ],
                    "and",
                    [
                        safe_int(
                            item.get(
                                "plan_index"
                            )
                        )
                        for item in right
                    ],
                )

                execute_blueprint_batch(
                    left,
                    depth=(
                        depth
                        + 1
                    ),
                )

                execute_blueprint_batch(
                    right,
                    depth=(
                        depth
                        + 1
                    ),
                )

                return

            if (
                output_budget
                < GROQ_MAX_GENERATION_OUTPUT_TOKENS
            ):
                expanded_budget = min(
                    GROQ_MAX_GENERATION_OUTPUT_TOKENS,
                    max(
                        output_budget
                        + 4096,
                        output_budget
                        * 2,
                    ),
                )

                print(
                    "Groq single-question JSON was truncated; "
                    "retrying with max output tokens:",
                    expanded_budget,
                )

                execute_blueprint_batch(
                    blueprint_batch,
                    depth=(
                        depth
                        + 1
                    ),
                    forced_output_budget=(
                        expanded_budget
                    ),
                    strict_json_retry=(
                        strict_json_retry
                    ),
                )

                return

            raise

        except GroqMalformedJSONError as malformed_exc:
            if len(
                blueprint_batch
            ) > 1:
                midpoint = max(
                    1,
                    len(
                        blueprint_batch
                    )
                    // 2,
                )

                left = blueprint_batch[
                    :midpoint
                ]

                right = blueprint_batch[
                    midpoint:
                ]

                print(
                    "Groq returned malformed JSON; splitting batch",
                    plan_indexes,
                    "into",
                    [
                        safe_int(
                            item.get(
                                "plan_index"
                            )
                        )
                        for item in left
                    ],
                    "and",
                    [
                        safe_int(
                            item.get(
                                "plan_index"
                            )
                        )
                        for item in right
                    ],
                )

                execute_blueprint_batch(
                    left,
                    depth=(
                        depth
                        + 1
                    ),
                )

                execute_blueprint_batch(
                    right,
                    depth=(
                        depth
                        + 1
                    ),
                )

                return

            if not strict_json_retry:
                print(
                    "Groq single-question response contained malformed JSON; "
                    "retrying that same blueprint once in strict JSON-object mode."
                )

                execute_blueprint_batch(
                    blueprint_batch,
                    depth=(
                        depth
                        + 1
                    ),
                    forced_output_budget=(
                        output_budget
                    ),
                    strict_json_retry=True,
                )

                return

            raise GroqMalformedJSONError(
                "Groq returned malformed JSON again after the bounded "
                "single-question strict JSON-object retry. Failing closed "
                "instead of accepting invalid quiz data.",
                response_text=(
                    malformed_exc.response_text
                ),
                finish_reason=(
                    malformed_exc.finish_reason
                ),
            ) from malformed_exc

        batch_questions = payload.get(
            "questions",
            [],
        )

        if not isinstance(
            batch_questions,
            list,
        ):
            raise ValueError(
                "Groq returned questions in an invalid format."
            )

        # A deliberately shallow provider response schema can be accepted by the
        # provider yet still collapse array items to incomplete/empty objects.
        # That produces the frontend symptom "0 marks / only a few questions"
        # even though the generation request asked for the full blueprint.
        #
        # Keep the existing schema path, but if its returned batch is visibly
        # incomplete, retry ONLY that same batch once in JSON Object mode.
        # All deterministic structural/semantic/HITL validation below remains
        # unchanged and authoritative.
        schema_batch_incomplete = bool(
            call_metadata.get(
                "response_schema_used"
            )
            and (
                len(
                    batch_questions
                )
                != len(
                    blueprint_batch
                )
                or any(
                    (
                        not isinstance(
                            question,
                            dict,
                        )
                        or not str(
                            question.get(
                                "question_text",
                                "",
                            )
                            or ""
                        ).strip()
                        or safe_int(
                            question.get(
                                "marks"
                            )
                        )
                        <= 0
                    )
                    for question in batch_questions
                )
            )
        )

        if schema_batch_incomplete:
            print(
                "Groq structured-output schema returned an incomplete "
                "question batch; retrying the same blueprint batch once with "
                "JSON Object mode."
            )

            try:
                payload, json_object_metadata = (
                    _groq_json_call(
                        client,
                        model=GENERATION_MODEL,
                        messages=messages,
                        max_tokens=output_budget,
                        response_schema=(
                            generation_response_schema(
                                len(
                                    blueprint_batch
                                )
                            )
                        ),
                        force_json_object=True,
                    )
                )

            except GroqOutputTruncatedError:
                # The schema-integrity fallback is still a Groq generation
                # call, so it must use the same adaptive truncation handling as
                # the primary structured-output call. No quiz constraint is
                # changed by this transport retry.
                if len(
                    blueprint_batch
                ) > 1:
                    midpoint = max(
                        1,
                        len(
                            blueprint_batch
                        )
                        // 2,
                    )

                    left = blueprint_batch[
                        :midpoint
                    ]

                    right = blueprint_batch[
                        midpoint:
                    ]

                    print(
                        "Groq JSON-object retry was truncated; splitting batch",
                        plan_indexes,
                        "into",
                        [
                            safe_int(
                                item.get(
                                    "plan_index"
                                )
                            )
                            for item in left
                        ],
                        "and",
                        [
                            safe_int(
                                item.get(
                                    "plan_index"
                                )
                            )
                            for item in right
                        ],
                    )

                    execute_blueprint_batch(
                        left,
                        depth=(
                            depth
                            + 1
                        ),
                    )

                    execute_blueprint_batch(
                        right,
                        depth=(
                            depth
                            + 1
                        ),
                    )

                    return

                if (
                    output_budget
                    < GROQ_MAX_GENERATION_OUTPUT_TOKENS
                ):
                    expanded_budget = min(
                        GROQ_MAX_GENERATION_OUTPUT_TOKENS,
                        max(
                            output_budget
                            + 4096,
                            output_budget
                            * 2,
                        ),
                    )

                    print(
                        "Groq single-question JSON-object retry was "
                        "truncated; retrying with max output tokens:",
                        expanded_budget,
                    )

                    execute_blueprint_batch(
                        blueprint_batch,
                        depth=(
                            depth
                            + 1
                        ),
                        forced_output_budget=(
                            expanded_budget
                        ),
                        strict_json_retry=True,
                    )

                    return

                raise

            except GroqMalformedJSONError as malformed_exc:
                if len(
                    blueprint_batch
                ) > 1:
                    midpoint = max(
                        1,
                        len(
                            blueprint_batch
                        )
                        // 2,
                    )

                    left = blueprint_batch[
                        :midpoint
                    ]

                    right = blueprint_batch[
                        midpoint:
                    ]

                    print(
                        "Groq JSON-object fallback returned malformed JSON; "
                        "splitting batch",
                        plan_indexes,
                        "into",
                        [
                            safe_int(
                                item.get(
                                    "plan_index"
                                )
                            )
                            for item in left
                        ],
                        "and",
                        [
                            safe_int(
                                item.get(
                                    "plan_index"
                                )
                            )
                            for item in right
                        ],
                    )

                    execute_blueprint_batch(
                        left,
                        depth=(
                            depth
                            + 1
                        ),
                    )

                    execute_blueprint_batch(
                        right,
                        depth=(
                            depth
                            + 1
                        ),
                    )

                    return

                print(
                    "Groq single-question JSON-object fallback returned "
                    "malformed JSON; retrying that same blueprint once with "
                    "the strict JSON transport reminder."
                )

                execute_blueprint_batch(
                    blueprint_batch,
                    depth=(
                        depth
                        + 1
                    ),
                    forced_output_budget=(
                        output_budget
                    ),
                    strict_json_retry=True,
                )

                return

            batch_questions = payload.get(
                "questions",
                [],
            )

            if not isinstance(
                batch_questions,
                list,
            ):
                raise ValueError(
                    "Groq returned questions in an invalid format."
                )

            json_object_metadata[
                "schema_integrity_retry_used"
            ] = True

            json_object_metadata[
                "strict_json_retry"
            ] = False

            call_metadata = (
                json_object_metadata
            )

        instruction_batch_reports.append(
            _instruction_report_from_payload(
                payload,
                plan_indexes,
            )
        )

        generated_questions.extend(
            batch_questions
        )

        parsed_question_objects = [
            question
            for question in batch_questions
            if isinstance(question, dict)
        ]

        parsed_plan_indexes = [
            safe_int(question.get("plan_index"))
            for question in parsed_question_objects
        ]

        call_metadata_rows.append(
            {
                "blueprint_plan_indexes":
                    plan_indexes,

                "question_count":
                    len(
                        blueprint_batch
                    ),

                "parsed_question_count":
                    len(parsed_question_objects),

                "parsed_plan_indexes":
                    parsed_plan_indexes,

                "batch_cardinality_match":
                    len(parsed_question_objects)
                    == len(blueprint_batch),

                "adaptive_split_depth":
                    depth,

                **call_metadata,
            }
        )

    for blueprint_batch in initial_batches:
        execute_blueprint_batch(
            blueprint_batch
        )

    generated_questions.sort(
        key=lambda item: safe_int(
            item.get(
                "plan_index"
            )
        )
        if isinstance(
            item,
            dict,
        )
        else 0
    )

    merged_instruction_report = (
        _merge_instruction_batch_reports(
            instruction_batch_reports
        )
    )

    return {
        "questions":
            generated_questions,

        "instruction_interpretation":
            merged_instruction_report[
                "instruction_interpretation"
            ],

        "special_instruction_compliance":
            merged_instruction_report[
                "special_instruction_compliance"
            ],

        "_special_instruction_batch_reports":
            merged_instruction_report[
                "_special_instruction_batch_reports"
            ],

        "_generation_metadata": {
            "provider":
                GENERATION_PROVIDER,

            "model":
                GENERATION_MODEL,

            "reasoning_effort":
                GROQ_REASONING_EFFORT,

            "initial_batch_count":
                len(
                    initial_batches
                ),

            "completed_api_call_count":
                len(
                    call_metadata_rows
                ),

            "batch_soft_token_limit":
                LLM_BATCH_SOFT_TOKEN_LIMIT,

            "batches":
                call_metadata_rows,
        },
    }

# ================================================================
# OPENAI / GPT-5 MINI PROVIDER ADAPTER
# ================================================================

class OpenAIOutputTruncatedError(RuntimeError):
    """Raised when OpenAI stops before a complete structured JSON response."""


class OpenAIMalformedJSONError(RuntimeError):
    """Raised when OpenAI finishes but the returned JSON is not parseable."""

    def __init__(
        self,
        message: str,
        *,
        response_text: str = "",
        finish_reason: str = "",
    ) -> None:
        super().__init__(message)
        self.response_text = str(response_text or "")
        self.finish_reason = str(finish_reason or "").strip().upper()


def _openai_status_code(exc: Exception) -> int | None:
    status_code = getattr(exc, "status_code", None)
    if status_code is None:
        status_code = getattr(exc, "code", None)
    try:
        return int(status_code) if status_code is not None else None
    except (TypeError, ValueError):
        return None


def _is_retryable_openai_transient(exc: Exception) -> bool:
    """Retry only temporary server-side OpenAI failures; do not hide 429 quota errors."""
    status = _openai_status_code(exc)
    if status in {500, 502, 503, 504}:
        return True
    text = str(exc).upper()
    return any(code in text for code in ("500", "502", "503", "504")) and (
        "SERVER" in text or "UNAVAILABLE" in text or "TIMEOUT" in text
    )


def _openai_chat_completion_with_retry(
    client: Any,
    **kwargs: Any,
) -> Any:
    """Execute one OpenAI chat-completion call with bounded 5xx backoff."""
    delay_seconds = OPENAI_TRANSIENT_INITIAL_BACKOFF_SECONDS

    for attempt in range(1, OPENAI_TRANSIENT_MAX_ATTEMPTS + 1):
        try:
            _record_model_api_hit(
                call_kind="quiz_generation",
            )
            return client.chat.completions.create(**kwargs)
        except Exception as exc:
            if not _is_retryable_openai_transient(exc):
                raise
            if attempt >= OPENAI_TRANSIENT_MAX_ATTEMPTS:
                print(
                    "OpenAI transient server error persisted after "
                    f"{attempt} attempts; propagating the final service error."
                )
                raise
            print(
                "OpenAI transient server error; retrying attempt "
                f"{attempt + 1}/{OPENAI_TRANSIENT_MAX_ATTEMPTS} after "
                f"{delay_seconds:g}s."
            )
            if delay_seconds > 0:
                time.sleep(delay_seconds)
            delay_seconds = min(
                OPENAI_TRANSIENT_MAX_BACKOFF_SECONDS,
                max(
                    OPENAI_TRANSIENT_INITIAL_BACKOFF_SECONDS,
                    delay_seconds * 2,
                ),
            )


def _openai_finish_reason(response: Any) -> str:
    try:
        reason = response.choices[0].finish_reason
    except Exception:
        return ""
    return str(reason or "").strip().upper()


def _openai_json_call(
    client: Any,
    *,
    model: str,
    messages: list[dict[str, Any]],
    max_tokens: int,
    response_schema: dict[str, Any],
    force_json_object: bool = False,
) -> tuple[dict[str, Any], dict[str, Any]]:
    """
    Execute one OpenAI GPT-5 mini JSON call.

    Primary transport uses OpenAI JSON Schema mode (best-effort shallow schema)
    because Notebook 06's deterministic validators enforce the full contract.
    If the provider rejects/cannot satisfy that transport schema, retry the same
    request in JSON Object mode. A LENGTH finish reason is treated as truncation.
    """
    if not messages:
        raise ValueError("OpenAI request has no messages.")

    estimated_prompt_tokens = estimate_messages_tokens(messages)

    preflight = model_context_preflight(
        messages,
        max_output_tokens=max_tokens,
        call_kind="quiz_generation",
    )

    started = time.perf_counter()
    schema_fallback_used = False

    schema_response_format = {
        "type": "json_schema",
        "json_schema": {
            "name": "agent2_quiz_batch",
            "strict": False,
            "schema": response_schema,
        },
    }
    json_object_response_format = {"type": "json_object"}

    def make_call(response_format: dict[str, Any]) -> Any:
        return _openai_chat_completion_with_retry(
            client,
            model=model,
            messages=messages,
            max_completion_tokens=max_tokens,
            reasoning_effort=OPENAI_REASONING_EFFORT,
            response_format=response_format,
        )

    if force_json_object:
        response = make_call(json_object_response_format)
    else:
        try:
            response = make_call(schema_response_format)
        except Exception as schema_exc:
            status = _openai_status_code(schema_exc)
            text = str(schema_exc).casefold()
            schema_related = (
                status == 400
                and any(
                    token in text
                    for token in (
                        "schema",
                        "response_format",
                        "structured output",
                        "generated json",
                        "json does not match",
                    )
                )
            )
            if not schema_related:
                raise

            print(
                "OpenAI JSON Schema transport was rejected/not satisfied; "
                "retrying the same batch in JSON Object mode."
            )
            schema_fallback_used = True
            response = make_call(json_object_response_format)

    elapsed = time.perf_counter() - started
    finish_reason = _openai_finish_reason(response)

    try:
        response_text = str(response.choices[0].message.content or "")
    except Exception as exc:
        raise OpenAIMalformedJSONError(
            "OpenAI returned no readable assistant content.",
            response_text="",
            finish_reason=finish_reason,
        ) from exc

    record_model_response_usage(
        response,
        preflight=preflight,
        call_kind="quiz_generation",
    )

    if finish_reason in {"LENGTH", "MAX_TOKENS"}:
        raise OpenAIOutputTruncatedError(
            "OpenAI stopped because the completion token ceiling was reached. "
            "The affected batch will be retried with more output headroom or split."
        )

    top_level_array_normalized = False
    conservative_json_repair_used = False

    try:
        payload = parse_json_response(response_text)
    except (json.JSONDecodeError, ValueError) as exc:
        repaired = _conservative_json_repair(
            response_text,
            allow_top_level_array=force_json_object,
        )
        if repaired is not None:
            payload, top_level_array_normalized = repaired
            conservative_json_repair_used = True
        else:
            # In JSON Object mode OpenAI should return an object, but preserve the
            # old provider-shape tolerance for a clearly valid question array.
            if force_json_object:
                value_text = str(response_text or "").strip()
                raw_value = None
                try:
                    raw_value = json.loads(value_text)
                except json.JSONDecodeError:
                    fenced_array = re.search(
                        r"```(?:json)?\s*(\[.*\])\s*```",
                        value_text,
                        flags=(re.DOTALL | re.IGNORECASE),
                    )
                    if fenced_array:
                        raw_value = json.loads(fenced_array.group(1))

                if (
                    isinstance(raw_value, list)
                    and all(isinstance(item, dict) for item in raw_value)
                ):
                    payload = {"questions": raw_value}
                    top_level_array_normalized = True
                else:
                    raise OpenAIMalformedJSONError(
                        "OpenAI returned JSON that could not be parsed.",
                        response_text=response_text,
                        finish_reason=finish_reason,
                    ) from exc
            else:
                raise OpenAIMalformedJSONError(
                    "OpenAI returned JSON that could not be parsed.",
                    response_text=response_text,
                    finish_reason=finish_reason,
                ) from exc

    usage = getattr(response, "usage", None)

    def usage_value(name: str) -> int | None:
        if usage is None:
            return None
        value = getattr(usage, name, None)
        try:
            return int(value) if value is not None else None
        except (TypeError, ValueError):
            return None

    reasoning_tokens = None
    if usage is not None:
        details = getattr(usage, "completion_tokens_details", None)
        if details is not None:
            value = getattr(details, "reasoning_tokens", None)
            try:
                reasoning_tokens = int(value) if value is not None else None
            except (TypeError, ValueError):
                reasoning_tokens = None

    return (
        payload,
        {
            "estimated_prompt_tokens": estimated_prompt_tokens,
            "max_output_tokens": max_tokens,
            "structured_schema_fallback_used": schema_fallback_used,
            "response_schema_used": bool(
                not force_json_object and not schema_fallback_used
            ),
            "json_object_forced": bool(force_json_object),
            "top_level_array_normalized": bool(top_level_array_normalized),
            "conservative_json_repair_used": bool(conservative_json_repair_used),
            "finish_reason": finish_reason,
            "prompt_token_count": usage_value("prompt_tokens"),
            "candidate_token_count": usage_value("completion_tokens"),
            "reasoning_token_count": reasoning_tokens,
            "total_token_count": usage_value("total_tokens"),
            "latency_seconds": round(elapsed, 4),
        },
    )


def generate_with_openai(
    request_payload: dict[str, Any],
    validation_feedback: list[str] | None = None,
) -> dict[str, Any]:

    try:
        from openai import OpenAI

    except ImportError as exc:
        raise RuntimeError(
            "Install the OpenAI SDK first: pip install -U openai"
        ) from exc

    api_key = str(
        os.getenv(
            "OPENAI_API_KEY"
        )
        or ""
    ).strip()

    if not api_key:
        raise RuntimeError(
            "OPENAI_API_KEY is not set in Agent2/.env."
        )

    client = OpenAI(
        api_key=api_key
    )

    initial_batches = build_generation_batches(
        request_payload,
        validation_feedback=(
            validation_feedback
        ),
    )

    if not initial_batches:
        return {
            "questions":
                [],

            "_generation_metadata": {
                "provider":
                    GENERATION_PROVIDER,

                "model":
                    GENERATION_MODEL,

                "reasoning_effort":
                    OPENAI_REASONING_EFFORT,

                "batch_count":
                    0,
            },
        }

    generated_questions = []

    call_metadata_rows = []

    instruction_batch_reports: list[
        dict[str, Any]
    ] = []

    def execute_blueprint_batch(
        blueprint_batch: list[
            dict[str, Any]
        ],
        *,
        depth: int = 0,
        forced_output_budget: int | None = None,
        strict_json_retry: bool = False,
    ) -> None:
        """
        Generate one batch.

        Transport recovery is deliberately bounded:
        1. true MAX_TOKENS truncation keeps the existing split/headroom logic;
        2. malformed JSON is treated separately;
        3. a malformed single-question response gets one strict JSON-object retry;
        4. repeated malformed output still fails closed.

        No quiz constraint is changed by these transport retries.
        """
        batch_request = (
            build_generation_batch_request(
                request_payload,
                blueprint_batch,
            )
        )

        targeted_transport = is_targeted_generation_request(
            batch_request
        )
        _set_active_model_call_audit_context(
            _generation_call_audit_context(
                batch_request,
                recovery_attempt=depth,
                transport_retry=(
                    "strict_transport_retry"
                    if strict_json_retry
                    else ""
                ),
            )
        )

        prompt = (
            build_targeted_repair_prompt(
                batch_request,
                validation_feedback=validation_feedback,
            )
            if targeted_transport
            else build_generation_prompt(
                batch_request,
                validation_feedback=(
                    validation_feedback
                ),
            )
        )
        system_prompt_for_call = (
            TARGETED_REPAIR_SYSTEM_PROMPT
            if targeted_transport
            else SYSTEM_PROMPT
        )

        messages = [
            {
                "role":
                    "system",

                "content":
                    system_prompt_for_call,
            },
            {
                "role":
                    "user",

                "content":
                    prompt,
            },
        ]

        if strict_json_retry:
            messages.append(
                {
                    "role":
                        "user",

                    "content":
                        (
                            "TRANSPORT RETRY ONLY. Return exactly one valid "
                            "JSON object matching the already supplied quiz "
                            "request. Do not use markdown or prose outside JSON. "
                            "Do not change the blueprint, marks, topics, paper "
                            "routing, visual requirements or question count. "
                            "Before returning, verify commas, quotes, braces and "
                            "brackets are syntactically complete."
                        ),
                }
            )

        normal_budget = (
            targeted_repair_output_token_budget(blueprint_batch)
            if targeted_transport
            else generation_output_token_budget(
                blueprint_batch
            )
        )

        output_budget = (
            int(
                forced_output_budget
            )
            if forced_output_budget
            is not None
            else normal_budget
        )

        output_budget = min(
            OPENAI_MAX_GENERATION_OUTPUT_TOKENS,
            max(
                (
                    TARGETED_REPAIR_MIN_OUTPUT_TOKENS
                    if targeted_transport
                    else OPENAI_MIN_GENERATION_OUTPUT_TOKENS
                ),
                output_budget,
            ),
        )

        plan_indexes = [
            safe_int(
                item.get(
                    "plan_index"
                )
            )
            for item in blueprint_batch
        ]

        print(
            "OpenAI GPT-5 mini generation batch",
            plan_indexes,
            "- questions:",
            len(
                blueprint_batch
            ),
            "- max output tokens:",
            output_budget,
        )

        try:
            payload, call_metadata = (
                _openai_json_call(
                    client,
                    model=GENERATION_MODEL,
                    messages=messages,
                    max_tokens=output_budget,
                    response_schema=(
                        generation_response_schema(
                            len(
                                blueprint_batch
                            )
                        )
                    ),
                    force_json_object=(
                        strict_json_retry
                    ),
                )
            )

            call_metadata[
                "strict_json_retry"
            ] = bool(
                strict_json_retry
            )

        except ModelContextWindowExceededError:
            # User-requested fail-fast behaviour: context-window overflow stops
            # locally before any OpenAI request is sent.
            raise

        except ModelProviderTokenBudgetExceededError as preflight_exc:
            if len(blueprint_batch) > 1:
                midpoint = max(1, len(blueprint_batch) // 2)
                left = blueprint_batch[:midpoint]
                right = blueprint_batch[midpoint:]

                print(
                    "OpenAI token preflight blocked batch; splitting locally BEFORE API:",
                    plan_indexes,
                    "->",
                    [safe_int(item.get("plan_index")) for item in left],
                    "and",
                    [safe_int(item.get("plan_index")) for item in right],
                )

                execute_blueprint_batch(left, depth=depth + 1)
                execute_blueprint_batch(right, depth=depth + 1)
                return

            safe_output_budget = provider_safe_output_budget_for_messages(
                messages,
                desired_output_tokens=output_budget,
            )

            if 768 <= safe_output_budget < output_budget:
                print(
                    "Single-question OpenAI request would exceed configured TPM. "
                    "Reducing only the reserved output budget locally BEFORE API to:",
                    safe_output_budget,
                )
                execute_blueprint_batch(
                    blueprint_batch,
                    depth=depth + 1,
                    forced_output_budget=safe_output_budget,
                    strict_json_retry=strict_json_retry,
                )
                return

            raise preflight_exc

        except OpenAIOutputTruncatedError:

            # Call-efficient recovery: first give the SAME planned batch one
            # bounded increase in completion headroom. The old path immediately
            # split any multi-question batch, turning one truncation into two
            # additional provider calls. Only split after the batch has already
            # reached the configured Notebook 06 completion ceiling.
            if output_budget < OPENAI_MAX_GENERATION_OUTPUT_TOKENS:
                expanded_budget = min(
                    OPENAI_MAX_GENERATION_OUTPUT_TOKENS,
                    max(
                        output_budget + 3072,
                        int(math.ceil(output_budget * 1.20)),
                    ),
                )

                print(
                    "OpenAI JSON was truncated; retrying the same planned batch "
                    "once with additional completion headroom:",
                    expanded_budget,
                )

                execute_blueprint_batch(
                    blueprint_batch,
                    depth=depth + 1,
                    forced_output_budget=expanded_budget,
                    strict_json_retry=strict_json_retry,
                )
                return

            if len(blueprint_batch) > 1:
                midpoint = max(1, len(blueprint_batch) // 2)
                left = blueprint_batch[:midpoint]
                right = blueprint_batch[midpoint:]

                print(
                    "OpenAI batch still truncated at the configured completion "
                    "ceiling; splitting only now:",
                    plan_indexes,
                    "->",
                    [safe_int(item.get("plan_index")) for item in left],
                    "and",
                    [safe_int(item.get("plan_index")) for item in right],
                )
                execute_blueprint_batch(left, depth=depth + 1)
                execute_blueprint_batch(right, depth=depth + 1)
                return

            raise

        except OpenAIMalformedJSONError as malformed_exc:
            if len(
                blueprint_batch
            ) > 1:
                midpoint = max(
                    1,
                    len(
                        blueprint_batch
                    )
                    // 2,
                )

                left = blueprint_batch[
                    :midpoint
                ]

                right = blueprint_batch[
                    midpoint:
                ]

                print(
                    "OpenAI returned malformed JSON; splitting batch",
                    plan_indexes,
                    "into",
                    [
                        safe_int(
                            item.get(
                                "plan_index"
                            )
                        )
                        for item in left
                    ],
                    "and",
                    [
                        safe_int(
                            item.get(
                                "plan_index"
                            )
                        )
                        for item in right
                    ],
                )

                execute_blueprint_batch(
                    left,
                    depth=(
                        depth
                        + 1
                    ),
                )

                execute_blueprint_batch(
                    right,
                    depth=(
                        depth
                        + 1
                    ),
                )

                return

            if not strict_json_retry:
                print(
                    "OpenAI single-question response contained malformed JSON; "
                    "retrying that same blueprint once in strict JSON-object mode."
                )

                execute_blueprint_batch(
                    blueprint_batch,
                    depth=(
                        depth
                        + 1
                    ),
                    forced_output_budget=(
                        output_budget
                    ),
                    strict_json_retry=True,
                )

                return

            raise OpenAIMalformedJSONError(
                "OpenAI returned malformed JSON again after the bounded "
                "single-question strict JSON-object retry. Failing closed "
                "instead of accepting invalid quiz data.",
                response_text=(
                    malformed_exc.response_text
                ),
                finish_reason=(
                    malformed_exc.finish_reason
                ),
            ) from malformed_exc

        batch_questions = payload.get(
            "questions",
            [],
        )

        if not isinstance(
            batch_questions,
            list,
        ):
            raise ValueError(
                "OpenAI returned questions in an invalid format."
            )

        # A deliberately shallow provider response schema can be accepted by the
        # provider yet still collapse array items to incomplete/empty objects.
        # That produces the frontend symptom "0 marks / only a few questions"
        # even though the generation request asked for the full blueprint.
        #
        # Keep the existing schema path, but if its returned batch is visibly
        # incomplete, retry ONLY that same batch once in JSON Object mode.
        # All deterministic structural/semantic/HITL validation below remains
        # unchanged and authoritative.
        schema_batch_incomplete = bool(
            call_metadata.get(
                "response_schema_used"
            )
            and (
                len(
                    batch_questions
                )
                != len(
                    blueprint_batch
                )
                or any(
                    (
                        not isinstance(
                            question,
                            dict,
                        )
                        or not str(
                            question.get(
                                "question_text",
                                "",
                            )
                            or ""
                        ).strip()
                        or safe_int(
                            question.get(
                                "marks"
                            )
                        )
                        <= 0
                    )
                    for question in batch_questions
                )
            )
        )

        if schema_batch_incomplete:
            print(
                "OpenAI structured-output schema returned an incomplete "
                "question batch; retrying the same blueprint batch once with "
                "JSON Object mode."
            )

            try:
                payload, json_object_metadata = (
                    _openai_json_call(
                        client,
                        model=GENERATION_MODEL,
                        messages=messages,
                        max_tokens=output_budget,
                        response_schema=(
                            generation_response_schema(
                                len(
                                    blueprint_batch
                                )
                            )
                        ),
                        force_json_object=True,
                    )
                )

            except OpenAIOutputTruncatedError:
                # The schema-integrity fallback is still a OpenAI generation
                # call, so it must use the same adaptive truncation handling as
                # the primary structured-output call. No quiz constraint is
                # changed by this transport retry.
                if len(
                    blueprint_batch
                ) > 1:
                    midpoint = max(
                        1,
                        len(
                            blueprint_batch
                        )
                        // 2,
                    )

                    left = blueprint_batch[
                        :midpoint
                    ]

                    right = blueprint_batch[
                        midpoint:
                    ]

                    print(
                        "OpenAI JSON-object retry was truncated; splitting batch",
                        plan_indexes,
                        "into",
                        [
                            safe_int(
                                item.get(
                                    "plan_index"
                                )
                            )
                            for item in left
                        ],
                        "and",
                        [
                            safe_int(
                                item.get(
                                    "plan_index"
                                )
                            )
                            for item in right
                        ],
                    )

                    execute_blueprint_batch(
                        left,
                        depth=(
                            depth
                            + 1
                        ),
                    )

                    execute_blueprint_batch(
                        right,
                        depth=(
                            depth
                            + 1
                        ),
                    )

                    return

                if (
                    output_budget
                    < OPENAI_MAX_GENERATION_OUTPUT_TOKENS
                ):
                    expanded_budget = min(
                        OPENAI_MAX_GENERATION_OUTPUT_TOKENS,
                        max(
                            output_budget
                            + 4096,
                            output_budget
                            * 2,
                        ),
                    )

                    print(
                        "OpenAI single-question JSON-object retry was "
                        "truncated; retrying with max output tokens:",
                        expanded_budget,
                    )

                    execute_blueprint_batch(
                        blueprint_batch,
                        depth=(
                            depth
                            + 1
                        ),
                        forced_output_budget=(
                            expanded_budget
                        ),
                        strict_json_retry=True,
                    )

                    return

                raise

            except OpenAIMalformedJSONError as malformed_exc:
                if len(
                    blueprint_batch
                ) > 1:
                    midpoint = max(
                        1,
                        len(
                            blueprint_batch
                        )
                        // 2,
                    )

                    left = blueprint_batch[
                        :midpoint
                    ]

                    right = blueprint_batch[
                        midpoint:
                    ]

                    print(
                        "OpenAI JSON-object fallback returned malformed JSON; "
                        "splitting batch",
                        plan_indexes,
                        "into",
                        [
                            safe_int(
                                item.get(
                                    "plan_index"
                                )
                            )
                            for item in left
                        ],
                        "and",
                        [
                            safe_int(
                                item.get(
                                    "plan_index"
                                )
                            )
                            for item in right
                        ],
                    )

                    execute_blueprint_batch(
                        left,
                        depth=(
                            depth
                            + 1
                        ),
                    )

                    execute_blueprint_batch(
                        right,
                        depth=(
                            depth
                            + 1
                        ),
                    )

                    return

                print(
                    "OpenAI single-question JSON-object fallback returned "
                    "malformed JSON; retrying that same blueprint once with "
                    "the strict JSON transport reminder."
                )

                execute_blueprint_batch(
                    blueprint_batch,
                    depth=(
                        depth
                        + 1
                    ),
                    forced_output_budget=(
                        output_budget
                    ),
                    strict_json_retry=True,
                )

                return

            batch_questions = payload.get(
                "questions",
                [],
            )

            if not isinstance(
                batch_questions,
                list,
            ):
                raise ValueError(
                    "OpenAI returned questions in an invalid format."
                )

            json_object_metadata[
                "schema_integrity_retry_used"
            ] = True

            json_object_metadata[
                "strict_json_retry"
            ] = False

            call_metadata = (
                json_object_metadata
            )

        instruction_batch_reports.append(
            _instruction_report_from_payload(
                payload,
                plan_indexes,
            )
        )

        generated_questions.extend(
            batch_questions
        )

        parsed_question_objects = [
            question
            for question in batch_questions
            if isinstance(question, dict)
        ]

        parsed_plan_indexes = [
            safe_int(question.get("plan_index"))
            for question in parsed_question_objects
        ]

        call_metadata_rows.append(
            {
                "blueprint_plan_indexes":
                    plan_indexes,

                "question_count":
                    len(
                        blueprint_batch
                    ),

                "parsed_question_count":
                    len(parsed_question_objects),

                "parsed_plan_indexes":
                    parsed_plan_indexes,

                "batch_cardinality_match":
                    len(parsed_question_objects)
                    == len(blueprint_batch),

                "adaptive_split_depth":
                    depth,

                **call_metadata,
            }
        )

    for blueprint_batch in initial_batches:
        execute_blueprint_batch(
            blueprint_batch
        )

    generated_questions.sort(
        key=lambda item: safe_int(
            item.get(
                "plan_index"
            )
        )
        if isinstance(
            item,
            dict,
        )
        else 0
    )

    merged_instruction_report = (
        _merge_instruction_batch_reports(
            instruction_batch_reports
        )
    )

    return {
        "questions":
            generated_questions,

        "instruction_interpretation":
            merged_instruction_report[
                "instruction_interpretation"
            ],

        "special_instruction_compliance":
            merged_instruction_report[
                "special_instruction_compliance"
            ],

        "_special_instruction_batch_reports":
            merged_instruction_report[
                "_special_instruction_batch_reports"
            ],

        "_generation_metadata": {
            "provider":
                GENERATION_PROVIDER,

            "model":
                GENERATION_MODEL,

            "reasoning_effort":
                OPENAI_REASONING_EFFORT,

            "initial_batch_count":
                len(
                    initial_batches
                ),

            "completed_api_call_count":
                len(
                    call_metadata_rows
                ),

            "batch_soft_token_limit":
                LLM_BATCH_SOFT_TOKEN_LIMIT,

            "batches":
                call_metadata_rows,
        },
    }

# ================================================================

# CONFIG-SELECTED GENERATION ROUTER
# ================================================================

def generate_with_selected_model(
    request_payload: dict[str, Any],
    validation_feedback: list[str] | None = None,
) -> dict[str, Any]:
    if GENERATION_PROVIDER == "google_gemini":
        return generate_with_gemini(
            request_payload,
            validation_feedback=validation_feedback,
        )

    if GENERATION_PROVIDER == "groq":
        return generate_with_groq(
            request_payload,
            validation_feedback=validation_feedback,
        )

    if GENERATION_PROVIDER == "openai":
        return generate_with_openai(
            request_payload,
            validation_feedback=validation_feedback,
        )

    raise RuntimeError(
        "Unsupported generation provider in quiz_model_config.json: "
        f"{GENERATION_PROVIDER!r}"
    )

## 9. Deterministic structural validation

In [ ]:
_QUESTION_COUNT_WORDS = {
    "one": 1,
    "two": 2,
    "three": 3,
    "four": 4,
    "five": 5,
    "six": 6,
    "seven": 7,
    "eight": 8,
    "nine": 9,
    "ten": 10,
    "eleven": 11,
    "twelve": 12,
    "thirteen": 13,
    "fourteen": 14,
    "fifteen": 15,
    "sixteen": 16,
    "seventeen": 17,
    "eighteen": 18,
    "nineteen": 19,
    "twenty": 20,
}


_RESPONSE_COUNT_CATEGORY_PATTERNS = {
    "reason": r"reasons?|justifications?",
    "advantage": r"advantages?",
    "disadvantage": r"disadvantages?",
    "benefit": r"benefits?",
    "drawback": r"drawbacks?",
    "difference": r"differences?",
    "similarity": r"similarities?",
    "example": r"examples?",
    "value": r"values?",
    "answer": r"answers?|responses?",
    "factor": r"factors?",
    "feature": r"features?",
    "step": r"steps?",
    "cause": r"causes?",
    "effect": r"effects?",
    "method": r"methods?",
    "way": r"ways?",
    "point": r"points?",
}


_CRITERION_CATEGORY_PATTERNS = {
    "reason": r"\b(?:reason|justification|because)\b",
    "advantage": r"\badvantage\b",
    "disadvantage": r"\bdisadvantage\b",
    "benefit": r"\bbenefit\b",
    "drawback": r"\bdrawback\b",
    "difference": r"\bdifference\b",
    "similarity": r"\bsimilarity\b",
    "example": r"\bexample\b",
    "value": r"\bvalue\b",
    "factor": r"\bfactor\b",
    "feature": r"\bfeature\b",
    "step": r"\bstep\b",
    "cause": r"\bcause\b",
    "effect": r"\beffect\b",
    "method": r"\bmethod\b",
    "way": r"\bway\b",
    "point": r"\bpoint\b",
}


def _question_count_token_to_int(value: Any) -> int | None:
    token = normalize_text(value)
    if token.isdigit():
        result = int(token)
        return result if result > 0 else None
    return _QUESTION_COUNT_WORDS.get(token)


def _explicit_response_count_constraints(
    question_text_value: Any,
) -> list[dict[str, Any]]:
    """
    Extract explicit learner-response counts such as "give two reasons",
    "five separate answers", or scoped forms such as:

        "For each of the three outputs, give one reason."

    Scoped forms are interpreted generically as:

        number of referenced items × responses required per item

    This deliberately ignores incidental numbers such as row/column sizes,
    loop bounds, marks and line numbers.
    """
    normalized = normalize_text(question_text_value)
    if not normalized:
        return []

    number_pattern = (
        r"(?:\d+|"
        + "|".join(
            sorted(
                _QUESTION_COUNT_WORDS,
                key=len,
                reverse=True,
            )
        )
        + r")"
    )
    category_pattern = (
        r"(?:"
        + "|".join(_RESPONSE_COUNT_CATEGORY_PATTERNS.values())
        + r")"
    )

    patterns = [
        # Command-led forms: "give two separate reasons".
        re.compile(
            rf"\b(?:give|state|write|list|name|identify|provide|describe|explain|record)\s+"
            rf"(?:exactly\s+)?(?P<count>{number_pattern})\s+"
            rf"(?:(?:separate|distinct|brief|valid|different|independent)\s+)*"
            rf"(?P<noun>{category_pattern})\b"
        ),
        # Embedded forms: "... and two separate brief reasons explaining ...".
        re.compile(
            rf"\b(?P<count>{number_pattern})\s+"
            rf"(?:(?:separate|distinct|brief|valid|different|independent)\s+)*"
            rf"(?P<noun>{category_pattern})\b"
        ),
    ]

    # Generic quantified "for each" scope, e.g.
    # "For each of those three printed lines, give one brief reason".
    #
    # We intentionally keep the item noun generic rather than topic-specific.
    each_scope_pattern = re.compile(
        rf"\bfor\s+each\s+of\s+"
        rf"(?:(?:the|these|those)\s+)?"
        rf"(?P<count>{number_pattern})\s+"
        rf"(?P<items>[a-z][a-z0-9_-]*(?:\s+[a-z][a-z0-9_-]*){{0,5}}?)"
        rf"\s*(?:,|:|;|\b(?:give|state|write|list|name|identify|provide|describe|explain|record)\b)",
        flags=re.IGNORECASE,
    )

    each_scopes: list[dict[str, Any]] = []
    for scope_match in each_scope_pattern.finditer(normalized):
        scope_count = _question_count_token_to_int(
            scope_match.group("count")
        )
        if scope_count is None or scope_count < 1:
            continue

        # The scope applies forward within the same instruction clause.
        clause_end_candidates = [
            position
            for token in (".", "\n")
            if (position := normalized.find(token, scope_match.end())) >= 0
        ]
        clause_end = min(clause_end_candidates) if clause_end_candidates else len(normalized)

        each_scopes.append(
            {
                "count": int(scope_count),
                "span": [scope_match.start(), clause_end],
                "text": scope_match.group(0).strip(),
            }
        )

    found: list[dict[str, Any]] = []
    seen: set[tuple[str, int, int, int]] = set()

    for pattern in patterns:
        for match in pattern.finditer(normalized):
            count = _question_count_token_to_int(match.group("count"))
            noun = normalize_text(match.group("noun"))
            if count is None:
                continue

            category = None
            for candidate, noun_pattern in _RESPONSE_COUNT_CATEGORY_PATTERNS.items():
                if re.fullmatch(noun_pattern, noun):
                    category = candidate
                    break

            if category is None:
                continue

            base_count = int(count)
            effective_count = base_count
            scope_multiplier = 1
            scope_text = ""

            # If this response instruction sits inside a quantified "for each"
            # clause, multiply responses-per-item by the number of referenced
            # items. Example: 3 lines × 1 reason each = 3 reasons.
            applicable_scopes = [
                scope
                for scope in each_scopes
                if int(scope["span"][0]) <= match.start() < int(scope["span"][1])
            ]

            if applicable_scopes:
                # Nearest enclosing/preceding scope wins.
                scope = max(
                    applicable_scopes,
                    key=lambda value: int(value["span"][0]),
                )
                scope_multiplier = int(scope["count"])
                effective_count = base_count * scope_multiplier
                scope_text = str(scope.get("text", "") or "")

            signature = (
                category,
                effective_count,
                match.start(),
                match.end(),
            )
            if signature in seen:
                continue
            seen.add(signature)

            found.append(
                {
                    "category": category,
                    "count": effective_count,
                    "base_count": base_count,
                    "scope_multiplier": scope_multiplier,
                    "scope_text": scope_text,
                    "text": match.group(0),
                    "span": [match.start(), match.end()],
                }
            )

    # Command-led and embedded regexes can capture the same phrase with
    # slightly different spans. De-duplicate by category/count and overlapping
    # source span, preserving the first strongest match.
    compact: list[dict[str, Any]] = []
    for item in sorted(
        found,
        key=lambda value: (
            value["span"][0],
            value["span"][1],
        ),
    ):
        duplicate = False
        for existing in compact:
            if (
                existing["category"] == item["category"]
                and existing["count"] == item["count"]
                and not (
                    item["span"][1] <= existing["span"][0]
                    or item["span"][0] >= existing["span"][1]
                )
            ):
                duplicate = True
                break

        if not duplicate:
            compact.append(item)

    return compact


def _question_marking_contract_checks(
    question_text_value: Any,
    marking_guidance: Any,
) -> dict[str, Any]:
    """
    Conservative question <-> marking-guidance contract check.

    Hard failures are raised only when the question explicitly requests N
    responses of a named type and the marking guidance itself clearly labels a
    conflicting number of criteria of that same type. Generic "answers" are
    compared against the total count of positive marking criteria.

    If category labels are absent from the mark scheme, the check does not guess
    semantic equivalence; HITL remains the qualitative safety net.
    """
    constraints = _explicit_response_count_constraints(question_text_value)
    guidance_items = [
        item
        for item in safe_list(marking_guidance)
        if isinstance(item, dict)
        and safe_int(item.get("marks")) > 0
        and str(item.get("criterion", "") or "").strip()
    ]

    errors: list[str] = []
    warnings: list[str] = []
    diagnostics: list[dict[str, Any]] = []

    for constraint in constraints:
        category = str(constraint["category"])
        expected_count = int(constraint["count"])

        if category == "answer":
            matched_count = len(guidance_items)
            enforce = True
        else:
            criterion_pattern = _CRITERION_CATEGORY_PATTERNS.get(category)
            matched_count = 0
            if criterion_pattern:
                for item in guidance_items:
                    criterion_text = normalize_text(item.get("criterion", ""))
                    if re.search(criterion_pattern, criterion_text):
                        matched_count += 1

            # A non-zero labelled count provides strong evidence that the mark
            # scheme is explicitly partitioning this response type. If there
            # are no labels, avoid guessing and leave it to HITL.
            enforce = matched_count > 0

        diagnostics.append(
            {
                "category": category,
                "requested_count": expected_count,
                "matched_marking_criteria": matched_count,
                "enforced": enforce,
                "source_text": constraint.get("text", ""),
                "base_count": constraint.get(
                    "base_count",
                    expected_count,
                ),
                "scope_multiplier": constraint.get(
                    "scope_multiplier",
                    1,
                ),
                "scope_text": constraint.get(
                    "scope_text",
                    "",
                ),
            }
        )

        if enforce and matched_count != expected_count:
            errors.append(
                "question/marking-guidance response-count mismatch: "
                f"question requests {expected_count} {category} response(s), "
                f"but marking guidance contains {matched_count} clearly "
                f"labelled {category} criterion/criteria"
            )

    # General numbered-code edit precision check. If the question asks which
    # existing line should be changed but the mark scheme actually requires an
    # insertion after a line, the learner instruction is ambiguous.
    q_norm = normalize_text(question_text_value)
    guidance_text = normalize_text(
        " ".join(
            str(item.get("criterion", "") or "")
            for item in guidance_items
        )
    )

    asks_change_line_number = bool(
        "line number" in q_norm
        and re.search(r"\b(?:change|changed|modify|modified|replace|replaced|correct)\b", q_norm)
    )
    answer_requires_insertion_location = bool(
        re.search(r"\b(?:insert|insertion|after line|line to add|add(?:ed)? after)\b", guidance_text)
    )
    question_already_asks_insertion_location = bool(
        re.search(
            r"\b(?:after which line|where should .* be inserted|where .* insert|insert .* after)\b",
            q_norm,
        )
    )

    if (
        asks_change_line_number
        and answer_requires_insertion_location
        and not question_already_asks_insertion_location
    ):
        errors.append(
            "numbered-code edit wording is ambiguous: the question asks for "
            "an existing line to change, but the marking guidance requires an "
            "insertion location. Ask where/after which line the new statement "
            "should be inserted."
        )

    return {
        "errors": list(dict.fromkeys(errors)),
        "warnings": list(dict.fromkeys(warnings)),
        "constraints": constraints,
        "diagnostics": diagnostics,
    }


_AQA_FORMAL_COMPLEXITY_PATTERNS = (
    r"\bbig\s*[- ]?o\b",
    r"\bo\s*\(\s*n(?:\s|\^|\*|\)|log)",
    r"\btime\s+complexity\b",
    r"\bspace\s+complexity\b",
    r"\bauxiliary\s+space\b",
    r"\basymptotic(?:\s+complexity|\s+notation)?\b",
)


def _aqa_formal_complexity_scope_errors(
    question_text_value: Any,
    marking_guidance: Any,
) -> list[str]:
    """
    AQA 8525 asks learners to consider/compare time efficiency but does not
    require formal asymptotic comparisons. Keep generated practice within that
    syllabus scope while still allowing concrete counts of comparisons, swaps,
    passes and other self-contained operations.
    """
    combined = " ".join(
        [str(question_text_value or "")]
        + [
            str(item.get("criterion", "") or "")
            for item in safe_list(marking_guidance)
            if isinstance(item, dict)
        ]
    ).casefold()

    matched = [
        pattern
        for pattern in _AQA_FORMAL_COMPLEXITY_PATTERNS
        if re.search(pattern, combined, flags=re.IGNORECASE)
    ]

    if not matched:
        return []

    return [
        "AQA syllabus-scope violation: formal Big-O/asymptotic time or space "
        "complexity is not required for GCSE 8525. Assess time efficiency "
        "qualitatively or through self-contained comparisons/passes/swaps/"
        "operations instead."
    ]


def validate_special_instruction_reporting(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:
    """
    Validate the SAME-CALL instruction interpretation/compliance report.

    The model's self-report is not treated as proof. Objective requirements are
    still independently enforced elsewhere in Python. This check only ensures
    that non-empty special instructions were explicitly interpreted and that
    the model did not admit to ignoring/conflicting with a mandatory request.
    Qualitative claims still require HITL confirmation.
    """
    errors: list[str] = []
    warnings: list[str] = []

    filters = request_payload.get(
        "assessment_filters",
        {},
    )
    if not isinstance(
        filters,
        dict,
    ):
        filters = {}

    raw_instruction = str(
        filters.get(
            "special_instructions",
            "",
        )
        or ""
    ).strip()

    interpretation = payload.get(
        "instruction_interpretation",
        {},
    )
    if not isinstance(
        interpretation,
        dict,
    ):
        interpretation = {}

    compliance = payload.get(
        "special_instruction_compliance",
        [],
    )
    if not isinstance(
        compliance,
        list,
    ):
        compliance = []

    status = str(
        interpretation.get(
            "status",
            "",
        )
        or ""
    ).strip().casefold()

    requirements = [
        item
        for item in safe_list(
            interpretation.get(
                "requirements",
                [],
            )
        )
        if isinstance(
            item,
            dict,
        )
    ]

    clean_compliance = [
        item
        for item in compliance
        if isinstance(
            item,
            dict,
        )
    ]

    if not raw_instruction:
        if status not in {
            "",
            "none",
        }:
            warnings.append(
                "Model returned a non-empty special-instruction interpretation "
                "even though the request contained no special instructions."
            )

        return {
            "valid":
                not errors,
            "errors":
                errors,
            "warnings":
                warnings,
            "status":
                status or "none",
            "requirements":
                requirements,
            "compliance":
                clean_compliance,
            "requires_hitl_confirmation":
                False,
        }

    if status not in {
        "applied",
        "conflict",
    }:
        errors.append(
            "Non-empty special_instructions were not explicitly interpreted in "
            "the same generation response."
        )

    if status == "conflict":
        errors.append(
            "The generation model reported that the user's special instructions "
            "conflict with the approved AQA scope or hard quiz controls."
        )

    if not requirements:
        errors.append(
            "Non-empty special_instructions produced no structured interpreted "
            "requirements."
        )

    compliance_by_id = {
        normalize_text(
            item.get(
                "requirement_id",
                "",
            )
        ): item
        for item in clean_compliance
        if normalize_text(
            item.get(
                "requirement_id",
                "",
            )
        )
    }

    for index, requirement in enumerate(
        requirements,
        start=1,
    ):
        requirement_id = str(
            requirement.get(
                "requirement_id",
                "",
            )
            or ""
        ).strip()

        instruction = str(
            requirement.get(
                "instruction",
                "",
            )
            or ""
        ).strip()

        priority = str(
            requirement.get(
                "priority",
                "mandatory",
            )
            or "mandatory"
        ).strip().casefold()

        key = normalize_text(
            requirement_id
        )

        if not requirement_id:
            errors.append(
                f"Special-instruction requirement {index} has no requirement_id."
            )
            continue

        if not instruction:
            errors.append(
                f"Special-instruction requirement {requirement_id} has no "
                "interpreted instruction text."
            )

        item = compliance_by_id.get(
            key
        )

        if item is None:
            errors.append(
                "Missing same-call compliance result for special-instruction "
                f"requirement {requirement_id}."
            )
            continue

        satisfied = item.get(
            "satisfied"
        )

        if not isinstance(
            satisfied,
            bool,
        ):
            errors.append(
                f"Special-instruction compliance for {requirement_id} must "
                "contain a boolean satisfied field."
            )
            continue

        if not satisfied:
            message = (
                f"Special instruction {requirement_id} was reported unsatisfied "
                "by the generation model."
            )

            if priority == "preference":
                warnings.append(
                    message
                )
            else:
                errors.append(
                    message
                )

        evidence = str(
            item.get(
                "evidence",
                "",
            )
            or ""
        ).strip()

        if not evidence:
            warnings.append(
                f"Special-instruction compliance for {requirement_id} has no "
                "evidence text for HITL."
            )

    return {
        "valid":
            not errors,
        "errors":
            list(
                dict.fromkeys(
                    errors
                )
            ),
        "warnings":
            list(
                dict.fromkeys(
                    warnings
                )
            ),
        "status":
            status,
        "requirements":
            requirements,
        "compliance":
            clean_compliance,
        "requires_hitl_confirmation":
            True,
    }



def validate_generated_payload(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:

    errors: list[
        str
    ] = []

    warnings: list[
        str
    ] = []

    question_contract_diagnostics: list[
        dict[str, Any]
    ] = []

    special_instruction_validation = (
        validate_special_instruction_reporting(
            payload,
            request_payload,
        )
    )

    errors.extend(
        special_instruction_validation.get(
            "errors",
            [],
        )
    )

    warnings.extend(
        special_instruction_validation.get(
            "warnings",
            [],
        )
    )

    questions = payload.get(
        "questions",
        [],
    )

    if not isinstance(
        questions,
        list,
    ):
        return {
            "valid":
                False,

            "errors": [
                "questions must be a list"
            ],

            "warnings":
                [],

            "generated_marks":
                0,

            "generated_questions":
                0,
        }

    blueprint = request_payload.get(
        "blueprint",
        [],
    )

    blueprint_by_index = {
        safe_int(
            item.get(
                "plan_index"
            )
        ):
            item
        for item in blueprint
        if isinstance(
            item,
            dict,
        )
    }

    if len(
        questions
    ) != len(
        blueprint
    ):
        errors.append(
            "Generated question count does not match the deterministic blueprint."
        )

    seen_plan_indexes: set[
        int
    ] = set()

    seen_question_texts: set[
        str
    ] = set()

    generated_marks = 0

    for position, question in enumerate(
        questions,
        start=1,
    ):
        prefix = (
            f"Question {position}"
        )

        if not isinstance(
            question,
            dict,
        ):
            errors.append(
                f"{prefix}: must be an object"
            )
            continue

        plan_index = safe_int(
            question.get(
                "plan_index"
            )
        )

        if plan_index not in blueprint_by_index:
            errors.append(
                f"{prefix}: invalid plan_index={plan_index}"
            )
            continue

        if plan_index in seen_plan_indexes:
            errors.append(
                f"{prefix}: duplicate plan_index={plan_index}"
            )

        seen_plan_indexes.add(
            plan_index
        )

        expected = blueprint_by_index[
            plan_index
        ]

        for field in [
            "topic",
            "official_reference",
            "paper_code",
            "paper_label",
            "assessment_pattern",
            "role",
        ]:
            actual_value = str(
                question.get(
                    field,
                    "",
                )
                or ""
            ).strip()

            expected_value = str(
                expected.get(
                    field,
                    "",
                )
                or ""
            ).strip()

            if normalize_text(
                actual_value
            ) != normalize_text(
                expected_value
            ):
                errors.append(
                    f"{prefix}: {field} mismatch "
                    f"({actual_value!r} != {expected_value!r})"
                )

        def coverage_signature(
            values: Any,
        ) -> set[
            tuple[
                str,
                str,
                str,
            ]
        ]:
            result = set()

            for item in safe_list(
                values
            ):
                if not isinstance(
                    item,
                    dict,
                ):
                    continue

                result.add(
                    (
                        normalize_text(
                            item.get(
                                "topic",
                                "",
                            )
                        ),
                        normalize_text(
                            item.get(
                                "official_reference",
                                "",
                            )
                        ),
                        str(
                            item.get(
                                "role",
                                "",
                            )
                            or ""
                        ).strip().casefold(),
                    )
                )

            return result

        actual_coverage = coverage_signature(
            question.get(
                "covered_topics",
                [],
            )
        )

        expected_coverage = coverage_signature(
            expected.get(
                "covered_topics",
                [],
            )
        )

        if actual_coverage != expected_coverage:
            errors.append(
                f"{prefix}: covered_topics mismatch "
                f"({sorted(actual_coverage)!r} != "
                f"{sorted(expected_coverage)!r})"
            )

        marks = safe_int(
            question.get(
                "marks"
            )
        )

        expected_marks = safe_int(
            expected.get(
                "marks"
            )
        )

        if marks != expected_marks:
            errors.append(
                f"{prefix}: marks mismatch "
                f"({marks} != {expected_marks})"
            )

        generated_marks += max(
            0,
            marks,
        )

        q_text = str(
            question.get(
                "question_text",
                "",
            )
            or ""
        ).strip()

        if not q_text:
            errors.append(
                f"{prefix}: question_text is empty"
            )

        normalized_q_text = normalize_text(
            q_text
        )

        if normalized_q_text in seen_question_texts:
            errors.append(
                f"{prefix}: duplicate generated question text"
            )

        if normalized_q_text:
            seen_question_texts.add(
                normalized_q_text
            )

        if str(
            question.get(
                "source_type",
                "",
            )
            or ""
        ).strip().casefold() != (
            "ai_generated_aqa_aligned"
        ):
            errors.append(
                f"{prefix}: source_type must be ai_generated_aqa_aligned"
            )

        if bool(
            question.get(
                "official_aqa_question",
                False,
            )
        ):
            errors.append(
                f"{prefix}: generated question cannot claim to be official AQA"
            )

        guidance = question.get(
            "marking_guidance",
            [],
        )

        if not isinstance(
            guidance,
            list,
        ) or not guidance:
            errors.append(
                f"{prefix}: marking_guidance must be a non-empty list"
            )
            guidance = []

        guidance_marks = 0

        for criterion_index, item in enumerate(
            guidance,
            start=1,
        ):
            if not isinstance(
                item,
                dict,
            ):
                errors.append(
                    f"{prefix}: marking criterion {criterion_index} is invalid"
                )
                continue

            criterion_marks = safe_int(
                item.get(
                    "marks"
                )
            )

            criterion_text = str(
                item.get(
                    "criterion",
                    "",
                )
                or ""
            ).strip()

            if criterion_marks <= 0:
                errors.append(
                    f"{prefix}: marking criterion {criterion_index} "
                    "must award positive marks"
                )

            if not criterion_text:
                errors.append(
                    f"{prefix}: marking criterion {criterion_index} is empty"
                )

            guidance_marks += max(
                0,
                criterion_marks,
            )

        if guidance_marks != marks:
            errors.append(
                f"{prefix}: marking-guidance marks "
                f"({guidance_marks}) do not equal question marks ({marks})"
            )

        aqa_scope_errors = _aqa_formal_complexity_scope_errors(
            q_text,
            guidance,
        )

        errors.extend(
            f"{prefix}: {message}"
            for message in aqa_scope_errors
        )

        contract_check = _question_marking_contract_checks(
            q_text,
            guidance,
        )

        question_contract_diagnostics.append(
            {
                "plan_index": plan_index,
                "generated_question_id": str(
                    question.get("generated_question_id", "") or ""
                ),
                **contract_check,
            }
        )

        errors.extend(
            f"{prefix}: {message}"
            for message in contract_check.get("errors", [])
        )
        warnings.extend(
            f"{prefix}: {message}"
            for message in contract_check.get("warnings", [])
        )

        requires_code = bool(
            question.get(
                "requires_code",
                False,
            )
        )

        requires_visual = bool(
            question.get(
                "requires_visual",
                False,
            )
        )

        expected_visual_requirement = normalize_visual_requirement(
            expected.get(
                "visual_requirement",
                "none",
            )
        )

        actual_visual_requirement = normalize_visual_requirement(
            question.get(
                "visual_requirement",
                "none",
            )
        )

        visual_gate = question.get(
            "visual_relevance_gate",
            {},
        )

        if not isinstance(
            visual_gate,
            dict,
        ):
            visual_gate = {}

        gate_removed_irrelevant = bool(
            actual_visual_requirement == "none"
            and expected_visual_requirement != "none"
            and str(
                visual_gate.get(
                    "status",
                    "",
                )
                or ""
            ).strip().upper()
            == "REMOVED_IRRELEVANT"
        )

        if (
            actual_visual_requirement
            != expected_visual_requirement
            and not gate_removed_irrelevant
        ):
            errors.append(
                f"{prefix}: visual_requirement mismatch "
                f"({actual_visual_requirement!r} != "
                f"{expected_visual_requirement!r})"
            )

        if gate_removed_irrelevant:
            warnings.append(
                f"{prefix}: planned {expected_visual_requirement} visual was "
                "removed by the deterministic visual relevance gate because "
                "the generated question did not depend on it."
            )

        effective_requires_visual = bool(
            actual_visual_requirement
            != "none"
        )

        if requires_visual != effective_requires_visual:
            errors.append(
                f"{prefix}: requires_visual mismatch "
                f"({requires_visual} != {effective_requires_visual})"
            )

        visual_spec_errors = validate_visual_spec(
            question.get(
                "visual"
            ),
            actual_visual_requirement,
        )

        for visual_error in visual_spec_errors:
            errors.append(
                f"{prefix}: {visual_error}"
            )

        if (
            actual_visual_requirement
            == "code_block"
            and not requires_code
        ):
            errors.append(
                f"{prefix}: code_block visual requires requires_code=True"
            )

        filters = request_payload.get(
            "assessment_filters",
            {},
        )

        if (
            not filters.get(
                "include_code_questions",
                True,
            )
            and requires_code
        ):
            errors.append(
                f"{prefix}: code question violates include_code_questions=False"
            )

        if (
            not filters.get(
                "include_visual_questions",
                True,
            )
            and requires_visual
        ):
            errors.append(
                f"{prefix}: visual question violates include_visual_questions=False"
            )

        requested_language = str(
            filters.get(
                "programming_language",
                "",
            )
            or ""
        ).strip()

        if (
            requires_code
            and requested_language
        ):
            actual_language = str(
                question.get(
                    "programming_language",
                    "",
                )
                or ""
            ).strip()

            if normalize_text(
                actual_language
            ) != normalize_text(
                requested_language
            ):
                errors.append(
                    f"{prefix}: programming language mismatch "
                    f"({actual_language!r} != {requested_language!r})"
                )

    expected_marks_total = safe_int(
        request_payload.get(
            "target_generated_marks"
        )
    )

    if generated_marks != expected_marks_total:
        errors.append(
            "Generated total marks do not match the requested generation budget: "
            f"{generated_marks} != {expected_marks_total}"
        )

    filters = request_payload.get(
        "assessment_filters",
        {},
    )

    if not isinstance(
        filters,
        dict,
    ):
        filters = {}

    special_directives = filters.get(
        "special_instruction_directives",
        {},
    )

    if not isinstance(
        special_directives,
        dict,
    ):
        special_directives = {}

    role_counts = {
        "primary":
            sum(
                1
                for item in questions
                if isinstance(
                    item,
                    dict,
                )
                and str(
                    item.get(
                        "role",
                        "",
                    )
                    or ""
                ).strip().casefold()
                == "primary"
            ),
        "supporting":
            sum(
                1
                for item in questions
                if isinstance(
                    item,
                    dict,
                )
                and str(
                    item.get(
                        "role",
                        "",
                    )
                    or ""
                ).strip().casefold()
                == "supporting"
            ),
    }

    for role, directive_key in [
        (
            "primary",
            "exact_primary_questions",
        ),
        (
            "supporting",
            "exact_supporting_questions",
        ),
    ]:
        exact_value = special_directives.get(
            directive_key
        )

        if exact_value is None:
            continue

        expected_exact = safe_int(
            exact_value,
            -1,
        )

        if role_counts[
            role
        ] != expected_exact:
            errors.append(
                "Mandatory special-instruction role count violated: "
                f"expected exactly {expected_exact} {role} questions, "
                f"received {role_counts[role]}."
            )

    distinct_style_roles = special_directives.get(
        "distinct_styles_for_roles",
        [],
    )

    if isinstance(
        distinct_style_roles,
        list,
    ):
        for role in distinct_style_roles:
            role_norm = str(
                role or ""
            ).strip().casefold()

            role_patterns = [
                normalize_text(
                    item.get(
                        "assessment_pattern",
                        "",
                    )
                )
                for item in questions
                if isinstance(
                    item,
                    dict,
                )
                and str(
                    item.get(
                        "role",
                        "",
                    )
                    or ""
                ).strip().casefold()
                == role_norm
            ]

            role_patterns = [
                value
                for value in role_patterns
                if value
            ]

            if len(
                role_patterns
            ) != len(
                set(
                    role_patterns
                )
            ):
                errors.append(
                    "Mandatory special-instruction distinct-style constraint "
                    f"violated for role {role_norm!r}: assessment_pattern "
                    "labels are not unique."
                )

    return {
        "valid":
            not errors,

        "errors":
            errors,

        "warnings":
            warnings,

        "generated_marks":
            generated_marks,

        "generated_questions":
            len(
                questions
            ),

        "question_marking_contract_checks":
            question_contract_diagnostics,

        "special_instruction_validation":
            special_instruction_validation,
    }

# ================================================================
# v2.37 — VISUAL INTEGRITY / LEAKAGE / CONSISTENCY PRECHECKS
# ================================================================

_VISUAL_REFERENCE_PATTERNS = [
    r"\brefer to (?:the|this|that)?\s*(?:array\s+grid|grid|diagram|visual|table|trace table|truth table|flowchart|pseudocode|code block|cpu diagram|network diagram)\b",
    r"\blook at (?:the|this|that)?\s*(?:array\s+grid|grid|diagram|visual|table|trace table|truth table|flowchart|pseudocode|code block|cpu diagram|network diagram)\b",
    r"\buse (?:the|this|that)?\s*(?:array\s+grid|grid|diagram|visual|table|trace table|truth table|flowchart|pseudocode|code block|cpu diagram|network diagram)\b",
    r"\bshown in (?:the|this|that)?\s*(?:visual|diagram|table|grid|flowchart|pseudocode|code block)\b",
    r"\b(?:diagram|visual|grid|flowchart|pseudocode|table|code block)\s+(?:shown|provided|below|above|in the visual)\b",
    r"\bthe\s+(?:array\s+grid|cpu\s+diagram|network\s+diagram|logic\s+gate\s+diagram|pseudocode)\b",
    r"\bin the visual\b",
]


def _question_references_external_visual(
    question_text_value: Any,
) -> bool:
    # Use the same precise, embedded-content-aware detector as the active
    # generic visual-integrity gate. Function lookup occurs at validation time
    # after this notebook cell has finished loading.
    detector = globals().get(
        "_generic_question_references_visual"
    )

    if callable(detector):
        return bool(
            detector(
                question_text_value
            )
        )

    # Conservative fallback used only if called before the generic helper has
    # been defined.
    text_value = normalize_text(
        question_text_value
    )

    if not text_value:
        return False

    return any(
        re.search(
            pattern,
            text_value,
            flags=re.IGNORECASE,
        )
        is not None
        for pattern in _VISUAL_REFERENCE_PATTERNS
    )


def _question_asks_to_identify_gate(
    question_text_value: Any,
) -> bool:
    text_value = normalize_text(
        question_text_value
    )
    return bool(
        re.search(
            r"\b(?:name|identify|state the name of|what (?:logic )?gate)\b"
            r".{0,80}\bgate\b",
            text_value,
            flags=re.IGNORECASE,
        )
        or re.search(
            r"\bgate\b.{0,80}\b(?:name|identify)\b",
            text_value,
            flags=re.IGNORECASE,
        )
    )


def _logic_gate_visible_text(
    spec: dict[str, Any],
) -> str:
    pieces: list[str] = []

    caption = str(
        spec.get(
            "caption",
            "",
        )
        or ""
    ).strip()
    if caption:
        pieces.append(caption)

    output = spec.get(
        "output",
        {},
    )
    if isinstance(output, dict):
        output_label = str(
            output.get(
                "label",
                "",
            )
            or ""
        ).strip()
        if output_label:
            pieces.append(output_label)

    # Gate `type` is renderer metadata, not student-visible text.
    return " ".join(pieces)


def _visual_answer_leakage_errors(
    question: dict[str, Any],
) -> list[str]:
    errors: list[str] = []

    q_text = str(
        question.get(
            "question_text",
            "",
        )
        or ""
    )

    visual_type = normalize_visual_requirement(
        question.get(
            "visual_requirement",
            "none",
        )
    )

    visual = question.get(
        "visual",
        {},
    )
    spec = (
        visual.get("spec", {})
        if isinstance(visual, dict)
        else {}
    )
    if not isinstance(spec, dict):
        spec = {}

    if (
        visual_type == "logic_gate_diagram"
        and _question_asks_to_identify_gate(q_text)
    ):
        visible_text = normalize_text(
            _logic_gate_visible_text(
                spec
            )
        )

        for gate in spec.get(
            "gates",
            [],
        ):
            if not isinstance(gate, dict):
                continue

            gate_type = str(
                gate.get(
                    "type",
                    "",
                )
                or ""
            ).strip().upper()

            if (
                gate_type
                and normalize_text(gate_type)
                in visible_text
            ):
                errors.append(
                    "logic-gate identification question exposes the gate "
                    f"type {gate_type!r} in student-visible visual text"
                )

    # Network/device identification leakage: if the learner is explicitly
    # asked to name/identify a device, do not put that exact device name in a
    # visible label.
    if visual_type == "network_diagram":
        identify_device = bool(
            re.search(
                r"\b(?:name|identify|state the name of)\b.{0,90}"
                r"\b(?:device|network device|component|hardware)\b",
                normalize_text(q_text),
                flags=re.IGNORECASE,
            )
        )

        if identify_device:
            guidance_text = normalize_text(
                " ".join(
                    str(item.get("criterion", "") or "")
                    for item in question.get(
                        "marking_guidance",
                        [],
                    )
                    if isinstance(item, dict)
                )
            )

            labels = [
                str(node.get("label", "") or "")
                for node in spec.get("nodes", [])
                if isinstance(node, dict)
            ]

            for device in [
                "router",
                "switch",
                "server",
                "printer",
                "firewall",
                "access point",
            ]:
                if (
                    device in guidance_text
                    and any(
                        device
                        in normalize_text(label)
                        for label in labels
                    )
                ):
                    errors.append(
                        "network-device identification question exposes the "
                        f"expected device name {device!r} in a visible node label"
                    )

    return errors


def _question_visual_consistency_errors(
    question: dict[str, Any],
) -> list[str]:
    errors: list[str] = []

    q_text = str(
        question.get(
            "question_text",
            "",
        )
        or ""
    )

    visual_type = normalize_visual_requirement(
        question.get(
            "visual_requirement",
            "none",
        )
    )

    visual = question.get(
        "visual",
        {},
    )
    spec = (
        visual.get("spec", {})
        if isinstance(visual, dict)
        else {}
    )
    if not isinstance(spec, dict):
        spec = {}

    # Hard fail: the student text explicitly depends on a visual but the
    # blueprint/response says there is no visual.
    if (
        _question_references_external_visual(q_text)
        and visual_type == "none"
    ):
        errors.append(
            "question_text explicitly refers to a diagram/grid/table/visual "
            "but visual_requirement='none'"
        )
        return errors

    if visual_type == "network_diagram":
        node_terms: set[str] = set()
        for node in spec.get("nodes", []):
            if not isinstance(node, dict):
                continue

            node_type = normalize_text(
                node.get("type", "")
            )
            node_label = normalize_text(
                node.get("label", "")
            )

            for term in [
                "router",
                "switch",
                "server",
                "printer",
                "firewall",
                "access point",
                "computer",
                "laptop",
            ]:
                if (
                    term in node_type
                    or term in node_label
                ):
                    node_terms.add(term)

        question_terms = {
            term
            for term in [
                "router",
                "switch",
                "server",
                "printer",
                "firewall",
                "access point",
            ]
            if term in normalize_text(q_text)
        }

        missing_terms = sorted(
            question_terms - node_terms
        )

        if missing_terms:
            errors.append(
                "network question terminology is inconsistent with the "
                "visual nodes; missing from diagram: "
                + ", ".join(missing_terms)
            )

    elif visual_type == "logic_gate_diagram":
        gate_types = {
            str(gate.get("type", "") or "")
            .strip()
            .upper()
            for gate in spec.get("gates", [])
            if isinstance(gate, dict)
        }

        q_upper = q_text.upper()

        explicitly_named = {
            gate
            for gate in [
                "AND",
                "OR",
                "NOT",
                "NAND",
                "NOR",
                "XOR",
                "XNOR",
            ]
            if re.search(
                rf"\b{gate}\s+gate\b",
                q_upper,
            )
        }

        if explicitly_named and not explicitly_named.issubset(
            gate_types
        ):
            errors.append(
                "logic-gate terminology in question_text does not match "
                f"visual gate types: question={sorted(explicitly_named)}, "
                f"visual={sorted(gate_types)}"
            )

    elif visual_type == "array_grid":
        values = spec.get(
            "values",
            [],
        )

        if (
            isinstance(values, list)
            and values
        ):
            if isinstance(
                values[0],
                list,
            ):
                row_count = len(values)
                col_count = max(
                    (
                        len(row)
                        for row in values
                        if isinstance(row, list)
                    ),
                    default=0,
                )
            else:
                row_count = 1
                col_count = len(values)

            for match in re.finditer(
                r"\[\s*(\d+)\s*\]\s*\[\s*(\d+)\s*\]",
                q_text,
            ):
                row_index = int(
                    match.group(1)
                )
                col_index = int(
                    match.group(2)
                )

                if (
                    row_index >= row_count
                    or col_index >= col_count
                ):
                    errors.append(
                        "question_text references array index outside the "
                        f"visual grid: [{row_index}][{col_index}] for "
                        f"{row_count}x{col_count} grid"
                    )


    compatibility_errors = _visual_topic_compatibility_errors(
        question
    )
    errors.extend(
        compatibility_errors
    )

    if visual_type == "cpu_block_diagram":
        component_labels = _cpu_visual_component_labels(
            spec
        )

        required_terms = {
            "control unit": "control unit",
            "alu": "alu",
            "register": "registers",
            "registers": "registers",
            "memory": "memory",
        }

        mentioned = {
            normalized
            for term, normalized in required_terms.items()
            if term in normalize_text(q_text)
        }

        missing = sorted(
            value
            for value in mentioned
            if value not in component_labels
        )

        if missing:
            errors.append(
                "CPU diagram terminology is inconsistent with the visual "
                "components; missing from diagram: " + ", ".join(missing)
            )

    return errors


_validate_generated_payload_v236 = validate_generated_payload


def validate_generated_payload(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:
    result = _validate_generated_payload_v236(
        payload,
        request_payload,
    )

    extra_errors: list[str] = []
    diagnostics: list[dict[str, Any]] = []

    questions = payload.get(
        "questions",
        [],
    )

    if not isinstance(
        questions,
        list,
    ):
        questions = []

    for position, question in enumerate(
        questions,
        start=1,
    ):
        if not isinstance(
            question,
            dict,
        ):
            continue

        qid = str(
            question.get(
                "generated_question_id",
                f"Q{position}",
            )
            or f"Q{position}"
        )

        leakage_errors = _visual_answer_leakage_errors(
            question
        )

        consistency_errors = _question_visual_consistency_errors(
            question
        )

        dependency_warnings = _visual_dependency_warnings(
            question
        )

        question_errors = (
            leakage_errors
            + consistency_errors
        )

        diagnostics.append(
            {
                "generated_question_id": qid,
                "visual_requirement": normalize_visual_requirement(
                    question.get(
                        "visual_requirement",
                        "none",
                    )
                ),
                "references_visual_in_text":
                    _question_references_external_visual(
                        question.get(
                            "question_text",
                            "",
                        )
                    ),
                "answer_leakage_errors":
                    leakage_errors,
                "question_visual_consistency_errors":
                    consistency_errors,
                "visual_dependency_warnings":
                    dependency_warnings,
            }
        )

        extra_errors.extend(
            f"{qid}: {message}"
            for message in question_errors
        )

    if extra_errors:
        result.setdefault(
            "errors",
            [],
        ).extend(
            extra_errors
        )
        result["errors"] = list(
            dict.fromkeys(
                result["errors"]
            )
        )
        result["valid"] = False

    result[
        "visual_quality_preflight"
    ] = {
        "status": (
            "PASS"
            if not extra_errors
            else "FAIL"
        ),
        "diagnostics": diagnostics,
        "errors": extra_errors,
    }

    return result

# ================================================================
# v2.38 — VISUAL COMPATIBILITY + DEPENDENCY HELPERS
# ================================================================

def _question_visual_topic_blob(
    question: dict[str, Any],
) -> str:
    pieces = [
        str(question.get("topic", "") or ""),
        str(question.get("official_reference", "") or ""),
        str(question.get("question_text", "") or ""),
    ]
    return normalize_text(" ".join(pieces))


def _compatible_visual_types_for_question(
    question: dict[str, Any],
) -> set[str]:
    blob = _question_visual_topic_blob(question)

    if any(
        phrase in blob
        for phrase in [
            "logic gates",
            "logic gate",
            "logic circuit",
            "truth table",
            "boolean logic",
        ]
    ):
        return {"logic_gate_diagram", "truth_table"}

    if "boolean operations in programming" in blob:
        return {"code_block", "trace_table"}

    if any(
        phrase in blob
        for phrase in [
            "star topology",
            "bus topology",
            "computer networks",
            "network topology",
        ]
    ):
        return {"network_diagram"}

    if any(
        phrase in blob
        for phrase in [
            "one- and two-dimensional arrays",
            "one dimensional array",
            "one-dimensional array",
            "two dimensional array",
            "two-dimensional array",
            "2d array",
            "array",
        ]
    ):
        return {"array_grid", "trace_table", "code_block"}

    if any(
        phrase in blob
        for phrase in [
            "cpu",
            "processor",
            "control unit",
            "alu",
            "registers",
            "von neumann",
        ]
    ):
        return {"cpu_block_diagram", "code_block", "trace_table"}

    if any(
        phrase in blob
        for phrase in [
            "algorithm",
            "flowchart",
            "pseudocode",
            "iteration",
            "selection",
            "sequence",
        ]
    ):
        return {"code_block", "simple_flowchart", "trace_table"}

    return set(VISUAL_TYPES) - {"none"}


def _visual_topic_compatibility_errors(
    question: dict[str, Any],
) -> list[str]:
    visual_type = normalize_visual_requirement(
        question.get(
            "visual_requirement",
            "none",
        )
    )

    if visual_type == "none":
        return []

    compatible = _compatible_visual_types_for_question(
        question
    )

    if visual_type not in compatible:
        return [
            f"visual type {visual_type!r} is not pedagogically compatible "
            f"with topic/question context (allowed={sorted(compatible)})"
        ]

    return []


def _cpu_visual_component_labels(
    spec: dict[str, Any],
) -> set[str]:
    labels: set[str] = set()

    for component in spec.get("components", []):
        if not isinstance(component, dict):
            continue

        blob = normalize_text(
            f"{component.get('id', '')} {component.get('label', '')}"
        )

        for term in [
            "control unit",
            "alu",
            "register",
            "registers",
            "memory",
            "cache",
            "accumulator",
        ]:
            if term in blob:
                labels.add("registers" if term == "register" else term)

    return labels


def _visual_dependency_warnings(
    question: dict[str, Any],
) -> list[str]:
    warnings: list[str] = []

    visual_type = normalize_visual_requirement(
        question.get("visual_requirement", "none")
    )
    if visual_type == "none":
        return warnings

    q_text = normalize_text(
        question.get("question_text", "")
    )

    visual = question.get("visual", {})
    spec = visual.get("spec", {}) if isinstance(visual, dict) else {}
    if not isinstance(spec, dict):
        spec = {}

    if visual_type == "logic_gate_diagram":
        if (
            "and gate" in q_text or "or gate" in q_text or "not gate" in q_text
        ) and ("what is the value of q" in q_text or "state the output q" in q_text):
            warnings.append(
                "Logic-gate diagram may be decorative because the gate type and "
                "required inputs/outputs are fully described in question_text."
            )

    elif visual_type == "network_diagram":
        if (
            "preferred over a bus topology" in q_text
            or "advantages of using a star topology" in q_text
        ):
            warnings.append(
                "Network diagram may be decorative because the learner task is "
                "generic topology evaluation rather than diagram interpretation."
            )

    elif visual_type == "array_grid":
        if "seats = [[" in q_text or "scores = [[" in q_text or "numbers = [" in q_text:
            warnings.append(
                "Array visual may be decorative because the array contents are "
                "fully restated in question_text."
            )

    elif visual_type == "cpu_block_diagram":
        component_labels = _cpu_visual_component_labels(spec)
        if "memory" in q_text and "memory" not in component_labels:
            warnings.append(
                "CPU diagram is missing a Memory component even though the "
                "question_text relies on memory access."
            )
        if ("refer to the cpu" in q_text or "cpu diagram" in q_text) and not (
            "which component" in q_text or "identify" in q_text or "label" in q_text
        ):
            warnings.append(
                "CPU block diagram may be decorative because the learner task can "
                "likely be answered from text/code without using the diagram."
            )

    elif visual_type == "code_block":
        if "the pseudocode shown in the visual" in q_text and "line" in q_text:
            # Often okay, but warn if all code needed is already visible through text
            pass

    return warnings

# ================================================================
# v2.41 — GENERIC VISUAL INVARIANTS
# ================================================================

_GENERIC_VISUAL_NOUN_PATTERN = (
    r"(?:"
    r"visual|diagram|figure|flowchart|flow\s+chart|grid|"
    r"truth\s+table|trace\s+table|table|"
    r"pseudocode|code\s+block|code\s+listing|code"
    r")"
)


def _strip_embedded_question_artifacts(
    question_text_value: Any,
) -> str:
    """
    Remove learner-facing artifacts already embedded inside question_text.

    A fenced code block or inline Markdown table is self-contained content,
    not an external visual dependency. Removing the artifact itself prevents
    ordinary code/table words inside it from causing visual-reference false
    positives while preserving the surrounding instruction text.
    """
    text_value = str(
        question_text_value or ""
    )

    # Preserve a neutral placeholder so sentence boundaries remain sensible.
    text_value = re.sub(
        r"```[\s\S]*?```",
        " [embedded content] ",
        text_value,
        flags=re.MULTILINE,
    )

    # Remove consecutive Markdown-table rows.
    lines = text_value.splitlines()
    cleaned_lines: list[str] = []
    in_markdown_table = False

    for line in lines:
        stripped = line.strip()

        is_table_row = bool(
            stripped.startswith("|")
            and stripped.endswith("|")
            and stripped.count("|") >= 2
        )

        is_table_separator = bool(
            re.fullmatch(
                r"\|?\s*:?-{3,}:?\s*"
                r"(?:\|\s*:?-{3,}:?\s*)+\|?",
                stripped,
            )
        )

        if is_table_row or is_table_separator:
            if not in_markdown_table:
                cleaned_lines.append(
                    "[embedded content]"
                )
            in_markdown_table = True
            continue

        in_markdown_table = False
        cleaned_lines.append(
            line
        )

    return "\n".join(
        cleaned_lines
    )


def _question_contains_self_contained_code_artifact(
    question_text_value: Any,
) -> bool:
    """
    Return True when the learner-facing question already contains enough
    code/pseudocode content to be self-contained.

    This prevents phrases such as "Using the pseudocode above" from being
    mistaken for a reference to a separately rendered visual when the actual
    pseudocode is already present inside question_text.
    """
    text_value = str(
        question_text_value or ""
    )

    # Explicit fenced code is definitely embedded learner-facing content.
    if re.search(
        r"```[\s\S]*?```",
        text_value,
        flags=re.MULTILINE,
    ):
        return True

    # Assignment arrows / pseudocode assignment operators are strong evidence
    # that the code body itself is present in the question.
    if re.search(
        r"(?:←|<-|:=)",
        text_value,
    ):
        return True

    # Explicit structured terminators are also strong evidence.
    if re.search(
        r"\b(?:END\s+IF|END\s+FOR|END\s+WHILE|ENDIF|ENDFOR|ENDWHILE)\b",
        text_value,
        flags=re.IGNORECASE,
    ):
        return True

    # For less distinctive code, require multiple independent syntax signals
    # so ordinary prose containing one programming word is not misclassified.
    syntax_signals = 0

    signal_patterns = [
        r"\bIF\b.{0,80}\bTHEN\b",
        r"\bELSE\b",
        r"\bFOR\b.{0,80}\b(?:DO|TO|IN)\b",
        r"\bWHILE\b.{0,80}\bDO\b",
        r"\b(?:INPUT|OUTPUT|RETURN)\b",
        r"\b[A-Za-z_]\w*\s*=\s*(?:-?\d+|[A-Za-z_]\w*)\b",
    ]

    for pattern in signal_patterns:
        if re.search(
            pattern,
            text_value,
            flags=re.IGNORECASE | re.DOTALL,
        ):
            syntax_signals += 1

    return syntax_signals >= 2


def _generic_question_references_visual(
    question_text_value: Any,
) -> bool:
    """
    Detect a genuine dependency on a separately supplied visual/artifact.

    Graphical objects (diagram, grid, table, flowchart, etc.) are treated as
    external when the wording explicitly points the learner to them.

    Code/pseudocode is handled separately: if the actual code body is already
    embedded in question_text, wording such as "Using the pseudocode above"
    remains self-contained and does NOT require visual_requirement != 'none'.
    A true external code-block reference still counts as a visual/artifact
    dependency.
    """
    original_text = str(
        question_text_value or ""
    )

    text_value = _strip_embedded_question_artifacts(
        original_text
    )

    normalized = re.sub(
        r"\s+",
        " ",
        str(
            text_value or ""
        ),
    ).strip()

    if not normalized:
        return False

    graphical_noun = (
        r"(?:"
        r"visual|diagram|figure|flowchart|flow\s+chart|grid|"
        r"truth\s+table|trace\s+table|table"
        r")"
    )

    code_noun = (
        r"(?:"
        r"pseudocode|code\s+block|code\s+listing|code"
        r")"
    )

    def has_explicit_reference(
        noun_pattern: str,
    ) -> bool:
        explicit_patterns = [
            # Direct action on a named supplied object.
            rf"\b(?:refer\s+to|look\s+at|inspect|study|use|using|complete)\s+"
            rf"(?:(?:the|this|that|following|given|provided)\s+)?"
            rf"{noun_pattern}\b",

            # Explicit deictic/location reference.
            rf"\b(?:the|this|that|following|given|provided)\s+"
            rf"{noun_pattern}\s+"
            rf"(?:below|above|shown|provided|given)\b",

            # "diagram shown below", "code provided above".
            rf"\b{noun_pattern}\s+"
            rf"(?:shown|provided|given|displayed)\s+"
            rf"(?:below|above|here|in\s+the\s+question)\b",

            # "shown in the diagram/figure/table".
            rf"\b(?:shown|provided|given|displayed)\s+in\s+"
            rf"(?:(?:the|this|that)\s+)?"
            rf"{noun_pattern}\b",

            # "according to / based on the diagram".
            rf"\b(?:according\s+to|based\s+on)\s+"
            rf"(?:(?:the|this|that|following)\s+)?"
            rf"{noun_pattern}\b",

            # "in the diagram", "from the table" as an explicit answer source.
            rf"\b(?:in|from)\s+"
            rf"(?:(?:the|this|that|following|given|provided)\s+)"
            rf"{noun_pattern}\b",
        ]

        return any(
            re.search(
                pattern,
                normalized,
                flags=re.IGNORECASE,
            )
            is not None
            for pattern in explicit_patterns
        )

    # Genuine graphical references remain fail-closed.
    if has_explicit_reference(
        graphical_noun
    ):
        return True

    # Code/pseudocode is only external when the code body is not already
    # present inside the learner-facing question itself.
    if has_explicit_reference(
        code_noun
    ):
        return not _question_contains_self_contained_code_artifact(
            original_text
        )

    return False


def _generic_unique_ids(
    rows: Any,
    *,
    id_key: str = "id",
) -> tuple[set[str], list[str]]:
    ids: list[str] = []

    if not isinstance(
        rows,
        list,
    ):
        return set(), []

    for row in rows:
        if not isinstance(
            row,
            dict,
        ):
            continue

        value = str(
            row.get(
                id_key,
                "",
            )
            or ""
        ).strip()

        if value:
            ids.append(
                value
            )

    duplicates = sorted(
        {
            value
            for value in ids
            if ids.count(value) > 1
        }
    )

    return set(ids), duplicates


def _generic_graph_reference_errors(
    *,
    nodes: Any,
    edges: Any,
    node_label: str,
) -> list[str]:
    errors: list[str] = []

    node_ids, duplicate_ids = _generic_unique_ids(
        nodes
    )

    if duplicate_ids:
        errors.append(
            f"{node_label} IDs must be unique; duplicates="
            + ", ".join(
                duplicate_ids
            )
        )

    if not isinstance(
        edges,
        list,
    ):
        return errors

    for edge_index, edge in enumerate(
        edges,
        start=1,
    ):
        if not isinstance(
            edge,
            dict,
        ):
            continue

        source = str(
            edge.get(
                "from",
                "",
            )
            or ""
        ).strip()

        target = str(
            edge.get(
                "to",
                "",
            )
            or ""
        ).strip()

        if source and source not in node_ids:
            errors.append(
                f"edge {edge_index} references unknown source ID {source!r}"
            )

        if target and target not in node_ids:
            errors.append(
                f"edge {edge_index} references unknown target ID {target!r}"
            )

    return errors


def _generic_logic_reference_errors(
    spec: dict[str, Any],
) -> list[str]:
    errors: list[str] = []

    input_ids = {
        str(value).strip()
        for value in spec.get(
            "inputs",
            [],
        )
        if str(value).strip()
    }

    gates = spec.get(
        "gates",
        [],
    )

    gate_ids, duplicate_ids = _generic_unique_ids(
        gates
    )

    if duplicate_ids:
        errors.append(
            "logic gate IDs must be unique; duplicates="
            + ", ".join(
                duplicate_ids
            )
        )

    supported_gate_types = {
        "AND",
        "OR",
        "NOT",
        "NAND",
        "NOR",
        "XOR",
        "XNOR",
    }

    known_refs = set(
        input_ids
    )

    if isinstance(
        gates,
        list,
    ):
        for gate_index, gate in enumerate(
            gates,
            start=1,
        ):
            if not isinstance(
                gate,
                dict,
            ):
                continue

            gate_id = str(
                gate.get(
                    "id",
                    "",
                )
                or ""
            ).strip()

            gate_type = str(
                gate.get(
                    "type",
                    "",
                )
                or ""
            ).strip().upper()

            if gate_type not in supported_gate_types:
                errors.append(
                    f"logic gate {gate_index} has unsupported type={gate_type!r}; "
                    "type must be one of the renderer-supported gate enums"
                )

            refs = gate.get(
                "inputs",
                [],
            )

            if isinstance(
                refs,
                list,
            ):
                for ref in refs:
                    ref_value = str(
                        ref
                    ).strip()

                    if (
                        ref_value
                        and ref_value not in known_refs
                    ):
                        errors.append(
                            f"logic gate {gate_index} references unknown "
                            f"input/gate ID {ref_value!r}"
                        )

            if gate_id:
                known_refs.add(
                    gate_id
                )

    output = spec.get(
        "output",
        {},
    )

    if isinstance(
        output,
        dict,
    ):
        output_from = str(
            output.get(
                "from",
                "",
            )
            or ""
        ).strip()

        if (
            output_from
            and output_from not in known_refs
        ):
            errors.append(
                f"logic output references unknown ID {output_from!r}"
            )

    return errors


def _generic_visual_integrity_errors(
    question: dict[str, Any],
) -> list[str]:
    errors: list[str] = []

    visual_type = normalize_visual_requirement(
        question.get(
            "visual_requirement",
            "none",
        )
    )

    question_text_value = str(
        question.get(
            "question_text",
            "",
        )
        or ""
    )

    references_visual = _generic_question_references_visual(
        question_text_value
    )

    if (
        references_visual
        and visual_type == "none"
    ):
        errors.append(
            "question_text refers to a visual but visual_requirement='none'"
        )
        return errors

    if visual_type == "none":
        return errors

    # A generated visual must be integrated into the learner task, otherwise it
    # is likely decorative. This rule is topic-agnostic.
    if not references_visual:
        errors.append(
            "visual_requirement is non-none but question_text does not "
            "explicitly direct the learner to use/inspect the visual"
        )

    visual = question.get(
        "visual",
        {},
    )

    spec = (
        visual.get(
            "spec",
            {},
        )
        if isinstance(
            visual,
            dict,
        )
        else {}
    )

    if not isinstance(
        spec,
        dict,
    ):
        return errors

    if visual_type in {
        "network_diagram",
        "simple_flowchart",
    }:
        errors.extend(
            _generic_graph_reference_errors(
                nodes=spec.get(
                    "nodes",
                    [],
                ),
                edges=spec.get(
                    "edges",
                    [],
                ),
                node_label="node",
            )
        )

    elif visual_type == "cpu_block_diagram":
        errors.extend(
            _generic_graph_reference_errors(
                nodes=spec.get(
                    "components",
                    [],
                ),
                edges=spec.get(
                    "connections",
                    [],
                ),
                node_label="component",
            )
        )

    elif visual_type == "logic_gate_diagram":
        errors.extend(
            _generic_logic_reference_errors(
                spec
            )
        )

    elif visual_type in {
        "truth_table",
        "trace_table",
        "database_table",
    }:
        columns = spec.get(
            "columns",
            [],
        )

        rows = spec.get(
            "rows",
            [],
        )

        if isinstance(
            columns,
            list,
        ) and isinstance(
            rows,
            list,
        ):
            width = len(
                columns
            )

            for row_index, row in enumerate(
                rows,
                start=1,
            ):
                if (
                    isinstance(
                        row,
                        list,
                    )
                    and len(row) != width
                ):
                    errors.append(
                        f"table row {row_index} width does not match columns"
                    )

    elif visual_type == "array_grid":
        values = spec.get(
            "values",
            [],
        )

        if isinstance(
            values,
            list,
        ) and values:
            if isinstance(
                values[0],
                list,
            ):
                row_count = len(
                    values
                )

                col_count = max(
                    (
                        len(row)
                        for row in values
                        if isinstance(
                            row,
                            list,
                        )
                    ),
                    default=0,
                )

                for row_index, row in enumerate(
                    values,
                    start=1,
                ):
                    if (
                        isinstance(
                            row,
                            list,
                        )
                        and len(row) != col_count
                    ):
                        errors.append(
                            f"array grid row {row_index} is not rectangular"
                        )

                for match in re.finditer(
                    r"\[\s*(\d+)\s*\]\s*\[\s*(\d+)\s*\]",
                    question_text_value,
                ):
                    r_index = int(
                        match.group(1)
                    )

                    c_index = int(
                        match.group(2)
                    )

                    if (
                        r_index >= row_count
                        or c_index >= col_count
                    ):
                        errors.append(
                            "question references an array index outside the "
                            f"visual grid: [{r_index}][{c_index}]"
                        )

    return list(
        dict.fromkeys(
            errors
        )
    )


def validate_generated_payload(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:
    """
    Generic v2.41 validation wrapper.

    IMPORTANT:
    `_validate_generated_payload_v236` is the captured original/base validator.
    We call it directly to avoid wrapper recursion.
    """
    result = _validate_generated_payload_v236(
        payload,
        request_payload,
    )

    extra_errors: list[str] = []
    diagnostics: list[dict[str, Any]] = []

    questions = payload.get(
        "questions",
        [],
    )

    if not isinstance(
        questions,
        list,
    ):
        questions = []

    for position, question in enumerate(
        questions,
        start=1,
    ):
        if not isinstance(
            question,
            dict,
        ):
            continue

        qid = str(
            question.get(
                "generated_question_id",
                f"Q{position}",
            )
            or f"Q{position}"
        )

        question_errors = _generic_visual_integrity_errors(
            question
        )

        diagnostics.append(
            {
                "generated_question_id": qid,
                "visual_requirement": normalize_visual_requirement(
                    question.get(
                        "visual_requirement",
                        "none",
                    )
                ),
                "errors": question_errors,
            }
        )

        extra_errors.extend(
            f"{qid}: {message}"
            for message in question_errors
        )

    if extra_errors:
        result.setdefault(
            "errors",
            [],
        ).extend(
            extra_errors
        )

        result["errors"] = list(
            dict.fromkeys(
                result["errors"]
            )
        )

        result["valid"] = False

    result[
        "generic_visual_integrity"
    ] = {
        "status": (
            "PASS"
            if not extra_errors
            else "FAIL"
        ),
        "errors": extra_errors,
        "diagnostics": diagnostics,
    }

    return result

# ================================================================
# v2.42 — GENERIC REFERENTIAL CONSISTENCY
# ================================================================

_VISUAL_OBJECT_TO_TYPE = {
    "truth table": "truth_table",
    "trace table": "trace_table",
    "flowchart": "simple_flowchart",
    "flow chart": "simple_flowchart",
    "array grid": "array_grid",
    "network diagram": "network_diagram",
    "cpu block diagram": "cpu_block_diagram",
    "processor block diagram": "cpu_block_diagram",
    "logic gate diagram": "logic_gate_diagram",
    "logic circuit diagram": "logic_gate_diagram",
    "database table": "database_table",
    "memory grid": "memory_grid",
    "binary register": "binary_register",
    "code block": "code_block",
    "code listing": "code_block",
    "pseudocode": "code_block",
}


def _extract_commanded_line_references(
    text_value: Any,
) -> set[int]:
    """
    Extract line numbers only from clauses that ask the learner to
    complete/correct/change/replace/identify/write specific line(s).
    """
    text = str(
        text_value or ""
    )

    command_pattern = re.compile(
        r"\b(?:complete|correct|change|replace|identify|write|give|state|"
        r"evaluate|explain|describe|trace|determine|calculate|find)"
        r"[^.!?\n]{0,120}\blines?\s+"
        r"((?:\d+\s*(?:,|and|&)?\s*)+)",
        flags=re.IGNORECASE,
    )

    references: set[int] = set()

    for match in command_pattern.finditer(
        text
    ):
        references.update(
            int(value)
            for value in re.findall(
                r"\d+",
                match.group(1),
            )
        )

    return references


def _extract_insertion_after_line_references(
    text_value: Any,
) -> set[int]:
    """
    Extract existing-code anchor lines only when the learner is explicitly
    asked to INSERT/ADD a new statement after that line.

    Example:
        "Insert a new statement after line 10."
    means line 10 is the anchor and the inserted statement may become line 11
    in the resulting program.

    This is generic code-edit semantics; no particular line number is
    hardcoded.
    """
    text = str(
        text_value or ""
    )

    patterns = [
        re.compile(
            r"\b(?:insert|add|place|write)\b"
            r"[^.!?\n]{0,120}\bafter\s+line\s+(\d+)\b",
            flags=re.IGNORECASE,
        ),
        re.compile(
            r"\bafter\s+line\s+(\d+)\b"
            r"[^.!?\n]{0,120}\b(?:insert|add|place|write)\b",
            flags=re.IGNORECASE,
        ),
    ]

    references: set[int] = set()

    for pattern in patterns:
        for match in pattern.finditer(
            text
        ):
            references.add(
                int(
                    match.group(1)
                )
            )

    return references


def _extract_marking_target_line_references(
    marking_guidance: Any,
) -> set[int]:
    references: set[int] = set()

    if not isinstance(
        marking_guidance,
        list,
    ):
        return references

    target_patterns = [
        re.compile(
            r"\bline\s+(\d+)\s+(?:should|must|is|=|to)\b",
            flags=re.IGNORECASE,
        ),
        re.compile(
            r"\bline\s+to\s+(?:change|correct|replace)\s*(?:=|is)?\s*"
            r"(?:line\s+)?(\d+)\b",
            flags=re.IGNORECASE,
        ),
        re.compile(
            r"\bcorrected\s+line\s+(\d+)\b",
            flags=re.IGNORECASE,
        ),
    ]

    for row in marking_guidance:
        if not isinstance(
            row,
            dict,
        ):
            continue

        criterion = str(
            row.get(
                "criterion",
                "",
            )
            or ""
        )

        for pattern in target_patterns:
            for match in pattern.finditer(
                criterion
            ):
                references.add(
                    int(
                        match.group(1)
                    )
                )

    return references


def _code_line_count(
    question: dict[str, Any],
) -> int:
    visual = question.get(
        "visual",
        {},
    )

    if not isinstance(
        visual,
        dict,
    ):
        return 0

    spec = visual.get(
        "spec",
        {},
    )

    if not isinstance(
        spec,
        dict,
    ):
        return 0

    code = str(
        spec.get(
            "code",
            "",
        )
        or ""
    )

    if not code.strip():
        return 0

    return len(
        code.rstrip("\n").splitlines()
    )


def _referenced_supplied_visual_types(
    question_text_value: Any,
) -> set[str]:
    """
    Detect visual object types that the learner-facing wording explicitly
    presents as supplied stimulus/reference material.

    Important distinction:
    - supplied stimulus/reference visual -> must agree with visual_requirement
    - learner response scaffold/output format -> may legitimately differ

    The detector is intentionally conservative. A bare instruction such as
    "complete the <object>" is NOT treated as proof that <object> is the
    externally supplied/rendered visual, because it may be a response scaffold.
    """
    text = normalize_text(
        question_text_value
    )

    references: set[str] = set()

    # This mapping is schema vocabulary, not syllabus/topic routing.
    for object_name, visual_type in sorted(
        _VISUAL_OBJECT_TO_TYPE.items(),
        key=lambda item: -len(
            item[0]
        ),
    ):
        object_pattern = re.escape(
            object_name
        )

        strong_supply_patterns = [
            # "Figure 2 shows/presents/contains a flowchart"
            rf"\bfigure\s*[a-z0-9._-]*\s+"
            rf"(?:shows?|presents?|contains?|depicts?|displays?)\b"
            rf"[^.!?\n]{{0,100}}\b{object_pattern}\b",

            # "the flowchart in Figure 2" / "the flowchart shown in Figure 2"
            rf"\b(?:the\s+)?{object_pattern}\b"
            rf"\s+(?:shown\s+)?in\s+figure\s*[a-z0-9._-]+\b",

            # "the supplied/provided/shown flowchart"
            rf"\b(?:shown|displayed|presented|provided|supplied)\b"
            rf"[^.!?\n]{{0,60}}\b{object_pattern}\b",

            # "flowchart shown/provided/supplied ..."
            rf"\b{object_pattern}\b"
            rf"[^.!?\n]{{0,60}}\b"
            rf"(?:shown|displayed|presented|provided|supplied)\b",

            # "use/study/inspect the flowchart shown/provided ..."
            rf"\b(?:use|study|inspect|refer\s+to|look\s+at)\b"
            rf"[^.!?\n]{{0,80}}\b{object_pattern}\b"
            rf"[^.!?\n]{{0,60}}\b"
            rf"(?:shown|displayed|presented|provided|supplied)\b",

            # "complete/label/annotate the flowchart shown/provided ..."
            # This is only treated as supplied when an explicit supply cue exists.
            rf"\b(?:complete|label|annotate)\b"
            rf"[^.!?\n]{{0,80}}\b{object_pattern}\b"
            rf"[^.!?\n]{{0,60}}\b"
            rf"(?:shown|displayed|presented|provided|supplied)\b",
        ]

        if any(
            re.search(
                pattern,
                text,
                flags=re.IGNORECASE,
            )
            for pattern in strong_supply_patterns
        ):
            references.add(
                visual_type
            )

    return references


def _generic_referential_consistency_errors(
    question: dict[str, Any],
) -> list[str]:
    errors: list[str] = []

    question_text_value = str(
        question.get(
            "question_text",
            "",
        )
        or ""
    )

    marking_guidance = question.get(
        "marking_guidance",
        [],
    )

    requested_lines = _extract_commanded_line_references(
        question_text_value
    )

    insertion_after_lines = _extract_insertion_after_line_references(
        question_text_value
    )

    marking_lines = _extract_marking_target_line_references(
        marking_guidance
    )

    # A statement inserted AFTER existing line N can legitimately be described
    # by marking guidance as resulting line N+1. Keep ordinary change/replace
    # questions strict: N+1 is allowed only when insertion wording is explicit.
    resulting_inserted_lines = {
        value + 1
        for value in insertion_after_lines
    }

    allowed_marking_lines = (
        requested_lines
        | resulting_inserted_lines
    )

    if (
        marking_lines
        and allowed_marking_lines
        and not marking_lines.issubset(
            allowed_marking_lines
        )
    ):
        errors.append(
            "marking_guidance targets line number(s) not requested by the "
            "question or implied by an explicit insertion: requested="
            + str(
                sorted(
                    requested_lines
                )
            )
            + ", insertion_after="
            + str(
                sorted(
                    insertion_after_lines
                )
            )
            + ", marking="
            + str(
                sorted(
                    marking_lines
                )
            )
        )

    line_count = _code_line_count(
        question
    )

    if line_count > 0:
        invalid_requested = sorted(
            value
            for value in (
                requested_lines
                | insertion_after_lines
            )
            if value < 1
            or value > line_count
        )

        invalid_marking = sorted(
            value
            for value in marking_lines
            if (
                value < 1
                or (
                    value > line_count
                    and value not in resulting_inserted_lines
                )
            )
        )

        if invalid_requested:
            errors.append(
                "question references code line(s) outside the supplied code "
                f"block: {invalid_requested}; line_count={line_count}"
            )

        if invalid_marking:
            errors.append(
                "marking_guidance references code line(s) outside the supplied "
                f"code block: {invalid_marking}; line_count={line_count}"
            )

    referenced_supplied_visual_types = _referenced_supplied_visual_types(
        question_text_value
    )

    actual_visual_type = normalize_visual_requirement(
        question.get(
            "visual_requirement",
            "none",
        )
    )

    incompatible_supplied_visual_types = sorted(
        visual_type
        for visual_type in referenced_supplied_visual_types
        if visual_type != actual_visual_type
    )

    if incompatible_supplied_visual_types:
        errors.append(
            "question explicitly refers to supplied visual stimulus type(s) "
            "that do not match visual_requirement: referenced="
            f"{incompatible_supplied_visual_types!r}, "
            f"supplied={actual_visual_type!r}"
        )

    return list(
        dict.fromkeys(
            errors
        )
    )


_validate_generated_payload_v241 = validate_generated_payload


def validate_generated_payload(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:
    """
    Add generic referential-consistency checks on top of the v2.41
    schema/visual validation chain.
    """
    result = _validate_generated_payload_v241(
        payload,
        request_payload,
    )

    extra_errors: list[str] = []
    diagnostics: list[dict[str, Any]] = []

    questions = payload.get(
        "questions",
        [],
    )

    if not isinstance(
        questions,
        list,
    ):
        questions = []

    for position, question in enumerate(
        questions,
        start=1,
    ):
        if not isinstance(
            question,
            dict,
        ):
            continue

        qid = str(
            question.get(
                "generated_question_id",
                f"Q{position}",
            )
            or f"Q{position}"
        )

        errors = _generic_referential_consistency_errors(
            question
        )

        diagnostics.append(
            {
                "generated_question_id": qid,
                "errors": errors,
            }
        )

        extra_errors.extend(
            f"{qid}: {message}"
            for message in errors
        )

    if extra_errors:
        result.setdefault(
            "errors",
            [],
        ).extend(
            extra_errors
        )

        result["errors"] = list(
            dict.fromkeys(
                result["errors"]
            )
        )

        result["valid"] = False

    result[
        "generic_referential_consistency"
    ] = {
        "status": (
            "PASS"
            if not extra_errors
            else "FAIL"
        ),
        "errors": extra_errors,
        "diagnostics": diagnostics,
    }

    return result

# ================================================================
# v2.44 — TRUTH-TABLE RESPONSE SCAFFOLD + NO UNKNOWN GATE SLOTS
# ================================================================
# Two deterministic release controls:
#   1) if learner wording asks to complete a truth table while another visual
#      (for example a logic circuit) is the supplied stimulus, attach a blank
#      response scaffold automatically;
#   2) reject L1/L2/L3/G1/... style unknown-gate-position tasks so the LLM
#      regenerates a renderable assessment form instead.

_TRUTH_TABLE_COMPLETION_PATTERN = re.compile(
    r"\b(?:complete|fill(?:\s+in)?|finish|draw|construct|create|produce)\s+"
    r"(?:the\s+|this\s+|a\s+|following\s+|given\s+|shown\s+|provided\s+)?"
    r"(?:truth\s+)?table\b"
    r"|\b(?:truth\s+)?table\b[^.!?\n]{0,45}"
    r"\b(?:is\s+|are\s+|should\s+be\s+|must\s+be\s+|to\s+be\s+)?"
    r"(?:complete(?:d)?|fill(?:ed)?(?:\s+in)?|finish(?:ed)?|drawn|constructed|created|produced)\b",
    flags=re.IGNORECASE,
)

_GATE_SLOT_LABEL_PATTERN = re.compile(
    r"\b(?:L|G)\s*\d+\b",
    flags=re.IGNORECASE,
)

_GATE_SLOT_ASSIGNMENT_PATTERN = re.compile(
    r"\b(?:L|G)\s*\d+\s*=\s*(?:AND|OR|NOT|NAND|NOR|XOR)\b",
    flags=re.IGNORECASE,
)

_GATE_SLOT_TASK_PATTERNS = [
    re.compile(
        r"\b(?:state|give|name|identify|choose|select|determine)\b"
        r"[^.!?\n]{0,120}\b(?:logic\s+)?gate\b"
        r"[^.!?\n]{0,120}\b(?:label|position|slot)s?\b",
        flags=re.IGNORECASE,
    ),
    re.compile(
        r"\b(?:logic\s+)?gate\b[^.!?\n]{0,120}"
        r"\b(?:placed|put|goes|belongs|used)\b"
        r"[^.!?\n]{0,100}\b(?:label|position|slot)s?\b",
        flags=re.IGNORECASE,
    ),
    re.compile(
        r"\b(?:label|position|slot)s?\b[^.!?\n]{0,120}"
        r"\b(?:AND|OR|NOT|NAND|NOR|XOR)\b",
        flags=re.IGNORECASE,
    ),
]

_TRUTH_ASSIGNMENT_PATTERN = re.compile(
    r"\b([A-Za-z][A-Za-z0-9_]*)\s*=\s*([01])\b"
)


def _question_requests_truth_table_completion(
    question_text_value: Any,
) -> bool:
    return bool(
        _TRUTH_TABLE_COMPLETION_PATTERN.search(
            str(question_text_value or "")
        )
    )


def _marking_assignment_rows(
    marking_guidance: Any,
) -> tuple[list[str], list[dict[str, int]]]:
    """Read only variable/value facts; never expose expected output values."""
    variable_order: list[str] = []
    rows: list[dict[str, int]] = []

    if not isinstance(marking_guidance, list):
        return variable_order, rows

    for item in marking_guidance:
        if not isinstance(item, dict):
            continue

        criterion = str(item.get("criterion", "") or "")
        matches = _TRUTH_ASSIGNMENT_PATTERN.findall(criterion)
        if len(matches) < 2:
            continue

        row: dict[str, int] = {}
        for name, value in matches:
            clean_name = str(name).strip()
            if clean_name not in variable_order:
                variable_order.append(clean_name)
            row[clean_name] = int(value)

        if row:
            rows.append(row)

    return variable_order, rows


def _truth_table_output_label(
    question: dict[str, Any],
    variable_order: list[str],
) -> str:
    visual = question.get("visual", {})
    if isinstance(visual, dict):
        spec = visual.get("spec", {})
        if isinstance(spec, dict):
            output = spec.get("output", {})
            if isinstance(output, dict):
                label = str(output.get("label", "") or "").strip()
                if label:
                    return label

    text_value = str(question.get("question_text", "") or "")
    text_patterns = [
        re.compile(r"\boutput\s+([A-Za-z][A-Za-z0-9_]*)\b", re.IGNORECASE),
        re.compile(r"\btruth\s+table\s+for\s+([A-Za-z][A-Za-z0-9_]*)\b", re.IGNORECASE),
        re.compile(r"\btable\s+(?:below\s+)?for\s+([A-Za-z][A-Za-z0-9_]*)\b", re.IGNORECASE),
    ]
    for pattern in text_patterns:
        match = pattern.search(text_value)
        if match:
            return match.group(1)

    return variable_order[-1] if variable_order else "Q"


def _truth_table_input_labels(
    question: dict[str, Any],
    output_label: str,
    variable_order: list[str],
) -> list[str]:
    visual = question.get("visual", {})
    if isinstance(visual, dict):
        spec = visual.get("spec", {})
        if isinstance(spec, dict):
            inputs = spec.get("inputs", [])
            if isinstance(inputs, list):
                clean_inputs = [
                    str(value or "").strip()
                    for value in inputs
                    if str(value or "").strip()
                ]
                if clean_inputs:
                    return clean_inputs

    output_norm = normalize_text(output_label)
    return [
        name
        for name in variable_order
        if normalize_text(name) != output_norm
    ]


def _binary_input_rows(input_count: int) -> list[list[int]]:
    if input_count <= 0 or input_count > 4:
        return []
    row_count = 2 ** input_count
    rows: list[list[int]] = []
    for number in range(row_count):
        bits = [
            (number >> shift) & 1
            for shift in range(input_count - 1, -1, -1)
        ]
        rows.append(bits)
    return rows


def _truth_table_response_scaffold_from_question(
    question: dict[str, Any],
) -> dict[str, Any] | None:
    """
    Build a learner-only blank truth-table response area.

    Input combinations may be taken from explicit marking facts or generated
    from the supplied logic-circuit input labels. Expected output values are
    NEVER copied into the scaffold.
    """
    if not _question_requests_truth_table_completion(
        question.get("question_text", "")
    ):
        return None

    required_type = normalize_visual_requirement(
        question.get("visual_requirement", "none")
    )

    # When truth_table is itself the supplied visual, that visual is already
    # the learner response object and must contain blank cells.
    if required_type == "truth_table":
        return None

    variable_order, marking_rows = _marking_assignment_rows(
        question.get("marking_guidance", [])
    )
    output_label = _truth_table_output_label(
        question,
        variable_order,
    )
    input_labels = _truth_table_input_labels(
        question,
        output_label,
        variable_order,
    )

    if not input_labels:
        return None

    rows: list[list[Any]] = []
    seen_inputs: set[tuple[int, ...]] = set()

    for row in marking_rows:
        if not all(label in row for label in input_labels):
            continue
        input_values = tuple(int(row[label]) for label in input_labels)
        if input_values in seen_inputs:
            continue
        seen_inputs.add(input_values)
        rows.append([*input_values, ""])

    if not rows:
        rows = [
            [*values, ""]
            for values in _binary_input_rows(len(input_labels))
        ]

    if not rows:
        return None

    return {
        "type": "truth_table",
        "columns": [*input_labels, output_label],
        "rows": rows,
        "caption": "Complete the truth table",
        "learner_completed_columns": [output_label],
        "deterministic": True,
        "contains_answers": False,
    }


def _ensure_truth_table_response_scaffolds(
    payload: dict[str, Any],
) -> None:
    questions = payload.get("questions", [])
    if not isinstance(questions, list):
        return

    for question in questions:
        if not isinstance(question, dict):
            continue

        scaffold = _truth_table_response_scaffold_from_question(question)
        if scaffold is not None:
            # Canonical cross-notebook contract. Keep the legacy alias while
            # existing PDF/frontend consumers migrate.
            question["learner_response_scaffold"] = scaffold
            question["response_scaffold"] = scaffold
        elif not _question_requests_truth_table_completion(
            question.get("question_text", "")
        ):
            # Avoid stale scaffold content after targeted regeneration.
            question.pop("learner_response_scaffold", None)
            question.pop("response_scaffold", None)


def _truth_table_response_scaffold_errors(
    question: dict[str, Any],
) -> list[str]:
    if not _question_requests_truth_table_completion(
        question.get("question_text", "")
    ):
        return []

    required_type = normalize_visual_requirement(
        question.get("visual_requirement", "none")
    )

    if required_type == "truth_table":
        visual = question.get("visual", {})
        spec = visual.get("spec", {}) if isinstance(visual, dict) else {}
        rows = spec.get("rows", []) if isinstance(spec, dict) else []
        has_blank = any(
            cell in ("", None)
            for row in rows
            if isinstance(row, list)
            for cell in row
        )
        return [] if has_blank else [
            "truth-table completion task must leave at least one learner-completed cell blank"
        ]

    scaffold = question.get(
        "learner_response_scaffold",
        question.get("response_scaffold"),
    )
    if not isinstance(scaffold, dict):
        return [
            "truth-table completion task requires a deterministic blank response_scaffold"
        ]

    columns = scaffold.get("columns", [])
    rows = scaffold.get("rows", [])
    if not isinstance(columns, list) or len(columns) < 2:
        return ["truth-table response_scaffold has invalid columns"]
    if not isinstance(rows, list) or not rows:
        return ["truth-table response_scaffold has no rows"]

    width = len(columns)
    for row in rows:
        if not isinstance(row, list) or len(row) != width:
            return ["truth-table response_scaffold rows are not rectangular"]
        if row[-1] not in ("", None):
            return [
                "truth-table response_scaffold leaks a completed learner output value"
            ]

    return []


def _forbidden_unknown_gate_slot_errors(
    question: dict[str, Any],
) -> list[str]:
    q_text = str(question.get("question_text", "") or "")
    guidance = question.get("marking_guidance", [])
    guidance_text = " ".join(
        str(item.get("criterion", "") or "")
        for item in guidance
        if isinstance(item, dict)
    )
    corpus = q_text + "\n" + guidance_text

    explicit_assignment = bool(
        _GATE_SLOT_ASSIGNMENT_PATTERN.search(corpus)
    )
    gate_slot_command = any(
        pattern.search(q_text)
        for pattern in _GATE_SLOT_TASK_PATTERNS
    )
    labelled_slots = bool(
        _GATE_SLOT_LABEL_PATTERN.search(corpus)
    )

    if explicit_assignment or (gate_slot_command and labelled_slots):
        return [
            "unknown labelled logic-gate slot tasks (for example L1/L2/L3/L4 or G1/G2 gate-type assignment) are forbidden because they are not reliably renderable; regenerate using a fully specified circuit task"
        ]

    return []


_validate_generated_payload_v244 = validate_generated_payload


def validate_generated_payload(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:
    """v2.44 final wrapper: response scaffolds + no unknown gate slots."""
    _ensure_truth_table_response_scaffolds(payload)

    result = _validate_generated_payload_v244(
        payload,
        request_payload,
    )

    extra_errors: list[str] = []
    diagnostics: list[dict[str, Any]] = []

    questions = payload.get("questions", [])
    if not isinstance(questions, list):
        questions = []

    for position, question in enumerate(questions, start=1):
        if not isinstance(question, dict):
            continue

        qid = str(
            question.get("generated_question_id", f"Q{position}")
            or f"Q{position}"
        )

        scaffold_errors = _truth_table_response_scaffold_errors(question)
        gate_slot_errors = _forbidden_unknown_gate_slot_errors(question)
        question_errors = scaffold_errors + gate_slot_errors

        diagnostics.append(
            {
                "generated_question_id": qid,
                "truth_table_completion": _question_requests_truth_table_completion(
                    question.get("question_text", "")
                ),
                "response_scaffold_present": isinstance(
                    question.get(
                        "learner_response_scaffold",
                        question.get("response_scaffold"),
                    ),
                    dict,
                ),
                "errors": question_errors,
            }
        )

        extra_errors.extend(
            f"{qid}: {message}"
            for message in question_errors
        )

    if extra_errors:
        result.setdefault("errors", []).extend(extra_errors)
        result["errors"] = list(dict.fromkeys(result["errors"]))
        result["valid"] = False

    result["v244_truth_table_and_gate_slot_validation"] = {
        "status": "PASS" if not extra_errors else "FAIL",
        "errors": extra_errors,
        "diagnostics": diagnostics,
    }

    return result


# ================================================================
# v2.45 — SUPPORTED LOGIC-CIRCUIT SHAPE GATE
# ================================================================
_MULTI_CIRCUIT_CLAIM_PATTERN = re.compile(
    r"\b(?:two|three|four|five|multiple|several|\d+)\s+"
    r"(?:separate\s+|different\s+|labelled\s+|labeled\s+)?"
    r"(?:logic\s+)?circuits?\b",
    flags=re.IGNORECASE,
)

_CIRCUIT_CHOICE_PATTERN = re.compile(
    r"\b(?:which|choose|select|identify|write\s+the\s+label)\b"
    r"[^.!?\n]{0,140}\bcircuits?\b"
    r"|\bcircuits?\b[^.!?\n]{0,140}"
    r"\b(?:which|choose|select|identify|label)\b",
    flags=re.IGNORECASE,
)


def _unsupported_multi_circuit_choice_errors(
    question: dict[str, Any],
) -> list[str]:
    if normalize_visual_requirement(
        question.get("visual_requirement", "none")
    ) != "logic_gate_diagram":
        return []

    question_text_value = str(
        question.get("question_text", "") or ""
    )

    if not (
        _MULTI_CIRCUIT_CLAIM_PATTERN.search(question_text_value)
        or _CIRCUIT_CHOICE_PATTERN.search(question_text_value)
    ):
        return []

    return [
        "question requests a choice between multiple separate logic circuits, "
        "but logic_gate_diagram supports one output-connected circuit per question; "
        "regenerate as a fully specified single-circuit task"
    ]


_validate_generated_payload_v245_base = validate_generated_payload


def validate_generated_payload(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:
    result = _validate_generated_payload_v245_base(
        payload,
        request_payload,
    )

    extra_errors: list[str] = []
    diagnostics: list[dict[str, Any]] = []
    questions = payload.get("questions", [])
    if not isinstance(questions, list):
        questions = []

    for position, question in enumerate(questions, start=1):
        if not isinstance(question, dict):
            continue
        qid = str(
            question.get("generated_question_id", f"Q{position}")
            or f"Q{position}"
        )
        errors = _unsupported_multi_circuit_choice_errors(question)
        diagnostics.append({"generated_question_id": qid, "errors": errors})
        extra_errors.extend(f"{qid}: {message}" for message in errors)

    if extra_errors:
        result.setdefault("errors", []).extend(extra_errors)
        result["errors"] = list(dict.fromkeys(result["errors"]))
        result["valid"] = False

    result["v245_supported_logic_circuit_shape"] = {
        "status": "PASS" if not extra_errors else "FAIL",
        "errors": extra_errors,
        "diagnostics": diagnostics,
    }
    return result


# ================================================================
# v2.46 — EXPLICIT ALLOWED-GATE SPECIAL-INSTRUCTION ENFORCEMENT
# ================================================================
_SUPPORTED_LOGIC_GATE_TYPES = ("AND", "OR", "NOT", "NAND", "NOR", "XOR")
_GATE_TYPE_ALTERNATION = "|".join(_SUPPORTED_LOGIC_GATE_TYPES)
_GATE_TYPE_LIST = (
    rf"(?:{_GATE_TYPE_ALTERNATION})"
    rf"(?:\s*(?:,|/|&|\band\b|\bor\b)?\s+"
    rf"(?:{_GATE_TYPE_ALTERNATION}))*"
)
_EXPLICIT_GATE_ONLY_PATTERNS = [
    re.compile(
        rf"\bonly\s+(?P<gates>{_GATE_TYPE_LIST})\s+(?:logic\s+)?gates?\b",
        flags=re.IGNORECASE,
    ),
    re.compile(
        rf"\b(?P<gates>{_GATE_TYPE_LIST})\s+(?:logic\s+)?gates?"
        r"(?:\s+questions?)?\s+only\b",
        flags=re.IGNORECASE,
    ),
    re.compile(
        rf"\bonly\s+questions?\s+(?:about|on|using|for)\s+"
        rf"(?P<gates>{_GATE_TYPE_LIST})\s+(?:logic\s+)?gates?\b",
        flags=re.IGNORECASE,
    ),
]


def _request_special_instruction_text(
    request_payload: dict[str, Any],
) -> str:
    if not isinstance(request_payload, dict):
        return ""

    direct = str(request_payload.get("special_instructions", "") or "").strip()
    if direct:
        return direct

    filters = request_payload.get("assessment_filters", {})
    if isinstance(filters, dict):
        nested = str(filters.get("special_instructions", "") or "").strip()
        if nested:
            return nested

    policy = request_payload.get("special_instruction_policy", {})
    if isinstance(policy, dict):
        return str(policy.get("raw_instruction", "") or "").strip()

    return ""


def _explicit_allowed_gate_types(
    request_payload: dict[str, Any],
) -> tuple[str, ...]:
    instruction = _request_special_instruction_text(request_payload)
    matched_groups = [
        match.group("gates")
        for pattern in _EXPLICIT_GATE_ONLY_PATTERNS
        for match in pattern.finditer(instruction)
    ]

    if not matched_groups:
        return ()

    matched_types: set[str] = set()
    for group in matched_groups:
        tokens = re.findall(
            rf"\b(?:{_GATE_TYPE_ALTERNATION})\b",
            group,
            flags=re.IGNORECASE,
        )
        for token_index, gate_type in enumerate(tokens):
            # Lower-case 'and' between already-listed gate names is a
            # conjunction, while a leading/upper-case AND is a gate type.
            if gate_type == "and" and token_index > 0:
                continue
            matched_types.add(gate_type.upper())
    return tuple(
        gate_type
        for gate_type in _SUPPORTED_LOGIC_GATE_TYPES
        if gate_type in matched_types
    )


def _explicit_gate_only_instruction_errors(
    question: dict[str, Any],
    allowed_gate_types: tuple[str, ...],
) -> list[str]:
    if not allowed_gate_types:
        return []

    if normalize_visual_requirement(
        question.get("visual_requirement", "none")
    ) != "logic_gate_diagram":
        return []

    visual = question.get("visual", {})
    spec = visual.get("spec", {}) if isinstance(visual, dict) else {}
    gates = spec.get("gates", []) if isinstance(spec, dict) else []
    actual_types = sorted({
        str(gate.get("type", "") or "").strip().upper()
        for gate in gates
        if isinstance(gate, dict)
        and str(gate.get("type", "") or "").strip()
    })

    if actual_types and all(
        gate_type in allowed_gate_types
        for gate_type in actual_types
    ):
        return []

    allowed_label = ", ".join(allowed_gate_types)
    actual_label = ", ".join(actual_types) if actual_types else "no gate type"
    return [
        f"special instructions allow only these gate types: {allowed_label}; but "
        f"the logic_gate_diagram contains {actual_label}; regenerate this "
        f"question using only the allowed gate spec types: {allowed_label}"
    ]


_validate_generated_payload_v246_base = validate_generated_payload


def validate_generated_payload(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:
    result = _validate_generated_payload_v246_base(payload, request_payload)
    allowed_gate_types = _explicit_allowed_gate_types(request_payload)
    extra_errors: list[str] = []
    diagnostics: list[dict[str, Any]] = []
    questions = payload.get("questions", [])
    if not isinstance(questions, list):
        questions = []

    for position, question in enumerate(questions, start=1):
        if not isinstance(question, dict):
            continue
        qid = str(
            question.get("generated_question_id", f"Q{position}")
            or f"Q{position}"
        )
        errors = _explicit_gate_only_instruction_errors(
            question,
            allowed_gate_types,
        )
        diagnostics.append({
            "generated_question_id": qid,
            "allowed_gate_types": list(allowed_gate_types),
            "errors": errors,
        })
        extra_errors.extend(f"{qid}: {message}" for message in errors)

    if extra_errors:
        result.setdefault("errors", []).extend(extra_errors)
        result["errors"] = list(dict.fromkeys(result["errors"]))
        result["valid"] = False

    result["v246_explicit_gate_only_enforcement"] = {
        "status": "PASS" if not extra_errors else "FAIL",
        "allowed_gate_types": list(allowed_gate_types),
        "errors": extra_errors,
        "diagnostics": diagnostics,
    }
    return result



## 10. Semantic quality checks

This version also enforces **question answerability** and **mark-scheme specificity**:
- a state variable updated from its previous value must have an explicit initial value/state;
- any requested next/final state must state the operation that causes the transition;
- objective marking points must include the exact expected answer/value/code, not generic "correct value" wording;
- question-local semantic failures are regenerated selectively by `plan_index`, so already-valid questions are not re-generated.



## Final Notebook 06 quality/cost architecture — v2.29

The agreed production architecture is **one intelligent generation LLM
surrounded by cheap deterministic safeguards**.

```text
Approved Agent 1 topics + hard UI controls + raw special instructions
                              ↓
                   Deterministic preflight
          (counts/marks/paper/scope + explicit hints)
                              ↓
                 Compact AQA RAG grounding
                              ↓
                    Token/context preflight
                              ↓
             ONE CONFIG-SELECTED LLM CALL
          ┌──────────────────────────────────┐
          │ understand special instructions  │
          │ generate questions + MS          │
          │ return visual specs              │
          │ return instruction interpretation│
          │ return compliance report         │
          └──────────────────────────────────┘
                              ↓
                 Deterministic Python checks
                              +
            local MiniLM topic grounding/diversity
                              +
                deterministic visual rendering
                              ↓
                     PASS / REVIEW / FAIL
                         ↓            ↓
                       HITL      targeted repair
                         ↓            ↓
                      RELEASE ← revalidate
```

### Cost policy

- **No separate LLM instruction-interpreter call.**
- **No secondary LLM semantic-review call.**
- A normal 10-question quiz targets **one generation call**.
- Operational batch splitting happens only if token/context/provider safeguards
  require it.
- If a question-local validation failure is found, only the affected
  `plan_index` question(s) are regenerated when they can be resolved safely.
- MiniLM, lexical checks, task-family checks, structural validation and visual
  rendering run locally.

### Special-instruction policy

`special_instructions` can contain arbitrary natural language. The LLM, not a
large regex rule-set, owns semantic understanding. A small deterministic helper
only promotes high-confidence exact constraints before generation so obvious
requirements can shape the blueprint without another API call.

The same LLM response must return:
- `instruction_interpretation` — what it understood;
- `special_instruction_compliance` — whether each interpreted requirement was
  followed and where.

These self-reported fields are diagnostic, not proof. Python independently
checks everything objective (counts, marks, roles when explicitly resolved,
paper/topic/code/visual constraints, etc.). Qualitative requirements remain
visible to the mandatory HITL reviewer.

### Topic grounding and diversity

Topic grounding remains evidence-based rather than keyword-only. Local MiniLM
compares each generated question with approved topic/lesson evidence and
reviewed same-reference assessment-bank examples. This avoids brittle
topic-specific exceptions.

Quiz diversity is also checked with MiniLM + lexical/task-family signals so
different `assessment_pattern` labels cannot hide near-duplicate learner tasks.
No current transcript, question, array value, RLE example, or 3/7 split is
hardcoded into the validator.

### Final qualitative gate

HITL remains mandatory. A candidate cannot be released merely because the LLM
claims its own instructions were satisfied. The human reviewer sees the same
special-instruction interpretation/compliance report alongside deterministic
validation signals.


In [ ]:
def semantic_token_set(
    value: Any,
) -> set[str]:
    return set(
        re.findall(
            r"[a-z0-9_]+",
            normalize_text(
                value
            ),
        )
    )


def token_jaccard(
    left: Any,
    right: Any,
) -> float:
    left_tokens = semantic_token_set(
        left
    )

    right_tokens = semantic_token_set(
        right
    )

    if (
        not left_tokens
        and not right_tokens
    ):
        return 1.0

    if (
        not left_tokens
        or not right_tokens
    ):
        return 0.0

    return (
        len(
            left_tokens
            & right_tokens
        )
        / len(
            left_tokens
            | right_tokens
        )
    )


def lexical_similarity(
    left: Any,
    right: Any,
) -> float:
    left_text = normalize_text(
        left
    )

    right_text = normalize_text(
        right
    )

    if not left_text or not right_text:
        return 0.0

    return SequenceMatcher(
        None,
        left_text,
        right_text,
    ).ratio()



# ================================================================
# LOCAL MINILM SEMANTIC QUALITY BACKEND
# ================================================================
# Local sentence-transformer inference only; no extra LLM/API reviewer.
# ================================================================

MINILM_MODEL_NAME = str(
    os.getenv(
        "AGENT2_MINILM_MODEL",
        "sentence-transformers/all-MiniLM-L6-v2",
    )
    or "sentence-transformers/all-MiniLM-L6-v2"
).strip()

_MINILM_MODEL: Any = None
_MINILM_MODEL_LOAD_ATTEMPTED = False
_MINILM_MODEL_ERROR = ""
_MINILM_EMBED_CACHE: dict[str, np.ndarray] = {}


def _get_minilm_model() -> Any:
    global _MINILM_MODEL
    global _MINILM_MODEL_LOAD_ATTEMPTED
    global _MINILM_MODEL_ERROR

    if _MINILM_MODEL_LOAD_ATTEMPTED:
        return _MINILM_MODEL

    _MINILM_MODEL_LOAD_ATTEMPTED = True

    try:
        from sentence_transformers import SentenceTransformer

        _MINILM_MODEL = SentenceTransformer(
            MINILM_MODEL_NAME
        )
    except Exception as exc:
        _MINILM_MODEL = None
        _MINILM_MODEL_ERROR = (
            f"{type(exc).__name__}: {exc}"
        )

    return _MINILM_MODEL


def _minilm_embeddings(
    values: list[Any],
) -> tuple[np.ndarray | None, str]:
    texts = [
        re.sub(
            r"\s+",
            " ",
            str(
                value or ""
            ).strip(),
        )
        for value in values
    ]

    if not texts:
        return (
            np.zeros(
                (
                    0,
                    0,
                ),
                dtype=float,
            ),
            "minilm",
        )

    model = _get_minilm_model()

    if model is None:
        return (
            None,
            (
                "lexical_fallback"
                + (
                    f": {_MINILM_MODEL_ERROR}"
                    if _MINILM_MODEL_ERROR
                    else ""
                )
            ),
        )

    missing = [
        text
        for text in dict.fromkeys(
            texts
        )
        if text
        and text not in _MINILM_EMBED_CACHE
    ]

    if missing:
        try:
            encoded = model.encode(
                missing,
                convert_to_numpy=True,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
        except Exception as exc:
            return (
                None,
                f"lexical_fallback: {type(exc).__name__}: {exc}",
            )

        for text, vector in zip(
            missing,
            encoded,
        ):
            _MINILM_EMBED_CACHE[
                text
            ] = np.asarray(
                vector,
                dtype=float,
            )

    nonblank_vector = next(
        (
            vector
            for vector in _MINILM_EMBED_CACHE.values()
        ),
        None,
    )

    if nonblank_vector is None:
        return (
            None,
            "lexical_fallback: no nonblank text available",
        )

    zero = np.zeros_like(
        nonblank_vector,
        dtype=float,
    )

    matrix = np.vstack(
        [
            (
                _MINILM_EMBED_CACHE[
                    text
                ]
                if text
                else zero
            )
            for text in texts
        ]
    )

    return (
        matrix,
        "minilm",
    )


def question_requests_final_response(
    question_text_value: str,
) -> bool:
    normalized = normalize_text(
        question_text_value
    )

    final_terms = [
        "final value",
        "final values",
        "final result",
        "final results",
        "final answer",
        "final output",
        "final outputs",
        "value printed",
        "values printed",
    ]

    if not any(
        normalize_text(
            term
        )
        in normalized
        for term in final_terms
    ):
        return False

    response_verbs = [
        "state",
        "give",
        "write",
        "identify",
        "record",
        "enter",
        "provide",
        "determine",
        "calculate",
    ]

    if any(
        re.search(
            rf"\b{re.escape(verb)}\b",
            normalized,
        )
        for verb in response_verbs
    ):
        return True

    question_patterns = [
        r"\bwhat\s+(?:is|are|will\s+be)\s+(?:the\s+)?(?:final|printed|output)",
        r"\bwhat\s+value(?:s)?\b",
        r"\bwhich\s+value(?:s)?\b",
    ]

    return any(
        re.search(
            pattern,
            normalized,
        )
        for pattern in question_patterns
    )


def official_style_comparison_text(example: Any) -> str:
    """Compare novelty against the source question itself, not its calibration metadata."""
    text = str(
        example
        or ""
    ).strip()

    marker = "\nQuestion:"

    if marker in text:
        candidate = text.rsplit(
            marker,
            1,
        )[-1].strip()

        if candidate:
            return candidate

    return text


SPECIALISED_RUNTIME_TERM_PATTERNS = {
    "short-circuit evaluation": r"\bshort[- ]circuit(?:ing| evaluation)?\b",
    "left-to-right evaluation": r"\bleft[- ]to[- ]right\s+evaluation\b",
    "pass-by-reference semantics": r"\bpass(?:ed)?\s+by\s+reference\b",
    "pass-by-value semantics": r"\bpass(?:ed)?\s+by\s+value\b",
    "memory-address semantics": r"\bmemory\s+address(?:es)?\b",
    "runtime stack semantics": r"\b(?:call\s+stack|runtime\s+stack|stack\s+frame)\b",
}


def _question_support_corpus(
    blueprint_item: dict[str, Any],
    request_payload: dict[str, Any],
) -> str:
    pieces: list[str] = []

    # Scope support comes from the approved lesson/topic evidence only. Style
    # examples calibrate difficulty/task shape and must NEVER expand content.
    covered_norms = {
        normalize_text(
            item.get(
                "topic",
                "",
            )
        )
        for item in safe_list(
            blueprint_item.get(
                "covered_topics",
                [],
            )
        )
        if isinstance(item, dict)
    }

    anchor_norm = normalize_text(
        blueprint_item.get(
            "topic",
            "",
        )
    )

    if anchor_norm:
        covered_norms.add(
            anchor_norm
        )

    pieces.extend(
        str(item.get("topic", "") or "")
        for item in safe_list(
            blueprint_item.get(
                "covered_topics",
                [],
            )
        )
        if isinstance(item, dict)
    )
    pieces.append(
        str(
            blueprint_item.get(
                "topic",
                "",
            )
            or ""
        )
    )

    for grounding in safe_list(
        request_payload.get(
            "topic_grounding",
            [],
        )
    ):
        if not isinstance(grounding, dict):
            continue

        if normalize_text(
            grounding.get(
                "topic",
                "",
            )
        ) not in covered_norms:
            continue

        pieces.extend(
            str(value or "")
            for value in safe_list(
                grounding.get(
                    "lesson_evidence",
                    [],
                )
            )
        )

    return "\n".join(
        piece
        for piece in pieces
        if str(piece).strip()
    )


def _explicit_one_mark_multi_demand(
    question_text_value: str,
) -> bool:
    text = normalize_text(
        question_text_value
    )

    # High-confidence cases only: a selection/identification/state task followed
    # by an additional justification/explanation/reason demand.
    return bool(
        re.search(
            r"\b(?:state|identify|select|choose|name)\b[^.?!;]{0,140}\b(?:and|then)\s+(?:justify|explain|describe|give\s+(?:a|one)\s+reason)\b",
            text,
        )
        or re.search(
            r"\b(?:justify|explain)\b[^.?!;]{0,140}\b(?:and|then)\s+(?:state|identify|select|choose|name)\b",
            text,
        )
    )


def _collapsed_multistatement_code(
    question_text_value: str,
) -> bool:
    text = str(
        question_text_value or ""
    )

    code_keyword_hits = len(
        re.findall(
            r"\b(?:IF|ELSE|WHILE|FOR|RETURN|PRINT|INPUT|END\s*IF|END\s*WHILE|END\s*FOR)\b",
            text,
            flags=re.IGNORECASE,
        )
    )

    assignment_hits = len(
        re.findall(
            r"(?:<-|←|(?<![<>=!])=(?!=))",
            text,
        )
    )

    statement_signal = (
        code_keyword_hits
        + min(assignment_hits, 3)
    )

    if statement_signal < 3:
        return False

    # A proper code fence or multiple real line breaks is enough.
    if "```" in text:
        return False

    return text.count("\n") < 2



_OBJECTIVE_MARKING_GENERIC_PATTERNS = [
    r"\bcorrect\s+(?:decimal|denary|binary|final|initial|new|next|row|column|index|count|value|number|bit|output|result|length|place\s+value|pattern)\b",
    r"\bcorrect\s+(?:count|value|number|index|pattern)\s+of\b",
    r"\bidentify\s+the\s+(?:decimal|denary|binary|row|column|index|count|value|number|bit|place\s+value)\b",
    r"\bcalculate\s+the\s+(?:decimal|denary|binary|row|column|index|count|value|number)\b",
    r"\bstate\s+the\s+(?:decimal|denary|binary|row|column|index|count|value|number|bit|place\s+value)\b",
]


def _criterion_without_mark_prefix(value: Any) -> str:
    text_value = str(value or "").strip()
    text_value = re.sub(
        r"^\s*\d+\s*marks?\s*(?:for|:|-)?\s*",
        "",
        text_value,
        flags=re.IGNORECASE,
    )
    return text_value.strip()


def _criterion_is_objective(value: Any) -> bool:
    text_value = normalize_text(
        _criterion_without_mark_prefix(value)
    )

    if not text_value:
        return False

    return any(
        re.search(
            pattern,
            text_value,
            flags=re.IGNORECASE,
        )
        for pattern in _OBJECTIVE_MARKING_GENERIC_PATTERNS
    )


def _criterion_has_exact_expected_answer(value: Any) -> bool:
    """
    Conservative specificity check for objectively answerable mark points.

    A criterion is specific when it contains an actual value/literal/code
    response rather than only saying that the student's value must be correct.
    """
    raw = _criterion_without_mark_prefix(value)
    lower = raw.casefold()

    if not raw:
        return False

    # "e.g." / "etc." are explicitly non-exact for a single objective answer.
    if (
        re.search(r"\be\.?\s*g\.?\b", lower)
        or "for example" in lower
        or re.search(r"\betc\.?\b", lower)
    ):
        return False

    # Concrete assignment / correction line.
    if re.search(
        r"\b[A-Za-z_][A-Za-z0-9_]*\s*(?:<-|←|=)\s*[^,;]+",
        raw,
    ):
        return True

    # Explicit binary pattern.
    if re.search(r"\b[01]{2,}\b", raw):
        return True

    # Concrete number. Leading "1 mark" has already been removed.
    if re.search(r"(?<![A-Za-z_])-?\d+(?:\.\d+)?(?![A-Za-z_])", raw):
        return True

    # Common exact literals / selections.
    if re.search(
        r"\b(?:true|false|yes|no|null|none)\b",
        lower,
    ):
        return True

    if re.search(
        r"\b(?:option|answer)\s+[A-F]\b",
        raw,
        flags=re.IGNORECASE,
    ):
        return True

    # Quoted literal/string/token.
    if re.search(r"""["'][^"']+["']""", raw):
        return True

    return False


def _marking_specificity_failures(
    question: dict[str, Any],
) -> list[str]:
    failures: list[str] = []

    guidance = question.get(
        "marking_guidance",
        [],
    )

    if not isinstance(guidance, list):
        return failures

    for index, item in enumerate(
        guidance,
        start=1,
    ):
        if not isinstance(item, dict):
            continue

        criterion = str(
            item.get(
                "criterion",
                "",
            )
            or ""
        ).strip()

        if not criterion:
            continue

        if (
            _criterion_is_objective(criterion)
            and not _criterion_has_exact_expected_answer(criterion)
        ):
            failures.append(
                "Objective marking criterion "
                f"{index} does not contain one exact expected answer/value/code. "
                "Replace generic wording such as 'correct value' with the actual "
                "answer (for example '178', 'row index = 17', or an exact code line)."
            )

    return failures


def _code_assignment_records(
    question_text_value: str,
) -> list[tuple[str, str]]:
    """
    Extract assignment-like statements conservatively from multiline code/prose.
    Comparisons (==, >=, <=, !=) are ignored.
    """
    text_value = str(question_text_value or "").replace("←", "<-")
    records: list[tuple[str, str]] = []

    assignment_pattern = re.compile(
        r"^\s*(?:\d+\s+)?([A-Za-z_][A-Za-z0-9_]*)\s*(<-|=)\s*(.+?)\s*$"
    )

    for raw_line in text_value.splitlines():
        line = raw_line.strip()

        if not line:
            continue

        upper = line.upper()

        if upper.startswith(
            (
                "IF ",
                "ELSE",
                "ENDIF",
                "END IF",
                "WHILE ",
                "ENDWHILE",
                "END WHILE",
                "FOR ",
                "ENDFOR",
                "END FOR",
            )
        ):
            continue

        if any(
            token in line
            for token in [
                "==",
                ">=",
                "<=",
                "!=",
            ]
        ):
            continue

        match = assignment_pattern.match(line)

        if match:
            records.append(
                (
                    match.group(1),
                    match.group(3),
                )
            )

    return records


def _explicitly_initialized_variables(
    question_text_value: str,
) -> set[str]:
    text_value = str(question_text_value or "").replace("←", "<-")
    initialized: set[str] = set()

    # Explicit assignments that do not depend on the same variable.
    for variable, rhs in _code_assignment_records(text_value):
        if not re.search(
            rf"\b{re.escape(variable)}\b",
            rhs,
            flags=re.IGNORECASE,
        ):
            initialized.add(
                variable.casefold()
            )

    # FOR-loop control variables are explicitly initialised by their header.
    for match in re.finditer(
        r"\bFOR\s+([A-Za-z_][A-Za-z0-9_]*)\s*(?:<-|=)\s*",
        text_value,
        flags=re.IGNORECASE,
    ):
        initialized.add(
            match.group(1).casefold()
        )

    # Prose can initialise data/state without an assignment symbol.
    assigned_names = {
        variable
        for variable, _ in _code_assignment_records(text_value)
    }

    for variable in assigned_names:
        pattern = (
            rf"\b{re.escape(variable)}\b"
            r"[^.\n]{0,70}\b"
            r"(?:starts?|begins?|initiali[sz]ed|set|contains?|stores?|holds?|has)\b"
        )

        if re.search(
            pattern,
            text_value,
            flags=re.IGNORECASE,
        ):
            initialized.add(
                variable.casefold()
            )

    return initialized


def _uninitialised_self_referential_updates(
    question_text_value: str,
) -> list[str]:
    initialized = _explicitly_initialized_variables(
        question_text_value
    )

    missing: list[str] = []

    for variable, rhs in _code_assignment_records(
        question_text_value
    ):
        variable_norm = variable.casefold()

        if not re.search(
            rf"\b{re.escape(variable)}\b",
            rhs,
            flags=re.IGNORECASE,
        ):
            continue

        if variable_norm in initialized:
            continue

        if variable_norm not in missing:
            missing.append(
                variable_norm
            )

    return missing


def _missing_state_transition_definition(
    question_text_value: str,
) -> bool:
    """
    Catch questions that ask for a next/final state after an iteration but do
    not state the operation that changes the state.
    """
    text_value = str(question_text_value or "").replace("←", "<-")
    lower = normalize_text(text_value)

    asks_transition = bool(
        re.search(
            r"\b(?:new|next|final)\s+(?:binary\s+)?(?:value|pattern|contents?)\b"
            r".{0,100}\bafter\b.{0,60}\b(?:iteration|step|update)\b",
            lower,
            flags=re.IGNORECASE | re.DOTALL,
        )
        or re.search(
            r"\bwhich\s+bit\b.{0,120}\bchange\b.{0,80}\b(?:iteration|iterates?)\b",
            lower,
            flags=re.IGNORECASE | re.DOTALL,
        )
    )

    if not asks_transition:
        return False

    has_assignment_update = any(
        re.search(
            rf"\b{re.escape(variable)}\b",
            rhs,
            flags=re.IGNORECASE,
        )
        for variable, rhs in _code_assignment_records(text_value)
    )

    if has_assignment_update:
        return False

    explicit_transition_patterns = [
        r"\bincrement(?:s|ed|ing)?\b",
        r"\bdecrement(?:s|ed|ing)?\b",
        r"\bincreas(?:e|es|ed|ing)\s+by\b",
        r"\bdecreas(?:e|es|ed|ing)\s+by\b",
        r"\badd(?:s|ed|ing)?\s+-?\d+\b",
        r"\bsubtract(?:s|ed|ing)?\s+-?\d+\b",
        r"\bright[-\s]?shift\b",
        r"\bleft[-\s]?shift\b",
        r"\bdivid(?:e|es|ed|ing)\s+by\b",
        r"\bmultip(?:ly|lies|lied|lying)\s+by\b",
        r"\btoggl(?:e|es|ed|ing)\b",
        r"\bflip(?:s|ped|ping)?\b",
        r"\bset(?:s|ting)?\b.{0,40}\bto\b",
        r"\bupdat(?:e|es|ed|ing)\b.{0,40}\bby\b",
    ]

    return not any(
        re.search(
            pattern,
            lower,
            flags=re.IGNORECASE,
        )
        for pattern in explicit_transition_patterns
    )


def _question_self_containment_failures(
    question_text_value: str,
) -> list[str]:
    failures: list[str] = []

    missing_variables = _uninitialised_self_referential_updates(
        question_text_value
    )

    for variable in missing_variables:
        failures.append(
            "Question updates "
            f"'{variable}' using its previous value but never gives an initial "
            "value/state for that variable. The question is not fully answerable "
            "without an unstated assumption."
        )

    if _missing_state_transition_definition(
        question_text_value
    ):
        failures.append(
            "Question asks for a changed/new state after an iteration but does "
            "not explicitly state the operation that changes the state (for "
            "example increment by 1, decrement, shift, or another update)."
        )

    return failures


def _open_ended_marking_scheme_warning(
    question_text_value: Any,
    marking_guidance: Any,
) -> str | None:
    """
    Detect when a task clearly permits multiple valid answers but its marking
    guidance looks like a rigid list of sample answers. The caller treats this
    as a targeted regeneration guard; flexible acceptance-based schemes pass.
    """
    q_text = normalize_text(question_text_value)

    open_ended_signals = [
        r"\b(?:suggest|propose)\b",
        r"\b(?:give|state|provide|list|write)\b[^.?!]{0,120}\b(?:example|examples|suitable|valid|possible|different)\b",
        r"\b(?:test data|test inputs?|test cases?|input strings?)\b",
        r"\bfor example\b",
        r"\banswers? may vary\b",
    ]

    if not any(re.search(pattern, q_text) for pattern in open_ended_signals):
        return None

    criteria = [
        normalize_text(item.get("criterion", ""))
        for item in safe_list(marking_guidance)
        if isinstance(item, dict)
        and safe_int(item.get("marks")) > 0
        and str(item.get("criterion", "") or "").strip()
    ]

    if not criteria:
        return None

    guidance_text = " ".join(criteria)
    flexibility_signals = [
        r"\baccept\b",
        r"\ballow\b",
        r"\bany (?:valid|suitable|equivalent|correct)\b",
        r"\bequivalent\b",
        r"\bother (?:valid|suitable|correct)\b",
        r"\banswers? may vary\b",
        r"\bexample answers?\b",
        r"\be\.g\.\b",
        r"\bfor example\b",
    ]

    if any(re.search(pattern, guidance_text) for pattern in flexibility_signals):
        return None

    return (
        "Open-ended task may allow multiple valid responses, but the marking "
        "guidance appears to list fixed sample answers without an acceptance "
        "rule."
    )


def _exam_layout_phrasing_warning(
    question_text_value: Any,
) -> str | None:
    """Flag source-paper UI wording that the generated PDF does not render."""
    q_text = normalize_text(question_text_value)

    layout_phrases = [
        r"\bshade (?:one|a) lozenge\b",
        r"\btick (?:one|a) box\b",
        r"\bcross (?:one|a) box\b",
        r"\bshade (?:one|a) box\b",
        r"\bmark (?:one|a) box\b",
        r"\bwrite (?:your )?answer in the box(?: below)?\b",
    ]

    matched = next(
        (pattern for pattern in layout_phrases if re.search(pattern, q_text)),
        None,
    )

    if matched is None:
        return None

    return (
        "Question contains exam-specific response-layout wording (for example "
        "a lozenge/box instruction) that is not represented by the generated "
        "student material."
    )


_TOPIC_GROUNDING_STOP_TOKENS = {
    "a", "an", "and", "or", "the", "to", "of", "in", "on", "for", "with",
    "from", "by", "as", "is", "are", "be", "this", "that", "these", "those",
    "give", "state", "write", "identify", "explain", "describe", "answer",
    "question", "questions", "mark", "marks", "value", "values", "result",
    "results", "using", "shown", "below", "above", "following", "computer",
    "science", "gcse", "aqa", "paper",
}


def _topic_grounding_tokens(value: Any) -> set[str]:
    return {
        token
        for token in re.findall(r"[a-z0-9_]+", normalize_text(value))
        if len(token) >= 3 and token not in _TOPIC_GROUNDING_STOP_TOKENS
    }


def _anchor_topic_grounding_warning(
    question_text_value: Any,
    blueprint_item: dict[str, Any],
    request_payload: dict[str, Any],
) -> str | None:
    """
    Soft lexical grounding check only.

    This never rejects a quiz. It only surfaces questions that have no clear
    lexical connection to either the approved anchor/covered-topic names or the
    approved lesson evidence. HITL decides whether the question is genuinely
    off-scope. Keeping this soft avoids constraining legitimate paraphrases.
    """
    q_tokens = _topic_grounding_tokens(question_text_value)
    if len(q_tokens) < 3:
        return None

    topic_pieces = [str(blueprint_item.get("topic", "") or "")]
    topic_pieces.extend(
        str(item.get("topic", "") or "")
        for item in safe_list(blueprint_item.get("covered_topics", []))
        if isinstance(item, dict)
    )
    topic_tokens = _topic_grounding_tokens(" ".join(topic_pieces))

    support_corpus = _question_support_corpus(
        blueprint_item,
        request_payload,
    )
    support_tokens = _topic_grounding_tokens(support_corpus)

    topic_overlap = q_tokens & topic_tokens
    support_overlap = q_tokens & support_tokens

    if topic_overlap or len(support_overlap) >= 2:
        return None

    # Do not manufacture semantic certainty from regex/token matching. A zero
    # overlap is only a review signal; HITL remains authoritative.
    return (
        "Question has weak lexical grounding in the approved anchor/covered "
        "topics and lesson evidence. HITL should confirm that the learner task "
        "has not drifted outside the approved topic scope."
    )



def _semantic_topic_grounding_signal(
    question_text_value: Any,
    blueprint_item: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:
    """
    Evidence-based topic grounding using approved lesson/topic evidence plus
    reviewed same-reference question-bank examples already selected for the slot.
    """
    q_text = str(
        question_text_value or ""
    ).strip()

    if not q_text:
        return {
            "status": "UNAVAILABLE",
            "backend": "none",
            "max_similarity": None,
            "lesson_similarity": None,
            "reviewed_bank_similarity": None,
            "reason": "Question text is empty.",
        }

    anchor_norm = normalize_text(
        blueprint_item.get(
            "topic",
            "",
        )
    )

    covered_norms = {
        anchor_norm
    }

    for item in safe_list(
        blueprint_item.get(
            "covered_topics",
            [],
        )
    ):
        if not isinstance(
            item,
            dict,
        ):
            continue

        value = normalize_text(
            item.get(
                "topic",
                "",
            )
        )

        if value:
            covered_norms.add(
                value
            )

    lesson_refs: list[str] = []

    anchor_label = str(
        blueprint_item.get(
            "topic",
            "",
        )
        or ""
    ).strip()

    if anchor_label:
        lesson_refs.append(
            anchor_label
        )

    for item in safe_list(
        blueprint_item.get(
            "covered_topics",
            [],
        )
    ):
        if isinstance(
            item,
            dict,
        ):
            label = str(
                item.get(
                    "topic",
                    "",
                )
                or ""
            ).strip()

            if label:
                lesson_refs.append(
                    label
                )

    for grounding in safe_list(
        request_payload.get(
            "topic_grounding",
            [],
        )
    ):
        if not isinstance(
            grounding,
            dict,
        ):
            continue

        grounding_norm = normalize_text(
            grounding.get(
                "topic",
                "",
            )
        )

        if grounding_norm not in covered_norms:
            continue

        lesson_refs.extend(
            str(
                value or ""
            ).strip()
            for value in safe_list(
                grounding.get(
                    "lesson_evidence",
                    [],
                )
            )
            if str(
                value or ""
            ).strip()
        )

    # These examples were loaded from reviewed/human-approved assessment-bank
    # records for the slot's official reference. They are evidence that a task
    # form is legitimate for that reference, not templates to copy.
    reviewed_refs = [
        official_style_comparison_text(
            value
        )
        for value in safe_list(
            blueprint_item.get(
                "official_question_style_examples",
                [],
            )
        )
        if str(
            value or ""
        ).strip()
    ]

    lesson_refs = list(
        dict.fromkeys(
            value
            for value in lesson_refs
            if value
        )
    )

    reviewed_refs = list(
        dict.fromkeys(
            value
            for value in reviewed_refs
            if value
        )
    )

    all_refs = (
        lesson_refs
        + reviewed_refs
    )

    if not all_refs:
        return {
            "status": "UNAVAILABLE",
            "backend": "none",
            "max_similarity": None,
            "lesson_similarity": None,
            "reviewed_bank_similarity": None,
            "reason": "No approved grounding evidence is available.",
        }

    matrix, backend = _minilm_embeddings(
        [
            q_text,
            *all_refs,
        ]
    )

    if matrix is None:
        return {
            "status": "UNAVAILABLE",
            "backend": backend,
            "max_similarity": None,
            "lesson_similarity": None,
            "reviewed_bank_similarity": None,
            "reason": (
                "MiniLM semantic grounding is unavailable; lexical grounding "
                "and HITL remain active."
            ),
        }

    q_vector = matrix[
        0
    ]

    similarities = np.clip(
        matrix[
            1:
        ]
        @ q_vector,
        -1.0,
        1.0,
    )

    lesson_count = len(
        lesson_refs
    )

    lesson_similarity = (
        float(
            np.max(
                similarities[
                    :lesson_count
                ]
            )
        )
        if lesson_count > 0
        else None
    )

    reviewed_similarity = (
        float(
            np.max(
                similarities[
                    lesson_count:
                ]
            )
        )
        if reviewed_refs
        else None
    )

    max_similarity = float(
        np.max(
            similarities
        )
    )

    q_tokens = _topic_grounding_tokens(
        q_text
    )

    topic_tokens = _topic_grounding_tokens(
        " ".join(
            [
                str(
                    blueprint_item.get(
                        "topic",
                        "",
                    )
                    or ""
                ),
                *[
                    str(
                        item.get(
                            "topic",
                            "",
                        )
                        or ""
                    )
                    for item in safe_list(
                        blueprint_item.get(
                            "covered_topics",
                            [],
                        )
                    )
                    if isinstance(
                        item,
                        dict,
                    )
                ],
            ]
        )
    )

    direct_topic_overlap = bool(
        q_tokens
        & topic_tokens
    )

    target_task_family = normalize_text(
        blueprint_item.get(
            "target_task_family",
            "",
        )
    )

    generated_task_family = normalize_text(
        _style_task_family(
            q_text
        )
    )

    task_family_mismatch = bool(
        target_task_family
        and generated_task_family
        and target_task_family != generated_task_family
    )

    if (
        direct_topic_overlap
        or max_similarity >= 0.43
    ):
        status = "PASS"
        reason = (
            "Question is grounded in approved topic/lesson or reviewed "
            "same-reference assessment-bank evidence."
        )

    elif max_similarity >= 0.30:
        status = "REVIEW"
        reason = (
            "Question has moderate semantic grounding in the approved "
            "topic/lesson/reviewed same-reference evidence. HITL should confirm "
            "that the learner task remains within scope."
        )

    elif (
        max_similarity >= 0.24
        and not task_family_mismatch
    ):
        # Conservative protection for legitimate AQA forms whose surface words
        # differ from the topic label (for example RLE-style questions that are
        # supported by reviewed same-reference assessment evidence).
        status = "REVIEW"
        reason = (
            "Surface similarity is low, but the learner task family is still "
            "consistent with the slot calibration. Keep for HITL instead of "
            "auto-rejecting a potentially legitimate AQA question form."
        )

    else:
        status = "FAIL"
        reason = (
            "Question has weak semantic grounding in the approved anchor/covered "
            "topics, lesson evidence, and reviewed same-reference question-bank "
            "examples, with no calibrated task-family support. Regenerate this "
            "plan_index to remove likely topic drift."
        )

    return {
        "status":
            status,
        "backend":
            backend,
        "max_similarity":
            round(
                max_similarity,
                4,
            ),
        "lesson_similarity":
            (
                round(
                    lesson_similarity,
                    4,
                )
                if lesson_similarity is not None
                else None
            ),
        "reviewed_bank_similarity":
            (
                round(
                    reviewed_similarity,
                    4,
                )
                if reviewed_similarity is not None
                else None
            ),
        "direct_topic_overlap":
            direct_topic_overlap,
        "target_task_family":
            target_task_family,
        "generated_task_family":
            generated_task_family,
        "task_family_mismatch":
            task_family_mismatch,
        "reason":
            reason,
    }



# ================================================================
# COMMAND-WORD / MARK-SCHEME + PATTERN / LEARNER-TASK ALIGNMENT
# ================================================================
# These checks are deliberately conservative and route uncertainty to HITL.
# They never make a second LLM call and they do not hard-code any current
# transcript, topic, RLE case, array values, or question IDs.
# ================================================================

_PATTERN_SIGNAL_RULES: tuple[tuple[str, float, tuple[str, ...]], ...] = (
    (
        "diagnose_or_correct",
        5.0,
        (
            r"\b(?:logic\s+error|error|bug|incorrect|wrong)\b",
            r"\b(?:correct|corrected)\s+(?:line|statement|code|algorithm|program|error|bug)\b|\b(?:fix|debug|diagnose)\b",
            r"\bline\s+number\b.*\b(?:error|correct|fix)\b",
        ),
    ),
    (
        "construct_or_complete",
        4.5,
        (
            r"\bmissing\s+(?:line|statement|code)\b",
            r"\b(?:complete|fill\s+in|finish)\b",
            r"\bwrite\b.*\b(?:line|statement|pseudocode|pseudo\s*code)\b",
            r"\b(?:state|give|provide|write)\b.{0,90}\b(?:test\s+(?:case|cases|input|inputs)|example\s+(?:input|inputs))\b",
        ),
    ),
    (
        "evaluate_or_justify",
        4.5,
        (
            r"\b(?:justify|evaluate)\b",
            r"\b(?:advantage|disadvantage|benefit|drawback)\b",
            r"\b(?:appropriate|suitable)\b.*\b(?:why|reason|justify)\b",
        ),
    ),
    (
        "compare_or_select",
        4.0,
        (
            r"\bcompare\b",
            r"\b(?:difference|similarity)\b",
            r"\b(?:choose|select|which\s+(?:one|option|method|type))\b",
        ),
    ),
    (
        "predict_consequence",
        4.0,
        (
            r"\bwhat\s+(?:will|would)\s+happen\b",
            r"\b(?:consequence|effect)\b",
            r"\bif\b.{0,100}\b(?:changed|removed|not\s+updated|never\s+updated)\b",
        ),
    ),
    (
        "explain_or_reason",
        4.0,
        (
            r"\bwhy\b",
            r"\b(?:reason|reasons)\b",
            r"\bexplain\b",
        ),
    ),
    (
        "adapt_or_modify",
        3.5,
        (
            r"\b(?:adapt|modify|update|change)\b",
            r"\brewrite\b.*\b(?:line|statement|algorithm|code)\b",
        ),
    ),
    (
        "classify_or_decide",
        3.5,
        (
            r"\bclassify\b",
            r"\b(?:category|classification)\b",
            r"\bdecide\s+whether\b",
        ),
    ),
    (
        "apply_or_predict",
        3.5,
        (
            r"\btrace\b",
            r"\bhow\s+many\s+times\b",
            r"\b(?:what|state|give|determine|calculate)\b.{0,80}\b(?:value|values|output|result)\b",
            r"\bafter\s+(?:execution|the\s+program|the\s+algorithm)\b",
        ),
    ),
    (
        "analyse_or_interpret",
        3.0,
        (
            r"\b(?:analyse|analyze|interpret)\b",
            r"\bexplanation\b.*\b(?:visual|table|diagram|data|index)\b",
            r"\buse\b.*\b(?:visual|table|diagram|data)\b.*\b(?:explain|interpret)\b",
        ),
    ),
    (
        "scenario_application",
        2.5,
        (
            r"\b(?:scenario|real[- ]world|context)\b",
            r"\b(?:program|system|developer|user|player|student|teacher)\b.{0,100}\b(?:write|use|store|set|output|apply)\b",
        ),
    ),
)

_PATTERN_COMPATIBILITY: dict[str, set[str]] = {
    "apply_or_predict": {
        "apply_or_predict",
        "analyse_or_interpret",
    },
    "analyse_or_interpret": {
        "analyse_or_interpret",
        "apply_or_predict",
        "explain_or_reason",
    },
    "explain_or_reason": {
        "explain_or_reason",
        "evaluate_or_justify",
    },
    "evaluate_or_justify": {
        "evaluate_or_justify",
        "explain_or_reason",
    },
    "construct_or_complete": {
        "construct_or_complete",
        "scenario_application",
        "adapt_or_modify",
        "diagnose_or_correct",
    },
    "scenario_application": {
        "scenario_application",
        "construct_or_complete",
        "apply_or_predict",
    },
    "adapt_or_modify": {
        "adapt_or_modify",
        "construct_or_complete",
        "diagnose_or_correct",
    },
    "diagnose_or_correct": {
        "diagnose_or_correct",
        "adapt_or_modify",
        "construct_or_complete",
    },
    "compare_or_select": {
        "compare_or_select",
        "classify_or_decide",
    },
    "classify_or_decide": {
        "classify_or_decide",
        "compare_or_select",
    },
    "predict_consequence": {
        "predict_consequence",
        "apply_or_predict",
    },
    "multi_step_synthesis": {
        "multi_step_synthesis",
    },
}


def infer_assessment_pattern_signal(
    question_text_value: Any,
) -> dict[str, Any]:
    """Infer the learner's actual task shape using topic-neutral command cues."""
    text = str(question_text_value or "").strip()
    normalized = normalize_text(text)

    scores: dict[str, float] = {
        pattern: 0.0
        for pattern in ASSESSMENT_PATTERN_CYCLE
    }
    evidence: dict[str, list[str]] = {
        pattern: []
        for pattern in ASSESSMENT_PATTERN_CYCLE
    }

    for pattern_name, weight, regexes in _PATTERN_SIGNAL_RULES:
        for regex in regexes:
            if re.search(regex, normalized, flags=re.IGNORECASE):
                scores[pattern_name] = scores.get(pattern_name, 0.0) + weight
                evidence.setdefault(pattern_name, []).append(regex)

    # Multi-step synthesis is signalled by several genuinely different learner
    # commands, not merely by a long question stem.
    command_groups = 0
    for regex in (
        r"\b(?:state|give|identify|name|list)\b",
        r"\b(?:calculate|determine|trace|predict)\b",
        r"\b(?:explain|justify|reason)\b",
        r"\b(?:write|construct|complete|correct|modify)\b",
        r"\b(?:compare|evaluate|classify|decide)\b",
    ):
        if re.search(regex, normalized):
            command_groups += 1

    if command_groups >= 3:
        scores["multi_step_synthesis"] = max(
            scores.get("multi_step_synthesis", 0.0),
            4.0,
        )
        evidence.setdefault("multi_step_synthesis", []).append(
            "three_or_more_distinct_command_groups"
        )

    ranked = sorted(
        (
            (pattern, score)
            for pattern, score in scores.items()
            if score > 0
        ),
        key=lambda item: (-item[1], ASSESSMENT_PATTERN_CYCLE.index(item[0])),
    )

    if not ranked:
        return {
            "status": "UNAVAILABLE",
            "suggested_pattern": None,
            "compatible_patterns": [],
            "scores": {},
            "evidence": {},
            "reason": "No high-confidence learner-task command cue was detected.",
        }

    top_pattern, top_score = ranked[0]
    near_top = {
        pattern
        for pattern, score in ranked
        if score >= max(2.5, top_score - 1.0)
    }

    compatible = set(
        _PATTERN_COMPATIBILITY.get(
            top_pattern,
            {top_pattern},
        )
    )

    # Treat declared compatibility as symmetric for HITL classification.
    # This reduces noisy flags for genuine mixed learner tasks without making
    # unrelated assessment patterns compatible.
    for candidate_pattern, family in _PATTERN_COMPATIBILITY.items():
        if top_pattern in family:
            compatible.add(candidate_pattern)

    compatible.update(near_top)

    return {
        "status": "AVAILABLE",
        "suggested_pattern": top_pattern,
        "compatible_patterns": sorted(compatible),
        "scores": {
            pattern: round(score, 3)
            for pattern, score in ranked
        },
        "evidence": {
            pattern: values
            for pattern, values in evidence.items()
            if values
        },
        "reason": (
            f"Highest learner-task signal is '{top_pattern}' "
            f"(score {top_score:.1f})."
        ),
    }


def assessment_pattern_alignment_signal(
    question: dict[str, Any],
) -> dict[str, Any]:
    assigned = normalize_text(
        question.get("assessment_pattern", "")
    )
    inferred = infer_assessment_pattern_signal(
        question.get("question_text", "")
    )

    if not assigned:
        return {
            **inferred,
            "status": "REVIEW",
            "assigned_pattern": "",
            "reason": "Question has no assigned assessment_pattern metadata.",
        }

    if inferred.get("status") == "UNAVAILABLE":
        return {
            **inferred,
            "status": "UNAVAILABLE",
            "assigned_pattern": assigned,
        }

    compatible = {
        normalize_text(value)
        for value in inferred.get("compatible_patterns", [])
    }

    suggested = normalize_text(
        inferred.get("suggested_pattern", "")
    )

    if assigned == suggested:
        status = "PASS"
        reason = (
            f"Assigned pattern '{assigned}' matches the strongest learner-task signal."
        )
    elif assigned in compatible:
        status = "PASS"
        reason = (
            f"Assigned pattern '{assigned}' is compatible with inferred learner task "
            f"'{suggested}'."
        )
    else:
        status = "REVIEW"
        reason = (
            f"Assigned pattern '{assigned}' may not match the actual learner task; "
            f"the strongest inferred pattern is '{suggested}'."
        )

    return {
        **inferred,
        "status": status,
        "assigned_pattern": assigned,
        "reason": reason,
    }


_FOCUS_STOPWORDS = {
    "a", "an", "the", "this", "that", "these", "those", "is", "are",
    "was", "were", "be", "been", "being", "used", "use", "using", "here",
    "why", "give", "state", "write", "provide", "one", "two", "three",
    "separate", "short", "brief", "answer", "answers", "reason", "reasons",
    "justify", "explain", "whether", "for", "of", "to", "in", "on", "with",
}

_FOCUS_CONCEPT_FAMILIES: dict[str, set[str]] = {
    "loop": {
        "loop", "loops", "iterate", "iterates", "iteration", "iterations",
        "repeat", "repeats", "repetition", "counter", "range",
    },
    "array": {
        "array", "arrays", "index", "indices", "element", "elements", "row",
        "rows", "column", "columns",
    },
    "selection": {
        "selection", "if", "else", "condition", "conditional", "branch",
    },
    "algorithm": {
        "algorithm", "algorithms", "procedure", "steps", "process",
    },
    "database": {
        "database", "table", "record", "records", "field", "fields", "key",
    },
    "network": {
        "network", "networks", "node", "nodes", "device", "devices",
        "connection", "connections", "topology",
    },
}


def _rationale_focus_text(question_text_value: Any) -> str:
    text = re.sub(r"\s+", " ", str(question_text_value or "").strip())
    candidates = [
        r"\bwhy\s+(?P<focus>.+?)\s+(?:is|are|was|were)\s+used\b",
        r"\b(?:advantage|disadvantage|benefit|drawback)s?\s+of\s+(?P<focus>.+?)(?:[.?!]|$)",
        r"\bjustify\s+(?:whether\s+)?(?P<focus>.+?)(?:[.?!]|$)",
        r"\breasons?\s+(?:why|for)\s+(?P<focus>.+?)(?:[.?!]|$)",
    ]
    for regex in candidates:
        match = re.search(regex, text, flags=re.IGNORECASE)
        if match:
            return str(match.group("focus") or "").strip()
    return ""


def _focus_concept_tokens(value: Any) -> set[str]:
    tokens = {
        token
        for token in semantic_token_set(value)
        if token not in _FOCUS_STOPWORDS
        and len(token) > 2
    }

    expanded = set(tokens)
    for family_tokens in _FOCUS_CONCEPT_FAMILIES.values():
        if tokens & family_tokens:
            expanded.update(family_tokens)
    return expanded


def _criterion_is_bare_answer(value: Any) -> bool:
    normalized = normalize_text(value)
    if not normalized:
        return True
    if re.fullmatch(r"(?:answer\s*)?(?:yes|no|true|false)", normalized):
        return True
    if re.fullmatch(r"(?:[a-z][a-z0-9_]*\s*)?=?\s*-?\d+(?:\.\d+)?", normalized):
        return True
    return False


def command_marking_alignment_signal(
    question: dict[str, Any],
) -> dict[str, Any]:
    """
    Check whether marking guidance rewards the learner action actually asked for.

    This is a REVIEW-oriented semantic contract, not an automatic correctness
    oracle. High-confidence structural mismatches are already handled elsewhere.
    """
    q_text = str(question.get("question_text", "") or "").strip()
    guidance_items = [
        item
        for item in safe_list(question.get("marking_guidance", []))
        if isinstance(item, dict)
        and safe_int(item.get("marks")) > 0
        and str(item.get("criterion", "") or "").strip()
    ]
    criteria = [
        str(item.get("criterion", "") or "").strip()
        for item in guidance_items
    ]

    if not q_text or not criteria:
        return {
            "status": "REVIEW",
            "focus": "",
            "issues": [
                "Question or positive marking guidance is missing, so command-word alignment cannot be confirmed."
            ],
            "diagnostics": {},
        }

    normalized = normalize_text(q_text)
    constraints = _explicit_response_count_constraints(q_text)
    rationale_categories = {
        "reason", "advantage", "disadvantage", "benefit", "drawback"
    }
    rationale_requested = bool(
        any(item.get("category") in rationale_categories for item in constraints)
        or re.search(r"\b(?:why|justify|explain)\b", normalized)
    )

    focus = _rationale_focus_text(q_text) if rationale_requested else ""
    focus_tokens = _focus_concept_tokens(focus)
    issues: list[str] = []
    rationale_diagnostics: list[dict[str, Any]] = []

    if rationale_requested and focus_tokens:
        nonbare = [criterion for criterion in criteria if not _criterion_is_bare_answer(criterion)]
        # If the question explicitly asks N reasons/advantages/etc., the positive
        # criteria are expected to reward those rationale points rather than only
        # the final program/result objective.
        for index, criterion in enumerate(nonbare, start=1):
            criterion_tokens = _focus_concept_tokens(criterion)
            overlap = sorted(focus_tokens & criterion_tokens)
            aligned = bool(overlap)
            rationale_diagnostics.append(
                {
                    "criterion_index": index,
                    "criterion": criterion,
                    "focus_overlap": overlap,
                    "aligned": aligned,
                }
            )
            if not aligned:
                issues.append(
                    "Marking criterion "
                    f"{index} may describe the program/result purpose rather than "
                    f"answering the requested rationale about '{focus}'."
                )

    inferred_pattern = infer_assessment_pattern_signal(q_text)
    suggested_pattern = str(inferred_pattern.get("suggested_pattern") or "")

    # Generic command-demand checks. They are warnings only because legitimate
    # mark schemes can phrase the same answer in many ways.
    if suggested_pattern in {"apply_or_predict", "predict_consequence"}:
        if not any(
            _criterion_has_exact_expected_answer(criterion)
            or re.search(r"\b(?:output|value|result|times|iterations?|consequence|infinite)\b", normalize_text(criterion))
            for criterion in criteria
        ):
            issues.append(
                "Question asks for a predicted/result response, but the marking guidance does not clearly state the expected outcome."
            )

    if suggested_pattern == "diagnose_or_correct":
        if not any(
            re.search(r"\b(?:correct|corrected|fix|error|line|replace)\b", normalize_text(criterion))
            for criterion in criteria
        ):
            issues.append(
                "Question asks the learner to diagnose/correct something, but the marking guidance does not clearly reward the correction."
            )

    if suggested_pattern == "construct_or_complete":
        if not any(
            _criterion_has_exact_expected_answer(criterion)
            or re.search(
                r"\b(?:line|statement|code|missing|complete|output|assignment|"
                r"step|row|column|value)\b",
                normalize_text(criterion),
            )
            or "<-" in criterion
            or bool(re.search(r"\b[a-z][a-z0-9_]*\s*=\s*[^,;]+", criterion, flags=re.IGNORECASE))
            for criterion in criteria
        ):
            issues.append(
                "Question asks the learner to construct/complete an answer, but the marking guidance does not clearly specify the expected constructed response."
            )

    return {
        "status": "REVIEW" if issues else "PASS",
        "focus": focus,
        "issues": list(dict.fromkeys(issues)),
        "diagnostics": {
            "explicit_response_constraints": constraints,
            "rationale_criteria": rationale_diagnostics,
            "inferred_pattern": inferred_pattern,
        },
    }



def _question_explicitly_requests_result_response(
    question_text_value: Any,
) -> bool:
    """
    High-recall guard used only to suppress noisy "final result" warnings.

    If the learner is explicitly asked to state/give/write/determine/calculate
    a value, output, result, count or number of executions, then a marking
    criterion containing that result is expected even when the stem does not
    literally say "final".
    """
    normalized = normalize_text(question_text_value)
    if not normalized:
        return False

    return bool(
        re.search(
            r"\b(?:state|give|write|provide|identify|determine|calculate|"
            r"complete|fill\s+in|what\s+is|what\s+are|how\s+many)\b"
            r".{0,120}\b(?:value|values|output|outputs|result|results|"
            r"times|iterations?|executions?|count|step|row|column)\b",
            normalized,
        )
        or re.search(
            r"\b(?:value|values|output|outputs|result|results)\b"
            r".{0,100}\b(?:state|give|write|provide|determine|calculate)\b",
            normalized,
        )
    )


def cognitive_demand_signal(
    question: dict[str, Any],
) -> dict[str, Any]:
    """
    Conservative low-demand detector for HITL only.

    It never hard-fails a question. It flags only very strong cases such as a
    one-mark bare Yes/No or True/False recognition item with no explanation or
    justification demand. This is topic-neutral and intentionally narrow.
    """
    q_text = str(question.get("question_text", "") or "").strip()
    marks = safe_int(question.get("marks"))
    guidance = [
        str(item.get("criterion", "") or "").strip()
        for item in safe_list(question.get("marking_guidance", []))
        if isinstance(item, dict)
        and safe_int(item.get("marks")) > 0
        and str(item.get("criterion", "") or "").strip()
    ]
    normalized = normalize_text(q_text)

    bare_binary_response = bool(
        marks <= 1
        and re.search(r"\b(?:yes\s+or\s+no|true\s+or\s+false)\b", normalized)
        and not re.search(r"\b(?:explain|justify|reason|why|describe)\b", normalized)
        and guidance
        and all(_criterion_is_bare_answer(item) for item in guidance)
    )

    if bare_binary_response:
        return {
            "status": "REVIEW",
            "reason": (
                "Very low cognitive demand: a one-mark bare binary recognition "
                "response is requested with no reasoning. Human review should "
                "confirm that this challenge level is intentional."
            ),
        }

    return {
        "status": "PASS",
        "reason": "",
    }


def deterministic_quality_signals(
    question: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:

    failures: list[
        str
    ] = []

    warnings: list[
        str
    ] = []

    q_text = str(
        question.get(
            "question_text",
            "",
        )
        or ""
    ).strip()

    open_ended_marking_warning = _open_ended_marking_scheme_warning(
        q_text,
        question.get("marking_guidance", []),
    )
    if open_ended_marking_warning:
        failures.append(
            open_ended_marking_warning
            + " Regenerate with acceptance-based marking guidance that allows "
            "equivalent valid responses."
        )

    exam_layout_warning = _exam_layout_phrasing_warning(q_text)
    if exam_layout_warning:
        failures.append(
            exam_layout_warning
            + " Regenerate using a normal response command unless the matching "
            "student response control is actually rendered."
        )

    plan_index = safe_int(
        question.get(
            "plan_index"
        )
    )

    blueprint_item = next(
        (
            item
            for item in request_payload.get(
                "blueprint",
                [],
            )
            if safe_int(
                item.get(
                    "plan_index"
                )
            )
            == plan_index
        ),
        {},
    )

    semantic_topic_grounding = _semantic_topic_grounding_signal(
        q_text,
        blueprint_item,
        request_payload,
    )

    anchor_topic_grounding_warning = _anchor_topic_grounding_warning(
        q_text,
        blueprint_item,
        request_payload,
    )

    grounding_status = str(
        semantic_topic_grounding.get(
            "status",
            "UNAVAILABLE",
        )
        or "UNAVAILABLE"
    ).strip().upper()

    if grounding_status == "FAIL":
        failures.append(
            str(
                semantic_topic_grounding.get(
                    "reason",
                    "Question may have drifted outside the approved topic scope.",
                )
            )
        )

    elif grounding_status == "REVIEW":
        warnings.append(
            str(
                semantic_topic_grounding.get(
                    "reason",
                    "Question requires topic-grounding review.",
                )
            )
        )

    elif (
        grounding_status == "UNAVAILABLE"
        and anchor_topic_grounding_warning
    ):
        warnings.append(
            anchor_topic_grounding_warning
        )

    examples = blueprint_item.get(
        "official_question_style_examples",
        [],
    ) or []

    max_lexical = 0.0
    max_jaccard = 0.0
    closest_example = None

    for example in examples:
        comparison_example = official_style_comparison_text(
            example
        )

        lexical = lexical_similarity(
            q_text,
            comparison_example,
        )

        jaccard = token_jaccard(
            q_text,
            comparison_example,
        )

        if lexical > max_lexical:
            max_lexical = lexical
            closest_example = comparison_example

        max_jaccard = max(
            max_jaccard,
            jaccard,
        )

    if max_lexical >= 0.92:
        failures.append(
            "Question is too lexically similar to an official style example."
        )

    elif (
        max_lexical >= 0.80
        or max_jaccard >= 0.70
    ):
        warnings.append(
            "Question may be too similar to an official style example."
        )

    guidance = question.get(
        "marking_guidance",
        [],
    )

    criteria = [
        str(
            item.get(
                "criterion",
                "",
            )
            or ""
        ).strip()
        for item in guidance
        if isinstance(
            item,
            dict,
        )
        and str(
            item.get(
                "criterion",
                "",
            )
            or ""
        ).strip()
    ]

    max_criterion_overlap = 0.0

    for left_index in range(
        len(
            criteria
        )
    ):
        for right_index in range(
            left_index + 1,
            len(
                criteria
            ),
        ):
            max_criterion_overlap = max(
                max_criterion_overlap,
                token_jaccard(
                    criteria[
                        left_index
                    ],
                    criteria[
                        right_index
                    ],
                ),
            )

    if max_criterion_overlap >= 0.75:
        warnings.append(
            "Marking criteria may reward substantially overlapping evidence."
        )

    lower_q = q_text.casefold()

    marks = safe_int(
        question.get(
            "marks"
        )
    )

    # ------------------------------------------------------------
    # Self-contained answerability guard
    # ------------------------------------------------------------
    self_containment_failures = _question_self_containment_failures(
        q_text
    )

    failures.extend(
        self_containment_failures
    )

    # ------------------------------------------------------------
    # Mark-scheme answer specificity guard
    # ------------------------------------------------------------
    marking_specificity_failures = _marking_specificity_failures(
        question
    )

    failures.extend(
        marking_specificity_failures
    )


    # ------------------------------------------------------------
    # Mark-demand guard
    # ------------------------------------------------------------
    if (
        marks <= 1
        and _explicit_one_mark_multi_demand(
            q_text
        )
    ):
        failures.append(
            "A 1-mark question asks for more than one independent response "
            "demand (for example identify/select plus justify/explain)."
        )

    # ------------------------------------------------------------
    # Source-grounding guard for implementation/runtime jargon
    # ------------------------------------------------------------
    support_corpus = _question_support_corpus(
        blueprint_item,
        request_payload,
    )

    normalized_support = normalize_text(
        support_corpus
    )

    for term_name, pattern in SPECIALISED_RUNTIME_TERM_PATTERNS.items():
        if not re.search(
            pattern,
            q_text,
            flags=re.IGNORECASE,
        ):
            continue

        if re.search(
            pattern,
            normalized_support,
            flags=re.IGNORECASE,
        ):
            continue

        failures.append(
            "Question introduces specialised implementation/runtime concept "
            f"'{term_name}' without support in lesson or assigned style "
            "evidence."
        )

    # ------------------------------------------------------------
    # Text/code formatting guard (visual architecture is validated separately)
    # ------------------------------------------------------------
    if _collapsed_multistatement_code(
        q_text
    ):
        warnings.append(
            "Multi-statement code/pseudocode appears collapsed into prose. "
            "Preserve a genuine multiline code block for student readability."
        )

    target_task_family = str(
        blueprint_item.get(
            "target_task_family",
            "",
        )
        or ""
    ).strip()

    generated_task_family = _style_task_family(
        q_text
    )

    if (
        target_task_family
        and generated_task_family != target_task_family
    ):
        warnings.append(
            "Generated learner task family "
            f"'{generated_task_family}' differs from the codebase-grounded "
            f"target '{target_task_family}'. Human review should confirm the "
            "question still matches the intended AQA-style task demand."
        )

    trace_heavy = bool(
        (
            "trace table"
            in lower_q
            or "trace the"
            in lower_q
        )
        and (
            "each iteration"
            in lower_q
            or "after each"
            in lower_q
            or "all iterations"
            in lower_q
            or "each step"
            in lower_q
        )
    )

    if (
        trace_heavy
        and marks <= 2
    ):
        warnings.append(
            "Multi-step trace workload may be high for the allocated marks."
        )

    final_guidance = [
        criterion
        for criterion in criteria
        if (
            "final value"
            in normalize_text(
                criterion
            )
            or "final values"
            in normalize_text(
                criterion
            )
            or "final output"
            in normalize_text(
                criterion
            )
            or re.search(
                r"\b[a-z][a-z0-9_]*\s*=\s*-?\d+\b",
                criterion.casefold(),
            )
        )
    ]

    if (
        final_guidance
        and not question_requests_final_response(
            q_text
        )
        and not _question_explicitly_requests_result_response(
            q_text
        )
        and not (
            "trace table"
            in lower_q
        )
    ):
        warnings.append(
            "Marking guidance rewards a final result but the learner task "
            "does not clearly request a final-result response."
        )

    # ------------------------------------------------------------
    # Command-word <-> marking-scheme alignment (HITL-oriented)
    # ------------------------------------------------------------
    command_marking_alignment = command_marking_alignment_signal(
        question
    )

    if command_marking_alignment.get("status") == "REVIEW":
        warnings.extend(
            "Command/mark-scheme alignment: " + str(issue)
            for issue in command_marking_alignment.get("issues", [])
            if str(issue).strip()
        )

    # ------------------------------------------------------------
    # Assigned assessment pattern <-> actual learner task alignment
    # ------------------------------------------------------------
    pattern_task_alignment = assessment_pattern_alignment_signal(
        question
    )

    if pattern_task_alignment.get("status") == "REVIEW":
        warnings.append(
            "Assessment-pattern alignment: "
            + str(pattern_task_alignment.get("reason", "Human review required."))
        )

    cognitive_demand = cognitive_demand_signal(
        question
    )

    if cognitive_demand.get("status") == "REVIEW":
        warnings.append(
            "Cognitive-demand review: "
            + str(cognitive_demand.get("reason", "Human review required."))
        )

    return {
        "hard_failures":
            list(
                dict.fromkeys(
                    failures
                )
            ),

        "warnings":
            list(
                dict.fromkeys(
                    warnings
                )
            ),

        "max_official_lexical_similarity":
            round(
                max_lexical,
                4,
            ),

        "max_official_token_jaccard":
            round(
                max_jaccard,
                4,
            ),

        "closest_official_example":
            closest_example,

        "max_marking_criterion_overlap":
            round(
                max_criterion_overlap,
                4,
            ),

        "generated_task_family":
            generated_task_family,

        "self_containment_failures":
            self_containment_failures,

        "marking_specificity_failures":
            marking_specificity_failures,

        "target_task_family":
            str(
                blueprint_item.get(
                    "target_task_family",
                    "",
                )
                or ""
            ).strip(),

        "anchor_topic_grounding_warning":
            anchor_topic_grounding_warning,

        "semantic_topic_grounding":
            semantic_topic_grounding,

        "command_marking_alignment":
            command_marking_alignment,

        "pattern_task_alignment":
            pattern_task_alignment,

        "cognitive_demand":
            cognitive_demand,
    }


def deterministic_quiz_diversity_signals(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:
    """
    Quiz-wide repetition check.

    Diversity is intentionally SOFT:
    - exact duplicate question text is already a structural error elsewhere;
    - broad pattern reuse alone is NOT an error;
    - a warning is raised only when two questions, especially on the same
      anchor topic, are strongly textually/template similar;
    - warnings route the candidate to HITL but do not automatically reject it.

    This avoids rejecting valid topics that naturally support a limited number
    of sensible assessment forms.
    """
    questions = [
        item
        for item in payload.get(
            "questions",
            [],
        )
        if isinstance(
            item,
            dict,
        )
    ]

    warnings: list[
        str
    ] = []

    hard_failures: list[
        str
    ] = []

    comparisons = []

    task_family_by_plan = {
        safe_int(
            item.get(
                "plan_index"
            )
        ): _style_task_family(
            item.get(
                "question_text",
                "",
            )
        )
        for item in questions
    }

    inferred_pattern_by_plan = {
        safe_int(item.get("plan_index")): infer_assessment_pattern_signal(
            item.get("question_text", "")
        ).get("suggested_pattern")
        for item in questions
    }

    blueprint_by_plan = {
        safe_int(item.get("plan_index")): item
        for item in request_payload.get("blueprint", [])
        if isinstance(item, dict)
    }

    target_family_by_plan = {
        plan_index: str(
            item.get("target_task_family", "") or ""
        ).strip()
        for plan_index, item in blueprint_by_plan.items()
    }

    question_text_values = [
        str(
            item.get(
                "question_text",
                "",
            )
            or ""
        ).strip()
        for item in questions
    ]

    semantic_matrix_raw, semantic_backend = _minilm_embeddings(
        question_text_values
    )

    semantic_similarity_matrix = (
        (
            semantic_matrix_raw
            @ semantic_matrix_raw.T
        )
        if semantic_matrix_raw is not None
        else None
    )

    filters = request_payload.get(
        "assessment_filters",
        {},
    )

    if not isinstance(
        filters,
        dict,
    ):
        filters = {}

    special_directives = filters.get(
        "special_instruction_directives",
        {},
    )

    if not isinstance(
        special_directives,
        dict,
    ):
        special_directives = {}

    strict_distinct_roles = {
        str(
            role or ""
        ).strip().casefold()
        for role in special_directives.get(
            "distinct_styles_for_roles",
            [],
        )
        if str(
            role or ""
        ).strip()
    }

    for left_index in range(
        len(
            questions
        )
    ):
        for right_index in range(
            left_index + 1,
            len(
                questions
            ),
        ):
            left = questions[
                left_index
            ]

            right = questions[
                right_index
            ]

            left_text = str(
                left.get(
                    "question_text",
                    "",
                )
                or ""
            ).strip()

            right_text = str(
                right.get(
                    "question_text",
                    "",
                )
                or ""
            ).strip()

            lexical = lexical_similarity(
                left_text,
                right_text,
            )

            jaccard = token_jaccard(
                left_text,
                right_text,
            )

            same_topic = (
                normalize_text(
                    left.get(
                        "topic",
                        "",
                    )
                )
                ==
                normalize_text(
                    right.get(
                        "topic",
                        "",
                    )
                )
            )

            same_pattern = (
                normalize_text(
                    left.get(
                        "assessment_pattern",
                        "",
                    )
                )
                ==
                normalize_text(
                    right.get(
                        "assessment_pattern",
                        "",
                    )
                )
            )

            left_role = str(
                left.get(
                    "role",
                    "",
                )
                or ""
            ).strip().casefold()

            right_role = str(
                right.get(
                    "role",
                    "",
                )
                or ""
            ).strip().casefold()

            same_role = bool(
                left_role
                and left_role == right_role
            )

            semantic_similarity = (
                float(
                    np.clip(
                        semantic_similarity_matrix[
                            left_index,
                            right_index,
                        ],
                        -1.0,
                        1.0,
                    )
                )
                if semantic_similarity_matrix is not None
                else None
            )

            comparisons.append(
                {
                    "left_plan_index":
                        safe_int(
                            left.get(
                                "plan_index"
                            )
                        ),

                    "right_plan_index":
                        safe_int(
                            right.get(
                                "plan_index"
                            )
                        ),

                    "same_topic":
                        same_topic,

                    "same_pattern":
                        same_pattern,

                    "left_task_family":
                        task_family_by_plan.get(
                            safe_int(left.get("plan_index")),
                            "other",
                        ),

                    "right_task_family":
                        task_family_by_plan.get(
                            safe_int(right.get("plan_index")),
                            "other",
                        ),

                    "left_inferred_pattern":
                        inferred_pattern_by_plan.get(
                            safe_int(left.get("plan_index"))
                        ),

                    "right_inferred_pattern":
                        inferred_pattern_by_plan.get(
                            safe_int(right.get("plan_index"))
                        ),

                    "lexical_similarity":
                        round(
                            lexical,
                            4,
                        ),

                    "token_jaccard":
                        round(
                            jaccard,
                            4,
                        ),

                    "semantic_similarity":
                        (
                            round(
                                semantic_similarity,
                                4,
                            )
                            if semantic_similarity is not None
                            else None
                        ),

                    "semantic_backend":
                        semantic_backend,

                    "same_role":
                        same_role,
                }
            )

            # Strong near-duplicate threshold.
            #
            # Same-topic pairs get a slightly more sensitive threshold because
            # superficial value/name changes are common there. These are still
            # WARNINGS only, not failures.
            strong_near_duplicate = bool(
                (
                    same_topic
                    and (
                        lexical >= 0.84
                        or jaccard >= 0.76
                        or (
                            semantic_similarity is not None
                            and semantic_similarity >= 0.86
                        )
                    )
                )
                or (
                    lexical >= 0.92
                    or jaccard >= 0.86
                    or (
                        semantic_similarity is not None
                        and semantic_similarity >= 0.93
                    )
                )
            )

            if strong_near_duplicate:
                left_plan = safe_int(
                    left.get(
                        "plan_index"
                    )
                )

                right_plan = safe_int(
                    right.get(
                        "plan_index"
                    )
                )

                warnings.append(
                    "Questions "
                    f"{left_plan} and {right_plan} may be strong "
                    "near-duplicate assessment templates. Human review should "
                    "confirm that the learner is genuinely doing different "
                    "work, rather than the same task with changed values or "
                    "names."
                )

            if (
                same_role
                and left_role in strict_distinct_roles
            ):
                left_family = task_family_by_plan.get(
                    safe_int(
                        left.get(
                            "plan_index"
                        )
                    ),
                    "other",
                )

                right_family = task_family_by_plan.get(
                    safe_int(
                        right.get(
                            "plan_index"
                        )
                    ),
                    "other",
                )

                semantic_style_collision = bool(
                    semantic_similarity is not None
                    and (
                        semantic_similarity >= 0.86
                        or (
                            semantic_similarity >= 0.76
                            and left_family == right_family
                        )
                    )
                )

                lexical_style_collision = bool(
                    left_family == right_family
                    and (
                        lexical >= 0.76
                        or jaccard >= 0.66
                    )
                )

                if (
                    semantic_style_collision
                    or lexical_style_collision
                ):
                    hard_failures.append(
                        "Mandatory special-instruction distinct-style violation "
                        f"for {left_role} questions "
                        f"{safe_int(left.get('plan_index'))} and "
                        f"{safe_int(right.get('plan_index'))}: learner tasks are "
                        "too semantically/template similar despite different "
                        "blueprint pattern labels. Regenerate with a genuinely "
                        "different learner action."
                    )

    # The generated metadata can contain different labels while the learner is
    # still being asked to do the same broad thing. Because this classifier is
    # heuristic, duplicate inferred patterns are routed to HITL rather than used
    # as an automatic failure.
    for role in strict_distinct_roles:
        role_items = [
            item
            for item in questions
            if str(item.get("role", "") or "").strip().casefold() == role
        ]
        inferred_values = [
            inferred_pattern_by_plan.get(safe_int(item.get("plan_index")))
            for item in role_items
        ]
        inferred_values = [value for value in inferred_values if value]
        if len(inferred_values) != len(set(inferred_values)):
            warnings.append(
                "Mandatory distinct-style instruction uses unique assigned pattern "
                f"labels for {role} questions, but the local learner-task classifier "
                "infers at least one repeated broad task pattern. Human review should "
                "confirm that the questions are genuinely different in student work."
            )

    # Task-family concentration is a SOFT guard. The deterministic blueprint and
    # prompt already try to vary work; this warning only surfaces a quiz that
    # still collapses into the same learner action too often.
    family_counts: dict[str, int] = {}
    topic_family_counts: dict[str, dict[str, int]] = {}

    for item in questions:
        family = _style_task_family(
            item.get(
                "question_text",
                "",
            )
        )
        family_counts[family] = (
            family_counts.get(family, 0)
            + 1
        )

        topic_norm = normalize_text(
            item.get(
                "topic",
                "",
            )
        )
        per_topic = topic_family_counts.setdefault(
            topic_norm,
            {},
        )
        per_topic[family] = (
            per_topic.get(family, 0)
            + 1
        )

    question_count = len(questions)

    if question_count >= 6 and family_counts:
        dominant_family, dominant_count = max(
            family_counts.items(),
            key=lambda pair: (
                pair[1],
                pair[0],
            ),
        )

        if dominant_count >= max(
            4,
            math.ceil(0.55 * question_count),
        ):
            warnings.append(
                "Quiz task-family variety may be too narrow: "
                f"'{dominant_family}' appears in {dominant_count}/"
                f"{question_count} questions."
            )

    for topic_norm, counts in topic_family_counts.items():
        topic_total = sum(
            counts.values()
        )

        if topic_total < 3:
            continue

        family, count = max(
            counts.items(),
            key=lambda pair: (
                pair[1],
                pair[0],
            ),
        )

        if count >= math.ceil(
            0.67 * topic_total
        ):
            warnings.append(
                "Repeated-topic task variety may be too narrow for "
                f"'{topic_norm}': '{family}' appears in {count}/"
                f"{topic_total} questions."
            )

    # Stronger general diversity guard: when the deterministic blueprint asks
    # repeated questions on the same topic to use genuinely different target
    # task families, the generated quiz must not collapse most of them into one
    # actual learner action. This catches value/name swaps that evade lexical
    # near-duplicate thresholds.
    questions_by_topic: dict[str, list[dict[str, Any]]] = {}
    for item in questions:
        topic_norm = normalize_text(item.get("topic", ""))
        questions_by_topic.setdefault(topic_norm, []).append(item)

    for topic_norm, topic_items in questions_by_topic.items():
        if len(topic_items) < 3:
            continue

        target_families = {
            target_family_by_plan.get(
                safe_int(item.get("plan_index")),
                "",
            )
            for item in topic_items
        }
        target_families.discard("")

        if len(target_families) < 2:
            continue

        actual_counts: dict[str, int] = {}
        for item in topic_items:
            actual = task_family_by_plan.get(
                safe_int(item.get("plan_index")),
                "other",
            )
            actual_counts[actual] = actual_counts.get(actual, 0) + 1

        dominant_family, dominant_count = max(
            actual_counts.items(),
            key=lambda pair: (pair[1], pair[0]),
        )

        collapse_threshold = max(
            3,
            math.ceil(0.67 * len(topic_items)),
        )

        if dominant_count >= collapse_threshold:
            hard_failures.append(
                "Quiz diversity collapsed for repeated topic "
                f"'{topic_norm}': the blueprint requested multiple target "
                f"task families {sorted(target_families)}, but "
                f"'{dominant_family}' appears in {dominant_count}/"
                f"{len(topic_items)} generated questions. Regenerate with "
                "genuinely different learner actions rather than superficial "
                "value/name changes."
            )

    return {
        "warnings":
            list(
                dict.fromkeys(
                    warnings
                )
            ),

        "comparisons":
            comparisons,

        "task_family_counts":
            family_counts,

        "topic_task_family_counts":
            topic_family_counts,

        "hard_failures":
            list(
                dict.fromkeys(
                    hard_failures
                )
            ),

        "semantic_backend":
            semantic_backend,
    }


def deterministic_manual_qa_spot_check(
    payload: dict[str, Any],
) -> dict[str, Any]:
    """
    Deterministically sample a configurable fraction of otherwise-PASS quiz
    candidates for an explicit human QA audit of the heuristic layer.

    This does not replace mandatory HITL and does not add another LLM call.
    The same quiz payload always receives the same sampling decision.
    """
    try:
        rate = float(
            os.getenv(
                "AGENT2_MANUAL_QA_SPOT_CHECK_RATE",
                "0.20",
            )
        )
    except (TypeError, ValueError):
        rate = 0.20

    rate = min(1.0, max(0.0, rate))
    questions = [
        item
        for item in payload.get("questions", [])
        if isinstance(item, dict)
    ]

    if not questions or rate <= 0.0:
        return {
            "selected": False,
            "rate": rate,
            "fingerprint_bucket": None,
            "purpose": "periodic_manual_audit_of_deterministic_quality_checks",
        }

    serialized = json.dumps(
        questions,
        sort_keys=True,
        ensure_ascii=False,
        default=str,
    ).encode("utf-8")
    digest = hashlib.sha256(serialized).hexdigest()
    bucket = int(digest[:8], 16) / 0xFFFFFFFF

    return {
        "selected": bool(bucket < rate),
        "rate": rate,
        "fingerprint_bucket": round(bucket, 6),
        "purpose": "periodic_manual_audit_of_deterministic_quality_checks",
    }


def validate_semantic_quality(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:
    """
    Backward-compatible quality-validation entry point.

    Despite the historical function name, this stage is now 100% deterministic
    Python. It makes NO Gemini/LLM call.

    Final qualitative acceptance is always handled by HITL.
    """
    questions = payload.get(
        "questions",
        [],
    )

    if not isinstance(
        questions,
        list,
    ):
        questions = []

    signals = {}

    all_failures: list[
        str
    ] = []

    all_warnings: list[
        str
    ] = []

    for question in questions:
        if not isinstance(
            question,
            dict,
        ):
            continue

        question_id = str(
            question.get(
                "generated_question_id",
                "UNKNOWN",
            )
            or "UNKNOWN"
        )

        result = deterministic_quality_signals(
            question,
            request_payload,
        )

        signals[
            question_id
        ] = result

        all_failures.extend(
            f"{question_id}: {reason}"
            for reason in result[
                "hard_failures"
            ]
        )

        all_warnings.extend(
            f"{question_id}: {reason}"
            for reason in result[
                "warnings"
            ]
        )

    quiz_diversity = (
        deterministic_quiz_diversity_signals(
            payload,
            request_payload,
        )
    )

    all_failures.extend(
        quiz_diversity.get(
            "hard_failures",
            [],
        )
    )

    all_warnings.extend(
        quiz_diversity.get(
            "warnings",
            [],
        )
    )

    filters = request_payload.get(
        "assessment_filters",
        {},
    )

    if isinstance(
        filters,
        dict,
    ):
        special_text = str(
            filters.get(
                "special_instructions",
                "",
            )
            or ""
        ).strip()

        if special_text:
            all_warnings.append(
                "Mandatory special quiz instructions were supplied. "
                "Machine-checkable parts are enforced deterministically; HITL "
                "must also confirm every qualitative wording/style/emphasis "
                "instruction before release."
            )

    if all_failures:
        final_status = "FAIL"

    elif all_warnings:
        final_status = "REVIEW"

    else:
        final_status = "PASS"

    reasons = list(
        dict.fromkeys(
            all_failures
            + all_warnings
        )
    )

    manual_qa_spot_check = deterministic_manual_qa_spot_check(
        payload
    )

    return {
        "status":
            final_status,

        # HITL is now the final acceptance gate even when this stage PASSes.
        "release_eligible":
            False,

        "validation_mode":
            "deterministic_python_plus_mandatory_hitl",

        "llm_semantic_review_used":
            False,

        "manual_qa_spot_check":
            manual_qa_spot_check,

        "reasons":
            reasons,

        "warnings":
            list(
                dict.fromkeys(
                    all_warnings
                )
            ),

        "deterministic_signals":
            signals,

        "quiz_diversity_signals":
            quiz_diversity,

        # Keep this key for current manifest/frontend compatibility.
        "llm_review": {
            "enabled":
                False,

            "provider":
                None,

            "model":
                None,

            "reason":
                (
                    "Secondary LLM semantic review was removed from the "
                    "normal flow. Gemini 3.5 Flash generates the quiz once; "
                    "Python performs deterministic checks; HITL performs the "
                    "final qualitative review."
                ),
        },
    }

# ================================================================
# v2.37 — DUPLICATION + TERMINOLOGY QUALITY EXTENSIONS
# ================================================================

_deterministic_quiz_diversity_v236 = deterministic_quiz_diversity_signals


def _student_visual_fingerprint(
    question: dict[str, Any],
) -> str:
    visual_type = normalize_visual_requirement(
        question.get(
            "visual_requirement",
            "none",
        )
    )

    if visual_type == "none":
        return ""

    visual = question.get(
        "visual",
        {},
    )

    spec = (
        visual.get(
            "spec",
            {},
        )
        if isinstance(
            visual,
            dict,
        )
        else {}
    )

    if not isinstance(
        spec,
        dict,
    ):
        return ""

    # Caption wording should not defeat duplicate-visual detection.
    cleaned_spec = {
        key: value
        for key, value in spec.items()
        if key not in {
            "caption",
            "_suppress_gate_type_labels",
        }
    }

    return stable_json_fingerprint(
        {
            "visual_type": visual_type,
            "spec": cleaned_spec,
        }
    )


def deterministic_quiz_diversity_signals(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:
    result = _deterministic_quiz_diversity_v236(
        payload,
        request_payload,
    )

    hard_failures = list(
        result.get(
            "hard_failures",
            [],
        )
    )

    warnings = list(
        result.get(
            "warnings",
            [],
        )
    )

    questions = [
        item
        for item in payload.get(
            "questions",
            [],
        )
        if isinstance(
            item,
            dict,
        )
    ]

    visual_fingerprints: dict[
        str,
        list[dict[str, Any]],
    ] = {}

    for question in questions:
        fingerprint = _student_visual_fingerprint(
            question
        )

        if not fingerprint:
            continue

        visual_fingerprints.setdefault(
            fingerprint,
            [],
        ).append(
            question
        )

    visual_duplicate_groups: list[
        dict[str, Any]
    ] = []

    for fingerprint, group in visual_fingerprints.items():
        if len(group) < 2:
            continue

        # Intentional shared source visual can be explicitly declared later.
        shared_ids = {
            str(
                item.get(
                    "shared_visual_id",
                    "",
                )
                or ""
            ).strip()
            for item in group
        }
        shared_ids.discard("")

        if (
            len(shared_ids) == 1
            and len(shared_ids)
            == len(
                {
                    str(
                        item.get(
                            "shared_visual_id",
                            "",
                        )
                        or ""
                    ).strip()
                    for item in group
                    if str(
                        item.get(
                            "shared_visual_id",
                            "",
                        )
                        or ""
                    ).strip()
                }
            )
            and all(
                str(
                    item.get(
                        "shared_visual_id",
                        "",
                    )
                    or ""
                ).strip()
                for item in group
            )
        ):
            continue

        plan_indexes = [
            safe_int(
                item.get(
                    "plan_index",
                    0,
                )
            )
            for item in group
        ]

        visual_types = sorted(
            {
                normalize_visual_requirement(
                    item.get(
                        "visual_requirement",
                        "none",
                    )
                )
                for item in group
            }
        )

        visual_duplicate_groups.append(
            {
                "fingerprint": fingerprint,
                "plan_indexes": plan_indexes,
                "visual_types": visual_types,
            }
        )

        hard_failures.append(
            "Duplicate student visual specification detected across "
            f"questions {plan_indexes}. Generate distinct values/scenario/"
            "structure unless the questions explicitly share one source "
            "visual via shared_visual_id."
        )

    # Escalate only extremely strong same-topic + same-pattern textual
    # duplicates. The existing softer thresholds still remain warnings.
    for comparison in result.get(
        "comparisons",
        [],
    ):
        if not isinstance(
            comparison,
            dict,
        ):
            continue

        lexical = float(
            comparison.get(
                "lexical_similarity",
                0.0,
            )
            or 0.0
        )

        semantic = comparison.get(
            "semantic_similarity"
        )

        semantic_value = (
            float(semantic)
            if semantic is not None
            else None
        )

        if (
            comparison.get(
                "same_topic",
                False,
            )
            and comparison.get(
                "same_pattern",
                False,
            )
            and (
                lexical >= 0.92
                or (
                    semantic_value is not None
                    and semantic_value >= 0.95
                )
            )
        ):
            hard_failures.append(
                "Questions "
                f"{comparison.get('left_plan_index')} and "
                f"{comparison.get('right_plan_index')} are too close in "
                "wording/learner task for the same topic and pattern. "
                "Regenerate one as a genuinely different assessment task."
            )

    hard_failures.extend(
        _cross_pattern_semantic_duplicate_failures(
            questions
        )
    )

    result["hard_failures"] = list(
        dict.fromkeys(
            hard_failures
        )
    )

    result["warnings"] = list(
        dict.fromkeys(
            warnings
        )
    )

    result[
        "duplicate_visual_groups"
    ] = visual_duplicate_groups

    return result


def _terminology_marking_warnings(
    question: dict[str, Any],
) -> list[str]:
    warnings: list[str] = []

    visual_type = normalize_visual_requirement(
        question.get(
            "visual_requirement",
            "none",
        )
    )

    if visual_type != "network_diagram":
        return warnings

    q_text = normalize_text(
        question.get(
            "question_text",
            "",
        )
    )

    visual = question.get(
        "visual",
        {},
    )

    spec = (
        visual.get(
            "spec",
            {},
        )
        if isinstance(
            visual,
            dict,
        )
        else {}
    )

    visual_terms: set[str] = set()

    if isinstance(
        spec,
        dict,
    ):
        for node in spec.get(
            "nodes",
            [],
        ):
            if not isinstance(
                node,
                dict,
            ):
                continue

            node_blob = normalize_text(
                f"{node.get('type', '')} {node.get('label', '')}"
            )

            for term in [
                "router",
                "switch",
                "server",
                "printer",
                "firewall",
                "access point",
            ]:
                if term in node_blob:
                    visual_terms.add(term)

    question_terms = {
        term
        for term in [
            "router",
            "switch",
            "server",
            "printer",
            "firewall",
            "access point",
        ]
        if term in q_text
    }

    marking_text = normalize_text(
        " ".join(
            str(
                item.get(
                    "criterion",
                    "",
                )
                or ""
            )
            for item in question.get(
                "marking_guidance",
                [],
            )
            if isinstance(
                item,
                dict,
            )
        )
    )

    marking_terms = {
        term
        for term in [
            "router",
            "switch",
            "server",
            "printer",
            "firewall",
            "access point",
        ]
        if term in marking_text
    }

    if (
        question_terms
        and visual_terms
        and question_terms.isdisjoint(
            visual_terms
        )
    ):
        warnings.append(
            "Network device terminology in question_text and rendered "
            "visual may be inconsistent."
        )

    # Marking guidance may mention acceptable alternatives, so only warn
    # rather than hard-fail when it introduces a different device term.
    introduced = (
        marking_terms
        - question_terms
        - visual_terms
    )

    if introduced:
        warnings.append(
            "Marking guidance introduces network-device terminology not "
            "present in the question/visual: "
            + ", ".join(
                sorted(
                    introduced
                )
            )
        )

    return warnings


_validate_semantic_quality_v236 = validate_semantic_quality


def validate_semantic_quality(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:
    result = _validate_semantic_quality_v236(
        payload,
        request_payload,
    )

    terminology_warnings: list[str] = []
    dependency_warnings: list[str] = []

    for question in payload.get(
        "questions",
        [],
    ):
        if not isinstance(
            question,
            dict,
        ):
            continue

        qid = str(
            question.get(
                "generated_question_id",
                "",
            )
            or ""
        )

        terminology_warnings.extend(
            f"{qid}: {warning}"
            for warning in _terminology_marking_warnings(
                question
            )
        )

        dependency_warnings.extend(
            f"{qid}: {warning}"
            for warning in _visual_dependency_warnings(
                question
            )
        )

    all_review_warnings = terminology_warnings + dependency_warnings

    if all_review_warnings:
        existing_warnings = list(
            result.get(
                "warnings",
                [],
            )
        )

        existing_reasons = list(
            result.get(
                "reasons",
                [],
            )
        )

        result["warnings"] = list(
            dict.fromkeys(
                existing_warnings
                + all_review_warnings
            )
        )

        result["reasons"] = list(
            dict.fromkeys(
                existing_reasons
                + all_review_warnings
            )
        )

        if result.get(
            "status"
        ) == "PASS":
            result["status"] = "REVIEW"

    result[
        "terminology_visual_consistency"
    ] = {
        "status": (
            "PASS"
            if not terminology_warnings
            else "REVIEW"
        ),
        "warnings": terminology_warnings,
    }

    result[
        "visual_dependency"
    ] = {
        "status": (
            "PASS"
            if not dependency_warnings
            else "REVIEW"
        ),
        "warnings": dependency_warnings,
    }

    return result

# ================================================================
# v2.38 — STRONGER CROSS-PATTERN DUPLICATE DETECTION
# ================================================================

_STOPWORDS_FOR_DUPLICATES = {
    "the", "a", "an", "of", "to", "in", "on", "for", "and", "or", "is",
    "are", "be", "with", "each", "give", "state", "what", "why", "use",
    "using", "shown", "visual", "diagram", "question", "mark", "marks",
    "this", "that", "these", "those", "it", "as", "at", "by", "from",
}


def _question_token_set(
    question: dict[str, Any],
) -> set[str]:
    text = normalize_text(
        question.get("question_text", "")
    )
    tokens = {
        token
        for token in re.findall(
            r"[a-z0-9]+",
            text,
        )
        if token not in _STOPWORDS_FOR_DUPLICATES
        and len(token) >= 3
    }
    return tokens


def _same_task_family_even_if_pattern_differs(
    left: dict[str, Any],
    right: dict[str, Any],
) -> bool:
    left_text = normalize_text(
        left.get("question_text", "")
    )
    right_text = normalize_text(
        right.get("question_text", "")
    )

    paired_phrases = [
        "preferred over a bus topology",
        "advantages of using a star topology",
        "value of q",
        "state the output q",
        "line should be changed",
        "corrected line",
        "stored at the two specified positions",
        "one likely consequence",
    ]

    for phrase in paired_phrases:
        if phrase in left_text and phrase in right_text:
            return True

    return False


def _cross_pattern_semantic_duplicate_failures(
    questions: list[dict[str, Any]],
) -> list[str]:
    failures: list[str] = []

    for i in range(len(questions)):
        for j in range(i + 1, len(questions)):
            left = questions[i]
            right = questions[j]

            left_id = str(left.get("generated_question_id", "") or f"Q{i+1}")
            right_id = str(right.get("generated_question_id", "") or f"Q{j+1}")

            left_ref = str(left.get("official_reference", "") or "")
            right_ref = str(right.get("official_reference", "") or "")

            left_tokens = _question_token_set(left)
            right_tokens = _question_token_set(right)

            union = left_tokens | right_tokens
            overlap = (
                len(left_tokens & right_tokens) / len(union)
                if union else 0.0
            )

            left_visual = normalize_visual_requirement(
                left.get("visual_requirement", "none")
            )
            right_visual = normalize_visual_requirement(
                right.get("visual_requirement", "none")
            )

            same_ref = bool(left_ref and left_ref == right_ref)
            same_visual = left_visual == right_visual and left_visual != "none"

            if same_ref and (same_visual or _same_task_family_even_if_pattern_differs(left, right)):
                if overlap >= 0.40:
                    failures.append(
                        f"{left_id} and {right_id}: learner-task wording/content "
                        "is too similar across different generated questions."
                    )

    return list(dict.fromkeys(failures))

# ================================================================
# v2.41 — GENERIC QUIZ DIVERSITY
# ================================================================

def _generic_student_visual_fingerprint(
    question: dict[str, Any],
) -> str:
    visual_type = normalize_visual_requirement(
        question.get(
            "visual_requirement",
            "none",
        )
    )

    if visual_type == "none":
        return ""

    visual = question.get(
        "visual",
        {},
    )

    spec = (
        visual.get(
            "spec",
            {},
        )
        if isinstance(
            visual,
            dict,
        )
        else {}
    )

    if not isinstance(
        spec,
        dict,
    ):
        return ""

    cleaned_spec = {
        key: value
        for key, value in spec.items()
        if key not in {
            "caption",
            "_suppress_gate_type_labels",
        }
    }

    return stable_json_fingerprint(
        {
            "visual_type": visual_type,
            "spec": cleaned_spec,
        }
    )


def deterministic_quiz_diversity_signals(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:
    """
    Generic duplicate detection.

    No topic-specific phrases or pattern-specific exceptions are used.
    """
    result = _deterministic_quiz_diversity_v236(
        payload,
        request_payload,
    )

    hard_failures = list(
        result.get(
            "hard_failures",
            [],
        )
    )

    warnings = list(
        result.get(
            "warnings",
            [],
        )
    )

    questions = [
        item
        for item in payload.get(
            "questions",
            [],
        )
        if isinstance(
            item,
            dict,
        )
    ]

    # Exact independent visual reuse.
    visual_groups: dict[
        str,
        list[dict[str, Any]],
    ] = {}

    for question in questions:
        fingerprint = _generic_student_visual_fingerprint(
            question
        )

        if fingerprint:
            visual_groups.setdefault(
                fingerprint,
                [],
            ).append(
                question
            )

    duplicate_visual_groups = []

    for fingerprint, group in visual_groups.items():
        if len(group) < 2:
            continue

        shared_ids = {
            str(
                item.get(
                    "shared_visual_id",
                    "",
                )
                or ""
            ).strip()
            for item in group
        }
        shared_ids.discard("")

        intentionally_shared = bool(
            len(shared_ids) == 1
            and all(
                str(
                    item.get(
                        "shared_visual_id",
                        "",
                    )
                    or ""
                ).strip()
                for item in group
            )
        )

        if intentionally_shared:
            continue

        plan_indexes = [
            safe_int(
                item.get(
                    "plan_index",
                    0,
                )
            )
            for item in group
        ]

        duplicate_visual_groups.append(
            {
                "fingerprint": fingerprint,
                "plan_indexes": plan_indexes,
            }
        )

        hard_failures.append(
            "Duplicate independent visual specification detected across "
            f"questions {plan_indexes}."
        )

    # Strong text/semantic similarity across ANY pair.
    # No hard-coded topic or phrase list is involved.
    for comparison in result.get(
        "comparisons",
        [],
    ):
        if not isinstance(
            comparison,
            dict,
        ):
            continue

        lexical = float(
            comparison.get(
                "lexical_similarity",
                0.0,
            )
            or 0.0
        )

        semantic_raw = comparison.get(
            "semantic_similarity"
        )

        semantic = (
            float(
                semantic_raw
            )
            if semantic_raw is not None
            else None
        )

        left_index = comparison.get(
            "left_plan_index"
        )

        right_index = comparison.get(
            "right_plan_index"
        )

        if (
            lexical >= 0.94
            or (
                semantic is not None
                and semantic >= 0.96
            )
        ):
            hard_failures.append(
                f"Questions {left_index} and {right_index} are near-duplicates "
                "based on generic lexical/semantic similarity."
            )

        elif (
            lexical >= 0.84
            or (
                semantic is not None
                and semantic >= 0.90
            )
        ):
            warnings.append(
                f"Questions {left_index} and {right_index} are similar enough "
                "to require HITL diversity review."
            )

    result[
        "hard_failures"
    ] = list(
        dict.fromkeys(
            hard_failures
        )
    )

    result[
        "warnings"
    ] = list(
        dict.fromkeys(
            warnings
        )
    )

    result[
        "duplicate_visual_groups"
    ] = duplicate_visual_groups

    return result


# Restore the original generic semantic-quality entry point. It resolves the
# latest global `deterministic_quiz_diversity_signals` at runtime, so it will
# use the v2.41 generic duplicate function above.
validate_semantic_quality = _validate_semantic_quality_v236

# ================================================================
# v2.42 — GENERIC MACHINE ANSWER VERIFICATION
# ================================================================

_SAFE_VERIFICATION_FUNCTIONS = {
    "len": len,
    "sum": sum,
    "min": min,
    "max": max,
    "sorted": sorted,
    "all": all,
    "any": any,
    "abs": abs,
    "round": round,
    "range": range,
    "int": int,
    "float": float,
    "str": str,
    "bool": bool,
}

_SAFE_VERIFICATION_AST_NODES = (
    ast.Expression,
    ast.Constant,
    ast.List,
    ast.Tuple,
    ast.Dict,
    ast.Set,
    ast.BinOp,
    ast.UnaryOp,
    ast.BoolOp,
    ast.Compare,
    ast.IfExp,
    ast.Subscript,
    ast.Slice,
    ast.Name,
    ast.Load,
    ast.Store,
    ast.Call,
    ast.GeneratorExp,
    ast.ListComp,
    ast.SetComp,
    ast.DictComp,
    ast.comprehension,
    ast.Add,
    ast.Sub,
    ast.Mult,
    ast.Div,
    ast.FloorDiv,
    ast.Mod,
    ast.Pow,
    ast.UAdd,
    ast.USub,
    ast.Not,
    ast.And,
    ast.Or,
    ast.Eq,
    ast.NotEq,
    ast.Lt,
    ast.LtE,
    ast.Gt,
    ast.GtE,
    ast.In,
    ast.NotIn,
    ast.Is,
    ast.IsNot,
)


def _visual_spec_for_verification(
    question: dict[str, Any],
) -> dict[str, Any]:
    visual = question.get(
        "visual",
        {},
    )

    if not isinstance(
        visual,
        dict,
    ):
        return {}

    spec = visual.get(
        "spec",
        {},
    )

    return (
        spec
        if isinstance(
            spec,
            dict,
        )
        else {}
    )


def _safe_verification_expression(
    expression: str,
    *,
    question: dict[str, Any],
) -> Any:
    """
    Evaluate a restricted pure expression.

    No attributes, imports, assignments, file/network/system access, lambdas,
    or arbitrary function calls are permitted.
    """
    tree = ast.parse(
        expression,
        mode="eval",
    )

    comprehension_locals = {
        node.id
        for node in ast.walk(
            tree
        )
        if isinstance(
            node,
            ast.Name,
        )
        and isinstance(
            node.ctx,
            ast.Store,
        )
    }

    allowed_names = (
        set(
            _SAFE_VERIFICATION_FUNCTIONS
        )
        | {
            "visual_spec",
            "question_text",
            "code",
            "True",
            "False",
            "None",
        }
        | comprehension_locals
    )

    for node in ast.walk(
        tree
    ):
        if not isinstance(
            node,
            _SAFE_VERIFICATION_AST_NODES,
        ):
            raise ValueError(
                "unsupported syntax in verification expression: "
                + type(
                    node
                ).__name__
            )

        if isinstance(
            node,
            ast.Call,
        ):
            if not isinstance(
                node.func,
                ast.Name,
            ) or node.func.id not in _SAFE_VERIFICATION_FUNCTIONS:
                raise ValueError(
                    "verification expression calls a non-whitelisted function"
                )

        if isinstance(
            node,
            ast.Name,
        ) and isinstance(
            node.ctx,
            ast.Load,
        ):
            if node.id not in allowed_names:
                raise ValueError(
                    f"verification expression uses unsupported name {node.id!r}"
                )

    # A literal-only answer is not independent verification.
    body = tree.body
    if isinstance(
        body,
        (
            ast.Constant,
            ast.List,
            ast.Tuple,
            ast.Dict,
            ast.Set,
        ),
    ):
        raise ValueError(
            "verification expression is constant-only and does not verify "
            "the generated evidence"
        )

    visual_spec = _visual_spec_for_verification(
        question
    )

    safe_locals = {
        **_SAFE_VERIFICATION_FUNCTIONS,
        "visual_spec": visual_spec,
        "question_text": str(
            question.get(
                "question_text",
                "",
            )
            or ""
        ),
        "code": str(
            visual_spec.get(
                "code",
                "",
            )
            or ""
        ),
    }

    return eval(
        compile(
            tree,
            "<answer_verification>",
            "eval",
        ),
        {
            "__builtins__": {},
        },
        safe_locals,
    )


def _verification_values_equal(
    actual: Any,
    expected: Any,
) -> bool:
    if isinstance(
        actual,
        bool,
    ) or isinstance(
        expected,
        bool,
    ):
        return actual is expected

    if isinstance(
        actual,
        (int, float),
    ) and isinstance(
        expected,
        (int, float),
    ):
        return math.isclose(
            float(actual),
            float(expected),
            rel_tol=1e-9,
            abs_tol=1e-9,
        )

    if isinstance(
        actual,
        str,
    ) and isinstance(
        expected,
        str,
    ):
        return actual.strip() == expected.strip()

    return actual == expected


def _generic_answer_verification_signals(
    question: dict[str, Any],
) -> dict[str, Any]:
    hard_failures: list[str] = []
    warnings: list[str] = []
    diagnostics: list[dict[str, Any]] = []

    verification = question.get(
        "answer_verification",
        {},
    )

    if not isinstance(
        verification,
        dict,
    ):
        return {
            "status": "REVIEW",
            "hard_failures": [],
            "warnings": [
                "answer_verification is missing; HITL must verify answer correctness"
            ],
            "diagnostics": [],
        }

    mode = str(
        verification.get(
            "mode",
            "manual",
        )
        or "manual"
    ).strip().casefold()

    if mode not in {
        "machine",
        "mixed",
        "manual",
    }:
        hard_failures.append(
            f"answer_verification.mode={mode!r} is unsupported"
        )

    checks = verification.get(
        "checks",
        [],
    )

    if not isinstance(
        checks,
        list,
    ):
        hard_failures.append(
            "answer_verification.checks must be a list"
        )
        checks = []

    marking_guidance = question.get(
        "marking_guidance",
        [],
    )

    criterion_count = (
        len(marking_guidance)
        if isinstance(
            marking_guidance,
            list,
        )
        else 0
    )

    seen_indexes: set[int] = set()

    for check_position, check in enumerate(
        checks,
        start=1,
    ):
        if not isinstance(
            check,
            dict,
        ):
            hard_failures.append(
                f"answer_verification check {check_position} must be an object"
            )
            continue

        criterion_index = safe_int(
            check.get(
                "criterion_index",
                0,
            )
        )

        if (
            criterion_index < 1
            or criterion_index > criterion_count
        ):
            hard_failures.append(
                f"answer_verification check {check_position} has invalid "
                f"criterion_index={criterion_index}"
            )
            continue

        if criterion_index in seen_indexes:
            hard_failures.append(
                f"answer_verification repeats criterion_index={criterion_index}"
            )
            continue

        seen_indexes.add(
            criterion_index
        )

        expression = str(
            check.get(
                "expression",
                "",
            )
            or ""
        ).strip()

        if not expression:
            hard_failures.append(
                f"criterion {criterion_index} machine check has no expression"
            )
            continue

        expected = check.get(
            "expected"
        )

        try:
            actual = _safe_verification_expression(
                expression,
                question=question,
            )
        except Exception as exc:
            hard_failures.append(
                f"criterion {criterion_index} verification expression is invalid: "
                f"{type(exc).__name__}: {exc}"
            )
            continue

        passed = _verification_values_equal(
            actual,
            expected,
        )

        diagnostics.append(
            {
                "criterion_index": criterion_index,
                "expression": expression,
                "expected": expected,
                "actual": actual,
                "passed": passed,
            }
        )

        if not passed:
            hard_failures.append(
                f"criterion {criterion_index} answer verification failed: "
                f"computed={actual!r}, marking_expected={expected!r}"
            )

    if mode == "machine" and not checks:
        hard_failures.append(
            "answer_verification.mode='machine' requires at least one check"
        )

    if mode == "manual" and checks:
        warnings.append(
            "answer_verification.mode='manual' should normally have no machine checks"
        )

    # Do not force every objective criterion into a machine expression:
    # this prevents overfitting and unsafe pseudo-verification.
    if mode in {
        "manual",
        "mixed",
    }:
        warnings.append(
            "Some answer correctness remains HITL-reviewed by design"
        )

    status = (
        "FAIL"
        if hard_failures
        else (
            "REVIEW"
            if warnings
            else "PASS"
        )
    )

    return {
        "status": status,
        "hard_failures": list(
            dict.fromkeys(
                hard_failures
            )
        ),
        "warnings": list(
            dict.fromkeys(
                warnings
            )
        ),
        "diagnostics": diagnostics,
    }


_deterministic_quality_signals_v241 = deterministic_quality_signals


def deterministic_quality_signals(
    question: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:
    result = _deterministic_quality_signals_v241(
        question,
        request_payload,
    )

    verification = _generic_answer_verification_signals(
        question
    )

    result.setdefault(
        "hard_failures",
        [],
    ).extend(
        verification.get(
            "hard_failures",
            [],
        )
    )

    result.setdefault(
        "warnings",
        [],
    ).extend(
        verification.get(
            "warnings",
            [],
        )
    )

    result["hard_failures"] = list(
        dict.fromkeys(
            result["hard_failures"]
        )
    )

    result["warnings"] = list(
        dict.fromkeys(
            result["warnings"]
        )
    )

    result[
        "answer_verification"
    ] = verification

    return result


# ================================================================
# v2.42 — GENERIC ANSWER-SPACE DIVERSITY
# ================================================================

def _marking_answer_space_text(
    question: dict[str, Any],
) -> str:
    marking_guidance = question.get(
        "marking_guidance",
        [],
    )

    if not isinstance(
        marking_guidance,
        list,
    ):
        return ""

    return " ".join(
        str(
            row.get(
                "criterion",
                "",
            )
            or ""
        )
        for row in marking_guidance
        if isinstance(
            row,
            dict,
        )
    ).strip()


_deterministic_quiz_diversity_v241 = deterministic_quiz_diversity_signals


def deterministic_quiz_diversity_signals(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:
    """
    Extend v2.41 generic question/visual diversity with answer-space overlap.

    No topic-specific phrase lists are used.
    """
    result = _deterministic_quiz_diversity_v241(
        payload,
        request_payload,
    )

    hard_failures = list(
        result.get(
            "hard_failures",
            [],
        )
    )

    warnings = list(
        result.get(
            "warnings",
            [],
        )
    )

    questions = [
        item
        for item in payload.get(
            "questions",
            [],
        )
        if isinstance(
            item,
            dict,
        )
    ]

    answer_space_comparisons: list[
        dict[str, Any]
    ] = []

    texts: list[str] = [
        _marking_answer_space_text(
            question
        )
        for question in questions
    ]

    semantic_matrix = None
    semantic_backend = "lexical_only"

    if len(
        [
            text
            for text in texts
            if text
        ]
    ) >= 2:
        try:
            embeddings, semantic_backend = _minilm_embeddings(
                texts
            )

            if embeddings is not None:
                semantic_matrix = (
                    embeddings
                    @ embeddings.T
                )

        except Exception as exc:
            semantic_matrix = None
            semantic_backend = (
                "lexical_fallback: "
                f"{type(exc).__name__}: {exc}"
            )

    for left_index in range(
        len(questions)
    ):
        for right_index in range(
            left_index + 1,
            len(questions),
        ):
            left = questions[
                left_index
            ]

            right = questions[
                right_index
            ]

            left_text = texts[
                left_index
            ]

            right_text = texts[
                right_index
            ]

            if not left_text or not right_text:
                continue

            lexical = lexical_similarity(
                left_text,
                right_text,
            )

            semantic = (
                float(
                    semantic_matrix[
                        left_index,
                        right_index,
                    ]
                )
                if semantic_matrix is not None
                else None
            )

            same_reference = bool(
                str(
                    left.get(
                        "official_reference",
                        "",
                    )
                    or ""
                ).strip()
                and str(
                    left.get(
                        "official_reference",
                        "",
                    )
                    or ""
                ).strip()
                == str(
                    right.get(
                        "official_reference",
                        "",
                    )
                    or ""
                ).strip()
            )

            comparison = {
                "semantic_backend": semantic_backend,
                "left_plan_index": safe_int(
                    left.get(
                        "plan_index",
                        left_index + 1,
                    )
                ),
                "right_plan_index": safe_int(
                    right.get(
                        "plan_index",
                        right_index + 1,
                    )
                ),
                "same_official_reference": same_reference,
                "lexical_similarity": lexical,
                "semantic_similarity": semantic,
            }

            answer_space_comparisons.append(
                comparison
            )

            # Conservative generic thresholds:
            # only strong overlap becomes a hard failure; moderate overlap
            # remains a HITL warning.
            if same_reference and (
                lexical >= 0.90
                or (
                    semantic is not None
                    and semantic >= 0.96
                )
            ):
                hard_failures.append(
                    "Generated questions "
                    f"{comparison['left_plan_index']} and "
                    f"{comparison['right_plan_index']} have near-duplicate "
                    "marking-answer spaces."
                )

            elif same_reference and (
                lexical >= 0.65
                or (
                    semantic is not None
                    and semantic >= 0.84
                )
            ):
                warnings.append(
                    "Generated questions "
                    f"{comparison['left_plan_index']} and "
                    f"{comparison['right_plan_index']} have overlapping "
                    "marking-answer spaces and should receive HITL diversity review."
                )

    result[
        "hard_failures"
    ] = list(
        dict.fromkeys(
            hard_failures
        )
    )

    result[
        "warnings"
    ] = list(
        dict.fromkeys(
            warnings
        )
    )

    result[
        "answer_space_comparisons"
    ] = answer_space_comparisons

    return result

# ================================================================
# v2.43 — MULTI-DIMENSIONAL GENERIC DIVERSITY
# ================================================================

def _diversity_signature_text(
    question: dict[str, Any],
    field: str,
) -> str:
    signature = question.get(
        "diversity_signature",
        {},
    )

    if not isinstance(
        signature,
        dict,
    ):
        return ""

    return str(
        signature.get(
            field,
            "",
        )
        or ""
    ).strip()


def _nonempty_lexical_similarity(
    left: Any,
    right: Any,
) -> float:
    left_text = str(
        left or ""
    ).strip()

    right_text = str(
        right or ""
    ).strip()

    if not left_text or not right_text:
        return 0.0

    return lexical_similarity(
        left_text,
        right_text,
    )


_deterministic_quiz_diversity_v242 = deterministic_quiz_diversity_signals


def deterministic_quiz_diversity_signals(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:
    """
    Extend existing generic wording/semantic/answer-space/visual checks with
    abstract task/scenario/answer-form diversity.

    No topic-specific phrases are used.
    """
    result = _deterministic_quiz_diversity_v242(
        payload,
        request_payload,
    )

    hard_failures = list(
        result.get(
            "hard_failures",
            [],
        )
    )

    warnings = list(
        result.get(
            "warnings",
            [],
        )
    )

    questions = [
        item
        for item in payload.get(
            "questions",
            [],
        )
        if isinstance(
            item,
            dict,
        )
    ]

    comparisons: list[
        dict[str, Any]
    ] = []

    for left_index in range(
        len(questions)
    ):
        for right_index in range(
            left_index + 1,
            len(questions),
        ):
            left = questions[
                left_index
            ]

            right = questions[
                right_index
            ]

            left_reference = str(
                left.get(
                    "official_reference",
                    "",
                )
                or ""
            ).strip()

            right_reference = str(
                right.get(
                    "official_reference",
                    "",
                )
                or ""
            ).strip()

            if (
                not left_reference
                or left_reference != right_reference
            ):
                continue

            scores = {
                "task_intent":
                    _nonempty_lexical_similarity(
                        _diversity_signature_text(
                            left,
                            "task_intent",
                        ),
                        _diversity_signature_text(
                            right,
                            "task_intent",
                        ),
                    ),
                "scenario":
                    _nonempty_lexical_similarity(
                        _diversity_signature_text(
                            left,
                            "scenario",
                        ),
                        _diversity_signature_text(
                            right,
                            "scenario",
                        ),
                    ),
                "answer_form":
                    _nonempty_lexical_similarity(
                        _diversity_signature_text(
                            left,
                            "answer_form",
                        ),
                        _diversity_signature_text(
                            right,
                            "answer_form",
                        ),
                    ),
                "question_text":
                    _nonempty_lexical_similarity(
                        left.get(
                            "question_text",
                            "",
                        ),
                        right.get(
                            "question_text",
                            "",
                        ),
                    ),
                "answer_space":
                    _nonempty_lexical_similarity(
                        _marking_answer_space_text(
                            left
                        ),
                        _marking_answer_space_text(
                            right
                        ),
                    ),
            }

            high_overlap = [
                dimension
                for dimension, value in scores.items()
                if value >= 0.72
            ]

            comparison = {
                "left_plan_index":
                    safe_int(
                        left.get(
                            "plan_index",
                            left_index + 1,
                        )
                    ),
                "right_plan_index":
                    safe_int(
                        right.get(
                            "plan_index",
                            right_index + 1,
                        )
                    ),
                "official_reference":
                    left_reference,
                "dimension_scores":
                    scores,
                "high_overlap_dimensions":
                    high_overlap,
            }

            comparisons.append(
                comparison
            )

            # Strong overlap across multiple independent dimensions is unlikely
            # to be a genuinely distinct question.
            if (
                len(
                    high_overlap
                ) >= 4
                and (
                    scores[
                        "task_intent"
                    ] >= 0.82
                    or scores[
                        "answer_space"
                    ] >= 0.82
                )
            ):
                hard_failures.append(
                    "Questions "
                    f"{comparison['left_plan_index']} and "
                    f"{comparison['right_plan_index']} overlap across too many "
                    "generic diversity dimensions for the same official reference."
                )

            elif len(
                high_overlap
            ) >= 3:
                warnings.append(
                    "Questions "
                    f"{comparison['left_plan_index']} and "
                    f"{comparison['right_plan_index']} overlap across "
                    f"{len(high_overlap)} generic diversity dimensions and "
                    "should receive HITL diversity review."
                )

    result[
        "hard_failures"
    ] = list(
        dict.fromkeys(
            hard_failures
        )
    )

    result[
        "warnings"
    ] = list(
        dict.fromkeys(
            warnings
        )
    )

    result[
        "multi_dimensional_diversity"
    ] = comparisons

    return result


## 10A. Generic deterministic QA layer — correctness before HITL

This layer is intentionally **thin and non-topic-specific**.

It does not try to encode the AQA syllabus in Python and it does not call a
second reviewer model. It only checks facts that can be proved safely from the
learner-facing question itself:

- a question references a trace table/code/visual that is actually present;
- a question that says a program/algorithm is run, executed, traced, or shown must actually supply that learner-facing source;
- executable Python does not call an undefined subroutine/function;
- small safe Python snippets can be executed with restricted builtins and a
  line-event limit;
- explicit objective marking targets such as printed outputs, final variable
  values and indexed array values can be compared with the independently
  computed result.

Every question receives one of three QA states:

- `PASS` — at least one independent local check was proved and no contradiction
  was found;
- `FAIL` — an objective contradiction or missing learner-facing dependency was
  proved;
- `NOT_VERIFIABLE` — the layer deliberately abstains and leaves correctness to
  mandatory HITL.

A `FAIL` is question-local, so the existing Plan C policy may use **one compact
targeted repair for that plan index only**. The QA layer itself uses **zero API
calls** and never triggers whole-quiz regeneration.


In [ ]:
# ================================================================
# v3.3 — THIN GENERIC QA LAYER
# ================================================================
# Purpose:
#   * verify only what Python can prove safely
#   * catch missing learner-facing resources/references
#   * never encode topic-specific answer rules
#   * never call another LLM
#   * leave uncertain/open-ended items to mandatory HITL
#
# The QA result is deliberately tri-state:
#   PASS            -> at least one independent local check succeeded
#   FAIL            -> an objective contradiction/resource defect was proven
#   NOT_VERIFIABLE  -> no safe independent check was available
# ================================================================

QA_VALIDATION_SCHEMA_VERSION = "agent2-generic-qa-v1.0.0"
QA_VALIDATION_REPORT_PATH = OUTPUT_DIR / "qa_validation_report.json"

_QA_SAFE_DIRECT_CALLS = {
    "print",
    "range",
    "len",
    "abs",
    "min",
    "max",
    "sum",
    "sorted",
    "all",
    "any",
    "int",
    "float",
    "str",
    "bool",
    "list",
    "tuple",
    "dict",
    "set",
    "enumerate",
    "zip",
}

_QA_KNOWN_BUT_NONEXECUTABLE_CALLS = {
    "input",
}

_QA_SAFE_METHOD_CALLS = {
    "append",
    "extend",
    "join",
    "strip",
    "lower",
    "upper",
    "split",
    "count",
    "index",
}

_QA_DANGEROUS_AST_NODES = (
    ast.Import,
    ast.ImportFrom,
    ast.With,
    ast.AsyncWith,
    ast.Try,
    ast.Raise,
    ast.ClassDef,
    ast.Global,
    ast.Nonlocal,
    ast.Delete,
    ast.Lambda,
    ast.Await,
    ast.Yield,
    ast.YieldFrom,
    ast.NamedExpr,
)


def _qa_visual_payload(
    question: dict[str, Any],
) -> tuple[str, dict[str, Any]]:
    visual = question.get("visual", {})
    if not isinstance(visual, dict):
        return "none", {}

    visual_type = normalize_visual_requirement(
        visual.get(
            "type",
            question.get("visual_requirement", "none"),
        )
    )

    spec = visual.get("spec", {})
    if not isinstance(spec, dict):
        spec = {}

    return visual_type, spec


def _qa_visible_code(
    question: dict[str, Any],
) -> str:
    _, spec = _qa_visual_payload(question)

    for key in (
        "code",
        "pseudocode",
        "program",
        "text",
    ):
        value = spec.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()

    # Embedded learner-facing code is allowed to live directly in question_text.
    text_value = str(question.get("question_text", "") or "")

    fenced = re.search(
        r"```(?:python|py)?\s*(.*?)```",
        text_value,
        flags=re.IGNORECASE | re.DOTALL,
    )

    if fenced:
        return str(fenced.group(1) or "").strip()

    return ""


def _qa_referenced_resource_signals(
    question: dict[str, Any],
) -> dict[str, Any]:
    hard_failures: list[str] = []
    warnings: list[str] = []
    diagnostics: list[dict[str, Any]] = []

    q_text = str(question.get("question_text", "") or "")
    lower_q = q_text.casefold()

    visual_type, _ = _qa_visual_payload(question)
    code = _qa_visible_code(question)

    trace_requested = bool(
        re.search(r"\btrace\s+table\b", lower_q)
    )

    # Generic learner-source dependency detection.
    #
    # This is intentionally based on the literal learner-facing wording, not
    # on topic names, marks, plan indexes, or question templates.
    #
    # Examples it catches generically:
    #   "Refer to the code..."
    #   "The following Python-like program is run..."
    #   "The algorithm below is executed..."
    #   "Using the pseudocode shown..."
    #
    # If the question claims that a program/algorithm/code resource exists and
    # the learner needs it to derive an answer, that resource must actually be
    # present in question text or the visual payload.
    explicit_code_reference = bool(
        re.search(
            r"\b(?:code|program|algorithm|pseudocode)\s+"
            r"(?:block|shown|visual|below|above)\b",
            lower_q,
        )
        or "refer to the code" in lower_q
        or "refer to the program" in lower_q
        or "refer to the algorithm" in lower_q
        or "use the code" in lower_q
        or "use the program" in lower_q
        or "use the algorithm" in lower_q
    )

    following_program_reference = bool(
        re.search(
            r"\b(?:the\s+)?following\s+"
            r"(?:python(?:-like)?\s+|python\s+|pseudo\s+)?"
            r"(?:code|program|algorithm|pseudocode)\b",
            lower_q,
        )
    )

    executed_program_reference = bool(
        re.search(
            r"\b(?:code|program|algorithm|pseudocode)\b"
            r".{0,60}\b(?:is|was|will\s+be)?\s*"
            r"(?:run|executed|traced)\b",
            lower_q,
        )
    )

    code_resource_requested = bool(
        explicit_code_reference
        or following_program_reference
        or executed_program_reference
    )

    generic_visual_requested = bool(
        "refer to the visual" in lower_q
        or "shown in the visual" in lower_q
        or "use the visual" in lower_q
    )

    if trace_requested:
        trace_present = visual_type == "trace_table"

        diagnostics.append(
            {
                "check": "trace_table_reference",
                "requested": True,
                "visual_type": visual_type,
                "passed": trace_present,
            }
        )

        if not trace_present:
            hard_failures.append(
                "Question explicitly references a trace table, but no trace_table "
                "visual is supplied for the learner."
            )

    if code_resource_requested:
        code_present = bool(code)

        diagnostics.append(
            {
                "check": "code_reference",
                "requested": True,
                "visual_type": visual_type,
                "code_present": code_present,
                "passed": code_present,
            }
        )

        if not code_present:
            hard_failures.append(
                "Question depends on learner-visible code/program/algorithm text "
                "to derive the answer, but that source is missing from both the "
                "question text and visual payload. Make the question self-contained "
                "by supplying the required learner-facing code/algorithm."
            )

    if generic_visual_requested:
        visual_present = visual_type not in {"", "none"}

        diagnostics.append(
            {
                "check": "generic_visual_reference",
                "requested": True,
                "visual_type": visual_type,
                "passed": visual_present,
            }
        )

        if not visual_present:
            hard_failures.append(
                "Question explicitly references a visual, but no learner-facing "
                "visual is supplied."
            )

    return {
        "hard_failures": list(dict.fromkeys(hard_failures)),
        "warnings": list(dict.fromkeys(warnings)),
        "diagnostics": diagnostics,
    }


def _qa_python_preflight(
    code: str,
) -> dict[str, Any]:
    """
    Conservative Python-only preflight.

    Syntax that is not valid Python is treated as NOT_APPLICABLE rather than
    wrong; this lets AQA pseudocode continue to HITL without false failure.
    """
    try:
        tree = ast.parse(
            code,
            mode="exec",
        )
    except SyntaxError as exc:
        return {
            "python_parseable": False,
            "safe_to_execute": False,
            "undefined_direct_calls": [],
            "reason": f"not executable Python: {exc.msg}",
        }

    defined_functions = {
        node.name
        for node in ast.walk(tree)
        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            ),
        )
    }

    undefined_direct_calls: list[str] = []
    nonexecuted_calls: list[str] = []

    for node in ast.walk(tree):
        if isinstance(
            node,
            _QA_DANGEROUS_AST_NODES,
        ):
            return {
                "python_parseable": True,
                "safe_to_execute": False,
                "undefined_direct_calls": [],
                "reason": (
                    "execution skipped because code contains unsupported "
                    f"{type(node).__name__}"
                ),
            }

        if (
            isinstance(node, ast.Name)
            and str(node.id).startswith("__")
        ):
            return {
                "python_parseable": True,
                "safe_to_execute": False,
                "undefined_direct_calls": [],
                "reason": "execution skipped because dunder names are not allowed",
            }

        if (
            isinstance(node, ast.Attribute)
            and str(node.attr).startswith("__")
        ):
            return {
                "python_parseable": True,
                "safe_to_execute": False,
                "undefined_direct_calls": [],
                "reason": (
                    "execution skipped because dunder attributes are not allowed"
                ),
            }

        if not isinstance(node, ast.Call):
            continue

        if isinstance(node.func, ast.Name):
            name = str(node.func.id)

            if name in _QA_KNOWN_BUT_NONEXECUTABLE_CALLS:
                nonexecuted_calls.append(name)
                continue

            if (
                name in _QA_SAFE_DIRECT_CALLS
                or name in defined_functions
            ):
                continue

            undefined_direct_calls.append(name)

        elif isinstance(node.func, ast.Attribute):
            method = str(node.func.attr)

            if method not in _QA_SAFE_METHOD_CALLS:
                return {
                    "python_parseable": True,
                    "safe_to_execute": False,
                    "undefined_direct_calls": [],
                    "reason": (
                        "execution skipped because method call "
                        f"{method!r} is not in the small safe-method allowlist"
                    ),
                }

        else:
            return {
                "python_parseable": True,
                "safe_to_execute": False,
                "undefined_direct_calls": [],
                "reason": (
                    "execution skipped because a dynamic call target is used"
                ),
            }

    undefined_direct_calls = sorted(
        set(undefined_direct_calls)
    )

    if undefined_direct_calls:
        return {
            "python_parseable": True,
            "safe_to_execute": False,
            "undefined_direct_calls": undefined_direct_calls,
            "reason": "undefined direct function/subroutine reference",
        }

    if nonexecuted_calls:
        return {
            "python_parseable": True,
            "safe_to_execute": False,
            "undefined_direct_calls": [],
            "reason": (
                "execution skipped because code requires external input via "
                + ", ".join(
                    sorted(
                        set(nonexecuted_calls)
                    )
                )
            ),
        }

    return {
        "python_parseable": True,
        "safe_to_execute": True,
        "undefined_direct_calls": [],
        "reason": "safe executable Python subset",
    }


def _qa_execute_python(
    code: str,
    *,
    max_line_events: int = 20000,
) -> dict[str, Any]:
    """
    Execute the small preflighted Python subset with restricted builtins.

    A line-event guard aborts accidental infinite loops. Because imports,
    dunder access and non-whitelisted method calls are rejected by preflight,
    generated code has no file/network/system capability.
    """
    import contextlib
    import io
    import sys

    safe_builtins = {
        "print": print,
        "range": range,
        "len": len,
        "abs": abs,
        "min": min,
        "max": max,
        "sum": sum,
        "sorted": sorted,
        "all": all,
        "any": any,
        "int": int,
        "float": float,
        "str": str,
        "bool": bool,
        "list": list,
        "tuple": tuple,
        "dict": dict,
        "set": set,
        "enumerate": enumerate,
        "zip": zip,
    }

    scope: dict[str, Any] = {
        "__builtins__": safe_builtins,
    }

    buffer = io.StringIO()
    line_events = 0

    def _guard(frame, event, arg):
        nonlocal line_events

        if event == "line":
            line_events += 1

            if line_events > max(
                100,
                int(max_line_events),
            ):
                raise RuntimeError(
                    "QA execution step limit exceeded"
                )

        return _guard

    previous_trace = sys.gettrace()

    try:
        sys.settrace(_guard)

        with contextlib.redirect_stdout(buffer):
            exec(
                compile(
                    code,
                    "<generated_quiz_code>",
                    "exec",
                ),
                scope,
                scope,
            )

    except Exception as exc:
        return {
            "ok": False,
            "stdout": buffer.getvalue().splitlines(),
            "globals": {},
            "error": f"{type(exc).__name__}: {exc}",
        }

    finally:
        sys.settrace(previous_trace)

    simple: dict[str, Any] = {}

    for key, value in scope.items():
        if key.startswith("__") or callable(value):
            continue

        try:
            json.dumps(value)
        except Exception:
            continue

        simple[key] = value

    return {
        "ok": True,
        "stdout": buffer.getvalue().splitlines(),
        "globals": simple,
        "error": None,
        "line_events": line_events,
    }


def _qa_parse_expected_literal(
    raw_value: str,
) -> Any:
    value = str(
        raw_value or ""
    ).strip().rstrip(".;")

    if (
        len(value) >= 2
        and value[0] == value[-1]
        and value[0] in {"'", '"'}
    ):
        value = value[1:-1]

    try:
        return ast.literal_eval(value)
    except Exception:
        pass

    folded = value.casefold()

    if folded == "true":
        return True

    if folded == "false":
        return False

    if re.fullmatch(r"-?\d+", value):
        return int(value)

    if re.fullmatch(
        r"-?(?:\d+\.\d*|\d*\.\d+)",
        value,
    ):
        return float(value)

    return value


_QA_ORDINAL_INDEX = {
    "first": 0,
    "second": 1,
    "third": 2,
    "fourth": 3,
    "fifth": 4,
    "sixth": 5,
    "seventh": 6,
    "eighth": 7,
    "ninth": 8,
    "tenth": 9,
}


def _qa_compare_values(
    actual: Any,
    expected: Any,
) -> bool:
    if isinstance(actual, str):
        actual = _qa_parse_expected_literal(
            actual
        )

    return _verification_values_equal(
        actual,
        expected,
    )


def _qa_objective_execution_checks(
    question: dict[str, Any],
    execution: dict[str, Any],
) -> dict[str, Any]:
    """
    Independently compare only highly explicit marking targets with observable
    program behaviour. Ambiguous criteria are ignored rather than guessed.
    """
    hard_failures: list[str] = []
    diagnostics: list[dict[str, Any]] = []

    stdout = execution.get(
        "stdout",
        [],
    )
    if not isinstance(stdout, list):
        stdout = []

    globals_map = execution.get(
        "globals",
        {},
    )
    if not isinstance(globals_map, dict):
        globals_map = {}

    marking = question.get(
        "marking_guidance",
        [],
    )
    if not isinstance(marking, list):
        marking = []

    q_text = str(
        question.get(
            "question_text",
            "",
        )
        or ""
    ).casefold()

    for criterion_index, row in enumerate(
        marking,
        start=1,
    ):
        if not isinstance(row, dict):
            continue

        criterion = str(
            row.get(
                "criterion",
                "",
            )
            or ""
        ).strip()

        if not criterion:
            continue

        checked = False
        passed = None
        actual: Any = None
        expected: Any = None
        check_type = ""

        # 1) Explicit ordinal stdout targets:
        #    "First printed value is 12"
        #    "Third printed line is exactly: Found"
        ordinal_match = re.search(
            r"\b(first|second|third|fourth|fifth|sixth|seventh|eighth|ninth|tenth)"
            r"\s+printed(?:\s+(?:line|value|integer|output))?"
            r".*?(?:\bis(?:\s+exactly)?\b|=)\s*:?\s*(.+?)\s*$",
            criterion,
            flags=re.IGNORECASE,
        )

        if ordinal_match:
            position = _QA_ORDINAL_INDEX[
                ordinal_match.group(1).casefold()
            ]

            expected = _qa_parse_expected_literal(
                ordinal_match.group(2)
            )

            actual = (
                stdout[position]
                if position < len(stdout)
                else None
            )

            checked = True
            check_type = "ordinal_stdout"
            passed = (
                position < len(stdout)
                and _qa_compare_values(
                    actual,
                    expected,
                )
            )

        # 2) Single printed-result target.
        if not checked:
            printed_match = re.search(
                r"\bprinted\s+(?:value|integer|output)"
                r".*?(?:\bis(?:\s+exactly)?\b|=)\s*:?\s*(.+?)\s*$",
                criterion,
                flags=re.IGNORECASE,
            )

            if (
                printed_match
                and len(stdout) == 1
            ):
                expected = _qa_parse_expected_literal(
                    printed_match.group(1)
                )
                actual = stdout[0]
                checked = True
                check_type = "single_stdout"
                passed = _qa_compare_values(
                    actual,
                    expected,
                )

        # 3) "Ping is printed 4 times" against captured stdout.
        if not checked:
            count_match = re.search(
                r"^[\"']?(.+?)[\"']?\s+is\s+printed\s+(\d+)\s+times\b",
                criterion,
                flags=re.IGNORECASE,
            )

            if count_match:
                token = str(
                    count_match.group(1)
                ).strip().strip("'\"")

                expected = int(
                    count_match.group(2)
                )

                actual = sum(
                    1
                    for line in stdout
                    if str(line).strip() == token
                )

                checked = True
                check_type = "stdout_occurrence_count"
                passed = actual == expected

        # 4) Explicit final variable value.
        if not checked:
            final_var_match = re.search(
                r"\bfinal\s+value\s+of\s+([A-Za-z_]\w*)"
                r"\s+(?:\bis\b|=)\s*:?\s*(.+?)\s*$",
                criterion,
                flags=re.IGNORECASE,
            )

            if final_var_match:
                variable = final_var_match.group(1)

                if variable in globals_map:
                    expected = _qa_parse_expected_literal(
                        final_var_match.group(2)
                    )
                    actual = globals_map.get(
                        variable
                    )
                    checked = True
                    check_type = "final_variable"
                    passed = _qa_compare_values(
                        actual,
                        expected,
                    )

        # 5) Explicit array/list index lookup.
        if not checked:
            index_match = re.search(
                r"\bvalue(?:\s+stored)?\s+at\s+"
                r"([A-Za-z_]\w*(?:\[-?\d+\])+)"
                r"\s+(?:\bis\b|=)\s*:?\s*(.+?)\s*$",
                criterion,
                flags=re.IGNORECASE,
            )

            if index_match:
                indexed_expression = index_match.group(1)

                expected = _qa_parse_expected_literal(
                    index_match.group(2)
                )

                try:
                    expr_tree = ast.parse(
                        indexed_expression,
                        mode="eval",
                    )

                    node = expr_tree.body
                    valid_shape = isinstance(
                        node,
                        (
                            ast.Name,
                            ast.Subscript,
                        ),
                    )

                    for subnode in ast.walk(
                        expr_tree
                    ):
                        if isinstance(
                            subnode,
                            ast.Name,
                        ):
                            if subnode.id not in globals_map:
                                valid_shape = False

                        elif isinstance(
                            subnode,
                            (
                                ast.Expression,
                                ast.Name,
                                ast.Load,
                                ast.Subscript,
                                ast.Constant,
                                ast.UnaryOp,
                                ast.USub,
                            ),
                        ):
                            continue

                        else:
                            valid_shape = False

                    if valid_shape:
                        actual = eval(
                            compile(
                                expr_tree,
                                "<qa_index_lookup>",
                                "eval",
                            ),
                            {
                                "__builtins__": {},
                            },
                            globals_map,
                        )

                        checked = True
                        check_type = "indexed_value"
                        passed = _qa_compare_values(
                            actual,
                            expected,
                        )

                except Exception:
                    pass

        # 6) Generic "printed number ... is X" only when the question itself
        #    explicitly says this is the first printed result.
        if not checked:
            first_print_match = re.search(
                r"\bprinted\s+number(?:\s+of\s+.+?)?"
                r"\s+(?:\bis\b|=)\s*:?\s*(.+?)\s*$",
                criterion,
                flags=re.IGNORECASE,
            )

            if (
                first_print_match
                and stdout
                and (
                    "printed first" in q_text
                    or "first printed" in q_text
                )
            ):
                expected = _qa_parse_expected_literal(
                    first_print_match.group(1)
                )

                actual = stdout[0]
                checked = True
                check_type = (
                    "first_stdout_from_question_contract"
                )

                passed = _qa_compare_values(
                    actual,
                    expected,
                )

        if checked:
            diagnostics.append(
                {
                    "criterion_index": criterion_index,
                    "check_type": check_type,
                    "criterion": criterion,
                    "expected": expected,
                    "actual": actual,
                    "passed": bool(passed),
                }
            )

            if not passed:
                hard_failures.append(
                    "Independent QA contradiction for marking criterion "
                    f"{criterion_index}: {check_type} computed {actual!r}, "
                    f"but marking guidance expects {expected!r}."
                )

    return {
        "hard_failures":
            list(
                dict.fromkeys(
                    hard_failures
                )
            ),

        "diagnostics":
            diagnostics,

        "verified_check_count":
            len(
                diagnostics
            ),
    }


def _generic_independent_qa_signals(
    question: dict[str, Any],
) -> dict[str, Any]:
    hard_failures: list[str] = []
    warnings: list[str] = []
    diagnostics: list[dict[str, Any]] = []
    verified_check_count = 0

    resource = _qa_referenced_resource_signals(
        question
    )

    hard_failures.extend(
        resource.get(
            "hard_failures",
            [],
        )
    )

    warnings.extend(
        resource.get(
            "warnings",
            [],
        )
    )

    diagnostics.extend(
        resource.get(
            "diagnostics",
            [],
        )
    )

    code = _qa_visible_code(
        question
    )

    execution_status = "NOT_APPLICABLE"
    execution_result: dict[str, Any] | None = None

    if code:
        preflight = _qa_python_preflight(
            code
        )

        diagnostics.append(
            {
                "check": "python_preflight",
                **preflight,
            }
        )

        undefined_calls = preflight.get(
            "undefined_direct_calls",
            [],
        )

        if undefined_calls:
            hard_failures.append(
                "Learner-visible Python references undefined function/subroutine "
                "name(s): "
                + ", ".join(
                    str(item)
                    for item in undefined_calls
                )
                + ". Define the function or supply its behaviour explicitly."
            )

            execution_status = "FAIL"

        elif preflight.get(
            "safe_to_execute"
        ):
            execution_result = _qa_execute_python(
                code
            )

            execution_status = (
                "PASS"
                if execution_result.get(
                    "ok"
                )
                else "NOT_VERIFIABLE"
            )

            diagnostics.append(
                {
                    "check":
                        "restricted_python_execution",

                    "ok":
                        bool(
                            execution_result.get(
                                "ok"
                            )
                        ),

                    "stdout":
                        execution_result.get(
                            "stdout",
                            [],
                        ),

                    "simple_globals":
                        execution_result.get(
                            "globals",
                            {},
                        ),

                    "error":
                        execution_result.get(
                            "error"
                        ),
                }
            )

            if execution_result.get(
                "ok"
            ):
                objective = _qa_objective_execution_checks(
                    question,
                    execution_result,
                )

                hard_failures.extend(
                    objective.get(
                        "hard_failures",
                        [],
                    )
                )

                diagnostics.extend(
                    objective.get(
                        "diagnostics",
                        [],
                    )
                )

                verified_check_count += safe_int(
                    objective.get(
                        "verified_check_count",
                        0,
                    )
                )

        else:
            execution_status = "NOT_VERIFIABLE"

    if hard_failures:
        status = "FAIL"

    elif verified_check_count > 0:
        status = "PASS"

    else:
        status = "NOT_VERIFIABLE"

    return {
        "schema_version":
            QA_VALIDATION_SCHEMA_VERSION,

        "status":
            status,

        "hard_failures":
            list(
                dict.fromkeys(
                    hard_failures
                )
            ),

        "warnings":
            list(
                dict.fromkeys(
                    warnings
                )
            ),

        "verified_check_count":
            verified_check_count,

        "execution_status":
            execution_status,

        "diagnostics":
            diagnostics,
    }


def _qa_relation_parser_regression_probe() -> dict[str, Any]:
    """
    Zero-API regression probe for objective marking criteria.

    Common forms using '=', 'is', and 'is exactly' must all be parsed.
    """
    pattern = re.compile(
        r"\b(first|second|third|fourth|fifth|sixth|seventh|eighth|ninth|tenth)"
        r"\s+printed(?:\s+(?:line|value|integer|output))?"
        r".*?(?:\bis(?:\s+exactly)?\b|=)\s*:?\s*(.+?)\s*$",
        flags=re.IGNORECASE,
    )

    examples = {
        "equals": "First printed value = 27",
        "is": "First printed value is 27",
        "is_exactly": "First printed value is exactly 27",
    }

    parsed = {}
    for name, text in examples.items():
        match = pattern.search(text)
        parsed[name] = match.group(2).strip() if match else None

    return {
        "all_relation_forms_parse": all(
            value == "27"
            for value in parsed.values()
        ),
        "parsed": parsed,
    }


def _qa_self_containment_regression_probe() -> dict[str, Any]:
    """
    Lightweight local regression probe for the generic self-containment rule.

    This is NOT used as a generation rule and makes zero provider calls.
    """
    missing_program_question = {
        "question_text": (
            "The following Python-like program is run. "
            "Use the trace table visual to give the values after step 1."
        ),
        "visual_requirement": "trace_table",
        "visual": {
            "type": "trace_table",
            "spec": {
                "columns": ["Step", "i", "total"],
                "rows": [
                    ["Initial", "0", "0"],
                    ["1", "", ""],
                ],
            },
        },
    }

    supplied_program_question = {
        "question_text": (
            "The following Python-like program is run. "
            "Use the code shown to give the printed value."
        ),
        "visual_requirement": "code_block",
        "visual": {
            "type": "code_block",
            "spec": {
                "code": "x = 1\\nprint(x)",
            },
        },
    }

    missing_result = _qa_referenced_resource_signals(
        missing_program_question
    )
    supplied_result = _qa_referenced_resource_signals(
        supplied_program_question
    )

    return {
        "missing_program_is_flagged":
            bool(
                missing_result.get(
                    "hard_failures"
                )
            ),

        "supplied_program_is_not_flagged":
            not bool(
                supplied_result.get(
                    "hard_failures"
                )
            ),

        "missing_program_failures":
            missing_result.get(
                "hard_failures",
                [],
            ),
    }


# Add independent QA to the existing deterministic semantic layer.
# This wrapper does not change generation, batching, token preflight, or HITL.
_deterministic_quality_signals_pre_generic_qa = (
    deterministic_quality_signals
)


def deterministic_quality_signals(
    question: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:
    result = _deterministic_quality_signals_pre_generic_qa(
        question,
        request_payload,
    )

    qa = _generic_independent_qa_signals(
        question
    )

    result.setdefault(
        "hard_failures",
        [],
    ).extend(
        qa.get(
            "hard_failures",
            [],
        )
    )

    result.setdefault(
        "warnings",
        [],
    ).extend(
        qa.get(
            "warnings",
            [],
        )
    )

    result["hard_failures"] = list(
        dict.fromkeys(
            result["hard_failures"]
        )
    )

    result["warnings"] = list(
        dict.fromkeys(
            result["warnings"]
        )
    )

    result["independent_qa"] = qa

    return result


_validate_semantic_quality_pre_generic_qa_report = (
    validate_semantic_quality
)


def validate_semantic_quality(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:
    result = _validate_semantic_quality_pre_generic_qa_report(
        payload,
        request_payload,
    )

    signals = result.get(
        "deterministic_signals",
        {},
    )

    if not isinstance(
        signals,
        dict,
    ):
        signals = {}

    per_question = {}

    status_counts = {
        "PASS": 0,
        "FAIL": 0,
        "NOT_VERIFIABLE": 0,
    }

    failed_plan_indexes: list[int] = []

    plan_by_id = {
        str(
            item.get(
                "generated_question_id",
                "",
            )
            or ""
        ): safe_int(
            item.get(
                "plan_index"
            )
        )
        for item in payload.get(
            "questions",
            [],
        )
        if isinstance(
            item,
            dict,
        )
    }

    for question_id, signal in signals.items():
        if not isinstance(
            signal,
            dict,
        ):
            continue

        qa = signal.get(
            "independent_qa",
            {},
        )

        if not isinstance(
            qa,
            dict,
        ):
            continue

        qa_status = str(
            qa.get(
                "status",
                "NOT_VERIFIABLE",
            )
            or "NOT_VERIFIABLE"
        )

        if qa_status not in status_counts:
            qa_status = "NOT_VERIFIABLE"

        status_counts[
            qa_status
        ] += 1

        plan_index = safe_int(
            plan_by_id.get(
                str(question_id),
                0,
            )
        )

        per_question[
            str(question_id)
        ] = {
            "plan_index":
                plan_index,
            **qa,
        }

        if (
            qa_status == "FAIL"
            and plan_index > 0
        ):
            failed_plan_indexes.append(
                plan_index
            )

    report = {
        "schema_version":
            QA_VALIDATION_SCHEMA_VERSION,

        "generated_at_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "pipeline_version":
            NOTEBOOK06_PIPELINE_VERSION,

        "quiz_mode":
            QUIZ_MODE,

        "question_count":
            len(
                [
                    item
                    for item in payload.get(
                        "questions",
                        [],
                    )
                    if isinstance(
                        item,
                        dict,
                    )
                ]
            ),

        "status_counts":
            status_counts,

        "failed_plan_indexes":
            sorted(
                set(
                    failed_plan_indexes
                )
            ),

        "per_question":
            per_question,

        "policy": {
            "llm_calls_used_by_qa":
                0,

            "uncertain_items":
                "NOT_VERIFIABLE -> mandatory HITL",

            "hard_failure_repair":
                "existing one-question targeted repair policy",

            "whole_quiz_regeneration":
                False,
        },
    }

    QA_VALIDATION_REPORT_PATH.write_text(
        json.dumps(
            report,
            indent=2,
            ensure_ascii=False,
            default=str,
        ),
        encoding="utf-8",
    )

    result[
        "independent_qa_report"
    ] = {
        "path":
            str(
                QA_VALIDATION_REPORT_PATH
            ),

        "status_counts":
            status_counts,

        "failed_plan_indexes":
            sorted(
                set(
                    failed_plan_indexes
                )
            ),

        "schema_version":
            QA_VALIDATION_SCHEMA_VERSION,
    }

    return result


## v2.43.1 — MiniLM diversity hotfix

The answer-space diversity layer now uses the notebook's existing
`_minilm_embeddings()` contract correctly:

```text
embeddings, backend = _minilm_embeddings(texts)
```

There is no separate `ENABLE_MINILM_SIMILARITY` flag. If MiniLM cannot load,
the existing lexical fallback continues automatically instead of crashing.


## Visual quality gates — v2.38

Before HITL/PDF release, generated questions now pass these additional checks:

```text
question generation
      ↓
missing-visual fail-closed
      ↓
answer-leakage check
      ↓
question ↔ visual compatibility
      ↓
question ↔ visual consistency
      ↓
duplicate question / duplicate visual guard
      ↓
visual dependency / relevance check
      ↓
terminology consistency
      ↓
visual rendering
      ↓
PDF visual-integrity preflight
```

Knowledge-base exemplar metadata still informs **assessment form**, but v2.38 adds a compatibility matrix so a reviewed exemplar visual cannot force an implausible visual for the approved topic.

Examples:
- arrays → array grid / trace table / code block
- network topology → network diagram
- logic gates → logic gate diagram / truth table
- algorithms → code block / flowchart / trace table
- CPU/processor topics → CPU block diagram


## Final visual materialization, validation and release integration

This layer is deterministic Python only. It validates each generated `visual_spec`, renders a black-and-white PNG, hashes the asset, attaches `visual_path` / render metadata to the candidate, and records per-phase/per-type counts.

A required visual that is invalid or cannot be rendered turns structural validation into `FAIL`, so mandatory HITL never receives a broken visual. No image-generation API or second semantic-review LLM is introduced.


In [ ]:
# ================================================================
# VISUAL RENDERING DEPENDENCY BOOTSTRAP
# ================================================================
# Notebook 06 is also executed from fresh project virtual environments.
# Install matplotlib only when it is missing so visual questions do not fail
# with: ModuleNotFoundError: No module named 'matplotlib'.
# The non-interactive Agg backend is used because Notebook 06 renders PNG
# assets in the background and does not require a desktop GUI.
# ================================================================

import subprocess
import sys

try:
    import matplotlib
except ImportError:
    print("matplotlib is not installed; installing it for deterministic visual rendering...")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "matplotlib",
        ]
    )
    import matplotlib

matplotlib.use("Agg", force=True)

def _visual_asset_path(question: dict[str, Any]) -> Path:
    generated_id = str(question.get("generated_question_id", "") or "").strip()
    if not generated_id:
        generated_id = "GEN_" + str(safe_int(question.get("plan_index"))).zfill(3)

    safe_id = re.sub(r"[^A-Za-z0-9_-]+", "_", generated_id).strip("_") or "GEN"
    visual_hash = stable_json_fingerprint(question.get("visual", {}))[:12]
    return VISUAL_DIR / f"{safe_id}_{visual_hash}.png"


# Backward-compatible path helper.
def _phase1_visual_asset_path(question: dict[str, Any]) -> Path:
    return _visual_asset_path(question)


def _save_visual_figure(fig: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=180, bbox_inches="tight", facecolor="white")
    from matplotlib import pyplot as plt
    plt.close(fig)


def _phase1_save_figure(fig: Any, path: Path) -> None:
    _save_visual_figure(fig, path)


def _render_code_block(spec: dict[str, Any], path: Path) -> None:
    from matplotlib import pyplot as plt
    from matplotlib.patches import Rectangle

    code = str(spec.get("code", "") or "").rstrip()
    caption = str(spec.get("caption", "") or "").strip()
    lines = code.splitlines() or [""]
    longest = max(len(line) for line in lines)

    # Size to content rather than using a large minimum canvas. This keeps
    # short pseudocode/code visuals compact in both Streamlit and PDF export.
    width = min(10.5, max(4.8, 1.25 + 0.082 * longest))
    height = min(7.0, max(1.55, 0.60 + 0.30 * len(lines)))

    fig, ax = plt.subplots(figsize=(width, height))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.add_patch(
        Rectangle(
            (0.015, 0.06),
            0.97,
            0.88,
            linewidth=1.1,
            edgecolor="black",
            facecolor="white",
        )
    )
    ax.text(
        0.04,
        0.90,
        code,
        ha="left",
        va="top",
        family="monospace",
        fontsize=10.0,
        color="black",
        linespacing=1.12,
    )
    if caption:
        ax.set_title(caption, fontsize=10.0, pad=5)
    _save_visual_figure(fig, path)


def _render_table(
    *,
    columns: list[Any] | None,
    rows: list[list[Any]],
    path: Path,
    caption: str = "",
    row_labels: list[Any] | None = None,
    fontsize: float = 10.0,
    scale_y: float = 1.45,
    min_width: float = 5.0,
) -> None:
    from matplotlib import pyplot as plt

    text_rows = [["" if value is None else str(value) for value in row] for row in rows]
    column_count = len(columns or []) or (len(text_rows[0]) if text_rows else 1)
    width = max(min_width, min(13.0, 1.25 * max(1, column_count)))
    height = max(2.3, min(10.5, 1.5 + 0.48 * max(1, len(text_rows))))

    fig, ax = plt.subplots(figsize=(width, height))
    ax.axis("off")
    table = ax.table(
        cellText=text_rows,
        colLabels=[str(value) for value in columns] if columns else None,
        rowLabels=[str(value) for value in row_labels] if row_labels else None,
        cellLoc="center",
        loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(fontsize)
    table.scale(1.0, scale_y)
    for cell in table.get_celld().values():
        cell.set_edgecolor("black")
        cell.set_linewidth(1.0)
        cell.set_facecolor("white")
        cell.get_text().set_color("black")
    if caption:
        ax.set_title(caption, fontsize=10.5, pad=10)
    _save_visual_figure(fig, path)


def _render_trace_table(spec: dict[str, Any], path: Path) -> None:
    _render_table(
        columns=spec.get("columns", []),
        rows=spec.get("rows", []),
        path=path,
        caption=str(spec.get("caption", "") or "").strip(),
        fontsize=10,
        scale_y=1.45,
        min_width=6.0,
    )


def _render_array_grid(spec: dict[str, Any], path: Path) -> None:
    values = spec.get("values", [])
    rows = values if values and isinstance(values[0], list) else [values]
    _render_table(
        columns=spec.get("column_labels"),
        rows=rows,
        row_labels=spec.get("row_labels"),
        path=path,
        caption=str(spec.get("caption", "") or "").strip(),
        fontsize=11,
        scale_y=1.6,
        min_width=4.5,
    )


def _render_simple_flowchart(spec: dict[str, Any], path: Path) -> None:
    from matplotlib import pyplot as plt
    from matplotlib.patches import Ellipse, Polygon, Rectangle

    nodes = spec.get("nodes", [])
    edges = spec.get("edges", [])
    caption = str(spec.get("caption", "") or "").strip()
    node_ids = [str(node.get("id", "")) for node in nodes]
    y_step = 0.78 / max(1, len(nodes) - 1)
    positions = {node_id: [0.5, 0.9 - index * y_step] for index, node_id in enumerate(node_ids)}
    node_by_id = {str(node.get("id", "")): node for node in nodes}

    for edge in edges:
        source = str(edge.get("from", ""))
        target = str(edge.get("to", ""))
        label = str(edge.get("label", "") or "").strip().casefold()
        source_node = node_by_id.get(source, {})
        if str(source_node.get("type", "")).strip().casefold() == "decision":
            if label in {"yes", "true"} and target in positions:
                positions[target][0] = 0.3
            elif label in {"no", "false"} and target in positions:
                positions[target][0] = 0.7

    fig, ax = plt.subplots(figsize=(8.0, max(5.0, 1.05 * len(nodes))))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    for edge in edges:
        source = str(edge.get("from", ""))
        target = str(edge.get("to", ""))
        if source not in positions or target not in positions:
            continue
        x1, y1 = positions[source]
        x2, y2 = positions[target]
        ax.annotate("", xy=(x2, y2 + 0.045), xytext=(x1, y1 - 0.045), arrowprops={"arrowstyle": "->", "linewidth": 1.1, "color": "black"}, zorder=1)
        label = str(edge.get("label", "") or "").strip()
        if label:
            ax.text((x1 + x2) / 2 + 0.02, (y1 + y2) / 2, label, fontsize=9.5, ha="left", va="center", color="black")

    for node in nodes:
        node_id = str(node.get("id", ""))
        node_type = str(node.get("type", "process")).strip().casefold()
        text = str(node.get("text", ""))
        x, y = positions[node_id]
        width, height = 0.28, 0.09
        if node_type == "start_end":
            patch = Ellipse((x, y), width, height, edgecolor="black", facecolor="white", linewidth=1.2, zorder=2)
        elif node_type == "decision":
            patch = Polygon([(x, y + height * 0.75), (x + width * 0.58, y), (x, y - height * 0.75), (x - width * 0.58, y)], closed=True, edgecolor="black", facecolor="white", linewidth=1.2, zorder=2)
        elif node_type == "input_output":
            skew = 0.04
            patch = Polygon([(x - width / 2 + skew, y + height / 2), (x + width / 2, y + height / 2), (x + width / 2 - skew, y - height / 2), (x - width / 2, y - height / 2)], closed=True, edgecolor="black", facecolor="white", linewidth=1.2, zorder=2)
        else:
            patch = Rectangle((x - width / 2, y - height / 2), width, height, edgecolor="black", facecolor="white", linewidth=1.2, zorder=2)
        ax.add_patch(patch)
        ax.text(x, y, text, ha="center", va="center", fontsize=9.5, wrap=True, color="black", zorder=3)

    if caption:
        ax.set_title(caption, fontsize=10.5, pad=8)
    _save_visual_figure(fig, path)


def _logic_gate_depths(inputs: list[str], gates: list[dict[str, Any]]) -> dict[str, int]:
    depths: dict[str, int] = {name: 0 for name in inputs}
    for gate in gates:
        gate_id = str(gate.get("id", ""))
        refs = [str(ref) for ref in gate.get("inputs", [])]
        depths[gate_id] = 1 + max((depths.get(ref, 0) for ref in refs), default=0)
    return depths


def _draw_logic_gate_symbol(ax: Any, x: float, y: float, gate_type: str, width: float = 0.16, height: float = 0.12) -> tuple[float, float]:
    from matplotlib.patches import Arc, Circle, PathPatch, Polygon
    from matplotlib.path import Path as MplPath

    gate_type = gate_type.upper()
    base_type = {"NAND": "AND", "NOR": "OR"}.get(gate_type, gate_type)
    bubble = gate_type in {"NAND", "NOR"}

    if base_type == "NOT":
        patch = Polygon([(x - width / 2, y - height / 2), (x - width / 2, y + height / 2), (x + width / 2 - 0.02, y)], closed=True, edgecolor="black", facecolor="white", linewidth=1.2)
        ax.add_patch(patch)
        ax.add_patch(Circle((x + width / 2, y), 0.012, edgecolor="black", facecolor="white", linewidth=1.1))
        return x - width / 2, x + width / 2 + 0.012

    if base_type == "AND":
        verts = [
            (x - width / 2, y - height / 2),
            (x, y - height / 2),
            (x + width / 2, y - height / 2),
            (x + width / 2, y + height / 2),
            (x, y + height / 2),
            (x - width / 2, y + height / 2),
            (x - width / 2, y - height / 2),
        ]
        codes = [MplPath.MOVETO, MplPath.LINETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4, MplPath.LINETO, MplPath.CLOSEPOLY]
        ax.add_patch(PathPatch(MplPath(verts, codes), edgecolor="black", facecolor="white", linewidth=1.2))
    else:
        # Approximate OR/XOR outline using Bezier curves.
        verts = [
            (x - width / 2, y - height / 2),
            (x - width * 0.18, y - height * 0.58),
            (x + width * 0.25, y - height * 0.48),
            (x + width / 2, y),
            (x + width * 0.25, y + height * 0.48),
            (x - width * 0.18, y + height * 0.58),
            (x - width / 2, y + height / 2),
            (x - width * 0.30, y),
            (x - width / 2, y - height / 2),
        ]
        codes = [MplPath.MOVETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4, MplPath.CLOSEPOLY]
        ax.add_patch(PathPatch(MplPath(verts, codes), edgecolor="black", facecolor="white", linewidth=1.2))
        if base_type == "XOR":
            ax.add_patch(Arc((x - width * 0.47, y), width * 0.35, height * 1.05, theta1=-70, theta2=70, edgecolor="black", linewidth=1.0))

    output_x = x + width / 2
    if bubble:
        ax.add_patch(Circle((output_x + 0.012, y), 0.012, edgecolor="black", facecolor="white", linewidth=1.1))
        output_x += 0.024
    return x - width / 2, output_x


def _render_logic_gate_diagram(spec: dict[str, Any], path: Path) -> None:
    from matplotlib import pyplot as plt

    inputs = [str(value) for value in spec.get("inputs", [])]
    gates = spec.get("gates", [])
    output = spec.get("output", {})
    caption = str(spec.get("caption", "") or "").strip()
    depths = _logic_gate_depths(inputs, gates)
    max_depth = max(depths.values(), default=1)

    # Position inputs on the left and gates by topological depth.
    positions: dict[str, tuple[float, float]] = {}
    for index, name in enumerate(inputs):
        y = 0.82 - index * (0.64 / max(1, len(inputs) - 1)) if len(inputs) > 1 else 0.5
        positions[name] = (0.08, y)

    by_depth: dict[int, list[dict[str, Any]]] = {}
    for gate in gates:
        by_depth.setdefault(depths.get(str(gate.get("id", "")), 1), []).append(gate)
    for depth, items in by_depth.items():
        x = 0.2 + 0.62 * depth / max(1, max_depth)
        for index, gate in enumerate(items):
            refs = [str(ref) for ref in gate.get("inputs", [])]
            ref_y = [positions.get(ref, (0, 0.5))[1] for ref in refs]
            desired = sum(ref_y) / len(ref_y) if ref_y else 0.5
            offset = (index - (len(items) - 1) / 2) * 0.15
            positions[str(gate.get("id", ""))] = (x, min(0.86, max(0.14, desired + offset)))

    fig, ax = plt.subplots(figsize=(9.0, max(3.5, 1.0 + 0.7 * len(inputs))))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    # Draw wires first.
    for gate in gates:
        gate_id = str(gate.get("id", ""))
        gx, gy = positions[gate_id]
        refs = [str(ref) for ref in gate.get("inputs", [])]
        offsets = [0.025] if len(refs) == 1 else [0.028, -0.028]
        for ref, yoff in zip(refs, offsets):
            sx, sy = positions[ref]
            ax.plot([sx + (0.02 if ref in inputs else 0.08), gx - 0.08], [sy, gy + yoff], color="black", linewidth=1.1)

    for name in inputs:
        x, y = positions[name]
        ax.text(x, y, name, ha="center", va="center", fontsize=10.5, color="black")
        ax.plot([x + 0.02, x + 0.08], [y, y], color="black", linewidth=1.1)

    for gate in gates:
        gate_id = str(gate.get("id", ""))
        gate_type = str(gate.get("type", "AND")).upper()
        x, y = positions[gate_id]
        _draw_logic_gate_symbol(ax, x, y, gate_type)
        if not bool(
            spec.get(
                "_suppress_gate_type_labels",
                False,
            )
        ):
            ax.text(
                x,
                y - 0.09,
                gate_type,
                ha="center",
                va="top",
                fontsize=8.5,
                color="black",
            )

    output_from = str(output.get("from", ""))
    if output_from in positions:
        x, y = positions[output_from]
        ax.plot([x + 0.09, 0.93], [y, y], color="black", linewidth=1.1)
        ax.text(0.95, y, str(output.get("label", "Q") or "Q"), ha="left", va="center", fontsize=10.5, color="black")

    if caption:
        ax.set_title(caption, fontsize=10.5, pad=8)
    _save_visual_figure(fig, path)


def _render_truth_table(spec: dict[str, Any], path: Path) -> None:
    _render_table(
        columns=spec.get("columns", []),
        rows=spec.get("rows", []),
        path=path,
        caption=str(spec.get("caption", "") or "").strip(),
        fontsize=10.5,
        scale_y=1.5,
        min_width=5.5,
    )


def _render_network_diagram(spec: dict[str, Any], path: Path) -> None:
    from matplotlib import pyplot as plt
    from matplotlib.patches import Circle, FancyBboxPatch

    nodes = spec.get("nodes", [])
    edges = spec.get("edges", [])
    caption = str(spec.get("caption", "") or "").strip()
    n = len(nodes)

    positions: dict[str, tuple[float, float]] = {}
    # Central infrastructure nodes get the centre; remaining nodes are radial.
    central = [node for node in nodes if str(node.get("type", "")).casefold() in {"switch", "router", "access_point", "server"}]
    centre_node = central[0] if central else None
    radial_nodes = [node for node in nodes if node is not centre_node]
    if centre_node:
        positions[str(centre_node.get("id", ""))] = (0.5, 0.5)
    for index, node in enumerate(radial_nodes):
        angle = 2 * math.pi * index / max(1, len(radial_nodes)) + math.pi / 2
        positions[str(node.get("id", ""))] = (0.5 + 0.34 * math.cos(angle), 0.5 + 0.34 * math.sin(angle))

    fig, ax = plt.subplots(figsize=(8.5, 6.3))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    for edge in edges:
        source = str(edge.get("from", ""))
        target = str(edge.get("to", ""))
        if source not in positions or target not in positions:
            continue
        x1, y1 = positions[source]
        x2, y2 = positions[target]
        ax.plot([x1, x2], [y1, y2], color="black", linewidth=1.1, zorder=1)
        label = str(edge.get("label", "") or "").strip()
        if label:
            ax.text((x1 + x2) / 2, (y1 + y2) / 2 + 0.02, label, fontsize=8.5, ha="center", va="bottom", color="black")

    circle_types = {"router", "switch", "access_point", "cloud"}
    for node in nodes:
        node_id = str(node.get("id", ""))
        node_type = str(node.get("type", "device")).casefold()
        label = str(node.get("label", ""))
        x, y = positions[node_id]
        if node_type in circle_types:
            patch = Circle((x, y), 0.065, edgecolor="black", facecolor="white", linewidth=1.2, zorder=2)
        else:
            patch = FancyBboxPatch((x - 0.075, y - 0.045), 0.15, 0.09, boxstyle="round,pad=0.01,rounding_size=0.01", edgecolor="black", facecolor="white", linewidth=1.2, zorder=2)
        ax.add_patch(patch)
        ax.text(x, y, label, ha="center", va="center", fontsize=9.2, color="black", zorder=3, wrap=True)
        ax.text(x, y - 0.085, node_type.replace("_", " "), ha="center", va="top", fontsize=7.5, color="black")

    if caption:
        ax.set_title(caption, fontsize=10.5, pad=8)
    _save_visual_figure(fig, path)


def _render_database_table(spec: dict[str, Any], path: Path) -> None:
    columns = [str(value) for value in spec.get("columns", [])]
    primary_key = str(spec.get("primary_key", "") or "").strip()
    display_columns = [f"{column} (PK)" if primary_key and column == primary_key else column for column in columns]
    _render_table(
        columns=display_columns,
        rows=spec.get("rows", []),
        path=path,
        caption=str(spec.get("caption", "") or "").strip(),
        fontsize=9.5,
        scale_y=1.45,
        min_width=6.0,
    )


def _render_cpu_block_diagram(spec: dict[str, Any], path: Path) -> None:
    from matplotlib import pyplot as plt
    from matplotlib.patches import FancyBboxPatch

    components = spec.get("components", [])
    connections = spec.get("connections", [])
    caption = str(spec.get("caption", "") or "").strip()

    # Deterministic grid layout; common CPU components are placed predictably.
    preferred = {
        "control_unit": (0.30, 0.68),
        "alu": (0.62, 0.68),
        "register": (0.46, 0.42),
        "cache": (0.72, 0.42),
        "memory": (0.46, 0.15),
        "input": (0.10, 0.42),
        "output": (0.90, 0.42),
        "bus": (0.46, 0.28),
    }
    fallback_positions = [(0.18, 0.82), (0.50, 0.82), (0.82, 0.82), (0.18, 0.18), (0.82, 0.18)]
    used_type_count: dict[str, int] = {}
    positions: dict[str, tuple[float, float]] = {}
    fallback_index = 0
    for component in components:
        component_id = str(component.get("id", ""))
        component_type = str(component.get("type", "component")).casefold()
        used_type_count[component_type] = used_type_count.get(component_type, 0) + 1
        if component_type in preferred and used_type_count[component_type] == 1:
            positions[component_id] = preferred[component_type]
        else:
            positions[component_id] = fallback_positions[fallback_index % len(fallback_positions)]
            fallback_index += 1

    fig, ax = plt.subplots(figsize=(9.0, 6.2))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    for connection in connections:
        source = str(connection.get("from", ""))
        target = str(connection.get("to", ""))
        if source not in positions or target not in positions:
            continue
        x1, y1 = positions[source]
        x2, y2 = positions[target]
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops={"arrowstyle": "->", "linewidth": 1.0, "color": "black"}, zorder=1)
        label = str(connection.get("label", "") or "").strip()
        if label:
            ax.text((x1 + x2) / 2, (y1 + y2) / 2 + 0.02, label, fontsize=8.0, ha="center", color="black")

    for component in components:
        component_id = str(component.get("id", ""))
        label = str(component.get("label", ""))
        component_type = str(component.get("type", "component")).casefold()
        x, y = positions[component_id]
        patch = FancyBboxPatch((x - 0.09, y - 0.055), 0.18, 0.11, boxstyle="round,pad=0.012,rounding_size=0.01", edgecolor="black", facecolor="white", linewidth=1.2, zorder=2)
        ax.add_patch(patch)
        ax.text(x, y, label, ha="center", va="center", fontsize=9.2, color="black", zorder=3, wrap=True)
        if component_type not in label.casefold().replace(" ", "_"):
            ax.text(x, y - 0.075, component_type.replace("_", " "), ha="center", va="top", fontsize=7.2, color="black")

    if caption:
        ax.set_title(caption, fontsize=10.5, pad=8)
    _save_visual_figure(fig, path)


def _render_memory_grid(spec: dict[str, Any], path: Path) -> None:
    rows = [[address, value] for address, value in zip(spec.get("addresses", []), spec.get("values", []))]
    _render_table(
        columns=["Address", "Value"],
        rows=rows,
        path=path,
        caption=str(spec.get("caption", "") or "").strip(),
        fontsize=10.5,
        scale_y=1.45,
        min_width=5.0,
    )



def _normalized_binary_register_values(
    spec: dict[str, Any],
) -> tuple[list[Any], list[Any], list[Any] | None]:
    """
    Normalize compact model output such as "10110010" into one cell per bit.

    If an 8-bit register has missing/concatenated place values, derive the
    standard [128, 64, ..., 1] values deterministically for rendering only.
    """
    raw_bits = spec.get(
        "bits",
        [],
    )

    bits = safe_list(
        raw_bits
    )

    if len(bits) == 1 and isinstance(bits[0], str):
        compact_bits = re.sub(
            r"[\s,|]+",
            "",
            bits[0],
        )

        if re.fullmatch(
            r"[01]{2,32}",
            compact_bits,
        ):
            bits = [
                int(char)
                for char in compact_bits
            ]

    raw_place_values = spec.get(
        "place_values",
        [],
    )

    place_values = safe_list(
        raw_place_values
    )

    if len(place_values) == 1 and isinstance(place_values[0], str):
        split_values = [
            value
            for value in re.split(
                r"[\s,|]+",
                place_values[0].strip(),
            )
            if value
        ]

        if len(split_values) == len(bits):
            place_values = split_values

    if (
        len(bits) == 8
        and len(place_values) != 8
    ):
        place_values = [
            128,
            64,
            32,
            16,
            8,
            4,
            2,
            1,
        ]

    raw_labels = spec.get(
        "labels"
    )

    labels = (
        safe_list(raw_labels)
        if raw_labels is not None
        else None
    )

    if (
        labels is not None
        and len(labels) == 1
        and isinstance(labels[0], str)
    ):
        split_labels = [
            value
            for value in re.split(
                r"[\s,|]+",
                labels[0].strip(),
            )
            if value
        ]

        if len(split_labels) == len(bits):
            labels = split_labels

    return (
        bits,
        place_values,
        labels,
    )


def _render_binary_register(spec: dict[str, Any], path: Path) -> None:
    (
        bits,
        place_values,
        labels,
    ) = _normalized_binary_register_values(
        spec
    )

    columns = [
        str(value)
        for value in (
            labels
            or place_values
            or list(
                range(
                    len(bits) - 1,
                    -1,
                    -1,
                )
            )
        )
    ]

    rows = [
        [
            ""
            if bit is None
            else bit
            for bit in bits
        ]
    ]

    _render_table(
        columns=columns,
        rows=rows,
        path=path,
        caption=str(spec.get("caption", "") or "").strip(),
        fontsize=11.0,
        scale_y=1.7,
        min_width=6.0,
    )


# Backward-compatible renderer names from Phase 1.
_render_phase1_code_block = _render_code_block
_render_phase1_trace_table = _render_trace_table
_render_phase1_array_grid = _render_array_grid
_render_phase1_simple_flowchart = _render_simple_flowchart



_VISUAL_EXPLICIT_REFERENCE_RE = re.compile(
    r"\b(?:"
    r"shown|displayed|provided|pictured|illustrated|"
    r"refer(?:ring)?\s+to|use\s+the|using\s+the|"
    r"in\s+the\s+(?:grid|table|diagram|figure|visual|register)|"
    r"(?:grid|table|diagram|figure|visual|flowchart|trace\s+table|"
    r"truth\s+table|code\s+block|memory\s+grid|register)\s+(?:below|above|shown|provided)|"
    r"see\s+the\s+(?:grid|table|diagram|figure|visual|code\s+block)"
    r")\b",
    flags=re.IGNORECASE,
)


def _visual_question_text(question: dict[str, Any]) -> str:
    for key in [
        "question_text",
        "question_text_canonical",
        "question_text_postgres",
        "question_text_retrieval",
        "text",
    ]:
        value = str(
            question.get(
                key,
                "",
            )
            or ""
        ).strip()
        if value:
            return value
    return ""


def _has_inline_array_initialiser(text: str) -> bool:
    """
    Detect a self-contained inline array initialiser such as:
        scores <- [12, 15, 9]
        grid = [[1, 2], [3, 4]]

    If the values are already fully present in question text, an array_grid is
    not automatically needed merely because the topic is arrays.
    """
    return bool(
        re.search(
            r"\b[a-zA-Z_]\w*\s*(?:<-|=)\s*\[\s*(?:\[)?[^\n]{1,240}\]",
            str(text or ""),
            flags=re.IGNORECASE,
        )
    )


def visual_relevance_decision(
    question: dict[str, Any],
    planned_type: str,
) -> dict[str, Any]:
    """
    Cheap deterministic relevance gate.

    The pre-generation planner may propose a visual from topic/task metadata,
    but the final generated question must actually use that visual. This gate
    removes decorative/topic-generic visuals without any extra LLM call.

    It is intentionally conservative:
    - explicit references to a visual keep it;
    - visual-specific learner actions keep it;
    - otherwise the visual is removed before rendering.
    """
    visual_type = normalize_visual_requirement(
        planned_type
    )

    if visual_type == "none":
        return {
            "status": "NOT_REQUIRED",
            "necessity": "NONE",
            "planned_visual": "none",
            "effective_visual": "none",
            "relevant": True,
            "reason": "Blueprint did not plan a visual.",
        }

    text = _visual_question_text(
        question
    )

    normalized = normalize_text(
        text
    )

    explicit_visual_reference = bool(
        _VISUAL_EXPLICIT_REFERENCE_RE.search(
            text
        )
    )

    # For array grids, an explicit phrase such as "shown in the visual" is not
    # sufficient by itself. A question may mention the grid but still be fully
    # answerable from an already-specified index expression. Array-grid
    # necessity is therefore evaluated below from the actual learner demand.
    if (
        explicit_visual_reference
        and visual_type != "array_grid"
    ):
        return {
            "status": "KEPT_RELEVANT",
            "necessity": "USEFUL",
            "planned_visual": visual_type,
            "effective_visual": visual_type,
            "relevant": True,
            "reason": "Question explicitly refers to and operates on a supplied visual.",
        }

    # Type-specific dependencies that can be detected without another LLM.
    # These are generic task-shape checks, not topic/example exceptions.
    if visual_type == "array_grid":
        indexed_access = bool(
            re.search(
                r"\b[a-zA-Z_]\w*\s*\[\s*[^\]\n]+\s*\]",
                text,
            )
        )

        asks_numeric_or_data_result = bool(
            re.search(
                r"\b(?:what\s+is|what\s+are|which|state|give|determine|calculate|"
                r"output|sum|total|largest|smallest|maximum|minimum|trace|"
                r"compute|find)\b.{0,120}\b(?:value|values|output|sum|total|"
                r"result|element|row|column)\b",
                normalized,
            )
            or re.search(
                r"\b(?:sum|total|largest|smallest|maximum|minimum)\b",
                normalized,
            )
        )

        asks_only_to_write_index_expression = bool(
            re.search(
                r"\b(?:write|give|provide)\b.{0,100}\b(?:statement|"
                r"pseudocode|pseudo\s*code|assignment|line)\b",
                normalized,
            )
            and re.search(
                r"\b(?:row|column|index)\b",
                normalized,
            )
            and not asks_numeric_or_data_result
        )

        if asks_only_to_write_index_expression:
            return {
                "status": "REMOVED_IRRELEVANT",
                "necessity": "UNNECESSARY",
                "planned_visual": visual_type,
                "effective_visual": "none",
                "relevant": False,
                "reason": (
                    "The learner only needs to construct an index/assignment "
                    "expression from indices already stated in the text; the "
                    "grid values are not needed."
                ),
            }

        # Explicit grid reference + requested data result means the visual is
        # required when the array values are not otherwise initialised in text.
        # This covers natural wording such as "row 0, column 1" as well as A[0][1].
        if (
            explicit_visual_reference
            and asks_numeric_or_data_result
            and not _has_inline_array_initialiser(text)
        ):
            return {
                "status": "KEPT_RELEVANT",
                "necessity": "REQUIRED",
                "planned_visual": visual_type,
                "effective_visual": visual_type,
                "relevant": True,
                "reason": (
                    "Question explicitly requires values/relationships from the "
                    "supplied array grid, and those values are not initialised "
                    "in the question text."
                ),
            }

        if (
            indexed_access
            and asks_numeric_or_data_result
            and not _has_inline_array_initialiser(
                text
            )
        ):
            return {
                "status": "KEPT_RELEVANT",
                "necessity": "REQUIRED",
                "planned_visual": visual_type,
                "effective_visual": visual_type,
                "relevant": True,
                "reason": (
                    "Question requires array values/data that are not "
                    "initialised in the text, so the grid supplies required "
                    "state."
                ),
            }

        if (
            explicit_visual_reference
            and indexed_access
            and not _has_inline_array_initialiser(text)
        ):
            return {
                "status": "KEPT_RELEVANT",
                "necessity": "USEFUL",
                "planned_visual": visual_type,
                "effective_visual": visual_type,
                "relevant": True,
                "reason": (
                    "Question explicitly works with indexed array data from "
                    "the supplied grid."
                ),
            }

        # Conservative fail-safe: once the index-expression-only case above has
        # been ruled out, do not automatically delete a grid explicitly referenced
        # by the learner task. That could make the question unanswerable.
        if explicit_visual_reference and not _has_inline_array_initialiser(text):
            return {
                "status": "KEPT_RELEVANT",
                "necessity": "USEFUL",
                "planned_visual": visual_type,
                "effective_visual": visual_type,
                "relevant": True,
                "reason": (
                    "Question explicitly references the supplied array grid; "
                    "automatic removal would risk deleting required context."
                ),
            }

    elif visual_type == "trace_table":
        if re.search(
            r"\b(?:trace|step|iteration|complete|row|column)\b",
            normalized,
        ) and re.search(
            r"\b(?:value|state|output|variable)\b",
            normalized,
        ):
            return {
                "status": "KEPT_RELEVANT",
                "planned_visual": visual_type,
                "effective_visual": visual_type,
                "relevant": True,
                "reason": "Question requires trace/state information.",
            }

    elif visual_type == "code_block":
        if re.search(
            r"\b(?:line|code|program|pseudocode|missing|error|correct|"
            r"debug|statement|fragment)\b",
            normalized,
        ):
            return {
                "status": "KEPT_RELEVANT",
                "planned_visual": visual_type,
                "effective_visual": visual_type,
                "relevant": True,
                "reason": "Question directly operates on code/pseudocode.",
            }

    elif visual_type in {
        "truth_table",
        "database_table",
        "memory_grid",
        "binary_register",
    }:
        if re.search(
            r"\b(?:complete|fill|read|state|identify|select|calculate|"
            r"determine|value|values|row|column|bit|address|record|field)\b",
            normalized,
        ) and re.search(
            r"\b(?:table|truth|memory|register|binary|database|record|"
            r"field|address|bit)\b",
            normalized,
        ):
            return {
                "status": "KEPT_RELEVANT",
                "planned_visual": visual_type,
                "effective_visual": visual_type,
                "relevant": True,
                "reason": "Question performs a learner action on visual data.",
            }

    elif visual_type in {
        "simple_flowchart",
        "logic_gate_diagram",
        "network_diagram",
        "cpu_block_diagram",
    }:
        if re.search(
            r"\b(?:label|complete|identify|select|trace|follow|connect|"
            r"component|node|gate|path|arrow)\b",
            normalized,
        ):
            return {
                "status": "KEPT_RELEVANT",
                "planned_visual": visual_type,
                "effective_visual": visual_type,
                "relevant": True,
                "reason": "Question performs a learner action on the diagram.",
            }

    return {
        "status": "REMOVED_IRRELEVANT",
        "necessity": "UNNECESSARY",
        "planned_visual": visual_type,
        "effective_visual": "none",
        "relevant": False,
        "reason": (
            "The generated question is self-contained and does not explicitly "
            "depend on or materially use the planned visual."
        ),
    }


def apply_visual_relevance_gate(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:
    """
    Apply the final visual relevance gate before structural validation/render.

    A visual rejected here is downgraded to visual_requirement='none' and is
    never rendered. The original planned type remains recorded in
    visual_relevance_gate for diagnostics/HITL.
    """
    questions = payload.get(
        "questions",
        [],
    )

    blueprint = request_payload.get(
        "blueprint",
        [],
    )

    blueprint_by_index = {
        safe_int(
            item.get(
                "plan_index"
            )
        ):
            item
        for item in blueprint
        if isinstance(
            item,
            dict,
        )
    }

    kept = 0
    removed = 0
    not_required = 0
    decisions: list[dict[str, Any]] = []

    if not isinstance(
        questions,
        list,
    ):
        return {
            "status": "NOT_RUN",
            "kept_count": 0,
            "removed_count": 0,
            "not_required_count": 0,
            "decisions": [],
        }

    for position, question in enumerate(
        questions,
        start=1,
    ):
        if not isinstance(
            question,
            dict,
        ):
            continue

        plan_index = safe_int(
            question.get(
                "plan_index"
            )
        )

        expected = blueprint_by_index.get(
            plan_index,
            {},
        )

        planned_type = normalize_visual_requirement(
            expected.get(
                "visual_requirement",
                question.get(
                    "visual_requirement",
                    "none",
                ),
            )
        )

        decision = visual_relevance_decision(
            question,
            planned_type,
        )

        decision.update(
            {
                "question_position": position,
                "plan_index": plan_index,
            }
        )

        question[
            "visual_relevance_gate"
        ] = decision

        status = str(
            decision.get(
                "status",
                "",
            )
            or ""
        ).upper()

        if status == "REMOVED_IRRELEVANT":
            question[
                "visual_requirement"
            ] = "none"

            question[
                "requires_visual"
            ] = False

            question[
                "visual"
            ] = {
                "type": "none",
                "spec": {},
            }

            question[
                "visual_path"
            ] = None

            removed += 1

        elif planned_type == "none":
            not_required += 1

        else:
            kept += 1

        decisions.append(
            decision
        )

    result = {
        "status": "PASS",
        "kept_count": kept,
        "removed_count": removed,
        "not_required_count": not_required,
        "decisions": decisions,
    }

    payload[
        "visual_relevance_validation"
    ] = result

    return result



def render_visual(question: dict[str, Any]) -> Path | None:
    required_type = normalize_visual_requirement(
        question.get(
            "visual_requirement",
            "none",
        )
    )

    if required_type == "none":
        return None

    visual = question.get(
        "visual",
        {},
    )

    raw_spec = (
        visual.get(
            "spec",
            {},
        )
        if isinstance(
            visual,
            dict,
        )
        else {}
    )

    spec = (
        dict(raw_spec)
        if isinstance(raw_spec, dict)
        else {}
    )

    # Internal display policy only; never sent back to the LLM.
    if (
        required_type == "logic_gate_diagram"
        and "_question_asks_to_identify_gate" in globals()
        and _question_asks_to_identify_gate(
            question.get(
                "question_text",
                "",
            )
        )
    ):
        spec[
            "_suppress_gate_type_labels"
        ] = True

    path = _visual_asset_path(
        question
    )

    renderers = {
        "code_block": _render_code_block,
        "trace_table": _render_trace_table,
        "array_grid": _render_array_grid,
        "simple_flowchart": _render_simple_flowchart,
        "logic_gate_diagram": _render_logic_gate_diagram,
        "truth_table": _render_truth_table,
        "network_diagram": _render_network_diagram,
        "database_table": _render_database_table,
        "cpu_block_diagram": _render_cpu_block_diagram,
        "memory_grid": _render_memory_grid,
        "binary_register": _render_binary_register,
    }

    renderer = renderers.get(
        required_type
    )

    if renderer is None:
        raise ValueError(
            "Unsupported visual type: "
            + required_type
        )

    renderer(
        spec,
        path,
    )

    if (
        not path.is_file()
        or path.stat().st_size <= 0
    ):
        raise RuntimeError(
            "Visual renderer did not produce a non-empty PNG asset."
        )

    return path


def render_phase1_visual(question: dict[str, Any]) -> Path | None:
    return render_visual(question)


def _file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def materialize_visuals(payload: dict[str, Any]) -> dict[str, Any]:
    """Validate, render, hash and attach every relevance-approved visual."""
    errors: list[str] = []
    rendered_count = 0
    not_required_count = 0
    phase_counts = {"phase_1": 0, "phase_2": 0, "final": 0}
    type_counts = {visual_type: 0 for visual_type in VISUAL_TYPES if visual_type != "none"}

    questions = payload.get("questions", [])
    if not isinstance(questions, list):
        return {
            "valid": False,
            "errors": ["questions must be a list before visual rendering"],
            "rendered_count": 0,
            "not_required_count": 0,
            "visual_dir": str(VISUAL_DIR),
            "phase_counts": phase_counts,
            "type_counts": type_counts,
        }

    for position, question in enumerate(questions, start=1):
        if not isinstance(question, dict):
            continue

        required_type = normalize_visual_requirement(question.get("visual_requirement", "none"))
        phase = visual_phase_for_type(required_type)
        question["visual_schema_version"] = VISUAL_SCHEMA_VERSION
        question["visual_renderer"] = VISUAL_RENDERER
        question["visual_architecture_phase"] = phase
        question["visual_asset_sha256"] = None

        if required_type == "none":
            question["visual_path"] = None
            question["visual_render_status"] = "not_required"
            question["visual_render_error"] = None
            question["visual_release_eligible"] = True
            not_required_count += 1
            continue

        spec_errors = validate_visual_spec(question.get("visual"), required_type)
        if spec_errors:
            message = "; ".join(spec_errors)
            question["visual_path"] = None
            question["visual_render_status"] = "invalid_spec"
            question["visual_render_error"] = message
            question["visual_release_eligible"] = False
            errors.append(f"Question {position}: {message}")
            continue

        try:
            path = render_visual(question)
            assert path is not None
            asset_hash = _file_sha256(path)
        except Exception as exc:
            message = str(exc).strip() or exc.__class__.__name__
            question["visual_path"] = None
            question["visual_render_status"] = "render_failed"
            question["visual_render_error"] = message
            question["visual_release_eligible"] = False
            errors.append(f"Question {position}: visual render failed: {message}")
            continue

        question["visual_path"] = str(path)
        question["visual_render_status"] = "rendered"
        question["visual_render_error"] = None
        question["visual_asset_sha256"] = asset_hash
        question["visual_release_eligible"] = True
        rendered_count += 1
        if phase in phase_counts:
            phase_counts[phase] += 1
        if required_type in type_counts:
            type_counts[required_type] += 1

    relevance = payload.get(
        "visual_relevance_validation",
        {},
    )

    if not isinstance(
        relevance,
        dict,
    ):
        relevance = {}

    return {
        "valid": not errors,
        "errors": errors,
        "rendered_count": rendered_count,
        "not_required_count": not_required_count,
        "relevance_kept_count": safe_int(
            relevance.get(
                "kept_count",
                0,
            )
        ),
        "relevance_removed_count": safe_int(
            relevance.get(
                "removed_count",
                0,
            )
        ),
        "visual_relevance_gate_status": str(
            relevance.get(
                "status",
                "NOT_RUN",
            )
            or "NOT_RUN"
        ),
        "visual_dir": str(VISUAL_DIR),
        "renderer": VISUAL_RENDERER,
        "schema_version": VISUAL_SCHEMA_VERSION,
        "phase_counts": phase_counts,
        "type_counts": type_counts,
    }


def materialize_phase1_visuals(payload: dict[str, Any]) -> dict[str, Any]:
    return materialize_visuals(payload)


def validate_render_and_quality(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> tuple[dict[str, Any], dict[str, Any]]:
    """
    Final validation order:
        structural contract -> deterministic visual render -> semantic signals

    Any required visual spec/render failure blocks the candidate before HITL.
    """
    visual_relevance_validation = apply_visual_relevance_gate(
        payload,
        request_payload,
    )

    structural = validate_generated_payload(payload, request_payload)

    visual_validation = {
        "valid": False,
        "errors": [],
        "rendered_count": 0,
        "not_required_count": 0,
        "relevance_kept_count": safe_int(
            visual_relevance_validation.get(
                "kept_count",
                0,
            )
        ),
        "relevance_removed_count": safe_int(
            visual_relevance_validation.get(
                "removed_count",
                0,
            )
        ),
        "visual_relevance_gate_status": str(
            visual_relevance_validation.get(
                "status",
                "NOT_RUN",
            )
            or "NOT_RUN"
        ),
        "visual_dir": str(VISUAL_DIR),
        "renderer": VISUAL_RENDERER,
        "schema_version": VISUAL_SCHEMA_VERSION,
        "phase_counts": {"phase_1": 0, "phase_2": 0, "final": 0},
        "type_counts": {},
        "status": "NOT_RUN",
    }

    if structural.get("valid", False):
        visual_validation = materialize_visuals(payload)
        visual_validation["status"] = "PASS" if visual_validation.get("valid", False) else "FAIL"
        if not visual_validation.get("valid", False):
            structural["valid"] = False
            structural.setdefault("errors", []).extend(visual_validation.get("errors", []))
    else:
        visual_validation["status"] = "BLOCKED_BY_STRUCTURAL_VALIDATION"

    structural["visual_validation"] = visual_validation

    if structural.get("valid", False):
        semantic = validate_semantic_quality(payload, request_payload)
    else:
        semantic = {
            "status": "NOT_RUN",
            "release_eligible": False,
            "reasons": structural.get("errors", []),
            "warnings": [],
        }

    return structural, semantic

_validate_render_and_quality_v236 = validate_render_and_quality


def _materialized_visual_integrity_errors(
    payload: dict[str, Any],
) -> list[str]:
    errors: list[str] = []

    questions = payload.get(
        "questions",
        [],
    )

    if not isinstance(
        questions,
        list,
    ):
        return [
            "questions must be a list for final visual-integrity validation"
        ]

    for position, question in enumerate(
        questions,
        start=1,
    ):
        if not isinstance(
            question,
            dict,
        ):
            continue

        qid = str(
            question.get(
                "generated_question_id",
                f"Q{position}",
            )
            or f"Q{position}"
        )

        required_type = normalize_visual_requirement(
            question.get(
                "visual_requirement",
                "none",
            )
        )

        references_visual = (
            _question_references_external_visual(
                question.get(
                    "question_text",
                    "",
                )
            )
            if "_question_references_external_visual" in globals()
            else False
        )

        if (
            references_visual
            and required_type == "none"
        ):
            errors.append(
                f"{qid}: student text refers to a visual but no visual "
                "is planned"
            )
            continue

        if required_type == "none":
            continue

        render_status = str(
            question.get(
                "visual_render_status",
                "",
            )
            or ""
        ).strip().casefold()

        path_value = str(
            question.get(
                "visual_path",
                "",
            )
            or ""
        ).strip()

        path = (
            Path(path_value)
            if path_value
            else None
        )

        if render_status != "rendered":
            errors.append(
                f"{qid}: required {required_type} visual is not rendered "
                f"(status={render_status!r})"
            )

        if (
            path is None
            or not path.is_file()
            or path.stat().st_size <= 0
        ):
            errors.append(
                f"{qid}: required {required_type} visual asset is missing "
                "or empty"
            )

        if not bool(
            question.get(
                "visual_release_eligible",
                False,
            )
        ):
            errors.append(
                f"{qid}: required visual is not release eligible"
            )

    return list(
        dict.fromkeys(
            errors
        )
    )


def validate_render_and_quality(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> tuple[dict[str, Any], dict[str, Any]]:
    structural, semantic = _validate_render_and_quality_v236(
        payload,
        request_payload,
    )

    if structural.get(
        "valid",
        False,
    ):
        integrity_errors = _materialized_visual_integrity_errors(
            payload
        )
    else:
        integrity_errors = []

    structural[
        "materialized_visual_integrity"
    ] = {
        "status": (
            "PASS"
            if not integrity_errors
            else "FAIL"
        ),
        "errors": integrity_errors,
    }

    if integrity_errors:
        structural["valid"] = False
        structural.setdefault(
            "errors",
            [],
        ).extend(
            integrity_errors
        )
        structural["errors"] = list(
            dict.fromkeys(
                structural["errors"]
            )
        )

        semantic = {
            "status": "NOT_RUN",
            "release_eligible": False,
            "reasons": structural["errors"],
            "warnings": [],
        }

    return structural, semantic


## 11. Generate / restore candidate and apply the single corrective retry

In [ ]:
generated_path = (
    OUTPUT_DIR
    / "generated_questions.json"
)

semantic_review_path = (
    OUTPUT_DIR
    / "semantic_quality_review.json"
)

generation_state_path = (
    OUTPUT_DIR
    / "generation_run_state.json"
)

human_review_path = (
    OUTPUT_DIR
    / "generated_quality_human_review.json"
)


def payload_fingerprint(
    payload: dict[str, Any],
) -> str | None:
    questions = payload.get(
        "questions",
        [],
    )

    if not isinstance(
        questions,
        list,
    ) or not questions:
        return None

    return stable_json_fingerprint(
        questions
    )


def save_candidate_artifacts(
    payload: dict[str, Any],
    structural: dict[str, Any],
    semantic: dict[str, Any],
    attempts_used: int,
    attempt_log: list[
        dict[str, Any]
    ],
) -> None:

    wrapped_payload = {
        **payload,

        "_provenance": {
            "pipeline_version":
                NOTEBOOK06_PIPELINE_VERSION,

            "generation_request_fingerprint":
                generation_request_fingerprint,

            "quiz_mode":
                QUIZ_MODE,

            "payload_fingerprint":
                payload_fingerprint(
                    payload
                ),

            "saved_at_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),
        },
    }

    generated_path.write_text(
        json.dumps(
            wrapped_payload,
            indent=2,
            ensure_ascii=False,
            default=str,
        ),
        encoding="utf-8",
    )

    semantic_review_path.write_text(
        json.dumps(
            semantic,
            indent=2,
            ensure_ascii=False,
            default=str,
        ),
        encoding="utf-8",
    )

    generation_state_path.write_text(
        json.dumps(
            {
                "pipeline_version":
                    NOTEBOOK06_PIPELINE_VERSION,

                "generation_request_fingerprint":
                    generation_request_fingerprint,

                "quiz_mode":
                    QUIZ_MODE,

                "payload_fingerprint":
                    payload_fingerprint(
                        payload
                    ),

                "regeneration_attempts_used":
                    attempts_used,

                "max_regeneration_attempts":
                    MAX_REGENERATION_ATTEMPTS,

                "attempt_log":
                    attempt_log,

                "updated_at_utc":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),
            },
            indent=2,
            ensure_ascii=False,
            default=str,
        ),
        encoding="utf-8",
    )


def restore_matching_candidate() -> tuple[
    dict[str, Any],
    int,
    list[
        dict[str, Any]
    ],
]:
    if not generated_path.is_file():
        return (
            {},
            0,
            [],
        )

    stored = json.loads(
        generated_path.read_text(
            encoding="utf-8"
        )
    )

    if not isinstance(
        stored,
        dict,
    ):
        return (
            {},
            0,
            [],
        )

    provenance = stored.get(
        "_provenance",
        {},
    )

    if not isinstance(
        provenance,
        dict,
    ):
        return (
            {},
            0,
            [],
        )

    if provenance.get(
        "generation_request_fingerprint"
    ) != generation_request_fingerprint:
        return (
            {},
            0,
            [],
        )

    if provenance.get(
        "pipeline_version"
    ) != NOTEBOOK06_PIPELINE_VERSION:
        return (
            {},
            0,
            [],
        )

    payload = {
        key:
            value
        for key, value in stored.items()
        if key
        != "_provenance"
    }

    attempts_used = 0
    attempt_log: list[
        dict[str, Any]
    ] = []

    if generation_state_path.is_file():
        state = json.loads(
            generation_state_path.read_text(
                encoding="utf-8"
            )
        )

        if (
            isinstance(
                state,
                dict,
            )
            and state.get(
                "generation_request_fingerprint"
            )
            == generation_request_fingerprint
        ):
            attempts_used = safe_int(
                state.get(
                    "regeneration_attempts_used"
                )
            )

            raw_log = state.get(
                "attempt_log",
                [],
            )

            if isinstance(
                raw_log,
                list,
            ):
                attempt_log = raw_log

    return (
        payload,
        attempts_used,
        attempt_log,
    )



def semantic_failed_plan_indexes(
    payload: dict[str, Any],
    semantic_validation: dict[str, Any],
) -> list[int]:
    """
    Return only plan indexes with question-local deterministic hard failures.

    Quiz-wide/global failures intentionally return no plan index. They are
    escalated to HITL without an automatic full-quiz regeneration, because a
    Plan C full regeneration would replay every adaptive batch and multiply API
    calls/cost without identifying a question-local repair target.
    """
    questions = [
        item
        for item in payload.get(
            "questions",
            [],
        )
        if isinstance(
            item,
            dict,
        )
    ]

    plan_by_id = {
        str(
            item.get(
                "generated_question_id",
                "",
            )
            or ""
        ): safe_int(
            item.get(
                "plan_index"
            )
        )
        for item in questions
    }

    signals = semantic_validation.get(
        "deterministic_signals",
        {},
    )

    if not isinstance(
        signals,
        dict,
    ):
        return []

    failed: list[int] = []

    for question_id, signal in signals.items():
        if not isinstance(
            signal,
            dict,
        ):
            continue

        hard_failures = signal.get(
            "hard_failures",
            [],
        )

        if not isinstance(
            hard_failures,
            list,
        ) or not hard_failures:
            continue

        plan_index = safe_int(
            plan_by_id.get(
                str(question_id),
                0,
            )
        )

        if (
            plan_index > 0
            and plan_index not in failed
        ):
            failed.append(
                plan_index
            )

    return sorted(
        failed
    )


def structural_failed_plan_indexes(
    payload: dict[str, Any],
    structural_validation: dict[str, Any],
) -> list[int]:
    """
    Map question-local structural errors back to plan_index for cheap repair.

    Supports both validator styles:
      - "Question 10: ..."
      - "GEN_010: ..."

    Global failures still return no guessed plan index and use the existing
    full corrective-regeneration path.
    """
    questions = [
        item
        for item in payload.get(
            "questions",
            [],
        )
        if isinstance(
            item,
            dict,
        )
    ]

    position_to_plan = {
        position: safe_int(
            item.get(
                "plan_index",
                position,
            )
        )
        for position, item in enumerate(
            questions,
            start=1,
        )
    }

    id_to_plan: dict[str, int] = {}

    for position, item in enumerate(
        questions,
        start=1,
    ):
        generated_id = str(
            item.get(
                "generated_question_id",
                "",
            )
            or ""
        ).strip()

        plan_index = safe_int(
            item.get(
                "plan_index",
                position,
            )
        )

        if generated_id and plan_index > 0:
            id_to_plan[
                generated_id.casefold()
            ] = plan_index

    failed: list[int] = []

    for error in safe_list(
        structural_validation.get(
            "errors",
            [],
        )
    ):
        text_value = str(
            error or ""
        ).strip()

        plan_index = 0

        # Existing validator form: "Question 10: ..."
        match = re.match(
            r"Question\s+(\d+)\s*:",
            text_value,
            flags=re.IGNORECASE,
        )

        if match:
            plan_index = safe_int(
                position_to_plan.get(
                    safe_int(
                        match.group(1)
                    ),
                    0,
                )
            )

        # v2.37+ visual-quality form: "GEN_010: ..."
        if plan_index <= 0:
            id_match = re.match(
                r"(GEN_\d+)\s*:",
                text_value,
                flags=re.IGNORECASE,
            )

            if id_match:
                generated_id = str(
                    id_match.group(1)
                ).casefold()

                plan_index = safe_int(
                    id_to_plan.get(
                        generated_id,
                        0,
                    )
                )

                # Deterministic fallback for normal GEN_### IDs.
                if plan_index <= 0:
                    numeric_match = re.search(
                        r"(\d+)$",
                        generated_id,
                    )
                    if numeric_match:
                        plan_index = safe_int(
                            numeric_match.group(1)
                        )

        if (
            plan_index > 0
            and plan_index not in failed
        ):
            failed.append(
                plan_index
            )

    return sorted(
        failed
    )



def _feedback_for_plan_indexes(
    feedback: list[str],
    plan_indexes: list[int],
) -> list[str]:
    if not plan_indexes:
        return feedback

    ids = {
        f"GEN_{index:03d}"
        for index in plan_indexes
    }

    plan_patterns = [
        re.compile(
            rf"\bplan[_ ]?index\s*(?:=|:)?\s*{int(index)}\b",
            flags=re.IGNORECASE,
        )
        for index in plan_indexes
    ]

    selected: list[str] = []

    for item in feedback:
        text_value = str(
            item
            or ""
        ).strip()

        if not text_value:
            continue

        if (
            any(
                question_id in text_value
                for question_id in ids
            )
            or any(
                pattern.search(text_value)
                for pattern in plan_patterns
            )
        ):
            selected.append(
                text_value
            )

    selected.append(
        "Regenerate only the listed plan_index items. Preserve every blueprint "
        "field exactly. Treat this as MINIMAL REPAIR, not free replacement: use "
        "targeted_regeneration_context.original_questions as the baseline and preserve "
        "learner-task intent, scenario/concept, task family, and all wording that remains "
        "valid. CURRENT HUMAN FEEDBACK and CURRENT DETERMINISTIC VALIDATION FEEDBACK are "
        "authoritative over invalid baseline content. If feedback targets marking guidance, "
        "acceptance rules, mark allocation, or metadata/pattern only, keep question_text "
        "unchanged and repair that component. For visual repairs, preserve the blueprint "
        "visual_requirement exactly and use the supplied visual contract as the source of "
        "truth. If learner-facing wording names, requests, or depends on a visual object "
        "inconsistent with that contract, rewrite only the conflicting wording; do not "
        "change the blueprint to match the mistake. If no visual is planned, remove the "
        "external-visual dependency generically and make the task self-contained rather "
        "than inventing a visual. If a visual is planned, return a complete valid visual.spec "
        "for the required type and make the student task genuinely depend on it. Past "
        "feedback memory is advisory only and must never override current feedback, the "
        "approved AQA scope, or deterministic blueprint. Make each regenerated question "
        "fully self-contained and make every objective marking criterion include its exact "
        "expected answer/value/code."
    )

    return selected


def regenerate_only_failed_questions(
    *,
    request_payload: dict[str, Any],
    current_payload: dict[str, Any],
    failed_plan_indexes: list[int],
    validation_feedback: list[str],
    repair_trigger: str = "validator_targeted_regeneration",
) -> dict[str, Any]:
    """
    Regenerate only question-local failures and merge them back by plan_index.

    This avoids spending a second model call on already-valid questions.
    """
    requested = {
        safe_int(value)
        for value in failed_plan_indexes
        if safe_int(value) > 0
    }

    if not requested:
        return generate_with_selected_model(
            request_payload,
            validation_feedback=validation_feedback,
        )

    targeted_blueprint = [
        item
        for item in request_payload.get(
            "blueprint",
            [],
        )
        if (
            isinstance(
                item,
                dict,
            )
            and safe_int(
                item.get(
                    "plan_index"
                )
            ) in requested
        )
    ]

    if len(targeted_blueprint) != len(requested):
        raise RuntimeError(
            "Targeted regeneration could not resolve every failed plan_index "
            "against the deterministic blueprint."
        )

    original_question_context: list[dict[str, Any]] = []
    for current_question in current_payload.get("questions", []):
        if not isinstance(current_question, dict):
            continue
        current_plan_index = safe_int(current_question.get("plan_index"))
        if current_plan_index not in requested:
            continue
        original_question_context.append(
            {
                "plan_index": current_plan_index,
                "generated_question_id": current_question.get("generated_question_id"),
                "question_text": current_question.get("question_text", ""),
                "marking_guidance": safe_list(current_question.get("marking_guidance", [])),
                "answer_verification": current_question.get(
                    "answer_verification",
                    {
                        "mode": "manual",
                        "checks": [],
                        "manual_reason": "",
                    },
                ),
                "assessment_pattern": current_question.get("assessment_pattern", ""),
                "question_type": current_question.get("question_type", ""),
                "visual_requirement": current_question.get("visual_requirement", "none"),
                "visual": current_question.get("visual", {"type": "none", "spec": {}}),
                "requires_code": bool(current_question.get("requires_code")),
                "requires_visual": bool(current_question.get("requires_visual")),
                "programming_language": current_question.get("programming_language"),
            }
        )

    targeted_request = {
        **request_payload,
        "blueprint": targeted_blueprint,
        "targeted_regeneration_plan_indexes": sorted(requested),
        "targeted_regeneration_trigger": str(
            repair_trigger or "validator_targeted_regeneration"
        ),
        "targeted_regeneration_context": {
            "repair_mode": "minimal_change",
            "original_questions": original_question_context,
            "policy": (
                "Preserve the original learner task, scenario, task family, and every part "
                "of the wording that remains valid. Apply the smallest repair required by "
                "current human feedback or current deterministic validation feedback. "
                "Immutable blueprint fields are authoritative. If learner-facing wording "
                "contradicts an immutable blueprint field or supplied contract, preserve "
                "the blueprint and rewrite only the conflicting wording. Marking-guidance-"
                "only feedback should leave question_text unchanged."
            ),
        },
    }

    targeted_feedback = _feedback_for_plan_indexes(
        validation_feedback,
        sorted(
            requested
        ),
    )

    replacement_payload = generate_with_selected_model(
        targeted_request,
        validation_feedback=targeted_feedback,
    )

    replacement_questions = [
        item
        for item in replacement_payload.get(
            "questions",
            [],
        )
        if isinstance(
            item,
            dict,
        )
    ]

    replacement_by_plan = {
        safe_int(
            item.get(
                "plan_index"
            )
        ): item
        for item in replacement_questions
        if safe_int(
            item.get(
                "plan_index"
            )
        ) > 0
    }

    missing = sorted(
        requested
        - set(
            replacement_by_plan
        )
    )

    if missing:
        raise RuntimeError(
            "Targeted regeneration returned no replacement for plan_index(es): "
            + ", ".join(
                str(value)
                for value in missing
            )
        )

    current_questions = [
        item
        for item in current_payload.get(
            "questions",
            [],
        )
        if isinstance(
            item,
            dict,
        )
    ]

    merged_questions: list[
        dict[str, Any]
    ] = []

    for item in current_questions:
        plan_index = safe_int(
            item.get(
                "plan_index"
            )
        )

        replacement = replacement_by_plan.get(
            plan_index
        )

        merged_questions.append(
            replacement
            if replacement is not None
            else item
        )

    merged_payload = {
        **current_payload,
        "questions":
            merged_questions,
        "targeted_regeneration": {
            "plan_indexes":
                sorted(
                    requested
                ),
            "replacement_generation_metadata":
                replacement_payload.get(
                    "generation_metadata",
                    {},
                ),
        },
    }

    return merged_payload


# ================================================================
# v2.43 — MARKING-SCHEME-FIRST REPAIR
# ================================================================

def _question_level_quality_split(
    question: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:
    """
    Separate learner-facing failures from marking-only failures.

    No topic-specific logic is used.
    """
    learner_facing_errors: list[str] = []
    marking_only_errors: list[str] = []
    verification_result: dict[str, Any] = {}

    if "_generic_visual_integrity_errors" in globals():
        learner_facing_errors.extend(
            _generic_visual_integrity_errors(
                question
            )
        )

    if "_generic_referential_consistency_errors" in globals():
        for error in _generic_referential_consistency_errors(
            question
        ):
            if (
                "marking_guidance targets line number(s) not requested"
                in error
                or "marking_guidance references code line(s) outside"
                in error
            ):
                marking_only_errors.append(
                    error
                )
            else:
                learner_facing_errors.append(
                    error
                )

    if "_generic_answer_verification_signals" in globals():
        verification_result = _generic_answer_verification_signals(
            question
        )

        # Machine-answer failures are marking/verification defects unless a
        # separate structural check has already shown learner-facing evidence
        # itself to be invalid.
        marking_only_errors.extend(
            verification_result.get(
                "hard_failures",
                [],
            )
        )

    return {
        "learner_facing_errors": list(
            dict.fromkeys(
                learner_facing_errors
            )
        ),
        "marking_only_errors": list(
            dict.fromkeys(
                marking_only_errors
            )
        ),
        "answer_verification_result": verification_result,
    }


def marking_only_failed_plan_indexes(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> list[int]:
    failed: list[int] = []

    questions = payload.get(
        "questions",
        [],
    )

    if not isinstance(
        questions,
        list,
    ):
        return failed

    for position, question in enumerate(
        questions,
        start=1,
    ):
        if not isinstance(
            question,
            dict,
        ):
            continue

        split = _question_level_quality_split(
            question,
            request_payload,
        )

        if (
            split["marking_only_errors"]
            and not split["learner_facing_errors"]
        ):
            plan_index = safe_int(
                question.get(
                    "plan_index",
                    position,
                )
            )

            if plan_index > 0:
                failed.append(
                    plan_index
                )

    return sorted(
        set(
            failed
        )
    )


def build_marking_repair_request(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
    plan_indexes: list[int],
) -> dict[str, Any]:
    question_map = {
        safe_int(
            question.get(
                "plan_index",
                position,
            )
        ): question
        for position, question in enumerate(
            payload.get(
                "questions",
                [],
            ),
            start=1,
        )
        if isinstance(
            question,
            dict,
        )
    }

    rows: list[dict[str, Any]] = []

    for plan_index in plan_indexes:
        question = question_map.get(
            plan_index
        )

        if not isinstance(
            question,
            dict,
        ):
            continue

        split = _question_level_quality_split(
            question,
            request_payload,
        )

        rows.append(
            {
                "plan_index": plan_index,
                "generated_question_id": question.get(
                    "generated_question_id",
                    "",
                ),
                "marks": question.get(
                    "marks",
                    0,
                ),
                "question_text": question.get(
                    "question_text",
                    "",
                ),
                "visual_requirement": question.get(
                    "visual_requirement",
                    "none",
                ),
                "visual": question.get(
                    "visual",
                    {
                        "type": "none",
                        "spec": {},
                    },
                ),
                "current_marking_guidance": question.get(
                    "marking_guidance",
                    [],
                ),
                "current_answer_verification": question.get(
                    "answer_verification",
                    {},
                ),
                "validation_errors": split[
                    "marking_only_errors"
                ],
                "verification_diagnostics": (
                    split.get(
                        "answer_verification_result",
                        {},
                    )
                    or {}
                ).get(
                    "diagnostics",
                    [],
                ),
            }
        )

    return {
        "schema_version":
            "agent2-marking-repair-request-v1.0.0",
        "questions":
            rows,
    }


MARKING_REPAIR_SYSTEM_PROMPT = """
You repair ONLY the marking scheme and answer-verification metadata for
already-generated GCSE Computer Science assessment questions.

Hard rules:
1. question_text is immutable.
2. visual_requirement and visual.spec are immutable.
3. marks and plan_index are immutable.
4. Return exactly one repair row per supplied plan_index.
5. You may change ONLY marking_guidance and answer_verification.
6. Use question_text + visual/code evidence + supplied verification diagnostics
   as the source of truth.
7. Objective marking criteria must contain the exact expected answer/value/code/
   line reference.
8. If machine verification computed a value that conflicts with the current
   expected answer, correct the marking answer; do NOT rewrite the question.
9. If a conceptual criterion cannot be safely machine verified, use manual mode.
10. Do not introduce new learner demands or new syllabus content.

Return JSON only:
{
  "questions": [
    {
      "plan_index": 1,
      "marking_guidance": [
        {"marks": 1, "criterion": "exact criterion"}
      ],
      "answer_verification": {
        "mode": "machine | mixed | manual",
        "checks": [
          {
            "criterion_index": 1,
            "expression": "restricted pure Python expression",
            "expected": "JSON scalar/list/dict"
          }
        ],
        "manual_reason": ""
      }
    }
  ]
}
""".strip()


def _marking_repair_response_schema() -> dict[str, Any]:
    return {
        "type": "object",
        "properties": {
            "questions": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "plan_index": {
                            "type": "integer",
                        },
                        "marking_guidance": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "marks": {
                                        "type": "integer",
                                    },
                                    "criterion": {
                                        "type": "string",
                                    },
                                },
                                "required": [
                                    "marks",
                                    "criterion",
                                ],
                            },
                        },
                        "answer_verification": {
                            "type": "object",
                        },
                    },
                    "required": [
                        "plan_index",
                        "marking_guidance",
                        "answer_verification",
                    ],
                },
            },
        },
        "required": [
            "questions",
        ],
    }


def call_marking_repair_model(
    repair_request: dict[str, Any],
) -> tuple[dict[str, Any], dict[str, Any]]:
    """
    Small same-model repair call.

    This is invoked only when learner-facing content is valid and the detected
    problem is marking/answer-verification-only.
    """
    messages = [
        {
            "role": "system",
            "content": MARKING_REPAIR_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": json.dumps(
                repair_request,
                ensure_ascii=False,
                default=str,
            ),
        },
    ]

    question_count = max(
        1,
        len(
            repair_request.get(
                "questions",
                [],
            )
        ),
    )

    desired_output_tokens = min(
        3200,
        max(
            900,
            650 * question_count,
        ),
    )

    safe_output_budget = provider_safe_output_budget_for_messages(
        messages,
        desired_output_tokens=desired_output_tokens,
    )

    if safe_output_budget < 512:
        raise ModelProviderTokenBudgetExceededError(
            "Marking-repair request does not have enough safe provider output "
            "budget. Candidate remains blocked for HITL rather than regenerating "
            "the learner-facing question."
        )

    response_schema = _marking_repair_response_schema()

    _set_active_model_call_audit_context(
        {
            "call_stage": "marking_scheme_repair",
            "trigger": "marking_only_validator_failure",
            "plan_indexes": [
                safe_int(item.get("plan_index"))
                for item in safe_list(repair_request.get("questions", []))
                if isinstance(item, dict) and safe_int(item.get("plan_index")) > 0
            ],
            "is_initial_generation": False,
            "is_targeted_regeneration": True,
            "prompt_profile": "marking_scheme_repair",
            "repair_transport_profile": "marking_only_compact",
            "recovery_attempt": 0,
            "transport_retry": "",
        }
    )

    if GENERATION_PROVIDER == "openai":
        try:
            from openai import OpenAI
        except ImportError as exc:
            raise RuntimeError(
                "Install the OpenAI SDK first: pip install -U openai"
            ) from exc

        api_key = str(
            os.getenv(
                "OPENAI_API_KEY"
            )
            or ""
        ).strip()

        if not api_key:
            raise RuntimeError(
                "OPENAI_API_KEY is not set in Agent2/.env."
            )

        client = OpenAI(
            api_key=api_key
        )

        return _openai_json_call(
            client,
            model=GENERATION_MODEL,
            messages=messages,
            max_tokens=min(
                safe_output_budget,
                OPENAI_MAX_GENERATION_OUTPUT_TOKENS,
            ),
            response_schema=response_schema,
            force_json_object=True,
        )

    if GENERATION_PROVIDER == "google_gemini":
        try:
            from google import genai
        except ImportError as exc:
            raise RuntimeError(
                "Install the Google GenAI SDK first: pip install -U google-genai"
            ) from exc

        api_key = str(
            os.getenv(
                "GEMINI_API_KEY"
            )
            or os.getenv(
                "GOOGLE_API_KEY"
            )
            or ""
        ).strip()

        if not api_key:
            raise RuntimeError(
                "GEMINI_API_KEY is not set in Agent2/.env."
            )

        client = genai.Client(
            api_key=api_key
        )

        return _gemini_json_call(
            client,
            model=GENERATION_MODEL,
            messages=messages,
            max_tokens=min(
                safe_output_budget,
                GEMINI_MAX_GENERATION_OUTPUT_TOKENS,
            ),
            response_schema=response_schema,
            force_mime_only=False,
        )

    if GENERATION_PROVIDER == "groq":
        try:
            from groq import Groq
        except ImportError as exc:
            raise RuntimeError(
                "Install the Groq SDK first: pip install -U groq"
            ) from exc

        api_key = str(
            os.getenv(
                "GROQ_API_KEY"
            )
            or ""
        ).strip()

        if not api_key:
            raise RuntimeError(
                "GROQ_API_KEY is not set in Agent2/.env."
            )

        client = Groq(
            api_key=api_key
        )

        return _groq_json_call(
            client,
            model=GENERATION_MODEL,
            messages=messages,
            max_tokens=min(
                safe_output_budget,
                GROQ_MAX_GENERATION_OUTPUT_TOKENS,
            ),
            response_schema=response_schema,
            force_json_object=True,
        )

    raise RuntimeError(
        "Unsupported provider for marking repair: "
        f"{GENERATION_PROVIDER!r}"
    )


def apply_marking_repair_response(
    payload: dict[str, Any],
    repair_response: dict[str, Any],
) -> dict[str, Any]:
    rows = repair_response.get(
        "questions",
        [],
    )

    if not isinstance(
        rows,
        list,
    ):
        return payload

    repair_map = {
        safe_int(
            row.get(
                "plan_index",
                0,
            )
        ): row
        for row in rows
        if isinstance(
            row,
            dict,
        )
        and safe_int(
            row.get(
                "plan_index",
                0,
            )
        ) > 0
    }

    for position, question in enumerate(
        payload.get(
            "questions",
            [],
        ),
        start=1,
    ):
        if not isinstance(
            question,
            dict,
        ):
            continue

        plan_index = safe_int(
            question.get(
                "plan_index",
                position,
            )
        )

        repair = repair_map.get(
            plan_index
        )

        if not isinstance(
            repair,
            dict,
        ):
            continue

        repaired_guidance = repair.get(
            "marking_guidance"
        )

        repaired_verification = repair.get(
            "answer_verification"
        )

        if isinstance(
            repaired_guidance,
            list,
        ):
            question[
                "marking_guidance"
            ] = repaired_guidance

        if isinstance(
            repaired_verification,
            dict,
        ):
            question[
                "answer_verification"
            ] = repaired_verification

        question[
            "marking_scheme_repair"
        ] = {
            "status":
                "APPLIED",
            "plan_index":
                plan_index,
            "question_text_frozen":
                True,
            "visual_frozen":
                True,
        }

    return payload


def attempt_marking_scheme_repairs(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> tuple[
    dict[str, Any],
    list[int],
    dict[str, Any],
]:
    plan_indexes = marking_only_failed_plan_indexes(
        payload,
        request_payload,
    )

    if not plan_indexes:
        return (
            payload,
            [],
            {},
        )

    repair_request = build_marking_repair_request(
        payload,
        request_payload,
        plan_indexes,
    )

    repair_response, metadata = call_marking_repair_model(
        repair_request
    )

    payload = apply_marking_repair_response(
        payload,
        repair_response,
    )

    return (
        payload,
        plan_indexes,
        metadata,
    )


# ================================================================
# v2.47 — TOPIC-INDEPENDENT SPECIAL-INSTRUCTION COMPLIANCE REVIEW
# ================================================================
SPECIAL_INSTRUCTION_REVIEW_SYSTEM_PROMPT = (
    "You are an independent assessment compliance validator. Judge the actual "
    "generated learner questions, supplied visuals/code/data, and marking guidance "
    "against the user's raw special instruction. Do not trust or reuse the generation "
    "model's self-reported interpretation or compliance claims. Apply the instruction "
    "regardless of assessment topic. Mark non-compliant when any objective part is "
    "violated. Mark conflict only when the instruction cannot coexist with the supplied "
    "hard controls or approved scope. Return every affected plan_index and concise, "
    "actionable reasons. If a quiz-wide failure cannot be isolated, return all plan indexes."
)


def _special_instruction_review_schema() -> dict[str, Any]:
    return {
        "type": "object",
        "properties": {
            "status": {"type": "string", "enum": ["compliant", "non_compliant", "conflict"]},
            "failed_plan_indexes": {"type": "array", "items": {"type": "integer"}},
            "reasons": {"type": "array", "items": {"type": "string"}},
            "per_question": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "plan_index": {"type": "integer"},
                        "compliant": {"type": "boolean"},
                        "reason": {"type": "string"},
                    },
                    "required": ["plan_index", "compliant", "reason"],
                },
            },
        },
        "required": ["status", "failed_plan_indexes", "reasons", "per_question"],
    }


def _special_instruction_review_request(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:
    questions = []
    for position, question in enumerate(payload.get("questions", []), start=1):
        if not isinstance(question, dict):
            continue
        questions.append({
            "plan_index": safe_int(question.get("plan_index", position)),
            "generated_question_id": question.get("generated_question_id", f"Q{position}"),
            "topic": question.get("topic", ""),
            "role": question.get("role", ""),
            "question_text": question.get("question_text", ""),
            "marking_guidance": safe_list(question.get("marking_guidance", [])),
            "visual_requirement": question.get("visual_requirement", "none"),
            "visual": question.get("visual", {"type": "none", "spec": {}}),
            "requires_code": bool(question.get("requires_code")),
        })
    filters = request_payload.get("assessment_filters", {})
    return {
        "raw_special_instruction": _request_special_instruction_text(request_payload),
        "hard_assessment_controls": filters if isinstance(filters, dict) else {},
        "questions": questions,
    }


def call_special_instruction_review_model(
    review_request: dict[str, Any],
) -> tuple[dict[str, Any], dict[str, Any]]:
    messages = [
        {"role": "system", "content": SPECIAL_INSTRUCTION_REVIEW_SYSTEM_PROMPT},
        {"role": "user", "content": json.dumps(review_request, ensure_ascii=False, default=str)},
    ]
    desired_output_tokens = min(3000, max(900, 450 * max(1, len(review_request.get("questions", [])))))
    safe_output_budget = provider_safe_output_budget_for_messages(
        messages,
        desired_output_tokens=desired_output_tokens,
    )
    if safe_output_budget < 512:
        raise ModelProviderTokenBudgetExceededError(
            "Independent special-instruction review has insufficient safe output budget."
        )
    schema = _special_instruction_review_schema()
    _set_active_model_call_audit_context({
        "call_stage": "independent_special_instruction_review",
        "trigger": "mandatory_post_generation_compliance",
        "plan_indexes": [safe_int(q.get("plan_index")) for q in review_request.get("questions", [])],
        "is_initial_generation": False,
        "is_targeted_regeneration": False,
        "prompt_profile": "independent_special_instruction_review_v1",
        "repair_transport_profile": "review_only",
        "recovery_attempt": 0,
        "transport_retry": "",
    })

    if GENERATION_PROVIDER == "openai":
        from openai import OpenAI
        api_key = str(os.getenv("OPENAI_API_KEY") or "").strip()
        if not api_key:
            raise RuntimeError("OPENAI_API_KEY is not set in Agent2/.env.")
        return _openai_json_call(
            OpenAI(api_key=api_key), model=GENERATION_MODEL, messages=messages,
            max_tokens=min(safe_output_budget, OPENAI_MAX_GENERATION_OUTPUT_TOKENS),
            response_schema=schema, force_json_object=True,
        )
    if GENERATION_PROVIDER == "google_gemini":
        from google import genai
        api_key = str(os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY") or "").strip()
        if not api_key:
            raise RuntimeError("GEMINI_API_KEY is not set in Agent2/.env.")
        return _gemini_json_call(
            genai.Client(api_key=api_key), model=GENERATION_MODEL, messages=messages,
            max_tokens=min(safe_output_budget, GEMINI_MAX_GENERATION_OUTPUT_TOKENS),
            response_schema=schema, force_mime_only=False,
        )
    if GENERATION_PROVIDER == "groq":
        from groq import Groq
        api_key = str(os.getenv("GROQ_API_KEY") or "").strip()
        if not api_key:
            raise RuntimeError("GROQ_API_KEY is not set in Agent2/.env.")
        return _groq_json_call(
            Groq(api_key=api_key), model=GENERATION_MODEL, messages=messages,
            max_tokens=min(safe_output_budget, GROQ_MAX_GENERATION_OUTPUT_TOKENS),
            response_schema=schema, force_json_object=True,
        )
    raise RuntimeError(f"Unsupported provider for special-instruction review: {GENERATION_PROVIDER!r}")


_SPECIAL_INSTRUCTION_REVIEW_CACHE: dict[str, dict[str, Any]] = {}


def review_special_instruction_compliance(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> dict[str, Any]:
    raw_instruction = _request_special_instruction_text(request_payload)
    if not raw_instruction:
        return {"status": "not_required", "failed_plan_indexes": [], "reasons": [], "per_question": []}
    review_request = _special_instruction_review_request(payload, request_payload)
    cache_key = stable_json_fingerprint(review_request)
    if cache_key not in _SPECIAL_INSTRUCTION_REVIEW_CACHE:
        review, metadata = call_special_instruction_review_model(review_request)
        review["model_call_metadata"] = metadata
        _SPECIAL_INSTRUCTION_REVIEW_CACHE[cache_key] = review
    return dict(_SPECIAL_INSTRUCTION_REVIEW_CACHE[cache_key])


_validate_render_and_quality_before_special_instruction_review = validate_render_and_quality


def validate_render_and_quality(
    payload: dict[str, Any],
    request_payload: dict[str, Any],
) -> tuple[dict[str, Any], dict[str, Any]]:
    structural, semantic = _validate_render_and_quality_before_special_instruction_review(
        payload, request_payload
    )
    raw_instruction = _request_special_instruction_text(request_payload)
    if not raw_instruction or not structural.get("valid", False):
        structural["independent_special_instruction_review"] = {
            "status": "not_required" if not raw_instruction else "blocked_by_structural_validation",
            "failed_plan_indexes": [], "reasons": [], "per_question": [],
        }
        return structural, semantic

    review = review_special_instruction_compliance(payload, request_payload)
    status = str(review.get("status", "") or "").strip().casefold()
    all_plan_indexes = [
        safe_int(q.get("plan_index", position))
        for position, q in enumerate(payload.get("questions", []), start=1)
        if isinstance(q, dict)
    ]
    failed_indexes = sorted({
        safe_int(value) for value in safe_list(review.get("failed_plan_indexes", []))
        if safe_int(value) > 0
    })
    if status != "compliant" and not failed_indexes:
        failed_indexes = sorted({value for value in all_plan_indexes if value > 0})
    review["failed_plan_indexes"] = failed_indexes
    structural["independent_special_instruction_review"] = review

    if status != "compliant":
        reasons = [str(value).strip() for value in safe_list(review.get("reasons", [])) if str(value).strip()]
        if not reasons:
            reasons = ["generated assessment did not satisfy the raw special instruction"]
        id_by_plan = {
            safe_int(q.get("plan_index", position)): str(q.get("generated_question_id", f"Q{position}"))
            for position, q in enumerate(payload.get("questions", []), start=1)
            if isinstance(q, dict)
        }
        if status == "conflict":
            # Keep conflicts quiz-global so the targeted repair mapper does
            # not waste a regeneration call on an impossible instruction.
            errors = [
                f"Special-instruction conflict: {reason}"
                for reason in reasons
            ]
        else:
            errors = [
                f"{id_by_plan.get(plan_index, f'Question {plan_index}')}: special-instruction compliance failure: {reason}"
                for plan_index in failed_indexes
                for reason in reasons
            ]
        structural.setdefault("errors", []).extend(errors)
        structural["errors"] = list(dict.fromkeys(structural["errors"]))
        structural["valid"] = False
        semantic["status"] = "NOT_RUN"
        semantic["release_eligible"] = False
        semantic.setdefault("reasons", []).extend(errors)

    return structural, semantic

generated_payload: dict[
    str,
    Any
] = {}

generation_validation: dict[
    str,
    Any
] = {}

semantic_quality_validation: dict[
    str,
    Any
] = {}

regeneration_attempts_used = 0

# v2.43.2:
# Extra targeted repair budget used ONLY when structural validation would
# otherwise block the final PDF. It does not regenerate already-valid questions.
STRUCTURAL_PDF_REPAIR_ATTEMPTS = max(
    0,
    safe_int(
        os.getenv(
            "AGENT2_STRUCTURAL_PDF_REPAIR_ATTEMPTS",
            "0",
        )
    ),
)

generation_attempt_log: list[
    dict[str, Any]
] = []


if PAPER_ROUTING_BLOCKED:

    generation_status = (
        "BLOCKED_PAPER_SELECTION_MISMATCH"
    )


elif not generation_required:

    generation_status = (
        "NOT_REQUIRED"
    )


elif not USER_APPROVED_GENERATION:

    generation_status = (
        "APPROVAL_REQUIRED"
    )


else:

    generation_status = (
        "GENERATION_APPROVED"
    )


if (
    generation_status
    == "GENERATION_APPROVED"
    and RUN_GENERATION
):

    try:
        generated_payload = generate_with_selected_model(
            generation_request
        )

    except ModelPreflightBlockedError as exc:
        # Graceful local stop: Notebook 06 continues to write its normal
        # manifest/report/usage artifacts instead of crashing the whole run.
        generation_status = "BLOCKED_MODEL_TOKEN_LIMIT"

        generated_payload = {}

        generation_validation = {
            "valid": False,
            "errors": [str(exc)],
            "warnings": [],
            "generated_marks": 0,
            "generated_questions": 0,
            "visual_validation": {
                "valid": False,
                "errors": [],
                "status": "NOT_RUN",
            },
        }

        semantic_quality_validation = {
            "status": "NOT_RUN",
            "release_eligible": False,
            "reasons": [str(exc)],
            "warnings": [],
        }

        print()
        print("QUIZ GENERATION BLOCKED LOCALLY")
        print(str(exc))
        print("No provider generation API call was sent for the blocked request.")
        print()

    else:
        (
            generation_validation,
            semantic_quality_validation,
        ) = validate_render_and_quality(
            generated_payload,
            generation_request,
        )


# --------------------------------------------------------
# v3 — one shared automatic LLM repair budget for the entire fresh run.
# A marking-only repair and a learner-facing corrective repair no longer have
# separate automatic budgets. Whichever path spends the one call first consumes
# the automatic repair allowance; any remaining issue goes to HITL.
# --------------------------------------------------------
automatic_repair_calls_used = 0
marking_repair_plan_indexes: list[int] = []
marking_repair_metadata: dict[str, Any] = {}

initial_marking_only_indexes = set(
    marking_only_failed_plan_indexes(
        generated_payload,
        generation_request,
    )
)
initial_structural_failed_indexes = set(
    structural_failed_plan_indexes(
        generated_payload,
        generation_validation,
    )
    if generation_validation and not generation_validation.get("valid", False)
    else []
)
marking_repair_is_exclusive = bool(
    initial_marking_only_indexes
    and not (initial_structural_failed_indexes - initial_marking_only_indexes)
    and semantic_quality_validation.get("status") != "FAIL"
)

try:
    if (
        marking_repair_is_exclusive
        and automatic_repair_calls_used < MAX_REGENERATION_ATTEMPTS
    ):
        # This is an actual LLM repair call, so it consumes the same one-call
        # automatic budget as any learner-facing corrective regeneration.
        automatic_repair_calls_used += 1
        (
            generated_payload,
            marking_repair_plan_indexes,
            marking_repair_metadata,
        ) = attempt_marking_scheme_repairs(
            generated_payload,
            generation_request,
        )
    elif initial_marking_only_indexes:
        print(
            "Skipping separate marking-only LLM repair because other learner-facing "
            "failures exist; the single corrective repair round will handle them together."
        )

except Exception as marking_repair_exc:
    print()
    print(
        "Marking-scheme-only repair was not completed:",
        f"{type(marking_repair_exc).__name__}: {marking_repair_exc}",
    )
    print(
        "The learner-facing question will not be regenerated solely "
        "for this marking-only issue. It remains blocked for HITL."
    )
    print()

else:
    if marking_repair_plan_indexes:
        (
            generation_validation,
            semantic_quality_validation,
        ) = validate_render_and_quality(
            generated_payload,
            generation_request,
        )

        generation_attempt_log.append(
            {
                "attempt_type":
                    "marking_scheme_only_repair",
                "timestamp_utc":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),
                "payload_fingerprint":
                    payload_fingerprint(
                        generated_payload
                    ),
                "targeted_plan_indexes":
                    marking_repair_plan_indexes,
                "structural_valid":
                    generation_validation.get(
                        "valid",
                        False,
                    ),
                "semantic_status":
                    semantic_quality_validation.get(
                        "status",
                        "NOT_RUN",
                    ),
                "model_call_metadata":
                    marking_repair_metadata,
            }
        )

        generated_payload[
            "_marking_repair_metadata"
        ] = {
            "plan_indexes":
                marking_repair_plan_indexes,
            "call_metadata":
                marking_repair_metadata,
        }

if RUN_GENERATION:
    generation_attempt_log.append(
        {
            "attempt_type":
                "initial_generation",
            "timestamp_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),
            "payload_fingerprint":
                payload_fingerprint(
                    generated_payload
                ),
            "structural_valid":
                generation_validation.get(
                    "valid",
                    False,
                ),
            "semantic_status":
                semantic_quality_validation.get(
                    "status"
                ),
        }
    )

# Persist the fresh candidate BEFORE deciding whether a corrective retry is
# needed. A valid clean run must survive the next HITL/Streamlit rerun without
# requiring another model call.
if (
    RUN_GENERATION
    and generation_status == "GENERATION_APPROVED"
    and generated_payload
):
    save_candidate_artifacts(
        generated_payload,
        generation_validation,
        semantic_quality_validation,
        regeneration_attempts_used,
        generation_attempt_log,
    )

needs_retry = bool(
    RUN_GENERATION
    and (
        (
            not generation_validation.get(
                "valid",
                False,
            )
        )
        or (
            AUTO_REGENERATE_ON_FAIL
            and semantic_quality_validation.get(
                "status"
            )
            == "FAIL"
        )
    )
)

if (
    needs_retry
    and automatic_repair_calls_used < MAX_REGENERATION_ATTEMPTS
):
    feedback = list(
        generation_validation.get(
            "errors",
            [],
        )
    ) + list(
        semantic_quality_validation.get(
            "reasons",
            [],
        )
    )

    semantic_failed_indexes = (
        semantic_failed_plan_indexes(
            generated_payload,
            semantic_quality_validation,
        )
        if semantic_quality_validation.get(
            "status"
        )
        == "FAIL"
        else []
    )

    structural_failed_indexes = (
        structural_failed_plan_indexes(
            generated_payload,
            generation_validation,
        )
        if not generation_validation.get(
            "valid",
            False,
        )
        else []
    )

    failed_plan_indexes = sorted(
        set(
            semantic_failed_indexes
            + structural_failed_indexes
        )
    )

    remaining_marking_only_indexes = set(
        marking_only_failed_plan_indexes(
            generated_payload,
            generation_request,
        )
    )

    if remaining_marking_only_indexes:
        failed_plan_indexes = [
            plan_index
            for plan_index in failed_plan_indexes
            if plan_index
            not in remaining_marking_only_indexes
        ]

        print(
            "Marking-only plan indexes excluded from full question "
            "regeneration:",
            sorted(
                remaining_marking_only_indexes
            ),
        )

    marking_only_blocks_full_regeneration = bool(
        remaining_marking_only_indexes
        and not failed_plan_indexes
    )

    if marking_only_blocks_full_regeneration:
        print(
            "No full learner-facing regeneration will run because "
            "the remaining failures are marking-scheme-only."
        )

    elif failed_plan_indexes:
        # Plan C automatic repair is question-local only.
        # One failed question may use one compact targeted provider request;
        # already-valid adaptive batches are never replayed.
        automatic_repair_calls_used += 1
        regeneration_attempts_used += 1

        try:
            print(
                "Automatic corrective regeneration will replace only "
                "failed plan_index(es):",
                failed_plan_indexes,
            )

            regenerated = regenerate_only_failed_questions(
                request_payload=
                    generation_request,
                current_payload=
                    generated_payload,
                failed_plan_indexes=
                    failed_plan_indexes,
                validation_feedback=
                    feedback,
                repair_trigger=
                    "automatic_corrective_validation",
            )

        except ModelPreflightBlockedError as exc:
            generation_status = (
                "BLOCKED_MODEL_TOKEN_LIMIT"
            )

            print()
            print(
                "TARGETED CORRECTIVE REGENERATION BLOCKED LOCALLY"
            )
            print(str(exc))
            print(
                "No provider generation API call was sent."
            )
            print()

        else:
            (
                regenerated_structural,
                regenerated_semantic,
            ) = validate_render_and_quality(
                regenerated,
                generation_request,
            )

            generated_payload = regenerated
            generation_validation = regenerated_structural
            semantic_quality_validation = regenerated_semantic

            generation_attempt_log.append(
                {
                    "attempt_type":
                        "automatic_targeted_regeneration",
                    "timestamp_utc":
                        datetime.now(
                            timezone.utc
                        ).isoformat(),
                    "payload_fingerprint":
                        payload_fingerprint(
                            generated_payload
                        ),
                    "structural_valid":
                        generation_validation[
                            "valid"
                        ],
                    "semantic_status":
                        semantic_quality_validation.get(
                            "status"
                        ),
                    "targeted_plan_indexes":
                        failed_plan_indexes,
                    "provider_call_used":
                        True,
                }
            )

        if generated_payload:
            save_candidate_artifacts(
                generated_payload,
                generation_validation,
                semantic_quality_validation,
                regeneration_attempts_used,
                generation_attempt_log,
            )

    else:
        # IMPORTANT COST CONTROL:
        # A global/quiz-wide validation failure has no safe question-local
        # repair target. Re-running generate_with_selected_model() here would
        # replay EVERY adaptive batch. For a 2-batch quiz that turns 2 calls
        # into 4; for a larger quiz it scales even worse.
        #
        # Preserve the generated candidate, keep validation blockers visible,
        # and escalate to HITL instead. No provider call is made.
        print(
            "Global/quiz-wide validation issue detected with no "
            "question-local plan_index. Automatic full-quiz regeneration "
            "is skipped; the current candidate is preserved for HITL."
        )

        generation_attempt_log.append(
            {
                "attempt_type":
                    "automatic_global_regeneration_skipped",
                "timestamp_utc":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),
                "payload_fingerprint":
                    payload_fingerprint(
                        generated_payload
                    ),
                "structural_valid":
                    generation_validation.get(
                        "valid",
                        False,
                    ),
                "semantic_status":
                    semantic_quality_validation.get(
                        "status"
                    ),
                "targeted_plan_indexes":
                    [],
                "provider_call_used":
                    False,
                "reason":
                    "No question-local failed plan_index; preserve candidate and escalate to HITL.",
            }
        )

        if generated_payload:
            save_candidate_artifacts(
                generated_payload,
                generation_validation,
                semantic_quality_validation,
                regeneration_attempts_used,
                generation_attempt_log,
            )


elif (
    generation_status
    == "GENERATION_APPROVED"
    and not RUN_GENERATION
):

    # Restore is ONLY for a non-generation rerun (for example HITL/resume).
    # A clean fresh generation must never be replaced by an older/absent
    # persisted candidate merely because no corrective retry was required.
    (
        generated_payload,
        regeneration_attempts_used,
        generation_attempt_log,
    ) = restore_matching_candidate()

    if generated_payload:
        (
            generation_validation,
            semantic_quality_validation,
        ) = validate_render_and_quality(
            generated_payload,
            generation_request,
        )

        print(
            "Restored candidate revalidated locally. No automatic LLM repair "
            "call is allowed on restore; unresolved issues remain for HITL."
        )


# ================================================================
# v2.43.2 — AUTO-REPAIR STRUCTURAL PDF BLOCKERS
# ================================================================
# If a generated question would block the PDF because of question-local
# structural/visual/referential validation, repair ONLY the failing question(s)
# before the later HITL/PDF stages.
#
# Existing valid questions remain untouched.
# ================================================================

structural_pdf_repair_attempts_used = 0
structural_pdf_repair_history: list[dict[str, Any]] = []

if (
    generation_status == "GENERATION_APPROVED"
    and isinstance(
        generated_payload,
        dict,
    )
    and generated_payload.get(
        "questions"
    )
):
    while (
        not bool(
            generation_validation.get(
                "valid",
                False,
            )
        )
        and structural_pdf_repair_attempts_used
        < STRUCTURAL_PDF_REPAIR_ATTEMPTS
    ):
        # First give the existing marking-only repair path one chance.
        # If it fixes the structural state, no learner-facing question changes.
        try:
            (
                generated_payload,
                pre_structural_marking_indexes,
                pre_structural_marking_metadata,
            ) = attempt_marking_scheme_repairs(
                generated_payload,
                generation_request,
            )
        except Exception as exc:
            pre_structural_marking_indexes = []
            pre_structural_marking_metadata = {
                "status": "not_applied",
                "reason": f"{type(exc).__name__}: {exc}",
            }
        else:
            if pre_structural_marking_indexes:
                (
                    generation_validation,
                    semantic_quality_validation,
                ) = validate_render_and_quality(
                    generated_payload,
                    generation_request,
                )

                if generation_validation.get(
                    "valid",
                    False,
                ):
                    generation_attempt_log.append(
                        {
                            "attempt_type":
                                "pre_pdf_marking_only_repair",
                            "timestamp_utc":
                                datetime.now(
                                    timezone.utc
                                ).isoformat(),
                            "targeted_plan_indexes":
                                pre_structural_marking_indexes,
                            "structural_valid":
                                True,
                            "model_call_metadata":
                                pre_structural_marking_metadata,
                        }
                    )
                    break

        blocker_indexes = structural_failed_plan_indexes(
            generated_payload,
            generation_validation,
        )

        # Only question-local blockers are automatically regenerated.
        # Global request/total/scope failures are not guessed at.
        if not blocker_indexes:
            print()
            print(
                "Structural validation is still failing, but no question-level "
                "plan_index could be identified. Automatic targeted repair stops "
                "without changing valid questions."
            )
            print()
            break

        structural_pdf_repair_attempts_used += 1

        blocker_feedback = list(
            generation_validation.get(
                "errors",
                [],
            )
        )

        print()
        print(
            "PDF structural blocker detected. Automatically regenerating only "
            "plan_index(es):",
            blocker_indexes,
            f"(repair {structural_pdf_repair_attempts_used}/"
            f"{STRUCTURAL_PDF_REPAIR_ATTEMPTS})",
        )

        try:
            repaired_payload = regenerate_only_failed_questions(
                request_payload=
                    generation_request,
                current_payload=
                    generated_payload,
                failed_plan_indexes=
                    blocker_indexes,
                validation_feedback=
                    blocker_feedback,
                repair_trigger=
                    "structural_pdf_blocker",
            )

        except ModelPreflightBlockedError as exc:
            structural_pdf_repair_history.append(
                {
                    "attempt":
                        structural_pdf_repair_attempts_used,
                    "targeted_plan_indexes":
                        blocker_indexes,
                    "status":
                        "blocked_model_token_limit",
                    "reason":
                        str(exc),
                }
            )
            print(
                "Targeted structural repair was blocked locally:",
                str(exc),
            )
            break

        except Exception as exc:
            structural_pdf_repair_history.append(
                {
                    "attempt":
                        structural_pdf_repair_attempts_used,
                    "targeted_plan_indexes":
                        blocker_indexes,
                    "status":
                        "repair_call_failed",
                    "reason":
                        f"{type(exc).__name__}: {exc}",
                }
            )
            print(
                "Targeted structural repair failed:",
                f"{type(exc).__name__}: {exc}",
            )
            break

        (
            repaired_validation,
            repaired_semantic,
        ) = validate_render_and_quality(
            repaired_payload,
            generation_request,
        )

        generated_payload = repaired_payload
        generation_validation = repaired_validation
        semantic_quality_validation = repaired_semantic

        structural_pdf_repair_history.append(
            {
                "attempt":
                    structural_pdf_repair_attempts_used,
                "targeted_plan_indexes":
                    blocker_indexes,
                "status":
                    (
                        "valid"
                        if generation_validation.get(
                            "valid",
                            False,
                        )
                        else "still_invalid"
                    ),
                "remaining_errors":
                    list(
                        generation_validation.get(
                            "errors",
                            [],
                        )
                    ),
            }
        )

        generation_attempt_log.append(
            {
                "attempt_type":
                    "automatic_pre_pdf_structural_repair",
                "timestamp_utc":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),
                "targeted_plan_indexes":
                    blocker_indexes,
                "repair_attempt":
                    structural_pdf_repair_attempts_used,
                "structural_valid":
                    generation_validation.get(
                        "valid",
                        False,
                    ),
                "semantic_status":
                    semantic_quality_validation.get(
                        "status",
                        "NOT_RUN",
                    ),
                "payload_fingerprint":
                    payload_fingerprint(
                        generated_payload
                    ),
            }
        )

    if structural_pdf_repair_history:
        generated_payload[
            "_structural_pdf_repair"
        ] = {
            "attempts_used":
                structural_pdf_repair_attempts_used,
            "max_attempts":
                STRUCTURAL_PDF_REPAIR_ATTEMPTS,
            "history":
                structural_pdf_repair_history,
            "final_structural_valid":
                bool(
                    generation_validation.get(
                        "valid",
                        False,
                    )
                ),
        }

        save_candidate_artifacts(
            generated_payload,
            generation_validation,
            semantic_quality_validation,
            regeneration_attempts_used,
            generation_attempt_log,
        )

# ================================================================
# PLAN C POST-GENERATION TRACE
# ================================================================
# Local diagnostics only — zero provider/API calls.
# This makes it explicit whether generated questions survived batching,
# parsing, aggregation and the later validation/repair path.
HYBRID_GENERATION_TRACE_PATH = (
    OUTPUT_DIR
    / "hybrid_generation_trace.json"
)

_trace_questions = (
    generated_payload.get("questions", [])
    if isinstance(generated_payload, dict)
    else []
)
if not isinstance(_trace_questions, list):
    _trace_questions = []

_trace_question_objects = [
    item
    for item in _trace_questions
    if isinstance(item, dict)
]

_trace_generation_metadata = (
    generated_payload.get("_generation_metadata", {})
    if isinstance(generated_payload, dict)
    else {}
)
if not isinstance(_trace_generation_metadata, dict):
    _trace_generation_metadata = {}

_trace_batches = _trace_generation_metadata.get("batches", [])
if not isinstance(_trace_batches, list):
    _trace_batches = []

_trace_payload = {
    "schema_version":
        "agent2-plan-c-hybrid-generation-trace-v1.0.0",

    "pipeline_version":
        NOTEBOOK06_PIPELINE_VERSION,

    "generated_at_utc":
        datetime.now(timezone.utc).isoformat(),

    "quiz_mode":
        QUIZ_MODE,

    "run_generation":
        bool(RUN_GENERATION),

    "generation_status":
        generation_status,

    "blueprint_question_count":
        len(
            [
                item
                for item in generation_request.get("blueprint", [])
                if isinstance(item, dict)
            ]
        ),

    "planned_batch_count":
        safe_int(
            _trace_generation_metadata.get(
                "initial_batch_count",
                0,
            )
        ),

    "completed_provider_batches":
        safe_int(
            _trace_generation_metadata.get(
                "completed_api_call_count",
                0,
            )
        ),

    "batches":
        [
            {
                "batch_number":
                    position,
                "expected_plan_indexes":
                    row.get("blueprint_plan_indexes", []),
                "expected_question_count":
                    row.get("question_count"),
                "parsed_question_count":
                    row.get("parsed_question_count"),
                "parsed_plan_indexes":
                    row.get("parsed_plan_indexes", []),
                "batch_cardinality_match":
                    row.get("batch_cardinality_match"),
                "adaptive_split_depth":
                    row.get("adaptive_split_depth"),
            }
            for position, row in enumerate(
                _trace_batches,
                start=1,
            )
            if isinstance(row, dict)
        ],

    "post_generation": {
        "combined_question_count":
            len(_trace_question_objects),
        "combined_plan_indexes":
            [
                safe_int(item.get("plan_index"))
                for item in _trace_question_objects
            ],
        "combined_marks":
            sum(
                max(0, safe_int(item.get("marks")))
                for item in _trace_question_objects
            ),
        "payload_fingerprint":
            payload_fingerprint(generated_payload)
            if generated_payload
            else None,
    },

    "post_structural": {
        "valid":
            bool(
                generation_validation.get(
                    "valid",
                    False,
                )
            ),
        "reported_question_count":
            safe_int(
                generation_validation.get(
                    "generated_questions",
                    0,
                )
            ),
        "reported_marks":
            safe_int(
                generation_validation.get(
                    "generated_marks",
                    0,
                )
            ),
        "failed_plan_indexes":
            (
                structural_failed_plan_indexes(
                    generated_payload,
                    generation_validation,
                )
                if generated_payload
                and generation_validation
                and not generation_validation.get("valid", False)
                else []
            ),
        "errors":
            list(
                generation_validation.get(
                    "errors",
                    [],
                )
            ),
    },

    "post_semantic": {
        "status":
            semantic_quality_validation.get(
                "status",
                "NOT_RUN",
            ),
        "failed_plan_indexes":
            (
                semantic_failed_plan_indexes(
                    generated_payload,
                    semantic_quality_validation,
                )
                if generated_payload
                and semantic_quality_validation.get("status") == "FAIL"
                else []
            ),
        "reasons":
            list(
                semantic_quality_validation.get(
                    "reasons",
                    [],
                )
            ),
    },

    "candidate_persisted":
        bool(generated_path.is_file()),

    "candidate_preserved_in_memory":
        bool(_trace_question_objects),

    "automatic_repair_policy":
        "question_local_targeted_only",

    "global_full_quiz_regeneration_allowed":
        False,

    "global_regeneration_skipped":
        any(
            isinstance(item, dict)
            and item.get("attempt_type")
            == "automatic_global_regeneration_skipped"
            for item in generation_attempt_log
        ),

    "automatic_repair_calls_used":
        safe_int(automatic_repair_calls_used),
}

HYBRID_GENERATION_TRACE_PATH.write_text(
    json.dumps(
        _trace_payload,
        indent=2,
        ensure_ascii=False,
        default=str,
    ),
    encoding="utf-8",
)

print(
    "Generation status:",
    generation_status,
)

print(
    "Generated candidate available:",
    bool(
        generated_payload
    ),
)

print(
    "Structural valid:",
    generation_validation.get(
        "valid",
        False,
    ),
)

print(
    "Deterministic quality status:",
    semantic_quality_validation.get(
        "status",
        "NOT_RUN",
    ),
)

print(
    "Secondary LLM reviewer used:",
    str(
        generation_validation.get(
            "independent_special_instruction_review",
            {},
        ).get("status", "not_required")
    ).casefold() not in {"not_required", "blocked_by_structural_validation"},
)

print(
    "Regeneration attempts:",
    regeneration_attempts_used,
    "/",
    MAX_REGENERATION_ATTEMPTS,
)

## Automatic structural repair policy — cost bounded

The fail-closed PDF gate remains intact, but Notebook 06C no longer runs a second independent multi-round repair loop by default.

The normal automatic flow is:

```text
initial generation
        ↓
deterministic validation
        ↓
question-local learner-facing failure?
        ↓
ONE targeted corrective repair round
        ↓
fresh validation
        ↓
still invalid → HITL / release blocked
```

`AGENT2_STRUCTURAL_PDF_REPAIR_ATTEMPTS` now defaults to **0**. It can be raised explicitly for experiments, but production does not spend extra LLM calls after the normal corrective round.

Already-valid questions are never regenerated. Global/non-question errors are not guessed at.

## 12. Human quality review — quiz-level + question-level HITL

Notebook 06 keeps the existing mandatory whole-quiz human gate and adds an optional granular review layer. The review queue surfaces topic grounding, command-word/mark-scheme alignment, assigned-pattern/learner-task alignment, and other deterministic warnings per generated question. A reviewer can approve a question, edit its text or marking guidance, relabel its assessment pattern, remove an irrelevant visual, regenerate only that question, or reject it. Every non-approve question-level action requires a written reason and is recorded in an audit/feedback JSONL. Deterministic edits add **no LLM cost**; selecting multiple questions for regeneration uses one targeted repair call where possible. Any content-changing action resets whole-quiz approval so the changed candidate must be seen again before release.


In [ ]:
ALLOWED_HUMAN_DECISIONS = {
    "pending",
    "approve",
    "regenerate",
    "reject",
}

ALLOWED_QUESTION_REVIEW_ACTIONS = {
    "approve",
    "edit_question",
    "edit_marking_guidance",
    "change_pattern",
    "remove_visual",
    "regenerate",
    "reject",
}

HUMAN_REVIEW_QUEUE_PATH = (
    OUTPUT_DIR
    / "generated_human_review_queue.json"
)

HUMAN_REVIEW_FEEDBACK_LOG_PATH = (
    OUTPUT_DIR
    / "generated_human_review_feedback.jsonl"
)


def _load_question_review_actions() -> tuple[list[dict[str, Any]], list[str]]:
    errors: list[str] = []
    raw_value: Any = None

    if HUMAN_REVIEW_ACTIONS_PATH:
        path = Path(HUMAN_REVIEW_ACTIONS_PATH).expanduser()
        if not path.is_file():
            return [], [
                "AGENT2_QUIZ_REVIEW_ACTIONS_PATH does not exist: "
                + str(path)
            ]
        try:
            raw_value = json.loads(path.read_text(encoding="utf-8"))
        except Exception as exc:
            return [], [
                "Could not parse question-level HITL actions file: "
                f"{type(exc).__name__}: {exc}"
            ]

    elif HUMAN_REVIEW_ACTIONS_JSON_RAW:
        try:
            raw_value = json.loads(HUMAN_REVIEW_ACTIONS_JSON_RAW)
        except Exception as exc:
            return [], [
                "Could not parse AGENT2_QUIZ_REVIEW_ACTIONS_JSON: "
                f"{type(exc).__name__}: {exc}"
            ]

    else:
        return [], []

    if isinstance(raw_value, dict):
        raw_value = raw_value.get("actions", raw_value.get("question_actions", []))

    if not isinstance(raw_value, list):
        return [], [
            "Question-level HITL actions must be a JSON list or an object containing an 'actions' list."
        ]

    actions = [item for item in raw_value if isinstance(item, dict)]
    if len(actions) != len(raw_value):
        errors.append(
            "Ignored one or more question-level HITL action entries because they were not JSON objects."
        )

    return actions, errors


def _question_identifier(question: dict[str, Any]) -> str:
    return str(
        question.get(
            "generated_question_id",
            question.get("question_id", ""),
        )
        or ""
    ).strip()


def _action_matches_question(
    action: dict[str, Any],
    question: dict[str, Any],
) -> bool:
    action_id = str(
        action.get("question_id", action.get("generated_question_id", ""))
        or ""
    ).strip()
    if action_id and action_id == _question_identifier(question):
        return True

    action_plan = safe_int(action.get("plan_index"))
    question_plan = safe_int(question.get("plan_index"))
    return bool(action_plan > 0 and action_plan == question_plan)


def _build_question_review_queue(
    payload: dict[str, Any],
    semantic_validation: dict[str, Any],
) -> list[dict[str, Any]]:
    """
    Build a LOW-NOISE question-level HITL queue.

    Only actionable validator findings move a question to REVIEW/HIGH.
    Generic informational warnings remain visible as notes but do not force the
    human to inspect an otherwise clean question.
    """
    signals_by_id = semantic_validation.get("deterministic_signals", {})
    if not isinstance(signals_by_id, dict):
        signals_by_id = {}

    queue: list[dict[str, Any]] = []

    for question in payload.get("questions", []) if isinstance(payload, dict) else []:
        if not isinstance(question, dict):
            continue

        question_id = _question_identifier(question) or f"PLAN_{safe_int(question.get('plan_index'))}"
        signals = signals_by_id.get(question_id, {})
        if not isinstance(signals, dict):
            signals = {}

        command_alignment = signals.get("command_marking_alignment", {})
        if not isinstance(command_alignment, dict):
            command_alignment = {}

        pattern_alignment = signals.get("pattern_task_alignment", {})
        if not isinstance(pattern_alignment, dict):
            pattern_alignment = {}

        grounding = signals.get("semantic_topic_grounding", {})
        if not isinstance(grounding, dict):
            grounding = {}

        cognitive = signals.get("cognitive_demand", {})
        if not isinstance(cognitive, dict):
            cognitive = {}

        visual_gate = question.get("visual_relevance_gate", {})
        if not isinstance(visual_gate, dict):
            visual_gate = {}

        hard_failures = list(signals.get("hard_failures", []) or [])
        warnings = list(signals.get("warnings", []) or [])

        issue_tags: list[str] = []

        if command_alignment.get("status") == "REVIEW":
            issue_tags.append("mark_scheme_command_alignment")

        if pattern_alignment.get("status") == "REVIEW":
            issue_tags.append("pattern_task_alignment")

        grounding_status = str(
            grounding.get("status", "")
            or ""
        ).strip().upper()
        if grounding_status in {"REVIEW", "FAIL"}:
            issue_tags.append("topic_grounding")

        if cognitive.get("status") == "REVIEW":
            issue_tags.append("low_cognitive_demand")

        # A visual that the gate already removed is not itself a blocking
        # review issue; the fix is deterministic and already applied. Keep the
        # status visible so Streamlit can explain why no visual was rendered.
        visual_necessity = str(
            visual_gate.get("necessity", "")
            or (
                "UNNECESSARY"
                if str(visual_gate.get("status", "")).upper()
                == "REMOVED_IRRELEVANT"
                else "NONE"
            )
        ).strip().upper()

        if hard_failures:
            issue_tags.append("hard_quality_failure")

        if hard_failures:
            priority = "HIGH"
            recommended_action = "regenerate_or_edit"
        elif issue_tags:
            priority = "REVIEW"
            recommended_action = "inspect_then_approve_edit_or_regenerate"
        else:
            priority = "NORMAL"
            recommended_action = "approve"

        queue.append(
            {
                "question_id": question_id,
                "plan_index": safe_int(question.get("plan_index")),
                "topic": str(question.get("topic", "") or "").strip(),
                "official_reference": str(
                    question.get("official_reference", "") or ""
                ).strip(),
                "role": str(question.get("role", "") or "").strip(),
                "marks": safe_int(question.get("marks")),
                "question_text": str(
                    question.get("question_text", "") or ""
                ).strip(),
                "marking_guidance": safe_list(
                    question.get("marking_guidance", [])
                ),
                "assigned_pattern": str(
                    question.get("assessment_pattern", "") or ""
                ).strip(),
                "suggested_pattern": pattern_alignment.get("suggested_pattern"),
                "pattern_alignment_status": pattern_alignment.get(
                    "status", "UNAVAILABLE"
                ),
                "pattern_alignment_reason": pattern_alignment.get(
                    "reason", ""
                ),
                "command_marking_alignment_status": command_alignment.get(
                    "status", "UNAVAILABLE"
                ),
                "command_marking_issues": command_alignment.get(
                    "issues", []
                ),
                "topic_grounding_status": grounding.get(
                    "status", "UNAVAILABLE"
                ),
                "topic_grounding_reason": grounding.get("reason", ""),
                "cognitive_demand_status": cognitive.get(
                    "status", "UNAVAILABLE"
                ),
                "cognitive_demand_reason": cognitive.get("reason", ""),
                "visual_relevance_status": visual_gate.get(
                    "status", "NOT_RUN"
                ),
                "visual_necessity": visual_necessity,
                "visual_relevance_reason": visual_gate.get("reason", ""),
                "hard_failures": hard_failures,
                "notes": warnings,
                "issue_tags": list(dict.fromkeys(issue_tags)),
                "priority": priority,
                "recommended_action": recommended_action,
                "available_actions": [
                    "approve",
                    "edit_question",
                    "edit_marking_guidance",
                    "change_pattern",
                    "remove_visual",
                    "regenerate",
                    "reject",
                ],
                "reason_required_for": [
                    "edit_question",
                    "edit_marking_guidance",
                    "change_pattern",
                    "remove_visual",
                    "regenerate",
                    "reject",
                ],
                "reason_prompt": (
                    "Add a short reason. The reason is saved to PostgreSQL and "
                    "may be reused as generation memory for future similar "
                    "targeted regenerations."
                ),
            }
        )

    return queue


def _append_human_feedback_events(
    events: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """
    Persist the local JSONL audit and return the canonical enriched events.

    The returned list is then reused for PostgreSQL/Qdrant persistence so every
    store shares the same deterministic event_id.
    """
    if not events:
        return []

    existing_ids: set[str] = set()
    if HUMAN_REVIEW_FEEDBACK_LOG_PATH.is_file():
        for line in HUMAN_REVIEW_FEEDBACK_LOG_PATH.read_text(encoding="utf-8").splitlines():
            try:
                item = json.loads(line)
            except Exception:
                continue
            event_id = str(item.get("event_id", "") or "")
            if event_id:
                existing_ids.add(event_id)

    enriched_events: list[dict[str, Any]] = []
    lines_to_append: list[str] = []

    for event in events:
        canonical = {
            key: value
            for key, value in event.items()
            if key not in {"event_id", "recorded_at_utc"}
        }
        event_id = stable_json_fingerprint(canonical)
        enriched = {
            **event,
            "event_id": event_id,
            "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        enriched_events.append(enriched)

        if event_id in existing_ids:
            continue

        existing_ids.add(event_id)
        lines_to_append.append(
            json.dumps(enriched, ensure_ascii=False, default=str)
        )

    if lines_to_append:
        HUMAN_REVIEW_FEEDBACK_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
        with HUMAN_REVIEW_FEEDBACK_LOG_PATH.open("a", encoding="utf-8") as handle:
            for line in lines_to_append:
                handle.write(line + "\n")

    return enriched_events


_HITL_MEMORY_ACTIONS = {
    "reject",
    "regenerate",
    "edit_question",
    "edit_marking_guidance",
    "change_pattern",
    "remove_visual",
}


def _feedback_memory_text(event: dict[str, Any]) -> str:
    before = event.get("before_question", {})
    after = event.get("after_question", {})
    if not isinstance(before, dict):
        before = {}
    if not isinstance(after, dict):
        after = {}

    issue_tags = [
        str(value).strip()
        for value in (event.get("issue_tags") or [])
        if str(value).strip()
    ]
    hard_failures = [
        str(value).strip()
        for value in (event.get("hard_failures") or [])
        if str(value).strip()
    ]

    pieces = [
        "Agent 2 human feedback memory.",
        "Topic: " + str(event.get("topic", "") or "").strip(),
        "AQA reference: " + str(event.get("official_reference", "") or "").strip(),
        "Role: " + str(event.get("role", "") or "").strip(),
        "Assessment pattern: " + str(event.get("assigned_pattern", "") or "").strip(),
        "Action: " + str(event.get("action", "") or "").strip(),
        "Issue tags: " + ", ".join(issue_tags),
        "Validator failures: " + " | ".join(hard_failures),
        "Human reason: " + str(event.get("reason", "") or "").strip(),
        "Original question: " + str(before.get("question_text", "") or "").strip(),
    ]

    after_text = str(after.get("question_text", "") or "").strip()
    if after_text and after_text != str(before.get("question_text", "") or "").strip():
        pieces.append("Corrected/revised question: " + after_text)

    before_guidance = before.get("marking_guidance", [])
    after_guidance = after.get("marking_guidance", [])
    if after_guidance and after_guidance != before_guidance:
        pieces.append(
            "Corrected/revised marking guidance: "
            + json.dumps(after_guidance, ensure_ascii=False, default=str)
        )

    return "\n".join(piece for piece in pieces if piece.strip())


def _feedback_memory_eligible(event: dict[str, Any]) -> bool:
    return bool(
        str(event.get("action", "") or "").strip().casefold()
        in _HITL_MEMORY_ACTIONS
        and str(event.get("reason", "") or "").strip()
        and str(event.get("official_reference", "") or "").strip()
    )


def _persist_human_feedback_to_postgres(
    events: list[dict[str, Any]],
) -> dict[str, Any]:
    """
    PostgreSQL is the durable source of truth for question-level HITL.

    Failure is fail-soft because the JSONL audit has already been written. The
    status is surfaced to the frontend/report so an operator can retry storage.
    """
    database_url = str(os.getenv("AGENT2_DATABASE_URL", "") or "").strip()

    if not HITL_FEEDBACK_DB_ENABLED:
        return {
            "status": "DISABLED",
            "persisted": 0,
            "error": None,
        }

    if not database_url:
        return {
            "status": "SKIPPED_NO_DATABASE_URL",
            "persisted": 0,
            "error": None,
        }

    if not events:
        return {
            "status": "NO_EVENTS",
            "persisted": 0,
            "error": None,
        }

    try:
        from sqlalchemy import create_engine, text

        engine = create_engine(
            database_url,
            pool_pre_ping=True,
            future=True,
        )

        ddl = """
        CREATE TABLE IF NOT EXISTS agent2_question_feedback (
            id BIGSERIAL PRIMARY KEY,
            event_id TEXT UNIQUE NOT NULL,
            generation_request_fingerprint TEXT,
            payload_fingerprint TEXT,
            question_id TEXT,
            plan_index INTEGER,
            topic TEXT,
            official_reference TEXT,
            role TEXT,
            assigned_pattern TEXT,
            action TEXT NOT NULL,
            reason TEXT,
            before_question JSONB,
            after_question JSONB,
            model_key TEXT,
            model_id TEXT,
            pipeline_version TEXT,
            memory_eligible BOOLEAN NOT NULL DEFAULT FALSE,
            recorded_at_utc TIMESTAMPTZ NOT NULL,
            created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
        )
        """

        insert_sql = text(
            """
            INSERT INTO agent2_question_feedback (
                event_id,
                generation_request_fingerprint,
                payload_fingerprint,
                question_id,
                plan_index,
                topic,
                official_reference,
                role,
                assigned_pattern,
                action,
                reason,
                before_question,
                after_question,
                model_key,
                model_id,
                pipeline_version,
                memory_eligible,
                recorded_at_utc
            )
            VALUES (
                :event_id,
                :generation_request_fingerprint,
                :payload_fingerprint,
                :question_id,
                :plan_index,
                :topic,
                :official_reference,
                :role,
                :assigned_pattern,
                :action,
                :reason,
                CAST(:before_question AS JSONB),
                CAST(:after_question AS JSONB),
                :model_key,
                :model_id,
                :pipeline_version,
                :memory_eligible,
                CAST(:recorded_at_utc AS TIMESTAMPTZ)
            )
            ON CONFLICT (event_id) DO NOTHING
            """
        )

        persisted = 0
        with engine.begin() as connection:
            connection.execute(text(ddl))
            connection.execute(
                text(
                    """
                    CREATE INDEX IF NOT EXISTS idx_agent2_question_feedback_reference
                    ON agent2_question_feedback (official_reference)
                    """
                )
            )
            connection.execute(
                text(
                    """
                    CREATE INDEX IF NOT EXISTS idx_agent2_question_feedback_action
                    ON agent2_question_feedback (action)
                    """
                )
            )

            for event in events:
                result = connection.execute(
                    insert_sql,
                    {
                        "event_id": str(event.get("event_id", "") or ""),
                        "generation_request_fingerprint": str(
                            event.get("generation_request_fingerprint", "") or ""
                        ),
                        "payload_fingerprint": str(
                            event.get("final_payload_fingerprint", "") or ""
                        ),
                        "question_id": str(event.get("question_id", "") or ""),
                        "plan_index": safe_int(event.get("plan_index")),
                        "topic": str(event.get("topic", "") or ""),
                        "official_reference": str(
                            event.get("official_reference", "") or ""
                        ),
                        "role": str(event.get("role", "") or ""),
                        "assigned_pattern": str(
                            event.get("assigned_pattern", "") or ""
                        ),
                        "action": str(event.get("action", "") or ""),
                        "reason": str(event.get("reason", "") or ""),
                        "before_question": json.dumps(
                            event.get("before_question", {}),
                            ensure_ascii=False,
                            default=str,
                        ),
                        "after_question": json.dumps(
                            event.get("after_question", {}),
                            ensure_ascii=False,
                            default=str,
                        ),
                        "model_key": SELECTED_MODEL_KEY,
                        "model_id": GENERATION_MODEL,
                        "pipeline_version": NOTEBOOK06_PIPELINE_VERSION,
                        "memory_eligible": _feedback_memory_eligible(event),
                        "recorded_at_utc": str(
                            event.get("recorded_at_utc", "")
                            or datetime.now(timezone.utc).isoformat()
                        ),
                    },
                )
                persisted += max(0, int(result.rowcount or 0))

        return {
            "status": "OK",
            "persisted": persisted,
            "error": None,
        }

    except Exception as exc:
        return {
            "status": "ERROR",
            "persisted": 0,
            "error": f"{type(exc).__name__}: {exc}",
        }


def _qdrant_client_for_hitl_memory():
    if (
        not HITL_MEMORY_QDRANT_ENABLED
        or not HITL_MEMORY_QDRANT_URL
    ):
        return None, "disabled_or_unconfigured"

    try:
        from qdrant_client import QdrantClient
    except Exception as exc:
        return None, f"qdrant_client_unavailable: {type(exc).__name__}: {exc}"

    kwargs: dict[str, Any] = {
        "url": HITL_MEMORY_QDRANT_URL,
    }
    if HITL_MEMORY_QDRANT_API_KEY:
        kwargs["api_key"] = HITL_MEMORY_QDRANT_API_KEY

    try:
        return QdrantClient(**kwargs), "ok"
    except Exception as exc:
        return None, f"qdrant_connection_error: {type(exc).__name__}: {exc}"


def _ensure_hitl_memory_collection(
    client: Any,
    vector_size: int,
) -> None:
    from qdrant_client.models import Distance, VectorParams

    try:
        client.get_collection(HITL_MEMORY_COLLECTION)
        return
    except Exception:
        pass

    client.create_collection(
        collection_name=HITL_MEMORY_COLLECTION,
        vectors_config=VectorParams(
            size=int(vector_size),
            distance=Distance.COSINE,
        ),
    )


def _promote_human_feedback_to_qdrant(
    events: list[dict[str, Any]],
) -> dict[str, Any]:
    eligible = [
        event
        for event in events
        if _feedback_memory_eligible(event)
    ]

    if not eligible:
        return {
            "status": "NO_ELIGIBLE_EVENTS",
            "promoted": 0,
            "error": None,
        }

    client, client_status = _qdrant_client_for_hitl_memory()
    if client is None:
        return {
            "status": client_status,
            "promoted": 0,
            "error": None,
        }

    texts = [
        _feedback_memory_text(event)
        for event in eligible
    ]

    vectors, backend = _minilm_embeddings(texts)
    if vectors is None or len(vectors) != len(eligible):
        return {
            "status": "EMBEDDING_UNAVAILABLE",
            "promoted": 0,
            "error": backend,
        }

    try:
        import uuid
        from qdrant_client.models import PointStruct

        _ensure_hitl_memory_collection(
            client,
            int(vectors.shape[1]),
        )

        points = []
        for event, text_value, vector in zip(
            eligible,
            texts,
            vectors,
        ):
            point_id = str(
                uuid.uuid5(
                    uuid.NAMESPACE_URL,
                    "agent2-hitl:" + str(event.get("event_id", "")),
                )
            )
            points.append(
                PointStruct(
                    id=point_id,
                    vector=np.asarray(vector, dtype=float).tolist(),
                    payload={
                        "event_id": str(event.get("event_id", "") or ""),
                        "topic": str(event.get("topic", "") or ""),
                        "official_reference": str(
                            event.get("official_reference", "") or ""
                        ),
                        "role": str(event.get("role", "") or ""),
                        "assigned_pattern": str(
                            event.get("assigned_pattern", "") or ""
                        ),
                        "action": str(event.get("action", "") or ""),
                        "reason": str(event.get("reason", "") or ""),
                        "issue_tags": [
                            str(value).strip()
                            for value in (event.get("issue_tags") or [])
                            if str(value).strip()
                        ],
                        "hard_failures": [
                            str(value).strip()
                            for value in (event.get("hard_failures") or [])
                            if str(value).strip()
                        ],
                        "memory_text": text_value,
                        "pipeline_version": NOTEBOOK06_PIPELINE_VERSION,
                        "recorded_at_utc": str(
                            event.get("recorded_at_utc", "") or ""
                        ),
                    },
                )
            )

        client.upsert(
            collection_name=HITL_MEMORY_COLLECTION,
            points=points,
            wait=True,
        )

        return {
            "status": "OK",
            "promoted": len(points),
            "embedding_backend": backend,
            "error": None,
        }

    except Exception as exc:
        return {
            "status": "ERROR",
            "promoted": 0,
            "error": f"{type(exc).__name__}: {exc}",
        }


HITL_MEMORY_LOOKUP_AUDIT: list[dict[str, Any]] = []


def _record_hitl_memory_lookup(
    *,
    question: dict[str, Any],
    source: str,
    hits: list[dict[str, Any]],
    status: str = "OK",
    detail: str = "",
) -> None:
    HITL_MEMORY_LOOKUP_AUDIT.append(
        {
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "question_id": str(
                question.get("generated_question_id")
                or question.get("question_id")
                or ""
            ).strip(),
            "plan_index": safe_int(question.get("plan_index")),
            "official_reference": str(
                question.get("official_reference", "") or ""
            ).strip(),
            "assigned_pattern": str(
                question.get("assessment_pattern", "") or ""
            ).strip(),
            "source": source,
            "status": status,
            "hit_count": len(hits),
            "used_in_prompt": bool(hits),
            "hits": hits,
            "detail": detail,
        }
    )


def _hitl_review_issue_text(
    review_item: dict[str, Any] | None,
    *,
    current_action: str = "",
    human_reason: str = "",
) -> str:
    """
    Focused CURRENT problem description used for memory compatibility.

    Important:
    - current_action is scored separately by _hitl_action_compatibility;
    - when validator findings already describe the problem, a generic human
      reason is NOT mixed into this embedding because that dilutes issue
      similarity;
    - the human reason still remains highest-priority generation guidance in
      the actual targeted-regeneration prompt.
    """
    review_item = review_item if isinstance(review_item, dict) else {}

    issue_parts: list[str] = []

    for tag in review_item.get("issue_tags") or []:
        value = str(tag or "").strip()
        if value:
            issue_parts.append("Issue tag: " + value)

    for failure in review_item.get("hard_failures") or []:
        value = str(failure or "").strip()
        if value:
            issue_parts.append("Validator failure: " + value)

    for issue in review_item.get("command_marking_issues") or []:
        value = str(issue or "").strip()
        if value:
            issue_parts.append("Mark-scheme issue: " + value)

    if str(review_item.get("pattern_alignment_status") or "").upper() == "REVIEW":
        value = str(review_item.get("pattern_alignment_reason") or "").strip()
        if value:
            issue_parts.append("Pattern issue: " + value)

    grounding_status = str(
        review_item.get("topic_grounding_status") or ""
    ).upper()
    if grounding_status in {"REVIEW", "FAIL"}:
        value = str(review_item.get("topic_grounding_reason") or "").strip()
        if value:
            issue_parts.append("Grounding issue: " + value)

    if str(review_item.get("cognitive_demand_status") or "").upper() == "REVIEW":
        value = str(review_item.get("cognitive_demand_reason") or "").strip()
        if value:
            issue_parts.append("Cognitive-demand issue: " + value)

    # If validators have no issue signal, the human reason becomes the issue
    # description. Otherwise it is kept out of the issue embedding to avoid
    # diluting a concrete validator problem with generic wording.
    if not issue_parts:
        reason_value = str(human_reason or "").strip()
        if reason_value:
            issue_parts.append("Human correction request: " + reason_value)

    return "\n".join(issue_parts)


def _hitl_action_compatibility(
    current_action: str,
    memory_action: str,
) -> float:
    """
    Generic compatibility prior between the action being performed now and
    the type of correction that produced the old memory.

    It is deliberately only one component of ranking; semantic/issue
    similarity remains dominant.
    """
    current = str(current_action or "").strip().casefold()
    memory = str(memory_action or "").strip().casefold()

    if not current or not memory:
        return 0.5
    if current == memory:
        return 1.0

    compatibility: dict[str, dict[str, float]] = {
        "regenerate": {
            "edit_marking_guidance": 0.90,
            "edit_question": 0.85,
            "remove_visual": 0.55,
            "change_pattern": 0.35,
            "reject": 0.15,
        },
        "edit_marking_guidance": {
            "regenerate": 0.85,
            "edit_question": 0.55,
            "change_pattern": 0.20,
            "remove_visual": 0.15,
            "reject": 0.10,
        },
        "edit_question": {
            "regenerate": 0.85,
            "edit_marking_guidance": 0.55,
            "change_pattern": 0.30,
            "remove_visual": 0.25,
            "reject": 0.10,
        },
        "change_pattern": {
            "regenerate": 0.55,
            "edit_question": 0.30,
            "edit_marking_guidance": 0.20,
            "remove_visual": 0.15,
            "reject": 0.10,
        },
        "remove_visual": {
            "regenerate": 0.55,
            "edit_question": 0.25,
            "edit_marking_guidance": 0.15,
            "change_pattern": 0.15,
            "reject": 0.10,
        },
    }
    return float(compatibility.get(current, {}).get(memory, 0.25))


def _cosine_from_vectors(left: Any, right: Any) -> float:
    left_arr = np.asarray(left, dtype=float)
    right_arr = np.asarray(right, dtype=float)
    denominator = float(
        np.linalg.norm(left_arr) * np.linalg.norm(right_arr)
    )
    if denominator <= 0:
        return 0.0
    return float(np.dot(left_arr, right_arr) / denominator)


def _rank_hitl_memory_candidates(
    *,
    question: dict[str, Any],
    candidates: list[dict[str, Any]],
    current_action: str,
    review_item: dict[str, Any] | None,
    human_reason: str,
) -> tuple[list[str], list[dict[str, Any]], str]:
    """
    Rerank same-reference candidates using:
      1) current-question semantic similarity,
      2) current validator-issue similarity,
      3) human-action compatibility,
      4) assessment-pattern compatibility.

    This prevents a semantically similar but correction-type-incompatible
    memory (for example a pattern relabel) from overriding a more useful past
    quality/mark-scheme correction.
    """
    if not candidates:
        return [], [], "No same-reference memory candidates."

    question_query = "\n".join(
        [
            "Topic: " + str(question.get("topic", "") or "").strip(),
            "AQA reference: "
            + str(question.get("official_reference", "") or "").strip(),
            "Assessment pattern: "
            + str(question.get("assessment_pattern", "") or "").strip(),
            "Question: "
            + str(question.get("question_text", "") or "").strip(),
        ]
    )

    issue_query = _hitl_review_issue_text(
        review_item,
        current_action=current_action,
        human_reason=human_reason,
    )

    memory_texts = [
        str(
            candidate.get("memory_text")
            or candidate.get("reason")
            or ""
        ).strip()
        for candidate in candidates
    ]

    question_vectors, question_backend = _minilm_embeddings(
        [question_query, *memory_texts]
    )
    if (
        question_vectors is None
        or len(question_vectors) != len(memory_texts) + 1
    ):
        return [], [], (
            "Local semantic rerank unavailable: "
            + str(question_backend or "embedding failure")
        )

    question_vector = question_vectors[0]
    memory_vectors = question_vectors[1:]

    # Semantic similarity uses the full memory text. Issue similarity uses a
    # focused correction profile so old memories remain useful even when their
    # revised-question text later drifted away from the original correction.
    candidate_issue_texts = []
    for candidate in candidates:
        issue_tags = [
            str(value).strip()
            for value in (candidate.get("issue_tags") or [])
            if str(value).strip()
        ]
        hard_failures = [
            str(value).strip()
            for value in (candidate.get("hard_failures") or [])
            if str(value).strip()
        ]
        candidate_issue_texts.append(
            "\n".join(
                part
                for part in (
                    "Human correction reason: "
                    + str(candidate.get("reason", "") or "").strip(),
                    (
                        "Stored issue tags: " + ", ".join(issue_tags)
                        if issue_tags
                        else ""
                    ),
                    (
                        "Stored validator failures: "
                        + " | ".join(hard_failures)
                        if hard_failures
                        else ""
                    ),
                )
                if part.strip()
            )
        )

    issue_similarities = [0.0 for _ in candidates]
    issue_backend = "not_needed"

    if issue_query.strip():
        issue_vectors, issue_backend = _minilm_embeddings(
            [issue_query, *candidate_issue_texts]
        )
        if (
            issue_vectors is not None
            and len(issue_vectors) == len(candidate_issue_texts) + 1
        ):
            issue_vector = issue_vectors[0]
            issue_similarities = [
                max(
                    0.0,
                    min(
                        1.0,
                        _cosine_from_vectors(issue_vector, vector),
                    ),
                )
                for vector in issue_vectors[1:]
            ]

    current_pattern = str(
        question.get("assessment_pattern", "") or ""
    ).strip().casefold()

    ranked: list[dict[str, Any]] = []

    for index, candidate in enumerate(candidates):
        local_semantic = max(
            0.0,
            min(
                1.0,
                _cosine_from_vectors(
                    question_vector,
                    memory_vectors[index],
                ),
            ),
        )
        issue_similarity = float(issue_similarities[index])

        memory_action = str(
            candidate.get("action", "") or ""
        ).strip().casefold()
        action_compatibility = _hitl_action_compatibility(
            current_action,
            memory_action,
        )

        memory_pattern = str(
            candidate.get("memory_assigned_pattern")
            or candidate.get("assigned_pattern")
            or ""
        ).strip().casefold()

        if current_pattern and memory_pattern:
            pattern_compatibility = (
                1.0 if current_pattern == memory_pattern else 0.0
            )
        else:
            pattern_compatibility = 0.5

        if issue_query.strip():
            compatibility_score = (
                0.35 * local_semantic
                + 0.45 * issue_similarity
                + 0.15 * action_compatibility
                + 0.05 * pattern_compatibility
            )
            compatible = bool(
                local_semantic >= HITL_MEMORY_CANDIDATE_MIN_SIMILARITY
                and compatibility_score
                >= HITL_MEMORY_COMPATIBILITY_THRESHOLD
                and (
                    issue_similarity >= HITL_MEMORY_MIN_ISSUE_SIMILARITY
                    or (
                        action_compatibility >= 0.95
                        and local_semantic >= HITL_MEMORY_MIN_SIMILARITY
                    )
                )
            )
        else:
            compatibility_score = (
                0.75 * local_semantic
                + 0.20 * action_compatibility
                + 0.05 * pattern_compatibility
            )
            compatible = bool(
                local_semantic >= HITL_MEMORY_MIN_SIMILARITY
                and compatibility_score
                >= HITL_MEMORY_COMPATIBILITY_THRESHOLD
            )

        ranked.append(
            {
                **candidate,
                "local_semantic_similarity": round(local_semantic, 4),
                "issue_similarity": round(issue_similarity, 4),
                "issue_similarity_basis": "human_reason+stored_issue_context",
                "candidate_issue_profile": candidate_issue_texts[index],
                "action_compatibility": round(action_compatibility, 4),
                "pattern_compatibility": round(pattern_compatibility, 4),
                "compatibility_score": round(
                    float(compatibility_score),
                    4,
                ),
                "selected": bool(compatible),
            }
        )

    ranked.sort(
        key=lambda item: (
            bool(item.get("selected")),
            float(item.get("compatibility_score") or 0.0),
            float(item.get("local_semantic_similarity") or 0.0),
        ),
        reverse=True,
    )

    selected = [
        item
        for item in ranked
        if bool(item.get("selected"))
    ][:HITL_MEMORY_MAX_PROMPT_HINTS]

    hints: list[str] = []

    for item in selected:
        reason = str(item.get("reason", "") or "").strip()
        action = str(item.get("action", "") or "").strip()
        hints.append(
            "Compatible previous human feedback "
            f"(compatibility={float(item.get('compatibility_score') or 0.0):.3f}, "
            f"semantic={float(item.get('local_semantic_similarity') or 0.0):.3f}, "
            f"issue={float(item.get('issue_similarity') or 0.0):.3f}, "
            f"action={action}): {reason}"
        )

    audit_hits: list[dict[str, Any]] = []
    for item in selected:
        audit_hits.append(
            {
                "point_id": str(item.get("point_id", "") or ""),
                "event_id": str(item.get("event_id", "") or ""),
                "retrieval_similarity": round(
                    float(item.get("retrieval_similarity") or 0.0),
                    4,
                ),
                "local_semantic_similarity": item.get(
                    "local_semantic_similarity"
                ),
                "issue_similarity": item.get("issue_similarity"),
                "action_compatibility": item.get(
                    "action_compatibility"
                ),
                "pattern_compatibility": item.get(
                    "pattern_compatibility"
                ),
                "compatibility_score": item.get(
                    "compatibility_score"
                ),
                "action": str(item.get("action", "") or ""),
                "reason": str(item.get("reason", "") or ""),
                "memory_assigned_pattern": str(
                    item.get("memory_assigned_pattern")
                    or item.get("assigned_pattern")
                    or ""
                ),
                "issue_tags": item.get("issue_tags") or [],
            }
        )

    detail = (
        f"candidate_count={len(candidates)}; "
        f"selected_count={len(selected)}; "
        "strategy=question_semantic+reason_focused_issue+action+pattern; "
        f"embedding={question_backend}; issue_embedding={issue_backend}; "
        f"candidate_min_semantic={HITL_MEMORY_CANDIDATE_MIN_SIMILARITY}; "
        f"min_issue_similarity={HITL_MEMORY_MIN_ISSUE_SIMILARITY}; "
        f"compatibility_threshold={HITL_MEMORY_COMPATIBILITY_THRESHOLD}."
    )

    return hints, audit_hits, detail


def _retrieve_hitl_memory_hints_from_postgres(
    question: dict[str, Any],
    *,
    current_action: str = "",
    review_item: dict[str, Any] | None = None,
    human_reason: str = "",
) -> tuple[list[str], list[dict[str, Any]], str]:
    """
    Exact-reference fallback when Qdrant is unavailable.

    It retrieves a broader same-reference set and applies the SAME local
    issue/action compatibility reranker used for Qdrant candidates.
    """
    database_url = str(
        os.getenv("AGENT2_DATABASE_URL", "") or ""
    ).strip()
    reference = str(
        question.get("official_reference", "") or ""
    ).strip()

    if (
        not HITL_FEEDBACK_DB_ENABLED
        or not database_url
        or not reference
    ):
        return [], [], "PostgreSQL feedback memory unavailable."

    try:
        from sqlalchemy import create_engine, text

        engine = create_engine(
            database_url,
            pool_pre_ping=True,
            future=True,
        )

        with engine.connect() as connection:
            rows = connection.execute(
                text(
                    """
                    SELECT event_id, action, reason, assigned_pattern,
                           before_question, after_question, recorded_at_utc
                    FROM agent2_question_feedback
                    WHERE official_reference = :official_reference
                      AND memory_eligible = TRUE
                      AND COALESCE(reason, '') <> ''
                    ORDER BY recorded_at_utc DESC
                    LIMIT :limit_value
                    """
                ),
                {
                    "official_reference": reference,
                    "limit_value": int(max(HITL_MEMORY_TOP_K * 3, 12)),
                },
            ).mappings().all()

        candidates: list[dict[str, Any]] = []

        for row in rows:
            before_question = row.get("before_question")
            after_question = row.get("after_question")

            if isinstance(before_question, str):
                try:
                    before_question = json.loads(before_question)
                except Exception:
                    before_question = {}
            if isinstance(after_question, str):
                try:
                    after_question = json.loads(after_question)
                except Exception:
                    after_question = {}

            event_like = {
                "topic": str(question.get("topic", "") or ""),
                "official_reference": reference,
                "role": str(question.get("role", "") or ""),
                "assigned_pattern": str(
                    row.get("assigned_pattern", "") or ""
                ),
                "action": str(row.get("action", "") or ""),
                "reason": str(row.get("reason", "") or ""),
                "before_question": (
                    before_question
                    if isinstance(before_question, dict)
                    else {}
                ),
                "after_question": (
                    after_question
                    if isinstance(after_question, dict)
                    else {}
                ),
            }

            candidates.append(
                {
                    "point_id": "",
                    "event_id": str(row.get("event_id", "") or ""),
                    "retrieval_similarity": 0.0,
                    "action": event_like["action"],
                    "reason": event_like["reason"],
                    "memory_assigned_pattern": event_like[
                        "assigned_pattern"
                    ],
                    "issue_tags": [],
                    "memory_text": _feedback_memory_text(event_like),
                }
            )

        return _rank_hitl_memory_candidates(
            question=question,
            candidates=candidates,
            current_action=current_action,
            review_item=review_item,
            human_reason=human_reason,
        )

    except Exception as exc:
        return [], [], (
            "PostgreSQL fallback failed: "
            f"{type(exc).__name__}: {exc}"
        )


def _retrieve_hitl_memory_hints(
    question: dict[str, Any],
    *,
    current_action: str = "",
    review_item: dict[str, Any] | None = None,
    human_reason: str = "",
) -> list[str]:
    """
    Retrieve same-reference memories, then select only correction-compatible
    memories using local MiniLM + deterministic action compatibility.

    Current human feedback remains highest priority. Retrieved memory is only
    secondary guidance.
    """
    reference = str(
        question.get("official_reference", "") or ""
    ).strip()

    if not reference:
        _record_hitl_memory_lookup(
            question=question,
            source="none",
            hits=[],
            status="SKIPPED",
            detail=(
                "No official_reference available for safe memory filtering."
            ),
        )
        return []

    client, client_status = _qdrant_client_for_hitl_memory()

    if client is None:
        hints, audit_hits, detail = (
            _retrieve_hitl_memory_hints_from_postgres(
                question,
                current_action=current_action,
                review_item=review_item,
                human_reason=human_reason,
            )
        )
        _record_hitl_memory_lookup(
            question=question,
            source="postgres_fallback",
            hits=audit_hits,
            status="OK" if hints else "NO_MATCH",
            detail=(
                str(client_status or "Qdrant unavailable. ")
                + " "
                + detail
            ).strip(),
        )
        return hints

    query_text = "\n".join(
        [
            "Topic: " + str(question.get("topic", "") or "").strip(),
            "AQA reference: " + reference,
            "Assessment pattern: "
            + str(question.get("assessment_pattern", "") or "").strip(),
            "Question: "
            + str(question.get("question_text", "") or "").strip(),
        ]
    )

    vectors, embedding_backend = _minilm_embeddings([query_text])

    if vectors is None or len(vectors) != 1:
        hints, audit_hits, detail = (
            _retrieve_hitl_memory_hints_from_postgres(
                question,
                current_action=current_action,
                review_item=review_item,
                human_reason=human_reason,
            )
        )
        _record_hitl_memory_lookup(
            question=question,
            source="postgres_fallback",
            hits=audit_hits,
            status="OK" if hints else "NO_MATCH",
            detail=(
                "MiniLM query embedding unavailable; PostgreSQL fallback. "
                + detail
            ),
        )
        return hints

    try:
        from qdrant_client.models import (
            Filter,
            FieldCondition,
            MatchValue,
        )

        query_filter = Filter(
            must=[
                FieldCondition(
                    key="official_reference",
                    match=MatchValue(value=reference),
                )
            ]
        )

        # IMPORTANT:
        # Do not apply the old 0.78 threshold at Qdrant-candidate retrieval.
        # Same-reference candidates are cheap and safe to retrieve broadly;
        # the stricter issue/action compatibility gate below decides what is
        # actually allowed into the LLM prompt.
        if hasattr(client, "query_points"):
            response = client.query_points(
                collection_name=HITL_MEMORY_COLLECTION,
                query=np.asarray(
                    vectors[0],
                    dtype=float,
                ).tolist(),
                query_filter=query_filter,
                limit=int(HITL_MEMORY_TOP_K),
                with_payload=True,
            )
            points = list(
                getattr(response, "points", []) or []
            )
        else:
            points = list(
                client.search(
                    collection_name=HITL_MEMORY_COLLECTION,
                    query_vector=np.asarray(
                        vectors[0],
                        dtype=float,
                    ).tolist(),
                    query_filter=query_filter,
                    limit=int(HITL_MEMORY_TOP_K),
                    with_payload=True,
                )
            )

        candidates: list[dict[str, Any]] = []

        for point in points:
            payload = getattr(point, "payload", {}) or {}
            reason = str(
                payload.get("reason", "") or ""
            ).strip()
            if not reason:
                continue

            candidates.append(
                {
                    "point_id": str(
                        getattr(point, "id", "") or ""
                    ),
                    "event_id": str(
                        payload.get("event_id", "") or ""
                    ),
                    "retrieval_similarity": float(
                        getattr(point, "score", 0.0) or 0.0
                    ),
                    "action": str(
                        payload.get("action", "") or ""
                    ).strip(),
                    "reason": reason,
                    "memory_assigned_pattern": str(
                        payload.get("assigned_pattern", "") or ""
                    ),
                    "issue_tags": [
                        str(value).strip()
                        for value in (
                            payload.get("issue_tags") or []
                        )
                        if str(value).strip()
                    ],
                    "hard_failures": [
                        str(value).strip()
                        for value in (
                            payload.get("hard_failures") or []
                        )
                        if str(value).strip()
                    ],
                    "memory_text": str(
                        payload.get("memory_text")
                        or reason
                    ).strip(),
                }
            )

        hints, audit_hits, rerank_detail = (
            _rank_hitl_memory_candidates(
                question=question,
                candidates=candidates,
                current_action=current_action,
                review_item=review_item,
                human_reason=human_reason,
            )
        )

        _record_hitl_memory_lookup(
            question=question,
            source="qdrant",
            hits=audit_hits,
            status="OK" if hints else "NO_MATCH",
            detail=(
                f"Collection={HITL_MEMORY_COLLECTION}; "
                f"qdrant_embedding={embedding_backend}; "
                + rerank_detail
            ),
        )

        return hints

    except Exception as exc:
        hints, audit_hits, detail = (
            _retrieve_hitl_memory_hints_from_postgres(
                question,
                current_action=current_action,
                review_item=review_item,
                human_reason=human_reason,
            )
        )
        _record_hitl_memory_lookup(
            question=question,
            source="postgres_fallback",
            hits=audit_hits,
            status="OK" if hints else "ERROR",
            detail=(
                "Qdrant lookup failed; PostgreSQL fallback used. "
                f"{type(exc).__name__}: {exc}. "
                + detail
            ),
        )
        return hints


if HUMAN_REVIEW_DECISION not in ALLOWED_HUMAN_DECISIONS:
    raise ValueError(
        "HUMAN_REVIEW_DECISION must be pending, approve, regenerate, or reject."
    )

if HUMAN_REVIEW_DECISION != "pending" and not HUMAN_REVIEW_REASON:
    raise ValueError(
        "A written HUMAN_REVIEW_REASON is required for a human decision."
    )

question_review_actions, question_review_action_errors = _load_question_review_actions()

# ----------------------------------------------------------------
# Human structural overrides use an EFFECTIVE blueprint.
#
# The original deterministic blueprint is preserved forever for audit.
# Reviewer-approved structural changes (currently pattern / visual removal)
# are persisted separately and are the blueprint used by subsequent
# structural validation for the same generation request.
# ----------------------------------------------------------------
EFFECTIVE_BLUEPRINT_PATH = OUTPUT_DIR / "effective_generation_blueprint.json"

generation_blueprint_original = json.loads(
    json.dumps(
        generation_request.get("blueprint", generation_blueprint),
        ensure_ascii=False,
        default=str,
    )
)

generation_blueprint_effective = json.loads(
    json.dumps(
        generation_blueprint_original,
        ensure_ascii=False,
        default=str,
    )
)

if EFFECTIVE_BLUEPRINT_PATH.is_file():
    try:
        persisted_effective = json.loads(
            EFFECTIVE_BLUEPRINT_PATH.read_text(encoding="utf-8")
        )
    except Exception:
        persisted_effective = {}
    if (
        isinstance(persisted_effective, dict)
        and str(
            persisted_effective.get("generation_request_fingerprint", "")
            or ""
        ).strip()
        == str(generation_request_fingerprint).strip()
        and isinstance(
            persisted_effective.get("effective_blueprint"),
            list,
        )
    ):
        generation_blueprint_effective = json.loads(
            json.dumps(
                persisted_effective["effective_blueprint"],
                ensure_ascii=False,
                default=str,
            )
        )

# Structural validation always uses the effective blueprint.
generation_request["blueprint"] = generation_blueprint_effective
generation_blueprint = generation_blueprint_effective


def _persist_effective_generation_blueprint() -> None:
    EFFECTIVE_BLUEPRINT_PATH.write_text(
        json.dumps(
            {
                "schema_version":
                    "agent2-effective-generation-blueprint-v1.0.0",
                "pipeline_version":
                    NOTEBOOK06_PIPELINE_VERSION,
                "generation_request_fingerprint":
                    generation_request_fingerprint,
                "original_blueprint":
                    generation_blueprint_original,
                "effective_blueprint":
                    generation_blueprint_effective,
                "updated_at_utc":
                    datetime.now(timezone.utc).isoformat(),
            },
            indent=2,
            ensure_ascii=False,
            default=str,
        ),
        encoding="utf-8",
    )


def _update_effective_blueprint_slot(
    plan_index: int,
    **updates: Any,
) -> None:
    for blueprint_item in generation_blueprint_effective:
        if (
            isinstance(blueprint_item, dict)
            and safe_int(blueprint_item.get("plan_index"))
            == safe_int(plan_index)
        ):
            blueprint_item.update(updates)
            break
    # generation_request points to this effective list, but assign again
    # explicitly to make the validation contract obvious.
    generation_request["blueprint"] = generation_blueprint_effective
    _persist_effective_generation_blueprint()


question_review_audit: list[dict[str, Any]] = []
question_level_content_changed = False
question_level_regeneration_requested = False
unresolved_question_rejections: list[int] = []
question_action_state: dict[int, str] = {}

# A candidate restored from disk may already contain a reviewer-approved
# structural override from an earlier HITL turn. Cell 34 initially validates
# against the immutable request blueprint, so re-run validation here against
# the persisted effective blueprint before processing the next human action.
if (
    generated_payload
    and generation_blueprint_effective
    != generation_blueprint_original
):
    generation_validation, semantic_quality_validation = (
        validate_render_and_quality(
            generated_payload,
            generation_request,
        )
    )

# Snapshot the CURRENT validator queue before applying this round of human
# actions. This issue context is stored with feedback memory and is also used
# to select compatible past memories for targeted regeneration.
pre_action_question_review_queue = _build_question_review_queue(
    generated_payload,
    semantic_quality_validation,
) if generated_payload else []

pre_action_review_by_plan = {
    safe_int(item.get("plan_index")): item
    for item in pre_action_question_review_queue
    if isinstance(item, dict)
    and safe_int(item.get("plan_index")) > 0
}

# ----------------------------------------------------------------
# Apply deterministic question-level edits first. These add no LLM cost.
# ----------------------------------------------------------------
if generated_payload and question_review_actions:
    questions = [
        item
        for item in generated_payload.get("questions", [])
        if isinstance(item, dict)
    ]

    for raw_action in question_review_actions:
        action_name = str(raw_action.get("action", "") or "").strip().casefold()
        if action_name not in ALLOWED_QUESTION_REVIEW_ACTIONS:
            question_review_action_errors.append(
                "Unsupported question-level HITL action: " + repr(action_name)
            )
            continue

        target = next(
            (question for question in questions if _action_matches_question(raw_action, question)),
            None,
        )
        if target is None:
            question_review_action_errors.append(
                "Question-level HITL action could not resolve its question_id/plan_index."
            )
            continue

        question_id = _question_identifier(target)
        plan_index = safe_int(target.get("plan_index"))
        reason = str(raw_action.get("reason", "") or "").strip()

        if action_name != "approve" and not reason:
            question_review_action_errors.append(
                f"{question_id or plan_index}: action '{action_name}' requires a written reason."
            )
            continue

        before_snapshot = json.loads(
            json.dumps(
                target,
                ensure_ascii=False,
                default=str,
            )
        )
        before_fingerprint = stable_json_fingerprint(before_snapshot)
        audit_item = {
            "question_id": question_id,
            "plan_index": plan_index,
            "topic": str(target.get("topic", "") or "").strip(),
            "official_reference": str(
                target.get("official_reference", "") or ""
            ).strip(),
            "role": str(target.get("role", "") or "").strip(),
            "assigned_pattern": str(
                target.get("assessment_pattern", "") or ""
            ).strip(),
            "action": action_name,
            "reason": reason,
            "issue_tags": list(
                (
                    pre_action_review_by_plan.get(plan_index, {})
                    or {}
                ).get("issue_tags")
                or []
            ),
            "hard_failures": list(
                (
                    pre_action_review_by_plan.get(plan_index, {})
                    or {}
                ).get("hard_failures")
                or []
            ),
            "before_question": before_snapshot,
            "before_question_fingerprint": before_fingerprint,
            "generation_request_fingerprint": generation_request_fingerprint,
        }

        if action_name == "approve":
            question_action_state[plan_index] = "approve"

        elif action_name == "reject":
            question_action_state[plan_index] = "reject"

        elif action_name == "change_pattern":
            new_pattern = normalize_text(
                raw_action.get("assessment_pattern", raw_action.get("new_pattern", ""))
            )
            if new_pattern not in ASSESSMENT_PATTERN_CYCLE:
                question_review_action_errors.append(
                    f"{question_id or plan_index}: invalid assessment pattern {new_pattern!r}."
                )
                continue

            # Explicit human structural correction:
            # - candidate is relabelled;
            # - ORIGINAL blueprint remains immutable for audit;
            # - EFFECTIVE blueprint is persisted and becomes the validator truth
            #   for this same generation request on all future reruns.
            target["assessment_pattern"] = new_pattern
            _update_effective_blueprint_slot(
                plan_index,
                assessment_pattern=new_pattern,
            )

            question_action_state[plan_index] = "change_pattern"
            question_level_content_changed = True

        elif action_name == "remove_visual":
            target["visual_requirement"] = "none"
            target["visual_spec"] = {}
            _update_effective_blueprint_slot(
                plan_index,
                visual_requirement="none",
            )
            for key in (
                "visual_asset_path",
                "visual_asset_sha256",
                "visual_render_error",
                "rendered_visual_path",
            ):
                target.pop(key, None)
            question_action_state[plan_index] = "remove_visual"
            question_level_content_changed = True

        elif action_name == "edit_question":
            new_text = str(
                raw_action.get(
                    "question_text",
                    (raw_action.get("updates") or {}).get("question_text", "")
                    if isinstance(raw_action.get("updates"), dict)
                    else "",
                )
                or ""
            ).strip()
            if not new_text:
                question_review_action_errors.append(
                    f"{question_id or plan_index}: edit_question requires non-empty question_text."
                )
                continue
            target["question_text"] = new_text
            question_action_state[plan_index] = "edit_question"
            question_level_content_changed = True

        elif action_name == "edit_marking_guidance":
            guidance = raw_action.get(
                "marking_guidance",
                (raw_action.get("updates") or {}).get("marking_guidance")
                if isinstance(raw_action.get("updates"), dict)
                else None,
            )
            if not isinstance(guidance, list) or not guidance:
                question_review_action_errors.append(
                    f"{question_id or plan_index}: edit_marking_guidance requires a non-empty list."
                )
                continue
            if not all(
                isinstance(item, dict)
                and safe_int(item.get("marks")) > 0
                and str(item.get("criterion", "") or "").strip()
                for item in guidance
            ):
                question_review_action_errors.append(
                    f"{question_id or plan_index}: every marking-guidance item must contain positive marks and criterion text."
                )
                continue
            target["marking_guidance"] = guidance
            question_action_state[plan_index] = "edit_marking_guidance"
            question_level_content_changed = True

        elif action_name == "regenerate":
            question_action_state[plan_index] = "regenerate"
            question_level_regeneration_requested = True

        after_snapshot = json.loads(
            json.dumps(
                target,
                ensure_ascii=False,
                default=str,
            )
        )
        audit_item["after_question"] = after_snapshot
        audit_item["after_question_fingerprint"] = stable_json_fingerprint(
            after_snapshot
        )
        audit_item["content_changed"] = bool(
            audit_item["after_question_fingerprint"] != before_fingerprint
        )
        question_review_audit.append(audit_item)

    generated_payload["questions"] = questions

    # Re-run all local checks after deterministic human edits/removals/relabels.
    if question_level_content_changed:
        generation_validation, semantic_quality_validation = validate_render_and_quality(
            generated_payload,
            generation_request,
        )

# ----------------------------------------------------------------
# Human-targeted regeneration: one small repair call for all selected questions.
# ----------------------------------------------------------------
human_targeted_regeneration_attempts_used = sum(
    1
    for item in generation_attempt_log
    if str(item.get("attempt_type", "") or "")
    == "human_targeted_question_regeneration"
)

if generated_payload and question_level_regeneration_requested and not question_review_action_errors:
    targeted_indexes = sorted(
        plan_index
        for plan_index, state in question_action_state.items()
        if state == "regenerate" and plan_index > 0
    )

    if targeted_indexes:
        if (
            human_targeted_regeneration_attempts_used
            >= MAX_HUMAN_TARGETED_REGENERATION_ATTEMPTS
        ):
            question_review_action_errors.append(
                "Question-level regeneration was requested but the separate "
                "human-targeted regeneration budget is exhausted."
            )
        else:
            # Human-targeted repair has its own budget and must not inflate the
            # automatic/whole-quiz regeneration counter.
            human_targeted_regeneration_attempts_used += 1

            targeted_feedback: list[str] = []

            for action in question_review_actions:
                if (
                    str(action.get("action", "") or "").strip().casefold()
                    != "regenerate"
                ):
                    continue

                target_question = next(
                    (
                        q
                        for q in generated_payload.get("questions", [])
                        if isinstance(q, dict)
                        and _action_matches_question(action, q)
                    ),
                    None,
                )

                if not isinstance(target_question, dict):
                    continue

                plan_index_value = safe_int(
                    target_question.get("plan_index")
                )

                human_reason = str(
                    action.get("reason", "")
                    or ""
                ).strip()

                targeted_feedback.append(
                    "CURRENT HUMAN FEEDBACK — highest priority: "
                    f"regenerate plan_index {plan_index_value}. "
                    f"Reason: {human_reason}"
                )

                # Previous memory is secondary guidance only. It never overrides
                # the current human reason, blueprint, or approved AQA scope.
                #
                # Memory selection is issue/action-aware: the current validator
                # findings are used to rerank a broader same-reference Qdrant
                # candidate pool so a semantically similar but correction-type-
                # incompatible memory is not injected into the prompt.
                current_review_item = pre_action_review_by_plan.get(
                    plan_index_value,
                    {},
                )
                for hint in _retrieve_hitl_memory_hints(
                    target_question,
                    current_action="regenerate",
                    review_item=current_review_item,
                    human_reason=human_reason,
                ):
                    targeted_feedback.append(
                        "PAST HUMAN FEEDBACK MEMORY — secondary guidance "
                        f"for plan_index {plan_index_value}: "
                        + hint
                    )

            # Snapshot deterministic structure BEFORE the LLM repair call.
            # A normal "regenerate" action repairs question wording / marking
            # quality; it is NOT permission for the model to change topic, role,
            # marks, paper, pattern, coverage, or visual requirement.
            locked_structure_by_plan: dict[int, dict[str, Any]] = {}
            structural_lock_fields = (
                "plan_index",
                "generated_question_id",
                "topic",
                "official_reference",
                "paper_code",
                "paper_label",
                "assessment_pattern",
                "role",
                "marks",
                "covered_topics",
                "visual_requirement",
            )

            for current_question in generated_payload.get("questions", []):
                if not isinstance(current_question, dict):
                    continue
                current_plan = safe_int(current_question.get("plan_index"))
                if current_plan not in targeted_indexes:
                    continue
                locked_structure_by_plan[current_plan] = {
                    field: json.loads(
                        json.dumps(
                            current_question.get(field),
                            ensure_ascii=False,
                            default=str,
                        )
                    )
                    for field in structural_lock_fields
                    if field in current_question
                }

            try:
                regenerated = regenerate_only_failed_questions(
                    request_payload=generation_request,
                    current_payload=generated_payload,
                    failed_plan_indexes=targeted_indexes,
                    validation_feedback=targeted_feedback,
                )
            except ModelPreflightBlockedError as exc:
                question_review_action_errors.append(str(exc))
                print(
                    "Question-level human regeneration blocked locally; "
                    "no provider call was sent."
                )
            else:
                # Re-apply deterministic structural metadata after the LLM call.
                # Only explicit HITL structural actions (e.g. change_pattern)
                # may modify the effective blueprint.
                for regenerated_question in regenerated.get("questions", []):
                    if not isinstance(regenerated_question, dict):
                        continue
                    regenerated_plan = safe_int(
                        regenerated_question.get("plan_index")
                    )
                    locked = locked_structure_by_plan.get(regenerated_plan)
                    if not isinstance(locked, dict):
                        continue
                    for field, value in locked.items():
                        regenerated_question[field] = json.loads(
                            json.dumps(
                                value,
                                ensure_ascii=False,
                                default=str,
                            )
                        )

                generated_payload = regenerated
                generation_validation, semantic_quality_validation = (
                    validate_render_and_quality(
                        generated_payload,
                        generation_request,
                    )
                )
                question_level_content_changed = True

                # Update the audit snapshots so PostgreSQL/Qdrant receive the
                # actual regenerated question rather than the pre-regeneration
                # copy.
                for audit_item in question_review_audit:
                    if str(
                        audit_item.get("action", "")
                        or ""
                    ).strip().casefold() != "regenerate":
                        continue

                    plan_index_value = safe_int(
                        audit_item.get("plan_index")
                    )

                    regenerated_question = next(
                        (
                            q
                            for q in generated_payload.get("questions", [])
                            if isinstance(q, dict)
                            and safe_int(q.get("plan_index"))
                            == plan_index_value
                        ),
                        None,
                    )

                    if not isinstance(regenerated_question, dict):
                        continue

                    after_snapshot = json.loads(
                        json.dumps(
                            regenerated_question,
                            ensure_ascii=False,
                            default=str,
                        )
                    )
                    audit_item["after_question"] = after_snapshot
                    audit_item["after_question_fingerprint"] = (
                        stable_json_fingerprint(after_snapshot)
                    )
                    audit_item["content_changed"] = bool(
                        audit_item["after_question_fingerprint"]
                        != audit_item.get("before_question_fingerprint")
                    )

                generation_attempt_log.append(
                    {
                        "attempt_type": "human_targeted_question_regeneration",
                        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                        "targeted_plan_indexes": targeted_indexes,
                        "payload_fingerprint": payload_fingerprint(
                            generated_payload
                        ),
                        "structural_valid": generation_validation.get("valid"),
                        "semantic_status": semantic_quality_validation.get(
                            "status"
                        ),
                        "memory_guidance_items": len(
                            [
                                item
                                for item in targeted_feedback
                                if item.startswith(
                                    "PAST HUMAN FEEDBACK MEMORY"
                                )
                            ]
                        ),
                        "memory_lookup_audit": [
                            item for item in HITL_MEMORY_LOOKUP_AUDIT
                            if safe_int(item.get("plan_index")) in targeted_indexes
                        ],
                    }
                )

# Last action wins for rejection state. A regenerated/edited question is no
# longer an unresolved rejection; a final explicit reject blocks release.
unresolved_question_rejections = sorted(
    plan_index
    for plan_index, state in question_action_state.items()
    if state == "reject"
)

if generated_payload and question_level_content_changed:
    save_candidate_artifacts(
        generated_payload,
        generation_validation,
        semantic_quality_validation,
        regeneration_attempts_used,
        generation_attempt_log,
    )

current_payload_fingerprint = payload_fingerprint(generated_payload)
semantic_status = str(
    semantic_quality_validation.get("status", "NOT_RUN") or "NOT_RUN"
).strip().upper()
structural_valid = bool(generation_validation.get("valid", False))

# A question-level edit/regeneration creates a NEW candidate. Even if the same
# request also carried a whole-quiz approve, approval is intentionally reset so
# the human sees the changed quiz before release.
approval_reset_due_to_question_changes = bool(question_level_content_changed)

if not generated_payload:
    human_review_state = "NO_GENERATED_CANDIDATE"
elif question_review_action_errors:
    human_review_state = "BLOCKED_HITL_ACTION_ERROR"
elif unresolved_question_rejections:
    human_review_state = "BLOCKED_HUMAN_QUESTION_REJECTION"
elif not structural_valid:
    human_review_state = "BLOCKED_STRUCTURAL_VALIDATION"
elif semantic_status == "FAIL":
    # Repairable question-level quality failures escalate to HITL instead of
    # terminating the workflow. Structural/scope failures still block above.
    human_review_state = "AWAITING_QUESTION_LEVEL_REVIEW"
elif approval_reset_due_to_question_changes:
    human_review_state = "AWAITING_HUMAN_REVIEW"
elif HUMAN_REVIEW_DECISION == "pending":
    human_review_state = "AWAITING_HUMAN_REVIEW"
elif HUMAN_REVIEW_DECISION == "approve":
    human_review_state = "HUMAN_APPROVED"
elif HUMAN_REVIEW_DECISION == "reject":
    human_review_state = "HUMAN_REJECTED"
else:
    # Backward-compatible whole-quiz regeneration remains available. Granular
    # question regeneration above is preferred because it is cheaper.
    if regeneration_attempts_used >= MAX_REGENERATION_ATTEMPTS:
        human_review_state = "REGENERATION_BUDGET_EXHAUSTED"
    else:
        regeneration_attempts_used += 1
        feedback = list(semantic_quality_validation.get("reasons", []))
        feedback.append("Human reviewer feedback: " + HUMAN_REVIEW_REASON)

        try:
            regenerated = generate_with_selected_model(
                generation_request,
                validation_feedback=feedback,
            )
        except ModelPreflightBlockedError as exc:
            print()
            print("HUMAN-REQUESTED REGENERATION BLOCKED LOCALLY")
            print(str(exc))
            print("No provider generation API call was sent.")
            print()
            regenerated = {}
            regenerated_structural = {
                "valid": False,
                "errors": [str(exc)],
                "warnings": [],
            }
            regenerated_semantic = {
                "status": "NOT_RUN",
                "release_eligible": False,
                "reasons": [str(exc)],
                "warnings": [],
            }
        else:
            regenerated_structural, regenerated_semantic = validate_render_and_quality(
                regenerated,
                generation_request,
            )

        generated_payload = regenerated
        generation_validation = regenerated_structural
        semantic_quality_validation = regenerated_semantic
        current_payload_fingerprint = payload_fingerprint(generated_payload)
        semantic_status = str(
            semantic_quality_validation.get("status", "NOT_RUN") or "NOT_RUN"
        ).strip().upper()
        structural_valid = bool(generation_validation.get("valid", False))

        generation_attempt_log.append(
            {
                "attempt_type": "human_requested_regeneration",
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                "human_reason": HUMAN_REVIEW_REASON,
                "payload_fingerprint": current_payload_fingerprint,
                "structural_valid": structural_valid,
                "semantic_status": semantic_status,
            }
        )

        save_candidate_artifacts(
            generated_payload,
            generation_validation,
            semantic_quality_validation,
            regeneration_attempts_used,
            generation_attempt_log,
        )

        if not structural_valid:
            human_review_state = "REGENERATION_FAILED_STRUCTURAL"
        elif semantic_status == "FAIL":
            human_review_state = "AWAITING_QUESTION_LEVEL_REVIEW"
        else:
            human_review_state = "AWAITING_HUMAN_REVIEW"

# Rebuild the queue from the FINAL candidate/validation state for this run.
question_review_queue = _build_question_review_queue(
    generated_payload,
    semantic_quality_validation,
)

HUMAN_REVIEW_QUEUE_PATH.write_text(
    json.dumps(
        {
            "schema_version": "agent2-question-review-queue-v2.0.0",
            "pipeline_version": NOTEBOOK06_PIPELINE_VERSION,
            "generation_request_fingerprint": generation_request_fingerprint,
            "payload_fingerprint": current_payload_fingerprint,
            "human_review_state": human_review_state,
            "questions": question_review_queue,
            "updated_at_utc": datetime.now(timezone.utc).isoformat(),
        },
        indent=2,
        ensure_ascii=False,
        default=str,
    ),
    encoding="utf-8",
)

feedback_events = [
    {
        **item,
        "pipeline_version": NOTEBOOK06_PIPELINE_VERSION,
        "final_payload_fingerprint": current_payload_fingerprint,
    }
    for item in question_review_audit
]

enriched_feedback_events = _append_human_feedback_events(
    feedback_events
)

feedback_db_persistence = _persist_human_feedback_to_postgres(
    enriched_feedback_events
)

feedback_qdrant_memory = _promote_human_feedback_to_qdrant(
    enriched_feedback_events
)

automatic_corrective_regeneration_attempts_used = sum(
    1
    for item in generation_attempt_log
    if str(item.get("attempt_type", "") or "")
    == "automatic_corrective_regeneration"
)

whole_quiz_human_regeneration_attempts_used = sum(
    1
    for item in generation_attempt_log
    if str(item.get("attempt_type", "") or "")
    == "human_requested_regeneration"
)

generated_human_review = {
    "schema_version": "agent2-generated-quality-human-review-v3.2.0",
    "pipeline_version": NOTEBOOK06_PIPELINE_VERSION,
    "generation_request_fingerprint": generation_request_fingerprint,
    "payload_fingerprint": current_payload_fingerprint,
    "decision": HUMAN_REVIEW_DECISION,
    "reason": HUMAN_REVIEW_REASON,
    "structural_valid": structural_valid,
    "semantic_status": semantic_status,
    "state": human_review_state,
    # Backward-compatible combined/legacy counter:
    "regeneration_attempts_used": regeneration_attempts_used,
    "max_regeneration_attempts": MAX_REGENERATION_ATTEMPTS,
    # Clear separated counters for reporting / UI:
    "automatic_corrective_regeneration_attempts_used":
        automatic_corrective_regeneration_attempts_used,
    "whole_quiz_human_regeneration_attempts_used":
        whole_quiz_human_regeneration_attempts_used,
    "human_targeted_regeneration_attempts_used":
        human_targeted_regeneration_attempts_used,
    "max_human_targeted_regeneration_attempts":
        MAX_HUMAN_TARGETED_REGENERATION_ATTEMPTS,
    "question_level_actions": question_review_actions,
    "question_level_action_audit": question_review_audit,
    "question_level_action_errors": question_review_action_errors,
    "question_level_content_changed": question_level_content_changed,
    "approval_reset_due_to_question_changes": approval_reset_due_to_question_changes,
    "unresolved_question_rejections": unresolved_question_rejections,
    "question_review_queue_path": str(HUMAN_REVIEW_QUEUE_PATH),
    "human_feedback_log_path": str(HUMAN_REVIEW_FEEDBACK_LOG_PATH),
    "feedback_db_persistence": feedback_db_persistence,
    "feedback_qdrant_memory": feedback_qdrant_memory,
    "feedback_memory_lookup_audit": HITL_MEMORY_LOOKUP_AUDIT,
    "generation_blueprint_original": generation_blueprint_original,
    "generation_blueprint_effective": generation_blueprint_effective,
    "effective_blueprint_path": str(EFFECTIVE_BLUEPRINT_PATH),
    "question_review_queue": question_review_queue,
    "special_instructions": str(request.get("special_instructions", "") or "").strip(),
    "same_call_instruction_interpretation": (
        generated_payload.get("instruction_interpretation", {})
        if isinstance(generated_payload, dict)
        else {}
    ),
    "same_call_special_instruction_compliance": (
        generated_payload.get("special_instruction_compliance", [])
        if isinstance(generated_payload, dict)
        else []
    ),
    "qualitative_special_instruction_confirmation_required": bool(
        str(request.get("special_instructions", "") or "").strip()
    ),
    "updated_at_utc": datetime.now(timezone.utc).isoformat(),
}

human_review_path.write_text(
    json.dumps(
        generated_human_review,
        indent=2,
        ensure_ascii=False,
        default=str,
    ),
    encoding="utf-8",
)

if question_review_queue:
    display(
        pd.DataFrame(
            [
                {
                    "question_id": item.get("question_id"),
                    "plan_index": item.get("plan_index"),
                    "topic": item.get("topic"),
                    "role": item.get("role"),
                    "assigned_pattern": item.get("assigned_pattern"),
                    "suggested_pattern": item.get("suggested_pattern"),
                    "pattern_alignment": item.get("pattern_alignment_status"),
                    "mark_scheme_alignment": item.get("command_marking_alignment_status"),
                    "topic_grounding": item.get("topic_grounding_status"),
                    "priority": item.get("priority"),
                    "recommended_action": item.get("recommended_action"),
                }
                for item in question_review_queue
            ]
        )
    )

# Keep the compact run summary used by the existing frontend/log flow.
display(
    pd.DataFrame(
        [
            {
                "semantic_status": semantic_status,
                "human_review_state": human_review_state,
                "regeneration_attempts_used": regeneration_attempts_used,
                "question_review_actions": len(question_review_actions),
                "question_review_action_errors": len(question_review_action_errors),
                "review_queue_items": len(question_review_queue),
                "special_instruction_status": str(
                    generated_payload.get("instruction_interpretation", {}).get("status", "none")
                    if isinstance(generated_payload, dict)
                    and isinstance(generated_payload.get("instruction_interpretation", {}), dict)
                    else "none"
                ),
            }
        ]
    )
)


## 13. Final acceptance: OFFICIAL_ONLY / HYBRID / GENERATED_ONLY

## Candidate-visible HITL contract — v2.10

A generated candidate and an accepted/released quiz are now represented
separately.

When the selected model successfully generates the requested quiz and deterministic Python
validation allows it to reach HITL:

```text
generated candidate exists
        ↓
manifest assessment_type = generated_candidate
        ↓
actual_total_marks / actual_question_count show the candidate for UI display
        ↓
candidate_questions contains the reviewable questions
        ↓
accepted_total_marks / accepted_question_count remain 0
        ↓
questions remains empty
        ↓
release_ready = False
```

After human approval:

```text
generated_quality_accepted = True
        ↓
assessment_type = generated_only (or hybrid)
        ↓
questions contains the accepted quiz
        ↓
accepted metrics equal the quiz metrics
        ↓
release_ready can become True if all other constraints pass
```

This prevents the Streamlit HITL screen from incorrectly displaying
`EMPTY / 0 marks / 0 questions` when a real generated candidate exists, while
still ensuring that an unapproved candidate is never treated as released.


In [ ]:
generated_candidate_questions = (
    generated_payload.get(
        "questions",
        [],
    )
    if isinstance(
        generated_payload,
        dict,
    )
    else []
)

if not isinstance(
    generated_candidate_questions,
    list,
):
    generated_candidate_questions = []

generated_candidate_questions = [
    question
    for question in generated_candidate_questions
    if isinstance(
        question,
        dict,
    )
]


generated_human_approved = bool(
    structural_valid
    and semantic_status
    in {
        "PASS",
        "REVIEW",
    }
    and human_review_state
    == "HUMAN_APPROVED"
)

# Deterministic PASS means the system checks passed; it does NOT bypass HITL.
generated_semantic_ready = bool(
    semantic_status
    in {
        "PASS",
        "REVIEW",
    }
)

generated_quality_accepted = bool(
    generated_human_approved
)

generated_questions = (
    generated_candidate_questions
    if generated_quality_accepted
    else []
)


if (
    official_questions
    and generated_questions
):
    assessment_type = (
        "hybrid"
    )

elif (
    official_questions
    and generated_candidate_questions
):
    # Candidate exists but is not yet accepted/released.
    assessment_type = (
        "hybrid_candidate"
    )

elif official_questions:
    assessment_type = (
        "official_only"
    )

elif generated_questions:
    assessment_type = (
        "generated_only"
    )

elif generated_candidate_questions:
    # Complete quiz was generated successfully, but mandatory HITL
    # has not accepted it yet.
    assessment_type = (
        "generated_candidate"
    )

else:
    assessment_type = (
        "empty"
    )


def enrich_question_paper_metadata(
    question: dict[str, Any],
) -> dict[str, Any]:
    enriched = dict(
        question
    )

    raw_code = (
        enriched.get(
            "paper_code"
        )
        or enriched.get(
            "paper"
        )
    )

    paper_code = None

    if raw_code is not None:
        try:
            paper_code = normalize_paper_code(
                raw_code
            )
        except ValueError:
            paper_code = None

    if paper_code is None:
        reference = (
            question_reference(
                enriched
            )
            if enriched.get(
                "source_type"
            )
            == "official_aqa"
            else str(
                enriched.get(
                    "official_reference",
                    "",
                )
                or ""
            ).strip()
        )

        paper_code = paper_code_for_reference(
            reference
        )

    paper_label = paper_label_for_code(
        paper_code
    )

    if paper_code:
        enriched[
            "paper_code"
        ] = paper_code

    if paper_label:
        enriched[
            "paper_label"
        ] = paper_label

    return enriched


candidate_generated_questions_enriched = [
    enrich_question_paper_metadata(
        question
    )
    for question in generated_candidate_questions
    if isinstance(
        question,
        dict,
    )
]


# Accepted/released questions only.
final_questions = [
    enrich_question_paper_metadata(
        question
    )
    for question in (
        official_questions
        + generated_questions
    )
    if isinstance(
        question,
        dict,
    )
]


actual_total_marks = sum(
    question_marks(
        question
    )
    if question.get(
        "source_type"
    )
    == "official_aqa"
    else safe_int(
        question.get(
            "marks"
        )
    )
    for question in final_questions
)

actual_question_count = len(
    final_questions
)


# ------------------------------------------------------------
# Candidate metrics are intentionally separate from accepted metrics.
#
# During mandatory HITL the accepted quiz is still empty, but the candidate
# really exists and must be visible to Streamlit so that a human can review it.
# These display values NEVER make the quiz release-ready.
# ------------------------------------------------------------
candidate_total_marks = sum(
    safe_int(
        question.get(
            "marks"
        )
    )
    for question in candidate_generated_questions_enriched
)

candidate_question_count = len(
    candidate_generated_questions_enriched
)


# ------------------------------------------------------------
# Fill-shortfall UI metrics.
#
# Keep three concepts explicit so Streamlit never confuses the AI-only
# candidate with the whole hybrid quiz:
#   1) official retrieval already selected by Notebook 05
#   2) AI-generated missing coverage
#   3) combined quiz after the AI coverage is approved
# ------------------------------------------------------------
official_total_marks = sum(
    question_marks(question)
    for question in official_questions
    if isinstance(question, dict)
)
official_question_count = len(
    [
        question
        for question in official_questions
        if isinstance(question, dict)
    ]
)

accepted_generated_total_marks = sum(
    safe_int(question.get("marks"))
    for question in generated_questions
    if isinstance(question, dict)
)
accepted_generated_question_count = len(
    [
        question
        for question in generated_questions
        if isinstance(question, dict)
    ]
)

# While HITL is pending, "combined after approval" is the projected hybrid
# using the current reviewable candidate. After approval it uses accepted AI
# questions instead.
if generated_quality_accepted:
    shortfall_display_marks = accepted_generated_total_marks
    shortfall_display_questions = accepted_generated_question_count
else:
    shortfall_display_marks = candidate_total_marks
    shortfall_display_questions = candidate_question_count

combined_after_approval_total_marks = (
    official_total_marks
    + shortfall_display_marks
)
combined_after_approval_question_count = (
    official_question_count
    + shortfall_display_questions
)


if generated_quality_accepted:
    display_questions = list(
        final_questions
    )

elif generated_candidate_questions:
    # For complete_quiz this is the generated candidate.
    # For fill_shortfall this is official content + pending generated coverage.
    display_questions = [
        enrich_question_paper_metadata(
            question
        )
        for question in official_questions
        if isinstance(
            question,
            dict,
        )
    ] + list(
        candidate_generated_questions_enriched
    )

else:
    display_questions = list(
        final_questions
    )


display_total_marks = sum(
    (
        question_marks(
            question
        )
        if question.get(
            "source_type"
        )
        == "official_aqa"
        else safe_int(
            question.get(
                "marks"
            )
        )
    )
    for question in display_questions
)

display_question_count = len(
    display_questions
)


final_roles = [
    question_role(
        question
    )
    if question.get(
        "source_type"
    )
    == "official_aqa"
    else str(
        question.get(
            "role",
            "",
        )
        or ""
    ).strip().casefold()
    for question in final_questions
]

final_refs: set[
    str
] = set()

for question in final_questions:
    if question.get(
        "source_type"
    ) == "official_aqa":
        reference = question_reference(
            question
        )

        if reference:
            final_refs.add(
                reference
            )

        continue

    anchor_reference = str(
        question.get(
            "official_reference",
            "",
        )
        or ""
    ).strip()

    if anchor_reference:
        final_refs.add(
            anchor_reference
        )

    for covered in safe_list(
        question.get(
            "covered_topics",
            [],
        )
    ):
        if not isinstance(
            covered,
            dict,
        ):
            continue

        reference = str(
            covered.get(
                "official_reference",
                "",
            )
            or ""
        ).strip()

        if reference:
            final_refs.add(
                reference
            )


final_topic_norms: set[
    str
] = set()

for question in final_questions:
    if question.get(
        "source_type"
    ) == "official_aqa":
        topic_norm = normalize_text(
            question_topic(
                question
            )
        )

        if topic_norm:
            final_topic_norms.add(
                topic_norm
            )

        continue

    anchor_topic_norm = normalize_text(
        question.get(
            "topic",
            "",
        )
    )

    if anchor_topic_norm:
        final_topic_norms.add(
            anchor_topic_norm
        )

    for covered in safe_list(
        question.get(
            "covered_topics",
            [],
        )
    ):
        if not isinstance(
            covered,
            dict,
        ):
            continue

        covered_norm = normalize_text(
            covered.get(
                "topic",
                "",
            )
        )

        if covered_norm:
            final_topic_norms.add(
                covered_norm
            )



mark_target_met = bool(
    actual_total_marks
    >= request[
        "target_total_marks"
    ]
)

if request.get(
    "exact_question_count",
    False,
):
    question_count_target_met = bool(
        actual_question_count
        == request[
            "number_of_questions"
        ]
    )
else:
    question_count_target_met = bool(
        actual_question_count
        >= request[
            "number_of_questions"
        ]
    )

special_directives = request.get(
    "special_instruction_directives",
    {},
)

if not isinstance(
    special_directives,
    dict,
):
    special_directives = {}

actual_primary_count = sum(
    1
    for role in final_roles
    if role
    == "primary"
)

actual_supporting_count = sum(
    1
    for role in final_roles
    if role
    == "supporting"
)

exact_primary_target = special_directives.get(
    "exact_primary_questions"
)

exact_supporting_target = special_directives.get(
    "exact_supporting_questions"
)

primary_target_met = bool(
    (
        actual_primary_count
        == safe_int(
            exact_primary_target
        )
    )
    if exact_primary_target is not None
    else (
        actual_primary_count
        >= request[
            "minimum_primary_questions"
        ]
    )
)

supporting_target_met = bool(
    (
        actual_supporting_count
        == safe_int(
            exact_supporting_target
        )
    )
    if exact_supporting_target is not None
    else (
        actual_supporting_count
        >= request[
            "minimum_supporting_questions"
        ]
    )
)

distinct_style_special_target_met = True

for role in special_directives.get(
    "distinct_styles_for_roles",
    [],
):
    role_norm = str(
        role or ""
    ).strip().casefold()

    role_patterns = [
        normalize_text(
            question.get(
                "assessment_pattern",
                "",
            )
        )
        for question in final_questions
        if isinstance(
            question,
            dict,
        )
        and str(
            question.get(
                "role",
                "",
            )
            or ""
        ).strip().casefold()
        == role_norm
    ]

    role_patterns = [
        value
        for value in role_patterns
        if value
    ]

    if len(
        role_patterns
    ) != len(
        set(
            role_patterns
        )
    ):
        distinct_style_special_target_met = False
        break

distinct_reference_target_met = bool(
    len(
        final_refs
    )
    >= request[
        "minimum_distinct_official_references"
    ]
)

if request[
    "cover_all_approved_topics"
]:
    required_topic_norms = {
        item[
            "topic_norm"
        ]
        for item in approved_topics
    }

    coverage_target_met = bool(
        required_topic_norms.issubset(
            final_topic_norms
        )
    )

else:
    coverage_target_met = True


final_constraints_met = bool(
    mark_target_met
    and question_count_target_met
    and primary_target_met
    and supporting_target_met
    and distinct_style_special_target_met
    and distinct_reference_target_met
    and coverage_target_met
)


official_release_failures = [
    {
        "question_id":
            question.get(
                "question_id"
            ),

        "question_number":
            question.get(
                "question_number_postgres",
                question.get(
                    "question_number_retrieval"
                ),
            ),

        "question_text":
            question_text(
                question
            ),

        "visual_render_error":
            question.get(
                "visual_render_error"
            ),
    }
    for question in official_questions
    if question.get(
        "student_release_eligible"
    )
    is False
]


official_release_ready = bool(
    not official_release_failures
)

if QUIZ_MODE == "complete_quiz":
    generated_release_ready = bool(
        generated_quality_accepted
        and generated_questions
    )

else:
    generated_release_ready = bool(
        (
            not generation_required
        )
        or generated_quality_accepted
    )


release_ready = bool(
    final_constraints_met
    and official_release_ready
    and generated_release_ready
)


manifest = {
    "schema_version":
        "agent2-final-quiz-manifest-v2.7.0",

    "pipeline_version":
        NOTEBOOK06_PIPELINE_VERSION,

    "generated_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "quiz_mode":
        QUIZ_MODE,

    "assessment_type":
        assessment_type,

    "release_ready":
        release_ready,

    "generation_model":
        GENERATION_MODEL,

    "generation_provider":
        GENERATION_PROVIDER,

    "selected_model_key":
        SELECTED_MODEL_KEY,

    "generation_strategy":
        QUIZ_GENERATION_STRATEGY,

    "hybrid_batching_version":
        HYBRID_BATCHING_VERSION,

    "hybrid_batching_diagnostics_path":
        (
            str(OUTPUT_DIR / "hybrid_batching_diagnostics.json")
            if (OUTPUT_DIR / "hybrid_batching_diagnostics.json").is_file()
            else None
        ),

    "generation_model_display_name":
        GENERATION_MODEL_DISPLAY_NAME,

    "model_limits": {
        "context_window_tokens":
            ACTIVE_MODEL_CONTEXT_WINDOW_TOKENS,
        "hard_max_output_tokens":
            ACTIVE_MODEL_HARD_MAX_OUTPUT_TOKENS,
        "provider_tpm_limit_tokens":
            ACTIVE_MODEL_PROVIDER_TPM_LIMIT_TOKENS,
        "notebook_generation_output_min_tokens":
            MODEL_MIN_GENERATION_OUTPUT_TOKENS,
        "notebook_generation_output_max_tokens":
            MODEL_MAX_GENERATION_OUTPUT_TOKENS,
    },

    "model_token_usage":
        model_usage_summary(),

    "model_call_usage":
        list(
            MODEL_CALL_USAGE_ROWS
        ),

    "llm_architecture": {
        "instruction_interpreter":
            "same_generation_call",
        "separate_instruction_llm_call":
            False,
        "secondary_llm_reviewer":
            False,
        "normal_target_generation_calls":
            "adaptive_not_fixed",
        "batching_policy":
            "generic_token_complexity_adaptive",
        "quiz_size_mapping_used":
            False,
        "adaptive_transport_split_allowed":
            True,
        "corrective_strategy":
            "targeted_failed_plan_indexes_when_resolvable",
        "local_semantic_checks":
            "MiniLM + lexical + task-family + deterministic Python",
        "final_qualitative_gate":
            "mandatory_hitl",
    },

    "special_instruction_interpretation":
        generated_payload.get(
            "instruction_interpretation",
            {},
        )
        if isinstance(
            generated_payload,
            dict,
        )
        else {},

    "special_instruction_compliance":
        generated_payload.get(
            "special_instruction_compliance",
            [],
        )
        if isinstance(
            generated_payload,
            dict,
        )
        else [],

    "visual_architecture": {
        "phase": "final",
        "schema_version": VISUAL_SCHEMA_VERSION,
        "renderer": VISUAL_RENDERER,
        "phase_1_types": list(VISUAL_PHASE1_TYPES),
        "phase_2_types": list(VISUAL_PHASE2_TYPES),
        "final_phase_types": list(VISUAL_FINAL_TYPES),
        "supported_types": [
            value
            for value in VISUAL_TYPES
            if value != "none"
        ],
        "visual_dir": str(VISUAL_DIR),
        "image_generation_api_used": False,
        "visual_selection_policy": "deterministic_no_fixed_quota",
        "render_validation": generation_validation.get(
            "visual_validation",
            {},
        ),
        "candidate_visual_count": sum(
            1
            for question in candidate_generated_questions_enriched
            if normalize_visual_requirement(
                question.get("visual_requirement", "none")
            ) != "none"
        ),
    },

    "generation_request_fingerprint":
        generation_request_fingerprint,

    "notebook05_used":
        bool(
            QUIZ_MODE
            == "fill_shortfall"
        ),

    "notebook05_run_timestamp":
        notebook05_run_timestamp,

    "target_marks":
        request[
            "target_total_marks"
        ],

    # UI-compatible current content metrics.
    #
    # While HITL is pending these reflect the real generated candidate so the
    # frontend does not incorrectly show 0 marks / 0 questions. They do NOT
    # imply acceptance or release readiness.
    "actual_total_marks":
        display_total_marks,

    "target_question_count":
        request[
            "number_of_questions"
        ],

    "actual_question_count":
        display_question_count,

    # Explicit accepted/released metrics remain separate.
    "accepted_total_marks":
        actual_total_marks,

    "accepted_question_count":
        actual_question_count,

    "candidate_total_marks":
        candidate_total_marks,

    "candidate_question_count":
        candidate_question_count,

    # Explicit hybrid/shortfall metrics for Streamlit. These avoid treating
    # the AI-only candidate as if it were the whole quiz.
    "official_total_marks":
        official_total_marks,

    "official_question_count":
        official_question_count,

    "ai_shortfall_candidate_marks":
        candidate_total_marks,

    "ai_shortfall_candidate_question_count":
        candidate_question_count,

    "ai_shortfall_accepted_marks":
        accepted_generated_total_marks,

    "ai_shortfall_accepted_question_count":
        accepted_generated_question_count,

    "combined_after_approval_total_marks":
        combined_after_approval_total_marks,

    "combined_after_approval_question_count":
        combined_after_approval_question_count,

    "ui_shortfall_summary": {
        "official_retrieval": {
            "marks": official_total_marks,
            "questions": official_question_count,
        },
        "ai_shortfall": {
            "marks": shortfall_display_marks,
            "questions": shortfall_display_questions,
            "state": (
                "accepted"
                if generated_quality_accepted
                else "candidate"
                if candidate_question_count
                else "none"
            ),
        },
        "combined_after_approval": {
            "marks": combined_after_approval_total_marks,
            "questions": combined_after_approval_question_count,
        },
    },

    "candidate_available":
        bool(
            generated_candidate_questions
        ),

    "candidate_awaiting_human_review":
        bool(
            generated_candidate_questions
            and not generated_quality_accepted
            and human_review_state
            == "AWAITING_HUMAN_REVIEW"
        ),

    "official_question_count":
        len(
            official_questions
        ),

    "generated_candidate_question_count":
        len(
            generated_candidate_questions
        ),

    "generated_question_count":
        len(
            generated_questions
        ),

    "generation_required":
        generation_required,

    "paper_routing_blocked":
        PAPER_ROUTING_BLOCKED,

    "paper_routing_preflight":
        paper_routing_preflight_result,

    "paper_routing_message":
        PAPER_ROUTING_BLOCK_REASON,

    "generation_status":
        generation_status,

    "generated_structural_ready":
        structural_valid,

    "generated_semantic_status":
        semantic_status,

    # Backward-compatible field name above; this is now deterministic
    # Python quality validation, not a second LLM semantic-review request.
    "quality_validation_mode":
        "deterministic_python_plus_mandatory_hitl",

    "llm_semantic_review_used":
        False,

    "generated_human_review_state":
        human_review_state,

    "generated_quality_accepted":
        generated_quality_accepted,

    "regeneration_attempts_used":
        regeneration_attempts_used,

    "automatic_corrective_regeneration_attempts_used":
        automatic_corrective_regeneration_attempts_used,

    "whole_quiz_human_regeneration_attempts_used":
        whole_quiz_human_regeneration_attempts_used,

    "human_targeted_regeneration_attempts_used":
        human_targeted_regeneration_attempts_used,

    "max_regeneration_attempts":
        MAX_REGENERATION_ATTEMPTS,

    "final_constraints_met":
        final_constraints_met,

    "mark_target_met":
        mark_target_met,

    "question_count_target_met":
        question_count_target_met,

    "primary_target_met":
        primary_target_met,

    "supporting_target_met":
        supporting_target_met,

    "distinct_style_special_target_met":
        distinct_style_special_target_met,

    "special_instruction_directives":
        special_directives,

    "distinct_reference_target_met":
        distinct_reference_target_met,

    "coverage_target_met":
        coverage_target_met,

    "official_release_ready":
        official_release_ready,

    "official_release_failures":
        official_release_failures,

    "generated_release_ready":
        generated_release_ready,

    "assessment_filters":
        request,

    "special_instructions_applied":
        str(
            request.get(
                "special_instructions",
                "",
            )
            or ""
        ).strip(),

    "approved_topics": [
        {
            "topic":
                item[
                    "topic"
                ],

            "role":
                item[
                    "role"
                ],

            "official_reference":
                item[
                    "official_reference"
                ],
        }
        for item in approved_topics
    ],

    # Backward-compatible field now represents the effective validation blueprint.
    "generation_blueprint":
        generation_blueprint_effective,

    "generation_blueprint_original":
        generation_blueprint_original,

    "generation_blueprint_effective":
        generation_blueprint_effective,

    "effective_blueprint_path":
        str(EFFECTIVE_BLUEPRINT_PATH),

    "generation_validation":
        generation_validation,

    "semantic_quality_validation":
        semantic_quality_validation,

    "generated_human_review":
        generated_human_review,

    "question_level_human_review": {
        "queue": question_review_queue,
        "queue_path": str(HUMAN_REVIEW_QUEUE_PATH),
        "feedback_log_path": str(HUMAN_REVIEW_FEEDBACK_LOG_PATH),
        "actions_received": question_review_actions,
        "action_errors": question_review_action_errors,
        "approval_reset_due_to_changes": approval_reset_due_to_question_changes,
        "unresolved_rejections": unresolved_question_rejections,
        "memory_lookup_audit": HITL_MEMORY_LOOKUP_AUDIT,
    },

    "source_artifacts": {
        "notebook05_package":
            (
                str(
                    notebook05_package_path
                )
                if notebook05_package_path
                else None
            ),

        "notebook05_selected_csv":
            (
                str(
                    notebook05_selected_path
                )
                if notebook05_selected_path
                else None
            ),

        "notebook05_student_question_paper_pdf":
            (
                str(notebook05_student_pdf_path)
                if notebook05_student_pdf_path
                else None
            ),

        "notebook05_student_pdf_resolution":
            (
                str(notebook05_pdf_resolution_path)
                if (
                    "notebook05_pdf_resolution_path" in globals()
                    and notebook05_pdf_resolution_path.is_file()
                )
                else None
            ),

        "generation_request":
            str(
                generation_request_path
            ),

        "generated_candidate":
            (
                str(
                    generated_path
                )
                if generated_path.is_file()
                else None
            ),

        "generated_visual_dir":
            str(
                VISUAL_DIR
            ),

        "model_call_usage":
            str(
                MODEL_CALL_USAGE_PATH
            ),

        "hybrid_batching_diagnostics":
            (
                str(OUTPUT_DIR / "hybrid_batching_diagnostics.json")
                if (OUTPUT_DIR / "hybrid_batching_diagnostics.json").is_file()
                else None
            ),

        "hybrid_generation_trace":
            (
                str(HYBRID_GENERATION_TRACE_PATH)
                if HYBRID_GENERATION_TRACE_PATH.is_file()
                else None
            ),

        "qa_validation_report":
            (
                str(QA_VALIDATION_REPORT_PATH)
                if QA_VALIDATION_REPORT_PATH.is_file()
                else None
            ),

        "human_review_queue":
            str(HUMAN_REVIEW_QUEUE_PATH),

        "human_review_feedback_log":
            str(HUMAN_REVIEW_FEEDBACK_LOG_PATH),
    },

    # Candidate questions are visible for HITL but are not released.
    "candidate_questions":
        (
            candidate_generated_questions_enriched
            if generated_candidate_questions
            else []
        ),

    # Accepted/released questions only.
    "questions":
        final_questions,
}


manifest_path = (
    OUTPUT_DIR
    / "final_quiz_manifest.json"
)

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False,
        default=str,
    ),
    encoding="utf-8",
)


summary_df = pd.DataFrame(
    [
        {
            "quiz_mode":
                QUIZ_MODE,

            "assessment_type":
                assessment_type,

            "release_ready":
                release_ready,

            "marks":
                f"{display_total_marks}/{request['target_total_marks']}",

            "questions":
                f"{display_question_count}/{request['number_of_questions']}",

            "accepted_marks":
                actual_total_marks,

            "accepted_questions":
                actual_question_count,

            "candidate_marks":
                candidate_total_marks,

            "candidate_questions":
                candidate_question_count,

            "official_questions":
                len(
                    official_questions
                ),

            "generated_questions":
                len(
                    generated_questions
                ),

            "quality_status":
                semantic_status,

            "llm_semantic_review_used":
                False,

            "human_state":
                human_review_state,
        }
    ]
)

display(
    summary_df
)

print(
    "Saved final manifest:",
    manifest_path,
)


## 13A. Export three-backend visual-tool handoff for Notebook 08

This cell creates the stable bridge between quiz generation and the dedicated
visual renderer layer.

It does **not** call MCP and it does **not** make another LLM request. It only
serializes the already-generated/validated visual intent and specification into:

```text
visual_tool_handoff.json
```

The routing hint identifies one of three future tool contracts:

```text
render_logic_visual
render_technical_visual
render_structured_visual
```

Notebook 08 executes these contracts locally/through Kroki for now. MCP will
later become the invocation layer without changing the handoff schema.


In [ ]:
# ================================================================
# NOTEBOOK 08 VISUAL-TOOL HANDOFF
# ================================================================
# No MCP call is made here.
# No additional LLM call is made here.
# This exports the visual intent/spec already produced by Notebook 06.
# ================================================================

VISUAL_TOOL_HANDOFF_SCHEMA_VERSION = "agent2-visual-tool-handoff-v1.3.0"
VISUAL_TOOL_ARCHITECTURE_VERSION = "agent2-visual-tool-routing-v2.0.0"
VISUAL_TOOL_HANDOFF_PATH = OUTPUT_DIR / "visual_tool_handoff.json"

VISUAL_TOOL_ROUTING_HINTS = {
    "none": {
        "tool_name": None,
        "backend": None,
        "engine": None,
        "renderer": None,
    },

    # Specialist Boolean / logic-circuit renderer.
    "logic_gate_diagram": {
        "tool_name": "render_logic_visual",
        "backend": "schemdraw",
        "engine": "schemdraw_logic",
        "renderer": "schemdraw",
    },

    # Technical diagrams use Kroki. GraphViz is the initial engine because it
    # covers the current network / flowchart / CPU visual contracts without
    # requiring a separate local Graphviz installation.
    "network_diagram": {
        "tool_name": "render_technical_visual",
        "backend": "kroki",
        "engine": "graphviz",
        "renderer": "kroki_graphviz",
    },
    "simple_flowchart": {
        "tool_name": "render_technical_visual",
        "backend": "kroki",
        "engine": "graphviz",
        "renderer": "kroki_graphviz",
    },
    "cpu_block_diagram": {
        "tool_name": "render_technical_visual",
        "backend": "kroki",
        "engine": "graphviz",
        "renderer": "kroki_graphviz",
    },

    # Exact-value assessment visuals remain deterministic/local.
    "truth_table": {
        "tool_name": "render_structured_visual",
        "backend": "local_structured",
        "engine": "table",
        "renderer": "local_structured",
    },
    "code_block": {
        "tool_name": "render_structured_visual",
        "backend": "local_structured",
        "engine": "code",
        "renderer": "local_structured",
    },
    "trace_table": {
        "tool_name": "render_structured_visual",
        "backend": "local_structured",
        "engine": "table",
        "renderer": "local_structured",
    },
    "array_grid": {
        "tool_name": "render_structured_visual",
        "backend": "local_structured",
        "engine": "grid",
        "renderer": "local_structured",
    },
    "database_table": {
        "tool_name": "render_structured_visual",
        "backend": "local_structured",
        "engine": "table",
        "renderer": "local_structured",
    },
    "memory_grid": {
        "tool_name": "render_structured_visual",
        "backend": "local_structured",
        "engine": "grid",
        "renderer": "local_structured",
    },
    "binary_register": {
        "tool_name": "render_structured_visual",
        "backend": "local_structured",
        "engine": "grid",
        "renderer": "local_structured",
    },
}

_visual_handoff_questions = (
    generated_questions
    if generated_questions
    else generated_candidate_questions
)

_visual_handoff_rows = []

for position, question in enumerate(_visual_handoff_questions, start=1):
    if not isinstance(question, dict):
        continue

    visual_type = normalize_visual_requirement(
        question.get("visual_requirement", "none")
    )

    visual_payload = question.get(
        "visual",
        {
            "type": visual_type,
            "spec": {},
        },
    )

    if not isinstance(visual_payload, dict):
        visual_payload = {
            "type": visual_type,
            "spec": {},
        }

    generated_question_id = str(
        question.get("generated_question_id", "")
        or f"GEN_{position:03d}"
    ).strip()

    _visual_handoff_rows.append(
        {
            "question_index": position,
            "generated_question_id": generated_question_id,
            "question_text": _pdf_question_text(question)
                if "_pdf_question_text" in globals()
                else str(
                    question.get(
                        "question_text",
                        question.get("text", ""),
                    )
                    or ""
                ),
            "topic": str(question.get("topic", "") or ""),
            "official_reference": str(
                question.get("official_reference", "") or ""
            ),
            "marks": safe_int(
                question.get(
                    "marks",
                    question.get("marks_numeric", 0),
                )
            ),
            "requires_visual": bool(
                visual_type != "none"
            ),
            "visual_requirement": visual_type,
            "visual": {
                "type": visual_type,
                "spec": visual_payload.get("spec", {}),
            },
            "legacy_visual_path": question.get("visual_path"),
            "legacy_visual_renderer": question.get("visual_renderer"),
            "visual_requirement_source": str(
                question.get(
                    "visual_requirement_source",
                    "",
                )
                or ""
            ),
            "learner_response_scaffold": question.get(
                "learner_response_scaffold",
                question.get("response_scaffold"),
            ),
            "kb_exemplar_visual": question.get(
                "kb_exemplar_visual",
                {},
            ),
            "routing_hint": VISUAL_TOOL_ROUTING_HINTS.get(
                visual_type,
                {
                    "tool_name": None,
                    "backend": None,
                    "engine": None,
                    "renderer": None,
                },
            ),
        }
    )

visual_tool_handoff = {
    "schema_version": VISUAL_TOOL_HANDOFF_SCHEMA_VERSION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "quiz_mode": QUIZ_MODE,
    "generation_model": GENERATION_MODEL,
    "generation_model_display_name": GENERATION_MODEL_DISPLAY_NAME,
    "visual_spec_schema_version": VISUAL_SCHEMA_VERSION,
    "mcp_status": "NOT_WIRED_YET",
    "mcp_note": (
        "Notebook 08 stabilizes three approved renderer contracts first: "
        "SchemDraw logic, Kroki technical diagrams, and local structured visuals. "
        "Expose the same registry through the existing MCP controller later."
    ),
    "visual_tool_architecture_version": VISUAL_TOOL_ARCHITECTURE_VERSION,
    "visual_quality_gate_version": "v2.38",
    "renderer_policy": {
        "logic": "schemdraw",
        "technical": "kroki",
        "structured": "local_structured",
    },
    "routing_hints": VISUAL_TOOL_ROUTING_HINTS,
    "questions": _visual_handoff_rows,
}

VISUAL_TOOL_HANDOFF_PATH.write_text(
    json.dumps(
        visual_tool_handoff,
        indent=2,
        ensure_ascii=False,
        default=str,
    ),
    encoding="utf-8",
)

# Keep the final manifest aware of the new artifact without changing
# release/HITL semantics.
if isinstance(manifest, dict):
    manifest.setdefault("output_files", {})[
        "visual_tool_handoff"
    ] = str(VISUAL_TOOL_HANDOFF_PATH)

    manifest.setdefault("source_artifacts", {})[
        "visual_tool_handoff"
    ] = str(VISUAL_TOOL_HANDOFF_PATH)

    manifest["visual_tool_architecture"] = {
        "handoff_schema_version": VISUAL_TOOL_HANDOFF_SCHEMA_VERSION,
        "renderer_notebook": "08_visual_generation_tool_layer.ipynb",
        "mcp_status": "NOT_WIRED_YET",
        "separate_image_generation_api_used": False,
        "visual_tool_architecture_version": VISUAL_TOOL_ARCHITECTURE_VERSION,
        "renderer_policy": {
            "logic": "schemdraw",
            "technical": "kroki",
            "structured": "local_structured",
        },
    }

    manifest_path.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False,
            default=str,
        ),
        encoding="utf-8",
    )

print("Saved Notebook 08 visual handoff:", VISUAL_TOOL_HANDOFF_PATH)
print("Visual questions in handoff:", sum(
    1
    for row in _visual_handoff_rows
    if row.get("requires_visual")
))


## 14. Human-readable final report

In [ ]:
report_lines = [
    "AGENT 2 — FINAL QUIZ GENERATION",
    "=" * 72,
    "",
    f"Quiz mode: {QUIZ_MODE}",
    (
        "Notebook 05 used: NO"
        if QUIZ_MODE
        == "complete_quiz"
        else (
            "Notebook 05 used: YES"
            + (
                f" ({notebook05_run_timestamp})"
                if notebook05_run_timestamp
                else ""
            )
        )
    ),
    f"Generator: {GENERATION_MODEL_DISPLAY_NAME} ({GENERATION_MODEL})",
    f"Provider: {GENERATION_PROVIDER}",
    "LLM architecture: same generation call interprets special instructions + generates quiz",
    "Separate special-instruction interpreter LLM call: NO",
    "Secondary LLM semantic reviewer: NO",
    "Generation batching: generic token/complexity-adaptive; no fixed quiz-size split",
    "Corrective strategy: targeted failed-plan repair when question-local failures are resolvable",
    "HITL architecture: mandatory question-level review/action contract + final whole-quiz approval gate",
    "HITL cost policy: deterministic edits/relabels/remove-visual are local; only explicit question regeneration calls the LLM",
    f"Question-level HITL actions received: {len(question_review_actions)}",
    f"Question-level HITL action errors: {len(question_review_action_errors)}",
    f"Question review queue items: {len(question_review_queue)}",
    f"Approval reset due to question changes: {approval_reset_due_to_question_changes}",
    f"Human-targeted regeneration attempts: "
    f"{human_targeted_regeneration_attempts_used}/"
    f"{MAX_HUMAN_TARGETED_REGENERATION_ATTEMPTS}",
    (
        "HITL PostgreSQL persistence: "
        + json.dumps(
            generated_human_review.get("feedback_db_persistence", {}),
            ensure_ascii=False,
            default=str,
        )
    ),
    (
        "HITL Qdrant memory promotion: "
        + json.dumps(
            generated_human_review.get("feedback_qdrant_memory", {}),
            ensure_ascii=False,
            default=str,
        )
    ),
    (
        "HITL past-memory retrieval audit: "
        + json.dumps(
            generated_human_review.get("feedback_memory_lookup_audit", []),
            ensure_ascii=False,
            default=str,
        )
    ),
    f"Model context window: {ACTIVE_MODEL_CONTEXT_WINDOW_TOKENS} tokens",
    f"Model hard max output: {ACTIVE_MODEL_HARD_MAX_OUTPUT_TOKENS} tokens",
    (
        f"Provider/service-tier TPM limit: {ACTIVE_MODEL_PROVIDER_TPM_LIMIT_TOKENS} tokens"
        if ACTIVE_MODEL_PROVIDER_TPM_LIMIT_TOKENS is not None
        else "Provider/service-tier TPM limit: not configured"
    ),
    f"Notebook output budget: {MODEL_MIN_GENERATION_OUTPUT_TOKENS}-"
    f"{MODEL_MAX_GENERATION_OUTPUT_TOKENS} tokens",
    f"API token usage summary: {model_usage_summary()}",
    "Visual architecture: FINAL (Phase 1 + Phase 2 + systems/data final phase)",
    f"Visual renderer: {VISUAL_RENDERER}",
    f"Visual render status: "
    f"{generation_validation.get('visual_validation', {}).get('status', 'NOT_RUN')}",
    f"Rendered candidate visuals: "
    f"{generation_validation.get('visual_validation', {}).get('rendered_count', 0)}",
    f"Visual relevance gate: "
    f"{generation_validation.get('visual_validation', {}).get('visual_relevance_gate_status', 'NOT_RUN')}",
    f"Visuals kept by relevance gate: "
    f"{generation_validation.get('visual_validation', {}).get('relevance_kept_count', 0)}",
    f"Visuals removed as irrelevant: "
    f"{generation_validation.get('visual_validation', {}).get('relevance_removed_count', 0)}",
    f"Paper routing status: "
    f"{paper_routing_preflight_result.get('status', 'UNKNOWN')}",
    f"Paper routing blocked: {PAPER_ROUTING_BLOCKED}",
    (
        "Paper routing message: "
        + PAPER_ROUTING_BLOCK_REASON
        if PAPER_ROUTING_BLOCK_REASON
        else "Paper routing message: none"
    ),
    (
        "Special instructions: "
        + str(
            request.get(
                "special_instructions",
                "",
            )
            or ""
        ).strip()
        if str(
            request.get(
                "special_instructions",
                "",
            )
            or ""
        ).strip()
        else "Special instructions: none"
    ),
    (
        "Special instruction policy: "
        + str(
            request.get(
                "special_instruction_directives",
                {},
            ).get(
                "mode",
                "none",
            )
            if isinstance(
                request.get(
                    "special_instruction_directives",
                    {},
                ),
                dict,
            )
            else "none"
        )
    ),
    (
        "Deterministic preflight hints (not full interpretation): "
        + json.dumps(
            request.get(
                "special_instruction_directives",
                {},
            ),
            ensure_ascii=False,
            default=str,
        )
    ),
    (
        "Same-call LLM instruction interpretation: "
        + json.dumps(
            generated_payload.get(
                "instruction_interpretation",
                {},
            )
            if isinstance(
                generated_payload,
                dict,
            )
            else {},
            ensure_ascii=False,
            default=str,
        )
    ),
    (
        "Same-call special-instruction compliance: "
        + json.dumps(
            generated_payload.get(
                "special_instruction_compliance",
                [],
            )
            if isinstance(
                generated_payload,
                dict,
            )
            else [],
            ensure_ascii=False,
            default=str,
        )
    ),
    "",
    f"Target marks: {request['target_total_marks']}",
    f"Current visible marks: {display_total_marks}",
    f"Accepted marks: {actual_total_marks}",
    f"Candidate marks: {candidate_total_marks}",
    f"Target questions: {request['number_of_questions']}",
    f"Current visible questions: {display_question_count}",
    f"Accepted questions: {actual_question_count}",
    f"Candidate questions: {candidate_question_count}",
    "",
    f"Assessment type: {assessment_type}",
    f"Generation required: {generation_required}",
    f"Generation status: {generation_status}",
    f"Structural valid: {structural_valid}",
    f"Deterministic quality status: {semantic_status}",
    f"Human review state: {human_review_state}",
    "Secondary LLM semantic reviewer used: NO",
    "Final qualitative gate: HITL",
    f"Generated quality accepted: {generated_quality_accepted}",
    f"Automatic corrective regeneration attempts: "
    f"{automatic_corrective_regeneration_attempts_used}/{MAX_REGENERATION_ATTEMPTS}",
    f"Whole-quiz human regeneration attempts: "
    f"{whole_quiz_human_regeneration_attempts_used}",
    f"Human-targeted question regeneration attempts: "
    f"{human_targeted_regeneration_attempts_used}/"
    f"{MAX_HUMAN_TARGETED_REGENERATION_ATTEMPTS}",
    "",
    f"Final constraints met: {final_constraints_met}",
    f"Official release ready: {official_release_ready}",
    f"Generated release ready: {generated_release_ready}",
    f"FINAL RELEASE READY: {release_ready}",
    "",
    "Final outcome:",
]

if assessment_type == "official_only":
    report_lines.append(
        "- OFFICIAL_ONLY"
    )

elif assessment_type == "hybrid":
    report_lines.append(
        "- HYBRID = official AQA questions + AI-generated missing coverage"
    )

elif assessment_type == "generated_only":
    report_lines.append(
        f"- GENERATED_ONLY = complete quiz generated directly by {GENERATION_MODEL}"
    )

elif assessment_type == "generated_candidate":
    report_lines.append(
        "- GENERATED_CANDIDATE = quiz exists and is visible, but mandatory HITL approval is still pending"
    )

elif assessment_type == "hybrid_candidate":
    report_lines.append(
        "- HYBRID_CANDIDATE = official questions + generated candidate coverage exist, but generated coverage is not yet approved"
    )

else:
    report_lines.append(
        "- No accepted quiz yet"
    )


if (
    candidate_generated_questions_enriched
    and not generated_quality_accepted
):
    report_lines.extend(
        [
            "",
            "Generated candidate awaiting acceptance:",
        ]
    )

    for position, question in enumerate(
        candidate_generated_questions_enriched,
        start=1,
    ):
        report_lines.append(
            f"- Candidate Q{position}: "
            f"{question.get('paper_label') or 'Paper unknown'}"
            f" | {question.get('topic') or ''}"
            f" | {safe_int(question.get('marks'))} mark(s)"
        )


if final_questions:
    report_lines.extend(
        [
            "",
            "Final accepted question paper routing:",
        ]
    )

    for position, question in enumerate(
        final_questions,
        start=1,
    ):
        report_lines.append(
            f"- Q{position}: "
            f"{question.get('paper_label') or 'Paper unknown'}"
            f" | {question.get('topic') or question.get('detected_topic') or ''}"
        )


if semantic_quality_validation.get(
    "reasons"
):
    report_lines.extend(
        [
            "",
            "Generated deterministic quality/review reasons:",
        ]
    )

    for reason in semantic_quality_validation[
        "reasons"
    ]:
        report_lines.append(
            "- "
            + str(
                reason
            )
        )


if official_release_failures:
    report_lines.extend(
        [
            "",
            "Official release blockers:",
        ]
    )

    for blocker in official_release_failures:
        report_lines.append(
            "- "
            + str(
                blocker.get(
                    "question_number"
                )
            )
            + ": "
            + str(
                blocker.get(
                    "visual_render_error"
                )
                or "student_release_eligible=false"
            )
        )


report_text = "\n".join(
    report_lines
)

print(
    report_text
)


report_path = (
    OUTPUT_DIR
    / "quiz_generation_report.txt"
)

report_path.write_text(
    report_text,
    encoding="utf-8",
)


print()
print(
    "Saved report:",
    report_path,
)


## 15. Automatic questions + marking-scheme PDF

Notebook 06 now creates a presentation-ready combined PDF after every quiz run.

**Output file**

```text
Agent2_Quiz_Output_Questions_and_Marking_Schemes.pdf
```

Behavior:

- after HITL acceptance, the PDF uses the accepted final quiz;
- before acceptance / when structural review blocks release, the PDF uses the current visible candidate so it can still be demonstrated;
- this export does **not** make the quiz release-ready and does not bypass validation or HITL;
- existing rendered visual PNGs are embedded when available;
- if a binary-register visual could not be rendered, the PDF reconstructs it directly from the stored visual JSON specification;
- each question is followed by its marking scheme / AI-generated marking guidance;
- the generated PDF path is registered in `final_quiz_manifest.json` under `output_files` and `source_artifacts`.

PDF hardening now normalizes unsupported dash/space glyphs before ReportLab rendering, renders simple Markdown cleanly, and normalizes compact binary-register values so each bit/place value occupies its own cell.



## v2.39 — graceful fail-closed behavior

A visual-integrity failure is now treated as a **quality-blocked run**, not a notebook crash.

Example:

```text
GEN_010:
question_text refers to a visual
but visual_requirement = none
        ↓
targeted regeneration of plan_index 10
        ↓
if repaired → continue normally
        ↓
if still invalid → no student PDF
                    release_ready = false
                    manifest records errors
                    Notebook 06 exits normally
```

This preserves the fail-closed principle without making Streamlit report the entire Agent 2 notebook as an execution failure.


## v2.41 — generic anti-overfitting design

The final visual planner no longer contains topic-name → renderer rules.

```text
reviewed KB exemplar
        ↓
explicit visual-form evidence?
        │
        ├─ yes → plan that visual type
        └─ no  → text-only by default
```

Python then validates **universal invariants only**:

- valid renderer schema;
- graph references point to real IDs;
- tables/grids are structurally valid;
- question refers to a visual only when one exists;
- generated visual must be integrated into the learner task;
- duplicate visuals/questions are blocked using fingerprints and generic
  lexical/semantic similarity;
- final qualitative relevance remains HITL.

This is intentionally conservative: when evidence is ambiguous, the system
prefers a valid text-only question rather than guessing a visual from topic
keywords.


## v2.42 — why these checks are generic

No rule in this layer maps a syllabus topic to a fixed answer or visual.

### Machine answer verification

The generator may provide a restricted pure expression only when an objective
answer can be calculated from data already present in that generated question.

Examples of the **mechanism** (not topic rules):

```text
explicit values in generated evidence
        ↓
pure expression
        ↓
Python computes independently
        ↓
compare with generated marking answer
```

If this cannot be done safely, the question remains `manual` and HITL reviews it.

### Referential consistency

The validator only checks generic references:
- requested line numbers vs marking-target line numbers;
- line numbers vs actual code length;
- "complete X" vs the actual visual-object type.

### Answer-space diversity

Marking-guidance text is compared lexically/semantically across questions.
There are no hand-written phrases for particular topics. Moderate similarity is
a HITL warning; only very strong overlap is a hard failure.


## v2.43 — marking scheme first, question regeneration second

### Marking-only failure

```text
valid learner-facing question
        +
valid visual/code
        +
wrong mark-scheme answer
        ↓
small same-model repair call
        ↓
ONLY marking_guidance + answer_verification may change
        ↓
question_text + visual stay frozen
```

If that small repair cannot complete, the candidate stays blocked for HITL.
The system does **not** regenerate a valid learner-facing question solely to
repair its mark scheme.

### Generic diversity

Same-reference questions are compared across independent dimensions:
- abstract learner task;
- scenario/context;
- answer form;
- question wording;
- marking-answer space;
- exact visual fingerprints from the existing validator.

Strong multi-dimensional overlap can trigger targeted regeneration; moderate
overlap remains a HITL diversity warning. No topic-specific phrase list is
used.


In [ ]:

# ================================================================
# AUTOMATIC QUIZ PDF — QUESTIONS + MARKING SCHEME
# ================================================================
# Produces one clean, presentation-ready PDF from the current quiz content.
#
# Design:
# - uses accepted questions after HITL when available;
# - otherwise uses the visible generated candidate so blocked/pending runs
#   can still be demonstrated without making them release-ready;
# - embeds deterministic visual PNGs when they exist;
# - reconstructs binary-register visuals directly from their JSON spec when
#   the matplotlib renderer is unavailable;
# - supports questions-first output with a separate marking-schemes section.
#
# The PDF is an OUTPUT artifact only. It does not change validation,
# human-review state, acceptance, or release readiness.
# ================================================================

import html
import subprocess
import sys

try:
    from reportlab.lib import colors
    from reportlab.lib.enums import TA_CENTER
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import mm
    from reportlab.platypus import (
        SimpleDocTemplate,
        Paragraph,
        Spacer,
        Table,
        TableStyle,
        PageBreak,
        Image as RLImage,
    )
except ImportError:
    # Notebook 06 is commonly executed from a fresh project venv.
    # Install only this lightweight PDF dependency if it is missing.
    print("reportlab is not installed; installing it for quiz PDF export...")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "reportlab",
        ]
    )
    from reportlab.lib import colors
    from reportlab.lib.enums import TA_CENTER
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import mm
    from reportlab.platypus import (
        SimpleDocTemplate,
        Paragraph,
        Spacer,
        Table,
        TableStyle,
        PageBreak,
        Image as RLImage,
    )


QUIZ_PDF_FILENAME = (
    "Agent2_Quiz_Output_Questions_and_Marking_Schemes.pdf"
)

quiz_pdf_path = (
    OUTPUT_DIR
    / QUIZ_PDF_FILENAME
)


def _pdf_clean_text(value: Any) -> str:
    """Normalize common glyphs before ReportLab rendering."""
    # Preserve legitimate falsy learner/table values such as 0 and False.
    # Only None represents an absent PDF value.
    text_value = (
        ""
        if value is None
        else str(value)
    )

    replacements = {
        "🡨": "<-",
        "←": "<-",
        "≤": "<=",
        "≥": ">=",
        "‐": "-",
        "‑": "-",
        "‒": "-",
        "–": "-",
        "—": "-",
        "−": "-",
        "\u00a0": " ",
        "\u200b": "",
        "\u2060": "",
        "’": "'",
        "‘": "'",
        "“": '"',
        "”": '"',
        "•": "-",
        "×": "x",
    }

    for old, new in replacements.items():
        text_value = text_value.replace(
            old,
            new,
        )

    return text_value


def _pdf_escape(value: Any) -> str:
    return html.escape(
        _pdf_clean_text(
            value
        )
    )


def _pdf_question_marks(
    question: dict[str, Any],
) -> int:
    if str(
        question.get(
            "source_type",
            "",
        )
        or ""
    ).strip().casefold() == "official_aqa":
        try:
            return int(
                question_marks(
                    question
                )
            )
        except Exception:
            pass

    return safe_int(
        question.get(
            "marks",
            question.get(
                "marks_numeric",
                0,
            ),
        )
    )


def _pdf_question_text(
    question: dict[str, Any],
) -> str:
    if str(
        question.get(
            "source_type",
            "",
        )
        or ""
    ).strip().casefold() == "official_aqa":
        try:
            value = question_text(
                question
            )
            if str(
                value
                or ""
            ).strip():
                return str(
                    value
                )
        except Exception:
            pass

    for key in [
        "question_text",
        "question_text_canonical",
        "question_text_postgres",
        "question_text_retrieval",
        "text",
    ]:
        value = str(
            question.get(
                key,
                "",
            )
            or ""
        ).strip()

        if value:
            return value

    return "Question text unavailable."


def _pdf_question_topic(
    question: dict[str, Any],
) -> str:
    for key in [
        "topic",
        "detected_topic",
        "topic_label",
    ]:
        value = str(
            question.get(
                key,
                "",
            )
            or ""
        ).strip()

        if value:
            return value

    return "Topic"


def _pdf_question_reference(
    question: dict[str, Any],
) -> str:
    for key in [
        "official_reference",
        "agent1_official_reference",
        "official_reference_canonical",
    ]:
        value = str(
            question.get(
                key,
                "",
            )
            or ""
        ).strip()

        if value:
            return value

    try:
        return str(
            question_reference(
                question
            )
            or ""
        ).strip()
    except Exception:
        return ""


def _pdf_question_paper(
    question: dict[str, Any],
) -> str:
    explicit = str(
        question.get(
            "paper_label",
            "",
        )
        or ""
    ).strip()

    if explicit:
        return explicit

    raw = str(
        question.get(
            "paper_code",
            question.get(
                "paper",
                "",
            ),
        )
        or ""
    ).strip().casefold()

    if raw in {
        "1",
        "1a",
        "1b",
        "paper 1",
        "paper1",
        "p1",
    }:
        return "Paper 1"

    if raw in {
        "2",
        "2a",
        "2b",
        "paper 2",
        "paper2",
        "p2",
    }:
        return "Paper 2"

    reference = _pdf_question_reference(
        question
    )

    match = re.match(
        r"^3\.(\d+)",
        reference,
    )

    if match:
        section = int(
            match.group(
                1
            )
        )
        return (
            "Paper 1"
            if section in {
                1,
                2,
            }
            else "Paper 2"
        )

    return ""


def _pdf_marking_rows(
    question: dict[str, Any],
) -> list[dict[str, Any]]:
    """
    Normalize generated or official marking guidance into rows:
        {"marks": int|str, "criterion": str}
    """
    guidance = question.get(
        "marking_guidance"
    )

    rows: list[
        dict[str, Any]
    ] = []

    if isinstance(
        guidance,
        list,
    ):
        for item in guidance:
            if isinstance(
                item,
                dict,
            ):
                criterion = str(
                    item.get(
                        "criterion",
                        item.get(
                            "text",
                            "",
                        ),
                    )
                    or ""
                ).strip()

                if criterion:
                    rows.append(
                        {
                            "marks":
                                item.get(
                                    "marks",
                                    "",
                                ),
                            "criterion":
                                criterion,
                        }
                    )

            elif str(
                item
                or ""
            ).strip():
                rows.append(
                    {
                        "marks":
                            "",
                        "criterion":
                            str(
                                item
                            ).strip(),
                    }
                )

    if rows:
        return rows

    # Official-question fallback shapes.
    mark_scheme = question.get(
        "mark_scheme"
    )

    if isinstance(
        mark_scheme,
        dict,
    ):
        raw = str(
            mark_scheme.get(
                "raw_marking_guidance",
                mark_scheme.get(
                    "marking_guidance",
                    "",
                ),
            )
            or ""
        ).strip()

        if raw:
            return [
                {
                    "marks":
                        "",
                    "criterion":
                        raw,
                }
            ]

        structured = mark_scheme.get(
            "phase3_structured"
        )

        if isinstance(
            structured,
            dict,
        ):
            structured_points = structured.get(
                "marking_points",
                [],
            )

            if isinstance(
                structured_points,
                list,
            ):
                for item in structured_points:
                    if isinstance(
                        item,
                        dict,
                    ):
                        criterion = str(
                            item.get(
                                "criterion",
                                item.get(
                                    "text",
                                    item.get(
                                        "point",
                                        "",
                                    ),
                                ),
                            )
                            or ""
                        ).strip()

                        if criterion:
                            rows.append(
                                {
                                    "marks":
                                        item.get(
                                            "marks",
                                            "",
                                        ),
                                    "criterion":
                                        criterion,
                                }
                            )

                    elif str(
                        item
                        or ""
                    ).strip():
                        rows.append(
                            {
                                "marks":
                                    "",
                                "criterion":
                                    str(
                                        item
                                    ).strip(),
                            }
                        )

    if rows:
        return rows

    return [
        {
            "marks":
                "",
            "criterion":
                "No marking guidance was available in the current quiz artifact.",
        }
    ]


def _pdf_visual_image_path(
    question: dict[str, Any],
) -> Path | None:
    raw = str(
        question.get(
            "visual_path",
            "",
        )
        or ""
    ).strip()

    if not raw:
        return None

    path_value = Path(
        raw
    )

    if path_value.is_file():
        return path_value

    candidate = (
        OUTPUT_DIR
        / raw
    )

    if candidate.is_file():
        return candidate

    return None


def _pdf_register_visual(
    visual_spec: dict[str, Any],
    *,
    styles: Any,
) -> list[Any]:
    """Fallback visual matching the clean binary-register table used in demo PDFs."""
    (
        bits,
        place_values,
        _labels,
    ) = _normalized_binary_register_values(
        visual_spec
    )

    if not bits:
        return []

    caption = str(
        visual_spec.get(
            "caption",
            "Binary register",
        )
        or "Binary register"
    ).strip()

    elements: list[
        Any
    ] = [
        Paragraph(
            "<b>"
            + _pdf_escape(
                caption
            )
            + "</b>",
            styles[
                "QuizPDFSmall"
            ],
        ),
        Spacer(
            1,
            4,
        ),
    ]

    table_rows: list[
        list[Any]
    ] = []

    if (
        place_values
        and len(
            place_values
        )
        == len(
            bits
        )
    ):
        table_rows.append(
            [
                "Place value",
                *[
                    str(
                        value
                    )
                    for value in place_values
                ],
            ]
        )

    table_rows.append(
        [
            "Bit",
            *[
                str(
                    value
                )
                for value in bits
            ],
        ]
    )

    usable_width = (
        A4[
            0
        ]
        - 32
        * mm
    )

    label_width = (
        22
        * mm
    )

    cell_width = max(
        9
        * mm,
        (
            usable_width
            - label_width
        )
        / max(
            1,
            len(
                bits
            ),
        ),
    )

    register_table = Table(
        table_rows,
        colWidths=[
            label_width,
            *[
                cell_width
                for _ in bits
            ],
        ],
    )

    register_table.setStyle(
        TableStyle(
            [
                (
                    "GRID",
                    (
                        0,
                        0,
                    ),
                    (
                        -1,
                        -1,
                    ),
                    0.8,
                    colors.HexColor(
                        "#475569"
                    ),
                ),
                (
                    "BACKGROUND",
                    (
                        0,
                        0,
                    ),
                    (
                        0,
                        -1,
                    ),
                    colors.HexColor(
                        "#e2e8f0"
                    ),
                ),
                (
                    "FONTNAME",
                    (
                        0,
                        0,
                    ),
                    (
                        0,
                        -1,
                    ),
                    "Helvetica-Bold",
                ),
                (
                    "FONTSIZE",
                    (
                        0,
                        0,
                    ),
                    (
                        -1,
                        -1,
                    ),
                    8.4,
                ),
                (
                    "ALIGN",
                    (
                        1,
                        0,
                    ),
                    (
                        -1,
                        -1,
                    ),
                    "CENTER",
                ),
                (
                    "VALIGN",
                    (
                        0,
                        0,
                    ),
                    (
                        -1,
                        -1,
                    ),
                    "MIDDLE",
                ),
                (
                    "TOPPADDING",
                    (
                        0,
                        0,
                    ),
                    (
                        -1,
                        -1,
                    ),
                    4,
                ),
                (
                    "BOTTOMPADDING",
                    (
                        0,
                        0,
                    ),
                    (
                        -1,
                        -1,
                    ),
                    4,
                ),
                (
                    "LEFTPADDING",
                    (
                        0,
                        0,
                    ),
                    (
                        -1,
                        -1,
                    ),
                    2,
                ),
                (
                    "RIGHTPADDING",
                    (
                        0,
                        0,
                    ),
                    (
                        -1,
                        -1,
                    ),
                    2,
                ),
            ]
        )
    )

    elements.extend(
        [
            register_table,
            Spacer(
                1,
                6,
            ),
        ]
    )

    return elements


def _pdf_native_table_visual(
    *,
    columns: list[Any] | None,
    rows: list[list[Any]],
    styles: Any,
    row_labels: list[Any] | None = None,
    caption: str = "",
) -> list[Any]:
    """ReportLab-native fallback for compact table/grid visual contracts."""
    normalized_rows = [
        ["" if value is None else _pdf_clean_text(value) for value in row]
        for row in rows
        if isinstance(row, list)
    ]
    if not normalized_rows:
        return []

    table_rows: list[list[Any]] = []
    if columns:
        header = [
            _pdf_clean_text(value)
            for value in columns
        ]
        if row_labels is not None:
            header = [""] + header
        table_rows.append(header)

    for index, row in enumerate(normalized_rows):
        rendered = list(row)
        if row_labels is not None:
            label = (
                row_labels[index]
                if index < len(row_labels)
                else ""
            )
            rendered = [_pdf_clean_text(label)] + rendered
        table_rows.append(rendered)

    usable_width = A4[0] - 36 * mm
    column_count = max(1, max(len(row) for row in table_rows))
    table = Table(
        table_rows,
        colWidths=[usable_width / column_count] * column_count,
        hAlign="CENTER",
    )
    table.setStyle(
        TableStyle(
            [
                ("GRID", (0, 0), (-1, -1), 0.75, colors.black),
                ("ALIGN", (0, 0), (-1, -1), "CENTER"),
                ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
                ("FONTNAME", (0, 0), (-1, -1), "Helvetica"),
                ("FONTSIZE", (0, 0), (-1, -1), 8.5),
                ("TOPPADDING", (0, 0), (-1, -1), 5),
                ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
            ]
        )
    )

    elements: list[Any] = []
    if caption:
        elements.extend(
            [
                Paragraph(
                    "<b>" + _pdf_escape(caption) + "</b>",
                    styles["QuizPDFSmall"],
                ),
                Spacer(1, 4),
            ]
        )
    elements.extend([table, Spacer(1, 6)])
    return elements


def _pdf_native_code_visual(
    spec: dict[str, Any],
    *,
    styles: Any,
) -> list[Any]:
    code = _pdf_clean_text(spec.get("code", "")).strip()
    if not code:
        return []
    caption = str(spec.get("caption", "") or "").strip()
    elements: list[Any] = []
    if caption:
        elements.append(
            Paragraph(
                "<b>" + _pdf_escape(caption) + "</b>",
                styles["QuizPDFSmall"],
            )
        )
        elements.append(Spacer(1, 4))
    elements.append(
        Paragraph(
            html.escape(code).replace("\n", "<br/>"),
            styles["QuizPDFCode"],
        )
    )
    return elements


def _pdf_visual_fallback_elements(
    required_type: str,
    spec: dict[str, Any],
    *,
    styles: Any,
) -> list[Any]:
    """
    Native PDF fallback for simple/table visual types.

    Complex diagrams are first re-rendered using the existing deterministic
    renderer. If that also fails, PDF export fails closed instead of silently
    omitting a required student visual.
    """
    if required_type == "code_block":
        return _pdf_native_code_visual(spec, styles=styles)

    if required_type == "trace_table":
        return _pdf_native_table_visual(
            columns=spec.get("columns", []),
            rows=spec.get("rows", []),
            caption=str(spec.get("caption", "") or "").strip(),
            styles=styles,
        )

    if required_type == "truth_table":
        return _pdf_native_table_visual(
            columns=spec.get("columns", []),
            rows=spec.get("rows", []),
            caption=str(spec.get("caption", "") or "").strip(),
            styles=styles,
        )

    if required_type == "database_table":
        columns = [str(value) for value in spec.get("columns", [])]
        pk = str(spec.get("primary_key", "") or "").strip()
        display_columns = [
            f"{column} (PK)" if pk and column == pk else column
            for column in columns
        ]
        return _pdf_native_table_visual(
            columns=display_columns,
            rows=spec.get("rows", []),
            caption=str(spec.get("caption", "") or "").strip(),
            styles=styles,
        )

    if required_type == "array_grid":
        values = spec.get("values", [])
        rows = values if values and isinstance(values[0], list) else [values]
        return _pdf_native_table_visual(
            columns=spec.get("column_labels"),
            rows=rows,
            row_labels=spec.get("row_labels"),
            caption=str(spec.get("caption", "") or "").strip(),
            styles=styles,
        )

    if required_type == "memory_grid":
        addresses = list(spec.get("addresses", []))
        values = list(spec.get("values", []))
        rows = [
            [address, values[index] if index < len(values) else ""]
            for index, address in enumerate(addresses)
        ]
        return _pdf_native_table_visual(
            columns=["Address", "Value"],
            rows=rows,
            caption=str(spec.get("caption", "") or "").strip(),
            styles=styles,
        )

    if required_type == "binary_register":
        return _pdf_register_visual(spec, styles=styles)

    return []



def _pdf_add_response_scaffold(
    story: list[Any],
    question: dict[str, Any],
    *,
    styles: Any,
) -> None:
    """Render learner response areas that are separate from the supplied visual."""
    scaffold = question.get("response_scaffold")

    # Older/restored candidates may predate the attached field. Build the same
    # deterministic blank scaffold on demand so PDF output still remains safe.
    if not isinstance(scaffold, dict):
        builder = globals().get("_truth_table_response_scaffold_from_question")
        if callable(builder):
            scaffold = builder(question)
            if isinstance(scaffold, dict):
                question["response_scaffold"] = scaffold

    if not isinstance(scaffold, dict):
        return

    scaffold_type = str(scaffold.get("type", "") or "").strip().casefold()
    if scaffold_type != "truth_table":
        return

    columns = scaffold.get("columns", [])
    rows = scaffold.get("rows", [])
    caption = str(
        scaffold.get("caption", "Complete the truth table")
        or "Complete the truth table"
    ).strip()

    if not isinstance(columns, list) or not isinstance(rows, list) or not rows:
        raise RuntimeError(
            "Truth-table response scaffold is malformed and cannot be rendered."
        )

    # Fail closed if a supposedly learner-completed final column contains data.
    for row in rows:
        if not isinstance(row, list) or len(row) != len(columns):
            raise RuntimeError(
                "Truth-table response scaffold rows are not rectangular."
            )
        if row[-1] not in ("", None):
            raise RuntimeError(
                "Truth-table response scaffold contains a completed learner output."
            )

    story.extend(
        _pdf_native_table_visual(
            columns=columns,
            rows=rows,
            caption=caption,
            styles=styles,
        )
    )

def _pdf_add_visual(
    story: list[Any],
    question: dict[str, Any],
    *,
    styles: Any,
) -> None:
    required_type = normalize_visual_requirement(
        question.get(
            "visual_requirement",
            "none",
        )
    )

    requires_visual = bool(
        question.get(
            "requires_visual",
            required_type != "none",
        )
    )

    if required_type == "none" or not requires_visual:
        return

    visual = question.get("visual")
    if not isinstance(visual, dict):
        raise RuntimeError(
            f"Required PDF visual {required_type!r} has no visual object."
        )

    visual_type = normalize_visual_requirement(
        visual.get("type", "")
    )
    if visual_type != required_type:
        raise RuntimeError(
            "Required PDF visual type mismatch: "
            f"{visual_type!r} != {required_type!r}."
        )

    spec = visual.get("spec")
    if not isinstance(spec, dict):
        spec = {}

    visual_path = _pdf_visual_image_path(question)

    # If an asset path was lost/stale, reconstruct it from the same validated
    # deterministic visual spec before falling back to native ReportLab tables.
    if visual_path is None or not visual_path.is_file():
        try:
            rebuilt_path = render_visual(question)
            if rebuilt_path is not None and rebuilt_path.is_file():
                visual_path = rebuilt_path
        except Exception as rebuild_exc:
            print(
                "Could not re-render required visual for PDF; trying native "
                "fallback where supported:",
                required_type,
                rebuild_exc,
            )

    if visual_path is not None and visual_path.is_file():
        try:
            image = RLImage(str(visual_path))
            max_width = A4[0] - 40 * mm
            max_height = (
                60 * mm
                if required_type == "code_block"
                else 95 * mm
            )
            width = float(image.imageWidth)
            height = float(image.imageHeight)
            scale = min(
                1.0,
                max_width / max(1.0, width),
                max_height / max(1.0, height),
            )
            image.drawWidth = width * scale
            image.drawHeight = height * scale
            story.extend([
                Spacer(1, 4),
                image,
                Spacer(1, 6),
            ])
            return
        except Exception as exc:
            print(
                "Could not embed visual image in PDF; trying native fallback:",
                visual_path,
                exc,
            )

    fallback = _pdf_visual_fallback_elements(
        required_type,
        spec,
        styles=styles,
    )
    if fallback:
        story.extend(fallback)
        return

    # Do not silently produce a student PDF with a missing required diagram.
    raise RuntimeError(
        "Required visual could not be embedded, re-rendered, or represented "
        f"by a native PDF fallback: {required_type}."
    )


def _pdf_format_prose_markup(
    value: Any,
) -> str:
    text_value = _pdf_clean_text(
        value
    )

    escaped = html.escape(
        text_value
    )

    # Minimal Markdown support used by generated question text.
    escaped = re.sub(
        r"\*\*([^*]+)\*\*",
        r"<b>\1</b>",
        escaped,
    )

    escaped = re.sub(
        r"(?<!\*)\*([^*\n]+)\*(?!\*)",
        r"<i>\1</i>",
        escaped,
    )

    escaped = re.sub(
        r"`([^`\n]+)`",
        r'<font name="Courier">\1</font>',
        escaped,
    )

    return escaped.replace(
        "\n",
        "<br/>",
    )


def _pdf_strip_fence_language(value: Any) -> str:
    """Remove a Markdown fence language tag without touching real code."""
    text_value = _pdf_clean_text(value).strip()
    if not text_value:
        return text_value

    lines = text_value.splitlines()
    if not lines:
        return text_value

    first = lines[0].strip().casefold()
    known_fence_languages = {
        "text", "plaintext", "plain", "pseudocode", "pseudo",
        "python", "py", "sql", "java", "javascript", "js",
        "c", "cpp", "c++", "csharp", "cs",
    }
    if first in known_fence_languages:
        return "\n".join(lines[1:]).strip()

    return text_value


def _pdf_add_question_text(
    story: list[Any],
    text_value: str,
    *,
    styles: Any,
) -> None:
    """
    Preserve code fences while keeping prose compact, glyph-safe and readable.
    """
    parts = re.split(
        r"```",
        _pdf_clean_text(
            text_value
        ),
    )

    for index, part in enumerate(
        parts
    ):
        part = part.strip()

        if not part:
            continue

        if index % 2 == 1:
            code_part = _pdf_strip_fence_language(
                part
            )
            if not code_part:
                continue

            escaped = html.escape(
                code_part
            ).replace(
                "\n",
                "<br/>",
            )

            story.append(
                Paragraph(
                    escaped,
                    styles[
                        "QuizPDFCode"
                    ],
                )
            )

        else:
            formatted = _pdf_format_prose_markup(
                part
            )

            formatted = formatted.replace(
                "<br/>* ",
                "<br/>&bull; ",
            ).replace(
                "<br/>- ",
                "<br/>&bull; ",
            )

            story.append(
                Paragraph(
                    formatted,
                    styles[
                        "QuizPDFBody"
                    ],
                )
            )



def _pdf_visual_integrity_preflight(
    questions: list[dict[str, Any]],
    *,
    styles: Any,
) -> dict[str, Any]:
    """
    Fail closed before ReportLab writes a student-facing PDF.

    This catches the exact class of failure where question_text says
    "the network diagram shows..." but no visual is attached.
    """
    errors: list[str] = []
    diagnostics: list[dict[str, Any]] = []

    for position, question in enumerate(
        questions,
        start=1,
    ):
        if not isinstance(
            question,
            dict,
        ):
            continue

        qid = str(
            question.get(
                "generated_question_id",
                question.get(
                    "question_id",
                    f"Q{position}",
                ),
            )
            or f"Q{position}"
        )

        required_type = normalize_visual_requirement(
            question.get(
                "visual_requirement",
                "none",
            )
        )

        references_visual = (
            _question_references_external_visual(
                _pdf_question_text(
                    question
                )
            )
            if "_question_references_external_visual" in globals()
            else False
        )

        if (
            references_visual
            and required_type == "none"
        ):
            errors.append(
                f"{qid}: question text refers to a visual but "
                "visual_requirement='none'"
            )
            continue

        if required_type == "none":
            diagnostics.append(
                {
                    "question_id": qid,
                    "status": "not_required",
                }
            )
            continue

        visual = question.get(
            "visual",
            {},
        )

        spec_errors = validate_visual_spec(
            visual,
            required_type,
        )

        if spec_errors:
            errors.extend(
                f"{qid}: {message}"
                for message in spec_errors
            )
            continue

        path = _pdf_visual_image_path(
            question
        )

        if (
            path is None
            or not path.is_file()
        ):
            try:
                rebuilt = render_visual(
                    question
                )
                if (
                    rebuilt is not None
                    and rebuilt.is_file()
                ):
                    path = rebuilt
            except Exception:
                path = None

        if (
            path is None
            or not path.is_file()
        ):
            spec = (
                visual.get(
                    "spec",
                    {},
                )
                if isinstance(
                    visual,
                    dict,
                )
                else {}
            )
            fallback = _pdf_visual_fallback_elements(
                required_type,
                spec if isinstance(
                    spec,
                    dict,
                )
                else {},
                styles=styles,
            )
            if not fallback:
                errors.append(
                    f"{qid}: required {required_type} visual cannot be "
                    "embedded, rebuilt, or represented by a PDF-native fallback"
                )
                continue

        diagnostics.append(
            {
                "question_id": qid,
                "status": "ready",
                "visual_requirement": required_type,
                "path": (
                    str(path)
                    if path is not None
                    else None
                ),
            }
        )

    if errors:
        raise RuntimeError(
            "PDF visual-integrity preflight failed: "
            + " | ".join(
                errors
            )
        )

    return {
        "status": "PASS",
        "errors": [],
        "diagnostics": diagnostics,
    }

def build_quiz_questions_marking_scheme_pdf(
    *,
    questions: list[dict[str, Any]],
    output_path: Path,
    manifest_payload: dict[str, Any],
    allow_review_draft: bool = False,
    include_questions: bool = True,
    include_marking_schemes: bool = True,
    marking_scheme_section_title: str = "Marking Schemes",
) -> Path:
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    pdf_styles = getSampleStyleSheet()

    pdf_styles.add(
        ParagraphStyle(
            name="QuizPDFTitle",
            parent=pdf_styles[
                "Title"
            ],
            fontName="Helvetica-Bold",
            fontSize=20,
            leading=24,
            alignment=TA_CENTER,
            spaceAfter=12,
        )
    )

    pdf_styles.add(
        ParagraphStyle(
            name="QuizPDFSubtitle",
            parent=pdf_styles[
                "Normal"
            ],
            fontName="Helvetica",
            fontSize=10,
            leading=14,
            alignment=TA_CENTER,
            textColor=colors.HexColor(
                "#444444"
            ),
            spaceAfter=14,
        )
    )

    pdf_styles.add(
        ParagraphStyle(
            name="QuizPDFQuestionHeading",
            parent=pdf_styles[
                "Heading2"
            ],
            fontName="Helvetica-Bold",
            fontSize=14,
            leading=17,
            spaceAfter=7,
            textColor=colors.HexColor(
                "#1f2937"
            ),
        )
    )

    pdf_styles.add(
        ParagraphStyle(
            name="QuizPDFSection",
            parent=pdf_styles[
                "Heading3"
            ],
            fontName="Helvetica-Bold",
            fontSize=11.5,
            leading=14,
            spaceBefore=8,
            spaceAfter=6,
        )
    )

    pdf_styles.add(
        ParagraphStyle(
            name="QuizPDFBody",
            parent=pdf_styles[
                "BodyText"
            ],
            fontName="Helvetica",
            fontSize=9.6,
            leading=13.2,
            spaceAfter=6,
        )
    )

    pdf_styles.add(
        ParagraphStyle(
            name="QuizPDFCode",
            parent=pdf_styles[
                "BodyText"
            ],
            fontName="Courier",
            fontSize=8.7,
            leading=11.5,
            leftIndent=8,
            rightIndent=8,
            borderColor=colors.HexColor(
                "#d1d5db"
            ),
            borderWidth=0.5,
            borderPadding=6,
            backColor=colors.HexColor(
                "#f8fafc"
            ),
            spaceBefore=5,
            spaceAfter=7,
        )
    )

    pdf_styles.add(
        ParagraphStyle(
            name="QuizPDFSmall",
            parent=pdf_styles[
                "BodyText"
            ],
            fontName="Helvetica",
            fontSize=8.6,
            leading=11,
            textColor=colors.HexColor(
                "#4b5563"
            ),
        )
    )

    pdf_styles.add(
        ParagraphStyle(
            name="QuizPDFMarkScheme",
            parent=pdf_styles[
                "BodyText"
            ],
            fontName="Helvetica",
            fontSize=8.8,
            leading=11.5,
        )
    )

    if include_questions:
        try:
            pdf_visual_integrity_preflight = (
                _pdf_visual_integrity_preflight(
                    questions,
                    styles=pdf_styles,
                )
            )
        except RuntimeError as exc:
            if (
                not allow_review_draft
                or "PDF visual-integrity preflight failed:" not in str(exc)
            ):
                raise

            # Review PDFs are diagnostic/HITL artifacts, not release artifacts.
            # Preserve the validator warning but still render the available content.
            pdf_visual_integrity_preflight = {
                "status": "DRAFT_WITH_WARNINGS",
                "errors": [str(exc)],
                "diagnostics": [],
            }
    else:
        # A mark-scheme-only PDF does not render question visuals, so visual
        # integrity preflight is not required for this internal section artifact.
        pdf_visual_integrity_preflight = {
            "status": "NOT_REQUIRED_MARK_SCHEME_ONLY",
            "errors": [],
            "diagnostics": [],
        }

    pdf_document = SimpleDocTemplate(
        str(
            output_path
        ),
        pagesize=A4,
        rightMargin=16
        * mm,
        leftMargin=16
        * mm,
        topMargin=16
        * mm,
        bottomMargin=16
        * mm,
        title=(
            "Agent 2 Quiz Output - "
            "Questions and Marking Schemes"
        ),
        author="EDTech Agent 2",
    )

    story: list[
        Any
    ] = []

    generated_model = str(
        manifest_payload.get(
            "generation_model",
            GENERATION_MODEL,
        )
        or GENERATION_MODEL
    ).strip()

    quiz_mode_value = str(
        manifest_payload.get(
            "quiz_mode",
            QUIZ_MODE,
        )
        or QUIZ_MODE
    ).strip()

    total_marks = sum(
        _pdf_question_marks(
            question
        )
        for question in questions
        if isinstance(
            question,
            dict,
        )
    )

    question_count = len(
        [
            question
            for question in questions
            if isinstance(
                question,
                dict,
            )
        ]
    )

    target_marks_value = safe_int(
        manifest_payload.get(
            "target_marks",
            request.get(
                "target_total_marks",
                0,
            ),
        )
    )

    target_questions_value = safe_int(
        manifest_payload.get(
            "target_question_count",
            request.get(
                "number_of_questions",
                0,
            ),
        )
    )

    review_draft_mode = bool(
        manifest_payload.get("pdf_review_draft", False)
        or not bool(manifest_payload.get("release_ready", False))
    )

    story.append(
        Paragraph(
            (
                "Agent 2 - AI Quiz Review Draft"
                if review_draft_mode
                else "Agent 2 - AI Quiz Generation Output"
            ),
            pdf_styles[
                "QuizPDFTitle"
            ],
        )
    )

    story.append(
        Paragraph(
            (
                "Model: "
                + _pdf_escape(
                    generated_model
                )
                + " | Quiz mode: "
                + _pdf_escape(
                    quiz_mode_value
                )
                + " | Generated content: "
                + str(
                    question_count
                )
                + " questions / "
                + str(
                    total_marks
                )
                + " marks"
            ),
            pdf_styles[
                "QuizPDFSubtitle"
            ],
        )
    )

    summary_rows = [
        [
            "Target marks",
            str(
                target_marks_value
            ),
            "PDF marks",
            str(
                total_marks
            ),
        ],
        [
            "Target questions",
            str(
                target_questions_value
            ),
            "PDF questions",
            str(
                question_count
            ),
        ],
    ]

    summary_table = Table(
        summary_rows,
        colWidths=[
            34
            * mm,
            28
            * mm,
            38
            * mm,
            48
            * mm,
        ],
    )

    summary_table.setStyle(
        TableStyle(
            [
                (
                    "GRID",
                    (
                        0,
                        0,
                    ),
                    (
                        -1,
                        -1,
                    ),
                    0.5,
                    colors.HexColor(
                        "#cbd5e1"
                    ),
                ),
                (
                    "BACKGROUND",
                    (
                        0,
                        0,
                    ),
                    (
                        0,
                        -1,
                    ),
                    colors.HexColor(
                        "#f1f5f9"
                    ),
                ),
                (
                    "BACKGROUND",
                    (
                        2,
                        0,
                    ),
                    (
                        2,
                        -1,
                    ),
                    colors.HexColor(
                        "#f1f5f9"
                    ),
                ),
                (
                    "FONTNAME",
                    (
                        0,
                        0,
                    ),
                    (
                        0,
                        -1,
                    ),
                    "Helvetica-Bold",
                ),
                (
                    "FONTNAME",
                    (
                        2,
                        0,
                    ),
                    (
                        2,
                        -1,
                    ),
                    "Helvetica-Bold",
                ),
                (
                    "FONTSIZE",
                    (
                        0,
                        0,
                    ),
                    (
                        -1,
                        -1,
                    ),
                    9,
                ),
                (
                    "VALIGN",
                    (
                        0,
                        0,
                    ),
                    (
                        -1,
                        -1,
                    ),
                    "MIDDLE",
                ),
                (
                    "LEFTPADDING",
                    (
                        0,
                        0,
                    ),
                    (
                        -1,
                        -1,
                    ),
                    6,
                ),
                (
                    "RIGHTPADDING",
                    (
                        0,
                        0,
                    ),
                    (
                        -1,
                        -1,
                    ),
                    6,
                ),
                (
                    "TOPPADDING",
                    (
                        0,
                        0,
                    ),
                    (
                        -1,
                        -1,
                    ),
                    5,
                ),
                (
                    "BOTTOMPADDING",
                    (
                        0,
                        0,
                    ),
                    (
                        -1,
                        -1,
                    ),
                    5,
                ),
            ]
        )
    )

    story.extend(
        [
            summary_table,
            Spacer(
                1,
                9,
            ),
        ]
    )

    visual_errors = safe_list(
        (
            manifest_payload.get(
                "generation_validation",
                {},
            )
            or {}
        ).get(
            "visual_validation",
            {},
        ).get(
            "errors",
            [],
        )
    )

    if visual_errors:
        story.append(
            Paragraph(
                (
                    "<b>Demonstration note:</b> "
                    "Questions and marking guidance were generated. "
                    "Where an original rendered visual is unavailable, "
                    "this PDF uses the stored visual specification when "
                    "a deterministic fallback is supported. This PDF does "
                    "not override the notebook's validation or release gate."
                ),
                pdf_styles[
                    "QuizPDFSmall"
                ],
            )
        )

    story.append(
        PageBreak()
    )

    valid_questions = [
        question
        for question in questions
        if isinstance(
            question,
            dict,
        )
    ]

    def _question_metadata_values(
        question: dict[str, Any],
    ) -> tuple[int, str, str, str, str, str, str]:
        marks = _pdf_question_marks(question)
        topic = _pdf_question_topic(question)
        reference = _pdf_question_reference(question)
        paper_label = _pdf_question_paper(question)
        role = str(
            question.get(
                "role",
                question.get("topic_role", ""),
            )
            or ""
        ).strip()
        assessment_pattern = str(
            question.get("assessment_pattern", "")
            or ""
        ).strip()
        generated_question_id = str(
            question.get(
                "generated_question_id",
                question.get("question_id", ""),
            )
            or ""
        ).strip()
        return (
            marks,
            topic,
            reference,
            paper_label,
            role,
            assessment_pattern,
            generated_question_id,
        )

    def _question_metadata_table(
        question: dict[str, Any],
    ) -> Table:
        (
            _marks,
            topic,
            reference,
            paper_label,
            role,
            assessment_pattern,
            generated_question_id,
        ) = _question_metadata_values(question)

        metadata_rows = [
            ["Topic", topic, "AQA ref", reference],
            ["Paper", paper_label, "Role", role],
            ["Pattern", assessment_pattern, "ID", generated_question_id],
        ]

        metadata_table = Table(
            metadata_rows,
            colWidths=[18 * mm, 62 * mm, 18 * mm, 62 * mm],
        )
        metadata_table.setStyle(
            TableStyle(
                [
                    ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#d1d5db")),
                    ("BACKGROUND", (0, 0), (0, -1), colors.HexColor("#f8fafc")),
                    ("BACKGROUND", (2, 0), (2, -1), colors.HexColor("#f8fafc")),
                    ("FONTNAME", (0, 0), (0, -1), "Helvetica-Bold"),
                    ("FONTNAME", (2, 0), (2, -1), "Helvetica-Bold"),
                    ("FONTSIZE", (0, 0), (-1, -1), 8.3),
                    ("VALIGN", (0, 0), (-1, -1), "TOP"),
                    ("LEFTPADDING", (0, 0), (-1, -1), 4),
                    ("RIGHTPADDING", (0, 0), (-1, -1), 4),
                    ("TOPPADDING", (0, 0), (-1, -1), 3),
                    ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
                ]
            )
        )
        return metadata_table

    def _append_question_block(
        position: int,
        question: dict[str, Any],
    ) -> None:
        marks, *_ = _question_metadata_values(question)

        story.append(
            Paragraph(
                (
                    "Question "
                    + str(position)
                    + " ("
                    + str(marks)
                    + " mark"
                    + ("" if marks == 1 else "s")
                    + ")"
                ),
                pdf_styles["QuizPDFQuestionHeading"],
            )
        )
        story.extend(
            [
                _question_metadata_table(question),
                Spacer(1, 7),
                Paragraph("Question", pdf_styles["QuizPDFSection"]),
            ]
        )
        _pdf_add_question_text(
            story,
            _pdf_question_text(question),
            styles=pdf_styles,
        )
        _pdf_add_visual(
            story,
            question,
            styles=pdf_styles,
        )
        _pdf_add_response_scaffold(
            story,
            question,
            styles=pdf_styles,
        )

    def _append_mark_scheme_block(
        position: int,
        question: dict[str, Any],
    ) -> None:
        (
            marks,
            topic,
            reference,
            paper_label,
            role,
            _assessment_pattern,
            generated_question_id,
        ) = _question_metadata_values(question)

        story.append(
            Paragraph(
                (
                    "Question "
                    + str(position)
                    + " Marking Scheme ("
                    + str(marks)
                    + " mark"
                    + ("" if marks == 1 else "s")
                    + ")"
                ),
                pdf_styles["QuizPDFQuestionHeading"],
            )
        )

        compact_meta = [
            ["Topic", topic, "AQA ref", reference],
            ["Paper", paper_label, "Role", role],
            ["Question ID", generated_question_id, "Marks", str(marks)],
        ]
        compact_table = Table(
            compact_meta,
            colWidths=[20 * mm, 60 * mm, 20 * mm, 60 * mm],
        )
        compact_table.setStyle(
            TableStyle(
                [
                    ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#d1d5db")),
                    ("BACKGROUND", (0, 0), (0, -1), colors.HexColor("#f8fafc")),
                    ("BACKGROUND", (2, 0), (2, -1), colors.HexColor("#f8fafc")),
                    ("FONTNAME", (0, 0), (0, -1), "Helvetica-Bold"),
                    ("FONTNAME", (2, 0), (2, -1), "Helvetica-Bold"),
                    ("FONTSIZE", (0, 0), (-1, -1), 8.3),
                    ("VALIGN", (0, 0), (-1, -1), "TOP"),
                    ("LEFTPADDING", (0, 0), (-1, -1), 4),
                    ("RIGHTPADDING", (0, 0), (-1, -1), 4),
                    ("TOPPADDING", (0, 0), (-1, -1), 3),
                    ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
                ]
            )
        )
        story.extend(
            [
                compact_table,
                Spacer(1, 8),
            ]
        )

        marking_rows = [
            [
                Paragraph("<b>Mark(s)</b>", pdf_styles["QuizPDFMarkScheme"]),
                Paragraph("<b>Criterion</b>", pdf_styles["QuizPDFMarkScheme"]),
            ]
        ]
        for item in _pdf_marking_rows(question):
            marking_rows.append(
                [
                    Paragraph(
                        _pdf_escape(item.get("marks", "")),
                        pdf_styles["QuizPDFMarkScheme"],
                    ),
                    Paragraph(
                        _pdf_escape(item.get("criterion", "")).replace("\n", "<br/>"),
                        pdf_styles["QuizPDFMarkScheme"],
                    ),
                ]
            )

        marking_table = Table(
            marking_rows,
            colWidths=[19 * mm, 142 * mm],
            repeatRows=1,
        )
        marking_table.setStyle(
            TableStyle(
                [
                    ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#cbd5e1")),
                    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#e5e7eb")),
                    ("ALIGN", (0, 1), (0, -1), "CENTER"),
                    ("VALIGN", (0, 0), (-1, -1), "TOP"),
                    ("TOPPADDING", (0, 0), (-1, -1), 5),
                    ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
                    ("LEFTPADDING", (0, 0), (-1, -1), 5),
                    ("RIGHTPADDING", (0, 0), (-1, -1), 5),
                ]
            )
        )
        story.append(marking_table)

    if include_questions:
        for position, question in enumerate(valid_questions, start=1):
            _append_question_block(position, question)
            if position != len(valid_questions):
                story.append(PageBreak())

    if include_marking_schemes and valid_questions:
        if include_questions:
            story.append(PageBreak())
            story.append(
                Paragraph(
                    _pdf_escape(marking_scheme_section_title),
                    pdf_styles["QuizPDFTitle"],
                )
            )
            story.append(
                Paragraph(
                    "Marking guidance for all questions in this assessment.",
                    pdf_styles["QuizPDFSubtitle"],
                )
            )
            story.append(PageBreak())

        for position, question in enumerate(valid_questions, start=1):
            _append_mark_scheme_block(position, question)
            if position != len(valid_questions):
                story.append(PageBreak())

    def _footer(
        canvas: Any,
        document: Any,
    ) -> None:
        canvas.saveState()
        canvas.setFont(
            "Helvetica",
            8,
        )
        canvas.setFillColor(
            colors.HexColor(
                "#6b7280"
            )
        )
        canvas.drawString(
            16
            * mm,
            9
            * mm,
            (
                "EDTech Agent 2 - "
                "Generated Quiz Output"
            ),
        )
        canvas.drawRightString(
            A4[
                0
            ]
            - 16
            * mm,
            9
            * mm,
            (
                "Page "
                + str(
                    document.page
                )
            ),
        )
        canvas.restoreState()

    pdf_document.build(
        story,
        onFirstPage=_footer,
        onLaterPages=_footer,
    )

    return output_path


# PDF export policy:
#
# complete_quiz
#   -> current Notebook 06 generated quiz PDF (unchanged)
#
# fill_shortfall
#   -> ONE final student-facing hybrid PDF only:
#        1) final hybrid summary
#        2) Official AQA Questions heading
#        3) original Notebook 05 official question pages
#        4) AI-Generated Missing Coverage heading
#        5) Notebook 06 AI question pages (questions only)
#        6) Marking Schemes heading
#        7) Notebook 05 official mark schemes + Notebook 06 AI marking guidance
#
# Notebook 05's original PDF remains an internal source artifact, but the
# Notebook 06 shortfall UI should expose only the final combined PDF.
# While HITL is pending, the AI section uses the reviewable candidate. After
# approval, it uses the accepted AI coverage.

import shutil


def _build_hybrid_heading_pdf(
    *,
    output_path: Path,
    title: str,
    subtitle: str = "",
    lines: list[str] | None = None,
) -> Path:
    """Create one clean A4 heading/summary page for the hybrid PDF."""
    from reportlab.pdfgen import canvas as rl_canvas

    output_path.parent.mkdir(parents=True, exist_ok=True)
    c = rl_canvas.Canvas(str(output_path), pagesize=A4)
    width, height = A4

    c.setFillColor(colors.HexColor("#111827"))
    c.setFont("Helvetica-Bold", 22)
    c.drawCentredString(width / 2, height - 52 * mm, _pdf_clean_text(title))

    if subtitle:
        c.setFillColor(colors.HexColor("#4b5563"))
        c.setFont("Helvetica", 11)
        c.drawCentredString(width / 2, height - 62 * mm, _pdf_clean_text(subtitle))

    y = height - 88 * mm
    for line in (lines or []):
        c.setFillColor(colors.HexColor("#1f2937"))
        c.setFont("Helvetica", 11)
        c.drawString(28 * mm, y, _pdf_clean_text(line))
        y -= 10 * mm

    c.setStrokeColor(colors.HexColor("#d1d5db"))
    c.line(24 * mm, 24 * mm, width - 24 * mm, 24 * mm)
    c.setFillColor(colors.HexColor("#6b7280"))
    c.setFont("Helvetica", 8)
    c.drawString(24 * mm, 16 * mm, "EDTech Agent 2")
    c.save()
    return output_path


def _pdf_content_start_page(
    path: Path,
    *,
    cover_markers: list[str],
) -> int:
    """Skip a generated cover page only when its text clearly matches."""
    try:
        import fitz
        with fitz.open(str(path)) as doc:
            if doc.page_count <= 1:
                return 0
            first_text = (doc[0].get_text("text") or "").casefold()
    except Exception:
        return 0

    return 1 if any(marker.casefold() in first_text for marker in cover_markers) else 0


def _pdf_page_count(
    path: Path | None,
) -> int:
    if path is None:
        return 0

    try:
        import fitz
        with fitz.open(str(path)) as doc:
            return int(doc.page_count)
    except Exception:
        return 0


def _merge_pdf_segments(
    segments: list[tuple[Path, int | None, int | None]],
    output_path: Path,
) -> Path:
    """Merge selected page ranges into one final PDF using PyMuPDF."""
    try:
        import fitz
    except ImportError as exc:
        raise RuntimeError(
            "PyMuPDF is required to build the final hybrid quiz PDF."
        ) from exc

    output_path.parent.mkdir(parents=True, exist_ok=True)
    temp_output = output_path.with_name(output_path.stem + "__merge_tmp.pdf")
    if temp_output.exists():
        temp_output.unlink()

    merged = fitz.open()
    try:
        for input_path, from_page, to_page in segments:
            input_path = Path(input_path)
            if not input_path.is_file():
                continue
            with fitz.open(str(input_path)) as source_doc:
                if source_doc.page_count <= 0:
                    continue
                start_page = 0 if from_page is None else max(0, int(from_page))
                end_page = (
                    source_doc.page_count - 1
                    if to_page is None
                    else min(source_doc.page_count - 1, int(to_page))
                )
                if start_page <= end_page:
                    merged.insert_pdf(
                        source_doc,
                        from_page=start_page,
                        to_page=end_page,
                    )
        if merged.page_count <= 0:
            raise RuntimeError("No PDF pages were available for hybrid export.")
        merged.save(str(temp_output))
    finally:
        merged.close()

    if not temp_output.is_file() or temp_output.stat().st_size <= 0:
        raise RuntimeError("Final hybrid PDF was not created correctly.")

    if output_path.exists():
        output_path.unlink()
    temp_output.replace(output_path)
    return output_path


if generated_quality_accepted:
    ai_pdf_questions = [
        question
        for question in generated_questions
        if isinstance(question, dict)
    ]
else:
    ai_pdf_questions = [
        question
        for question in candidate_generated_questions_enriched
        if isinstance(question, dict)
    ]


# ================================================================
# v2.39 — GRACEFUL PDF FAIL-CLOSED GATE
# ================================================================
# A structurally invalid generated candidate is useful for diagnostics/HITL,
# but it must never be turned into a student-facing quiz PDF.
#
# Previously the PDF preflight correctly detected the problem but raised an
# exception that crashed the entire Notebook 06 process. We now block the PDF
# artifact while allowing the notebook/manifest to finish normally.
# ================================================================

pdf_review_draft_required = bool(
    not bool(manifest.get("release_ready", False))
    or (
        generation_required
        and bool(generated_payload)
        and not bool(
            generation_validation.get(
                "valid",
                False,
            )
        )
    )
)

pdf_review_warning_reasons = list(
    generation_validation.get(
        "errors",
        [],
    )
) if (
    generation_required
    and bool(generated_payload)
    and not bool(
        generation_validation.get(
            "valid",
            False,
        )
    )
) else []

# IMPORTANT:
# PDF availability is independent from release approval.
# A pending/invalid candidate gets a clearly-labelled REVIEW DRAFT,
# while release_ready remains False and HITL still controls release.
manifest["pdf_review_draft"] = bool(
    pdf_review_draft_required
)
manifest["pdf_review_warning_reasons"] = (
    pdf_review_warning_reasons
)
manifest["pdf_generation_status"] = (
    "DRAFT_PENDING_REVIEW"
    if pdf_review_draft_required
    else "FINAL_RELEASE_ARTIFACT"
)

if QUIZ_MODE == "fill_shortfall" and notebook05_student_pdf_path is not None:
    # SAME shortfall PDF strategy as the original Notebook 06:
    #   1) hybrid summary
    #   2) Official AQA Questions heading
    #   3) ORIGINAL Notebook 05 student-paper pages (NOT reconstructed text)
    #   4) AI-Generated Missing Coverage heading
    #   5) AI-generated extension pages
    #
    # Direct page insertion preserves every retrieval diagram, figure, table,
    # code crop, answer grid and official layout exactly as Notebook 05 rendered it.
    official_marks_for_pdf = safe_int(manifest.get("official_total_marks", 0))
    official_questions_for_pdf = safe_int(manifest.get("official_question_count", 0))
    ai_marks_for_pdf = sum(_pdf_question_marks(question) for question in ai_pdf_questions)
    ai_questions_for_pdf = len(ai_pdf_questions)
    combined_marks_for_pdf = official_marks_for_pdf + ai_marks_for_pdf
    combined_questions_for_pdf = official_questions_for_pdf + ai_questions_for_pdf
    target_marks_for_pdf = safe_int(manifest.get("target_marks", request.get("target_total_marks", 0)))
    target_questions_for_pdf = safe_int(
        manifest.get("target_question_count", request.get("number_of_questions", 0))
    )

    hybrid_cover_path = OUTPUT_DIR / "__hybrid_summary.pdf"
    official_heading_path = OUTPUT_DIR / "__official_questions_heading.pdf"
    ai_heading_path = OUTPUT_DIR / "__ai_questions_heading.pdf"
    marking_scheme_heading_path = OUTPUT_DIR / "__marking_schemes_heading.pdf"

    _build_hybrid_heading_pdf(
        output_path=hybrid_cover_path,
        title=(
            "Agent 2 - Hybrid Quiz Review Draft"
            if pdf_review_draft_required
            else "Agent 2 - Final Hybrid Quiz"
        ),
        subtitle=(
            "NOT RELEASE READY - pending HITL review"
            if pdf_review_draft_required
            else "Official AQA questions followed by AI-generated missing coverage"
        ),
        lines=[
            (
                f"Official retrieval: {official_marks_for_pdf}/{target_marks_for_pdf} marks | "
                f"{official_questions_for_pdf}/{target_questions_for_pdf} questions"
            ),
            (
                f"AI-generated missing coverage: +{ai_marks_for_pdf} marks | "
                f"+{ai_questions_for_pdf} question(s)"
            ),
            (
                (
                    f"Current review candidate: {combined_marks_for_pdf}/{target_marks_for_pdf} marks | "
                    f"{combined_questions_for_pdf} questions"
                    if pdf_review_draft_required
                    else
                    f"Combined after approval: {combined_marks_for_pdf}/{target_marks_for_pdf} marks | "
                    f"{combined_questions_for_pdf} questions"
                )
            ),
            f"Generation model: {manifest.get('generation_model_display_name') or manifest.get('generation_model') or GENERATION_MODEL}",
        ],
    )

    _build_hybrid_heading_pdf(
        output_path=official_heading_path,
        title="Official AQA Questions",
        subtitle="Retrieved by Notebook 05",
        lines=[
            f"Official questions: {official_questions_for_pdf}",
            f"Official marks: {official_marks_for_pdf}",
        ],
    )

    # Preserve Notebook 05's original question rendering. Its generated title
    # page is skipped when confidently detected because the final hybrid PDF
    # already has its own summary + section heading.
    official_start_page = _pdf_content_start_page(
        notebook05_student_pdf_path,
        cover_markers=[
            "Student Question Paper",
            "AQA GCSE Computer Science",
        ],
    )

    merge_segments: list[tuple[Path, int | None, int | None]] = [
        (hybrid_cover_path, 0, None),
        (official_heading_path, 0, None),
        (notebook05_student_pdf_path, official_start_page, None),
    ]

    # AI questions are rendered without their marking guidance so every
    # question appears before the single Marking Schemes section.
    ai_questions_pdf_path: Path | None = None
    ai_marking_schemes_pdf_path: Path | None = None

    if ai_pdf_questions:
        _build_hybrid_heading_pdf(
            output_path=ai_heading_path,
            title="AI-Generated Missing Coverage",
            subtitle="AQA-aligned practice questions generated by Notebook 06",
            lines=[
                f"AI-generated questions: {ai_questions_for_pdf}",
                f"AI-generated marks: {ai_marks_for_pdf}",
                "These questions remain subject to the mandatory human quality review gate.",
            ],
        )

        ai_questions_pdf_path = OUTPUT_DIR / "__ai_shortfall_questions_internal.pdf"
        build_quiz_questions_marking_scheme_pdf(
            questions=ai_pdf_questions,
            output_path=ai_questions_pdf_path,
            manifest_payload=manifest,
            allow_review_draft=True,
            include_questions=True,
            include_marking_schemes=False,
        )
        ai_questions_start_page = _pdf_content_start_page(
            ai_questions_pdf_path,
            cover_markers=[
                "AI Quiz Generation Output",
                "Generated content",
            ],
        )
        merge_segments.extend(
            [
                (ai_heading_path, 0, None),
                (ai_questions_pdf_path, ai_questions_start_page, None),
            ]
        )

        ai_marking_schemes_pdf_path = OUTPUT_DIR / "__ai_shortfall_marking_schemes_internal.pdf"
        build_quiz_questions_marking_scheme_pdf(
            questions=ai_pdf_questions,
            output_path=ai_marking_schemes_pdf_path,
            manifest_payload=manifest,
            allow_review_draft=True,
            include_questions=False,
            include_marking_schemes=True,
        )

    # One common marking-scheme section comes after ALL official + AI questions.
    # Official AQA mark schemes are preserved from Notebook 05; AI marking
    # guidance is appended immediately afterwards in the same section.
    if notebook05_teacher_pdf_path is not None or ai_marking_schemes_pdf_path is not None:
        _build_hybrid_heading_pdf(
            output_path=marking_scheme_heading_path,
            title="Marking Schemes",
            subtitle="Official AQA mark schemes followed by AI-generated marking guidance",
            lines=[
                f"Official mark schemes: {official_questions_for_pdf}",
                f"AI-generated marking schemes: {ai_questions_for_pdf}",
                f"Total marking schemes: {official_questions_for_pdf + ai_questions_for_pdf}",
            ],
        )
        merge_segments.append((marking_scheme_heading_path, 0, None))

        if notebook05_teacher_pdf_path is not None:
            official_mark_scheme_start_page = _pdf_content_start_page(
                notebook05_teacher_pdf_path,
                cover_markers=[
                    "Teacher Mark Scheme",
                    "AQA GCSE Computer Science",
                ],
            )
            merge_segments.append(
                (
                    notebook05_teacher_pdf_path,
                    official_mark_scheme_start_page,
                    None,
                )
            )

        if ai_marking_schemes_pdf_path is not None:
            ai_marking_scheme_start_page = _pdf_content_start_page(
                ai_marking_schemes_pdf_path,
                cover_markers=[
                    "AI Quiz Generation Output",
                    "Generated content",
                ],
            )
            merge_segments.append(
                (
                    ai_marking_schemes_pdf_path,
                    ai_marking_scheme_start_page,
                    None,
                )
            )

    _merge_pdf_segments(
        merge_segments,
        quiz_pdf_path,
    )

    # Only ONE student-facing PDF is registered for Notebook 06 shortfall mode.
    manifest.setdefault("output_files", {})[
        "questions_and_marking_schemes_pdf"
    ] = str(quiz_pdf_path)
    manifest.setdefault("output_files", {}).pop(
        "official_retrieval_student_pdf",
        None,
    )
    manifest.setdefault("output_files", {}).pop(
        "ai_shortfall_extension_pdf",
        None,
    )

    # Source artifacts remain auditable/internal but are not exposed as final
    # student-facing PDFs by the Streamlit shortfall UI.
    manifest.setdefault("source_artifacts", {})[
        "official_retrieval_student_pdf"
    ] = str(notebook05_student_pdf_path)

    if "notebook05_pdf_resolution_path" in globals():
        manifest.setdefault("source_artifacts", {})[
            "notebook05_student_pdf_resolution"
        ] = str(notebook05_pdf_resolution_path)
    # Backward-compatible key now points to the AI question-only internal PDF.
    manifest.setdefault("source_artifacts", {})[
        "ai_shortfall_internal_pdf"
    ] = str(ai_questions_pdf_path) if ai_questions_pdf_path is not None else None
    manifest.setdefault("source_artifacts", {})[
        "ai_shortfall_questions_internal_pdf"
    ] = str(ai_questions_pdf_path) if ai_questions_pdf_path is not None else None
    manifest.setdefault("source_artifacts", {})[
        "ai_shortfall_marking_schemes_internal_pdf"
    ] = (
        str(ai_marking_schemes_pdf_path)
        if ai_marking_schemes_pdf_path is not None
        else None
    )
    manifest.setdefault("source_artifacts", {})[
        "questions_and_marking_schemes_pdf"
    ] = str(quiz_pdf_path)

    manifest["pdf_review_draft"] = bool(
        pdf_review_draft_required
    )
    manifest["pdf_generation_status"] = (
        "DRAFT_SAVED_PENDING_REVIEW"
        if pdf_review_draft_required
        else "FINAL_SAVED"
    )

    manifest["official_pdf_rendering"] = {
        "mode": "ORIGINAL_NOTEBOOK05_PDF_PAGES",
        "source_pdf": str(notebook05_student_pdf_path),
        "source_page_count": _pdf_page_count(
            notebook05_student_pdf_path
        ),
        "diagrams_preserved_by_page_merge": True,
        "text_reconstruction_used": False,
    }

    manifest["hybrid_pdf_structure"] = {
        "single_student_facing_pdf": True,
        "section_order": [
            "final_hybrid_summary",
            "official_aqa_questions",
            "ai_generated_missing_coverage",
            "marking_schemes",
        ],
        "marking_scheme_order": [
            "official_aqa_mark_schemes",
            "ai_generated_marking_guidance",
        ],
        "official_source_pdf": str(notebook05_student_pdf_path),
        "final_pdf": str(quiz_pdf_path),
    }

    manifest_path.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False,
            default=str,
        ),
        encoding="utf-8",
    )

    # Remove temporary student-facing-looking PDF fragments after the final
    # merge. The original Notebook 05 source remains untouched in its own run.
    for temp_path in [
        hybrid_cover_path,
        official_heading_path,
        ai_heading_path,
        marking_scheme_heading_path,
        ai_questions_pdf_path,
        ai_marking_schemes_pdf_path,
    ]:
        if temp_path is not None and Path(temp_path).is_file():
            try:
                Path(temp_path).unlink()
            except OSError:
                pass

    print(
        (
            "Saved hybrid REVIEW DRAFT PDF:"
            if pdf_review_draft_required
            else "Saved ONE final hybrid PDF:"
        ),
        quiz_pdf_path,
    )

else:
    # Complete-quiz behavior is unchanged.
    #
    # Shortfall fallback is REVIEW-ONLY. The preferred/proper shortfall export
    # is the original-Notebook05-page merge above. If the original Notebook 05
    # PDF cannot be found we keep the preview available (per HITL requirement),
    # but record loudly that official diagrams/layout could not be preserved.
    if QUIZ_MODE == "fill_shortfall":
        manifest["official_pdf_rendering"] = {
            "mode": "TEXT_RECONSTRUCTION_REVIEW_FALLBACK",
            "source_pdf": None,
            "source_page_count": 0,
            "diagrams_preserved_by_page_merge": False,
            "text_reconstruction_used": True,
            "warning": (
                "Notebook 05 original student PDF could not be resolved. "
                "This is a review fallback and may omit retrieval diagrams."
            ),
        }
        manifest["pdf_review_draft"] = True
        manifest["pdf_generation_status"] = (
            "DRAFT_OFFICIAL_SOURCE_PDF_MISSING"
        )

    # Existing complete_quiz behaviour, plus a review fallback if an older
    # Notebook 05 package genuinely has no resolvable official student PDF.
    if (
        QUIZ_MODE == "fill_shortfall"
        and candidate_generated_questions_enriched
        and not generated_quality_accepted
    ):
        pdf_source_questions = (
            [
                enrich_question_paper_metadata(question)
                for question in official_questions
                if isinstance(question, dict)
            ]
            + list(candidate_generated_questions_enriched)
        )
    elif final_questions:
        pdf_source_questions = list(final_questions)
    else:
        pdf_source_questions = list(display_questions)

    pdf_questions = [
        question
        for question in pdf_source_questions
        if isinstance(question, dict)
    ]

    if pdf_questions:
        try:
            build_quiz_questions_marking_scheme_pdf(
                questions=pdf_questions,
                output_path=quiz_pdf_path,
                manifest_payload=manifest,
                allow_review_draft=True,
            )
        except RuntimeError as exc:
            if "PDF visual-integrity preflight failed:" not in str(exc):
                raise

            manifest["release_ready"] = False
            manifest["pdf_generation_status"] = (
                "DRAFT_RENDER_FAILED"
            )
            manifest["pdf_generation_block_reason"] = str(exc)

            manifest.setdefault("output_files", {}).pop(
                "questions_and_marking_schemes_pdf",
                None,
            )
            manifest.setdefault("source_artifacts", {}).pop(
                "questions_and_marking_schemes_pdf",
                None,
            )

            if quiz_pdf_path.is_file():
                try:
                    quiz_pdf_path.unlink()
                except OSError:
                    pass

            manifest_path.write_text(
                json.dumps(
                    manifest,
                    indent=2,
                    ensure_ascii=False,
                    default=str,
                ),
                encoding="utf-8",
            )

            print()
            print("QUIZ REVIEW PDF RENDER WARNING")
            print(str(exc))
            print(
                "Notebook 06 will finish normally. "
                "No invalid student-facing PDF was created."
            )
            print()

        else:
            manifest.setdefault("output_files", {})[
                "questions_and_marking_schemes_pdf"
            ] = str(quiz_pdf_path)
            manifest.setdefault("source_artifacts", {})[
                "questions_and_marking_schemes_pdf"
            ] = str(quiz_pdf_path)

            manifest_path.write_text(
                json.dumps(
                    manifest,
                    indent=2,
                    ensure_ascii=False,
                    default=str,
                ),
                encoding="utf-8",
            )

            manifest["pdf_generation_status"] = "SAVED"

            print("Saved questions + marking schemes PDF:", quiz_pdf_path)
    else:
        print(
            "Quiz PDF was not generated because no candidate or accepted "
            "questions are available in the current run."
        )


# Final Streamlit contract

The frontend can call **this same Notebook 06** in two different modes.

## Button 1 — Generate Complete Quiz

Set:

```text
AGENT2_QUIZ_MODE=complete_quiz
AGENT2_AGENT1_TOPICS_JSON=<approved Agent 1 topic list>
AGENT2_ASSESSMENT_REQUEST_JSON=<same filter dictionary>
AGENT2_LESSON_SUMMARY=<lesson evidence>
AGENT2_QUIZ_USER_APPROVED_GENERATION=1
AGENT2_QUIZ_RUN_GENERATION=1
```

Result:

```text
GENERATED_ONLY
```

**Notebook 05 is never loaded in this branch.**

---

## Button 2 — Generate Missing Quiz Coverage with AI

This button is shown only after Notebook 05 has returned an insufficient official assessment.

Set:

```text
AGENT2_QUIZ_MODE=fill_shortfall
AGENT2_NOTEBOOK05_PACKAGE_PATH=<exact current package>
AGENT2_NOTEBOOK05_SELECTED_CSV_PATH=<exact matching selected CSV>
AGENT2_QUIZ_USER_APPROVED_GENERATION=1
AGENT2_QUIZ_RUN_GENERATION=1
```

Result:

```text
OFFICIAL_ONLY
```

when official retrieval was already sufficient, otherwise:

```text
HYBRID
```

after accepted AI shortfall generation.

In `fill_shortfall`, `target_total_marks` and `number_of_questions` are minimum
coverage targets. If either target is still short, the same config-selected
Notebook 06 generation model completes only the missing coverage. A marks-only
shortfall may add one or more questions beyond the requested question count; a
question-count-only shortfall may minimally exceed the requested mark total.
Model selection uses the same priority as complete-quiz mode: environment
override -> current-run `quiz_model_selection.json` -> config default.

---

## Human REVIEW rerun

For a semantic `REVIEW`, rerun the same request with generation disabled and one of:

```text
AGENT2_QUIZ_REVIEW_DECISION=approve
AGENT2_QUIZ_REVIEW_REASON=<required reason>
```

or:

```text
AGENT2_QUIZ_REVIEW_DECISION=regenerate
AGENT2_QUIZ_REVIEW_REASON=<required reason>
```

or:

```text
AGENT2_QUIZ_REVIEW_DECISION=reject
AGENT2_QUIZ_REVIEW_REASON=<required reason>
```

Keep:

```text
AGENT2_QUIZ_RUN_GENERATION=0
```

The candidate is restored only when its generation-request fingerprint matches the current request.

### Visual fields exposed to Streamlit / PDF

Generated candidate/final question objects may now include `visual_requirement`, `visual`, `visual_path`, `visual_render_status`, `visual_release_eligible`, `visual_asset_sha256`, `visual_schema_version`, `visual_renderer`, `visual_architecture_phase`, and an optional deterministic `response_scaffold`. For a learner task that says to complete a truth table while the supplied stimulus is a different visual (for example a logic circuit), `response_scaffold` carries the blank truth-table columns/rows used by the PDF/frontend. Existing text-only consumers remain compatible because `visual_path=None` when no visual is required.

### Automatic PDF artifact

Every run with visible quiz content now also writes:

```text
Agent2_Quiz_Output_Questions_and_Marking_Schemes.pdf
```

The path is recorded in `final_quiz_manifest.json` as:

```text
output_files.questions_and_marking_schemes_pdf
source_artifacts.questions_and_marking_schemes_pdf
```

This PDF export is presentation-only and never changes HITL, structural validation, semantic validation, or `release_ready`.



### Paper-routing and AQA efficiency-scope preflight

Before quiz generation, Notebook 06 now checks the selected paper against approved Agent 1 topic references. An all-paper mismatch, or a selected paper with no compatible primary topic, stops before any LLM/API call and writes `paper_routing_preflight.json` with a user-facing message. Mixed-paper inputs continue using only compatible topics and record which topics were excluded.

Generated AQA 8525 questions also remain within the specification's time-efficiency scope: qualitative comparison and concrete operation/pass/swap counts are allowed, while formal Big-O/asymptotic or space-complexity requirements trigger regeneration.

### Special-instruction contract — v2.29

`AGENT2_ASSESSMENT_REQUEST_JSON.special_instructions` is passed as raw natural
language to the **same selected generation-model call**. Notebook 06 does not
make a separate instruction-interpreter LLM request.

For `complete_quiz`, explicit deterministic hints such as
`only/exactly/remaining` role counts can be promoted into the blueprint before
generation. Other wording is left for the LLM to interpret semantically.

The generated candidate now records:
- `instruction_interpretation`
- `special_instruction_compliance`

in the candidate payload, HITL record, final manifest and text report. Objective
constraints are still independently checked by Python; qualitative
special-instruction compliance requires HITL confirmation.



---

## Notebook 08 visual-tool handoff — v2.35

Notebook 06 additionally writes:

```text
visual_tool_handoff.json
```

The handoff contains:
- generated question ID;
- question text/topic/reference;
- `visual_requirement`;
- compact validated `visual.spec`;
- one of three renderer/tool routing hints;
- the legacy Notebook 06 visual path only for migration comparison.

The canonical renderer notebook is:

```text
08_visual_generation_tool_layer.ipynb
```

Current pre-MCP path:

```text
Notebook 06
   ↓
visual_tool_handoff.json
   ↓
Notebook 08 router
   ↓
   ├─ logic circuit → SchemDraw
   ├─ technical diagram → Kroki → GraphViz
   └─ structured visual → local deterministic renderer
```

Future MCP path:

```text
Controller / LLM
   ↓
MCP tool selection
   ↓
render_logic_visual()
render_technical_visual()
render_structured_visual()
```

**No MCP implementation has been added in v2.35.**
